# Observable deep hedging under calibrated Heston and Lifted Heston

## Configurable lag-consistent final research notebook: H = 0.35 or H = 0.25

This notebook implements one research pipeline under two calibrated Lifted-Heston specifications. The H = 0.35 mode loads the lag-consistent synthetic banks and trained checkpoints already produced by the current run, then regenerates evaluation, inference, diagnostics and dissertation exports. The H = 0.25 mode loads the separately calibrated H = 0.25 cache, rebuilds all dependent observable banks and retrains the same policy cells under the same economic and machine-learning protocol.

The daily-data timing contract is fixed in both modes:

1. observe the completed end-of-day option-quote snapshot and CRSP index level at date \(t\);
2. construct policy features using only information available by that snapshot, with any explicit sequence ending at \(t-1\);
3. compute the requested hedge after the date-\(t\) snapshot;
4. execute that request at the next recorded daily underlying observation, \(t+1\);
5. apply the same delay in simulation pretraining, historical fine-tuning, validation, benchmark selection and evaluation.

The next CRSP index level is a transparent daily execution proxy. It is not described as a timestamped tradable SPX fill, because the data stubbornly decline to contain information they never recorded.

## Binding estimands and confirmatory contrast family

All effects are reported as paired relative-MSE improvements, with positive values favouring the first policy named.

1. **Complete source-pipeline effect:** Lifted-Heston-only small current MLP versus Heston-only small current MLP.
2. **Fixed-support source-composition effect:** fixed-size 50:50 mixed small current MLP versus Heston-only small current MLP.
3. **Capacity effect (secondary identification diagnostic):** matched current MLP versus small current MLP.
4. **Incremental supplied-history effect:** lag-stack MLP versus capacity-matched current MLP.
5. **Recurrent-processing effect:** GRU versus lag-stack MLP with the identical history and mask.
6. **Complete sequence-policy effect:** GRU versus capacity-matched current MLP.
7. **Selected neural policy versus Black-Scholes delta:** the neural policy selected only on 2020 validation is compared with the analytic benchmark.

The six confirmatory hypotheses are items 1, 2, 4, 5, 6 and 7. The capacity contrast is reported separately because it diagnoses the architecture comparison rather than testing the economic source or sequence hypotheses. A null interval is inconclusive, not equivalence, sufficiency or proof of no effect.


## Binding design and reporting amendments

The notebook enforces the following research design in both calibration modes.

- Latent Heston variance and Lifted-Heston factors may generate synthetic prices but never enter policy tensors, scalers, recurrent initialisation, targets or model selection.
- Heston, Lifted Heston and the fixed-size 50:50 mixture have equal total simulated support.
- The architecture ladder contains a small current-feature MLP, a parameter-matched current-feature MLP, a ten-observation lagged-input MLP and a ten-observation GRU encoder.
- The lagged-input MLP and GRU receive the identical history and mask.
- Every policy predicts a bounded correction to Black-Scholes straddle delta.
- The reporting layer exports every seed, selected stage, date-level contrast, maturity/cost/year diagnostic, feature-clipping rate, action-bound saturation rate and Heston-pricer fallback count.
- The lower-tail training term is reported accurately as a minibatch worst-outcome squared penalty. With batch size 16 and a 5% tail fraction, it acts on one episode.
- The 2021-2024 panel is labelled post-development frozen evaluation rather than an analyst-blind holdout.

The H = 0.35 mode preserves the existing checkpoint protocol so the currently trained models can be loaded without being invalidated by reporting-only amendments. H = 0.25 receives its own protocol namespace and may use an expanded common-seed panel.

In [1]:
from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import random
import re
import shutil
import time
import warnings
import zipfile
import zlib
from functools import lru_cache
from datetime import datetime, timezone
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path
from typing import Any, Callable, Iterable, Literal

import numpy as np
import pandas as pd
from IPython.display import display

try:
    import pyarrow.dataset as ds
    import pyarrow.parquet as pq
    PYARROW_AVAILABLE = True
except ImportError as exc:  # optional dependency
    PYARROW_AVAILABLE = False
    ds = None
    pq = None
    warnings.warn(f"PyArrow unavailable; memory-safe parquet streaming is disabled: {exc}")

try:
    from scipy.optimize import brentq, lsq_linear
    from scipy.special import gamma
    SCIPY_AVAILABLE = True
except ImportError as exc:  # optional dependency
    SCIPY_AVAILABLE = False
    warnings.warn(f"SciPy unavailable: {exc}")

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except ImportError as exc:  # optional dependency
    MATPLOTLIB_AVAILABLE = False
    warnings.warn(f"Matplotlib unavailable: {exc}")

try:
    from sklearn.decomposition import PCA
    from sklearn.linear_model import Ridge
    from sklearn.metrics import r2_score
    SKLEARN_AVAILABLE = True
except ImportError as exc:  # optional dependency
    SKLEARN_AVAILABLE = False
    warnings.warn(f"scikit-learn unavailable: {exc}")

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    TORCH_AVAILABLE = True
except ImportError as exc:  # optional dependency
    TORCH_AVAILABLE = False
    torch = None
    nn = None
    F = None
    warnings.warn(f"PyTorch unavailable: {exc}")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
warnings.filterwarnings("default")

try:
    # Use the standard text progress bar. It works in notebooks and terminals
    # without requiring the optional Jupyter widget extension.
    from tqdm.std import tqdm
except ImportError:  # optional progress display
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else []


# Run modes and safe switching

Set only the environment variables in the configuration cell below.

### Reuse the completed H = 0.35 lag-consistent models

```text
FINAL_CALIBRATION_SPEC=H035
FINAL_REBUILD_FROM=reporting
FINAL_RUN_PROFILE=full
```

Use `evaluation` instead of `reporting` when the episode-level evaluation export is absent or must be regenerated. The default H = 0.35 seed panel is the three seeds used by the current run.

### Run the final H = 0.25 specification

```text
FINAL_CALIBRATION_SPEC=H025
FINAL_REBUILD_FROM=banks
FINAL_RUN_PROFILE=full
FINAL_TRAINING_SEEDS=20260710,20260711,20260712,20260713,20260714,20260715,20260716,20260717,20260718,20260719
```

H = 0.25 reuses the separately calibrated cache but rebuilds every dependent synthetic bank, scaler, checkpoint and evaluation artefact. `auto` resumes valid partial work. `policies` reuses valid banks and retrains missing or invalid checkpoints. Calibration is frozen by default in this notebook; set `ALLOW_UPSTREAM_REBUILD=1` only when deliberately rebuilding the source data and calibrations in the same environment.

The two modes write to separate cache and output namespaces. Apparently even files deserve protection from accidental identity theft.

In [2]:
# ============================================================
# USER CONFIGURATION — THIS IS THE ONLY CELL TO EDIT
# ============================================================

@dataclass(frozen=True)
class RunRequest:
    profile: Literal["smoke", "full"] = "full"
    calibration_spec: Literal["preliminary_h035", "final_h025"] = "preliminary_h035"
    rebuild_from: Literal[
        "auto", "reuse", "surface", "hedging_panel", "calibration",
        "feasibility", "banks", "policies", "evaluation", "reporting",
    ] = "reporting"
    project_root: Path = Path(".")
    data_dir: Path = Path("")
    cache_dir_override: Path | None = None
    output_dir_override: Path | None = None
    legacy_hedging_panel_dir: Path | None = None
    compatibility_mode: Literal["none", "legacy_h035_v53_reuse"] = "none"
    allow_upstream_rebuild: bool = False

    # Retained only for inherited helper compatibility. This notebook does not
    # import the old immediate-execution policy checkpoints.
    legacy_h035_output_dir: Path = Path("outputs_observable_preliminary_h035_full")
    legacy_h035_output_zip: Path = Path("outputs_observable_preliminary_h035_full.zip")
    auto_extract_legacy_output: bool = False
    legacy_h035_expected_protocol_hash: str = "f1be3a61adddefac6b8bc9f4bd219fd164151950b8e2c80fc4a626ea4224ba85"

    @property
    def calibration_H(self) -> float:
        return 0.35 if self.calibration_spec == "preliminary_h035" else 0.25

    @property
    def calibration_tag(self) -> str:
        return "H035" if self.calibration_spec == "preliminary_h035" else "H025"

    @property
    def is_final_spec(self) -> bool:
        # Keep False for H035 to preserve the protocol identity of the models
        # currently being trained. H025 is the final calibrated specification.
        return self.calibration_spec == "final_h025"

    @property
    def is_legacy_h035_reuse(self) -> bool:
        return self.compatibility_mode == "legacy_h035_v53_reuse"

    @property
    def cache_dir(self) -> Path:
        if self.cache_dir_override is not None:
            return Path(self.cache_dir_override)
        return Path("cache_final" if self.calibration_spec == "preliminary_h035" else "cache_final_H025")

    @property
    def output_dir(self) -> Path:
        if self.output_dir_override is not None:
            return Path(self.output_dir_override)
        if self.calibration_spec == "preliminary_h035":
            return Path(f"outputs_observable_h035_lag_consistent_retrained_{self.profile}")
        return Path(f"outputs_observable_h025_lag_consistent_retrained_{self.profile}")


def _optional_env_path(name: str) -> Path | None:
    value = os.environ.get(name, "").strip()
    return Path(value) if value else None


def _env_flag(name: str, default: bool = False) -> bool:
    value = os.environ.get(name, "").strip().lower()
    if not value:
        return bool(default)
    if value in {"1", "true", "yes", "on"}:
        return True
    if value in {"0", "false", "no", "off"}:
        return False
    raise ValueError(f"{name} must be a boolean flag; received {value!r}.")


def _training_seeds_from_env(default: tuple[int, ...]) -> tuple[int, ...]:
    raw = os.environ.get("FINAL_TRAINING_SEEDS", "").strip()
    if not raw:
        return tuple(int(seed) for seed in default)
    seeds = tuple(int(part.strip()) for part in raw.split(",") if part.strip())
    if not seeds:
        raise ValueError("FINAL_TRAINING_SEEDS was supplied but contained no integer seeds.")
    if len(set(seeds)) != len(seeds):
        raise ValueError("FINAL_TRAINING_SEEDS contains duplicate seeds.")
    return seeds


_SPEC_TOKEN = os.environ.get("FINAL_CALIBRATION_SPEC", "H035").strip().upper().replace("=", "")
if _SPEC_TOKEN in {"H035", "035", "0.35", "PRELIMINARY_H035"}:
    _CALIBRATION_SPEC = "preliminary_h035"
    _DEFAULT_REBUILD = "reporting"
    _DEFAULT_SEEDS = (20260710, 20260711, 20260712)
    _CACHE_ENV = "H035_CACHE_DIR"
    _OUTPUT_ENV = "H035_LAG_OUTPUT_DIR"
elif _SPEC_TOKEN in {"H025", "025", "0.25", "FINAL_H025"}:
    _CALIBRATION_SPEC = "final_h025"
    _DEFAULT_REBUILD = "banks"
    _DEFAULT_SEEDS = tuple(range(20260710, 20260720))
    _CACHE_ENV = "H025_CACHE_DIR"
    _OUTPUT_ENV = "H025_LAG_OUTPUT_DIR"
else:
    raise ValueError("FINAL_CALIBRATION_SPEC must be H035 or H025.")

RUN = RunRequest(
    profile=os.environ.get("FINAL_RUN_PROFILE", "full").strip().lower(),
    calibration_spec=_CALIBRATION_SPEC,
    rebuild_from=os.environ.get("FINAL_REBUILD_FROM", _DEFAULT_REBUILD).strip().lower(),
    project_root=Path(os.environ.get("PROJECT_ROOT", ".")),
    data_dir=Path(os.environ.get("DATA_DIR", "")),
    cache_dir_override=_optional_env_path(_CACHE_ENV),
    output_dir_override=_optional_env_path(_OUTPUT_ENV),
    legacy_hedging_panel_dir=_optional_env_path("LEGACY_HEDGING_PANEL_DIR"),
    compatibility_mode="none",
    auto_extract_legacy_output=False,
    allow_upstream_rebuild=_env_flag("ALLOW_UPSTREAM_REBUILD", False),
)

_ALLOWED_PROFILES = {"smoke", "full"}
_ALLOWED_REBUILD = {"auto", "reuse", "surface", "hedging_panel", "calibration", "feasibility", "banks", "policies", "evaluation", "reporting"}
if RUN.profile not in _ALLOWED_PROFILES:
    raise ValueError(f"profile must be one of {sorted(_ALLOWED_PROFILES)}; got {RUN.profile!r}.")
if RUN.rebuild_from not in _ALLOWED_REBUILD:
    raise ValueError(f"rebuild_from must be one of {sorted(_ALLOWED_REBUILD)}; got {RUN.rebuild_from!r}.")
if RUN.compatibility_mode != "none":
    raise AssertionError("Immediate-execution legacy checkpoint import is disabled in the final notebook.")
if RUN.rebuild_from in {"evaluation", "reporting", "reuse"}:
    print("Checkpoint reuse mode: no policy retraining will occur unless an artefact is missing and auto rebuilding is enabled.")

RUN_PROFILES = {
    "smoke": {
        "training_seeds": _training_seeds_from_env((_DEFAULT_SEEDS[0],)),
        "transaction_cost_bps": (1.0,),
        "source_train_dates_per_bucket": 1,
        "source_validation_dates_per_bucket": 1,
        "exact_train_paths_per_seed_row": 1,
        "exact_validation_paths_per_seed_row": 1,
        "market_max_cohorts_per_bucket_per_split": 1,
        "common_pretrain_steps": 3,
        "hybrid_head_steps": 1,
        "hybrid_full_steps": 2,
        "batch_episodes": 2,
        "validation_interval": 1,
        "early_stopping_patience": 2,
        "bootstrap_repetitions": 50,
        "bootstrap_sensitivity_weeks": (2, 4),
    },
    "full": {
        "training_seeds": _training_seeds_from_env(_DEFAULT_SEEDS),
        "transaction_cost_bps": (1.0, 5.0, 10.0),
        "source_train_dates_per_bucket": 20,
        "source_validation_dates_per_bucket": 8,
        "exact_train_paths_per_seed_row": 16,
        "exact_validation_paths_per_seed_row": 8,
        "market_max_cohorts_per_bucket_per_split": None,
        "common_pretrain_steps": 1800,
        "hybrid_head_steps": 250,
        "hybrid_full_steps": 750,
        "batch_episodes": 16,
        "validation_interval": 50,
        "early_stopping_patience": 12,
        "bootstrap_repetitions": 10_000,
        "bootstrap_sensitivity_weeks": (4, 8, 13, 26),
    },
}
RUN_PROFILE = dict(RUN_PROFILES[RUN.profile])

FINAL_RUN_PROFILE = RUN.profile
IS_SMOKE = RUN.profile == "smoke"
IS_FULL = RUN.profile == "full"
IS_FINAL_H025_SPEC = RUN.calibration_spec == "final_h025"
IS_PRELIMINARY_H035 = RUN.calibration_spec == "preliminary_h035"  # cache/protocol label only
LEGACY_H035_REUSE = False

# Preserve episode-weighted selection for the already-trained H035 checkpoints.
# H025 uses the headline equal-maturity-within-date, equal-date validation criterion.
MODEL_SELECTION_WEIGHTING = "episode_weighted" if IS_PRELIMINARY_H035 else "equal_maturity_within_date_then_equal_date"

print({
    "profile": RUN.profile,
    "calibration_spec": RUN.calibration_spec,
    "calibration_H": RUN.calibration_H,
    "rebuild_from": RUN.rebuild_from,
    "cache_dir": str(RUN.cache_dir),
    "output_dir": str(RUN.output_dir),
    "training_seeds": RUN_PROFILE["training_seeds"],
    "model_selection_weighting": MODEL_SELECTION_WEIGHTING,
    "execution_design": "observe EOD t; decide after snapshot; execute next recorded node t+1",
})


Checkpoint reuse mode: no policy retraining will occur unless an artefact is missing and auto rebuilding is enabled.
{'profile': 'full', 'calibration_spec': 'preliminary_h035', 'calibration_H': 0.35, 'rebuild_from': 'reporting', 'cache_dir': 'cache_final', 'output_dir': 'outputs_observable_h035_lag_consistent_retrained_full', 'training_seeds': (20260710, 20260711, 20260712), 'model_selection_weighting': 'episode_weighted', 'execution_design': 'observe EOD t; decide after snapshot; execute next recorded node t+1'}


## Timing contract and rerun boundary

**Observation time:** completed end-of-day OptionMetrics quote snapshot and CRSP index level at date \(t\).

**Feature availability:** current features use information available by that snapshot; explicit sequence inputs contain only dates strictly earlier than \(t\).

**Decision time:** immediately after constructing the date-\(t\) observable information set.

**Execution time:** the next recorded daily underlying observation, \(t+1\).

**Consistency requirement:** the same one-observation delay is applied in simulation pretraining, simulation validation, historical fine-tuning, 2020 model selection, benchmark tuning and final evaluation.

**Reused:** validated market data, historical episode construction inputs and H = 0.35 calibration artefacts.

**Rebuilt:** synthetic observable banks, scalers, all neural checkpoints, validation selections, benchmark selection, evaluation tables, inference and figures.


In [3]:

# ============================================================
# Dissertation-wide Matplotlib theme, visual grammar, and save/show contract
# ============================================================

# The theme is deliberately print-oriented: thin lines, small legends, high-contrast
# colour-blind-safe model colours and explicit line styles. It is not grayscale, because
# apparently dissertation figures should not resemble security-camera footage of a spreadsheet.
DISSERTATION_MPL_STYLE = {
    "figure.figsize": (7.2, 4.2),
    "figure.dpi": 130,
    "savefig.dpi": 320,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.06,
    "font.size": 9.0,
    "axes.titlesize": 10.0,
    "axes.labelsize": 9.0,
    "xtick.labelsize": 8.0,
    "ytick.labelsize": 8.0,
    "legend.fontsize": 6.8,
    "legend.title_fontsize": 7.2,
    "legend.frameon": True,
    "legend.framealpha": 0.88,
    "legend.edgecolor": "#cbd5e1",
    "legend.borderpad": 0.25,
    "legend.handlelength": 1.65,
    "legend.handletextpad": 0.45,
    "legend.labelspacing": 0.28,
    "legend.columnspacing": 0.75,
    "lines.linewidth": 1.05,
    "lines.markersize": 3.2,
    "lines.markeredgewidth": 0.55,
    "patch.linewidth": 0.55,
    "axes.linewidth": 0.65,
    "axes.grid": True,
    "grid.alpha": 0.18,
    "grid.linewidth": 0.42,
    "grid.color": "#94a3b8",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.axisbelow": True,
    "axes.facecolor": "white",
    "figure.facecolor": "white",
    "axes.titlepad": 6.0,
    "axes.labelpad": 4.0,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 3.0,
    "ytick.major.size": 3.0,
}

# Okabe-Ito-inspired palette, extended for policy families. Only no-hedge stays neutral.
MODEL_ALPHA = 0.80
BAR_ALPHA = 0.78
SCATTER_ALPHA = 0.72

# Light dissertation palette reused consistently throughout the notebook.
MODEL_COLORS = {
    "market": "#5B6472",
    "observed": "#5B6472",
    "heston": "#7BB6E6",
    "lifted": "#F3A26D",
    "lifted_heston": "#F3A26D",
    "black_scholes": "#7CCBA2",
    "bs_delta": "#7CCBA2",
    "neural_scalar": "#C8A2D8",
    "neural_factors": "#F4C06A",
    "gru": "#79C8C2",
    "lstm": "#9DB4E0",
    "no_hedge": "#B5BDC8",
    "no_trade_band": "#D8B07C",
    "model_delta": "#74B8B0",
    "residual_negative": "#6FA8DC",
    "residual_positive": "#F4A261",
}

MODEL_MARKERS = {
    "market": "o",
    "observed": "o",
    "heston": "s",
    "lifted": "^",
    "lifted_heston": "^",
    "black_scholes": "D",
    "bs_delta": "D",
    "neural_scalar": "P",
    "neural_factors": "X",
    "gru": "v",
    "lstm": "<",
    "no_hedge": "x",
    "no_trade_band": "h",
    "model_delta": "d",
}

MODEL_LINESTYLES = {
    "market": "None",
    "observed": "None",
    "heston": "-",
    "lifted": "-",
    "lifted_heston": "-",
    "black_scholes": "-",
    "bs_delta": "-",
    "neural_scalar": "-",
    "neural_factors": "-",
    "gru": "-",
    "lstm": "-",
    "no_hedge": "-",
    "no_trade_band": "-",
    "model_delta": "-",
}

POLICY_DISPLAY_ORDER = [
    "no_hedge",
    "black_scholes",
    "bs_delta",
    "model_delta",
    "no_trade_band",
    "heston",
    "lifted_heston",
    "neural_scalar",
    "neural_factors",
    "gru",
    "lstm",
]

MODEL_DISPLAY = globals().get("MODEL_DISPLAY", {}) | {
    "market": "Market",
    "observed": "Observed market",
    "heston": "Heston",
    "lifted": "Lifted Heston",
    "lifted_heston": "Lifted Heston",
    "black_scholes": "Black-Scholes",
    "bs_delta": "Black-Scholes delta",
    "neural_scalar": "Neural scalar",
    "neural_factors": "Neural factors",
    "gru": "GRU",
    "lstm": "LSTM",
    "no_hedge": "No hedge",
    "no_trade_band": "No-trade band",
    "model_delta": "Model delta",
}

try:
    from matplotlib.colors import LinearSegmentedColormap
except Exception:
    LinearSegmentedColormap = None

if LinearSegmentedColormap is not None:
    BLUE_ORANGE_SEQ = LinearSegmentedColormap.from_list("blue_orange_seq", ["#EAF2FB", "#BFD7EE", "#7BB6E6", "#F5CBA7", "#F3A26D"])
    BLUE_ORANGE_DIVERGING = LinearSegmentedColormap.from_list("blue_orange_div", ["#7BB6E6", "#EAF2FB", "#FFF7ED", "#F8C7A4", "#F3A26D"])
else:
    BLUE_ORANGE_SEQ = "coolwarm"
    BLUE_ORANGE_DIVERGING = "coolwarm"

SEQUENTIAL_CMAP = BLUE_ORANGE_SEQ
SIGNED_RESIDUAL_CMAP = BLUE_ORANGE_DIVERGING
COUNT_CMAP = BLUE_ORANGE_SEQ
ERROR_CMAP = BLUE_ORANGE_SEQ
WINRATE_CMAP = BLUE_ORANGE_DIVERGING


def _normalise_style_key(name: object) -> str:
    s = str(name).strip().lower().replace("-", "_").replace(" ", "_")
    if "lifted" in s:
        return "lifted_heston"
    if "heston" in s:
        return "heston"
    if "black" in s or s in {"bs", "bs_delta"}:
        return "black_scholes"
    if "no_trade" in s or "band" in s:
        return "no_trade_band"
    if "no_hedge" in s or s == "none":
        return "no_hedge"
    if "gru" in s:
        return "gru"
    if "lstm" in s:
        return "lstm"
    if "factor" in s:
        return "neural_factors"
    if "neural" in s or "scalar" in s:
        return "neural_scalar"
    if "market" in s or "observed" in s:
        return "market"
    if "delta" in s:
        return "model_delta"
    return s


def model_color(name: object, default: str = "#4b5563") -> str:
    return MODEL_COLORS.get(_normalise_style_key(name), default)


def model_marker(name: object, default: str = "o") -> str:
    return MODEL_MARKERS.get(_normalise_style_key(name), default)


def model_linestyle(name: object, default: str = "-") -> str | tuple:
    return MODEL_LINESTYLES.get(_normalise_style_key(name), default)


def set_dissertation_plot_theme() -> None:
    """Apply the notebook-wide academic Matplotlib style."""
    if not MATPLOTLIB_AVAILABLE:
        return
    plt.rcParams.update(DISSERTATION_MPL_STYLE)
    try:
        from cycler import cycler
        cycle = [
            MODEL_COLORS["market"],
            MODEL_COLORS["heston"],
            MODEL_COLORS["lifted_heston"],
            MODEL_COLORS["black_scholes"],
            MODEL_COLORS["neural_scalar"],
            MODEL_COLORS["neural_factors"],
            MODEL_COLORS["gru"],
            MODEL_COLORS["lstm"],
            MODEL_COLORS["no_trade_band"],
        ]
        plt.rcParams["axes.prop_cycle"] = cycler(color=cycle)
    except Exception:
        pass



try:
    from matplotlib.colors import TwoSlopeNorm, Normalize
except Exception:
    TwoSlopeNorm = None
    Normalize = None

FIGURE_ARTIFACT_REGISTRY: list[dict[str, Any]] = []


def register_figure_artifact(
    path: Path | str,
    *,
    figure_id: str | None = None,
    description: str = "",
    notebook_section: str = "",
    source_data: str = "",
    displayed_in_notebook: bool = True,
    recommendation: str = "main/appendix",
    notes: str = "",
) -> Path:
    p = Path(path)
    row = {
        "figure_id": figure_id or p.stem,
        "description": description or p.stem.replace("_", " "),
        "notebook_section": notebook_section,
        "source_data_table_cache": source_data,
        "saved_filename": str(p),
        "displayed_in_notebook": "yes" if displayed_in_notebook else "no",
        "main_text_or_appendix": recommendation,
        "notes": notes,
    }
    FIGURE_ARTIFACT_REGISTRY.append(row)
    return p


def save_and_show_figure(
    fig,
    path: Path | str,
    *,
    dpi: int = 320,
    bbox_inches: str = "tight",
    close: bool = False,
    figure_id: str | None = None,
    description: str = "",
    notebook_section: str = "",
    source_data: str = "",
    recommendation: str = "main/appendix",
    notes: str = "",
) -> Path:
    """Save a figure and display it in the notebook output. Default: do not close it."""
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    try:
        fig.tight_layout()
    except Exception:
        pass
    fig.savefig(p, dpi=dpi, bbox_inches=bbox_inches)
    plt.show()
    register_figure_artifact(
        p,
        figure_id=figure_id,
        description=description,
        notebook_section=notebook_section,
        source_data=source_data,
        displayed_in_notebook=True,
        recommendation=recommendation,
        notes=notes,
    )
    if close and MATPLOTLIB_AVAILABLE:
        # Explicit exception only; the dissertation notebook should normally show figures.
        plt.close(fig)
    return p


def compact_legend(ax, *, ncol: int | None = None, loc: str = "best", title: str | None = None):
    """Small, print-safe legend used across dissertation figures."""
    handles, labels = ax.get_legend_handles_labels()
    labels = [lab for lab in labels if lab and not lab.startswith("_")]
    if not labels:
        return None
    if ncol is None:
        ncol = 2 if len(labels) > 4 else 1
    return ax.legend(
        loc=loc,
        ncol=ncol,
        title=title,
        fontsize=DISSERTATION_MPL_STYLE["legend.fontsize"],
        title_fontsize=DISSERTATION_MPL_STYLE["legend.title_fontsize"],
        frameon=True,
        framealpha=DISSERTATION_MPL_STYLE["legend.framealpha"],
        borderpad=DISSERTATION_MPL_STYLE["legend.borderpad"],
        handlelength=DISSERTATION_MPL_STYLE["legend.handlelength"],
        handletextpad=DISSERTATION_MPL_STYLE["legend.handletextpad"],
        labelspacing=DISSERTATION_MPL_STYLE["legend.labelspacing"],
        columnspacing=DISSERTATION_MPL_STYLE["legend.columnspacing"],
    )


def standardise_axis(ax, *, title: str | None = None, xlabel: str | None = None, ylabel: str | None = None, legend: bool = False, ncol: int | None = None) -> None:
    if title:
        ax.set_title(title)
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    ax.grid(True, alpha=float(DISSERTATION_MPL_STYLE.get("grid.alpha", 0.18)))
    if legend:
        compact_legend(ax, ncol=ncol)


def sort_policy_labels(labels: Iterable[object]) -> list[str]:
    labels = [str(x) for x in labels]
    order = {name: i for i, name in enumerate(POLICY_DISPLAY_ORDER)}
    return sorted(labels, key=lambda x: (order.get(_normalise_style_key(x), 999), x))


def _format_heatmap_value(value: float, fmt: str | None = None) -> str:
    if not np.isfinite(value):
        return ""
    if fmt is not None:
        return format(value, fmt)
    if abs(value - round(value)) < 1e-9 and abs(value) < 10000:
        return f"{value:.0f}"
    if abs(value) >= 100:
        return f"{value:.0f}"
    if abs(value) >= 10:
        return f"{value:.1f}"
    if abs(value) >= 1:
        return f"{value:.2f}"
    if abs(value) >= 0.01:
        return f"{value:.3f}"
    return f"{value:.2g}"


def annotate_heatmap_values(
    ax,
    im,
    values: pd.DataFrame | np.ndarray,
    *,
    color_reference: pd.DataFrame | np.ndarray | None = None,
    fmt: str | None = None,
    fontsize: float | None = None,
    min_alpha: float = 0.95,
) -> None:
    """Annotate every heatmap cell with its numeric value."""
    label_arr = values.to_numpy(float) if isinstance(values, pd.DataFrame) else np.asarray(values, dtype=float)
    if color_reference is None:
        color_arr = label_arr
    else:
        color_arr = color_reference.to_numpy(float) if isinstance(color_reference, pd.DataFrame) else np.asarray(color_reference, dtype=float)
    nrows, ncols = label_arr.shape
    if fontsize is None:
        fontsize = max(4.2, min(6.6, 34.0 / max(nrows, ncols, 1)))
    cmap = im.cmap
    norm = im.norm
    for i in range(nrows):
        for j in range(ncols):
            value = label_arr[i, j]
            if not np.isfinite(value):
                continue
            try:
                colour_value = color_arr[i, j]
                rgba = cmap(norm(colour_value))
                luminance = 0.299 * rgba[0] + 0.587 * rgba[1] + 0.114 * rgba[2]
                color = "white" if luminance < 0.45 else "#111827"
            except Exception:
                color = "#111827"
            ax.text(
                j,
                i,
                _format_heatmap_value(float(value), fmt),
                ha="center",
                va="center",
                fontsize=fontsize,
                color=color,
                alpha=min_alpha,
            )


def plot_annotated_heatmap(
    matrix: pd.DataFrame,
    *,
    ax=None,
    fig=None,
    title: str = "",
    xlabel: str = "",
    ylabel: str = "",
    cbar_label: str = "",
    cmap: str = SEQUENTIAL_CMAP,
    norm=None,
    vmin: float | None = None,
    vmax: float | None = None,
    color_values: pd.DataFrame | np.ndarray | None = None,
    text_values: pd.DataFrame | np.ndarray | None = None,
    fmt: str | None = None,
    rotation: int = 35,
):
    """Draw a heatmap with explicit cell values. Colour values and text values may differ."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(7.0, 4.2))
    elif fig is None:
        fig = ax.figure
    color_matrix = color_values if color_values is not None else matrix
    color_arr = color_matrix.to_numpy(float) if isinstance(color_matrix, pd.DataFrame) else np.asarray(color_matrix, dtype=float)
    im = ax.imshow(color_arr, aspect="auto", cmap=cmap, norm=norm, vmin=vmin, vmax=vmax)
    ax.set_xticks(np.arange(len(matrix.columns)))
    ax.set_xticklabels(matrix.columns.astype(str), rotation=rotation, ha="right")
    ax.set_yticks(np.arange(len(matrix.index)))
    ax.set_yticklabels(matrix.index.astype(str))
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    annotate_heatmap_values(ax, im, text_values if text_values is not None else matrix, color_reference=color_matrix, fmt=fmt)
    cbar = fig.colorbar(im, ax=ax, label=cbar_label) if cbar_label else fig.colorbar(im, ax=ax)
    cbar.ax.tick_params(labelsize=DISSERTATION_MPL_STYLE["ytick.labelsize"])
    return im




def bar_colors_from_labels(labels: Iterable[object]) -> list[str]:
    labels = [str(x) for x in labels]
    return [model_color(lbl) for lbl in labels]
def plot_confidence_interval_bars(
    data: pd.DataFrame,
    *,
    x_col: str,
    y_col: str,
    lower_col: str | None = None,
    upper_col: str | None = None,
    title: str = "",
    ylabel: str = "",
    output_path: Path | str | None = None,
):
    if not MATPLOTLIB_AVAILABLE or data is None or data.empty:
        return None
    p = data.copy()
    p[x_col] = p[x_col].astype(str)
    ordered = [x for x in sort_policy_labels(p[x_col].unique()) if x in set(p[x_col])]
    if ordered:
        p["_order"] = p[x_col].map({v: i for i, v in enumerate(ordered)})
        p = p.sort_values(["_order", x_col]).drop(columns="_order")
    fig, ax = plt.subplots(figsize=(max(6.2, 0.40 * len(p)), 3.8))
    x = np.arange(len(p))
    y = pd.to_numeric(p[y_col], errors="coerce")
    colors = [model_color(v) for v in p[x_col]]
    if lower_col and upper_col and {lower_col, upper_col}.issubset(p.columns):
        yerr = np.vstack([y - pd.to_numeric(p[lower_col], errors="coerce"), pd.to_numeric(p[upper_col], errors="coerce") - y])
        ax.errorbar(x, y, yerr=yerr, fmt="none", capsize=2.5, color="#111827", linewidth=0.75)
    ax.bar(x, y, color=colors, alpha=0.82, edgecolor="white", linewidth=0.45)
    ax.set_xticks(x)
    ax.set_xticklabels(p[x_col], rotation=45, ha="right")
    standardise_axis(ax, title=title, xlabel=x_col.replace("_", " "), ylabel=ylabel or y_col.replace("_", " "))
    if output_path is not None:
        return save_and_show_figure(fig, output_path, dpi=320, description=title, notes="Confidence intervals use available lower/upper columns where present.")
    return fig


set_dissertation_plot_theme()


In [4]:
# Configuration: one ordered execution plan
REBUILD_STAGES = (
    "surface", "hedging_panel", "calibration", "feasibility",
    "banks", "policies", "evaluation", "reporting",
)
_STAGE_INDEX = {name: i for i, name in enumerate(REBUILD_STAGES)}


@dataclass(frozen=True)
class ExecutionPlan:
    """Derived execution controls. Users configure only `RUN` above."""

    profile: Literal["smoke", "full"]
    rebuild_from: str
    calibration_spec: Literal["preliminary_h035", "final_h025"]
    allow_upstream_rebuild: bool = False
    implementation_version: str = "execution_plan_v8_dual_calibration_lag_consistent"

    def force_rebuild(self, stage: str) -> bool:
        if stage not in _STAGE_INDEX:
            raise KeyError(f"Unknown rebuild stage {stage!r}.")
        if self.rebuild_from in {"auto", "reuse"}:
            return False
        return _STAGE_INDEX[stage] >= _STAGE_INDEX[self.rebuild_from]

    def can_rebuild(self, stage: str) -> bool:
        if stage not in _STAGE_INDEX:
            raise KeyError(f"Unknown rebuild stage {stage!r}.")
        if _STAGE_INDEX[stage] <= _STAGE_INDEX["calibration"] and not self.allow_upstream_rebuild:
            return False
        if self.rebuild_from == "auto":
            return True
        if self.rebuild_from == "reuse":
            return False
        return self.force_rebuild(stage)

    @property
    def auto_rebuild_missing(self) -> bool:
        return self.rebuild_from == "auto"

    @property
    def REBUILD_SURFACE(self) -> bool: return self.can_rebuild("surface") and self.force_rebuild("surface")
    @property
    def REBUILD_HEDGING_PANEL(self) -> bool: return self.can_rebuild("hedging_panel") and self.force_rebuild("hedging_panel")
    @property
    def RUN_HESTON_CALIBRATION(self) -> bool: return self.can_rebuild("calibration") and self.force_rebuild("calibration")
    @property
    def RUN_LIFTED_CALIBRATION(self) -> bool: return self.can_rebuild("calibration") and self.force_rebuild("calibration")
    @property
    def FREEZE_CALIBRATION_ARTIFACTS(self) -> bool: return not self.can_rebuild("calibration")
    @property
    def ALLOW_CALIBRATION_REBUILD_FROM_DOWNSTREAM(self) -> bool: return self.can_rebuild("calibration")
    @property
    def RUN_SIMULATION_FEASIBILITY_SCAN(self) -> bool: return self.can_rebuild("feasibility")
    @property
    def REUSE_SIMULATION_FEASIBILITY_MANIFEST(self) -> bool: return not self.force_rebuild("feasibility")
    @property
    def REBUILD_SIMULATION_FEASIBILITY_MANIFEST(self) -> bool: return self.force_rebuild("feasibility")
    @property
    def RUN_TRAINING(self) -> bool: return self.can_rebuild("policies")
    @property
    def RUN_EVALUATION(self) -> bool: return self.can_rebuild("evaluation")
    @property
    def run_data_build(self) -> bool: return self.REBUILD_SURFACE
    @property
    def run_calibration(self) -> bool: return self.RUN_HESTON_CALIBRATION or self.RUN_LIFTED_CALIBRATION
    @property
    def run_deep_hedging_training(self) -> bool: return self.RUN_TRAINING
    @property
    def run_evaluation(self) -> bool: return self.RUN_EVALUATION
    @property
    def rebuild_caches(self) -> bool: return self.rebuild_from not in {"auto", "reuse", "reporting"}
    @property
    def strict_methodology_gates(self) -> bool: return True

    AUTO_PRUNE_REDUNDANT_DERIVED_CACHES: bool = False
    BUILD_TEST_RAW_TENSOR_BANKS: bool = False
    CACHE_FEASIBILITY_PATH_ARRAYS: bool = False
    CACHE_SCALED_TENSOR_BATCHES: bool = False
    RUN_CAPACITY_ROBUSTNESS: bool = False
    RUN_H_SENSITIVITY: bool = False
    RUN_PATHWISE_DIAGNOSTIC_EXPORT: bool = False
    RUN_REPRESENTATION_MEMORY_DIAGNOSTIC: bool = False


def execution_plan_signature(plan: Any) -> tuple[str, str, str, bool, str | None]:
    return (
        str(getattr(plan, "profile", "<missing>")),
        str(getattr(plan, "rebuild_from", "<missing>")),
        str(getattr(plan, "calibration_spec", "<missing>")),
        bool(getattr(plan, "allow_upstream_rebuild", False)),
        getattr(plan, "implementation_version", None),
    )


EXPECTED_EXECUTION_SIGNATURE = (
    RUN.profile,
    RUN.rebuild_from,
    RUN.calibration_spec,
    RUN.allow_upstream_rebuild,
    "execution_plan_v8_dual_calibration_lag_consistent",
)
EXECUTION = ExecutionPlan(
    profile=RUN.profile,
    rebuild_from=RUN.rebuild_from,
    calibration_spec=RUN.calibration_spec,
    allow_upstream_rebuild=RUN.allow_upstream_rebuild,
)
if execution_plan_signature(EXECUTION) != EXPECTED_EXECUTION_SIGNATURE:
    raise AssertionError("Failed to resolve the execution plan from RUN. Restart the kernel and run all cells from the top.")

RUN_PATHWISE_DIAGNOSTIC_EXPORT = False
RUN_REPRESENTATION_MEMORY_DIAGNOSTIC = False
RUN_CAPACITY_ROBUSTNESS = False
RUN_H_SENSITIVITY = False
TRAIN_SECONDARY_POLICIES = False
BUILD_TEST_RAW_TENSOR_BANKS = False
CACHE_SCALED_TENSOR_BATCHES = False
CACHE_FEASIBILITY_PATH_ARRAYS = False
AUTO_PRUNE_REDUNDANT_DERIVED_CACHES = False

print(pd.DataFrame([
    {"stage": stage, "forced_rebuild": EXECUTION.force_rebuild(stage), "rebuild_allowed": EXECUTION.can_rebuild(stage)}
    for stage in REBUILD_STAGES
]))


           stage  forced_rebuild  rebuild_allowed
0        surface           False            False
1  hedging_panel           False            False
2    calibration           False            False
3    feasibility           False            False
4          banks           False            False
5       policies           False            False
6     evaluation           False            False
7      reporting            True             True


In [5]:
# Configuration: paths
@dataclass(frozen=True)
class PathConfig:
    project_root: Path = RUN.project_root
    data_dir: Path = RUN.data_dir
    cache_dir: Path = RUN.cache_dir
    output_dir: Path = RUN.output_dir
    raw_option_file: str = "OptionMetrics ATM IV 2010 - 2025.parquet"
    raw_crsp_file: str = "CRSP S&P 2010 - 2024.parquet"
    surface_cache_file: str = "full_surface_panel_partA_surface_discipline.parquet"
    heston_calibration_file: str = "heston_hedging_panel_calibration_5bucket.parquet"
    lifted_calibration_file: str = "lifted_heston_hedging_panel_calibration_5bucket.parquet"
    lifted_fit_file: str = "lifted_heston_hedging_panel_fit_table_5bucket.parquet"
    lifted_history_file: str = "lifted_heston_hedging_panel_history_5bucket.parquet"
    lifted_timing_file: str = "lifted_heston_hedging_panel_timing_5bucket.parquet"
    heston_fit_residuals_file: str = "heston_hedging_panel_fit_residuals_5bucket.parquet"
    heston_bucket_summary_file: str = "heston_hedging_residual_bucket_summary_5bucket.parquet"
    calibration_comparison_file: str = "heston_vs_lifted_hedging_panel_calibration_comparison_5bucket.parquet"
    forward_variance_file: str = "hedging_panel_forward_variance_curve_regularised_5bucket.parquet"
    no_trade_band_file: str = "no_trade_band_validation_5bucket.parquet"
    factor_panel_file: str = "factor_diagnostic_panel_5bucket.parquet"
    factor_pca_file: str = "factor_pca_meta_5bucket.parquet"
    factor_regression_file: str = "factor_information_regression_5bucket.parquet"
    lifted_validation_file: str = "lifted_riccati_fourier_validation_5bucket.parquet"
    model_delta_validation_file: str = "model_delta_validation_5bucket.parquet"
    seed_panel_file: str = "near_atm_straddle_seed_panel_5bucket.parquet"
    candidate_seed_panel_file: str = "candidate_near_atm_straddle_seed_panel_5bucket.parquet"
    final_feasible_seed_panel_file: str = "final_feasible_near_atm_straddle_seed_panel_5bucket.parquet"
    panel_selection_dir_name: str = "panel_selection_5bucket"
    seed_candidate_universe_file: str = "seed_candidate_universe.parquet"
    seed_panel_exclusion_manifest_file: str = "seed_panel_exclusion_manifest.parquet"
    deterministic_stratified_seed_panel_file: str = "deterministic_stratified_seed_panel.parquet"
    simulation_feasibility_manifest_file: str = "simulation_feasibility_manifest_5bucket.parquet"
    raw_tensor_bank_manifest_file: str = "raw_tensor_bank_manifest_5bucket.parquet"
    simulation_path_cache_dir_name: str = "simulation_path_cache_5bucket"
    evaluation_file: str = "hedging_evaluation_results.parquet"
    training_history_file: str = "neural_training_history.parquet"
    paired_inference_file: str = "paired_date_clustered_inference.parquet"
    simulation_validation_file: str = "simulation_validation_5bucket.parquet"
    calibration_quality_file: str = "calibration_quality_gate_5bucket.parquet"
    h_sensitivity_file: str = "lifted_h_sensitivity.parquet"
    full_methodology_audit_file: str = "full_methodology_audit.csv"
    bug_sweep_file: str = "bug_sweep_report.csv"
    readiness_file: str = "readiness_table.csv"
    methodology_insert_panel_file: str = "methodology_insert_deterministic_panel.md"
    preprocessing_temp_dir_name: str = "_preprocessing_parts_partA_surface_discipline"
    surface_year_dir_name: str = "surface_year_parts_partA_surface_discipline"
    heston_parts_dir_name: str = "heston_hedging_panel_date_parts_5bucket"
    lifted_parts_dir_name: str = "lifted_heston_hedging_panel_date_parts_5bucket"

    @property
    def raw_option_path(self) -> Path:
        return self.project_root / self.data_dir / self.raw_option_file

    @property
    def raw_crsp_path(self) -> Path:
        return self.project_root / self.data_dir / self.raw_crsp_file

    @property
    def surface_cache_path(self) -> Path:
        return self.project_root / self.cache_dir / self.surface_cache_file

    @property
    def preprocessing_tmp_dir(self) -> Path:
        return self.project_root / self.cache_dir / self.preprocessing_temp_dir_name

    @property
    def surface_year_dir(self) -> Path:
        return self.project_root / self.cache_dir / self.surface_year_dir_name

    @property
    def heston_calibration_path(self) -> Path:
        return self.project_root / self.cache_dir / self.heston_calibration_file

    @property
    def heston_calibration_parts_dir(self) -> Path:
        return self.project_root / self.cache_dir / self.heston_parts_dir_name

    @property
    def lifted_calibration_path(self) -> Path:
        return self.project_root / self.cache_dir / self.lifted_calibration_file

    @property
    def lifted_fit_path(self) -> Path:
        return self.project_root / self.cache_dir / self.lifted_fit_file

    @property
    def lifted_history_path(self) -> Path:
        return self.project_root / self.cache_dir / self.lifted_history_file

    @property
    def lifted_timing_path(self) -> Path:
        return self.project_root / self.cache_dir / self.lifted_timing_file

    @property
    def lifted_calibration_parts_dir(self) -> Path:
        return self.project_root / self.cache_dir / self.lifted_parts_dir_name

    @property
    def seed_panel_path(self) -> Path:
        return self.project_root / self.cache_dir / self.seed_panel_file

    @property
    def candidate_seed_panel_path(self) -> Path:
        return self.project_root / self.cache_dir / self.candidate_seed_panel_file

    @property
    def final_feasible_seed_panel_path(self) -> Path:
        return self.project_root / self.cache_dir / self.final_feasible_seed_panel_file

    @property
    def panel_selection_dir(self) -> Path:
        return self.project_root / self.cache_dir / self.panel_selection_dir_name

    @property
    def seed_candidate_universe_path(self) -> Path:
        return self.panel_selection_dir / self.seed_candidate_universe_file

    @property
    def seed_panel_exclusion_manifest_path(self) -> Path:
        return self.panel_selection_dir / self.seed_panel_exclusion_manifest_file

    @property
    def deterministic_stratified_seed_panel_path(self) -> Path:
        return self.panel_selection_dir / self.deterministic_stratified_seed_panel_file

    @property
    def methodology_insert_deterministic_panel_path(self) -> Path:
        return self.project_root / self.output_dir / self.methodology_insert_panel_file

    @property
    def simulation_feasibility_manifest_path(self) -> Path:
        return self.project_root / self.cache_dir / self.simulation_feasibility_manifest_file

    @property
    def raw_tensor_bank_manifest_path(self) -> Path:
        return self.project_root / self.cache_dir / self.raw_tensor_bank_manifest_file

    @property
    def simulation_path_cache_dir(self) -> Path:
        return self.project_root / self.cache_dir / self.simulation_path_cache_dir_name

    @property
    def heston_fit_residuals_path(self) -> Path:
        return self.project_root / self.cache_dir / self.heston_fit_residuals_file

    @property
    def heston_bucket_summary_path(self) -> Path:
        return self.project_root / self.cache_dir / self.heston_bucket_summary_file

    @property
    def calibration_comparison_path(self) -> Path:
        return self.project_root / self.cache_dir / self.calibration_comparison_file

    @property
    def forward_variance_path(self) -> Path:
        return self.project_root / self.cache_dir / self.forward_variance_file

    @property
    def no_trade_band_path(self) -> Path:
        return self.project_root / self.cache_dir / self.no_trade_band_file

    @property
    def factor_panel_path(self) -> Path:
        return self.project_root / self.cache_dir / self.factor_panel_file

    @property
    def factor_pca_path(self) -> Path:
        return self.project_root / self.cache_dir / self.factor_pca_file

    @property
    def factor_regression_path(self) -> Path:
        return self.project_root / self.cache_dir / self.factor_regression_file

    @property
    def lifted_validation_path(self) -> Path:
        return self.project_root / self.cache_dir / self.lifted_validation_file

    @property
    def model_delta_validation_path(self) -> Path:
        return self.project_root / self.cache_dir / self.model_delta_validation_file

    @property
    def evaluation_path(self) -> Path:
        return self.project_root / self.output_dir / self.evaluation_file

    @property
    def training_history_path(self) -> Path:
        return self.project_root / self.output_dir / self.training_history_file

    @property
    def paired_inference_path(self) -> Path:
        return self.project_root / self.output_dir / self.paired_inference_file

    @property
    def simulation_validation_path(self) -> Path:
        return self.project_root / self.cache_dir / self.simulation_validation_file

    @property
    def calibration_quality_path(self) -> Path:
        return self.project_root / self.cache_dir / self.calibration_quality_file

    @property
    def h_sensitivity_path(self) -> Path:
        return self.project_root / self.output_dir / self.h_sensitivity_file

    @property
    def figure_dir(self) -> Path:
        return self.project_root / self.output_dir / "figures"

    @property
    def table_dir(self) -> Path:
        return self.project_root / self.output_dir / "tables"

    @property
    def checkpoint_dir(self) -> Path:
        return self.project_root / self.output_dir / "checkpoints"


In [6]:
# Configuration: methodology parameters
@dataclass(frozen=True)
class MarketConfig:
    calendar_days_per_year: float = 365.25
    trading_days_per_year: int = 252
    optionmetrics_strike_scale: float = 1000.0
    auto_rescale_large_strikes: bool = True
    use_memory_safe_preprocessing: bool = True
    raw_batch_rows: int = 500_000
    keep_preprocessing_parts: bool = False
    preprocessing_start_date: str | None = None
    preprocessing_end_date: str | None = None
    min_dte: int = 7
    max_dte: int = 365
    # Headline hedging-focused SPX panel filters.
    hedging_min_dte: int = 7
    hedging_max_dte: int = 365
    hedging_max_abs_log_moneyness: float = 0.10
    hedging_primary_atm_abs_log_moneyness: float = 0.05
    hedging_min_volume: int | None = None
    hedging_min_open_interest: int | None = None
    hedging_max_abs_spread: float | None = None
    hedging_max_rel_spread: float | None = 0.25
    hedging_stress_vix_threshold: float = 25.0
    hedging_stress_realised_vol_quantile: float = 0.80
    min_mid_price: float = 0.05
    min_best_bid: float = 0.05
    min_open_interest: int = 1
    # Strict spread discipline: zero/missing relative spreads imply infinite/undefined quote-quality weight.
    # Such rows are excluded rather than assigned an artificial unit/infinite weight.
    min_bid_ask_spread_frac: float = 1e-8
    max_bid_ask_spread_frac: float = 0.25
    min_iv: float = 0.02
    max_iv: float = 3.00
    max_abs_log_moneyness: float = 0.35
    max_abs_forward_basis: float = 0.25
    min_forward_pairs: int = 5
    max_forward_iqr_rel: float | None = None
    use_otm_quotes: bool = True
    collapse_to_strike_grid: bool = True
    drop_static_arb_violating_expiries: bool = True
    fit_weight_floor: float = 0.25
    fit_weight_cap: float = 4.0
    maturity_balance_power: float = 0.5
    moneyness_balance_power: float = 0.5
    moneyness_balance_bins: tuple[float, ...] = (-0.35, -0.20, -0.10, -0.03, 0.03, 0.10, 0.20, 0.35)
    vega_floor: float = 1e-4
    static_arb_tolerance: float = 1e-6
    # Lifted calibration uses a regularised initial forward-variance curve.
    # The curve is inferred from near-ATM implied total variance and used to initialise
    # and shape the deterministic input curve for the 5-parameter lifted-Heston calibration.
    forward_variance_estimator_version: str = "regularised_near_atm_total_variance_v1"
    forward_variance_smoothness: float = 2e-3
    forward_variance_curvature_smoothness: float = 2e-2
    forward_variance_min: float = 1e-8
    forward_variance_max: float = 3.0
    forward_variance_atm_kernel_bandwidth: float = 0.025
    forward_variance_min_maturities: int = 2
    forward_variance_use_fit_weights: bool = True
    forward_variance_max_relative_total_variance_adjustment: float = 0.75
    max_atm_forward_moneyness: float = 0.05
    # Risk-free-rate handling is explicit, not silent imputation.
    # The OptionMetrics quote extract used here may not contain rates. In that case,
    # use the configured constant-rate convention inherited from the validated Part A
    # pipeline. Set risk_free_rate_source="raw_column" only when the raw option table
    # genuinely contains a risk-free rate column/alias.
    risk_free_rate_source: Literal["configured_constant", "raw_column"] = "configured_constant"
    configured_risk_free_rate: float = 0.01

    # CRSP spot coverage policy. This is not spot imputation. When the option file
    # extends beyond the CRSP index-level file, uncovered option dates are excluded
    # and audited so that every retained row still uses verified CRSP spindx. Set
    # require_full_coverage to fail instead.
    crsp_spot_coverage_policy: Literal["drop_uncovered_dates", "require_full_coverage"] = "drop_uncovered_dates"
    # Default 0.0 because memory-safe streaming may encounter entire late batches
    # outside CRSP coverage, e.g. an option file extending into 2025 while the CRSP
    # spot file ends in 2024. Those rows are excluded, not imputed. Set a positive
    # value only when enforcing batch-level coverage deliberately.
    min_crsp_spot_coverage_fraction: float = 0.0


@dataclass(frozen=True)
class CalibrationConfig:
    heston_bounds: dict[str, tuple[float, float]] = field(default_factory=lambda: {
        "kappa": (0.02, 15.0),
        "theta": (1e-4, 2.0),
        "xi": (0.01, 5.0),
        "rho": (-0.999, 0.999),
        "v0": (1e-4, 2.0),
    })
    heston_initial: dict[str, float] = field(default_factory=lambda: {
        "kappa": 2.0,
        "theta": 0.04,
        "xi": 0.60,
        "rho": -0.70,
        "v0": 0.04,
    })
    max_options_per_surface: int | None = None
    optimizer_maxiter: int = 100
    pricing_integration_limit: int = 120
    pricing_integration_upper: float = 100.0
    heston_pricing_nodes: int = 64
    heston_checkpoint_by_date: bool = True
    heston_resume_date_parts: bool = True
    heston_warm_start_previous_date: bool = True
    heston_retry_failed_dates: bool = True
    heston_quality_warn_rmse: float = 0.50
    heston_quality_bad_rmse: float = 2.00
    vega_floor: float = 1.0


@dataclass(frozen=True)
class LiftedHestonConfig:
    """Final lifted-Heston configuration: N=20 with 5-parameter calibration and a smoothed forward-variance input curve."""

    H: float = 0.25  # overridden only by the explicit RUN.calibration_spec compatibility mode.
    n_factors: int = 20
    rn_scale: float = 2.5
    factor_grid_method: Literal["abi_zaugg_rn"] = "abi_zaugg_rn"
    kernel_t_min: float = 1 / 365.25
    kernel_t_max: float = 2.0
    kernel_grid_size: int = 500
    n_steps_per_year: int = 252
    variance_floor: float = 1e-10
    # Strict methodology: lifted calibration is Riccati ODE + Fourier inversion, not MC substitution.
    pricing_nodes: int = 64
    riccati_steps: int = 160
    optimizer_max_nfev: int = 120
    calibration_vega_floor: float = 1.0
    calibration_parameterization: Literal["full_5p_forward_curve_init"] = "full_5p_forward_curve_init"
    calibration_checkpoint_by_date: bool = True
    calibration_resume_date_parts: bool = True
    calibration_max_dates: int | None = None
    fit_vega_rmse_warn: float = 0.50
    fit_vega_rmse_bad: float = 2.00
    delta_bump_frac: float = 1e-4  # used only for independent validation against probability delta
    delta_validation_bump_fracs: tuple[float, ...] = (5e-5, 1e-4, 2e-4)
    delta_validation_rows: int = 6
    delta_coefficient_cache_size: int = 4096
    delta_probability_tolerance: float = 2e-3
    initial_curve_ridge: float = 1e-6
    initial_curve_max_rmse: float = 0.05
    pricing_validation_rows: int = 24
    pricing_validation_dates: int = 6
    no_trade_band_candidates: tuple[float, ...] = (0.0, 0.01, 0.025, 0.05, 0.075, 0.10)
    conditional_projection_steps: int = 64
    heston_limit_price_tolerance: float = 2e-3
    simulation_mean_variance_tolerance: float = 0.005
    h_sensitivity_values: tuple[float, ...] = (0.10, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45)
    robustness_max_dates: int = 6
    validation_paths: int = 20_000
    # Lifted simulation does not clip/project invalid states.
    # Invalid proposals are rejected and resampled; if acceptance fails, the notebook stops.
    valid_state_tolerance: float = 1e-9
    max_valid_state_resamples: int = 256
    martingale_relative_tolerance: float = 0.02
    delta_no_arbitrage_tolerance: float = 5e-3
    timestep_convergence_relative_tolerance: float = 0.10
    # Explicit lifted valid-state checks. These are validation tolerances, not simulator repairs.
    cumulant_convexity_bump: float = 1e-3
    cumulant_convexity_exponents: tuple[float, ...] = (-0.5, 0.0, 0.5, 1.0)
    cumulant_convexity_tolerance: float = 2e-4
    cumulant_validation_paths: int = 128
    cumulant_validation_time_points: int = 5
    delta_finite_difference_tolerance: float = 5e-2
    path_delta_validation_paths: int = 96
    path_delta_validation_time_points: int = 4
    one_factor_limit_paths: int = 2_000
    one_factor_limit_steps: int = 8
    one_factor_limit_mean_variance_tolerance: float = 0.01
    integrated_variance_min_tolerance: float = 0.0
    # C-LP simulator numerical refinement. This is same-model timestep refinement,
    # not a fallback simulator and not a projection/clipping repair. Daily hedge nodes
    # remain unchanged; only the internal simulation grid is refined if the C-LP
    # coefficient positivity constraints become numerically infeasible.
    simulation_internal_substeps_per_hedge_step: int = 1
    simulation_max_internal_substeps_per_hedge_step: int = 16


TARGET_FINAL_QUOTAS = {
    "train": 20,
    "val": 8,
    "test": 30,
}

TARGET_CANDIDATE_QUOTAS = {
    "train": 60,
    "val": 30,
    "test": 90,
}


def make_balanced_role_sequence(
    quotas: dict[str, int],
    priority: tuple[str, ...] = ("train", "test", "val"),
) -> tuple[str, ...]:
    """Spread split roles approximately evenly across chronological strata."""
    total = int(sum(quotas.values()))
    used = {role: 0 for role in quotas}
    sequence: list[str] = []

    for position in range(total):
        deficits = {
            role: ((position + 1) * quotas[role] / total) - used[role]
            for role in quotas
        }
        role = max(
            priority,
            key=lambda name: (
                deficits[name],
                -priority.index(name),
            ),
        )
        sequence.append(role)
        used[role] += 1

    observed = {role: sequence.count(role) for role in quotas}
    if observed != quotas:
        raise RuntimeError(
            f"Role-sequence construction failed: expected={quotas}, observed={observed}"
        )
    return tuple(sequence)


EXPANDED_ROLE_SEQUENCE = make_balanced_role_sequence(TARGET_FINAL_QUOTAS)

assert len(EXPANDED_ROLE_SEQUENCE) == 58
assert EXPANDED_ROLE_SEQUENCE.count("train") == 20
assert EXPANDED_ROLE_SEQUENCE.count("val") == 8
assert EXPANDED_ROLE_SEQUENCE.count("test") == 30


@dataclass(frozen=True)
class PanelSelectionConfig:
    """Deterministic maturity-stratified panel with an expanded test sample."""

    deterministic_panel_version: str = (
        "deterministic_stratified_chrono_hash_panel_v5_test30_5bucket_date_exclusive"
    )
    panel_selection_seed: int = 20260702
    panel_selection_mode: Literal["chrono_stratified_hash"] = "chrono_stratified_hash"

    # 20 train + 8 validation + 30 test dates per maturity bucket.
    strata_per_bucket: int = 58
    role_sequence: tuple[str, ...] = EXPANDED_ROLE_SEQUENCE
    quota_per_bucket: dict[str, int] = field(
        default_factory=lambda: dict(TARGET_FINAL_QUOTAS)
    )

    # Critical: do not silently restore the legacy four-date test split.
    preserve_existing_test_split: bool = False

    simulation_feasibility_required_worlds: tuple[str, ...] = ("heston", "lifted")
    headline_evaluation_mode: str = "production_10000_test_paths_expanded_test30_5bucket_date_exclusive"
    paired_bootstrap_reps: int = 10_000
    paired_bootstrap_seed: int = 20260702


@dataclass(frozen=True)
class SeedPanelSizeConfig:
    """Single source of truth for expanded-test candidate and final quotas."""

    candidate_quota_per_bucket: dict[str, int] = field(
        default_factory=lambda: dict(TARGET_CANDIDATE_QUOTAS)
    )
    final_quota_per_bucket: dict[str, int] = field(
        default_factory=lambda: dict(TARGET_FINAL_QUOTAS)
    )


@dataclass(frozen=True)
class HistoryMemoryConfig:
    history_windows: tuple[int, ...] = (1, 5, 10, 20)
    default_history_window: int = 10
    history_dim: int = 4
    memory_hidden_size: int = 32
    memory_embedding_size: int = 16


@dataclass(frozen=True)
class ResidualHedgeConfig:
    enabled: bool = True
    baseline: Literal["model_delta"] = "model_delta"
    adjustment_l2_lambda: float = 1e-3
    adjustment_limit: float = 0.5
    zero_initialise_output_head: bool = True

TARGET_MATURITY_BUCKETS: dict[str, tuple[int, int, int]] = {
    # label: (target calendar days, inclusive lower DTE, inclusive upper DTE)
    "1W": (7, 7, 10),
    "2W": (14, 11, 21),
    "1M": (30, 22, 45),
    "2M": (60, 46, 75),
    "3M": (90, 76, 120),
}


@dataclass(frozen=True)
class HedgingConfig:
    maturity_buckets: dict[str, tuple[int, int, int]] = field(
        default_factory=lambda: dict(TARGET_MATURITY_BUCKETS)
    )
    transaction_cost_bps: tuple[float, ...] = (1.0, 5.0, 10.0)
    primary_transaction_cost_bps: float = 1.0
    hedge_steps_per_day: float = 1.0
    n_paths_train: int = 4_096
    n_paths_val: int = 4_096
    n_paths_test: int = 10_000
    train_val_test_split: tuple[float, float, float] = (20 / 58, 8 / 58, 30 / 58)
    max_training_seeds: int | None = None
    max_evaluation_seeds: int | None = None
    training_seeds: tuple[int, ...] = (17_181_926, 17_181_927, 17_181_928)
    diagnostic_paths: int = 1_024
    max_diagnostic_seeds: int = 24
    diagnostic_time_points: int = 12
    validation_seeds_per_bucket: int = 8
    candidate_train_per_bucket: int = 60
    candidate_val_per_bucket: int = 30
    candidate_test_per_bucket: int = 90
    final_train_per_bucket: int = 20
    final_val_per_bucket: int = 8
    final_test_per_bucket: int = 30
    training_transaction_cost_bps: tuple[float, ...] = (1.0, 5.0, 10.0)
    # Optional production override. Do not enable casually; dates and path counts should not both increase in one patch.
    production_train_per_bucket: int = 20
    production_val_per_bucket: int = 8
    production_test_per_bucket: int = 30
    feasibility_scan_paths: int = 512
    cluster_bootstrap_reps: int = 10_000
    cross_world_evaluation: bool = True


@dataclass(frozen=True)
class ModelConfig:
    hidden_size: int = 64
    primary_gru_hidden_size: int = 32
    robustness_gru_hidden_size: int = 64
    mlp_depth: int = 2
    gru_layers: int = 1
    dropout: float = 0.00
    hedge_limit: float = 2.0
    factor_denominator_floor: float = 1e-8
    factor_audit_max_abs_threshold: float = 1e4
    factor_reconstruction_tolerance: float = 1e-5
    # Causal train-only bridge from scalar/observable histories to lifted-factor contributions.
    factor_filter_hidden_size: int = 48
    factor_filter_layers: int = 1
    factor_filter_training_seed: int = 17_181_926
    factor_filter_training_steps: int = 1_500
    factor_filter_learning_rate: float = 3e-4
    factor_filter_weight_decay: float = 1e-5
    factor_filter_batch_paths: int = 512
    factor_filter_validation_interval: int = 100
    factor_filter_early_stopping_patience: int = 5
    factor_filter_target_clip_z: float = 8.0
    learning_rate: float = 3e-4
    weight_decay: float = 1e-5
    batch_paths: int = 512
    training_steps: int = 2_000
    early_stopping_patience: int = 5
    validation_interval: int = 100
    cvar_tail_probability: float = 0.05
    cvar_weight: float = 0.0
    cost_weight: float = 0.0
    use_mixed_precision: bool = True
    gradient_clip_norm: float = 1.0
    # The default production run trains the information-matched observability ladder:
    # scalar, oracle-factor and causal filtered-factor MLP/GRU cells, plus classical and bucket controls.
    train_secondary_policies: bool = False
    validation_aggregation: Literal["equal_date_bucket"] = "equal_date_bucket"
    no_trade_band_candidates: tuple[float, ...] = (
        0.0, 0.01, 0.025, 0.05, 0.075, 0.10, 0.15, 0.20, 0.25,
    )


@dataclass(frozen=True)
class InferenceConfig:
    bootstrap_reps: int = 10_000
    bootstrap_seed: int = 20260711
    primary_method: Literal["stationary_block"] = "stationary_block"
    primary_block_length_selected_dates: int = 5
    block_length_sensitivity_selected_dates: tuple[int, ...] = (3, 5, 8)
    include_iid_sensitivity: bool = True
    hierarchical_bootstrap_reps: int = 10_000



@dataclass(frozen=True)
class DiagnosticsConfig:
    """Switches and numerical settings for dissertation-ready calibration diagnostics."""

    compute_global_iv_rmse: bool = True
    compute_atm_skew_fit: bool = True
    compute_near_atm_skew: bool = True
    compute_finite_difference_skew: bool = True
    representative_fit_dates: int = 6
    near_atm_skew_window: float = 0.05
    finite_difference_skew_bump: float = 0.01

@dataclass(frozen=True)
class ExperimentConfig:
    methodology_version: str = "observable_heston_lifted_config_v5_calibration_compatibility"
    empirical_mode: Literal["hedging_focused", "global_surface_diagnostic"] = "hedging_focused"
    random_seed: int = 17_181_926
    paths: PathConfig = field(default_factory=PathConfig)
    flags: ExecutionPlan = field(default_factory=lambda: EXECUTION)
    market: MarketConfig = field(default_factory=MarketConfig)
    calibration: CalibrationConfig = field(default_factory=CalibrationConfig)
    lifted: LiftedHestonConfig = field(default_factory=LiftedHestonConfig)
    panel: PanelSelectionConfig = field(default_factory=PanelSelectionConfig)
    seed_sizes: SeedPanelSizeConfig = field(default_factory=SeedPanelSizeConfig)
    hedging: HedgingConfig = field(default_factory=HedgingConfig)
    history_memory: HistoryMemoryConfig = field(default_factory=HistoryMemoryConfig)
    residual_hedge: ResidualHedgeConfig = field(default_factory=ResidualHedgeConfig)
    diagnostics: DiagnosticsConfig = field(default_factory=DiagnosticsConfig)
    inference: InferenceConfig = field(default_factory=InferenceConfig)
    model: ModelConfig = field(default_factory=ModelConfig)


In [7]:
# Instantiate the single authoritative configuration.
_RESOLVED_PROJECT_ROOT = RUN.project_root.expanduser().resolve()
_RESOLVED_DATA_DIR = (RUN.data_dir if RUN.data_dir.is_absolute() else _RESOLVED_PROJECT_ROOT / RUN.data_dir).resolve()
_RESOLVED_CACHE_DIR = (RUN.cache_dir if RUN.cache_dir.is_absolute() else _RESOLVED_PROJECT_ROOT / RUN.cache_dir).resolve()
_RESOLVED_OUTPUT_DIR = (RUN.output_dir if RUN.output_dir.is_absolute() else _RESOLVED_PROJECT_ROOT / RUN.output_dir).resolve()
_RESOLVED_LEGACY_H035_OUTPUT_DIR = (RUN.legacy_h035_output_dir if RUN.legacy_h035_output_dir.is_absolute() else _RESOLVED_PROJECT_ROOT / RUN.legacy_h035_output_dir).resolve()
LEGACY_H035_OUTPUT_DIR = _RESOLVED_LEGACY_H035_OUTPUT_DIR
_RESOLVED_LEGACY_H035_OUTPUT_ZIP = (RUN.legacy_h035_output_zip if RUN.legacy_h035_output_zip.is_absolute() else _RESOLVED_PROJECT_ROOT / RUN.legacy_h035_output_zip).resolve()
LEGACY_H035_OUTPUT_ZIP = _RESOLVED_LEGACY_H035_OUTPUT_ZIP

_RUN_METHOD_TAG = RUN.calibration_spec.replace("preliminary_", "").replace("final_", "")
CFG = ExperimentConfig(
    methodology_version=f"observable_{_RUN_METHOD_TAG}_lag_consistent_retrain_v7",
    paths=PathConfig(
        project_root=_RESOLVED_PROJECT_ROOT,
        data_dir=_RESOLVED_DATA_DIR,
        cache_dir=_RESOLVED_CACHE_DIR,
        output_dir=_RESOLVED_OUTPUT_DIR,
    ),
    flags=EXECUTION,
    lifted=LiftedHestonConfig(H=RUN.calibration_H, n_factors=20, rn_scale=2.5),
)

# Compatibility labels are inert and exist only for inherited helper code.
RUN_MODE_REQUESTED = f"observable_{RUN.calibration_spec}_{RUN.profile}"
RUN_MODE = RUN_MODE_REQUESTED

if execution_plan_signature(CFG.flags) != EXPECTED_EXECUTION_SIGNATURE:
    raise AssertionError(
        "CFG contains a stale execution plan. Restart the kernel and run all "
        "cells from the top; do not rerun isolated configuration cells."
    )
assert CFG.paths.output_dir == _RESOLVED_OUTPUT_DIR
assert abs(CFG.lifted.H - RUN.calibration_H) < 1e-12
assert CFG.lifted.n_factors == 20
assert abs(CFG.lifted.rn_scale - 2.5) < 1e-12
print("Authoritative configuration created from RUN only.")


Authoritative configuration created from RUN only.


In [8]:
# ============================================================
# Calibration-cache inspection and execution flags
# ============================================================
FINAL_PROTOCOL_VERSION = (
    f"observable_heston_lifted_{RUN.calibration_tag.lower()}_"
    "lag_consistent_retraining_v7" if IS_PRELIMINARY_H035
    else f"observable_heston_lifted_{RUN.calibration_tag.lower()}_lag_consistent_final_v8"
)


def _parquet_has_rows_and_columns(path: Path, required: set[str]) -> bool:
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        frame = pd.read_parquet(path)
    except Exception:
        return False
    return (not frame.empty) and required.issubset(frame.columns)


def _valid_heston_calibration_cache(path: Path) -> bool:
    return _parquet_has_rows_and_columns(path, {"date", "kappa", "theta", "xi", "rho", "v0"})


def _valid_lifted_cache_bundle(paths: PathConfig, expected_H: float) -> bool:
    bundle = [
        paths.lifted_calibration_path, paths.lifted_fit_path,
        paths.lifted_history_path, paths.lifted_timing_path,
        paths.forward_variance_path,
    ]
    if not all(path.exists() and path.stat().st_size > 0 for path in bundle):
        return False
    try:
        panel = pd.read_parquet(paths.lifted_calibration_path)
    except Exception:
        return False
    required = {
        "date", "kappa", "theta", "xi", "rho", "v0", "H", "n_factors",
        "rn_scale", "calibration_parameterization", "quality_flag",
    }
    if panel.empty or not required.issubset(panel.columns):
        return False
    observed_H = pd.to_numeric(panel["H"], errors="coerce")
    return bool(
        observed_H.notna().all()
        and np.allclose(observed_H, expected_H, atol=1e-12, rtol=0.0)
        and set(pd.to_numeric(panel["n_factors"], errors="coerce").dropna().astype(int).unique()) == {20}
        and np.allclose(pd.to_numeric(panel["rn_scale"], errors="coerce"), 2.5)
        and set(panel["calibration_parameterization"].dropna().astype(str).unique()) == {"full_5p_forward_curve_init"}
    )


_heston_valid_basic = _valid_heston_calibration_cache(CFG.paths.heston_calibration_path)
_lifted_valid_basic = _valid_lifted_cache_bundle(CFG.paths, RUN.calibration_H)
_force_calibration = EXECUTION.force_rebuild("calibration")
if _force_calibration and not RUN.allow_upstream_rebuild:
    raise RuntimeError("Calibration is frozen in the final research notebook. Set ALLOW_UPSTREAM_REBUILD=1 deliberately to rebuild it.")
if not _heston_valid_basic or not _lifted_valid_basic:
    if not EXECUTION.can_rebuild("calibration"):
        raise FileNotFoundError(
            f"A valid Heston and Lifted-Heston H={RUN.calibration_H:.2f} calibration cache is required in {CFG.paths.cache_dir}. "
            "Complete or materialise the calibration cache before rebuilding downstream banks."
        )

CFG = replace(CFG, methodology_version=FINAL_PROTOCOL_VERSION, flags=replace(EXECUTION))

print({
    "protocol": FINAL_PROTOCOL_VERSION,
    "profile": RUN.profile,
    "calibration_spec": RUN.calibration_spec,
    "rebuild_from": RUN.rebuild_from,
    "H": CFG.lifted.H,
    "n_factors": CFG.lifted.n_factors,
    "rn_scale": CFG.lifted.rn_scale,
    "calibration_forced": _force_calibration,
    "heston_cache_basic_validity": _heston_valid_basic,
    "lifted_cache_basic_validity": _lifted_valid_basic,
    "output_dir": str(CFG.paths.output_dir),
    "policy_access_to_latent_simulator_state": "None",
})


{'protocol': 'observable_heston_lifted_h035_lag_consistent_retraining_v7', 'profile': 'full', 'calibration_spec': 'preliminary_h035', 'rebuild_from': 'reporting', 'H': 0.35, 'n_factors': 20, 'rn_scale': 2.5, 'calibration_forced': False, 'heston_cache_basic_validity': True, 'lifted_cache_basic_validity': True, 'output_dir': 'C:\\Users\\lhkke\\OneDrive - Queen Mary, University of London\\Documents\\Dissertation\\Deep-Hedging-under-Lifted-Heston-MSc-Dissertation-\\outputs_observable_h035_lag_consistent_retrained_full', 'policy_access_to_latent_simulator_state': 'None'}


### Run preflight

This gate checks packages, proprietary inputs, disk space and the resolved rebuild chain before expensive work begins. Unlike the previous configuration, smoke mode may rebuild the surface or calibration when explicitly requested. A smoke run with `rebuild_from="surface"` is mechanically small downstream but the raw surface and calibration stages remain genuinely expensive.


In [9]:
# Fail early, before an expensive notebook discovers a missing dependency three hours later.
_missing_runtime = []
if not SCIPY_AVAILABLE: _missing_runtime.append("scipy")
if not TORCH_AVAILABLE: _missing_runtime.append("torch")
if not MATPLOTLIB_AVAILABLE: _missing_runtime.append("matplotlib")
try:
    pd.io.parquet.get_engine("auto")
except Exception:
    _missing_runtime.append("pyarrow or fastparquet")
if _missing_runtime:
    raise RuntimeError(f"Missing required runtime packages: {sorted(set(_missing_runtime))}")

_required_data_paths = [CFG.paths.raw_option_path, CFG.paths.raw_crsp_path]
_missing_data_paths = [str(path) for path in _required_data_paths if not Path(path).exists()]
if _missing_data_paths:
    raise FileNotFoundError(
        "Required proprietary source files are missing: " + ", ".join(_missing_data_paths)
        + ". Edit RUN.project_root/data_dir in the user configuration cell."
    )

_free_gb = shutil.disk_usage(CFG.paths.project_root).free / 1024**3
_minimum_free_gb = 2.0 if IS_SMOKE else 15.0
if _free_gb < _minimum_free_gb:
    raise OSError(
        f"Only {_free_gb:.1f} GB free; {RUN.profile} requires at least {_minimum_free_gb:.1f} GB."
    )

RUN_PREFLIGHT = pd.DataFrame([
    {"check": "profile", "value": RUN.profile, "status": "PASS"},
    {"check": "rebuild_from", "value": RUN.rebuild_from, "status": "PASS"},
    {"check": "raw_option_path", "value": str(CFG.paths.raw_option_path), "status": "PASS"},
    {"check": "raw_crsp_path", "value": str(CFG.paths.raw_crsp_path), "status": "PASS"},
    {"check": "cache_directory", "value": str(CFG.paths.cache_dir), "status": "PASS"},
    {"check": "output_directory", "value": str(CFG.paths.output_dir), "status": "PASS"},
    {"check": "calibration_spec", "value": RUN.calibration_spec, "status": "PASS"},
    {"check": "H", "value": CFG.lifted.H, "status": "PASS"},
    {"check": "policy_latent_state_access", "value": "None", "status": "PASS"},
    {"check": "heston_cache_basic_validity", "value": bool(_heston_valid_basic), "status": "INFO"},
    {"check": "lifted_cache_basic_validity", "value": bool(_lifted_valid_basic), "status": "INFO"},
    {"check": "free_disk_gb", "value": round(_free_gb, 2), "status": "PASS"},
])
if IS_PRELIMINARY_H035 and not _lifted_valid_basic:
    raise FileNotFoundError(
        f"The H=0.35 calibration bundle was not found or failed semantic validation in "
        f"{CFG.paths.cache_dir}. Set RUN.cache_dir_override to the existing TEST30 cache directory. "
        "This retraining notebook will not rebuild calibration."
    )
display(RUN_PREFLIGHT)



def _safe_extract_legacy_h035_zip(zip_path: Path, expected_dir: Path) -> None:
    """Extract a trusted run archive without permitting path traversal or silent overwrite."""
    import zipfile

    if expected_dir.exists():
        return
    if not zip_path.exists():
        raise FileNotFoundError(
            f"Neither the extracted legacy output directory nor its zip archive exists: "
            f"{expected_dir} ; {zip_path}"
        )

    destination = expected_dir.parent.resolve()
    expected_root = expected_dir.name
    with zipfile.ZipFile(zip_path) as archive:
        members = archive.infolist()
        roots = {Path(member.filename).parts[0] for member in members if Path(member.filename).parts}
        if expected_root not in roots:
            raise ValueError(
                f"Legacy archive root mismatch. Expected {expected_root!r}; observed roots={sorted(roots)}"
            )
        for member in members:
            target = (destination / member.filename).resolve()
            try:
                target.relative_to(destination)
            except ValueError as exc:
                raise ValueError(f"Unsafe archive member: {member.filename}") from exc
        archive.extractall(destination)
    if not expected_dir.exists():
        raise FileNotFoundError(f"Archive extraction completed but {expected_dir} was not created.")
    print("Extracted validated H=0.35 V5.3 output archive to:", expected_dir)


if LEGACY_H035_REUSE and not LEGACY_H035_OUTPUT_DIR.exists() and RUN.auto_extract_legacy_output:
    _safe_extract_legacy_h035_zip(LEGACY_H035_OUTPUT_ZIP, LEGACY_H035_OUTPUT_DIR)


if LEGACY_H035_REUSE:
    _legacy_required = [
        LEGACY_H035_OUTPUT_DIR / "run_manifest.json",
        LEGACY_H035_OUTPUT_DIR / "tables" / "checkpoint_manifest.csv",
        LEGACY_H035_OUTPUT_DIR / "observable_market_cache",
        LEGACY_H035_OUTPUT_DIR / "checkpoints",
    ]
    _legacy_missing = [str(path) for path in _legacy_required if not path.exists()]
    if _legacy_missing:
        raise FileNotFoundError(
            "The completed H=0.35 V5.3 output directory is incomplete: " + ", ".join(_legacy_missing)
            + ". Set LEGACY_H035_OUTPUT_DIR or RUN.legacy_h035_output_dir to the extracted/full output directory."
        )
    _legacy_manifest = json.loads((LEGACY_H035_OUTPUT_DIR / "run_manifest.json").read_text())
    _legacy_checks = {
        "protocol_hash": _legacy_manifest.get("protocol_hash") == RUN.legacy_h035_expected_protocol_hash,
        "H": abs(float(_legacy_manifest.get("actual_calibration_H", np.nan)) - 0.35) < 1e-12,
        "calibration_spec": _legacy_manifest.get("calibration_spec") == "preliminary_h035",
        "policy_observability": _legacy_manifest.get("policy_access_to_latent_simulator_state") == "None",
        "current_features": list(_legacy_manifest.get("current_features", [])) == list(CURRENT_FEATURES) if "CURRENT_FEATURES" in globals() else True,
    }
    if not all(_legacy_checks.values()):
        raise RuntimeError(f"Legacy H=0.35 run-manifest validation failed: {_legacy_checks}")
    print("Validated legacy H=0.35 run:", LEGACY_H035_OUTPUT_DIR)


,check,value,status
0,profile,full,PASS
1,rebuild_from,reporting,PASS
2,raw_option_path,"C:\Users\lhkke\OneDrive - Queen Mary, Universi...",PASS
3,raw_crsp_path,"C:\Users\lhkke\OneDrive - Queen Mary, Universi...",PASS
4,cache_directory,"C:\Users\lhkke\OneDrive - Queen Mary, Universi...",PASS
5,output_directory,"C:\Users\lhkke\OneDrive - Queen Mary, Universi...",PASS
6,calibration_spec,preliminary_h035,PASS
7,H,0.35,PASS
8,policy_latent_state_access,None,PASS
9,heston_cache_basic_validity,True,INFO


In [10]:

# ============================================================
# Cache provenance and stale-cache policy
# ============================================================

METHODOLOGY_VERSION = "final_methodology_hedging_focused_spx_panel_v2_polished_active_models"
CACHE_SCHEMA_VERSION = "strict-cache-hedging-focused-spx-panel-v2"
SURFACE_CACHE_VERSION = "partA_surface_discipline_v1"


# These upstream artefacts are expensive deterministic inputs. They are validated by
# artefact bytes, dependency fingerprints, required columns, and semantic version columns
# checked after loading. Notebook reporting edits must not invalidate them.
CONTENT_VALIDATED_CACHE_SCOPES = frozenset({
    "surface",
    "forward_variance",
    "heston",
    "lifted",
    "seed_panel_candidate",
    "seed_panel",
    "simulation_feasibility",
    "simulation_validation",
    "no_trade_band",
})


def cache_scope_is_content_validated(scope: str) -> bool:
    return scope in CONTENT_VALIDATED_CACHE_SCOPES


def calibration_rebuild_allowed(cfg: ExperimentConfig) -> bool:
    return bool(
        cfg.flags.RUN_HESTON_CALIBRATION
        or cfg.flags.RUN_LIFTED_CALIBRATION
        or cfg.flags.ALLOW_CALIBRATION_REBUILD_FROM_DOWNSTREAM
    )


# These fields affect runtime mechanics rather than the fitted lifted-Heston calibration panel.
# They do not force recalibration when changed.
LIFTED_CALIBRATION_CONFIG_FIELDS = (
    "H",
    "n_factors",
    "rn_scale",
    "factor_grid_method",
    "kernel_t_min",
    "kernel_t_max",
    "kernel_grid_size",
    "variance_floor",
    "pricing_nodes",
    "riccati_steps",
    "optimizer_max_nfev",
    "calibration_vega_floor",
    "calibration_parameterization",
    "calibration_max_dates",
    "fit_vega_rmse_warn",
    "fit_vega_rmse_bad",
    "initial_curve_ridge",
    "initial_curve_max_rmse",
)

# Fields introduced after the original lifted-calibration cache was written. A manifest
# produced by the old hash logic is still valid if it matches the old payload with only
# these runtime fields absent. Calibration-relevant changes are still rejected.
LIFTED_RUNTIME_ONLY_HASH_COMPAT_FIELDS = frozenset({
    "simulation_internal_substeps_per_hedge_step",
    "simulation_max_internal_substeps_per_hedge_step",
})


# Older lifted-Heston artefacts were sometimes written with a full lifted-config hash.
# Diagnostic-only fields such as H-sensitivity grids and reporting switches do not change
# fitted lifted calibrations. These artefacts may pass the config-hash gate only if the
# semantic validation below still matches H, N, r_N, backend, parameterisation, calibrated
# parameter names and forward-variance regularisation.
LIFTED_SEMANTICALLY_VALIDATED_CACHE_ARTIFACTS = frozenset({
    "lifted_calibration",
    "lifted_hedging_panel_calibration",
    "lifted_fit_table",
    "lifted_history",
    "lifted_timing",
})


def _allow_lifted_semantic_cache_compatibility(artifact_name: str, scope: str) -> bool:
    return scope == "lifted" and artifact_name in LIFTED_SEMANTICALLY_VALIDATED_CACHE_ARTIFACTS


def _jsonable(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, dict):
        return {str(k): _jsonable(v) for k, v in sorted(value.items(), key=lambda kv: str(kv[0]))}
    if isinstance(value, (list, tuple)):
        return [_jsonable(v) for v in value]
    return value


def _scope_payload(cfg: ExperimentConfig, scope: str) -> dict[str, Any]:
    sections_by_scope = {
        "surface": ("market",),
        "forward_variance": ("market",),
        "heston": ("market", "calibration"),
        "lifted": ("market", "calibration", "lifted_calibration"),
        "seed_panel_candidate": ("market", "calibration", "lifted", "hedging", "panel"),
        "seed_panel": ("market", "calibration", "lifted", "hedging", "panel"),
        "simulation_feasibility": ("lifted", "hedging", "panel"),
        "simulation_validation": ("lifted", "hedging", "panel"),
        "raw_tensor_bank": ("market", "calibration", "lifted", "hedging", "panel", "model"),
        "no_trade_band": ("market", "calibration", "lifted", "hedging", "panel"),
        "training": ("market", "calibration", "lifted", "hedging", "panel", "model"),
        "factor_diagnostics": ("lifted", "hedging", "panel"),
        "evaluation": ("market", "calibration", "lifted", "hedging", "panel", "model"),
        "inference": ("hedging", "panel", "model", "inference"),
        "robustness": ("market", "calibration", "lifted", "hedging", "panel", "model"),
    }
    if scope not in sections_by_scope:
        raise KeyError(f"Unknown cache scope: {scope}")
    payload = {
        "methodology_version": METHODOLOGY_VERSION,
        "cache_schema_version": CACHE_SCHEMA_VERSION,
        "scope": scope,
    }
    stochastic_scopes = {
        "seed_panel_candidate",
        "seed_panel",
        "simulation_feasibility",
        "simulation_validation",
        "raw_tensor_bank",
        "no_trade_band",
        "training",
        "factor_diagnostics",
        "evaluation",
        "inference",
    }
    if scope in stochastic_scopes:
        payload["random_seed"] = int(cfg.random_seed)
    for name in sections_by_scope[scope]:
        if name == "lifted_calibration":
            lifted_dict = asdict(cfg.lifted)
            payload["lifted"] = _jsonable({k: lifted_dict[k] for k in LIFTED_CALIBRATION_CONFIG_FIELDS})
        else:
            payload[name] = _jsonable(asdict(getattr(cfg, name)))
    return payload


def _legacy_scope_payload_full_lifted_without_runtime_refinement(cfg: ExperimentConfig, scope: str) -> dict[str, Any]:
    """Hash payload used before C-LP internal-timestep fields existed.

    This exists only to read already-built calibration caches. It does not allow
    calibration-method changes through the gate: every old calibration, market and lifted
    field must still match exactly, apart from the newly added runtime-only simulator knobs.
    """
    sections_by_scope = {
        "surface": ("market",),
        "forward_variance": ("market",),
        "heston": ("market", "calibration"),
        "lifted": ("market", "calibration", "lifted"),
        "seed_panel_candidate": ("market", "calibration", "lifted", "hedging", "panel"),
        "seed_panel": ("market", "calibration", "lifted", "hedging", "panel"),
        "simulation_feasibility": ("lifted", "hedging", "panel"),
        "simulation_validation": ("lifted", "hedging", "panel"),
        "raw_tensor_bank": ("market", "calibration", "lifted", "hedging", "panel", "model"),
        "no_trade_band": ("market", "calibration", "lifted", "hedging", "panel"),
        "training": ("market", "calibration", "lifted", "hedging", "panel", "model"),
        "factor_diagnostics": ("lifted", "hedging", "panel"),
        "evaluation": ("market", "calibration", "lifted", "hedging", "panel", "model"),
        "inference": ("hedging", "panel", "model", "inference"),
        "robustness": ("market", "calibration", "lifted", "hedging", "panel", "model"),
    }
    if scope not in sections_by_scope:
        raise KeyError(f"Unknown cache scope: {scope}")
    payload = {
        "methodology_version": METHODOLOGY_VERSION,
        "cache_schema_version": CACHE_SCHEMA_VERSION,
        "scope": scope,
    }
    stochastic_scopes = {
        "seed_panel_candidate",
        "seed_panel",
        "simulation_feasibility",
        "simulation_validation",
        "raw_tensor_bank",
        "no_trade_band",
        "training",
        "factor_diagnostics",
        "evaluation",
        "inference",
    }
    if scope in stochastic_scopes:
        payload["random_seed"] = int(cfg.random_seed)
    for name in sections_by_scope[scope]:
        section_payload = asdict(getattr(cfg, name))
        if name == "lifted":
            for runtime_key in LIFTED_RUNTIME_ONLY_HASH_COMPAT_FIELDS:
                section_payload.pop(runtime_key, None)
        payload[name] = _jsonable(section_payload)
    return payload


def _hash_payload(payload: dict[str, Any]) -> str:
    raw = json.dumps(payload, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def cache_config_hash(cfg: ExperimentConfig, scope: str) -> str:
    return _hash_payload(_scope_payload(cfg, scope))


def compatible_cache_config_hashes(cfg: ExperimentConfig, scope: str) -> set[str]:
    hashes = {cache_config_hash(cfg, scope)}
    if scope == "lifted":
        hashes.add(_hash_payload(_legacy_scope_payload_full_lifted_without_runtime_refinement(cfg, scope)))
    return hashes


def cache_manifest_path(path: Path) -> Path:
    return Path(str(path) + ".manifest.json")


def _sha256_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_bytes), b""):
            h.update(chunk)
    return h.hexdigest()


def _fingerprint_cache_path(path: Path, cfg: ExperimentConfig) -> Path:
    key = hashlib.sha256(str(Path(path).resolve()).encode("utf-8")).hexdigest()
    return cfg.paths.project_root / cfg.paths.cache_dir / "_source_hashes" / f"{key}.json"


def file_fingerprint(path: Path, cfg: ExperimentConfig) -> dict[str, Any]:
    """Return a cached content fingerprint, with progress for expensive hash builds.

    Several downstream benchmark cache checks depend on large parquet files. If the sidecar
    fingerprint is absent or stale, hashing can take long enough that the notebook appears
    frozen. That is not a methodology event; it is just Python silently chewing bytes like
    a bored termite. We therefore print before/after any real hash computation.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    st = path.stat()
    fp_path = _fingerprint_cache_path(path, cfg)
    if fp_path.exists():
        try:
            cached = json.loads(fp_path.read_text())
            if cached.get("size") == st.st_size and cached.get("mtime_ns") == st.st_mtime_ns:
                return cached
        except json.JSONDecodeError as exc:
            warnings.warn(f"Ignoring corrupt fingerprint sidecar {fp_path}: {exc}")
    size_mb = st.st_size / (1024.0 * 1024.0)
    t0 = time.perf_counter()
    print(f"[cache fingerprint] hashing {path.name} ({size_mb:.1f} MB); sidecar={fp_path.name}", flush=True)
    record = {
        "path": str(path.resolve()),
        "size": int(st.st_size),
        "mtime_ns": int(st.st_mtime_ns),
        "sha256": _sha256_file(path),
    }
    fp_path.parent.mkdir(parents=True, exist_ok=True)
    fp_path.write_text(json.dumps(record, indent=2, sort_keys=True))
    print(f"[cache fingerprint] complete {path.name} in {time.perf_counter() - t0:.2f}s", flush=True)
    return record

def dependency_fingerprints(paths: Iterable[Path], cfg: ExperimentConfig) -> list[dict[str, Any]]:
    return [file_fingerprint(Path(p), cfg) for p in paths]


def _stable_fingerprint_signature(fingerprint: dict[str, Any] | None) -> dict[str, Any] | None:
    """Return the content identity of a cache artefact.

    The full sidecar fingerprint intentionally records path and mtime for debugging, but
    those fields are not semantic cache identity. A copied cache can have a different
    absolute path or modification time while containing identical bytes. Rejecting that
    is not reproducibility, it is bureaucracy wearing a lab coat.
    """
    if not isinstance(fingerprint, dict):
        return None
    try:
        return {
            "size": int(fingerprint["size"]),
            "sha256": str(fingerprint["sha256"]),
        }
    except (KeyError, TypeError, ValueError):
        return None


def _fingerprints_match(current: dict[str, Any], recorded: dict[str, Any] | None) -> bool:
    return _stable_fingerprint_signature(current) == _stable_fingerprint_signature(recorded)


def _dependency_fingerprints_match(current: list[dict[str, Any]], recorded: list[dict[str, Any]] | None) -> bool:
    if recorded is None:
        recorded = []
    if len(current) != len(recorded):
        return False
    return all(_fingerprints_match(c, r) for c, r in zip(current, recorded))


def write_cache_manifest(path: Path, artifact_name: str, scope: str, cfg: ExperimentConfig, *, dependencies: Iterable[Path] = (), row_count: int | None = None, columns: Iterable[str] = (), extra: dict[str, Any] | None = None) -> None:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Cannot manifest missing artefact: {path}")
    manifest = {
        "artifact_name": artifact_name,
        "artifact_path": str(path.resolve()),
        "methodology_version": METHODOLOGY_VERSION,
        "cache_schema_version": CACHE_SCHEMA_VERSION,
        "scope": scope,
        "config_hash": cache_config_hash(cfg, scope),
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "artifact_fingerprint": file_fingerprint(path, cfg),
        "dependencies": dependency_fingerprints(dependencies, cfg),
        "row_count": None if row_count is None else int(row_count),
        "columns": sorted(map(str, columns)),
        "extra": _jsonable(extra or {}),
    }
    mp = cache_manifest_path(path)
    mp.parent.mkdir(parents=True, exist_ok=True)
    mp.write_text(json.dumps(manifest, indent=2, sort_keys=True))


def write_dataframe_cache(df: pd.DataFrame, path: Path, artifact_name: str, scope: str, cfg: ExperimentConfig, *, dependencies: Iterable[Path] = (), extra: dict[str, Any] | None = None) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False)
    write_cache_manifest(path, artifact_name, scope, cfg, dependencies=dependencies, row_count=len(df), columns=df.columns, extra=extra)


def validate_cache_manifest(path: Path, artifact_name: str, scope: str, cfg: ExperimentConfig, *, dependencies: Iterable[Path] = ()) -> dict[str, Any]:
    path = Path(path)
    mp = cache_manifest_path(path)
    if not path.exists() or not mp.exists():
        raise FileNotFoundError(f"Cache or manifest missing for {artifact_name}: {path}")
    manifest = json.loads(mp.read_text())
    if manifest.get("artifact_name") != artifact_name:
        raise ValueError(f"Wrong artifact manifest for {path}: {manifest.get('artifact_name')!r}")

    current_artifact_fp = file_fingerprint(path, cfg)
    if not _fingerprints_match(current_artifact_fp, manifest.get("artifact_fingerprint")):
        raise ValueError(f"Artefact content fingerprint mismatch for {artifact_name}.")

    current_dependency_fps = dependency_fingerprints(dependencies, cfg)
    if not _dependency_fingerprints_match(current_dependency_fps, manifest.get("dependencies", [])):
        raise ValueError(f"Dependency content fingerprint mismatch for {artifact_name}; rebuild required.")

    if cache_scope_is_content_validated(scope):
        return manifest

    if manifest.get("methodology_version") != METHODOLOGY_VERSION:
        raise ValueError(f"Methodology version mismatch for {artifact_name}.")
    if manifest.get("cache_schema_version") != CACHE_SCHEMA_VERSION:
        raise ValueError(f"Cache schema mismatch for {artifact_name}.")
    if manifest.get("config_hash") not in compatible_cache_config_hashes(cfg, scope):
        if _allow_lifted_semantic_cache_compatibility(artifact_name, scope):
            warnings.warn(
                f"{artifact_name} has a legacy lifted config hash. Accepting it only through "
                "semantic lifted-cache validation; calibration-relevant fields are checked after load.",
                RuntimeWarning,
            )
        else:
            raise ValueError(f"Configuration hash mismatch for {artifact_name}; rebuild required.")
    return manifest

def load_dataframe_cache(path: Path, artifact_name: str, scope: str, cfg: ExperimentConfig, *, required_columns: Iterable[str] = (), dependencies: Iterable[Path] = ()) -> pd.DataFrame:
    manifest = validate_cache_manifest(path, artifact_name, scope, cfg, dependencies=dependencies)
    df = pd.read_parquet(path)
    missing = set(required_columns) - set(df.columns)
    if missing:
        raise ValueError(f"{artifact_name} cache missing required columns: {sorted(missing)}")
    if manifest.get("row_count") is not None and len(df) != int(manifest["row_count"]):
        raise ValueError(f"{artifact_name} row-count mismatch against manifest.")
    return df


def cache_valid(path: Path, artifact_name: str, scope: str, cfg: ExperimentConfig, *, dependencies: Iterable[Path] = ()) -> bool:
    try:
        validate_cache_manifest(path, artifact_name, scope, cfg, dependencies=dependencies)
        return True
    except (FileNotFoundError, ValueError, json.JSONDecodeError):
        return False


def prepare_checkpoint_directory(directory: Path, artifact_name: str, scope: str, cfg: ExperimentConfig, dependencies: Iterable[Path]) -> None:
    directory = Path(directory)
    marker = directory / "_checkpoint_manifest.json"
    expected = {
        "artifact_name": artifact_name,
        "methodology_version": METHODOLOGY_VERSION,
        "cache_schema_version": CACHE_SCHEMA_VERSION,
        "config_hash": cache_config_hash(cfg, scope),
        "dependencies": dependency_fingerprints(dependencies, cfg),
    }
    existing = None
    if marker.exists():
        try:
            existing = json.loads(marker.read_text())
        except json.JSONDecodeError as exc:
            warnings.warn(f"Ignoring corrupt checkpoint manifest {marker}: {exc}")
    if directory.exists() and any(directory.iterdir()) and existing != expected:
        shutil.rmtree(directory, ignore_errors=True)
    directory.mkdir(parents=True, exist_ok=True)
    marker.write_text(json.dumps(expected, indent=2, sort_keys=True))


def checkpoint_dependency_payload(paths: Iterable[Path], cfg: ExperimentConfig) -> list[dict[str, Any]]:
    return dependency_fingerprints(paths, cfg)


def require_columns(df: pd.DataFrame, columns: Iterable[str], label: str) -> None:
    missing = set(columns) - set(df.columns)
    if missing:
        raise ValueError(f"{label} missing required columns: {sorted(missing)}")


In [11]:
def _rooted(path: Path, cfg: ExperimentConfig | None = None) -> Path:
    cfg = CFG if cfg is None else cfg
    path = Path(path)
    return path if path.is_absolute() else cfg.paths.project_root / path


def table_path(name: str, cfg: ExperimentConfig | None = None) -> Path:
    cfg = CFG if cfg is None else cfg
    return _rooted(cfg.paths.table_dir, cfg) / name


def figure_path(name: str, cfg: ExperimentConfig | None = None) -> Path:
    cfg = CFG if cfg is None else cfg
    return _rooted(cfg.paths.figure_dir, cfg) / name


def cache_path(name: str, cfg: ExperimentConfig | None = None) -> Path:
    cfg = CFG if cfg is None else cfg
    return _rooted(cfg.paths.cache_dir, cfg) / name


def ensure_output_dirs(cfg: ExperimentConfig) -> None:
    for path in [cfg.paths.cache_dir, cfg.paths.output_dir, cfg.paths.figure_dir, cfg.paths.table_dir, cfg.paths.checkpoint_dir]:
        _rooted(path, cfg).mkdir(parents=True, exist_ok=True)


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    if TORCH_AVAILABLE:
        torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


def get_device(cfg: ExperimentConfig) -> Any:
    if TORCH_AVAILABLE and torch.cuda.is_available(): return torch.device("cuda")
    if TORCH_AVAILABLE: return torch.device("cpu")
    return "cpu"


def describe_torch_device(device: Any, cfg: ExperimentConfig) -> None:
    if not TORCH_AVAILABLE:
        print("PyTorch unavailable. Neural training/evaluation cannot run.")
        return
    print(f"PyTorch device: {device}")
    if isinstance(device, torch.device) and device.type == "cuda":
        idx = torch.cuda.current_device(); props = torch.cuda.get_device_properties(idx)
        print({"cuda_available": True, "gpu_name": torch.cuda.get_device_name(idx), "gpu_memory_gb": round(props.total_memory / 1024**3, 2)})
    else:
        print({"cuda_available": False, "note": "PyTorch sections will run on CPU."})


def save_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, default=lambda x: str(x)))


def cache_status_table(cfg: ExperimentConfig) -> pd.DataFrame:
    entries = [
        ("surface", cfg.paths.surface_cache_path),
        ("heston", cfg.paths.heston_calibration_path),
        ("forward_variance", cfg.paths.forward_variance_path),
        ("lifted", cfg.paths.lifted_calibration_path),
        ("seed_panel", cfg.paths.seed_panel_path),
        ("training_history", cfg.paths.training_history_path),
        ("evaluation", cfg.paths.evaluation_path),
    ]
    return pd.DataFrame([
        {"artefact": name, "file": str(path), "exists": Path(path).exists(), "manifest": cache_manifest_path(Path(path)).exists()}
        for name, path in entries
    ])


def assert_final_config(cfg: ExperimentConfig) -> pd.DataFrame:
    expected_buckets = {"1W": (7, 7, 10), "2W": (14, 11, 21), "1M": (30, 22, 45), "2M": (60, 46, 75), "3M": (90, 76, 120)}
    if dict(cfg.hedging.maturity_buckets) != expected_buckets:
        raise AssertionError("Maturity-bucket specification drifted.")
    if abs(cfg.lifted.H - RUN.calibration_H) > 1e-12 or cfg.lifted.n_factors != 20 or abs(cfg.lifted.rn_scale - 2.5) > 1e-12:
        raise AssertionError("Lifted-Heston H/N/r_N specification disagrees with RUN.calibration_spec.")
    observed_plan = execution_plan_signature(cfg.flags)
    if observed_plan != EXPECTED_EXECUTION_SIGNATURE:
        raise AssertionError(
            "CFG execution-plan signature is stale or inconsistent. "
            f"expected={EXPECTED_EXECUTION_SIGNATURE}, observed={observed_plan}. "
            "Restart the kernel and run all cells from the top."
        )
    if cfg.paths.output_dir != _RESOLVED_OUTPUT_DIR or cfg.paths.cache_dir != _RESOLVED_CACHE_DIR:
        raise AssertionError("PathConfig disagrees with the user RUN request.")
    out = pd.DataFrame([
        {"gate": "profile", "status": "PASS", "value": RUN.profile},
        {"gate": "rebuild_from", "status": "PASS", "value": RUN.rebuild_from},
        {"gate": "calibration_spec", "status": "PASS", "value": RUN.calibration_spec},
        {"gate": "lag_consistent_h035_retraining", "status": "PASS", "value": True},
        {"gate": "H_N_rN", "status": "PASS", "value": (cfg.lifted.H, cfg.lifted.n_factors, cfg.lifted.rn_scale)},
        {"gate": "output_dir", "status": "PASS", "value": str(cfg.paths.output_dir)},
        {"gate": "cache_dir", "status": "PASS", "value": str(cfg.paths.cache_dir)},
        {"gate": "latent_policy_state", "status": "PASS", "value": "None"},
    ])
    display(out)
    return out



In [12]:
# Final output, reproducibility and runtime setup.
# All functions used here are defined in the immediately preceding cell.
ensure_output_dirs(CFG)
set_global_seed(CFG.random_seed)
CONFIG_AUDIT = assert_final_config(CFG)
display(cache_status_table(CFG))

DEVICE = get_device(CFG)
DTYPE = torch.float32 if TORCH_AVAILABLE else None
AMP_ENABLED = bool(
    TORCH_AVAILABLE
    and isinstance(DEVICE, torch.device)
    and DEVICE.type == "cuda"
    and CFG.model.use_mixed_precision
)

describe_torch_device(DEVICE, CFG)
print("Cache directory:", CFG.paths.cache_dir)
print("Output directory:", CFG.paths.output_dir)
print("Lifted-Heston calibration specification:", RUN.calibration_spec)
print("Lifted-Heston H:", CFG.lifted.H)
print("Evidence eligibility: H035_LAG_CONSISTENT_RETRAIN")
print("Policy access to latent simulator state: None")

save_json(
    CFG.paths.output_dir / "final_methodology_config.json",
    asdict(CFG),
)


,gate,status,value
0,profile,PASS,full
1,rebuild_from,PASS,reporting
2,calibration_spec,PASS,preliminary_h035
3,lag_consistent_h035_retraining,PASS,True
4,H_N_rN,PASS,"(0.35, 20, 2.5)"
5,output_dir,PASS,"C:\Users\lhkke\OneDrive - Queen Mary, Universi..."
6,cache_dir,PASS,"C:\Users\lhkke\OneDrive - Queen Mary, Universi..."
7,latent_policy_state,PASS,None


,artefact,file,exists,manifest
0,surface,"C:\Users\lhkke\OneDrive - Queen Mary, Universi...",True,True
1,heston,"C:\Users\lhkke\OneDrive - Queen Mary, Universi...",True,True
2,forward_variance,"C:\Users\lhkke\OneDrive - Queen Mary, Universi...",True,True
3,lifted,"C:\Users\lhkke\OneDrive - Queen Mary, Universi...",True,True
4,seed_panel,"C:\Users\lhkke\OneDrive - Queen Mary, Universi...",True,True
5,training_history,"C:\Users\lhkke\OneDrive - Queen Mary, Universi...",False,False
6,evaluation,"C:\Users\lhkke\OneDrive - Queen Mary, Universi...",False,False


PyTorch device: cuda
{'cuda_available': True, 'gpu_name': 'NVIDIA GeForce RTX 5070', 'gpu_memory_gb': 11.94}
Cache directory: C:\Users\lhkke\OneDrive - Queen Mary, University of London\Documents\Dissertation\Deep-Hedging-under-Lifted-Heston-MSc-Dissertation-\cache_final
Output directory: C:\Users\lhkke\OneDrive - Queen Mary, University of London\Documents\Dissertation\Deep-Hedging-under-Lifted-Heston-MSc-Dissertation-\outputs_observable_h035_lag_consistent_retrained_full
Lifted-Heston calibration specification: preliminary_h035
Lifted-Heston H: 0.35
Evidence eligibility: H035_LAG_CONSISTENT_RETRAIN
Policy access to latent simulator state: None


In [13]:
# Optional cache hygiene helper for interrupted PyTorch saves

def remove_interrupted_torch_temp_files(cfg: ExperimentConfig = CFG) -> list[str]:
    """Remove orphan temp files left by failed atomic PyTorch saves.

    This does not delete final .pt caches or manifests. It only removes hidden
    temporary files created during failed saves.
    """
    roots = [cfg.paths.cache_dir, cfg.paths.output_dir]
    removed: list[str] = []
    for root in roots:
        root = _rooted(root, cfg) if "_rooted" in globals() else Path(root)
        if not root.exists():
            continue
        for p in root.rglob(".*.pt.*.tmp"):
            try:
                p.unlink()
                removed.append(str(p))
            except OSError:
                pass
    print(f"Removed {len(removed)} interrupted torch-save temp files.")
    return removed


# 2. Economic setting and information filtration

The instrument is a short, near-ATM SPX straddle hedged daily using the underlying index proxy. Transaction costs are proportional to traded notional. The policy is a bounded residual correction around contemporaneous Black–Scholes straddle delta.

At hedge time \(t\), the policy may use only quantities observable at or before \(t\): contract state, contemporaneous Black–Scholes delta, contemporaneous implied-volatility-surface summaries, causal returns and realised-volatility summaries, the previous hedge and the transaction-cost rate. Latent simulator states are internal pricing and path-generation variables only.

In [14]:
def norm_cdf(x: np.ndarray | float) -> np.ndarray | float:
    if SCIPY_AVAILABLE:
        from scipy.stats import norm
        return norm.cdf(x)
    return 0.5 * (1.0 + np.vectorize(math.erf)(np.asarray(x) / math.sqrt(2.0)))


def norm_pdf(x: np.ndarray | float) -> np.ndarray | float:
    return np.exp(-0.5 * np.asarray(x) ** 2) / math.sqrt(2.0 * math.pi)


def bs_d1_d2(S: np.ndarray | float, K: np.ndarray | float, r: float, q: float, T: np.ndarray | float, sigma: np.ndarray | float) -> tuple[np.ndarray, np.ndarray]:
    S = np.asarray(S, dtype=float)
    K = np.asarray(K, dtype=float)
    T = np.maximum(np.asarray(T, dtype=float), 1e-12)
    sigma = np.maximum(np.asarray(sigma, dtype=float), 1e-12)
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return d1, d2


def bs_price(S: float | np.ndarray, K: float | np.ndarray, r: float, q: float, T: float | np.ndarray, sigma: float | np.ndarray, option_type: str = "call") -> np.ndarray:
    d1, d2 = bs_d1_d2(S, K, r, q, T, sigma)
    S = np.asarray(S, dtype=float)
    K = np.asarray(K, dtype=float)
    T = np.asarray(T, dtype=float)
    call = S * np.exp(-q * T) * norm_cdf(d1) - K * np.exp(-r * T) * norm_cdf(d2)
    if option_type.lower().startswith("c"):
        return np.asarray(call)
    put = call - S * np.exp(-q * T) + K * np.exp(-r * T)
    return np.asarray(put)


def bs_delta(S: float | np.ndarray, K: float | np.ndarray, r: float, q: float, T: float | np.ndarray, sigma: float | np.ndarray, option_type: str = "call") -> np.ndarray:
    d1, _ = bs_d1_d2(S, K, r, q, T, sigma)
    call_delta = np.exp(-q * np.asarray(T, dtype=float)) * norm_cdf(d1)
    if option_type.lower().startswith("c"):
        return np.asarray(call_delta)
    return np.asarray(call_delta - np.exp(-q * np.asarray(T, dtype=float)))


def bs_vega(S: float | np.ndarray, K: float | np.ndarray, r: float, q: float, T: float | np.ndarray, sigma: float | np.ndarray) -> np.ndarray:
    d1, _ = bs_d1_d2(S, K, r, q, T, sigma)
    S = np.asarray(S, dtype=float)
    T = np.asarray(T, dtype=float)
    return S * np.exp(-q * T) * np.sqrt(np.maximum(T, 1e-12)) * norm_pdf(d1)


def implied_vol_single(price: float, S: float, K: float, r: float, q: float, T: float, option_type: str = "call", low: float = 1e-4, high: float = 5.0) -> float:
    if not SCIPY_AVAILABLE or not np.isfinite(price) or price <= 0 or T <= 0:
        return np.nan
    def objective(sig: float) -> float:
        return float(bs_price(S, K, r, q, T, sig, option_type=option_type) - price)
    try:
        f_low, f_high = objective(low), objective(high)
        if f_low * f_high > 0:
            return np.nan
        return float(brentq(objective, low, high, maxiter=100))
    except (ValueError, RuntimeError, FloatingPointError, OverflowError):
        return np.nan


def implied_vol_vectorised(df: pd.DataFrame, price_col: str, option_col: str = "option_type") -> np.ndarray:
    vols = []
    for row in df.itertuples(index=False):
        vols.append(implied_vol_single(
            price=float(getattr(row, price_col)),
            S=float(getattr(row, "spot")),
            K=float(getattr(row, "strike")),
            r=float(getattr(row, "r")),
            q=float(getattr(row, "q")),
            T=float(getattr(row, "T")),
            option_type=str(getattr(row, option_col)),
        ))
    return np.asarray(vols, dtype=float)

In [15]:

REQUIRED_OPTION_COLUMNS = {
    "date", "exdate", "strike", "option_type", "bid", "ask"
}

RISK_FREE_RATE_ALIASES = {
    "risk_free_rate": "r",
    "riskfree_rate": "r",
    "risk_free": "r",
    "interest_rate": "r",
    "zero_rate": "r",
    "rf_rate": "r",
    "rf": "r",
    "rate": "r",
}

CRSP_SPOT_ALIASES = {
    "caldt": "date",
    "spindx": "spot",
    "spx": "spot",
    "index_level": "spot",
    "close": "spot",
    "close_price": "spot",
}

OPTION_COLUMN_ALIASES = {
    "cp_flag": "option_type",
    "type": "option_type",
    "expiry": "exdate",
    "expiration": "exdate",
    "best_bid": "bid",
    "best_offer": "ask",
    "impl_volatility": "iv",
    "implied_volatility": "iv",
    "forward_price": "vendor_forward",
    **RISK_FREE_RATE_ALIASES,
}

RAW_OPTION_CANDIDATE_COLUMNS = [
    "secid", "ticker", "date", "exdate", "cp_flag", "option_type", "type",
    "strike_price", "strike", "best_bid", "best_offer", "bid", "ask",
    "impl_volatility", "implied_volatility", "iv",
    "delta", "gamma", "vega", "volume", "open_interest", "optionid",
    "forward_price", "vendor_forward", "r", "risk_free_rate", "riskfree_rate", "risk_free",
    "interest_rate", "zero_rate", "rf_rate", "rf", "rate", "q",
    "underlying_price", "underlying_close", "S0",
]

MINIMAL_CLEAN_COLUMNS = [
    "date", "exdate", "option_type", "strike", "bid", "ask", "mid", "spread", "spread_frac",
    "dte", "T", "spot", "spot_source", "r", "r_source", "iv", "volume", "open_interest",
    "vendor_forward", "delta", "gamma", "vega", "optionid", "quote_quality_weight",
]

FINAL_SURFACE_COLUMNS = [
    "date", "exdate", "strike", "call_equiv_mid", "call_mid", "iv_market", "iv_proxy",
    "bs_vega", "fit_weight", "raw_fit_weight", "quote_quality_weight", "median_rel_spread",
    "dte", "T", "maturity_bucket", "spot_ref", "forward_ref", "forward_source",
    "vendor_forward_ref", "vendor_forward_rel_diff", "q", "q_proxy", "r", "r_source", "log_moneyness",
    "n_quotes", "put_via_parity_share", "quote_source_mix", "open_interest", "volume",
    "vertical_violations", "slope_lower_violations", "slope_upper_violations",
    "butterfly_violations", "has_static_arb_violation",
]


def table_columns(path: Path) -> list[str]:
    """Return available columns without loading the full table."""
    suffix = path.suffix.lower()
    if suffix in {".parquet", ".pq"} and PYARROW_AVAILABLE:
        return list(pq.read_schema(path).names)
    if suffix in {".csv", ".gz"}:
        return list(pd.read_csv(path, nrows=0).columns)
    return list(pd.read_parquet(path, engine="pyarrow").columns)


def existing_columns(path: Path, candidates: list[str]) -> list[str]:
    available = set(table_columns(path))
    cols = [c for c in candidates if c in available]
    if not cols:
        raise ValueError(f"None of the candidate columns were found in {path}.")
    return cols


def load_table(path: Path, columns: list[str] | None = None) -> pd.DataFrame:
    """Load a parquet/csv table from disk, optionally with a column projection."""
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    suffix = path.suffix.lower()
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path, columns=columns)
    if suffix in {".csv", ".gz"}:
        return pd.read_csv(path, usecols=columns)
    raise ValueError(f"Unsupported file extension for {path}: {path.suffix}")


def load_raw_option_data(path: Path, columns: list[str] | None = None) -> pd.DataFrame:
    """Load raw OptionMetrics option data from parquet or CSV."""
    return load_table(path, columns=columns)


def iter_raw_option_batches(path: Path, cfg: MarketConfig) -> Iterable[pd.DataFrame]:
    """
    Stream raw OptionMetrics rows in batches with only the required columns.

    This preserves the memory-safe behaviour of the new notebook while avoiding the
    giant full-table copies that previously turned pandas into a space heater.
    """
    columns = existing_columns(path, RAW_OPTION_CANDIDATE_COLUMNS)
    suffix = path.suffix.lower()
    if suffix in {".parquet", ".pq"} and PYARROW_AVAILABLE:
        dataset = ds.dataset(str(path), format="parquet")
        scanner = dataset.scanner(columns=columns, batch_size=cfg.raw_batch_rows)
        for batch in scanner.to_batches():
            if batch.num_rows:
                yield batch.to_pandas()
        return
    if suffix in {".csv", ".gz"}:
        for chunk in pd.read_csv(path, usecols=columns, chunksize=cfg.raw_batch_rows):
            yield chunk
        return
    yield load_raw_option_data(path, columns=columns)


def load_crsp_spx(path: Path) -> pd.DataFrame:
    """
    Load the CRSP S&P index-level file used as the daily SPX spot source.

    Expected old-notebook convention:
    - caldt  -> date
    - spindx -> spot
    """
    crsp = load_table(path)
    crsp = crsp.rename(columns={k: v for k, v in CRSP_SPOT_ALIASES.items() if k in crsp.columns})
    missing = {"date", "spot"} - set(crsp.columns)
    if missing:
        raise ValueError(
            "CRSP spot file must contain date/spot fields. "
            "Expected columns like caldt and spindx. "
            f"Missing after aliasing: {sorted(missing)}"
        )
    crsp = crsp[["date", "spot"]].copy()
    crsp["date"] = pd.to_datetime(crsp["date"])
    crsp["spot"] = pd.to_numeric(crsp["spot"], errors="coerce")
    crsp = (
        crsp.dropna()
        .query("spot > 0")
        .drop_duplicates("date")
        .sort_values("date")
        .reset_index(drop=True)
    )
    if crsp.empty:
        raise ValueError("CRSP spot file loaded but produced no positive spot rows.")
    return crsp


def _finite_median_or_nan(values: pd.Series | np.ndarray) -> float:
    """Finite median without NumPy all-NaN runtime warnings."""
    arr = pd.to_numeric(pd.Series(values), errors="coerce").to_numpy(dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return float("nan")
    return float(np.median(arr))


def _apply_optionmetrics_strike_scaling(df: pd.DataFrame, cfg: MarketConfig) -> pd.DataFrame:
    """Handle OptionMetrics strike_price, usually stored as strike * 1000."""
    out = df
    if "strike" not in out.columns and "strike_price" in out.columns:
        out["strike"] = pd.to_numeric(out["strike_price"], errors="coerce") / cfg.optionmetrics_strike_scale
    elif "strike" in out.columns:
        out["strike"] = pd.to_numeric(out["strike"], errors="coerce")
        if cfg.auto_rescale_large_strikes:
            med = _finite_median_or_nan(out["strike"])
            if np.isfinite(med) and med > 100_000:
                print(f"Large strike scale detected; dividing strike by {cfg.optionmetrics_strike_scale:g}.")
                out["strike"] = out["strike"] / cfg.optionmetrics_strike_scale
    return out


def standardise_option_schema(raw: pd.DataFrame, cfg: MarketConfig, crsp_spot: pd.DataFrame | None = None) -> pd.DataFrame:
    """Standardise OptionMetrics-style option data and merge CRSP SPX spot."""
    df = raw.rename(columns={k: v for k, v in OPTION_COLUMN_ALIASES.items() if k in raw.columns}, copy=False)
    df = _apply_optionmetrics_strike_scaling(df, cfg)

    missing = REQUIRED_OPTION_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f"Raw option data missing required option columns after aliasing/scaling: {sorted(missing)}")

    df["date"] = pd.to_datetime(df["date"])
    df["exdate"] = pd.to_datetime(df["exdate"])
    df["option_type"] = df["option_type"].astype(str).str.lower().str[0].map({"c": "call", "p": "put"})

    if crsp_spot is None:
        raise ValueError(
            "CRSP spot table is required. Provide CFG.paths.raw_crsp_file with CRSP S&P columns caldt and spindx."
        )
    if "spot" in df.columns:
        df = df.drop(columns=["spot"])
    df = df.merge(crsp_spot, on="date", how="left", validate="many_to_one")
    df["spot_source"] = "CRSP spindx"
    missing_spot_mask = df["spot"].isna()
    if missing_spot_mask.any():
        missing_rows = int(missing_spot_mask.sum())
        total_rows = int(len(df))
        missing_dates = df.loc[missing_spot_mask, "date"].drop_duplicates().sort_values()
        example_missing_dates = missing_dates.head(5).dt.strftime("%Y-%m-%d").tolist()
        coverage_fraction = 1.0 - (missing_rows / max(total_rows, 1))
        if cfg.crsp_spot_coverage_policy == "require_full_coverage":
            raise ValueError(
                "CRSP spot is missing for at least one option date under "
                "crsp_spot_coverage_policy='require_full_coverage'. "
                f"Missing rows={missing_rows:,}/{total_rows:,}; "
                f"missing_dates={len(missing_dates):,}; examples={example_missing_dates}."
            )
        if cfg.crsp_spot_coverage_policy != "drop_uncovered_dates":
            raise ValueError(f"Unknown crsp_spot_coverage_policy={cfg.crsp_spot_coverage_policy!r}")
        if coverage_fraction < float(cfg.min_crsp_spot_coverage_fraction):
            raise ValueError(
                "CRSP spot coverage is too low to continue without changing the explicit coverage policy. "
                f"Coverage={coverage_fraction:.4%}, required>={cfg.min_crsp_spot_coverage_fraction:.4%}; "
                f"missing rows={missing_rows:,}/{total_rows:,}; examples={example_missing_dates}."
            )
        print(
            "Dropping option rows whose trade dates are outside CRSP spindx coverage: "
            f"rows={missing_rows:,}/{total_rows:,}, dates={len(missing_dates):,}, examples={example_missing_dates}."
        )
        df = df.loc[~missing_spot_mask].copy()
        # If this streaming batch sits entirely outside CRSP coverage, return an
        # empty batch after adding the expected schema below. This is strict data
        # intersection, not spot imputation.

    if cfg.risk_free_rate_source == "raw_column":
        if "r" not in df.columns:
            raise ValueError(
                "CFG.market.risk_free_rate_source='raw_column' but the raw option data contains no recognised "
                "risk-free-rate column. Recognised aliases are: "
                f"{sorted(RISK_FREE_RATE_ALIASES)} plus 'r'."
            )
        df["r"] = pd.to_numeric(df["r"], errors="coerce")
        if df["r"].isna().any():
            raise ValueError("Raw risk-free-rate column `r` contains non-numeric or missing values; strict preprocessing will not impute rates.")
        df["r_source"] = "raw_column"
    elif cfg.risk_free_rate_source == "configured_constant":
        r_value = float(cfg.configured_risk_free_rate)
        if not np.isfinite(r_value):
            raise ValueError("CFG.market.configured_risk_free_rate must be finite.")
        df["r"] = r_value
        df["r_source"] = "configured_constant"
    else:
        raise ValueError(f"Unknown risk_free_rate_source={cfg.risk_free_rate_source!r}")

    for col, default in (("volume", np.nan), ("open_interest", np.nan), ("iv", np.nan), ("vendor_forward", np.nan), ("optionid", np.nan)):
        if col not in df.columns:
            df[col] = default

    for col in ["r", "vendor_forward", "volume", "open_interest", "iv", "delta", "gamma", "vega", "optionid"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if cfg.auto_rescale_large_strikes and "vendor_forward" in df.columns:
        vf_med = _finite_median_or_nan(df["vendor_forward"])
        k_med = _finite_median_or_nan(df["strike"])
        if np.isfinite(vf_med) and np.isfinite(k_med) and vf_med > 100_000 and k_med < 100_000:
            print(f"Large vendor forward scale detected; dividing vendor_forward by {cfg.optionmetrics_strike_scale:g}.")
            df["vendor_forward"] = df["vendor_forward"] / cfg.optionmetrics_strike_scale
    return df


def _downcast_surface_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Reduce memory footprint for cached surface panels."""
    float_cols = df.select_dtypes(include=["float64"]).columns
    int_cols = df.select_dtypes(include=["int64"]).columns
    for col in float_cols:
        df[col] = pd.to_numeric(df[col], downcast="float")
    for col in int_cols:
        if col not in {"optionid"}:
            df[col] = pd.to_numeric(df[col], downcast="integer")
    return df


def clean_option_quotes(raw: pd.DataFrame, cfg: MarketConfig, crsp_spot: pd.DataFrame | None = None) -> pd.DataFrame:
    """
    Part-A-style quote cleaning, adapted to the memory-safe batch path.

    Important differences from the earlier fresh notebook:
    - use best-bid and open-interest filters;
    - keep stricter spread control;
    - require vendor IV to exist and be positive before the surface stage;
    - use CRSP spot only;
    - use put-call parity forwards only for the primary surface.
    """
    df = standardise_option_schema(raw, cfg, crsp_spot=crsp_spot)
    df["bid"] = pd.to_numeric(df["bid"], errors="coerce")
    df["ask"] = pd.to_numeric(df["ask"], errors="coerce")
    df["strike"] = pd.to_numeric(df["strike"], errors="coerce")
    df["spot"] = pd.to_numeric(df["spot"], errors="coerce")
    df["r"] = pd.to_numeric(df["r"], errors="coerce")
    df["iv"] = pd.to_numeric(df["iv"], errors="coerce")
    df["open_interest"] = pd.to_numeric(df["open_interest"], errors="coerce")

    df["mid"] = 0.5 * (df["bid"] + df["ask"])
    df["spread"] = df["ask"] - df["bid"]
    df["spread_frac"] = df["spread"] / df["mid"].replace(0.0, np.nan)
    df["dte"] = (df["exdate"] - df["date"]).dt.days
    df["T"] = df["dte"] / cfg.calendar_days_per_year

    if cfg.preprocessing_start_date is not None:
        df = df[df["date"].ge(pd.Timestamp(cfg.preprocessing_start_date))]
    if cfg.preprocessing_end_date is not None:
        df = df[df["date"].le(pd.Timestamp(cfg.preprocessing_end_date))]

    mask = (
        df["option_type"].isin(["call", "put"])
        & np.isfinite(df["bid"])
        & np.isfinite(df["ask"])
        & np.isfinite(df["mid"])
        & np.isfinite(df["spread"])
        & np.isfinite(df["spread_frac"])
        & (df["ask"] >= df["bid"])
        & (df["bid"] >= cfg.min_best_bid)
        & (df["mid"] >= cfg.min_mid_price)
        & (df["spread"] > 0)
        & (df["spread_frac"] > cfg.min_bid_ask_spread_frac)
        & (df["spread_frac"] <= cfg.max_bid_ask_spread_frac)
        & (df["strike"] > 0)
        & (df["spot"] > 0)
        & df["r"].notna() & np.isfinite(df["r"])
        & (df["open_interest"].fillna(0) >= cfg.min_open_interest)
        & (df["iv"].notna() & (df["iv"] > 0))
        & (df["dte"].between(cfg.min_dte, cfg.max_dte))
    )
    keep_cols = [c for c in MINIMAL_CLEAN_COLUMNS if c in df.columns]
    cleaned = df.loc[mask, keep_cols].copy()
    cleaned["quote_quality_weight"] = 1.0 / cleaned["spread_frac"].clip(lower=1e-6)
    return _downcast_surface_frame(cleaned.reset_index(drop=True))


def _select_option_side(quotes: pd.DataFrame, option_type: str, price_name: str, spread_name: str) -> pd.DataFrame:
    """Aggregate duplicate quotes on one option side for parity matching."""
    side = quotes.loc[quotes["option_type"].eq(option_type)]
    if side.empty:
        return pd.DataFrame(columns=["date", "exdate", "strike", price_name, spread_name])
    return (
        side.groupby(["date", "exdate", "strike"], as_index=False)
        .agg(**{
            price_name: ("mid", "median"),
            spread_name: ("spread_frac", "median"),
        })
    )


def infer_forwards_put_call_parity(cleaned: pd.DataFrame, cfg: MarketConfig) -> pd.DataFrame:
    """
    Infer date-expiry forwards from matched call-put pairs using put-call parity.

    The primary methodology surface is parity-only. Expiries without enough valid matched call-put pairs are excluded.
    """
    key_cols = ["date", "exdate", "strike"]
    calls = _select_option_side(cleaned, "call", "call_mid", "call_rel_spread")
    puts = _select_option_side(cleaned, "put", "put_mid", "put_rel_spread")
    pairs = calls.merge(puts, on=key_cols, how="inner")

    if not pairs.empty:
        refs = (
            cleaned.groupby(key_cols, as_index=False)
            .agg(
                spot=("spot", "median"),
                T=("T", "median"),
                r=("r", "median"),
                r_source=("r_source", lambda x: ";".join(sorted(set(map(str, x.dropna()))))),
                n_raw_quotes=("option_type", "size"),
            )
        )
        pairs = pairs.merge(refs, on=key_cols, how="left")
        pairs["forward_pair"] = pairs["strike"] + np.exp(pairs["r"] * pairs["T"]) * (pairs["call_mid"] - pairs["put_mid"])
        pairs["forward_basis"] = pairs["forward_pair"] / pairs["spot"] - 1.0
        if pairs[["call_rel_spread", "put_rel_spread"]].isna().any().any():
            raise ValueError("Matched parity pairs contain missing bid-ask spreads; strict preprocessing does not impute them.")
        pairs["pair_rel_spread"] = pairs["call_rel_spread"] + pairs["put_rel_spread"]
        valid = (
            pairs["call_mid"].notna()
            & pairs["put_mid"].notna()
            & pairs["spot"].gt(0)
            & pairs["T"].gt(0)
            & pairs["forward_pair"].gt(0)
            & pairs["forward_basis"].between(-cfg.max_abs_forward_basis, cfg.max_abs_forward_basis)
        )
        pairs = pairs.loc[valid].reset_index(drop=True)

    if pairs.empty:
        forward_curve = pd.DataFrame(columns=[
            "date", "exdate", "spot_ref", "forward_ref", "T_ref", "r", "r_source", "q_proxy", "forward_pairs",
            "forward_basis", "forward_iqr", "forward_iqr_rel", "median_pair_rel_spread",
        ])
    else:
        forward_curve = (
            pairs.groupby(["date", "exdate"], as_index=False)
            .agg(
                spot_ref=("spot", "median"),
                forward_ref=("forward_pair", "median"),
                T_ref=("T", "median"),
                r=("r", "median"),
                r_source=("r_source", lambda x: ";".join(sorted(set(map(str, x.dropna()))))),
                forward_pairs=("forward_pair", "count"),
                forward_pair_p25=("forward_pair", lambda x: x.quantile(0.25)),
                forward_pair_p75=("forward_pair", lambda x: x.quantile(0.75)),
                median_pair_rel_spread=("pair_rel_spread", "median"),
            )
        )
        valid = forward_curve["spot_ref"].gt(0) & forward_curve["forward_ref"].gt(0) & forward_curve["T_ref"].gt(0)
        forward_curve["q_proxy"] = np.nan
        forward_curve.loc[valid, "q_proxy"] = (
            forward_curve.loc[valid, "r"]
            - np.log(forward_curve.loc[valid, "forward_ref"] / forward_curve.loc[valid, "spot_ref"])
            / forward_curve.loc[valid, "T_ref"]
        )
        forward_curve["forward_basis"] = forward_curve["forward_ref"] / forward_curve["spot_ref"] - 1.0
        forward_curve["forward_iqr"] = forward_curve["forward_pair_p75"] - forward_curve["forward_pair_p25"]
        forward_curve["forward_iqr_rel"] = forward_curve["forward_iqr"] / forward_curve["spot_ref"]
        keep = forward_curve["forward_pairs"] >= cfg.min_forward_pairs
        if cfg.max_forward_iqr_rel is not None:
            keep = keep & (forward_curve["forward_iqr_rel"] <= cfg.max_forward_iqr_rel)
        forward_curve = forward_curve.loc[keep].reset_index(drop=True)

    out = cleaned.merge(
        forward_curve[["date", "exdate", "spot_ref", "forward_ref", "T_ref", "r", "r_source", "q_proxy", "forward_pairs", "forward_basis", "forward_iqr", "median_pair_rel_spread"]],
        on=["date", "exdate"],
        how="inner",
        suffixes=("", "_curve"),
    )
    out["forward_source"] = "put_call_parity"

    out["vendor_forward_ref"] = np.nan
    out["vendor_forward_rel_diff"] = np.nan
    return _downcast_surface_frame(out)


def convert_to_call_equivalent(with_forwards: pd.DataFrame, cfg: MarketConfig) -> pd.DataFrame:
    """Apply OTM-side selection and convert puts to call-equivalent prices."""
    df = with_forwards.copy()
    df["log_moneyness"] = np.log(df["strike"] / df["forward_ref"])
    df["is_otm"] = (
        (df["option_type"].eq("call") & (df["strike"] >= df["forward_ref"]))
        | (df["option_type"].eq("put") & (df["strike"] <= df["forward_ref"]))
    )
    keep = df["log_moneyness"].abs().le(cfg.max_abs_log_moneyness)
    if cfg.use_otm_quotes:
        keep = keep & df["is_otm"]
    df = df.loc[keep].copy()

    discount = np.exp(-df["r"] * df["T"])
    df["parity_adjustment"] = discount * (df["forward_ref"] - df["strike"])
    df["call_equiv_mid"] = np.where(
        df["option_type"].eq("call"),
        df["mid"],
        df["mid"] + df["parity_adjustment"],
    )
    df["quote_source"] = np.where(df["option_type"].eq("call"), "direct_call", "put_via_parity")
    df["call_equiv_iv_proxy"] = df["iv"]
    return _downcast_surface_frame(df.reset_index(drop=True))


def collapse_to_strike_grid(quotes: pd.DataFrame) -> pd.DataFrame:
    """Collapse quote-level rows to one call-equivalent point per date-expiry-strike, as in Part A."""
    if quotes.empty:
        return pd.DataFrame(columns=FINAL_SURFACE_COLUMNS)

    out = (
        quotes.groupby(["date", "exdate", "strike"], as_index=False)
        .agg(
            call_equiv_mid=("call_equiv_mid", "median"),
            iv_market=("call_equiv_iv_proxy", "median"),
            spot_ref=("spot_ref", "median"),
            forward_ref=("forward_ref", "median"),
            T=("T", "median"),
            dte=("dte", "median"),
            r=("r", "median"),
            r_source=("r_source", lambda x: ";".join(sorted(set(map(str, x.dropna()))))),
            q_proxy=("q_proxy", "median"),
            log_moneyness=("log_moneyness", "median"),
            n_quotes=("quote_source", "size"),
            put_via_parity_share=("quote_source", lambda x: float((x == "put_via_parity").mean())),
            quote_source_mix=("quote_source", lambda x: ",".join(sorted(pd.Series(x).dropna().astype(str).unique()))),
            median_rel_spread=("spread_frac", "median"),
            quote_quality_weight=("quote_quality_weight", "median"),
            open_interest=("open_interest", "median"),
            volume=("volume", "median"),
            forward_source=("forward_source", "first"),
            vendor_forward_ref=("vendor_forward_ref", "median"),
            vendor_forward_rel_diff=("vendor_forward_rel_diff", "median"),
        )
    )
    out["q"] = out["q_proxy"]
    out["call_mid"] = out["call_equiv_mid"]
    out["iv_proxy"] = out["iv_market"]
    return _downcast_surface_frame(out)


def static_arbitrage_expiry_audit(surface: pd.DataFrame, cfg: MarketConfig) -> pd.DataFrame:
    """Part-A-style vertical, slope and butterfly diagnostics on the collapsed call surface."""
    rows = []
    for (date, exdate), group in surface.groupby(["date", "exdate"], sort=False):
        g = group.sort_values("strike")
        n = len(g)
        if n < 3:
            rows.append({
                "date": date, "exdate": exdate, "n_strikes": n,
                "vertical_violations": np.nan,
                "slope_lower_violations": np.nan,
                "slope_upper_violations": np.nan,
                "butterfly_violations": np.nan,
                "has_static_arb_violation": np.nan,
            })
            continue
        K = g["strike"].to_numpy(dtype=float)
        C = g["call_equiv_mid"].to_numpy(dtype=float)
        T = float(np.nanmedian(g["T"]))
        r = float(np.nanmedian(g["r"]))
        discount = np.exp(-r * T)
        dK = np.diff(K)
        dC = np.diff(C)
        valid = dK > 0
        if not valid.any():
            slopes = np.array([], dtype=float)
        else:
            slopes = dC[valid] / dK[valid]
        tol = cfg.static_arb_tolerance
        vertical = int((dC[valid] > tol).sum()) if len(slopes) else 0
        slope_lower = int((slopes < -discount - tol).sum()) if len(slopes) else 0
        slope_upper = int((slopes > tol).sum()) if len(slopes) else 0
        butterfly = int((np.diff(slopes) < -tol).sum()) if len(slopes) >= 2 else 0
        rows.append({
            "date": date,
            "exdate": exdate,
            "n_strikes": n,
            "vertical_violations": vertical,
            "slope_lower_violations": slope_lower,
            "slope_upper_violations": slope_upper,
            "butterfly_violations": butterfly,
            "has_static_arb_violation": bool(vertical + slope_lower + slope_upper + butterfly),
        })
    return pd.DataFrame(rows)



def add_part_a_style_fit_weights(surface: pd.DataFrame, cfg: MarketConfig) -> pd.DataFrame:
    """Balance quote quality, maturity density and moneyness density.

    Vega scaling is applied in the calibration residual, not here. Rows with
    missing/non-positive relative spreads are invalid calibration observations and
    are excluded with an explicit audit count, never assigned a substitute weight.
    """
    out = surface.copy()
    require_columns(out, {"date", "T", "log_moneyness", "median_rel_spread"}, "Calibration surface")

    spread = pd.to_numeric(out["median_rel_spread"], errors="coerce")
    finite_spread = np.isfinite(spread.to_numpy(float))
    valid_spread = finite_spread & (spread.to_numpy(float) > float(cfg.min_bid_ask_spread_frac))
    invalid_count = int((~valid_spread).sum())
    if invalid_count:
        print(
            f"Excluding {invalid_count:,}/{len(out):,} collapsed surface rows with "
            "missing/non-positive relative spreads before calibration weighting."
        )
        out = out.loc[valid_spread].copy()
        spread = pd.to_numeric(out["median_rel_spread"], errors="coerce")
    if out.empty:
        raise ValueError("All collapsed calibration rows have invalid relative spreads; cannot build strict weighted surface.")

    out["maturity_weight_bucket"] = pd.cut(
        out["T"], bins=[0, 14/365.25, 30/365.25, 60/365.25, 120/365.25, 365/365.25, np.inf],
        labels=False, include_lowest=True,
    )
    out["moneyness_weight_bucket"] = pd.cut(
        out["log_moneyness"], bins=list(cfg.moneyness_balance_bins), labels=False, include_lowest=True,
    )
    if out[["maturity_weight_bucket", "moneyness_weight_bucket"]].isna().any().any():
        bad = out.loc[out[["maturity_weight_bucket", "moneyness_weight_bucket"]].isna().any(axis=1)]
        raise ValueError(
            "Maturity/moneyness weighting buckets do not cover the calibration surface. "
            f"Bad rows={len(bad):,}; max_abs_log_moneyness={bad['log_moneyness'].abs().max() if len(bad) else np.nan}."
        )
    maturity_count = out.groupby(["date", "maturity_weight_bucket"], observed=True)["strike"].transform("size").astype(float)
    moneyness_count = out.groupby(["date", "moneyness_weight_bucket"], observed=True)["strike"].transform("size").astype(float)
    quality = 1.0 / spread.clip(lower=float(cfg.min_bid_ask_spread_frac))
    density = maturity_count.pow(-float(cfg.maturity_balance_power)) * moneyness_count.pow(-float(cfg.moneyness_balance_power))
    out["raw_fit_weight"] = quality * density
    if out["raw_fit_weight"].isna().any() or (~np.isfinite(out["raw_fit_weight"].to_numpy(float))).any() or out["raw_fit_weight"].le(0).any():
        raise ValueError("Raw calibration weights are missing/non-finite/non-positive after strict spread filtering.")
    scale = out.groupby("date")["raw_fit_weight"].transform("median")
    if scale.isna().any() or (scale <= 0).any() or (~np.isfinite(scale.to_numpy(float))).any():
        raise ValueError("Cannot normalise calibration weights by date.")
    out["fit_weight"] = (out["raw_fit_weight"] / scale).clip(lower=cfg.fit_weight_floor, upper=cfg.fit_weight_cap)
    if out["fit_weight"].isna().any() or (~np.isfinite(out["fit_weight"].to_numpy(float))).any() or out["fit_weight"].le(0).any():
        raise ValueError("Calibration fit weights are missing/non-finite/non-positive.")
    return out

def print_market_data_validation(surface: pd.DataFrame) -> None:
    print("Market data validation")
    validation = {
        "rows": int(len(surface)),
        "dates": int(surface["date"].nunique()) if "date" in surface and len(surface) else 0,
        "forward_sources": surface["forward_source"].value_counts(dropna=False).to_dict() if "forward_source" in surface else {},
        "median_spot": float(surface["spot_ref"].median()) if "spot_ref" in surface and len(surface) else np.nan,
        "median_forward_ref": float(surface["forward_ref"].median()) if "forward_ref" in surface and len(surface) else np.nan,
        "median_strike": float(surface["strike"].median()) if "strike" in surface and len(surface) else np.nan,
        "median_abs_log_moneyness": float(surface["log_moneyness"].abs().median()) if "log_moneyness" in surface and len(surface) else np.nan,
        "median_fit_weight": float(surface["fit_weight"].median()) if "fit_weight" in surface and len(surface) else np.nan,
        "duplicate_date_expiry_strike_rows": int(surface.duplicated(["date", "exdate", "strike"]).sum()) if all(c in surface.columns for c in ["date", "exdate", "strike"]) else np.nan,
        "static_arb_violation_share": float(surface["has_static_arb_violation"].fillna(False).mean()) if "has_static_arb_violation" in surface and len(surface) else np.nan,
    }
    print(validation)
    cols = [c for c in ["date", "exdate", "strike", "spot_ref", "forward_ref", "forward_source", "log_moneyness", "call_equiv_mid", "iv_market", "fit_weight"] if c in surface.columns]
    display(surface[cols].head())


def build_calibration_surface_from_cleaned(cleaned: pd.DataFrame, cfg: MarketConfig, *, validate: bool = True) -> pd.DataFrame:
    """
    Build the final full-surface calibration panel from already-cleaned quote rows.

    This is the intended hybrid:
    - current notebook: memory-safe streaming and full-surface methodology;
    - Part A: quote discipline, parity-only forwards, OTM call-equivalent quotes, collapsed strike grid, and spread-based weights.
    """
    with_forwards = infer_forwards_put_call_parity(cleaned, cfg)
    if with_forwards.empty:
        return pd.DataFrame(columns=FINAL_SURFACE_COLUMNS)

    call_equiv_quotes = convert_to_call_equivalent(with_forwards, cfg)
    surface = collapse_to_strike_grid(call_equiv_quotes) if cfg.collapse_to_strike_grid else call_equiv_quotes
    if surface.empty:
        return pd.DataFrame(columns=FINAL_SURFACE_COLUMNS)

    # After collapse, these aliases keep the surface compatible with the Part A calibration engine.
    if "iv_market" not in surface.columns and "iv_proxy" in surface.columns:
        surface["iv_market"] = surface["iv_proxy"]
    if "iv_proxy" not in surface.columns and "iv_market" in surface.columns:
        surface["iv_proxy"] = surface["iv_market"]
    if "q" not in surface.columns:
        if "q_proxy" not in surface.columns:
            raise KeyError("Surface must contain q or q_proxy inferred from put-call parity; no substitute dividend yield is used.")
        surface["q"] = surface["q_proxy"]

    need_iv = surface["iv_market"].isna() if "iv_market" in surface.columns else pd.Series(False, index=surface.index)
    if bool(need_iv.any()):
        iv_df = surface.loc[need_iv].copy()
        iv_df["option_type"] = "call"
        iv_df["spot"] = iv_df["spot_ref"].astype(float)
        surface.loc[need_iv, "iv_market"] = implied_vol_vectorised(iv_df, "call_equiv_mid", option_col="option_type")
        surface["iv_proxy"] = surface["iv_market"]
        del iv_df

    for required_col in ["r", "q", "iv_market"]:
        values = pd.to_numeric(surface[required_col], errors="coerce")
        if values.isna().any() or (~np.isfinite(values.to_numpy(float))).any():
            raise ValueError(f"Surface contains non-finite {required_col}; strict methodology does not use a substitute value.")
        surface[required_col] = values

    surface["bs_vega"] = bs_vega(
        surface["spot_ref"],
        surface["strike"],
        surface["r"],
        surface["q"],
        surface["T"],
        surface["iv_market"],
    )
    surface = add_part_a_style_fit_weights(surface, cfg)

    expiry_audit = static_arbitrage_expiry_audit(surface, cfg)
    if not expiry_audit.empty:
        surface = surface.merge(
            expiry_audit[["date", "exdate", "vertical_violations", "slope_lower_violations", "slope_upper_violations", "butterfly_violations", "has_static_arb_violation"]],
            on=["date", "exdate"],
            how="left",
        )
        if cfg.drop_static_arb_violating_expiries:
            surface = surface.loc[~surface["has_static_arb_violation"].fillna(False)].copy()

    surface["maturity_bucket"] = pd.cut(
        surface["dte"],
        bins=[0, 10, 21, 45, 75, 120, 10_000],
        labels=["1W", "2W", "1M", "2M", "3M", "LONG"],
        right=True,
    ).astype(str)

    mask = (
        surface["call_equiv_mid"].gt(0)
        & surface["iv_market"].between(cfg.min_iv, cfg.max_iv)
        & surface["log_moneyness"].abs().le(cfg.max_abs_log_moneyness)
        & np.isfinite(surface["fit_weight"])
        & surface["fit_weight"].gt(0)
        & surface["bs_vega"].gt(cfg.vega_floor)
        & surface["forward_ref"].gt(0)
        & surface["spot_ref"].gt(0)
        & surface["forward_source"].eq("put_call_parity")
    )
    keep_cols = [c for c in FINAL_SURFACE_COLUMNS if c in surface.columns]
    out = surface.loc[mask, keep_cols].reset_index(drop=True)
    out = _downcast_surface_frame(out)
    if validate:
        print_market_data_validation(out)
    return out


def build_calibration_surface(raw: pd.DataFrame, cfg: MarketConfig, crsp_spot: pd.DataFrame | None = None) -> pd.DataFrame:
    cleaned = clean_option_quotes(raw, cfg, crsp_spot=crsp_spot)
    return build_calibration_surface_from_cleaned(cleaned, cfg, validate=True)


def _write_cleaned_parts_by_year(option_path: Path, cfg: MarketConfig, crsp_spot: pd.DataFrame | None, tmp_dir: Path) -> list[int]:
    """Stream raw data once and write cleaned minimal rows to yearly parquet parts."""
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    tmp_dir.mkdir(parents=True, exist_ok=True)
    part_counters: dict[int, int] = {}
    total_in = 0
    total_cleaned = 0
    for batch_idx, raw_batch in enumerate(iter_raw_option_batches(option_path, cfg), start=1):
        total_in += len(raw_batch)
        cleaned = clean_option_quotes(raw_batch, cfg, crsp_spot=crsp_spot)
        if cleaned.empty:
            print(f"Batch {batch_idx:04d}: raw={len(raw_batch):,}, cleaned=0")
            continue
        cleaned["year"] = cleaned["date"].dt.year.astype(int)
        total_cleaned += len(cleaned)
        for year, group in cleaned.groupby("year", sort=True):
            year = int(year)
            part_id = part_counters.get(year, 0)
            out_dir = tmp_dir / f"year={year}"
            out_dir.mkdir(parents=True, exist_ok=True)
            out_path = out_dir / f"part_{part_id:05d}.parquet"
            group.drop(columns=["year"]).to_parquet(out_path, index=False)
            part_counters[year] = part_id + 1
        if batch_idx == 1 or batch_idx % 10 == 0:
            print(f"Batch {batch_idx:04d}: raw_seen={total_in:,}, cleaned_seen={total_cleaned:,}, years={sorted(part_counters)}")
        del raw_batch, cleaned
        gc.collect()
    print(f"Finished streaming raw options: raw_seen={total_in:,}, cleaned_written={total_cleaned:,}")
    return sorted(part_counters)


def build_calibration_surface_chunked(option_path: Path, cfg: MarketConfig, paths: PathConfig, crsp_spot: pd.DataFrame | None = None) -> pd.DataFrame:
    """
    Memory-safe full-surface construction.

    Workflow:
    1. Stream raw OptionMetrics in batches and clean with Part-A-style filters.
    2. Write cleaned rows to yearly parquet parts.
    3. Load one year at a time, infer parity forwards, build collapsed call-equivalent surfaces, and save yearly surfaces.
    4. Concatenate yearly surface files into the final full-surface panel cache.
    """
    tmp_dir = paths.preprocessing_tmp_dir
    surface_year_dir = paths.surface_year_dir
    if surface_year_dir.exists():
        shutil.rmtree(surface_year_dir)
    surface_year_dir.mkdir(parents=True, exist_ok=True)

    years = _write_cleaned_parts_by_year(option_path, cfg, crsp_spot, tmp_dir)
    year_surface_files: list[Path] = []
    for year in years:
        part_files = sorted((tmp_dir / f"year={year}").glob("part_*.parquet"))
        if not part_files:
            continue
        print(f"Building final Part-A-disciplined full surface for {year} from {len(part_files)} cleaned parts...")
        cleaned_year = pd.concat((pd.read_parquet(p) for p in part_files), ignore_index=True)
        surface_year = build_calibration_surface_from_cleaned(cleaned_year, cfg, validate=False)
        if surface_year.empty:
            print(f"Warning: no final calibration surface rows for {year} after parity/OTM/collapse filters.")
            continue
        out_path = surface_year_dir / f"surface_{year}.parquet"
        surface_year.to_parquet(out_path, index=False)
        print(f"Saved {year}: {len(surface_year):,} rows -> {out_path}")
        year_surface_files.append(out_path)
        del cleaned_year, surface_year
        gc.collect()

    if not year_surface_files:
        raise ValueError("No yearly surface files were produced. Check filters, dates and input data.")

    print("Combining yearly surface files into final cached surface panel...")
    surface = pd.concat((pd.read_parquet(p) for p in year_surface_files), ignore_index=True)
    surface = _downcast_surface_frame(surface)
    print_market_data_validation(surface)
    paths.surface_cache_path.parent.mkdir(parents=True, exist_ok=True)
    surface.to_parquet(paths.surface_cache_path, index=False)

    if not cfg.keep_preprocessing_parts and tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    return surface


In [16]:
def static_arbitrage_diagnostics(surface: pd.DataFrame, cfg: MarketConfig) -> pd.DataFrame:
    """Run Part-A-style static-arbitrage diagnostics on the collapsed full surface."""
    if "call_equiv_mid" in surface.columns:
        audit = static_arbitrage_expiry_audit(surface, cfg)
    elif "call_mid" in surface.columns:
        temp = surface.rename(columns={"call_mid": "call_equiv_mid"})
        audit = static_arbitrage_expiry_audit(temp, cfg)
    else:
        raise ValueError("Surface must contain call_equiv_mid or call_mid for static-arbitrage diagnostics.")
    return audit


def summarise_static_arbitrage(audit: pd.DataFrame) -> pd.DataFrame:
    clean = audit.dropna(subset=["has_static_arb_violation"]).copy()
    if clean.empty:
        return pd.DataFrame({"metric": ["expiry_slices_checked"], "value": [0]})
    return pd.DataFrame({
        "metric": [
            "expiry_slices_checked",
            "median_strikes_per_expiry",
            "share_expiry_slices_with_any_violation",
            "total_vertical_violations",
            "total_slope_lower_violations",
            "total_slope_upper_violations",
            "total_butterfly_violations",
        ],
        "value": [
            len(clean),
            clean["n_strikes"].median(),
            clean["has_static_arb_violation"].mean(),
            clean["vertical_violations"].sum(),
            clean["slope_lower_violations"].sum(),
            clean["slope_upper_violations"].sum(),
            clean["butterfly_violations"].sum(),
        ],
    })


def build_or_load_surface_panel(cfg: ExperimentConfig) -> pd.DataFrame:
    deps = [cfg.paths.raw_option_path, cfg.paths.raw_crsp_path]
    required = {"date", "exdate", "strike", "spot_ref", "forward_ref", "r", "q", "call_equiv_mid", "iv_market", "bs_vega", "fit_weight"}
    force = cfg.flags.force_rebuild("surface")
    load_error: Exception | None = None
    if not force:
        try:
            surface = load_dataframe_cache(
                cfg.paths.surface_cache_path, "full_surface_panel", "surface", cfg,
                required_columns=required, dependencies=deps,
            )
            print(f"[surface] loaded validated cache: {cfg.paths.surface_cache_path}")
            return surface
        except (FileNotFoundError, ValueError, json.JSONDecodeError) as exc:
            load_error = exc
            if not cfg.flags.can_rebuild("surface"):
                raise RuntimeError(
                    "The cleaned surface cache is missing or stale and RUN.rebuild_from='reuse' forbids rebuilding. "
                    "Set RUN = RunRequest(..., rebuild_from='surface') to force a clean rebuild, or use 'auto' to rebuild only when required."
                ) from exc
            print(f"[surface] cache unavailable/stale; rebuilding under {RUN.rebuild_from!r}. Reason: {type(exc).__name__}: {exc}", flush=True)
    else:
        print("[surface] forced rebuild selected; existing surface cache will be replaced.", flush=True)

    if not cfg.paths.raw_crsp_path.exists():
        raise FileNotFoundError(f"CRSP spot file not found: {cfg.paths.raw_crsp_path}")
    if not cfg.paths.raw_option_path.exists():
        raise FileNotFoundError(f"OptionMetrics option file not found: {cfg.paths.raw_option_path}")

    crsp_spot = load_crsp_spx(cfg.paths.raw_crsp_path)
    if cfg.market.use_memory_safe_preprocessing:
        surface = build_calibration_surface_chunked(cfg.paths.raw_option_path, cfg.market, cfg.paths, crsp_spot=crsp_spot)
        if not cfg.paths.surface_cache_path.exists():
            cfg.paths.surface_cache_path.parent.mkdir(parents=True, exist_ok=True)
            surface.to_parquet(cfg.paths.surface_cache_path, index=False)
        write_cache_manifest(
            cfg.paths.surface_cache_path, "full_surface_panel", "surface", cfg,
            dependencies=deps, row_count=len(surface), columns=surface.columns,
            extra={
                "forward_source": "put_call_parity", "spot_source": "CRSP spindx",
                "risk_free_rate_source": cfg.market.risk_free_rate_source,
                "configured_risk_free_rate": cfg.market.configured_risk_free_rate,
                "rebuild_request": RUN.rebuild_from,
            },
        )
        return surface

    raw_cols = existing_columns(cfg.paths.raw_option_path, RAW_OPTION_CANDIDATE_COLUMNS)
    raw = load_raw_option_data(cfg.paths.raw_option_path, columns=raw_cols)
    surface = build_calibration_surface(raw, cfg.market, crsp_spot=crsp_spot)
    write_dataframe_cache(
        surface, cfg.paths.surface_cache_path, "full_surface_panel", "surface", cfg,
        dependencies=deps,
        extra={
            "forward_source": "put_call_parity", "spot_source": "CRSP spindx",
            "risk_free_rate_source": cfg.market.risk_free_rate_source,
            "configured_risk_free_rate": cfg.market.configured_risk_free_rate,
            "rebuild_request": RUN.rebuild_from,
        },
    )
    return surface

surface_panel = build_or_load_surface_panel(CFG)
print(surface_panel.shape)
display(surface_panel.head())

audit_static_arb = static_arbitrage_diagnostics(surface_panel, CFG.market)
display(audit_static_arb.head())
display(summarise_static_arbitrage(audit_static_arb))


[surface] loaded validated cache: C:\Users\lhkke\OneDrive - Queen Mary, University of London\Documents\Dissertation\Deep-Hedging-under-Lifted-Heston-MSc-Dissertation-\cache_final\full_surface_panel_partA_surface_discipline.parquet
(254081, 35)


,date,exdate,strike,call_equiv_mid,call_mid,iv_market,iv_proxy,bs_vega,fit_weight,raw_fit_weight,quote_quality_weight,median_rel_spread,dte,T,maturity_bucket,spot_ref,forward_ref,forward_source,vendor_forward_ref,vendor_forward_rel_diff,q,q_proxy,r,r_source,log_moneyness,n_quotes,put_via_parity_share,quote_source_mix,open_interest,volume,vertical_violations,slope_lower_violations,slope_upper_violations,butterfly_violations,has_static_arb_violation
0,2010-01-04,2010-03-31,850.0,280.279633,280.279633,0.338953,0.338953,42.853626,0.401260,0.073879,4.666667,0.214286,86.0,0.235455,3M,1132.98999,1128.133789,put_call_parity,NaN,NaN,0.028243,0.028243,0.01,configured_constant,-0.283084,1,1.0,put_via_parity,17.0,0.0,0,0,0,0,False
1,2010-01-04,2010-03-31,900.0,232.197250,232.197250,0.312688,0.312688,64.038483,0.494409,0.091029,5.750000,0.173913,86.0,0.235455,3M,1132.98999,1128.133789,put_call_parity,NaN,NaN,0.028243,0.028243,0.01,configured_constant,-0.225925,1,1.0,put_via_parity,3.0,1.0,0,0,0,0,False
2,2010-01-04,2010-03-31,950.0,185.314850,185.314850,0.287530,0.287530,93.401573,0.515774,0.094963,7.600000,0.131579,86.0,0.235455,3M,1132.98999,1128.133789,put_call_parity,NaN,NaN,0.028243,0.028243,0.01,configured_constant,-0.171858,1,1.0,put_via_parity,1824.0,4.0,0,0,0,0,False
3,2010-01-04,2010-03-31,1000.0,140.182434,140.182434,0.261464,0.261464,130.333069,0.644717,0.118704,9.500000,0.105263,86.0,0.235455,3M,1132.98999,1128.133789,put_call_parity,NaN,NaN,0.028243,0.028243,0.01,configured_constant,-0.120565,1,1.0,put_via_parity,125.0,0.0,0,0,0,0,False
4,2010-01-04,2010-03-31,1050.0,98.100029,98.100029,0.235975,0.235975,172.418503,1.139306,0.209766,15.500000,0.064516,86.0,0.235455,3M,1132.98999,1128.133789,put_call_parity,NaN,NaN,0.028243,0.028243,0.01,configured_constant,-0.071775,1,1.0,put_via_parity,8.0,0.0,0,0,0,0,False


,date,exdate,n_strikes,vertical_violations,slope_lower_violations,slope_upper_violations,butterfly_violations,has_static_arb_violation
0,2010-01-04,2010-03-31,9,0,0,0,0,False
1,2010-01-04,2010-04-17,19,0,0,0,0,False
2,2010-01-05,2010-03-31,10,0,0,0,0,False
3,2010-01-05,2010-12-18,31,0,0,0,0,False
4,2010-01-06,2010-03-31,11,0,0,0,0,False


,metric,value
0,expiry_slices_checked,8451.0
1,median_strikes_per_expiry,29.0
2,share_expiry_slices_with_any_violation,0.0
3,total_vertical_violations,0.0
4,total_slope_lower_violations,0.0
5,total_slope_upper_violations,0.0
6,total_butterfly_violations,0.0


In [17]:
# Enforce final surface naming used by calibration, seed selection and diagnostics.
if surface_panel.empty:
    raise ValueError("surface_panel is empty after construction/loading.")
if "log_forward_moneyness" not in surface_panel.columns:
    if "log_moneyness" not in surface_panel.columns:
        raise ValueError("surface_panel must contain log_moneyness or log_forward_moneyness.")
    surface_panel["log_forward_moneyness"] = surface_panel["log_moneyness"].astype(float)
if "log_moneyness" not in surface_panel.columns:
    surface_panel["log_moneyness"] = surface_panel["log_forward_moneyness"].astype(float)
required_surface = {"spot_ref", "forward_ref", "strike", "call_equiv_mid", "bs_vega", "fit_weight", "log_forward_moneyness"}
missing_surface = required_surface - set(surface_panel.columns)
if missing_surface:
    raise ValueError(f"surface_panel missing required research columns: {sorted(missing_surface)}")


## 3b. SPX Hedging-Focused Empirical Panel

The hedging-focused panel is the headline empirical object. It is deliberately narrower than the full SPX surface. The purpose is to evaluate implementable stock-only hedging under liquid near-ATM option conditions, not to force lifted Heston to explain every wing, maturity and stale quote in the market.

The panel is built around near-ATM options and maturity buckets that correspond to the traded hedging experiments. This aligns calibration diagnostics with the risk exposures actually used in policy evaluation. The design avoids a common methodological trap: calibrating a model to a broad surface, finding poor global fit, and then declaring the model useless for a narrower hedging task without checking the relevant slice.

The distinction is:

$$
\text{global surface diagnostic} \neq \text{headline hedging panel}.
$$

The global panel remains useful as a stress test and limitation. The hedging-focused panel is used for the main calibration and policy-evaluation claims. This hierarchy also matches the dissertation's cautious claim: lifted Heston is assessed as a state representation for hedging, not as a universal SPX smile machine.

This section should produce coverage tables by date, maturity bucket and moneyness. Those tables tell the reader whether the panel is balanced enough for inference. If the seed panel is thin in some maturity/regime cells, later confidence intervals and robustness claims must be interpreted accordingly.


In [18]:
# ============================================================
# SPX Hedging-Focused Empirical Panel
# ============================================================

SPX_HEDGING_PANEL_VERSION = "spx_hedging_focused_panel_v2_1w_2w"
HEADLINE_EMPIRICAL_MODE = "HEDGING_FOCUSED_PANEL"
GLOBAL_DIAGNOSTIC_MODE = "GLOBAL_SURFACE_DIAGNOSTIC"

# Named, pre-specified quote filters. Do not hand-edit these after inspecting results.
SPX_HEDGING_UNDERLYING = "SPX"
SPX_HEDGING_MIN_DTE = int(CFG.market.hedging_min_dte)
SPX_HEDGING_MAX_DTE = int(CFG.market.hedging_max_dte)
SPX_HEDGING_MAX_ABS_LOG_MONEYNESS = float(CFG.market.hedging_max_abs_log_moneyness)
SPX_HEDGING_PRIMARY_ATM_ABS_LOG_MONEYNESS = float(CFG.market.hedging_primary_atm_abs_log_moneyness)
SPX_HEDGING_MIN_IV = 1e-8
SPX_HEDGING_MIN_MID = 1e-8
SPX_HEDGING_MIN_BID = 0.0
SPX_HEDGING_MIN_VOLUME = 0 if CFG.market.hedging_min_volume is None else int(CFG.market.hedging_min_volume)
SPX_HEDGING_MIN_OPEN_INTEREST = 0 if CFG.market.hedging_min_open_interest is None else int(CFG.market.hedging_min_open_interest)
SPX_HEDGING_MAX_ABS_SPREAD = np.inf if CFG.market.hedging_max_abs_spread is None else float(CFG.market.hedging_max_abs_spread)
SPX_HEDGING_MAX_REL_SPREAD = np.inf if CFG.market.hedging_max_rel_spread is None else float(CFG.market.hedging_max_rel_spread)
SPX_HEDGING_MIN_DATES = 20
SPX_HEDGING_SPLIT_RULE = "chronological_by_date_60_20_20"
SPX_HEDGING_STRESS_REALIZED_VOL_QUANTILE = float(CFG.market.hedging_stress_realised_vol_quantile)
SPX_HEDGING_VIX_STRESS_THRESHOLD = float(CFG.market.hedging_stress_vix_threshold)
SPX_HEDGING_MANUAL_DATE_DELETIONS_ALLOWED = False
SPX_HEDGING_REQUIRE_RAW_BID_ASK_FOR_HEADLINE = False  # final collapsed Part-A surface usually no longer carries raw bid/ask.

SPX_HEDGING_FILENAMES = {
    "global_diagnostic_panel": "spx_global_diagnostic_panel.parquet",
    "hedging_calibration_panel": "spx_hedging_calibration_panel.parquet",
    "hedging_seed_panel": "spx_hedging_seed_panel.parquet",
    "hedging_calm_panel": "spx_hedging_calm_panel.parquet",
    "hedging_stress_panel": "spx_hedging_stress_panel.parquet",
    "exclusion_manifest": "spx_hedging_panel_exclusion_manifest.csv",
    "regime_labels": "spx_regime_labels.csv",
    "split_manifest": "spx_hedging_train_val_test_manifest.csv",
    "methodology_audit": "spx_hedging_panel_methodology_audit.csv",
    "row_identity_audit": "spx_hedging_row_identity_audit.csv",
    "calibration_metrics": "spx_hedging_calibration_metrics.csv",
    "hedging_metrics": "spx_hedging_policy_metrics_by_regime.csv",
    "final_audit": "spx_hedging_final_methodology_audit.csv",
}

# New run-local destinations. Existing TEST30 files are inputs, never write targets.
SPX_HEDGING_OUTPUTS = {
    name: table_path(filename)
    for name, filename in SPX_HEDGING_FILENAMES.items()
}
for _path in SPX_HEDGING_OUTPUTS.values():
    _path.parent.mkdir(parents=True, exist_ok=True)
SPX_HEDGING_CALIBRATION_PANEL_PATH = SPX_HEDGING_OUTPUTS["hedging_calibration_panel"]
HEADLINE_CALIBRATION_CACHE_KIND = "hedging_focused_near_atm_spx_panel_1w_2w"

_SPX_PANEL_REQUIRED_KEYS = (
    "global_diagnostic_panel",
    "hedging_calibration_panel",
    "hedging_seed_panel",
    "hedging_calm_panel",
    "hedging_stress_panel",
    "regime_labels",
    "split_manifest",
    "exclusion_manifest",
)


def _normalise_project_path(path: Path | str) -> Path:
    path = Path(path).expanduser()
    if not path.is_absolute():
        path = CFG.paths.project_root / path
    return path.resolve()


def _spx_panel_paths_for_dir(directory: Path) -> dict[str, Path]:
    directory = _normalise_project_path(directory)
    return {
        name: directory / filename
        for name, filename in SPX_HEDGING_FILENAMES.items()
    }


def spx_hedging_panel_candidate_dirs() -> list[Path]:
    """Ordered candidate directories for an existing TEST30 panel bundle.

    Search is deliberately shallow and deterministic. Recursive OneDrive archaeology is
    not a reproducibility strategy, despite its popularity among deadlines.
    """
    candidates: list[Path] = [Path(CFG.paths.table_dir)]
    if RUN.legacy_hedging_panel_dir is not None:
        candidates.append(Path(RUN.legacy_hedging_panel_dir))

    root = Path(CFG.paths.project_root)
    candidates.extend([
        root / "outputs_final_test30_5bucket" / "tables",
        root / "outputs_final_test30_5bucket",
        root / "outputs_final" / "tables",
    ])
    candidates.extend(sorted(root.glob("outputs_final_test30*/tables")))

    unique: list[Path] = []
    seen: set[str] = set()
    for candidate in candidates:
        resolved = _normalise_project_path(candidate)
        key = os.path.normcase(str(resolved))
        if key not in seen:
            seen.add(key)
            unique.append(resolved)
    return unique


def resolve_spx_hedging_panel_inputs() -> tuple[dict[str, Path], Path, pd.DataFrame]:
    rows: list[dict[str, Any]] = []
    for directory in spx_hedging_panel_candidate_dirs():
        paths = _spx_panel_paths_for_dir(directory)
        missing = [name for name in _SPX_PANEL_REQUIRED_KEYS if not paths[name].exists()]
        rows.append({
            "candidate_directory": str(directory),
            "complete": not missing,
            "missing": ", ".join(missing),
        })
        if not missing:
            return paths, directory, pd.DataFrame(rows)
    audit = pd.DataFrame(rows)
    searched = "\n".join(f"  - {row['candidate_directory']}: missing {row['missing']}" for row in rows)
    raise FileNotFoundError(
        "No complete SPX hedging-panel bundle was found. Searched:\n"
        f"{searched}\n"
        "Set RUN.legacy_hedging_panel_dir to the TEST30 tables directory, or use "
        "rebuild_from='hedging_panel' to reconstruct it from the validated surface."
    )


def materialise_spx_hedging_panel_inputs(
    source_paths: dict[str, Path],
    source_directory: Path,
) -> None:
    """Copy validated legacy inputs into the isolated run output directory.

    This avoids later cells oscillating between legacy and run-local paths. The legacy
    TEST30 directory remains read-only.
    """
    target_directory = Path(CFG.paths.table_dir).resolve()
    source_directory = Path(source_directory).resolve()
    if source_directory == target_directory:
        return

    for name in _SPX_PANEL_REQUIRED_KEYS:
        src = Path(source_paths[name])
        dst = Path(SPX_HEDGING_OUTPUTS[name])
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists() or src.stat().st_size != dst.stat().st_size:
            shutil.copy2(src, dst)

    provenance = {
        "source_directory": str(source_directory),
        "target_directory": str(target_directory),
        "calibration_spec": RUN.calibration_spec,
        "calibration_H": RUN.calibration_H,
        "copied_at_utc": datetime.now(timezone.utc).isoformat(),
        "files": {
            name: {
                "source": str(source_paths[name]),
                "target": str(SPX_HEDGING_OUTPUTS[name]),
                "size": int(Path(source_paths[name]).stat().st_size),
            }
            for name in _SPX_PANEL_REQUIRED_KEYS
        },
    }
    save_json(target_directory / "spx_hedging_panel_legacy_provenance.json", provenance)
    print(f"[hedging panel] materialised legacy TEST30 bundle from {source_directory}", flush=True)


def _safe_unique_count(frame: pd.DataFrame, col: str) -> int:
    return int(frame[col].nunique()) if col in frame.columns and not frame.empty else 0


def _filter_audit_row(name: str, before: pd.DataFrame, after: pd.DataFrame, reason: str) -> dict[str, Any]:
    rows_before = int(len(before))
    rows_after = int(len(after))
    removed = rows_before - rows_after
    return {
        "filter_name": name,
        "rows_before": rows_before,
        "rows_after": rows_after,
        "rows_removed": int(removed),
        "pct_removed": float(removed / rows_before) if rows_before else 0.0,
        "unique_dates_before": _safe_unique_count(before, "date"),
        "unique_dates_after": _safe_unique_count(after, "date"),
        "unique_expiries_before": _safe_unique_count(before, "exdate"),
        "unique_expiries_after": _safe_unique_count(after, "exdate"),
        "reason": reason,
    }


def _apply_panel_filter(frame: pd.DataFrame, mask: pd.Series | np.ndarray, name: str, reason: str, audit_rows: list[dict[str, Any]]) -> pd.DataFrame:
    before = frame.copy()
    mask = pd.Series(mask, index=frame.index).fillna(False).astype(bool)
    after = frame.loc[mask].copy()
    audit_rows.append(_filter_audit_row(name, before, after, reason))
    return after


def _first_present(columns: Iterable[str], frame: pd.DataFrame) -> str | None:
    for c in columns:
        if c in frame.columns:
            return c
    return None


def _require_any(frame: pd.DataFrame, candidates: Iterable[str], label: str) -> str:
    found = _first_present(candidates, frame)
    if found is None:
        raise ValueError(f"SPX hedging panel requires {label}; none of {list(candidates)} found. No substitute data is fabricated.")
    return found


def _prepare_surface_for_spx_hedging(raw_surface: pd.DataFrame) -> pd.DataFrame:
    if raw_surface.empty:
        raise ValueError("Cannot build hedging-focused SPX panel from an empty global diagnostic surface.")
    frame = raw_surface.copy()
    frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()
    frame["exdate"] = pd.to_datetime(frame["exdate"]).dt.normalize()
    if "dte" not in frame.columns:
        if "T" not in frame.columns:
            raise ValueError("SPX hedging panel requires either dte or T to enforce 1W-to-1Y maturities.")
        frame["dte"] = np.round(pd.to_numeric(frame["T"], errors="coerce") * CFG.market.calendar_days_per_year).astype("float")
    if "T" not in frame.columns:
        frame["T"] = pd.to_numeric(frame["dte"], errors="coerce") / float(CFG.market.calendar_days_per_year)
    if "log_forward_moneyness" not in frame.columns:
        if "log_moneyness" in frame.columns:
            frame["log_forward_moneyness"] = pd.to_numeric(frame["log_moneyness"], errors="coerce")
        else:
            strike_col = _require_any(frame, ["strike"], "strike")
            fwd_col = _require_any(frame, ["forward_ref", "forward", "F"], "forward")
            frame["log_forward_moneyness"] = np.log(pd.to_numeric(frame[strike_col], errors="coerce") / pd.to_numeric(frame[fwd_col], errors="coerce"))
    if "log_moneyness" not in frame.columns:
        frame["log_moneyness"] = frame["log_forward_moneyness"]
    if "mid_price" not in frame.columns:
        mid_col = _require_any(frame, ["mid", "call_equiv_mid", "call_mid"], "mid/call-equivalent mid price")
        frame["mid_price"] = pd.to_numeric(frame[mid_col], errors="coerce")
        frame["mid_price_source"] = mid_col
    iv_col = _require_any(frame, ["iv_market", "iv_proxy", "impl_volatility", "implied_volatility", "iv"], "implied volatility")
    frame["iv_for_filter"] = pd.to_numeric(frame[iv_col], errors="coerce")
    if "iv_market" not in frame.columns:
        frame["iv_market"] = frame["iv_for_filter"]
    if "call_equiv_mid" not in frame.columns:
        if "mid_price" in frame.columns:
            frame["call_equiv_mid"] = frame["mid_price"]
        else:
            raise ValueError("Hedging calibration requires call_equiv_mid or a genuine mid_price column.")
    if "spot_ref" not in frame.columns:
        _require_any(frame, ["spot", "underlying_price", "S0"], "spot/index level")
    if "forward_ref" not in frame.columns:
        _require_any(frame, ["forward", "forward_price", "vendor_forward"], "forward price")
    for col in ["strike", "spot_ref", "forward_ref", "call_equiv_mid", "mid_price", "T", "dte", "r", "q", "bs_vega", "fit_weight"]:
        if col in frame.columns:
            frame[col] = pd.to_numeric(frame[col], errors="coerce")
    frame["abs_log_forward_moneyness"] = pd.to_numeric(frame["log_forward_moneyness"], errors="coerce").abs()
    frame["headline_empirical_mode"] = HEADLINE_EMPIRICAL_MODE
    frame["spx_hedging_panel_version"] = SPX_HEDGING_PANEL_VERSION
    frame["manual_date_deletion"] = False
    frame["crisis_window_excluded"] = False
    return frame


def _underlying_spx_mask(frame: pd.DataFrame) -> tuple[pd.Series, str]:
    if "ticker" in frame.columns:
        return frame["ticker"].astype(str).str.upper().eq(SPX_HEDGING_UNDERLYING), "ticker == SPX"
    if "secid" in frame.columns:
        # OptionMetrics SPX index option extracts commonly arrive pre-filtered. Do not guess a secid.
        return pd.Series(True, index=frame.index), "secid present but no SPX mapping supplied; upstream file is treated as the SPX extract and not remapped"
    return pd.Series(True, index=frame.index), "no underlying identifier in collapsed surface; upstream surface is treated as the SPX-only extract"


def _standard_contract_mask(frame: pd.DataFrame) -> tuple[pd.Series, str]:
    if "nonstandard" in frame.columns:
        return ~frame["nonstandard"].fillna(False).astype(bool), "exclude rows flagged nonstandard"
    if "contract_size" in frame.columns:
        return pd.to_numeric(frame["contract_size"], errors="coerce").isin([100, np.nan]) | frame["contract_size"].isna(), "standard 100 multiplier or missing multiplier"
    return pd.Series(True, index=frame.index), "no non-standard contract flag in collapsed surface; no synthetic standardness filter applied"


def _liquidity_mask(frame: pd.DataFrame) -> tuple[pd.Series, str]:
    masks = []
    notes = []
    if "open_interest" in frame.columns:
        masks.append(pd.to_numeric(frame["open_interest"], errors="coerce").fillna(0).ge(SPX_HEDGING_MIN_OPEN_INTEREST))
        notes.append(f"open_interest >= {SPX_HEDGING_MIN_OPEN_INTEREST}")
    if "volume" in frame.columns:
        masks.append(pd.to_numeric(frame["volume"], errors="coerce").fillna(0).ge(SPX_HEDGING_MIN_VOLUME))
        notes.append(f"volume >= {SPX_HEDGING_MIN_VOLUME}")
    if not masks:
        return pd.Series(True, index=frame.index), "volume/open_interest unavailable; no liquidity count filter applied"
    combined = masks[0]
    for m in masks[1:]:
        combined = combined | m
    return combined, " or ".join(notes)


def _spread_mask(frame: pd.DataFrame) -> tuple[pd.Series, str]:
    rel_col = _first_present(["spread_frac", "median_rel_spread", "relative_spread"], frame)
    abs_col = _first_present(["spread", "median_abs_spread", "absolute_spread"], frame)
    mask = pd.Series(True, index=frame.index)
    notes = []
    if rel_col is not None:
        rel = pd.to_numeric(frame[rel_col], errors="coerce")
        mask &= rel.gt(0) & rel.le(SPX_HEDGING_MAX_REL_SPREAD)
        notes.append(f"0 < {rel_col} <= {SPX_HEDGING_MAX_REL_SPREAD}")
    if abs_col is not None and np.isfinite(SPX_HEDGING_MAX_ABS_SPREAD):
        abs_spread = pd.to_numeric(frame[abs_col], errors="coerce")
        mask &= abs_spread.gt(0) & abs_spread.le(SPX_HEDGING_MAX_ABS_SPREAD)
        notes.append(f"0 < {abs_col} <= {SPX_HEDGING_MAX_ABS_SPREAD}")
    if not notes:
        return mask, "bid-ask spread columns unavailable in collapsed surface; no synthetic spread filter applied"
    return mask, "; ".join(notes)


def build_spx_hedging_panels(surface: pd.DataFrame, cfg: ExperimentConfig) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    audit_rows: list[dict[str, Any]] = []
    global_diag = surface.copy()
    global_diag["empirical_mode"] = GLOBAL_DIAGNOSTIC_MODE
    global_diag["headline_use"] = False
    global_diag["diagnostic_note"] = "GLOBAL SURFACE DIAGNOSTIC: not used for headline hedging results"

    panel = _prepare_surface_for_spx_hedging(surface)

    underlying_mask, underlying_reason = _underlying_spx_mask(panel)
    panel = _apply_panel_filter(panel, underlying_mask, "underlying_spx_only", underlying_reason, audit_rows)

    standard_mask, standard_reason = _standard_contract_mask(panel)
    panel = _apply_panel_filter(panel, standard_mask, "standard_listed_options", standard_reason, audit_rows)

    required_nonmissing = ["date", "exdate", "strike", "dte", "T", "forward_ref", "spot_ref", "call_equiv_mid", "mid_price", "iv_for_filter", "log_forward_moneyness"]
    panel = _apply_panel_filter(
        panel,
        panel[required_nonmissing].replace([np.inf, -np.inf], np.nan).notna().all(axis=1),
        "required_fields_present",
        "remove rows missing date, expiry, strike, maturity, forward/spot, mid/call-equivalent mid, IV or log-moneyness",
        audit_rows,
    )

    panel = _apply_panel_filter(
        panel,
        pd.to_numeric(panel["dte"], errors="coerce").between(SPX_HEDGING_MIN_DTE, SPX_HEDGING_MAX_DTE),
        "maturity_1m_to_1y",
        f"{SPX_HEDGING_MIN_DTE} <= calendar days to expiry <= {SPX_HEDGING_MAX_DTE}",
        audit_rows,
    )

    panel = _apply_panel_filter(
        panel,
        pd.to_numeric(panel["abs_log_forward_moneyness"], errors="coerce").le(SPX_HEDGING_MAX_ABS_LOG_MONEYNESS),
        "near_atm_window_abs_log_moneyness_le_010",
        f"abs(log(K/F)) <= {SPX_HEDGING_MAX_ABS_LOG_MONEYNESS}",
        audit_rows,
    )

    panel = _apply_panel_filter(
        panel,
        np.isfinite(pd.to_numeric(panel["iv_for_filter"], errors="coerce")) & pd.to_numeric(panel["iv_for_filter"], errors="coerce").gt(SPX_HEDGING_MIN_IV),
        "positive_finite_iv",
        "implied volatility must be finite and positive; no fake IVs",
        audit_rows,
    )

    panel = _apply_panel_filter(
        panel,
        np.isfinite(pd.to_numeric(panel["mid_price"], errors="coerce")) & pd.to_numeric(panel["mid_price"], errors="coerce").gt(SPX_HEDGING_MIN_MID),
        "positive_mid_price",
        "mid or call-equivalent mid price must be finite and positive",
        audit_rows,
    )

    if {"bid", "ask"}.issubset(panel.columns):
        bid = pd.to_numeric(panel["bid"], errors="coerce")
        ask = pd.to_numeric(panel["ask"], errors="coerce")
        panel = _apply_panel_filter(
            panel,
            bid.gt(SPX_HEDGING_MIN_BID) & ask.gt(bid),
            "raw_bid_ask_valid",
            f"bid > {SPX_HEDGING_MIN_BID} and ask > bid",
            audit_rows,
        )
        panel["raw_bid_ask_available"] = True
    else:
        if SPX_HEDGING_REQUIRE_RAW_BID_ASK_FOR_HEADLINE:
            raise ValueError("Raw bid/ask columns are absent from the collapsed surface and SPX_HEDGING_REQUIRE_RAW_BID_ASK_FOR_HEADLINE=True.")
        before = panel.copy()
        panel["raw_bid_ask_available"] = False
        audit_rows.append(_filter_audit_row(
            "raw_bid_ask_valid_unavailable",
            before,
            panel,
            "raw bid/ask not present in collapsed call-equivalent surface; no bid/ask values are synthesized, and headline wording must not claim raw bid/ask filtering",
        ))

    liq_mask, liq_reason = _liquidity_mask(panel)
    panel = _apply_panel_filter(panel, liq_mask, "volume_open_interest_liquidity", liq_reason, audit_rows)

    spread_mask, spread_reason = _spread_mask(panel)
    panel = _apply_panel_filter(panel, spread_mask, "bid_ask_spread_discipline", spread_reason, audit_rows)

    if "has_static_arb_violation" in panel.columns:
        panel = _apply_panel_filter(
            panel,
            ~panel["has_static_arb_violation"].fillna(False).astype(bool),
            "static_arbitrage_clean_expiries",
            "exclude rows from expiry slices flagged by static-arbitrage diagnostics",
            audit_rows,
        )

    if panel.empty:
        raise ValueError("Hedging-focused SPX calibration panel is empty after pre-specified filters.")
    if panel["date"].nunique() < SPX_HEDGING_MIN_DATES:
        raise ValueError(f"Hedging-focused SPX panel has only {panel['date'].nunique()} dates; minimum required is {SPX_HEDGING_MIN_DATES}.")
    if panel.get("manual_date_deletion", pd.Series(False, index=panel.index)).astype(bool).any():
        raise RuntimeError("Manual date deletion detected in the hedging-focused panel; headline results are blocked.")

    panel = panel.sort_values(["date", "exdate", "abs_log_forward_moneyness", "strike"]).reset_index(drop=True)
    panel["headline_use"] = True
    panel["empirical_mode"] = HEADLINE_EMPIRICAL_MODE
    panel["calibration_panel_label"] = "HEDGING-FOCUSED PANEL: used for headline calibration and policy evaluation"

    regime_labels = build_spx_regime_labels(global_diag, panel)
    panel = panel.merge(regime_labels[["date", "regime_label", "regime_variable", "regime_threshold", "source"]], on="date", how="left")
    if panel["regime_label"].isna().any():
        raise ValueError("Some hedging panel dates did not receive calm/stress regime labels.")

    split_manifest = build_spx_chronological_split(panel)
    date_to_split = split_manifest.set_index("date")["split"].to_dict()
    panel["split"] = panel["date"].map(date_to_split)
    panel["split_rule"] = SPX_HEDGING_SPLIT_RULE
    if panel["split"].isna().any():
        raise ValueError("Some hedging panel rows did not receive deterministic train/validation/test labels.")

    seed_panel = build_spx_hedging_seed_panel(panel)
    calm_panel = panel.loc[panel["regime_label"].eq("calm")].copy()
    stress_panel = panel.loc[panel["regime_label"].eq("stress")].copy()
    if calm_panel.empty or stress_panel.empty:
        raise ValueError("Calm/stress panel split produced an empty regime. Do not silently delete crisis/calm periods.")

    manifest = pd.DataFrame(audit_rows)
    return global_diag, panel, seed_panel, calm_panel, stress_panel, regime_labels, split_manifest, manifest


def build_spx_regime_labels(global_surface: pd.DataFrame, hedging_panel: pd.DataFrame) -> pd.DataFrame:
    frame = global_surface.copy()
    frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()
    vix_col = _first_present(["vix", "VIX", "vix_close", "VIXCLS"], frame)
    dates = pd.DataFrame({"date": sorted(pd.to_datetime(hedging_panel["date"]).dt.normalize().unique())})
    if vix_col is not None:
        vix = frame.groupby("date", as_index=False)[vix_col].first().rename(columns={vix_col: "regime_variable"})
        labels = dates.merge(vix, on="date", how="left")
        if labels["regime_variable"].isna().any():
            raise ValueError("VIX column exists but is missing for some hedging dates; regime labelling refuses partial fake coverage.")
        threshold = float(SPX_HEDGING_VIX_STRESS_THRESHOLD)
        labels["regime_label"] = np.where(pd.to_numeric(labels["regime_variable"], errors="coerce").ge(threshold), "stress", "calm")
        labels["regime_threshold"] = threshold
        labels["source"] = "VIX"
        return labels[["date", "regime_label", "regime_variable", "regime_threshold", "source"]]

    spot_col = _first_present(["spot_ref", "spot", "underlying_price", "S0"], frame)
    if spot_col is None:
        raise ValueError("No VIX and no spot/index level found; cannot build deterministic calm/stress regime labels.")
    spot = (
        frame[["date", spot_col]]
        .dropna()
        .drop_duplicates("date")
        .sort_values("date")
        .rename(columns={spot_col: "spot_for_regime"})
    )
    spot["log_return"] = np.log(pd.to_numeric(spot["spot_for_regime"], errors="coerce")).diff()
    spot["realized_vol_21d"] = spot["log_return"].rolling(21, min_periods=10).std() * math.sqrt(float(CFG.market.trading_days_per_year))
    labels = dates.merge(spot[["date", "realized_vol_21d"]], on="date", how="left")
    labels["regime_variable"] = labels["realized_vol_21d"].ffill().bfill()
    if labels["regime_variable"].isna().any():
        raise ValueError("Realised-volatility regime proxy could not be computed for hedging dates.")
    threshold = float(labels["regime_variable"].quantile(SPX_HEDGING_STRESS_REALIZED_VOL_QUANTILE))
    labels["regime_label"] = np.where(labels["regime_variable"].ge(threshold), "stress", "calm")
    labels["regime_threshold"] = threshold
    labels["source"] = "realised-vol proxy: 21-trading-day SPX log-return volatility, top 20% stress"
    return labels[["date", "regime_label", "regime_variable", "regime_threshold", "source"]]


def build_spx_chronological_split(panel: pd.DataFrame) -> pd.DataFrame:
    dates = pd.Series(sorted(pd.to_datetime(panel["date"]).dt.normalize().unique()), name="date")
    n = len(dates)
    if n < SPX_HEDGING_MIN_DATES:
        raise ValueError(f"Need at least {SPX_HEDGING_MIN_DATES} unique dates for deterministic split; got {n}.")
    ranks = np.arange(n)
    q = (ranks + 0.5) / float(n)
    split = np.where(q <= 0.60, "train", np.where(q <= 0.80, "validation", "test"))
    out = pd.DataFrame({"date": dates, "chronological_rank": ranks, "chronological_quantile": q, "split": split})
    out["split_rule"] = SPX_HEDGING_SPLIT_RULE
    date_counts = panel.groupby("date").size().rename("rows_for_date").reset_index()
    out = out.merge(date_counts, on="date", how="left")
    return out


def build_spx_hedging_seed_panel(panel: pd.DataFrame) -> pd.DataFrame:
    seed = panel.loc[pd.to_numeric(panel["abs_log_forward_moneyness"], errors="coerce").le(SPX_HEDGING_PRIMARY_ATM_ABS_LOG_MONEYNESS)].copy()
    if seed.empty:
        raise ValueError("No primary near-ATM seed candidates remain after abs(log(K/F)) <= 0.05.")
    seed = seed.sort_values(["date", "exdate", "abs_log_forward_moneyness", "dte", "strike"]).copy()
    seed = seed.groupby(["date", "exdate"], as_index=False, sort=True).head(1).reset_index(drop=True)
    seed["hedging_seed_panel_version"] = SPX_HEDGING_PANEL_VERSION
    seed["seed_selection_rule"] = "one closest-to-forward near-ATM row per date-expiry from pre-filtered liquid hedging panel"
    seed["seed_id"] = [f"{pd.Timestamp(r.date).date()}_{pd.Timestamp(r.exdate).date()}_{int(round(float(r.strike)))}" for r in seed.itertuples(index=False)]
    seed["selected_after_observing_hedging_performance"] = False
    seed["manual_override"] = False
    return seed


def write_spx_hedging_panel_outputs(global_diag, panel, seed_panel, calm_panel, stress_panel, regime_labels, split_manifest, manifest) -> None:
    global_diag.to_parquet(SPX_HEDGING_OUTPUTS["global_diagnostic_panel"], index=False)
    panel.to_parquet(SPX_HEDGING_OUTPUTS["hedging_calibration_panel"], index=False)
    seed_panel.to_parquet(SPX_HEDGING_OUTPUTS["hedging_seed_panel"], index=False)
    calm_panel.to_parquet(SPX_HEDGING_OUTPUTS["hedging_calm_panel"], index=False)
    stress_panel.to_parquet(SPX_HEDGING_OUTPUTS["hedging_stress_panel"], index=False)
    regime_labels.to_csv(SPX_HEDGING_OUTPUTS["regime_labels"], index=False)
    split_manifest.to_csv(SPX_HEDGING_OUTPUTS["split_manifest"], index=False)
    manifest.to_csv(SPX_HEDGING_OUTPUTS["exclusion_manifest"], index=False)




def _read_spx_panel_output(path: Path) -> pd.DataFrame:
    path = Path(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path)


def load_spx_hedging_panel_outputs() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    paths, source_directory, discovery_audit = resolve_spx_hedging_panel_inputs()
    display(discovery_audit)
    global_diag = _read_spx_panel_output(paths["global_diagnostic_panel"])
    panel = _read_spx_panel_output(paths["hedging_calibration_panel"])
    seed_panel = _read_spx_panel_output(paths["hedging_seed_panel"])
    calm_panel = _read_spx_panel_output(paths["hedging_calm_panel"])
    stress_panel = _read_spx_panel_output(paths["hedging_stress_panel"])
    regime_labels = _read_spx_panel_output(paths["regime_labels"])
    split_manifest = _read_spx_panel_output(paths["split_manifest"])
    manifest = _read_spx_panel_output(paths["exclusion_manifest"])

    require_columns(panel, {"date", "exdate", "strike", "T", "iv_market", "empirical_mode", "headline_use", "split", "regime_label"}, "cached SPX hedging calibration panel")
    require_columns(seed_panel, {"date", "exdate", "seed_id", "hedging_seed_panel_version"}, "cached SPX hedging seed panel")
    if set(panel["empirical_mode"].dropna().astype(str).unique()) != {HEADLINE_EMPIRICAL_MODE}:
        raise ValueError("Cached SPX hedging panel is not labelled as the headline hedging-focused panel.")
    if not panel["headline_use"].astype(bool).all():
        raise ValueError("Cached SPX hedging panel contains rows not marked for headline use.")
    if set(seed_panel["hedging_seed_panel_version"].dropna().astype(str).unique()) != {SPX_HEDGING_PANEL_VERSION}:
        raise ValueError("Cached SPX hedging seed panel has a stale version label.")
    for frame in [global_diag, panel, seed_panel, calm_panel, stress_panel, regime_labels, split_manifest]:
        if "date" in frame.columns:
            frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()
    if "exdate" in panel.columns:
        panel["exdate"] = pd.to_datetime(panel["exdate"]).dt.normalize()
    if "exdate" in seed_panel.columns:
        seed_panel["exdate"] = pd.to_datetime(seed_panel["exdate"]).dt.normalize()

    # Only after schema/version validation do we copy the bundle into this run's isolated output.
    materialise_spx_hedging_panel_inputs(paths, source_directory)
    return global_diag, panel, seed_panel, calm_panel, stress_panel, regime_labels, split_manifest, manifest


def make_spx_panel_methodology_audit(panel: pd.DataFrame, seed_panel: pd.DataFrame, regime_labels: pd.DataFrame, manifest: pd.DataFrame) -> pd.DataFrame:
    checks = [
        ("SPX-only panel built", not panel.empty, f"rows={len(panel):,}, dates={panel['date'].nunique():,}"),
        ("quote filters applied", not manifest.empty and (manifest["filter_name"].nunique() >= 8), f"filters={manifest['filter_name'].nunique() if not manifest.empty else 0}"),
        ("exclusion manifest saved", SPX_HEDGING_OUTPUTS["exclusion_manifest"].exists(), str(SPX_HEDGING_OUTPUTS["exclusion_manifest"])),
        ("hedging calibration panel saved", SPX_HEDGING_OUTPUTS["hedging_calibration_panel"].exists(), str(SPX_HEDGING_OUTPUTS["hedging_calibration_panel"])),
        ("hedging seed panel saved", SPX_HEDGING_OUTPUTS["hedging_seed_panel"].exists(), str(SPX_HEDGING_OUTPUTS["hedging_seed_panel"])),
        ("calm/stress labels saved", SPX_HEDGING_OUTPUTS["regime_labels"].exists() and {"calm", "stress"}.issubset(set(regime_labels["regime_label"])), str(SPX_HEDGING_OUTPUTS["regime_labels"])),
        ("train/validation/test split deterministic", set(panel["split"].dropna().unique()) == {"train", "validation", "test"}, SPX_HEDGING_SPLIT_RULE),
        ("no fake pricing detected", "placeholder_price" not in panel.columns and not panel.astype(str).apply(lambda c: c.str.contains("placeholder|fake", case=False, regex=True, na=False)).any().any(), "no placeholder/fake markers in panel"),
        ("no synthetic bid/ask data", not {"synthetic_bid", "synthetic_ask"}.intersection(panel.columns), "raw bid/ask not fabricated"),
        ("no manual date deletion detected", not panel.get("manual_date_deletion", pd.Series(False, index=panel.index)).astype(bool).any(), "manual_date_deletion=False"),
        ("no crisis-window exclusion without regime labelling", not panel.get("crisis_window_excluded", pd.Series(False, index=panel.index)).astype(bool).any() and "regime_label" in panel.columns, "regime labels attached"),
        ("global diagnostic separated from headline panel", GLOBAL_DIAGNOSTIC_MODE != HEADLINE_EMPIRICAL_MODE and "headline_use" in panel.columns and panel["headline_use"].all(), "separate artefacts and empirical_mode labels"),
    ]
    out = pd.DataFrame([{"check": c, "status": "PASS" if bool(ok) else "FAIL", "evidence": e} for c, ok, e in checks])
    out.to_csv(SPX_HEDGING_OUTPUTS["methodology_audit"], index=False)
    return out


# Preserve the original full-surface object as a diagnostic artefact, then route headline work to the hedging panel.
surface_panel_global_diagnostic = surface_panel.copy()
_force_panel = CFG.flags.force_rebuild("hedging_panel")
_panel_loaded = False
if not _force_panel:
    try:
        (
            spx_global_diagnostic_panel, spx_hedging_calibration_panel,
            spx_hedging_seed_panel, spx_hedging_calm_panel,
            spx_hedging_stress_panel, spx_regime_labels,
            spx_hedging_split_manifest, spx_hedging_exclusion_manifest,
        ) = load_spx_hedging_panel_outputs()
        _panel_loaded = True
        print(f"[hedging panel] loaded compatible artefacts into {CFG.paths.table_dir}.", flush=True)
    except (FileNotFoundError, ValueError) as exc:
        if not CFG.flags.can_rebuild("hedging_panel"):
            raise RuntimeError(
                "No compatible run-local or legacy TEST30 SPX hedging-panel bundle was found, "
                "and the current execution plan does not permit rebuilding the panel. "
                "Either set RUN.legacy_hedging_panel_dir to the directory containing the eight "
                "spx_hedging_* artefacts, or set rebuild_from='hedging_panel' to rebuild the panel "
                "from the already validated surface without recalibrating Heston or Lifted Heston."
            ) from exc
        print(f"[hedging panel] cache unavailable/stale; rebuilding. Reason: {type(exc).__name__}: {exc}", flush=True)

if not _panel_loaded:
    print("[hedging panel] rebuilding from the validated global surface.", flush=True)
    (
        spx_global_diagnostic_panel, spx_hedging_calibration_panel,
        spx_hedging_seed_panel, spx_hedging_calm_panel,
        spx_hedging_stress_panel, spx_regime_labels,
        spx_hedging_split_manifest, spx_hedging_exclusion_manifest,
    ) = build_spx_hedging_panels(surface_panel_global_diagnostic, CFG)
    write_spx_hedging_panel_outputs(
        spx_global_diagnostic_panel, spx_hedging_calibration_panel,
        spx_hedging_seed_panel, spx_hedging_calm_panel,
        spx_hedging_stress_panel, spx_regime_labels,
        spx_hedging_split_manifest, spx_hedging_exclusion_manifest,
    )

spx_hedging_panel_methodology_audit = make_spx_panel_methodology_audit(
    spx_hedging_calibration_panel,
    spx_hedging_seed_panel,
    spx_regime_labels,
    spx_hedging_exclusion_manifest,
)

# Critical routing line: all downstream headline calibration / seed construction consumes this panel.
surface_panel = spx_hedging_calibration_panel.copy()

print("GLOBAL SURFACE DIAGNOSTIC: not used for headline hedging results")
print("HEDGING-FOCUSED PANEL: used for headline calibration and policy evaluation")
print({
    "global_diagnostic_rows": int(len(spx_global_diagnostic_panel)),
    "hedging_calibration_rows": int(len(spx_hedging_calibration_panel)),
    "hedging_calibration_dates": int(spx_hedging_calibration_panel["date"].nunique()),
    "hedging_seed_rows": int(len(spx_hedging_seed_panel)),
    "calm_rows": int(len(spx_hedging_calm_panel)),
    "stress_rows": int(len(spx_hedging_stress_panel)),
    "calibration_panel_path": str(SPX_HEDGING_OUTPUTS["hedging_calibration_panel"]),
})
display(spx_hedging_exclusion_manifest)
display(spx_regime_labels.head())
display(spx_hedging_split_manifest.groupby("split", as_index=False).agg(dates=("date", "nunique"), rows=("rows_for_date", "sum")))
display(spx_hedging_panel_methodology_audit)


,candidate_directory,complete,missing
0,"C:\Users\lhkke\OneDrive - Queen Mary, Universi...",True,


[hedging panel] loaded compatible artefacts into C:\Users\lhkke\OneDrive - Queen Mary, University of London\Documents\Dissertation\Deep-Hedging-under-Lifted-Heston-MSc-Dissertation-\outputs_observable_h035_lag_consistent_retrained_full\tables.
GLOBAL SURFACE DIAGNOSTIC: not used for headline hedging results
HEDGING-FOCUSED PANEL: used for headline calibration and policy evaluation
{'global_diagnostic_rows': 254081, 'hedging_calibration_rows': 133868, 'hedging_calibration_dates': 2869, 'hedging_seed_rows': 8451, 'calm_rows': 112590, 'stress_rows': 21278, 'calibration_panel_path': 'C:\\Users\\lhkke\\OneDrive - Queen Mary, University of London\\Documents\\Dissertation\\Deep-Hedging-under-Lifted-Heston-MSc-Dissertation-\\outputs_observable_h035_lag_consistent_retrained_full\\tables\\spx_hedging_calibration_panel.parquet'}


,filter_name,rows_before,rows_after,rows_removed,pct_removed,unique_dates_before,unique_dates_after,unique_expiries_before,unique_expiries_after,reason
0,underlying_spx_only,254081,254081,0,0.000000,2869,2869,803,803,no underlying identifier in collapsed surface;...
1,standard_listed_options,254081,254081,0,0.000000,2869,2869,803,803,no non-standard contract flag in collapsed sur...
2,required_fields_present,254081,254081,0,0.000000,2869,2869,803,803,"remove rows missing date, expiry, strike, matu..."
3,maturity_1m_to_1y,254081,254081,0,0.000000,2869,2869,803,803,7 <= calendar days to expiry <= 365
4,near_atm_window_abs_log_moneyness_le_010,254081,133868,120213,0.473129,2869,2869,803,803,abs(log(K/F)) <= 0.1
5,positive_finite_iv,133868,133868,0,0.000000,2869,2869,803,803,implied volatility must be finite and positive...
6,positive_mid_price,133868,133868,0,0.000000,2869,2869,803,803,mid or call-equivalent mid price must be finit...
7,raw_bid_ask_valid_unavailable,133868,133868,0,0.000000,2869,2869,803,803,raw bid/ask not present in collapsed call-equi...
8,volume_open_interest_liquidity,133868,133868,0,0.000000,2869,2869,803,803,open_interest >= 0 or volume >= 0
9,bid_ask_spread_discipline,133868,133868,0,0.000000,2869,2869,803,803,0 < median_rel_spread <= 0.25


,date,regime_label,regime_variable,regime_threshold,source
0,2010-01-04,calm,0.112328,0.212914,realised-vol proxy: 21-trading-day SPX log-ret...
1,2010-01-05,calm,0.112328,0.212914,realised-vol proxy: 21-trading-day SPX log-ret...
2,2010-01-06,calm,0.112328,0.212914,realised-vol proxy: 21-trading-day SPX log-ret...
3,2010-01-07,calm,0.112328,0.212914,realised-vol proxy: 21-trading-day SPX log-ret...
4,2010-01-08,calm,0.112328,0.212914,realised-vol proxy: 21-trading-day SPX log-ret...


,split,dates,rows
0,test,574,28587
1,train,1721,80499
2,validation,574,24782


,check,status,evidence
0,SPX-only panel built,PASS,"rows=133,868, dates=2,869"
1,quote filters applied,PASS,filters=11
2,exclusion manifest saved,PASS,"C:\Users\lhkke\OneDrive - Queen Mary, Universi..."
3,hedging calibration panel saved,PASS,"C:\Users\lhkke\OneDrive - Queen Mary, Universi..."
4,hedging seed panel saved,PASS,"C:\Users\lhkke\OneDrive - Queen Mary, Universi..."
5,calm/stress labels saved,PASS,"C:\Users\lhkke\OneDrive - Queen Mary, Universi..."
6,train/validation/test split deterministic,PASS,chronological_by_date_60_20_20
7,no fake pricing detected,PASS,no placeholder/fake markers in panel
8,no synthetic bid/ask data,PASS,raw bid/ask not fabricated
9,no manual date deletion detected,PASS,manual_date_deletion=False


In [19]:
# ============================================================
# Regularised forward-variance approximation from near-ATM implied total variance
# ============================================================

FORWARD_VARIANCE_CURVE_VERSION = "regularised_near_atm_total_variance_final"


def _normalised_positive_weights(values: pd.Series | np.ndarray, n: int) -> np.ndarray:
    raw = np.asarray(values, dtype=float) if values is not None else np.ones(n, dtype=float)
    if raw.shape != (n,):
        raw = np.ones(n, dtype=float)
    raw = np.where(np.isfinite(raw) & (raw > 0), raw, 1.0)
    med = float(np.nanmedian(raw)) if np.any(np.isfinite(raw)) else 1.0
    med = max(med, 1e-12)
    return np.clip(raw / med, 0.25, 4.0)


def _near_atm_total_variance_points(surface: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    """Collapse each date/maturity to a regularised near-ATM total-variance observation.

    This deliberately does not take a single nearest-ATM quote. A single quote is too noisy for
    the initial curve that supplies v0/theta in the final 5-parameter lifted calibration. Instead,
    all quotes inside the configured near-ATM band are combined with a moneyness kernel and,
    when available, the existing surface fit weights.
    """
    required = {"date", "T", "iv_market", "log_forward_moneyness"}
    missing = required - set(surface.columns)
    if missing:
        raise ValueError(f"Forward-variance construction missing required columns: {sorted(missing)}")

    x = surface.copy()
    x["date"] = pd.to_datetime(x["date"]).dt.normalize()
    for col in ["T", "iv_market", "log_forward_moneyness"]:
        x[col] = pd.to_numeric(x[col], errors="coerce")
    bad = x[["T", "iv_market", "log_forward_moneyness"]].isna().any(axis=1) | (x["T"] <= 0) | (x["iv_market"] <= 0)
    if bad.any():
        raise ValueError(f"Forward-variance construction found {int(bad.sum())} invalid rows.")

    max_abs = float(cfg.market.max_atm_forward_moneyness)
    bandwidth = float(getattr(cfg.market, "forward_variance_atm_kernel_bandwidth", max_abs / 2.0))
    bandwidth = max(bandwidth, 1e-8)
    x = x.assign(abs_mny=x["log_forward_moneyness"].abs())
    x = x.loc[x["abs_mny"] <= max_abs].copy()
    if x.empty:
        raise ValueError("No sufficiently near-ATM observations are available for forward-variance estimation.")

    base_weight = np.ones(len(x), dtype=float)
    if bool(getattr(cfg.market, "forward_variance_use_fit_weights", True)) and "fit_weight" in x.columns:
        base_weight = _normalised_positive_weights(pd.to_numeric(x["fit_weight"], errors="coerce"), len(x))
    kernel_weight = np.exp(-0.5 * (x["abs_mny"].to_numpy(float) / bandwidth) ** 2)
    x["forward_curve_obs_weight"] = np.clip(base_weight * kernel_weight, 1e-8, np.inf)
    x["weighted_iv_variance"] = x["forward_curve_obs_weight"] * x["iv_market"].to_numpy(float) ** 2

    grouped = []
    for (date, T), g in x.groupby(["date", "T"], sort=True):
        w = g["forward_curve_obs_weight"].to_numpy(float)
        iv_var = float(np.sum(g["weighted_iv_variance"].to_numpy(float)) / max(np.sum(w), 1e-12))
        atm_iv = math.sqrt(max(iv_var, float(cfg.market.forward_variance_min)))
        grouped.append({
            "date": pd.Timestamp(date),
            "T": float(T),
            "atm_iv": float(atm_iv),
            "raw_atm_total_variance": float(float(T) * iv_var),
            "near_atm_quote_count": int(len(g)),
            "atm_abs_log_moneyness_min": float(g["abs_mny"].min()),
            "atm_abs_log_moneyness_weighted": float(np.sum(g["abs_mny"].to_numpy(float) * w) / max(np.sum(w), 1e-12)),
            "forward_curve_obs_weight_sum": float(np.sum(w)),
        })
    out = pd.DataFrame(grouped).sort_values(["date", "T"]).reset_index(drop=True)
    if out.empty:
        raise ValueError("Near-ATM total-variance aggregation returned no rows.")
    return out


def _difference_matrix(n: int, order: int) -> np.ndarray:
    eye = np.eye(n)
    if order == 1:
        return np.diff(eye, n=1, axis=0)
    if order == 2:
        return np.diff(eye, n=2, axis=0)
    raise ValueError("Only first- and second-difference regularisation is supported.")


def _fit_positive_forward_variance(
    maturities: np.ndarray,
    raw_total_variance: np.ndarray,
    observation_weights: np.ndarray,
    smoothness: float,
    curvature_smoothness: float,
    minimum: float,
    maximum: float,
) -> tuple[np.ndarray, np.ndarray, dict[str, Any]]:
    """Fit a positive piecewise-constant ξ₀ curve by constrained regularised LS.

    The fitted total variance W(T)=∫₀ᵀξ₀(s)ds is monotone by construction because ξ₀ is
    constrained positive. First- and second-difference penalties suppress jagged forward
    variance estimates caused by noisy ATM quotes.
    """
    if not SCIPY_AVAILABLE:
        raise RuntimeError("SciPy is required for constrained forward-variance estimation.")
    T = np.asarray(maturities, dtype=float)
    W = np.asarray(raw_total_variance, dtype=float)
    obs_w = _normalised_positive_weights(observation_weights, len(T))
    if len(T) == 0 or np.any(np.diff(T) <= 0) or np.any(~np.isfinite(W)) or np.any(W <= 0):
        raise ValueError("Invalid ATM total-variance term structure.")
    if not np.isfinite(maximum) or maximum <= minimum:
        maximum = np.inf

    dt = np.diff(np.concatenate([[0.0], T]))
    integration = np.tril(np.ones((len(T), len(T)), dtype=float)) * dt.reshape(1, -1)
    design_blocks = [integration * np.sqrt(obs_w).reshape(-1, 1)]
    target_blocks = [W * np.sqrt(obs_w)]

    # Penalise excessive changes in ξ₀. This is deliberately transparent: it is a
    # regularised curve estimate, not hidden endpoint recalibration.
    if len(T) > 1 and smoothness > 0:
        d1 = _difference_matrix(len(T), 1)
        scale = max(float(np.nanmedian(W / np.maximum(T, 1e-12))), minimum)
        design_blocks.append(math.sqrt(float(smoothness)) * d1 / scale)
        target_blocks.append(np.zeros(d1.shape[0], dtype=float))
    if len(T) > 2 and curvature_smoothness > 0:
        d2 = _difference_matrix(len(T), 2)
        scale = max(float(np.nanmedian(W / np.maximum(T, 1e-12))), minimum)
        design_blocks.append(math.sqrt(float(curvature_smoothness)) * d2 / scale)
        target_blocks.append(np.zeros(d2.shape[0], dtype=float))

    design = np.vstack(design_blocks)
    target = np.concatenate(target_blocks)
    result = lsq_linear(
        design,
        target,
        bounds=(float(minimum), float(maximum)),
        lsmr_tol="auto",
        max_iter=10_000,
        verbose=0,
    )
    if not result.success:
        raise RuntimeError(f"Positive forward-variance estimation failed: {result.message}")
    forward = np.asarray(result.x, dtype=float)
    fitted_total = integration @ forward
    if np.any(~np.isfinite(forward)) or np.any(forward < minimum) or np.any(fitted_total <= 0):
        raise FloatingPointError("Constrained forward-variance estimator returned invalid values.")

    total_error = fitted_total - W
    rel_adjustment = np.abs(total_error) / np.maximum(np.abs(W), 1e-12)
    d1_norm = float(np.sqrt(np.mean(np.diff(forward) ** 2))) if len(forward) > 1 else 0.0
    d2_norm = float(np.sqrt(np.mean(np.diff(forward, n=2) ** 2))) if len(forward) > 2 else 0.0
    diagnostics = {
        "forward_curve_version": FORWARD_VARIANCE_CURVE_VERSION,
        "forward_curve_cost": float(result.cost),
        "forward_curve_optimality": float(result.optimality),
        "forward_curve_status": int(result.status),
        "forward_curve_total_variance_rmse": float(np.sqrt(np.mean(total_error ** 2))),
        "forward_curve_total_variance_max_abs_error": float(np.max(np.abs(total_error))),
        "forward_curve_total_variance_max_rel_adjustment": float(np.max(rel_adjustment)),
        "forward_curve_d1_rmse": d1_norm,
        "forward_curve_d2_rmse": d2_norm,
        "forward_curve_min_xi0": float(np.min(forward)),
        "forward_curve_max_xi0": float(np.max(forward)),
        "forward_curve_total_variance_monotone": bool(np.all(np.diff(fitted_total) >= -1e-12)),
        "forward_curve_smoothness": float(smoothness),
        "forward_curve_curvature_smoothness": float(curvature_smoothness),
    }
    return forward, fitted_total, diagnostics


def build_atm_forward_variance_curve(surface: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    atm = _near_atm_total_variance_points(surface, cfg)
    min_maturities = int(getattr(cfg.market, "forward_variance_min_maturities", 2))
    maturity_counts = atm.groupby("date")["T"].nunique()
    eligible_dates = maturity_counts[maturity_counts >= max(min_maturities, 2)].index
    excluded_dates = maturity_counts[maturity_counts < max(min_maturities, 2)].index
    atm = atm.loc[atm["date"].isin(eligible_dates)].copy()
    if atm.empty:
        raise ValueError("No date has enough near-ATM maturities for regularised forward-variance estimation.")
    print({
        "forward_variance_dates": int(len(eligible_dates)),
        "excluded_insufficient_atm_maturities": int(len(excluded_dates)),
        "forward_curve_version": FORWARD_VARIANCE_CURVE_VERSION,
    })

    rows = []
    for date, g in atm.groupby("date", sort=True):
        g = g.sort_values("T").reset_index(drop=True)
        T = g["T"].to_numpy(float)
        raw_W = g["raw_atm_total_variance"].to_numpy(float)
        obs_w = g["forward_curve_obs_weight_sum"].to_numpy(float)
        forward, fitted_W, diag = _fit_positive_forward_variance(
            T,
            raw_W,
            observation_weights=obs_w,
            smoothness=float(cfg.market.forward_variance_smoothness),
            curvature_smoothness=float(getattr(cfg.market, "forward_variance_curvature_smoothness", 0.0)),
            minimum=float(cfg.market.forward_variance_min),
            maximum=float(getattr(cfg.market, "forward_variance_max", np.inf)),
        )
        max_rel_allowed = float(getattr(cfg.market, "forward_variance_max_relative_total_variance_adjustment", np.inf))
        adjustment_flag = "OK" if diag["forward_curve_total_variance_max_rel_adjustment"] <= max_rel_allowed else "REVIEW_LARGE_TOTAL_VARIANCE_ADJUSTMENT"
        for j, row in enumerate(g.itertuples(index=False)):
            rows.append({
                "date": pd.Timestamp(date),
                "T": float(T[j]),
                "atm_iv": float(row.atm_iv),
                "raw_atm_total_variance": float(raw_W[j]),
                "atm_total_variance": float(fitted_W[j]),
                "total_variance_adjustment": float(fitted_W[j] - raw_W[j]),
                "forward_variance": float(forward[j]),
                "near_atm_quote_count": int(row.near_atm_quote_count),
                "atm_abs_log_moneyness_min": float(row.atm_abs_log_moneyness_min),
                "atm_abs_log_moneyness_weighted": float(row.atm_abs_log_moneyness_weighted),
                "forward_curve_obs_weight_sum": float(row.forward_curve_obs_weight_sum),
                "forward_curve_adjustment_flag": adjustment_flag,
                "curve_regularization_version": FORWARD_VARIANCE_CURVE_VERSION,
                **diag,
            })
    out = pd.DataFrame(rows)
    if out.empty:
        raise ValueError("Forward-variance curve construction returned no rows.")
    if np.any(~np.isfinite(out["forward_variance"])) or np.any(out["forward_variance"] <= 0):
        raise FloatingPointError("Regularised forward-variance curve contains invalid values.")
    return out.sort_values(["date", "T"]).reset_index(drop=True)


def load_or_build_forward_variance_curve(surface: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    path = cfg.paths.forward_variance_path
    deps = [globals().get("SPX_HEDGING_CALIBRATION_PANEL_PATH", cfg.paths.surface_cache_path)]
    required = {
        "date", "T", "atm_total_variance", "forward_variance", "curve_regularization_version",
        "forward_curve_total_variance_max_rel_adjustment", "forward_curve_d1_rmse", "forward_curve_d2_rmse",
    }
    if not cfg.flags.force_rebuild("calibration"):
        try:
            curve = load_dataframe_cache(path, "atm_forward_variance_curve", "forward_variance", cfg, required_columns=required, dependencies=deps)
            versions = set(curve["curve_regularization_version"].dropna().astype(str).unique())
            if versions != {FORWARD_VARIANCE_CURVE_VERSION}:
                raise ValueError(f"Forward-variance cache version mismatch: {versions}")
            print(f"Loading validated regularised ATM forward-variance curve: {path}")
            return curve
        except (FileNotFoundError, ValueError) as exc:
            if not calibration_rebuild_allowed(cfg):
                raise RuntimeError(
                    "Forward-variance cache is missing or stale and strict reuse forbids rebuilding. "
                    "Set rebuild_from='calibration' or use rebuild_from='auto'."
                ) from exc
            print(f"[forward-variance] cache unavailable/stale; rebuilding. Reason: {exc}", flush=True)
    curve = build_atm_forward_variance_curve(surface, cfg)
    write_dataframe_cache(curve, path, "atm_forward_variance_curve", "forward_variance", cfg, dependencies=deps)
    print(f"Saved regularised ATM forward-variance curve: {path}")
    return curve


forward_variance_curve = load_or_build_forward_variance_curve(surface_panel, CFG)
display(forward_variance_curve.head())

forward_variance_curve_audit = (
    forward_variance_curve
    .groupby("date", as_index=False)
    .agg(
        maturities=("T", "nunique"),
        min_xi0=("forward_variance", "min"),
        max_xi0=("forward_variance", "max"),
        max_rel_total_variance_adjustment=("forward_curve_total_variance_max_rel_adjustment", "max"),
        d1_rmse=("forward_curve_d1_rmse", "max"),
        d2_rmse=("forward_curve_d2_rmse", "max"),
        adjustment_flag=("forward_curve_adjustment_flag", "first"),
    )
)
display(forward_variance_curve_audit.head())
print({
    "regularised_forward_variance_dates": int(forward_variance_curve_audit["date"].nunique()),
    "review_large_adjustment_dates": int(forward_variance_curve_audit["adjustment_flag"].ne("OK").sum()),
    "curve_version": FORWARD_VARIANCE_CURVE_VERSION,
})

if MATPLOTLIB_AVAILABLE:
    sample_dates = sorted(forward_variance_curve["date"].drop_duplicates())[:3]
    for d in sample_dates:
        g = forward_variance_curve[forward_variance_curve["date"].eq(d)].sort_values("T")
        fig, ax = plt.subplots(figsize=(7, 3))
        ax.plot(g["T"], g["forward_variance"], marker="o")
        ax.set_title(f"Regularised near-ATM forward variance | {pd.Timestamp(d).date()}")
        ax.set_xlabel("Maturity T")
        ax.set_ylabel("Forward variance")
        ax.grid(True, alpha=0.3)
        plt.show()


Loading validated regularised ATM forward-variance curve: C:\Users\lhkke\OneDrive - Queen Mary, University of London\Documents\Dissertation\Deep-Hedging-under-Lifted-Heston-MSc-Dissertation-\cache_final\hedging_panel_forward_variance_curve_regularised_5bucket.parquet


,date,T,atm_iv,raw_atm_total_variance,atm_total_variance,total_variance_adjustment,forward_variance,near_atm_quote_count,atm_abs_log_moneyness_min,atm_abs_log_moneyness_weighted,forward_curve_obs_weight_sum,forward_curve_adjustment_flag,curve_regularization_version,forward_curve_version,forward_curve_cost,forward_curve_optimality,forward_curve_status,forward_curve_total_variance_rmse,forward_curve_total_variance_max_abs_error,forward_curve_total_variance_max_rel_adjustment,forward_curve_d1_rmse,forward_curve_d2_rmse,forward_curve_min_xi0,forward_curve_max_xi0,forward_curve_total_variance_monotone,forward_curve_smoothness,forward_curve_curvature_smoothness
0,2010-01-04,0.235455,0.200034,0.009421,0.009655,0.000233,0.041004,2,0.019197,0.021961,0.915151,OK,regularised_near_atm_total_variance_final,regularised_near_atm_total_variance_final,1.917952e-08,1.494674e-17,3,0.000173,0.000233,0.024765,0.000004,0.000000,0.041004,0.041009,True,0.002,0.02
1,2010-01-04,0.281999,0.203152,0.011638,0.011563,-0.000075,0.041009,5,0.002178,0.017027,2.380460,OK,regularised_near_atm_total_variance_final,regularised_near_atm_total_variance_final,1.917952e-08,1.494674e-17,3,0.000173,0.000233,0.024765,0.000004,0.000000,0.041004,0.041009,True,0.002,0.02
2,2010-01-05,0.232717,0.192320,0.008607,0.011428,0.002821,0.049108,3,0.016139,0.024088,1.060561,OK,regularised_near_atm_total_variance_final,regularised_near_atm_total_variance_final,2.525288e-06,2.526191e-17,3,0.002006,0.002821,0.327707,0.000283,0.000000,0.049108,0.049390,True,0.002,0.02
3,2010-01-05,0.950034,0.222798,0.047159,0.046857,-0.000302,0.049390,5,0.004751,0.018109,2.427428,OK,regularised_near_atm_total_variance_final,regularised_near_atm_total_variance_final,2.525288e-06,2.526191e-17,3,0.002006,0.002821,0.327707,0.000283,0.000000,0.049108,0.049390,True,0.002,0.02
4,2010-01-06,0.229979,0.188804,0.008198,0.010181,0.001983,0.044269,3,0.015255,0.024142,1.165700,OK,regularised_near_atm_total_variance_final,regularised_near_atm_total_variance_final,4.041027e-06,7.819944e-16,3,0.001789,0.002189,0.241864,0.000314,0.000001,0.044269,0.044897,True,0.002,0.02


,date,maturities,min_xi0,max_xi0,max_rel_total_variance_adjustment,d1_rmse,d2_rmse,adjustment_flag
0,2010-01-04,2,0.041004,0.041009,0.024765,0.000004,0.000000,OK
1,2010-01-05,2,0.049108,0.049390,0.327707,0.000283,0.000000,OK
2,2010-01-06,3,0.044269,0.044897,0.241864,0.000314,0.000001,OK
3,2010-01-07,3,0.042361,0.042966,0.238627,0.000302,0.000002,OK
4,2010-01-08,3,0.040307,0.040880,0.245627,0.000287,0.000002,OK


{'regularised_forward_variance_dates': 2161, 'review_large_adjustment_dates': 147, 'curve_version': 'regularised_near_atm_total_variance_final'}


<Figure size 910x390 with 1 Axes>

<Figure size 910x390 with 1 Axes>

<Figure size 910x390 with 1 Axes>

## 3c. Surface Coverage and Forward-Variance Diagnostics

This section checks whether the surface and forward-variance inputs are empirically usable. Surface coverage diagnostics report the number of quotes, dates, expiries, strikes, maturity buckets and moneyness regions. These are methodological coverage measures. They describe the support of the calibration problem and the extent to which Heston and lifted Heston are being compared on the same evidence.

Forward-variance diagnostics are particularly important for lifted Heston because the model uses a forward-curve-like input to initialise variance structure. The proxy

$$
\xi_0(T)\approx \partial_T\left[T\sigma_{\mathrm{ATM}}^2(T)\right]
$$

is economically meaningful only if the ATM term structure is sufficiently smooth and positive after preprocessing. If the proxy is jagged, negative or driven by sparse maturities, lifted calibration may be unstable for reasons unrelated to the underlying model.

The example forward-variance curves in this section serve two purposes. They make the model initialisation tangible for the methodology chapter, and they provide a limitation check: the dissertation can state clearly that the forward curve is an ATM-IV-based proxy rather than a directly observed variance-swap curve.

Interpretation should remain modest. A smooth forward-variance proxy supports the implementation. It does not prove that the proxy is the true expected future variance. The notebook therefore reports forward-variance diagnostics as input-quality evidence, not as a separate forecasting result.


In [20]:

# Surface coverage and forward-variance evidence

def _choose_col(frame: pd.DataFrame, names: list[str]) -> str | None:
    return next((c for c in names if isinstance(frame, pd.DataFrame) and c in frame.columns), None)


def surface_coverage_table(surface: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    if surface is None or surface.empty:
        return pd.DataFrame([{"metric": "surface_rows", "value": 0, "notes": "surface_panel missing or empty"}])
    s = surface.copy()
    rows = [
        {"metric": "surface_rows", "value": len(s), "notes": "cleaned quote rows"},
        {"metric": "dates", "value": pd.to_datetime(s["date"], errors="coerce").dt.normalize().nunique() if "date" in s else np.nan, "notes": "distinct calibration dates"},
        {"metric": "expiries", "value": s["exdate"].nunique() if "exdate" in s else np.nan, "notes": "distinct option expiries"},
        {"metric": "strikes", "value": s["strike"].nunique() if "strike" in s else np.nan, "notes": "distinct strikes"},
    ]
    cp_col = _choose_col(s, ["cp_flag", "option_type", "call_put"])
    if cp_col:
        cp = s[cp_col].astype(str).str.upper().str[0].value_counts(normalize=True)
        rows += [
            {"metric": "call_share", "value": float(cp.get("C", np.nan)), "notes": f"from {cp_col}"},
            {"metric": "put_share", "value": float(cp.get("P", np.nan)), "notes": f"from {cp_col}"},
        ]
    mny = _choose_col(s, ["log_forward_moneyness", "log_moneyness", "abs_log_forward_moneyness"])
    if mny:
        vals = pd.to_numeric(s[mny], errors="coerce").abs()
        rows += [
            {"metric": "near_atm_rows_abs_log_moneyness_le_0_05", "value": int(vals.le(0.05).sum()), "notes": mny},
            {"metric": "hedging_panel_rows_abs_log_moneyness_le_config", "value": int(vals.le(cfg.market.hedging_max_abs_log_moneyness).sum()), "notes": mny},
        ]
    out = pd.DataFrame(rows)
    out.to_csv(table_path("surface_coverage_diagnostics.csv", cfg), index=False)
    return out


def maturity_coverage_table(surface: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    if surface is None or surface.empty or "maturity_bucket" not in surface.columns:
        return pd.DataFrame()
    s = surface.copy()
    s["date"] = pd.to_datetime(s.get("date"), errors="coerce").dt.normalize()
    out = s.groupby("maturity_bucket", dropna=False, as_index=False).agg(
        rows=("maturity_bucket", "size"),
        dates=("date", "nunique"),
        expiries=("exdate", "nunique") if "exdate" in s else ("maturity_bucket", "size"),
        median_T=("T", "median") if "T" in s else ("maturity_bucket", "size"),
        median_abs_log_moneyness=("log_forward_moneyness", lambda x: float(pd.to_numeric(x, errors="coerce").abs().median())) if "log_forward_moneyness" in s else ("maturity_bucket", "size"),
    )
    out.to_csv(table_path("surface_coverage_by_maturity_bucket.csv", cfg), index=False)
    return out


def forward_variance_diagnostics(curve: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    if curve is None or curve.empty:
        return pd.DataFrame([{"date": pd.NaT, "status": "missing", "notes": "forward_variance_curve unavailable"}])
    c = curve.copy()
    t_col = _choose_col(c, ["T", "maturity", "tau"])
    xi_col = _choose_col(c, ["forward_variance", "xi0", "xi", "variance"])
    if not t_col or not xi_col or "date" not in c.columns:
        return pd.DataFrame([{"date": pd.NaT, "status": "missing_columns", "notes": f"T={t_col}; xi={xi_col}"}])
    c["date"] = pd.to_datetime(c["date"], errors="coerce").dt.normalize()
    c[t_col] = pd.to_numeric(c[t_col], errors="coerce")
    c[xi_col] = pd.to_numeric(c[xi_col], errors="coerce")
    rows = []
    for d, g in c.dropna(subset=[t_col, xi_col]).sort_values(["date", t_col]).groupby("date"):
        xi = g[xi_col].to_numpy(float)
        roughness = float(np.nanmean(np.abs(np.diff(xi, n=2)))) if len(xi) >= 3 else np.nan
        rows.append({
            "date": d,
            "n_curve_points": int(len(g)),
            "min_forward_variance": float(np.nanmin(xi)),
            "median_forward_variance": float(np.nanmedian(xi)),
            "max_forward_variance": float(np.nanmax(xi)),
            "second_difference_smoothness": roughness,
            "positive_curve": bool(np.nanmin(xi) > 0),
        })
    out = pd.DataFrame(rows)
    out.to_csv(table_path("forward_variance_smoothness_positivity.csv", cfg), index=False)
    return out


def plot_forward_variance_examples(curve: pd.DataFrame, cfg: ExperimentConfig, n_dates: int = 4) -> Path | None:
    if not MATPLOTLIB_AVAILABLE or curve is None or curve.empty:
        return None
    t_col = _choose_col(curve, ["T", "maturity", "tau"])
    xi_col = _choose_col(curve, ["forward_variance", "xi0", "xi", "variance"])
    if not t_col or not xi_col or "date" not in curve.columns:
        return None
    c = curve.copy()
    c["date"] = pd.to_datetime(c["date"], errors="coerce").dt.normalize()
    dates = sorted(c["date"].dropna().unique())
    if not dates:
        return None
    pick = [dates[i] for i in np.linspace(0, len(dates) - 1, min(n_dates, len(dates))).astype(int)]
    fig, ax = plt.subplots(figsize=(7, 4))
    for d in pick:
        g = c.loc[c["date"].eq(d)].sort_values(t_col)
        ax.plot(pd.to_numeric(g[t_col], errors="coerce"), pd.to_numeric(g[xi_col], errors="coerce"), marker="o", label=pd.Timestamp(d).date())
    ax.set(title="Empirical forward-variance curve examples", xlabel="Maturity $T$ (years)", ylabel=r"Forward variance $\xi_0(T)$")
    ax.grid(True, alpha=0.3)
    ax.legend(title="date", fontsize=8)
    path = figure_path("forward_variance_curve_examples.png", cfg)
    save_and_show_figure(plt.gcf(), path, dpi=160)
    return path

surface_coverage_diagnostics = surface_coverage_table(surface_panel, CFG)
surface_coverage_by_maturity_bucket = maturity_coverage_table(surface_panel, CFG)
forward_variance_smoothness_positivity = forward_variance_diagnostics(forward_variance_curve, CFG)
forward_variance_curve_examples_path = plot_forward_variance_examples(forward_variance_curve, CFG)

display(surface_coverage_diagnostics)
display(surface_coverage_by_maturity_bucket.head())
display(forward_variance_smoothness_positivity.head())


<Figure size 910x520 with 1 Axes>

,metric,value,notes
0,surface_rows,133868,cleaned quote rows
1,dates,2869,distinct calibration dates
2,expiries,803,distinct option expiries
3,strikes,906,distinct strikes
4,near_atm_rows_abs_log_moneyness_le_0_05,78043,log_forward_moneyness
5,hedging_panel_rows_abs_log_moneyness_le_config,133868,log_forward_moneyness


,maturity_bucket,rows,dates,expiries,median_T,median_abs_log_moneyness
0,1M,23369,615,445,0.084873,0.025445
1,1W,3280,166,134,0.021903,0.014994
2,2M,3777,234,71,0.156057,0.037238
3,2W,4013,166,101,0.038330,0.020778
4,3M,14590,962,121,0.287474,0.046653


,date,n_curve_points,min_forward_variance,median_forward_variance,max_forward_variance,second_difference_smoothness,positive_curve
0,2010-01-04,2,0.041004,0.041007,0.041009,NaN,True
1,2010-01-05,2,0.049108,0.049249,0.049390,NaN,True
2,2010-01-06,3,0.044269,0.044582,0.044897,0.000001,True
3,2010-01-07,3,0.042361,0.042663,0.042966,0.000002,True
4,2010-01-08,3,0.040307,0.040593,0.040880,0.000002,True


## 3d. Panel Sufficiency and Data-Evidence Diagnostics

This section asks whether the final empirical panel is large and balanced enough to support the dissertation's claims. Panel sufficiency is not glamorous, which is probably why it is useful. A model comparison can be invalidated by lopsided coverage even when every plot looks professional.

The diagnostics summarise coverage by split, maturity bucket, regime label and moneyness. They also report date ranges and representative statistics such as median DTE and median absolute log-moneyness. The key question is whether train, validation and test data describe comparable market objects. If one policy is evaluated mostly on one maturity region while another is evaluated on another, the comparison becomes contaminated by panel composition.

The notebook uses deterministic date-based splits rather than random row-level splits. This matters because option rows from the same market date share the same volatility regime, spot level, calibration state and macro environment. Treating rows from the same date as independent evidence would inflate precision.

The interpretation rule is:

$$
\text{more paths per date} \rightarrow \text{lower Monte Carlo noise},
\qquad
\text{more dates} \rightarrow \text{more market-state evidence}.
$$

The dissertation should distinguish these two. A 10,000-path evaluation on twenty dates can be conditionally precise while still being limited in market-state diversity. This section makes that limitation explicit before the performance tables arrive looking authoritative.


In [21]:

# Panel sufficiency and data-evidence diagnostics

def _load_table_if_present(path: Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    return pd.read_parquet(path) if path.suffix.lower() == ".parquet" else pd.read_csv(path)


def _live_or_cached_frame(live_names: list[str], paths: list[Path]) -> pd.DataFrame:
    for name in live_names:
        obj = globals().get(name)
        if isinstance(obj, pd.DataFrame) and not obj.empty:
            return obj.copy()
    for path in paths:
        frame = _load_table_if_present(path)
        if not frame.empty:
            return frame
    return pd.DataFrame()


def _panel_summary_row(name: str, frame: pd.DataFrame) -> dict[str, Any]:
    if frame is None or frame.empty:
        return {"panel": name, "rows": 0, "n_dates": 0, "n_expiries": 0, "n_strikes": 0}
    f = frame.copy()
    date_col = _choose_col(f, ["date", "valuation_date"])
    expiry_col = _choose_col(f, ["exdate", "expiry", "expiration"])
    strike_col = _choose_col(f, ["strike", "K"])
    dte_col = _choose_col(f, ["dte", "days_to_expiry"])
    k_col = _choose_col(f, ["log_forward_moneyness", "k", "log_moneyness"])
    row = {
        "panel": name,
        "rows": int(len(f)),
        "n_dates": int(pd.to_datetime(f[date_col], errors="coerce").dt.normalize().nunique()) if date_col else 0,
        "n_expiries": int(pd.to_datetime(f[expiry_col], errors="coerce").dt.normalize().nunique()) if expiry_col else 0,
        "n_strikes": int(f[strike_col].nunique()) if strike_col else 0,
        "min_date": str(pd.to_datetime(f[date_col], errors="coerce").min().date()) if date_col and pd.to_datetime(f[date_col], errors="coerce").notna().any() else "",
        "max_date": str(pd.to_datetime(f[date_col], errors="coerce").max().date()) if date_col and pd.to_datetime(f[date_col], errors="coerce").notna().any() else "",
        "median_dte": float(pd.to_numeric(f[dte_col], errors="coerce").median()) if dte_col else np.nan,
        "median_abs_log_moneyness": float(pd.to_numeric(f[k_col], errors="coerce").abs().median()) if k_col else np.nan,
    }
    if "option_type" in f.columns:
        row["call_share"] = float(f["option_type"].astype(str).str.lower().str.startswith("c").mean())
    return row


def build_panel_sufficiency_diagnostics(cfg: ExperimentConfig) -> pd.DataFrame:
    frames = {
        "global_surface_panel": _live_or_cached_frame(["surface_panel", "global_diagnostic_panel"], [cfg.paths.surface_cache_path, SPX_HEDGING_OUTPUTS["global_diagnostic_panel"]]),
        "hedging_calibration_panel": _live_or_cached_frame(["spx_hedging_calibration_panel", "hedging_calibration_panel"], [SPX_HEDGING_OUTPUTS["hedging_calibration_panel"]]),
        "candidate_seed_panel": _live_or_cached_frame(["candidate_seed_panel"], [cfg.paths.candidate_seed_panel_path, cfg.paths.seed_candidate_universe_path]),
        "final_seed_panel": _live_or_cached_frame(["seed_panel", "spx_hedging_seed_panel"], [cfg.paths.seed_panel_path, cfg.paths.final_feasible_seed_panel_path, SPX_HEDGING_OUTPUTS["hedging_seed_panel"]]),
    }
    out = pd.DataFrame([_panel_summary_row(name, frame) for name, frame in frames.items()])
    out.to_csv(table_path("panel_sufficiency_diagnostics.csv", cfg), index=False)
    return out


def plot_panel_filter_waterfall(summary: pd.DataFrame, cfg: ExperimentConfig) -> Path | None:
    if not MATPLOTLIB_AVAILABLE or summary is None or summary.empty or "rows" not in summary.columns:
        return None
    p = summary.copy()
    p = p.loc[p["rows"].fillna(0).astype(float) > 0].copy()
    if p.empty:
        return None
    fig, ax = plt.subplots(figsize=(7, 4))
    colors = [plt.rcParams["axes.prop_cycle"].by_key().get("color", [MODEL_COLORS["heston"]])[i % len(plt.rcParams["axes.prop_cycle"].by_key().get("color", [MODEL_COLORS["heston"]]))] for i in range(len(p))]
    ax.bar(p["panel"].astype(str), pd.to_numeric(p["rows"], errors="coerce"), color=colors, alpha=BAR_ALPHA, edgecolor="white", linewidth=0.45)
    ax.set_yscale("log")
    ax.set(title="Panel construction evidence: rows retained by stage", xlabel="Panel stage", ylabel="Rows (log scale)")
    ax.tick_params(axis="x", rotation=30)
    ax.grid(True, axis="y", alpha=0.3)
    path = figure_path("panel_filter_waterfall_rows.png", cfg)
    save_and_show_figure(plt.gcf(), path, dpi=160)
    return path


def hedging_panel_bucket_coverage(cfg: ExperimentConfig) -> pd.DataFrame:
    panel = _live_or_cached_frame(["spx_hedging_calibration_panel", "hedging_calibration_panel"], [SPX_HEDGING_OUTPUTS["hedging_calibration_panel"]])
    if panel.empty:
        out = pd.DataFrame([{"status": "missing", "notes": "hedging calibration panel unavailable"}])
        out.to_csv(table_path("hedging_panel_bucket_coverage.csv", cfg), index=False)
        return out

    p = panel.copy()
    k_col = _choose_col(p, ["log_forward_moneyness", "k", "log_moneyness"])

    if "maturity_bucket" not in p.columns:
        dte_col = _choose_col(p, ["dte", "days_to_expiry"])
        if dte_col:
            bins = [0, 45, 90, 180, 365, np.inf]
            labels = ["<=45d", "46-90d", "91-180d", "181-365d", ">365d"]
            p["maturity_bucket"] = pd.cut(pd.to_numeric(p[dte_col], errors="coerce"), bins=bins, labels=labels)
        else:
            p["maturity_bucket"] = "all"

    if k_col:
        p["moneyness_bucket"] = pd.cut(
            pd.to_numeric(p[k_col], errors="coerce"),
            bins=[-np.inf, -0.10, -0.05, 0.05, 0.10, np.inf],
            labels=["left_wing", "left_near", "atm", "right_near", "right_wing"],
        )
    else:
        p["moneyness_bucket"] = "all"

    group_cols = ["maturity_bucket", "moneyness_bucket"]
    grouped = p.groupby(group_cols, dropna=False, observed=True)
    out = grouped.size().rename("n_rows").reset_index()

    if "date" in p.columns:
        date_counts = grouped["date"].apply(
            lambda x: pd.to_datetime(x, errors="coerce").dt.normalize().nunique()
        ).rename("n_dates").reset_index()
        out = out.merge(date_counts, on=group_cols, how="left")
    else:
        out["n_dates"] = out["n_rows"]

    out["maturity_bucket"] = out["maturity_bucket"].astype(str)
    out["moneyness_bucket"] = out["moneyness_bucket"].astype(str)
    out = out.sort_values(group_cols).reset_index(drop=True)
    out.to_csv(table_path("hedging_panel_bucket_coverage.csv", cfg), index=False)

    if MATPLOTLIB_AVAILABLE and not out.empty:
        heat = out.pivot_table(
            index="maturity_bucket",
            columns="moneyness_bucket",
            values="n_rows",
            aggfunc="sum",
            fill_value=0,
        )
        fig, ax = plt.subplots(figsize=(7, 4))
        plot_annotated_heatmap(
            heat,
            ax=ax,
            fig=fig,
            color_values=np.log1p(heat.astype(float)),
            text_values=heat.astype(float),
            cmap=COUNT_CMAP,
            title="Hedging-panel coverage by maturity and moneyness",
            xlabel="Moneyness bucket",
            ylabel="Maturity bucket",
            cbar_label="log(1 + rows)",
            fmt=".0f",
            rotation=30,
        )
        path = figure_path("hedging_panel_bucket_coverage_heatmap.png", cfg)
        save_and_show_figure(fig, path, dpi=260, description="Hedging-panel coverage by maturity and moneyness", notes="Cell labels show raw row counts; colour scale shows log(1 + rows).")

    return out


panel_sufficiency_diagnostics = build_panel_sufficiency_diagnostics(CFG)
panel_filter_waterfall_path = plot_panel_filter_waterfall(panel_sufficiency_diagnostics, CFG)
hedging_panel_bucket_coverage_table = hedging_panel_bucket_coverage(CFG)

display(panel_sufficiency_diagnostics)
display(hedging_panel_bucket_coverage_table.head())


<Figure size 910x520 with 1 Axes>

<Figure size 910x520 with 2 Axes>

,panel,rows,n_dates,n_expiries,n_strikes,min_date,max_date,median_dte,median_abs_log_moneyness
0,global_surface_panel,133868,2869,803,906,2010-01-04,2024-12-31,157.0,0.041567
1,hedging_calibration_panel,133868,2869,803,906,2010-01-04,2024-12-31,157.0,0.041567
2,candidate_seed_panel,845,725,364,308,2010-01-06,2024-12-19,33.0,0.001413
3,final_seed_panel,8451,2869,803,528,2010-01-04,2024-12-31,169.0,0.003037


,maturity_bucket,moneyness_bucket,n_rows,n_dates
0,1M,atm,18075,615
1,1M,left_near,3808,613
2,1M,right_near,1486,486
3,1W,atm,3035,166
4,1W,left_near,237,113


# 4. Calibration Engines

This section loads or constructs the calibrated Heston and lifted-Heston model states used downstream. Calibration is the bridge between market option data and simulated hedging environments. It determines the parameter vectors, variance states and model-implied volatilities used for diagnostics and path generation.

The methodological requirement is **paired fairness**. Heston and lifted Heston should be compared on identical dates and target rows wherever possible. If one model is evaluated on a different option subset, differences in RMSE may reflect data composition rather than model performance.

The calibration engine is not part of evaluation. In cached-evaluation mode, calibrated artefacts are frozen and loaded. Recalibration should occur only when explicitly requested. This prevents a reporting run from changing the empirical state of the notebook.

The calibration output has two roles:

1. **Static fit evidence:** model IVs, residuals, RMSEs, skew errors and parameter stability.
2. **State construction:** parameters and state variables used to simulate hedging paths and compute model-delta benchmarks.

The dissertation should phrase calibration results carefully. A good calibration fit makes a model more credible as a scenario generator, especially on the relevant hedging slice. It does not by itself show that the model produces better hedges.


## 4.1 Classical Heston

In [22]:
from scipy.optimize import least_squares


@dataclass(frozen=True)
class HestonParams:
    kappa: float
    theta: float
    xi: float
    rho: float
    v0: float

    def as_array(self) -> np.ndarray:
        return np.array([self.kappa, self.theta, self.xi, self.rho, self.v0], dtype=float)

    @classmethod
    def from_array(cls, x: Iterable[float]) -> "HestonParams":
        kappa, theta, xi, rho, v0 = [float(v) for v in x]
        return cls(kappa=kappa, theta=theta, xi=xi, rho=rho, v0=v0)


@dataclass(frozen=True)
class HestonMarketTarget:
    date: pd.Timestamp
    profile: str
    frame: pd.DataFrame
    S: np.ndarray
    K: np.ndarray
    T: np.ndarray
    r: np.ndarray
    q: np.ndarray
    market_prices: np.ndarray
    iv_proxy: np.ndarray
    vegas: np.ndarray
    weights: np.ndarray


@dataclass(frozen=True)
class HestonPanelConfig:
    """Part-A-style production panel controls, adapted from profile keys to full-surface date keys.

    backend controls which proven Part A path is used:
    - "cpu": canonical SciPy least_squares for every full-surface date target.
    - "gpu_if_available": batched Torch/GPU initialisation where CUDA exists, then CPU polish for every flagged date.
    - "gpu": require CUDA for the batched Part A path.
    """

    backend: Literal["cpu", "gpu_if_available", "gpu"] = "gpu_if_available"
    gpu_chunk_size: int = 128
    gpu_max_steps: int = 1000
    gpu_lr: float = 0.03
    cpu_polish_bad_fits: bool = True
    cpu_polish_validation_dates: int = 25
    cpu_polish_max_dates: int | None = None
    fit_vega_rmse_threshold: float = 0.50
    bad_vega_rmse_threshold: float = 2.00
    common_pricer_n_quad: int = 64


HESTON_PANEL_CFG = HestonPanelConfig(
    fit_vega_rmse_threshold=CFG.calibration.heston_quality_warn_rmse,
    bad_vega_rmse_threshold=CFG.calibration.heston_quality_bad_rmse,
    common_pricer_n_quad=CFG.calibration.heston_pricing_nodes,
)


class HestonPricer:
    """Validated Part A semi-analytic Heston call pricer using Gauss-Laguerre P1/P2 quadrature."""

    def __init__(self, n_quad: int = 64):
        self.nodes, self.weights = np.polynomial.laguerre.laggauss(n_quad)

    @staticmethod
    def _cf(
        u: np.ndarray | complex,
        S: float,
        T: float,
        r: float,
        q: float,
        params: HestonParams,
    ) -> np.ndarray:
        kappa = params.kappa
        theta = params.theta
        xi = params.xi
        rho = params.rho
        v0 = params.v0

        iu = 1j * u
        a = kappa * theta
        b = kappa

        d = np.sqrt((rho * xi * iu - b) ** 2 + xi ** 2 * (iu + u ** 2))
        g = (b - rho * xi * iu - d) / (b - rho * xi * iu + d)
        exp_neg_dT = np.exp(-d * T)

        C = (
            iu * (np.log(S) + (r - q) * T)
            + (a / xi ** 2)
            * ((b - rho * xi * iu - d) * T - 2.0 * np.log((1.0 - g * exp_neg_dT) / (1.0 - g)))
        )
        D = ((b - rho * xi * iu - d) / xi ** 2) * ((1.0 - exp_neg_dT) / (1.0 - g * exp_neg_dT))
        return np.exp(C + D * v0)

    def call_price_one(self, S: float, K: float, T: float, r: float, q: float, params: HestonParams) -> float:
        if S <= 0 or K <= 0 or T <= 0:
            return np.nan

        u = self.nodes
        w = self.weights
        log_K = np.log(K)

        with np.errstate(over="ignore", divide="ignore", invalid="ignore"):
            phi_minus_i = self._cf(-1j, S, T, r, q, params)
            if not np.all(np.isfinite(phi_minus_i)):
                return np.nan

            phi_1 = self._cf(u - 1j, S, T, r, q, params) / phi_minus_i
            phi_2 = self._cf(u, S, T, r, q, params)

            integrand_1 = np.real(np.exp(-1j * u * log_K) * phi_1 / (1j * u))
            integrand_2 = np.real(np.exp(-1j * u * log_K) * phi_2 / (1j * u))

            # Gauss-Laguerre integrates exp(-u)g(u), so multiply the original integrand by exp(u).
            integral_1 = np.sum(w * np.exp(u) * integrand_1)
            integral_2 = np.sum(w * np.exp(u) * integrand_2)

            P1 = 0.5 + integral_1 / np.pi
            P2 = 0.5 + integral_2 / np.pi

            price = S * np.exp(-q * T) * P1 - K * np.exp(-r * T) * P2

        if not np.isfinite(price):
            return np.nan
        return float(np.real(price))

    def call_probability_one(self, S: float, K: float, T: float, r: float, q: float, params: HestonParams) -> float:
        """Return Heston P1, so call delta is exp(-qT) P1."""
        if S <= 0 or K <= 0:
            raise ValueError("Heston probability requires positive S and K.")
        if T <= 1e-10:
            return 1.0 if S > K else (0.0 if S < K else 0.5)
        u, w = self.nodes, self.weights
        log_K = math.log(K)
        phi_minus_i = self._cf(-1j, S, T, r, q, params)
        if not np.all(np.isfinite(phi_minus_i)) or abs(phi_minus_i) < 1e-14:
            raise FloatingPointError("Invalid Heston P1 denominator.")
        phi_1 = self._cf(u - 1j, S, T, r, q, params) / phi_minus_i
        integrand_1 = np.real(np.exp(-1j * u * log_K) * phi_1 / (1j * u))
        P1 = 0.5 + float(np.sum(w * np.exp(u) * integrand_1) / np.pi)
        tol = 2e-3
        if not np.isfinite(P1) or P1 < -tol or P1 > 1.0 + tol:
            raise FloatingPointError(f"Heston P1 outside probability bounds: {P1}")
        return float(P1)

    def call_delta_one(self, S: float, K: float, T: float, r: float, q: float, params: HestonParams) -> float:
        P1 = self.call_probability_one(S, K, T, r, q, params)
        return float(math.exp(-q * max(T, 0.0)) * P1)

    def call_prices(self, S, K, T, r, q, params: HestonParams) -> np.ndarray:
        S = np.asarray(S, dtype=float)
        K = np.asarray(K, dtype=float)
        T = np.asarray(T, dtype=float)
        r = np.asarray(r, dtype=float)
        q = np.asarray(q, dtype=float)
        out = np.empty(len(K), dtype=float)
        for i in range(len(K)):
            out[i] = self.call_price_one(float(S[i]), float(K[i]), float(T[i]), float(r[i]), float(q[i]), params)
        return out


def prepare_surface_for_calibration(surface: pd.DataFrame, cfg: CalibrationConfig) -> pd.DataFrame:
    """Build the final full-surface target for one date.

    The dissertation methodology requires the Part-A full-surface weighting field
    ``fit_weight``. Older notebooks silently fell back to quote-quality weights or
    unit weights; that changes the calibration target and is therefore forbidden.
    """
    required = ["date", "spot_ref", "strike", "T", "r", "q", "call_equiv_mid", "bs_vega", "fit_weight"]
    missing = [c for c in required if c not in surface.columns]
    if missing:
        raise KeyError(f"Full-surface calibration target is missing required columns: {missing}")

    keep = [
        "date", "exdate", "maturity_bucket", "log_moneyness", "spot_ref", "strike", "T", "r", "q",
        "call_equiv_mid", "bs_vega", "iv_market", "impl_volatility", "fit_weight",
        "forward_ref", "dte",
    ]
    keep = [c for c in keep if c in surface.columns]
    out = surface.loc[:, keep].copy()
    out["target_weight"] = pd.to_numeric(out["fit_weight"], errors="coerce")

    if "iv_market" in out.columns:
        out["iv_proxy"] = pd.to_numeric(out["iv_market"], errors="coerce")
    elif "impl_volatility" in out.columns:
        out["iv_proxy"] = pd.to_numeric(out["impl_volatility"], errors="coerce")
    else:
        raise KeyError("Full-surface calibration requires iv_market or impl_volatility; no IV proxy is fabricated.")

    numeric_cols = ["spot_ref", "strike", "T", "r", "q", "call_equiv_mid", "bs_vega", "target_weight"]
    for col in numeric_cols + ["iv_proxy"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    out = out.dropna(subset=numeric_cols)
    valid = (
        np.isfinite(out["spot_ref"].to_numpy(float))
        & np.isfinite(out["strike"].to_numpy(float))
        & np.isfinite(out["T"].to_numpy(float))
        & np.isfinite(out["call_equiv_mid"].to_numpy(float))
        & np.isfinite(out["bs_vega"].to_numpy(float))
        & np.isfinite(out["target_weight"].to_numpy(float))
        & out["spot_ref"].to_numpy(float).__gt__(0.0)
        & out["strike"].to_numpy(float).__gt__(0.0)
        & out["T"].to_numpy(float).__gt__(0.0)
        & out["call_equiv_mid"].to_numpy(float).__gt__(0.0)
        & out["bs_vega"].to_numpy(float).__gt__(0.0)
        & out["target_weight"].to_numpy(float).__gt__(0.0)
    )
    out = out.loc[valid]

    if cfg.max_options_per_surface is not None and len(out) > cfg.max_options_per_surface:
        if "maturity_bucket" in out.columns and "log_moneyness" in out.columns:
            temp = out.copy()
            qbins = min(5, max(1, temp["log_moneyness"].nunique()))
            temp["_mny_bucket"] = pd.qcut(temp["log_moneyness"], q=qbins, duplicates="drop")
            n_groups = max(1, temp.groupby(["maturity_bucket", "_mny_bucket"], observed=False).ngroups)
            per_group = max(1, cfg.max_options_per_surface // n_groups)
            pieces = []
            for _, g in temp.groupby(["maturity_bucket", "_mny_bucket"], observed=False):
                pieces.append(g.sort_values("bs_vega", ascending=False).head(per_group))
            out = pd.concat(pieces, ignore_index=True).head(cfg.max_options_per_surface)
            out = out.drop(columns=["_mny_bucket"], errors="ignore")
        else:
            out = out.sort_values("bs_vega", ascending=False).head(cfg.max_options_per_surface)

    return out.sort_values([c for c in ["exdate", "strike"] if c in out.columns]).reset_index(drop=True)


def make_heston_market_target(surface_date: pd.DataFrame, cfg: CalibrationConfig, profile: str = "full_surface") -> HestonMarketTarget:
    frame = prepare_surface_for_calibration(surface_date, cfg)
    if frame.empty:
        raise ValueError("Cannot build Heston target from an empty filtered surface.")

    S = frame["spot_ref"].to_numpy(dtype=float)
    K = frame["strike"].to_numpy(dtype=float)
    T = frame["T"].to_numpy(dtype=float)
    r = frame["r"].to_numpy(dtype=float)
    q = frame["q"].to_numpy(dtype=float)
    market_prices = frame["call_equiv_mid"].to_numpy(dtype=float)
    vegas = np.maximum(frame["bs_vega"].to_numpy(dtype=float), cfg.vega_floor)
    weights = frame["target_weight"].to_numpy(dtype=float)

    iv_proxy = frame["iv_proxy"].to_numpy(dtype=float) if "iv_proxy" in frame.columns else np.full(len(frame), np.nan)
    if not np.isfinite(iv_proxy).all():
        raise ValueError("Full-surface calibration target contains non-finite implied volatility values.")

    return HestonMarketTarget(
        date=pd.Timestamp(frame["date"].iloc[0]),
        profile=profile,
        frame=frame,
        S=S,
        K=K,
        T=T,
        r=r,
        q=q,
        market_prices=market_prices,
        iv_proxy=iv_proxy,
        vegas=vegas,
        weights=weights,
    )


class HestonCalibrator:
    """Part A CPU canonical Heston calibrator using SciPy least_squares on residual vectors."""

    def __init__(self, cfg: CalibrationConfig, pricer: HestonPricer | None = None):
        self.cfg = cfg
        self.pricer = pricer if pricer is not None else HestonPricer(n_quad=cfg.heston_pricing_nodes)

    @staticmethod
    def _params_from_array(x: np.ndarray) -> HestonParams:
        return HestonParams.from_array(x)

    def _initial_guess(self, target: HestonMarketTarget, x0: np.ndarray | None = None) -> np.ndarray:
        if x0 is not None and np.all(np.isfinite(x0)):
            return np.asarray(x0, dtype=float)
        finite_iv = target.iv_proxy[np.isfinite(target.iv_proxy)]
        atm_var = float(np.nanmedian(finite_iv) ** 2) if len(finite_iv) else self.cfg.heston_initial["theta"]
        return np.array([
            self.cfg.heston_initial["kappa"],
            max(atm_var, self.cfg.heston_bounds["theta"][0]),
            self.cfg.heston_initial["xi"],
            self.cfg.heston_initial["rho"],
            max(atm_var, self.cfg.heston_bounds["v0"][0]),
        ], dtype=float)

    def _bounds(self) -> tuple[np.ndarray, np.ndarray]:
        lower = np.array([self.cfg.heston_bounds[k][0] for k in ["kappa", "theta", "xi", "rho", "v0"]], dtype=float)
        upper = np.array([self.cfg.heston_bounds[k][1] for k in ["kappa", "theta", "xi", "rho", "v0"]], dtype=float)
        return lower, upper

    def _residuals(self, x: np.ndarray, target: HestonMarketTarget) -> np.ndarray:
        params = self._params_from_array(x)
        model_prices = self.pricer.call_prices(target.S, target.K, target.T, target.r, target.q, params)
        if not np.all(np.isfinite(model_prices)):
            return np.full_like(target.market_prices, 1e6)
        residuals = (model_prices - target.market_prices) / np.maximum(np.abs(target.vegas), self.cfg.vega_floor)
        residuals = np.sqrt(target.weights) * residuals
        residuals[~np.isfinite(residuals)] = 1e6
        return residuals

    def calibrate(self, target: HestonMarketTarget, x0: np.ndarray | None = None) -> dict[str, Any]:
        lower, upper = self._bounds()
        x0_final = np.clip(self._initial_guess(target, x0=x0), lower + 1e-10, upper - 1e-10)

        result = least_squares(
            fun=lambda x: self._residuals(x, target),
            x0=x0_final,
            bounds=(lower, upper),
            max_nfev=self.cfg.optimizer_maxiter,
        )

        params = self._params_from_array(result.x)
        model_prices = self.pricer.call_prices(target.S, target.K, target.T, target.r, target.q, params)
        price_error = model_prices - target.market_prices
        vega_scaled_error = price_error / np.maximum(np.abs(target.vegas), self.cfg.vega_floor)
        weighted_vega_mse = float(np.average(vega_scaled_error ** 2, weights=target.weights))
        vega_scaled_rmse = float(np.sqrt(weighted_vega_mse))
        price_rmse = float(np.sqrt(np.average(price_error ** 2, weights=target.weights)))
        price_mae = float(np.average(np.abs(price_error), weights=target.weights))

        return {
            "date": target.date,
            "profile": target.profile,
            "success": bool(result.success),
            "message": str(result.message),
            "optimizer_cost": float(result.cost),
            "loss": weighted_vega_mse,
            "kappa": params.kappa,
            "theta": params.theta,
            "xi": params.xi,
            "rho": params.rho,
            "v0": params.v0,
            "n_options": int(len(target.K)),
            "n_points": int(len(target.K)),
            "price_rmse": price_rmse,
            "price_mae": price_mae,
            "vega_scaled_rmse": vega_scaled_rmse,
            "optimizer_nfev": int(result.nfev),
            "optimizer_nit": int(getattr(result, "nit", -1) if getattr(result, "nit", None) is not None else -1),
            "feller_margin": float(2.0 * params.kappa * params.theta - params.xi ** 2),
            "pricing_engine": "partA_cpu_gauss_laguerre_p1p2",
        }


@dataclass(frozen=True)
class BatchedHestonTargets:
    keys: pd.DataFrame
    S: torch.Tensor
    K: torch.Tensor
    T: torch.Tensor
    r: torch.Tensor
    q: torch.Tensor
    market_prices: torch.Tensor
    vegas: torch.Tensor
    weights: torch.Tensor
    mask: torch.Tensor
    iv_proxy: torch.Tensor
    device: torch.device


@dataclass(frozen=True)
class BatchedHestonGPUResult:
    params: pd.DataFrame
    fit_summary: pd.DataFrame
    history: pd.DataFrame


@dataclass(frozen=True)
class HestonPanelGPUOutput:
    params: pd.DataFrame
    fit_summary: pd.DataFrame
    history: pd.DataFrame
    timing: pd.DataFrame


HESTON_LOWER = torch.tensor([CFG.calibration.heston_bounds[k][0] for k in ["kappa", "theta", "xi", "rho", "v0"]], dtype=torch.float64) if TORCH_AVAILABLE else None
HESTON_UPPER = torch.tensor([CFG.calibration.heston_bounds[k][1] for k in ["kappa", "theta", "xi", "rho", "v0"]], dtype=torch.float64) if TORCH_AVAILABLE else None


def bounded_from_raw(raw: torch.Tensor, lower: torch.Tensor, upper: torch.Tensor) -> torch.Tensor:
    return lower + (upper - lower) * torch.sigmoid(raw)


def raw_from_bounded(params: torch.Tensor, lower: torch.Tensor, upper: torch.Tensor) -> torch.Tensor:
    scaled = (params - lower) / torch.clamp(upper - lower, min=1e-12)
    scaled = torch.clamp(scaled, 1e-6, 1.0 - 1e-6)
    return torch.log(scaled / (1.0 - scaled))


class BatchedTorchHestonPricer:
    """Part A fully batched Torch Heston pricer for GPU panel calibration."""

    def __init__(self, n_quad: int = 64, device: str = "cuda", dtype: torch.dtype = torch.float64):
        self.device = torch.device(device)
        self.real_dtype = dtype
        self.complex_dtype = torch.complex128 if dtype == torch.float64 else torch.complex64
        nodes, weights = np.polynomial.laguerre.laggauss(n_quad)
        self.u = torch.tensor(nodes, dtype=self.real_dtype, device=self.device)
        self.w = torch.tensor(weights, dtype=self.real_dtype, device=self.device)

    def _cf(self, u: torch.Tensor, S: torch.Tensor, T: torch.Tensor, r: torch.Tensor, q: torch.Tensor, params: torch.Tensor) -> torch.Tensor:
        kappa = params[:, 0].reshape(-1, 1, 1)
        theta = params[:, 1].reshape(-1, 1, 1)
        xi = params[:, 2].reshape(-1, 1, 1)
        rho = params[:, 3].reshape(-1, 1, 1)
        v0 = params[:, 4].reshape(-1, 1, 1)

        u = u.to(self.complex_dtype)
        S = S.to(self.complex_dtype)
        T = T.to(self.complex_dtype)
        r = r.to(self.complex_dtype)
        q = q.to(self.complex_dtype)
        kappa = kappa.to(self.complex_dtype)
        theta = theta.to(self.complex_dtype)
        xi = xi.to(self.complex_dtype)
        rho = rho.to(self.complex_dtype)
        v0 = v0.to(self.complex_dtype)

        iu = 1j * u
        d = torch.sqrt((rho * xi * iu - kappa) ** 2 + xi ** 2 * (iu + u ** 2))
        g = (kappa - rho * xi * iu - d) / (kappa - rho * xi * iu + d)
        exp_neg_dT = torch.exp(-d * T)
        C = (
            iu * (torch.log(S) + (r - q) * T)
            + (kappa * theta / xi ** 2)
            * ((kappa - rho * xi * iu - d) * T - 2.0 * torch.log((1.0 - g * exp_neg_dT) / (1.0 - g)))
        )
        D = ((kappa - rho * xi * iu - d) / xi ** 2) * ((1.0 - exp_neg_dT) / (1.0 - g * exp_neg_dT))
        return torch.exp(C + D * v0)

    def call_prices(self, batch: BatchedHestonTargets, params: torch.Tensor) -> torch.Tensor:
        S = torch.clamp(batch.S.unsqueeze(-1), min=1e-12)
        K = torch.clamp(batch.K.unsqueeze(-1), min=1e-12)
        T = torch.clamp(batch.T.unsqueeze(-1), min=1e-12)
        r = batch.r.unsqueeze(-1)
        q = batch.q.unsqueeze(-1)
        u = self.u.reshape(1, 1, -1)
        w = self.w.reshape(1, 1, -1)
        u_complex = u.to(self.complex_dtype)
        log_K = torch.log(K).to(self.complex_dtype)
        minus_i = torch.tensor(-1j, dtype=self.complex_dtype, device=self.device)
        phi_minus_i = self._cf(minus_i, S, T, r, q, params)
        phi_1 = self._cf(u_complex - 1j, S, T, r, q, params) / phi_minus_i
        phi_2 = self._cf(u_complex, S, T, r, q, params)
        integrand_1 = torch.real(torch.exp(-1j * u_complex * log_K) * phi_1 / (1j * u_complex))
        integrand_2 = torch.real(torch.exp(-1j * u_complex * log_K) * phi_2 / (1j * u_complex))
        integral_1 = torch.sum(w * torch.exp(u) * integrand_1, dim=-1)
        integral_2 = torch.sum(w * torch.exp(u) * integrand_2, dim=-1)
        P1 = 0.5 + integral_1 / np.pi
        P2 = 0.5 + integral_2 / np.pi
        price = batch.S * torch.exp(-batch.q * batch.T) * P1 - batch.K * torch.exp(-batch.r * batch.T) * P2
        return price * batch.mask


class BatchedHestonGPUOptimizer:
    """Part A batched GPU optimiser using bounded parameter transform and Adam."""

    def __init__(self, pricer: BatchedTorchHestonPricer, max_steps: int = 1000, lr: float = 0.03, vega_floor: float = 1.0):
        self.pricer = pricer
        self.max_steps = max_steps
        self.lr = lr
        self.vega_floor = vega_floor

    def _loss_per_target(self, batch: BatchedHestonTargets, params: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        model_prices = self.pricer.call_prices(batch, params)
        vega_scale = torch.clamp(torch.abs(batch.vegas), min=self.vega_floor)
        residuals = (model_prices - batch.market_prices) / vega_scale
        weighted = torch.sqrt(batch.weights) * residuals * batch.mask
        loss_per_target = torch.sum(weighted ** 2, dim=1) / torch.clamp(torch.sum(batch.mask, dim=1), min=1.0)
        return loss_per_target, model_prices, residuals

    def fit(self, batch: BatchedHestonTargets, initial_params: torch.Tensor) -> BatchedHestonGPUResult:
        device = batch.device
        lower = HESTON_LOWER.to(device)
        upper = HESTON_UPPER.to(device)
        raw = torch.nn.Parameter(raw_from_bounded(initial_params, lower, upper).clone())
        optimiser = torch.optim.Adam([raw], lr=self.lr)
        history_rows = []

        for step in range(self.max_steps):
            optimiser.zero_grad()
            params = bounded_from_raw(raw, lower, upper)
            loss_per_target, _, _ = self._loss_per_target(batch, params)
            loss = torch.mean(loss_per_target)
            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite batched Heston loss.")
            loss.backward()
            optimiser.step()
            if step % 50 == 0 or step == self.max_steps - 1:
                history_rows.append({
                    "step": step,
                    "mean_loss": float(loss.detach().cpu()),
                    "median_loss": float(torch.median(loss_per_target).detach().cpu()),
                    "max_loss": float(torch.max(loss_per_target).detach().cpu()),
                })

        with torch.no_grad():
            final_params = bounded_from_raw(raw, lower, upper)
            loss_per_target, model_prices, residuals = self._loss_per_target(batch, final_params)

        params_np = final_params.detach().cpu().numpy()
        params_df = batch.keys.copy()
        params_df["kappa"] = params_np[:, 0]
        params_df["theta"] = params_np[:, 1]
        params_df["xi"] = params_np[:, 2]
        params_df["rho"] = params_np[:, 3]
        params_df["v0"] = params_np[:, 4]
        params_df["loss"] = loss_per_target.detach().cpu().numpy()
        params_df["feller_margin"] = 2.0 * params_df["kappa"] * params_df["theta"] - params_df["xi"] ** 2

        model_np = model_prices.detach().cpu().numpy()
        residual_np = residuals.detach().cpu().numpy()
        market_np = batch.market_prices.detach().cpu().numpy()
        mask_np = batch.mask.detach().cpu().numpy()
        fit_rows = []
        for b, key in batch.keys.iterrows():
            valid = mask_np[b] > 0
            price_error = model_np[b, valid] - market_np[b, valid]
            vega_error = residual_np[b, valid]
            fit_rows.append({
                "date": pd.Timestamp(key["date"]),
                "profile": str(key["profile"]),
                "n_points": int(valid.sum()),
                "n_options": int(valid.sum()),
                "price_rmse": float(np.sqrt(np.nanmean(price_error ** 2))),
                "price_mae": float(np.nanmean(np.abs(price_error))),
                "vega_scaled_rmse": float(np.sqrt(np.nanmean(vega_error ** 2))),
                "max_abs_price_error": float(np.nanmax(np.abs(price_error))),
            })
        return BatchedHestonGPUResult(params=params_df, fit_summary=pd.DataFrame(fit_rows), history=pd.DataFrame(history_rows))


def _date_keys(surface: pd.DataFrame) -> list[pd.Timestamp]:
    return [pd.Timestamp(d).normalize() for d in sorted(pd.to_datetime(surface["date"]).dt.normalize().unique())]


def _chunk_list(xs: list[Any], n: int) -> list[list[Any]]:
    return [xs[i:i + n] for i in range(0, len(xs), n)]


def make_batched_heston_targets(surface: pd.DataFrame, dates: list[pd.Timestamp], cfg: CalibrationConfig, device: str) -> BatchedHestonTargets:
    frames = []
    key_rows = []
    for date in dates:
        g = surface[pd.to_datetime(surface["date"]).dt.normalize().eq(pd.Timestamp(date).normalize())]
        target = make_heston_market_target(g, cfg, profile="hedging_focused_near_atm")
        frames.append(target)
        key_rows.append({"date": pd.Timestamp(date).normalize(), "profile": "hedging_focused_near_atm"})

    max_n = max(len(t.K) for t in frames)
    B = len(frames)
    arrays = {}
    for name in ["S", "K", "T", "r", "q", "market_prices", "vegas", "weights", "iv_proxy"]:
        arr = np.zeros((B, max_n), dtype=np.float64)
        for b, t in enumerate(frames):
            values = getattr(t, name)
            arr[b, :len(values)] = values
        arrays[name] = arr
    mask = np.zeros((B, max_n), dtype=np.float64)
    for b, t in enumerate(frames):
        mask[b, :len(t.K)] = 1.0

    dev = torch.device(device)
    return BatchedHestonTargets(
        keys=pd.DataFrame(key_rows),
        S=torch.tensor(arrays["S"], dtype=torch.float64, device=dev),
        K=torch.tensor(arrays["K"], dtype=torch.float64, device=dev),
        T=torch.tensor(arrays["T"], dtype=torch.float64, device=dev),
        r=torch.tensor(arrays["r"], dtype=torch.float64, device=dev),
        q=torch.tensor(arrays["q"], dtype=torch.float64, device=dev),
        market_prices=torch.tensor(arrays["market_prices"], dtype=torch.float64, device=dev),
        vegas=torch.tensor(arrays["vegas"], dtype=torch.float64, device=dev),
        weights=torch.tensor(arrays["weights"], dtype=torch.float64, device=dev),
        mask=torch.tensor(mask, dtype=torch.float64, device=dev),
        iv_proxy=torch.tensor(arrays["iv_proxy"], dtype=torch.float64, device=dev),
        device=dev,
    )


def initial_heston_params_for_batch(batch: BatchedHestonTargets, cfg: CalibrationConfig) -> torch.Tensor:
    params = []
    lower = np.array([cfg.heston_bounds[k][0] for k in ["kappa", "theta", "xi", "rho", "v0"]], dtype=float)
    upper = np.array([cfg.heston_bounds[k][1] for k in ["kappa", "theta", "xi", "rho", "v0"]], dtype=float)
    iv_np = batch.iv_proxy.detach().cpu().numpy()
    mask_np = batch.mask.detach().cpu().numpy() > 0
    for b in range(iv_np.shape[0]):
        vals = iv_np[b, mask_np[b]]
        vals = vals[np.isfinite(vals)]
        atm_var = float(np.nanmedian(vals) ** 2) if len(vals) else cfg.heston_initial["theta"]
        x = np.array([
            cfg.heston_initial["kappa"],
            max(atm_var, cfg.heston_bounds["theta"][0]),
            cfg.heston_initial["xi"],
            cfg.heston_initial["rho"],
            max(atm_var, cfg.heston_bounds["v0"][0]),
        ], dtype=float)
        params.append(np.clip(x, lower + 1e-10, upper - 1e-10))
    return torch.tensor(np.vstack(params), dtype=torch.float64, device=batch.device)


def calibrate_heston_gpu_batch(surface: pd.DataFrame, dates: list[pd.Timestamp], cfg: ExperimentConfig, panel_cfg: HestonPanelConfig, device: str) -> tuple[BatchedHestonGPUResult, float]:
    batch = make_batched_heston_targets(surface, dates, cfg.calibration, device=device)
    initial = initial_heston_params_for_batch(batch, cfg.calibration)
    pricer = BatchedTorchHestonPricer(n_quad=panel_cfg.common_pricer_n_quad, device=device)
    optimiser = BatchedHestonGPUOptimizer(pricer=pricer, max_steps=panel_cfg.gpu_max_steps, lr=panel_cfg.gpu_lr, vega_floor=cfg.calibration.vega_floor)
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    result = optimiser.fit(batch, initial)
    if device == "cuda":
        torch.cuda.synchronize()
    return result, time.perf_counter() - t0


def combine_heston_params_and_fit(params: pd.DataFrame, fit: pd.DataFrame) -> pd.DataFrame:
    out = params.merge(fit, on=["date", "profile"], how="left", suffixes=("", "_fit"))
    out["date"] = pd.to_datetime(out["date"]).dt.normalize()
    return out


def add_heston_quality_flags(
    panel: pd.DataFrame,
    panel_cfg: HestonPanelConfig,
    calibration_cfg: CalibrationConfig,
) -> pd.DataFrame:
    out = panel.copy()
    required = ["kappa", "theta", "xi", "rho", "v0", "vega_scaled_rmse"]
    out["finite_params"] = np.isfinite(out[required]).all(axis=1)
    out["fit_ok"] = out["finite_params"] & out["vega_scaled_rmse"].lt(panel_cfg.fit_vega_rmse_threshold)
    bounds = {
        name: calibration_cfg.heston_bounds[name]
        for name in ["kappa", "theta", "xi", "rho", "v0"]
    }
    out["bound_hit_count"] = 0
    for col, (lo, hi) in bounds.items():
        out["bound_hit_count"] += np.isclose(out[col], lo, rtol=1e-4, atol=1e-6).astype(int)
        out["bound_hit_count"] += np.isclose(out[col], hi, rtol=1e-4, atol=1e-6).astype(int)
    out["feller_pass"] = out["feller_margin"] > 0
    out["quality_flag"] = np.where(
        out["vega_scaled_rmse"].gt(panel_cfg.bad_vega_rmse_threshold),
        "BADFIT",
        np.where(out["fit_ok"], "OK", "WARN"),
    )
    out["needs_cpu_polish"] = (~out["fit_ok"]) | out["bound_hit_count"].gt(0) | (~out["finite_params"])
    return out


def run_heston_gpu_panel(surface: pd.DataFrame, dates: list[pd.Timestamp], cfg: ExperimentConfig, panel_cfg: HestonPanelConfig, device: str) -> HestonPanelGPUOutput:
    chunks = _chunk_list(dates, panel_cfg.gpu_chunk_size)
    params_rows, fit_rows, history_rows, timing_rows = [], [], [], []
    for chunk_id, chunk_dates in enumerate(chunks, start=1):
        result, elapsed = calibrate_heston_gpu_batch(surface, chunk_dates, cfg, panel_cfg, device=device)
        params = result.params.copy(); params["chunk_id"] = chunk_id
        fit = result.fit_summary.copy(); fit["chunk_id"] = chunk_id
        hist = result.history.copy(); hist["chunk_id"] = chunk_id
        params_rows.append(params); fit_rows.append(fit); history_rows.append(hist)
        timing_rows.append({"chunk_id": chunk_id, "targets": len(chunk_dates), "seconds": elapsed, "seconds_per_target": elapsed / max(1, len(chunk_dates))})
        print(f"GPU Heston chunk {chunk_id}/{len(chunks)}: dates={len(chunk_dates)}, seconds={elapsed:.1f}, seconds/date={elapsed/max(1, len(chunk_dates)):.3f}")
    return HestonPanelGPUOutput(
        params=pd.concat(params_rows, ignore_index=True) if params_rows else pd.DataFrame(),
        fit_summary=pd.concat(fit_rows, ignore_index=True) if fit_rows else pd.DataFrame(),
        history=pd.concat(history_rows, ignore_index=True) if history_rows else pd.DataFrame(),
        timing=pd.DataFrame(timing_rows),
    )


def heston_date_part_path(cfg: ExperimentConfig, date: pd.Timestamp) -> Path:
    date_key = pd.Timestamp(date).normalize().strftime("%Y%m%d")
    return cfg.paths.heston_calibration_parts_dir / f"heston_full_surface_{date_key}.parquet"



def load_heston_date_part(cfg: ExperimentConfig, date: pd.Timestamp) -> pd.DataFrame | None:
    path = heston_date_part_path(cfg, date)
    if not cfg.calibration.heston_resume_date_parts:
        return None
    try:
        return load_dataframe_cache(
            path,
            f"heston_date_part_{pd.Timestamp(date):%Y%m%d}",
            "heston",
            cfg,
            required_columns={"date", "kappa", "theta", "xi", "rho", "v0", "success", "quality_flag"},
            dependencies=[cfg.paths.surface_cache_path],
        )
    except (FileNotFoundError, ValueError):
        return None


def save_heston_date_part(row: dict[str, Any] | pd.Series | pd.DataFrame, cfg: ExperimentConfig, date: pd.Timestamp) -> None:
    path = heston_date_part_path(cfg, date)
    if isinstance(row, pd.DataFrame):
        frame = row.copy()
    elif isinstance(row, pd.Series):
        frame = row.to_frame().T
    else:
        frame = pd.DataFrame([row])
    write_dataframe_cache(
        frame,
        path,
        f"heston_date_part_{pd.Timestamp(date):%Y%m%d}",
        "heston",
        cfg,
        dependencies=[cfg.paths.surface_cache_path],
    )

def calibrate_heston_cpu_dates(
    surface: pd.DataFrame,
    dates: list[pd.Timestamp],
    cfg: ExperimentConfig,
    panel_cfg: HestonPanelConfig = HESTON_PANEL_CFG,
    x0_lookup: dict[pd.Timestamp, np.ndarray] | None = None,
) -> tuple[pd.DataFrame, float]:
    """Canonical Part A CPU calibration for full-surface date targets, with reusable date checkpoints."""
    calibrator = HestonCalibrator(cfg=cfg.calibration, pricer=HestonPricer(n_quad=cfg.calibration.heston_pricing_nodes))
    rows = []
    t0 = time.perf_counter()
    prev_x = None
    cfg.paths.heston_calibration_parts_dir.mkdir(parents=True, exist_ok=True)

    for i, date in enumerate(dates, start=1):
        date_norm = pd.Timestamp(date).normalize()
        cached = load_heston_date_part(cfg, date_norm)
        if cached is not None and not cached.empty:
            row = cached.iloc[0].to_dict()
            row["date"] = pd.Timestamp(row["date"]).normalize()
            rows.append(row)
            prev_x = np.array([row["kappa"], row["theta"], row["xi"], row["rho"], row["v0"]], dtype=float)
            print(f"CPU Part A {i}/{len(dates)} {date_norm.date()}: loaded date checkpoint")
            continue

        g = surface[pd.to_datetime(surface["date"]).dt.normalize().eq(date_norm)]
        target = make_heston_market_target(g, cfg.calibration, profile="hedging_focused_near_atm")
        x0 = None
        if x0_lookup is not None and date_norm in x0_lookup:
            x0 = x0_lookup[date_norm]
        elif cfg.calibration.heston_warm_start_previous_date:
            x0 = prev_x

        t_date = time.perf_counter()
        row = calibrator.calibrate(target, x0=x0)
        row["backend"] = "cpu_scipy_least_squares"
        row["calibration_mode"] = "hedging_focused_near_atm_spx_cpu"
        row["cpu_seconds"] = time.perf_counter() - t_date
        row = add_heston_quality_flags(pd.DataFrame([row]), panel_cfg, cfg.calibration).iloc[0].to_dict()
        rows.append(row)
        save_heston_date_part(row, cfg, date_norm)
        prev_x = np.array([row["kappa"], row["theta"], row["xi"], row["rho"], row["v0"]], dtype=float)
        print(
            f"CPU Part A {i}/{len(dates)} {date_norm.date()}: "
            f"success={row['success']}, loss={row['loss']:.6g}, "
            f"vega_rmse={row['vega_scaled_rmse']:.4g}, n={row['n_options']}, "
            f"nfev={row['optimizer_nfev']}, seconds={row['cpu_seconds']:.1f}"
        )

    return pd.DataFrame(rows), time.perf_counter() - t0

def select_cpu_polish_dates(panel: pd.DataFrame, panel_cfg: HestonPanelConfig) -> list[pd.Timestamp]:
    bad = panel.loc[panel["needs_cpu_polish"], ["date"]].copy()
    validation = panel.loc[~panel["needs_cpu_polish"], ["date"]].sort_values("date").head(panel_cfg.cpu_polish_validation_dates)
    selected = pd.concat([bad, validation], ignore_index=True).drop_duplicates("date").sort_values("date")
    if panel_cfg.cpu_polish_max_dates is not None:
        selected = selected.head(panel_cfg.cpu_polish_max_dates)
    return [pd.Timestamp(d).normalize() for d in selected["date"]]


def build_final_heston_panel(gpu_panel: pd.DataFrame, cpu_polish: pd.DataFrame) -> pd.DataFrame:
    if gpu_panel.empty:
        return cpu_polish.sort_values("date").reset_index(drop=True)
    if cpu_polish.empty:
        return gpu_panel.sort_values("date").reset_index(drop=True)
    key_cols = ["date", "profile"]
    polish_keys = cpu_polish[key_cols].drop_duplicates()
    gpu_keep = gpu_panel.merge(polish_keys.assign(_has_cpu=True), on=key_cols, how="left")
    gpu_keep = gpu_keep[gpu_keep["_has_cpu"].isna()].drop(columns=["_has_cpu"])
    cols = sorted(set(gpu_keep.columns).union(cpu_polish.columns))
    return pd.concat([gpu_keep.reindex(columns=cols), cpu_polish.reindex(columns=cols)], ignore_index=True).sort_values("date").reset_index(drop=True)


def heston_arch_cache_path(cfg: ExperimentConfig) -> Path:
    # Use the standard path config so downstream sections and cache loading agree.
    return cfg.paths.heston_calibration_path


def load_or_run_heston_calibration(surface: pd.DataFrame, cfg: ExperimentConfig, panel_cfg: HestonPanelConfig = HESTON_PANEL_CFG) -> pd.DataFrame:
    """Load a frozen Heston calibration cache, or rebuild only when explicitly requested."""
    cache_path = heston_arch_cache_path(cfg)
    deps = [globals().get("SPX_HEDGING_CALIBRATION_PANEL_PATH", cfg.paths.surface_cache_path)]
    artifact_name = "heston_hedging_panel_calibration" if globals().get("HEADLINE_EMPIRICAL_MODE") == "HEDGING_FOCUSED_PANEL" else "heston_full_surface_calibration"
    required = {"date", "profile", "kappa", "theta", "xi", "rho", "v0"}
    if cfg.flags.FREEZE_CALIBRATION_ARTIFACTS and cfg.flags.RUN_HESTON_CALIBRATION:
        raise RuntimeError(
            "Internal execution-plan contradiction for Heston calibration. "
            "Restart the kernel and configure only RUN.rebuild_from in the user configuration cell."
        )
    if not cfg.flags.RUN_HESTON_CALIBRATION:
        try:
            loaded = load_dataframe_cache(cache_path, artifact_name, "heston", cfg, required_columns=required, dependencies=deps)
            loaded["date"] = pd.to_datetime(loaded["date"]).dt.normalize()
            print(f"Loading validated Heston hedging-panel calibration cache: {cache_path}")
            return loaded
        except (FileNotFoundError, ValueError) as exc:
            if cfg.flags.FREEZE_CALIBRATION_ARTIFACTS or not cfg.flags.ALLOW_CALIBRATION_REBUILD_FROM_DOWNSTREAM:
                raise RuntimeError(
                    "Heston calibration cache is missing/stale and strict reuse forbids rebuilding. "
                    "Set rebuild_from='calibration', an earlier stage, or use 'auto'."
                ) from exc
            print(f"[heston calibration] cache unavailable/stale; rebuilding. Reason: {exc}", flush=True)

    dates = _date_keys(surface)
    if not dates:
        raise ValueError("No dates available for Heston calibration.")
    prepare_checkpoint_directory(cfg.paths.heston_calibration_parts_dir, "heston_date_parts", "heston", cfg, deps)
    use_cuda = TORCH_AVAILABLE and torch.cuda.is_available()
    if panel_cfg.backend == "gpu" and not use_cuda:
        raise RuntimeError("HestonPanelConfig.backend='gpu' but CUDA is not available.")
    use_gpu = panel_cfg.backend in {"gpu", "gpu_if_available"} and use_cuda
    if use_gpu:
        gpu_output = run_heston_gpu_panel(surface, dates, cfg, panel_cfg, device="cuda")
        gpu_panel = combine_heston_params_and_fit(gpu_output.params, gpu_output.fit_summary)
        gpu_panel["backend"] = "gpu_batched_partA"
        gpu_panel["calibration_mode"] = "hedging_focused_near_atm_spx_gpu_batched"
        gpu_panel = add_heston_quality_flags(gpu_panel, panel_cfg, cfg.calibration)
        x0_lookup = {
            pd.Timestamp(row.date).normalize(): np.array([row.kappa, row.theta, row.xi, row.rho, row.v0], dtype=float)
            for row in gpu_panel.itertuples(index=False)
            if np.all(np.isfinite([row.kappa, row.theta, row.xi, row.rho, row.v0]))
        }
        polish_dates = select_cpu_polish_dates(gpu_panel, panel_cfg) if panel_cfg.cpu_polish_bad_fits else []
        if polish_dates:
            cpu_panel, _ = calibrate_heston_cpu_dates(surface, polish_dates, cfg, panel_cfg=panel_cfg, x0_lookup=x0_lookup)
            cpu_panel = add_heston_quality_flags(cpu_panel, panel_cfg, cfg.calibration)
        else:
            cpu_panel = pd.DataFrame()
        final = build_final_heston_panel(gpu_panel, cpu_panel)
    else:
        cpu_panel, _ = calibrate_heston_cpu_dates(surface, dates, cfg, panel_cfg=panel_cfg, x0_lookup=None)
        final = add_heston_quality_flags(cpu_panel, panel_cfg, cfg.calibration)

    final["date"] = pd.to_datetime(final["date"]).dt.normalize()
    final["profile"] = final.get("profile", "hedging_focused_near_atm")
    final["calibration_panel"] = globals().get("HEADLINE_EMPIRICAL_MODE", "unknown")
    final = final.sort_values("date").drop_duplicates(["date", "profile"], keep="last").reset_index(drop=True)
    missing_dates = sorted(set(pd.Series(dates).dt.normalize()).difference(set(final["date"])))
    if missing_dates:
        raise RuntimeError(f"Heston calibration missing {len(missing_dates)} dates; first: {missing_dates[:5]}")
    write_dataframe_cache(final, cache_path, artifact_name, "heston", cfg, dependencies=deps, extra={"headline_empirical_mode": globals().get("HEADLINE_EMPIRICAL_MODE", "unknown")})
    print(f"Saved Heston hedging-panel calibration cache: {cache_path} ({len(final)} rows)")
    return final

heston_calibration = load_or_run_heston_calibration(surface_panel, CFG)
display(heston_calibration.head())


Loading validated Heston hedging-panel calibration cache: C:\Users\lhkke\OneDrive - Queen Mary, University of London\Documents\Dissertation\Deep-Hedging-under-Lifted-Heston-MSc-Dissertation-\cache_final\heston_hedging_panel_calibration_5bucket.parquet


,backend,bound_hit_count,calibration_mode,chunk_id,chunk_id_fit,cpu_seconds,date,feller_margin,feller_pass,finite_params,fit_ok,kappa,loss,max_abs_price_error,message,n_options,n_points,needs_cpu_polish,optimizer_cost,optimizer_nfev,optimizer_nit,price_mae,price_rmse,pricing_engine,profile,quality_flag,rho,success,theta,v0,vega_scaled_rmse,xi,calibration_panel
0,cpu_scipy_least_squares,0,hedging_focused_near_atm_spx_cpu,NaN,NaN,0.398496,2010-01-04,-0.484471,False,True,True,4.840616,8.955512e-08,NaN,`gtol` termination condition is satisfied.,13,13,False,7.417031e-07,25.0,-1.0,0.051696,0.058337,partA_cpu_gauss_laguerre_p1p2,hedging_focused_near_atm,OK,-0.686512,1.0,0.067734,0.037066,0.000299,1.067813,HEDGING_FOCUSED_PANEL
1,cpu_scipy_least_squares,0,hedging_focused_near_atm_spx_cpu,NaN,NaN,0.313135,2010-01-05,-0.205100,False,True,True,1.154539,3.704908e-07,NaN,`gtol` termination condition is satisfied.,16,16,False,3.815972e-06,12.0,-1.0,0.149815,0.182779,partA_cpu_gauss_laguerre_p1p2,hedging_focused_near_atm,OK,-0.709870,1.0,0.112471,0.036306,0.000609,0.681765,HEDGING_FOCUSED_PANEL
2,cpu_scipy_least_squares,0,hedging_focused_near_atm_spx_cpu,NaN,NaN,0.583240,2010-01-06,-0.201449,False,True,True,1.035999,3.341898e-07,NaN,`gtol` termination condition is satisfied.,30,30,False,6.494967e-06,14.0,-1.0,0.108670,0.121961,partA_cpu_gauss_laguerre_p1p2,hedging_focused_near_atm,OK,-0.686329,1.0,0.124329,0.033162,0.000578,0.677538,HEDGING_FOCUSED_PANEL
3,cpu_scipy_least_squares,0,hedging_focused_near_atm_spx_cpu,NaN,NaN,0.706187,2010-01-07,-0.191157,False,True,True,1.310406,3.839909e-07,NaN,`ftol` termination condition is satisfied.,35,35,False,8.431027e-06,13.0,-1.0,0.110183,0.134450,partA_cpu_gauss_laguerre_p1p2,hedging_focused_near_atm,OK,-0.690385,1.0,0.105965,0.031657,0.000620,0.684742,HEDGING_FOCUSED_PANEL
4,cpu_scipy_least_squares,0,hedging_focused_near_atm_spx_cpu,NaN,NaN,1.115355,2010-01-08,-0.162642,False,True,True,0.803358,4.166474e-07,NaN,`gtol` termination condition is satisfied.,36,36,False,9.368457e-06,23.0,-1.0,0.113244,0.139311,partA_cpu_gauss_laguerre_p1p2,hedging_focused_near_atm,OK,-0.678488,1.0,0.136489,0.030173,0.000645,0.618013,HEDGING_FOCUSED_PANEL


In [23]:
# ============================================================
# Calibration fit inspection helpers
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


def _visual_model_key(model_name: str) -> str:
    text = str(model_name).strip().lower().replace('-', '_').replace(' ', '_')
    if 'lifted' in text:
        return 'lifted_heston'
    if 'heston' in text:
        return 'heston'
    if 'market' in text:
        return 'market'
    return text


def _normalise_date_col(df: pd.DataFrame, col: str = "date") -> pd.DataFrame:
    out = df.copy()
    out[col] = pd.to_datetime(out[col]).dt.normalize()
    return out


def add_maturity_bucket(df: pd.DataFrame, T_col: str = "T") -> pd.DataFrame:
    """
    Add maturity buckets used for calibration diagnostics.
    T is assumed to be in years.
    """
    out = df.copy()
    dte = out[T_col].astype(float) * 365.25

    bins = [0, 14, 30, 60, 120, 365, np.inf]
    labels = ["0-2W", "2W-1M", "1M-2M", "2M-4M", "4M-1Y", "1Y+"]

    out["dte"] = dte
    out["maturity_bucket"] = pd.cut(dte, bins=bins, labels=labels, right=True)
    return out


def add_moneyness_bucket(df: pd.DataFrame, k_col: str = "log_forward_moneyness") -> pd.DataFrame:
    """
    Add log-forward-moneyness buckets.
    """
    out = df.copy()
    bins = [-np.inf, -0.20, -0.10, -0.03, 0.03, 0.10, 0.20, np.inf]
    labels = ["deep_ITM_call", "ITM_call", "near_ATM_low", "ATM", "near_ATM_high", "OTM_call", "far_OTM_call"]
    out["moneyness_bucket"] = pd.cut(out[k_col].astype(float), bins=bins, labels=labels)
    return out


def weighted_rmse(x: np.ndarray, w: np.ndarray | None = None) -> float:
    x = np.asarray(x, dtype=float)
    mask = np.isfinite(x)
    if w is None:
        if mask.sum() == 0:
            return np.nan
        return float(np.sqrt(np.mean(x[mask] ** 2)))

    w = np.asarray(w, dtype=float)
    mask &= np.isfinite(w) & (w >= 0)
    if mask.sum() == 0 or np.sum(w[mask]) <= 0:
        return np.nan
    return float(np.sqrt(np.average(x[mask] ** 2, weights=w[mask])))


def summarise_calibration_panel(
    panel: pd.DataFrame,
    model_name: str,
    param_cols: list[str],
    metric_cols: list[str] | None = None,
) -> None:
    """
    Inspect date-level calibration output.
    """
    print("=" * 80)
    print(f"{model_name} calibration panel summary")
    print("=" * 80)

    panel = _normalise_date_col(panel)

    print(f"Rows: {len(panel):,}")
    print(f"Date range: {panel['date'].min()} -> {panel['date'].max()}")
    print(f"Unique dates: {panel['date'].nunique():,}")

    if "success" in panel.columns:
        print("\nSuccess rate:")
        display(panel["success"].value_counts(dropna=False).rename("count").to_frame())

    if "quality_flag" in panel.columns:
        print("\nQuality flags:")
        display(panel["quality_flag"].value_counts(dropna=False).rename("count").to_frame())

    if metric_cols is None:
        metric_cols = [
            c for c in [
                "loss",
                "price_rmse",
                "price_mae",
                "vega_scaled_rmse",
                "iv_rmse",
                "cpu_seconds",
                "gpu_seconds",
                "optimizer_nfev",
                "bound_hit_count",
                "feller_margin",
            ]
            if c in panel.columns
        ]

    if metric_cols:
        print("\nMetric summary:")
        display(panel[metric_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T)

    available_params = [c for c in param_cols if c in panel.columns]
    if available_params:
        print("\nParameter summary:")
        display(panel[available_params].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T)

    suspicious_cols = [c for c in ["vega_scaled_rmse", "price_rmse", "loss"] if c in panel.columns]
    if suspicious_cols:
        sort_col = "vega_scaled_rmse" if "vega_scaled_rmse" in panel.columns else suspicious_cols[0]
        print(f"\nWorst 20 dates by {sort_col}:")
        display(
            panel.sort_values(sort_col, ascending=False)
            .head(20)
            [["date"] + [c for c in ["success", "quality_flag", "loss", "vega_scaled_rmse", "price_rmse", "price_mae", "n_options", "optimizer_nfev", "cpu_seconds", "bound_hit_count"] if c in panel.columns] + available_params]
        )


def plot_calibration_metric_time_series(
    panel: pd.DataFrame,
    metric: str,
    title: str,
    model_name: str = "Heston",
) -> None:
    panel = _normalise_date_col(panel)
    if metric not in panel.columns:
        raise KeyError(f"{metric} not found in calibration panel.")

    style_key = _visual_model_key(model_name)
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(
        panel["date"],
        pd.to_numeric(panel[metric], errors="coerce"),
        color=model_color(style_key),
        linestyle=model_linestyle(style_key),
        linewidth=1.05,
        alpha=0.82,
    )
    ax.set_title(title)
    ax.set_xlabel("Date")
    ax.set_ylabel(metric)
    ax.grid(True, alpha=0.3)
    plt.show()


def plot_parameter_time_series(
    panel: pd.DataFrame,
    param_cols: list[str],
    title: str,
    model_name: str = "Heston",
) -> None:
    panel = _normalise_date_col(panel)
    available = [c for c in param_cols if c in panel.columns]
    if not available:
        raise KeyError("None of the requested parameter columns are available.")

    style_key = _visual_model_key(model_name)
    for col in available:
        fig, ax = plt.subplots(figsize=(12, 3))
        ax.plot(
            panel["date"],
            pd.to_numeric(panel[col], errors="coerce"),
            color=model_color(style_key),
            linestyle=model_linestyle(style_key),
            linewidth=1.0,
            alpha=0.82,
        )
        ax.set_title(f"{title}: {col}")
        ax.set_xlabel("Date")
        ax.set_ylabel(col)
        ax.grid(True, alpha=0.3)
        plt.show()


def residual_bucket_table(
    residuals: pd.DataFrame,
    residual_col: str = "vega_scaled_error",
    weight_col: str = "fit_weight",
) -> pd.DataFrame:
    """
    Summarise residuals by maturity and moneyness bucket.
    """
    x = add_maturity_bucket(residuals)
    x = add_moneyness_bucket(x)

    def _agg(g: pd.DataFrame) -> pd.Series:
        return pd.Series({
            "n": len(g),
            "mean_error": float(np.nanmean(g[residual_col])),
            "rmse": weighted_rmse(g[residual_col].to_numpy(), g[weight_col].to_numpy() if weight_col in g else None),
            "mae": float(np.nanmean(np.abs(g[residual_col]))),
            "p05": float(np.nanpercentile(g[residual_col], 5)),
            "p50": float(np.nanpercentile(g[residual_col], 50)),
            "p95": float(np.nanpercentile(g[residual_col], 95)),
        })

    return (
        x.groupby(["maturity_bucket", "moneyness_bucket"], observed=True)
        .apply(_agg)
        .reset_index()
        .sort_values(["maturity_bucket", "moneyness_bucket"])
    )


def plot_residual_heatmap(
    residual_summary: pd.DataFrame,
    value_col: str = "rmse",
    title: str = "Residual RMSE by maturity and moneyness",
) -> None:
    pivot = residual_summary.pivot(
        index="maturity_bucket",
        columns="moneyness_bucket",
        values=value_col,
    )

    fig, ax = plt.subplots(figsize=(12, 5))
    plot_annotated_heatmap(
        pivot,
        ax=ax,
        fig=fig,
        title=title,
        xlabel="Moneyness bucket",
        ylabel="Maturity bucket",
        cbar_label=value_col.replace("_", " "),
        cmap=ERROR_CMAP,
        fmt=None,
    )
    plt.tight_layout()
    plt.show()


def plot_model_vs_market_scatter(
    residuals: pd.DataFrame,
    model_name: str,
    date: pd.Timestamp | None = None,
    market_col: str = "call_equiv_mid",
    model_col: str = "model_price",
) -> None:
    x = residuals.copy()
    if date is not None:
        date = pd.Timestamp(date).normalize()
        x = x[pd.to_datetime(x["date"]).dt.normalize().eq(date)]

    if x.empty:
        raise ValueError("No rows available for requested date.")

    style_key = _visual_model_key(model_name)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(
        pd.to_numeric(x[market_col], errors="coerce"),
        pd.to_numeric(x[model_col], errors="coerce"),
        s=18,
        alpha=0.68,
        color=model_color(style_key),
        marker=model_marker(style_key),
        edgecolors="white",
        linewidths=0.30,
    )

    lo = float(np.nanmin([x[market_col].min(), x[model_col].min()]))
    hi = float(np.nanmax([x[market_col].max(), x[model_col].max()]))
    ax.plot([lo, hi], [lo, hi], linestyle="--", color="#6B7280", linewidth=0.95, alpha=0.75)

    title_date = "" if date is None else f" | {date.date()}"
    ax.set_title(f"{model_name}: model vs market prices{title_date}")
    ax.set_xlabel("Market call-equivalent price")
    ax.set_ylabel("Model price")
    ax.grid(True, alpha=0.3)
    plt.show()


def plot_smile_fit_for_date(
    residuals: pd.DataFrame,
    model_name: str,
    date: pd.Timestamp,
    maturity_bucket: str | None = None,
) -> None:
    x = residuals.copy()
    x["date"] = pd.to_datetime(x["date"]).dt.normalize()
    date = pd.Timestamp(date).normalize()
    x = x[x["date"].eq(date)]

    if maturity_bucket is not None:
        x = add_maturity_bucket(x)
        x = x[x["maturity_bucket"].astype(str).eq(maturity_bucket)]

    if x.empty:
        raise ValueError("No rows available for requested date / maturity bucket.")

    x = x.sort_values(["T", "log_forward_moneyness"])

    for T_value, g in x.groupby("T"):
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.scatter(g["log_forward_moneyness"], g["call_equiv_mid"], label="market", s=14)
        ax.plot(g["log_forward_moneyness"], g["model_price"], label="model")
        ax.set_title(f"{model_name} fit | date={date.date()} | T={T_value:.4f}y")
        ax.set_xlabel("log(K / F)")
        ax.set_ylabel("call-equivalent price")
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.show()

In [24]:
# ============================================================
# Section 4 fit inspection: Classical Heston
# ============================================================

HESTON_PARAM_COLS = ["kappa", "theta", "xi", "rho", "v0"]

summarise_calibration_panel(
    heston_calibration,
    model_name="Classical Heston",
    param_cols=HESTON_PARAM_COLS,
)

for metric in [c for c in ["vega_scaled_rmse", "price_rmse", "loss", "cpu_seconds", "optimizer_nfev"] if c in heston_calibration.columns]:
    plot_calibration_metric_time_series(
        heston_calibration,
        metric=metric,
        title=f"Heston {metric} through time",
        model_name="Heston",
    )

plot_parameter_time_series(
    heston_calibration,
    HESTON_PARAM_COLS,
    title="Heston calibrated parameters",
    model_name="Heston",
)


def enforce_surface_inspection_schema(surface: pd.DataFrame) -> pd.DataFrame:
    """Standardise deterministic derived fields; do not fabricate missing research data."""
    out = surface.copy()
    if "log_forward_moneyness" not in out.columns:
        if "log_moneyness" not in out.columns:
            raise ValueError("Surface must contain log_forward_moneyness or log_moneyness from parity-forward construction.")
        out["log_forward_moneyness"] = out["log_moneyness"].astype(float)

    required = [
        "date", "spot_ref", "forward_ref", "strike", "T", "r", "q",
        "call_equiv_mid", "bs_vega", "fit_weight", "log_forward_moneyness",
    ]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f"Surface is missing required inspection columns: {missing}")
    bad = (
        ~np.isfinite(out["forward_ref"].astype(float)) | (out["forward_ref"].astype(float) <= 0)
        | ~np.isfinite(out["strike"].astype(float)) | (out["strike"].astype(float) <= 0)
        | ~np.isfinite(out["spot_ref"].astype(float)) | (out["spot_ref"].astype(float) <= 0)
        | ~np.isfinite(out["log_forward_moneyness"].astype(float))
    )
    if bad.any():
        raise ValueError(f"Surface inspection schema found {int(bad.sum())} invalid rows.")
    return out


def build_heston_fit_residuals(surface: pd.DataFrame, calib: pd.DataFrame, cfg: ExperimentConfig, max_dates: int | None = None) -> pd.DataFrame:
    required_calib_cols = {"date", "kappa", "theta", "xi", "rho", "v0"}
    missing_calib = required_calib_cols - set(calib.columns)
    if missing_calib:
        raise ValueError(f"calib is missing required columns: {sorted(missing_calib)}")

    surface = _normalise_date_col(enforce_surface_inspection_schema(surface))
    calib = _normalise_date_col(calib)
    dates = sorted(set(surface["date"]).intersection(set(calib["date"])))
    if max_dates is not None:
        dates = dates[:max_dates]
    if not dates:
        raise ValueError("No overlapping dates between surface and Heston calibration.")

    pricer = HestonPricer(n_quad=cfg.calibration.heston_pricing_nodes)
    rows = []
    param_lookup = calib.set_index("date")[HESTON_PARAM_COLS].to_dict("index")
    for i, date in enumerate(dates, start=1):
        g = surface[surface["date"].eq(date)].copy()
        p = param_lookup[date]
        params = HestonParams(float(p["kappa"]), float(p["theta"]), float(p["xi"]), float(p["rho"]), float(p["v0"]))
        model_price = pricer.call_prices(
            S=g["spot_ref"].to_numpy(float),
            K=g["strike"].to_numpy(float),
            T=g["T"].to_numpy(float),
            r=g["r"].to_numpy(float),
            q=g["q"].to_numpy(float),
            params=params,
        )
        g["model_price"] = model_price
        g["price_error"] = g["model_price"] - g["call_equiv_mid"]
        g["vega_scaled_error"] = g["price_error"] / np.maximum(np.abs(g["bs_vega"].to_numpy(float)), cfg.calibration.vega_floor)
        g["abs_vega_scaled_error"] = np.abs(g["vega_scaled_error"])
        g["model_name"] = "heston"
        rows.append(g)
        if i % 250 == 0:
            print(f"Repriced Heston residuals for {i:,}/{len(dates):,} dates")
    out = pd.concat(rows, ignore_index=True)
    return out


def load_or_build_heston_diagnostics(surface: pd.DataFrame, calib: pd.DataFrame, cfg: ExperimentConfig) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    residual_path = cfg.paths.heston_fit_residuals_path
    bucket_path = cfg.paths.heston_bucket_summary_path
    deps = [cfg.paths.surface_cache_path, cfg.paths.heston_calibration_path]
    _rebuild_diagnostics = cfg.flags.force_rebuild("calibration")
    if not _rebuild_diagnostics:
        try:
            residuals = load_dataframe_cache(residual_path, "heston_fit_residuals", "heston", cfg, required_columns={"date", "model_price", "price_error", "vega_scaled_error"}, dependencies=deps)
            bucket = load_dataframe_cache(bucket_path, "heston_residual_bucket_summary", "heston", cfg, dependencies=[residual_path])
            print(f"Loading validated Heston diagnostics: {residual_path}")
        except (FileNotFoundError, ValueError) as exc:
            if not cfg.flags.can_rebuild("calibration"):
                raise RuntimeError(
                    "Heston diagnostics are missing/stale and strict reuse forbids rebuilding. "
                    "Set rebuild_from='calibration' or use 'auto'."
                ) from exc
            print(f"[Heston diagnostics] cache unavailable/stale; rebuilding. Reason: {exc}")
            _rebuild_diagnostics = True
    if _rebuild_diagnostics:
        residuals = build_heston_fit_residuals(surface, calib, cfg)
        bucket = residual_bucket_table(residuals)
        write_dataframe_cache(residuals, residual_path, "heston_fit_residuals", "heston", cfg, dependencies=deps)
        write_dataframe_cache(bucket, bucket_path, "heston_residual_bucket_summary", "heston", cfg, dependencies=[residual_path])
    date_fit = (
        residuals.groupby("date")
        .apply(lambda g: pd.Series({
            "n": len(g),
            "price_rmse": weighted_rmse(g["price_error"]),
            "vega_scaled_rmse": weighted_rmse(g["vega_scaled_error"], g["fit_weight"]),
            "mean_abs_error": float(np.nanmean(np.abs(g["price_error"]))),
        }))
        .reset_index()
        .sort_values("vega_scaled_rmse", ascending=False)
    )
    return residuals, bucket, date_fit


surface_panel = enforce_surface_inspection_schema(surface_panel)
heston_fit_residuals, heston_bucket_summary, heston_date_fit = load_or_build_heston_diagnostics(surface_panel, heston_calibration, CFG)

print("\nOverall Heston residual diagnostics:")
display(pd.DataFrame({
    "metric": ["n", "price_rmse", "price_mae", "vega_scaled_rmse", "vega_scaled_mae"],
    "value": [
        len(heston_fit_residuals),
        weighted_rmse(heston_fit_residuals["price_error"]),
        float(np.nanmean(np.abs(heston_fit_residuals["price_error"]))),
        weighted_rmse(heston_fit_residuals["vega_scaled_error"], heston_fit_residuals["fit_weight"]),
        float(np.nanmean(np.abs(heston_fit_residuals["vega_scaled_error"]))),
    ],
}))

print("\nHeston residuals by maturity and moneyness:")
display(heston_bucket_summary)
plot_residual_heatmap(heston_bucket_summary, value_col="rmse", title="Heston vega-scaled RMSE by maturity and moneyness")

print("\nWorst 30 Heston dates by repriced vega-scaled RMSE:")
display(heston_date_fit.head(30))
worst_heston_date = heston_date_fit.iloc[0]["date"]
print(f"Worst Heston fit date: {worst_heston_date}")
plot_model_vs_market_scatter(heston_fit_residuals, model_name="Heston", date=worst_heston_date)
plot_smile_fit_for_date(heston_fit_residuals, model_name="Heston", date=worst_heston_date)


Classical Heston calibration panel summary
Rows: 2,869
Date range: 2010-01-04 00:00:00 -> 2024-12-31 00:00:00
Unique dates: 2,869

Success rate:


,count
success,
NaN,2844
1.0,20
0.0,5



Quality flags:


,count
quality_flag,
OK,2869



Metric summary:


,count,mean,std,min,1%,5%,50%,95%,99%,max
loss,2869.0,0.000005,0.000016,1.023931e-09,2.281271e-08,7.289967e-08,0.000001,0.000024,0.000072,0.000302
price_rmse,2869.0,0.454807,0.403337,2.201633e-02,6.777541e-02,1.137081e-01,0.354321,1.083155,1.788709,9.733312
price_mae,2869.0,0.362620,0.291860,1.657234e-02,5.506532e-02,9.116927e-02,0.287494,0.864357,1.352348,5.215223
vega_scaled_rmse,2869.0,0.001438,0.001691,2.356283e-05,1.200831e-04,2.085016e-04,0.000828,0.004609,0.008510,0.021875
cpu_seconds,25.0,0.837385,0.705749,7.606610e-02,9.161923e-02,1.436985e-01,0.620141,2.358750,2.638849,2.665133
optimizer_nfev,25.0,33.200000,36.282457,3.000000e+00,3.000000e+00,3.200000e+00,14.000000,100.000000,100.000000,100.000000
bound_hit_count,2869.0,0.002091,0.064651,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,2.000000
feller_margin,2869.0,-0.511169,0.584083,-7.693201e+00,-2.982352e+00,-1.437795e+00,-0.337056,-0.133742,0.016048,1.094878


... (21 outputs left out)

,date,n,price_rmse,vega_scaled_rmse,mean_abs_error
1946,2017-12-26,26.0,1.038352,0.019338,0.889935
790,2013-03-20,36.0,1.952523,0.013596,1.682448
367,2011-07-01,38.0,0.807593,0.012740,0.662894
1098,2014-06-23,20.0,1.887894,0.012029,1.354112
251,2011-01-13,17.0,0.522518,0.011361,0.439641
309,2011-04-08,24.0,1.033509,0.011271,0.809917
786,2013-03-14,50.0,1.081895,0.010264,0.766818
843,2013-06-07,39.0,0.316292,0.009408,0.282992
1950,2018-01-08,28.0,0.704113,0.009350,0.556630
769,2013-02-19,46.0,1.317815,0.009041,0.940088


Worst Heston fit date: 2017-12-26 00:00:00


<Figure size 780x780 with 1 Axes>

<Figure size 1040x520 with 1 Axes>

## 4.2 Lifted Heston

The lifted-Heston calibration extends the scalar Heston state into a finite-dimensional factor state. The aggregate variance is

$$
V_t=\sum_{i=1}^{N}c_iU_t^i,
$$

with factors associated with different mean-reversion speeds. In the dissertation narrative, this is the structural-memory model: it approximates rough-volatility behaviour while preserving finite-dimensional Markovian simulation.

The calibration task is delicate because lifted Heston has a richer state representation but should not be allowed to win merely by being more flexible on a different target set. The notebook therefore reports common-row comparisons, ATM-skew diagnostics, residual heatmaps and parameter-stability checks. These outputs answer whether the model is fitting the relevant option-market objects and whether the fitted states are stable enough to use in hedging experiments.

The lifted model should be interpreted as a finite-factor approximation, not as proof that the true SPX volatility process is rough. Livieri-style roughness evidence motivates fixed-$H$ sensitivity, while Abi Jaber's lifted construction motivates the exponential factor grid. Abi Jaber and Li's later critique motivates caution: rough or lifted dynamics may help selected short-maturity objects without dominating the global SPX smile.

This section therefore prepares evidence for a conditional claim: lifted Heston may add hedge-relevant volatility-memory information, but the value must be shown in terminal hedging performance and mechanism diagnostics.


In [25]:
@dataclass(frozen=True)
class LiftedKernelGrid:
    H: float
    alpha: float
    n_factors: int
    rn_scale: float
    lambdas: np.ndarray
    weights: np.ndarray
    grid: pd.DataFrame
    diagnostics: pd.DataFrame

    @property
    def decays(self) -> np.ndarray:
        """Alias for the lifted-factor mean-reversion rates used by bucket features."""
        return self.lambdas


@dataclass(frozen=True)
class LiftedHestonParams:
    kappa: float
    theta: float
    xi: float
    rho: float
    v0: float


def build_abi_zaugg_lifted_kernel(cfg: LiftedHestonConfig) -> LiftedKernelGrid:
    if not SCIPY_AVAILABLE:
        raise RuntimeError("SciPy is required for gamma() in lifted kernel construction.")
    H = float(cfg.H)
    alpha = H + 0.5
    N = int(cfg.n_factors)
    R_N = float(cfg.rn_scale)
    i = np.arange(1, N + 1, dtype=float)

    weights = (
        ((R_N ** (1.0 - alpha) - 1.0) * R_N ** ((alpha - 1.0) * (1.0 + N / 2.0)))
        / (gamma(alpha) * gamma(2.0 - alpha))
        * R_N ** ((1.0 - alpha) * i)
    )
    lambdas = (
        ((1.0 - alpha) * (R_N ** (2.0 - alpha) - 1.0))
        / ((2.0 - alpha) * (R_N ** (1.0 - alpha) - 1.0))
        * R_N ** (i - 1.0 - N / 2.0)
    )

    t_grid = np.geomspace(cfg.kernel_t_min, cfg.kernel_t_max, cfg.kernel_grid_size)
    rough_kernel = t_grid ** (alpha - 1.0) / gamma(alpha)
    lifted_kernel = np.exp(-np.outer(t_grid, lambdas)) @ weights
    diagnostics = pd.DataFrame({
        "t": t_grid,
        "rough_kernel": rough_kernel,
        "lifted_kernel": lifted_kernel,
        "abs_error": np.abs(lifted_kernel - rough_kernel),
        "rel_error": (lifted_kernel - rough_kernel) / rough_kernel,
        "abs_rel_error": np.abs((lifted_kernel - rough_kernel) / rough_kernel),
    })
    grid = pd.DataFrame({
        "factor": np.arange(1, N + 1),
        "lambda": lambdas,
        "weight": weights,
        "time_scale_years": 1.0 / lambdas,
        "time_scale_days": 365.25 / lambdas,
        "rn_scale": R_N,
        "H": H,
        "alpha": alpha,
    })
    return LiftedKernelGrid(H=H, alpha=alpha, n_factors=N, rn_scale=R_N, lambdas=lambdas, weights=weights, grid=grid, diagnostics=diagnostics)


lifted_kernel = build_abi_zaugg_lifted_kernel(CFG.lifted)
print("Validated lifted-Heston kernel configuration")
print({"H": lifted_kernel.H, "alpha": lifted_kernel.alpha, "N": lifted_kernel.n_factors, "r_N": lifted_kernel.rn_scale})
display(lifted_kernel.grid.head())
display(lifted_kernel.diagnostics.describe())

assert lifted_kernel.n_factors == 20
assert abs(lifted_kernel.rn_scale - 2.5) < 1e-12


Validated lifted-Heston kernel configuration
{'H': 0.35, 'alpha': 0.85, 'N': 20, 'r_N': 2.5}


,factor,lambda,weight,time_scale_years,time_scale_days,rn_scale,H,alpha
0,1,0.000173,0.035909,5765.832356,2.105970e+06,2.5,0.35,0.85
1,2,0.000434,0.041200,2306.332942,8.423881e+05,2.5,0.35,0.85
2,3,0.001084,0.047270,922.533177,3.369552e+05,2.5,0.35,0.85
3,4,0.002710,0.054235,369.013271,1.347821e+05,2.5,0.35,0.85
4,5,0.006775,0.062226,147.605308,5.391284e+04,2.5,0.35,0.85


,t,rough_kernel,lifted_kernel,abs_error,rel_error,abs_rel_error
count,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000
mean,0.304305,1.383416,1.131458,0.251958,-0.197027,0.197027
std,0.463203,0.392994,0.390653,0.002342,0.054277,0.054277
min,0.002738,0.810125,0.561580,0.248544,-0.306798,0.117843
25%,0.014234,1.037380,0.787474,0.249905,-0.240901,0.149226
50%,0.074000,1.328384,1.076755,0.251629,-0.189425,0.189425
75%,0.384709,1.701019,1.447184,0.253835,-0.149226,0.240901
max,2.000000,2.178186,1.921502,0.256684,-0.117843,0.306798


In [26]:
from types import SimpleNamespace


class LiftedCalibrationNumericalFailure(RuntimeError):
    """Raised when no finite lifted-Heston calibration candidate is available for a date."""
    pass

# -----------------------------------------------------------------------------
# Final lifted-Heston full-surface calibration
# Main specification: five calibrated Heston-style parameters with a regularised
# forward-variance input curve used for initialisation and deterministic-curve shape.
# One model convention is used for pricing, simulation and conditional model deltas.
# -----------------------------------------------------------------------------

LIFTED_CALIBRATION_BACKEND = "lifted_riccati_fourier_regularised_forward_curve_5p_final"

@dataclass(frozen=True)
class ForwardVarianceInputCurve:
    """Positive piecewise-constant forward-variance curve ξ₀(t).

    For the lifted model
        V_t = g₀(t) + Σ c_i U^i_t,
        dU^i_t = (-x_i U^i_t - κ V_t)dt + ξ√V_t dW_t,
        U^i_0 = 0,
    the deterministic input is
        g₀(t) = ξ₀(t) + κ Σ c_i ∫₀ᵗ exp(-x_i(t-s)) ξ₀(s) ds.
    This makes E[V_t] = ξ₀(t), matching the methodology's regularised near-ATM forward-variance proxy.
    """

    maturities: np.ndarray
    forward_variances: np.ndarray

    @classmethod
    def from_frame(cls, frame: pd.DataFrame) -> "ForwardVarianceInputCurve":
        require_columns(frame, {"T", "forward_variance"}, "Forward-variance input curve")
        g = frame[["T", "forward_variance"]].copy().sort_values("T").drop_duplicates("T")
        T = g["T"].to_numpy(float)
        xi0 = g["forward_variance"].to_numpy(float)
        if len(T) < 2 or np.any(np.diff(T) <= 0):
            raise ValueError("Forward-variance input curve requires at least two strictly increasing maturities.")
        if np.any(~np.isfinite(xi0)) or np.any(xi0 <= 0):
            raise ValueError("Forward-variance input curve must be finite and positive.")
        return cls(T, xi0)

    @property
    def max_maturity(self) -> float:
        return float(self.maturities[-1])

    def xi0(self, t: np.ndarray | float) -> np.ndarray | float:
        arr = np.asarray(t, dtype=float)
        if np.any(arr < -1e-12):
            raise ValueError("Forward variance cannot be evaluated at negative time.")
        idx = np.searchsorted(self.maturities, np.maximum(arr, 0.0), side="left")
        idx = np.clip(idx, 0, len(self.maturities) - 1)
        out = self.forward_variances[idx]
        return float(out) if arr.ndim == 0 else out

    def exponential_convolution(self, t: np.ndarray | float, lambdas: np.ndarray) -> np.ndarray:
        """Exact convolution of the piecewise-constant ξ₀ with exp(-x(t-s))."""
        scalar = np.asarray(t).ndim == 0
        times = np.atleast_1d(np.asarray(t, dtype=float))
        x = np.asarray(lambdas, dtype=float).reshape(-1)
        if np.any(times < -1e-12) or np.any(x < 0):
            raise ValueError("Invalid convolution inputs.")
        bounds = np.concatenate([[0.0], self.maturities])
        out = np.zeros((len(times), len(x)), dtype=float)
        for k, tt in enumerate(times):
            tt = max(float(tt), 0.0)
            for m, xi_m in enumerate(self.forward_variances):
                a = float(bounds[m])
                b = min(float(bounds[m + 1]), tt)
                if b > a:
                    zero = x < 1e-14
                    integ = np.empty_like(x)
                    integ[zero] = b - a
                    integ[~zero] = (
                        np.exp(-x[~zero] * (tt - b)) - np.exp(-x[~zero] * (tt - a))
                    ) / x[~zero]
                    out[k] += float(xi_m) * integ
                if bounds[m + 1] >= tt:
                    break
            if tt > self.maturities[-1]:
                a = float(self.maturities[-1])
                b = tt
                zero = x < 1e-14
                integ = np.empty_like(x)
                integ[zero] = b - a
                integ[~zero] = (1.0 - np.exp(-x[~zero] * (b - a))) / x[~zero]
                out[k] += float(self.forward_variances[-1]) * integ
        return out[0] if scalar else out

    def g0(self, t: np.ndarray | float, kappa: float, kernel: LiftedKernelGrid) -> np.ndarray | float:
        conv = self.exponential_convolution(t, kernel.lambdas)
        deterministic = self.xi0(t) + float(kappa) * np.sum(conv * kernel.weights, axis=-1)
        if np.any(~np.isfinite(deterministic)) or np.any(np.asarray(deterministic) <= 0):
            raise FloatingPointError("Lifted deterministic input g0 is non-positive or non-finite.")
        return deterministic

    def with_endpoint_levels(self, v0: float, theta: float) -> "ForwardVarianceInputCurve":
        v0 = float(v0)
        theta = float(theta)
        if not np.isfinite(v0) or not np.isfinite(theta) or v0 <= 0.0 or theta <= 0.0:
            raise ValueError("Calibrated lifted curve endpoint levels must be finite and positive.")
        base = np.asarray(self.forward_variances, dtype=float)
        T = np.asarray(self.maturities, dtype=float)
        base0 = float(base[0])
        baseT = float(base[-1])
        if abs(baseT - base0) > 1e-10:
            adjusted = v0 + (base - base0) * ((theta - v0) / (baseT - base0))
        else:
            span = max(float(T[-1] - T[0]), 1e-12)
            adjusted = v0 + (theta - v0) * ((T - T[0]) / span)
        if np.any(~np.isfinite(adjusted)) or np.any(adjusted <= 0.0):
            raise ValueError("Calibrated lifted curve endpoint transform produced non-positive forward variances.")
        return ForwardVarianceInputCurve(T.copy(), adjusted)


@dataclass(frozen=True)
class LiftedCalibrationResult:
    date: pd.Timestamp
    profile: str
    params: LiftedHestonParams
    loss: float
    fit_table: pd.DataFrame
    timing: dict[str, Any]
    optimizer: dict[str, Any]


class LiftedRiccatiFourierPricer:
    """Abi-Jaber finite-factor Riccati/Fourier pricer with deterministic g₀ input."""

    def __init__(
        self,
        kernel: LiftedKernelGrid,
        n_quad: int = 64,
        riccati_steps: int = 160,
        delta_cache_size: int = 4096,
    ):
        self.kernel = kernel
        self.nodes, self.quad_weights = np.polynomial.laguerre.laggauss(int(n_quad))
        self.riccati_steps = int(riccati_steps)
        self.lambdas = np.asarray(kernel.lambdas, dtype=float)
        self.weights = np.asarray(kernel.weights, dtype=float)
        self._delta_coefficient_cache: dict[tuple[Any, ...], tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]] = {}
        self.delta_cache_size = int(delta_cache_size)
        if self.delta_cache_size < 1:
            raise ValueError("delta_cache_size must be positive.")
        if self.riccati_steps < 16:
            raise ValueError("riccati_steps must be at least 16.")

    @staticmethod
    def _F(z: np.ndarray, aggregate_psi: np.ndarray, params: LiftedHestonParams) -> np.ndarray:
        """Lifted Riccati forcing with explicit overflow/invalid checks.

        Invalid parameter candidates are common during nonlinear calibration. They
        must be rejected by the objective, not allowed to leak RuntimeWarnings and
        crash the final pricing pass. This is a numeric guard, not a substitute
        pricing model.
        """
        z = np.asarray(z, dtype=np.complex128)
        aggregate_psi = np.asarray(aggregate_psi, dtype=np.complex128)
        try:
            with np.errstate(over="raise", invalid="raise", divide="raise"):
                out = (
                    0.5 * (z ** 2 - z)
                    + (params.rho * params.xi * z - params.kappa) * aggregate_psi
                    + 0.5 * params.xi ** 2 * aggregate_psi ** 2
                )
        except FloatingPointError as exc:
            raise FloatingPointError("Lifted Riccati forcing overflow/invalid value.") from exc
        if np.any(~np.isfinite(out.real)) or np.any(~np.isfinite(out.imag)):
            raise FloatingPointError("Lifted Riccati forcing produced non-finite values.")
        return out

    @staticmethod
    def _phi1(a: np.ndarray) -> np.ndarray:
        """Stable (1-exp(-a))/a used by the exponential Riccati integrator."""
        a = np.asarray(a, dtype=float)
        out = np.empty_like(a)
        small = np.abs(a) < 1e-8
        out[small] = 1.0 - 0.5 * a[small] + a[small] ** 2 / 6.0
        out[~small] = -np.expm1(-a[~small]) / a[~small]
        return out

    def _coefficients(
        self,
        z: np.ndarray,
        tau: float,
        current_time: float,
        params: LiftedHestonParams,
        curve: ForwardVarianceInputCurve,
    ) -> tuple[np.ndarray, np.ndarray]:
        """Solve the stiff lifted Riccati system with exponential midpoint steps.

        The factor grid contains decay rates spanning several orders of magnitude.
        Treating ``-x_i psi_i`` exactly avoids the stability failure of explicit
        RK schemes while retaining the common nonlinear Riccati forcing.
        """
        z = np.atleast_1d(np.asarray(z, dtype=np.complex128))
        psi = np.zeros((len(z), self.kernel.n_factors), dtype=np.complex128)
        phi = np.zeros(len(z), dtype=np.complex128)
        if tau <= 0:
            return phi, psi
        h = float(tau) / self.riccati_steps
        xh = self.lambdas * h
        decay_full = np.exp(-xh).reshape(1, -1)
        decay_half = np.exp(-0.5 * xh).reshape(1, -1)
        force_full = (h * self._phi1(xh)).reshape(1, -1)
        force_half = (0.5 * h * self._phi1(0.5 * xh)).reshape(1, -1)
        c = self.weights
        riccati_abs_cap = 1e8
        for step in range(self.riccati_steps):
            s = step * h
            aggregate = psi @ c
            if np.any(~np.isfinite(aggregate.real)) or np.any(~np.isfinite(aggregate.imag)) or np.max(np.abs(aggregate)) > riccati_abs_cap:
                raise FloatingPointError("Lifted Riccati aggregate exploded before midpoint step.")
            f0 = self._F(z, aggregate, params)
            psi_half = decay_half * psi + force_half * f0.reshape(-1, 1)
            aggregate_half = psi_half @ c
            if np.any(~np.isfinite(aggregate_half.real)) or np.any(~np.isfinite(aggregate_half.imag)) or np.max(np.abs(aggregate_half)) > riccati_abs_cap:
                raise FloatingPointError("Lifted Riccati aggregate exploded at midpoint step.")
            f_half = self._F(z, aggregate_half, params)
            psi = decay_full * psi + force_full * f_half.reshape(-1, 1)
            g_mid = float(curve.g0(current_time + tau - (s + 0.5 * h), params.kappa, self.kernel))
            phi += h * f_half * g_mid
            if not all(np.all(np.isfinite(v)) for v in [psi.real, psi.imag, phi.real, phi.imag]):
                raise FloatingPointError("Non-finite lifted Riccati state encountered.")
        return phi, psi

    def characteristic_state_batch(
        self,
        u: np.ndarray,
        S: np.ndarray,
        tau: float,
        r: float,
        q: float,
        params: LiftedHestonParams,
        curve: ForwardVarianceInputCurve,
        U: np.ndarray,
        current_time: float,
    ) -> np.ndarray:
        """Characteristic function for many current states and common remaining maturity."""
        u = np.atleast_1d(np.asarray(u, dtype=np.complex128))
        S = np.asarray(S, dtype=float).reshape(-1)
        U = np.asarray(U, dtype=float)
        if U.shape != (len(S), self.kernel.n_factors):
            raise ValueError("Lifted state matrix U has incompatible shape.")
        z = 1j * u
        phi, psi = self._coefficients(z, tau, current_time, params, curve)
        state_loading = psi * self.weights.reshape(1, -1)
        exponent = (
            np.outer(np.log(S) + (float(r) - float(q)) * tau, z)
            + phi.reshape(1, -1)
            + U @ state_loading.T
        )
        out = np.exp(exponent)
        if not all(np.all(np.isfinite(v)) for v in [out.real, out.imag]):
            raise FloatingPointError("Non-finite lifted characteristic values.")
        return out

    def probabilities_state_batch(
        self,
        S: np.ndarray,
        K: float,
        tau: float,
        r: float,
        q: float,
        params: LiftedHestonParams,
        curve: ForwardVarianceInputCurve,
        U: np.ndarray,
        current_time: float = 0.0,
    ) -> tuple[np.ndarray, np.ndarray]:
        """Compute P1 and P2 from one combined Riccati solve."""
        S = np.asarray(S, dtype=float).reshape(-1)
        U = np.asarray(U, dtype=float)
        if U.shape != (len(S), self.kernel.n_factors):
            raise ValueError("Lifted state matrix U has incompatible shape.")
        if tau <= 1e-10:
            p = np.where(S > K, 1.0, np.where(S < K, 0.0, 0.5))
            return p, p
        u = self.nodes.astype(np.complex128)
        z = np.concatenate([np.array([1.0 + 0.0j]), 1.0 + 1j * u, 1j * u])
        phi, psi = self._coefficients(z, tau, current_time, params, curve)
        state_loading = psi * self.weights.reshape(1, -1)
        x = np.log(S) + (float(r) - float(q)) * tau
        cf = np.exp(np.outer(x, z) + phi.reshape(1, -1) + U @ state_loading.T)
        denominator = cf[:, 0]
        if np.any(np.abs(denominator) < 1e-14):
            raise FloatingPointError("Lifted P1 denominator is numerically zero.")
        n = len(u)
        phi1 = cf[:, 1:1+n] / denominator[:, None]
        phi2 = cf[:, 1+n:]
        kernel_term = np.exp(-1j * u * math.log(float(K))) / (1j * u)
        quadrature = self.quad_weights * np.exp(self.nodes)
        P1 = 0.5 + np.real(phi1 * kernel_term.reshape(1, -1)) @ quadrature / np.pi
        P2 = 0.5 + np.real(phi2 * kernel_term.reshape(1, -1)) @ quadrature / np.pi
        tol = 3e-3
        if np.any(~np.isfinite(P1)) or np.any(~np.isfinite(P2)) or np.any(P1 < -tol) or np.any(P1 > 1 + tol) or np.any(P2 < -tol) or np.any(P2 > 1 + tol):
            raise FloatingPointError("Lifted Fourier probabilities violate probability bounds.")
        return P1, P2

    def _delta_coefficients(
        self,
        tau: float,
        current_time: float,
        params: LiftedHestonParams,
        curve: ForwardVarianceInputCurve,
    ) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """Cache state-independent Riccati coefficients used by pathwise model delta."""
        key = (
            round(float(tau), 12), round(float(current_time), 12),
            *(round(float(v), 12) for v in (params.kappa, params.theta, params.xi, params.rho, params.v0)),
            tuple(np.round(curve.maturities, 12)),
            tuple(np.round(curve.forward_variances, 12)),
        )
        cached = self._delta_coefficient_cache.get(key)
        if cached is not None:
            return cached
        u = self.nodes.astype(np.complex128)
        z = np.concatenate([np.array([1.0 + 0.0j]), 1.0 + 1j * u])
        phi, psi = self._coefficients(z, tau, current_time, params, curve)
        value = (phi[:1], psi[:1], phi[1:], psi[1:])
        if len(self._delta_coefficient_cache) >= self.delta_cache_size:
            self._delta_coefficient_cache.pop(next(iter(self._delta_coefficient_cache)))
        self._delta_coefficient_cache[key] = value
        return value

    def call_prices_slice(
        self,
        S: float,
        strikes: np.ndarray,
        tau: float,
        r: float,
        q: float,
        params: LiftedHestonParams,
        curve: ForwardVarianceInputCurve,
    ) -> np.ndarray:
        """Price all strikes for one expiry from one combined Riccati solve."""
        K = np.asarray(strikes, dtype=float).reshape(-1)
        if np.any(K <= 0) or S <= 0:
            raise ValueError("Spot and strikes must be positive.")
        if tau <= 1e-10:
            return np.maximum(float(S) - K, 0.0)
        u = self.nodes.astype(np.complex128)
        z = np.concatenate([np.array([1.0 + 0.0j]), 1.0 + 1j * u, 1j * u])
        phi, _ = self._coefficients(z, tau, 0.0, params, curve)
        x = math.log(float(S)) + (float(r) - float(q)) * tau
        cf = np.exp(x * z + phi)
        denominator = cf[0]
        if abs(denominator) < 1e-14:
            raise FloatingPointError("Lifted P1 denominator is numerically zero.")
        n = len(u)
        phi1 = cf[1:1+n] / denominator
        phi2 = cf[1+n:]
        fourier_kernel = np.exp(-1j * np.outer(np.log(K), u)) / (1j * u.reshape(1, -1))
        quadrature = self.quad_weights * np.exp(self.nodes)
        P1 = 0.5 + np.real(fourier_kernel * phi1.reshape(1, -1)) @ quadrature / np.pi
        P2 = 0.5 + np.real(fourier_kernel * phi2.reshape(1, -1)) @ quadrature / np.pi
        tolerance = 3e-3
        if np.any(P1 < -tolerance) or np.any(P1 > 1.0 + tolerance) or np.any(P2 < -tolerance) or np.any(P2 > 1.0 + tolerance):
            raise FloatingPointError("Lifted Fourier probabilities violate numerical bounds.")
        price = float(S) * math.exp(-float(q) * tau) * P1 - K * math.exp(-float(r) * tau) * P2
        lower = np.maximum(float(S) * math.exp(-float(q) * tau) - K * math.exp(-float(r) * tau), 0.0)
        upper = float(S) * math.exp(-float(q) * tau)
        price_tolerance = 3e-3 * max(1.0, upper)  # coherent with Fourier probability tolerance; numeric quadrature guard, not pricing substitute
        if np.any(~np.isfinite(price)) or np.any(price < lower - price_tolerance) or np.any(price > upper + price_tolerance):
            raise FloatingPointError("Lifted calls violate no-arbitrage price bounds.")
        return np.asarray(price, dtype=float)

    def call_price_one(self, S: float, K: float, T: float, r: float, q: float, params: LiftedHestonParams, curve: ForwardVarianceInputCurve) -> float:
        return float(self.call_prices_slice(S, np.array([K]), T, r, q, params, curve)[0])

    def call_prices(self, S, K, T, r, q, params: LiftedHestonParams, curve: ForwardVarianceInputCurve) -> np.ndarray:
        arrays = [np.asarray(v, dtype=float).reshape(-1) for v in (S, K, T, r, q)]
        n = len(arrays[1])
        if any(len(v) != n for v in arrays):
            raise ValueError("Lifted pricing arrays must have equal length.")
        output = np.empty(n, dtype=float)
        groups: dict[tuple[float, float, float, float], list[int]] = {}
        for i in range(n):
            key = (float(arrays[0][i]), float(arrays[2][i]), float(arrays[3][i]), float(arrays[4][i]))
            groups.setdefault(key, []).append(i)
        for (spot, maturity, rate, dividend), indices in groups.items():
            idx = np.asarray(indices, dtype=int)
            output[idx] = self.call_prices_slice(spot, arrays[1][idx], maturity, rate, dividend, params, curve)
        return output

    def call_delta_state_batch(self, S: np.ndarray, K: float, tau: float, r: float, q: float, params: LiftedHestonParams, curve: ForwardVarianceInputCurve, U: np.ndarray, current_time: float) -> np.ndarray:
        """Pathwise call delta from the lifted Riccati/Fourier P1 coefficient.

        Raw Fourier probabilities are not used as a hard validation gate here. The
        model-delta policy applies the final finite/no-arbitrage delta gate after
        analytic evaluation and same-model price finite-difference recovery.
        """
        S = np.asarray(S, dtype=float).reshape(-1)
        U = np.asarray(U, dtype=float)
        if U.shape != (len(S), self.kernel.n_factors):
            raise ValueError("Lifted state matrix U has incompatible shape.")
        if tau <= 1e-10:
            return np.where(S > K, 1.0, np.where(S < K, 0.0, 0.5))
        phi_den, psi_den, phi_num, psi_num = self._delta_coefficients(tau, current_time, params, curve)
        x = np.log(S) + (float(r) - float(q)) * tau
        den_loading = psi_den * self.weights.reshape(1, -1)
        num_loading = psi_num * self.weights.reshape(1, -1)
        denominator = np.exp(x + phi_den[0] + U @ den_loading[0])
        if np.any(np.abs(denominator) < 1e-14):
            raise FloatingPointError("Lifted P1 denominator is numerically zero.")
        u = self.nodes.astype(np.complex128)
        z_num = 1.0 + 1j * u
        numerator = np.exp(np.outer(x, z_num) + phi_num.reshape(1, -1) + U @ num_loading.T)
        phi1 = numerator / denominator[:, None]
        kernel_term = np.exp(-1j * u * math.log(float(K))) / (1j * u)
        quadrature = self.quad_weights * np.exp(self.nodes)
        P1 = 0.5 + np.real(phi1 * kernel_term.reshape(1, -1)) @ quadrature / np.pi
        if np.any(~np.isfinite(P1)):
            raise FloatingPointError("Lifted P1 is non-finite.")
        return math.exp(-float(q) * max(float(tau), 0.0)) * P1

def lifted_calibration_parameter_names(cfg: ExperimentConfig) -> tuple[str, ...]:
    mode = getattr(cfg.lifted, "calibration_parameterization", "full_5p_forward_curve_init")
    if mode == "full_5p_forward_curve_init":
        return ("kappa", "theta", "xi", "rho", "v0")
    raise ValueError(f"Unknown lifted calibration parameterization: {mode!r}")


def effective_lifted_curve(base_curve: ForwardVarianceInputCurve, params: LiftedHestonParams, cfg: ExperimentConfig | None = None) -> ForwardVarianceInputCurve:
    """Return the deterministic input curve used by pricing, simulation and deltas.

    Final specification: keep the regularised forward-variance curve shape, then
    let calibrated v0 and theta set the short- and long-end levels. This keeps the
    smoothed forward-variance information without forcing the five-parameter
    reduction used in the paper's narrower ATM-skew illustration.
    """
    mode = getattr(cfg.lifted, "calibration_parameterization", "full_5p_forward_curve_init") if cfg is not None else "full_5p_forward_curve_init"
    if mode == "full_5p_forward_curve_init":
        return base_curve.with_endpoint_levels(params.v0, params.theta)
    raise ValueError(f"Unknown lifted calibration parameterization: {mode!r}")

# Final lifted specification audit after parameter-name helper is available.
lifted_specification_audit = pd.DataFrame([{
    "audit_item": "factor count and grid",
    "notebook_setting": f"N={CFG.lifted.n_factors}, rN={CFG.lifted.rn_scale}",
    "status": "PASS" if int(CFG.lifted.n_factors) == 20 and np.isclose(float(CFG.lifted.rn_scale), 2.5) else "REVIEW",
}, {
    "audit_item": "calibration degrees of freedom",
    "notebook_setting": f"{CFG.lifted.calibration_parameterization}: {', '.join(lifted_calibration_parameter_names(CFG))}",
    "status": "PASS" if lifted_calibration_parameter_names(CFG) == ("kappa", "theta", "xi", "rho", "v0") else "REVIEW",
}, {
    "audit_item": "forward-variance smoothing",
    "notebook_setting": "regularised near-ATM xi0 curve shapes lifted deterministic input and initialises endpoint levels",
    "status": "PASS",
}, {
    "audit_item": "roughness handling",
    "notebook_setting": f"fixed H={CFG.lifted.H}; N=20 lifted Markovian approximation",
    "status": "PASS",
}])
display(lifted_specification_audit)


class LiftedHestonCalibrator:
    def __init__(self, pricer: LiftedRiccatiFourierPricer, cfg: ExperimentConfig, curve: ForwardVarianceInputCurve):
        self.pricer = pricer
        self.cfg = cfg
        self.curve = curve
        self.param_names = lifted_calibration_parameter_names(cfg)
        self.lower = np.array([cfg.calibration.heston_bounds[name][0] for name in self.param_names], dtype=float)
        self.upper = np.array([cfg.calibration.heston_bounds[name][1] for name in self.param_names], dtype=float)
        self._invalid_eval_count = 0
        self._last_invalid_reason = ""
        self._start_count = 0
        self._valid_start_count = 0

    def _params(self, x: np.ndarray) -> LiftedHestonParams:
        x = np.asarray(x, dtype=float)
        if x.shape != (len(self.param_names),):
            raise ValueError("Lifted candidate has wrong parameter dimension.")
        values = dict(zip(self.param_names, map(float, x)))
        values.setdefault("theta", float(self.curve.forward_variances[-1]))
        values.setdefault("v0", float(self.curve.xi0(0.0)))
        return LiftedHestonParams(float(values["kappa"]), float(values["theta"]), float(values["xi"]), float(values["rho"]), float(values["v0"]))

    def _priced_candidate(self, x: np.ndarray, target: HestonMarketTarget) -> tuple[LiftedHestonParams, np.ndarray, np.ndarray, float]:
        x = np.asarray(x, dtype=float)
        if x.shape != (len(self.param_names),) or np.any(~np.isfinite(x)) or np.any(x < self.lower) or np.any(x > self.upper):
            raise ValueError("Lifted candidate is outside parameter bounds or non-finite.")
        params = self._params(x)
        curve = effective_lifted_curve(self.curve, params, self.cfg)
        model = self.pricer.call_prices(target.S, target.K, target.T, target.r, target.q, params, curve)
        if np.any(~np.isfinite(model)):
            raise FloatingPointError("Lifted pricing produced non-finite model prices.")
        denom = np.maximum(np.abs(target.vegas), self.cfg.lifted.calibration_vega_floor)
        vega_error = (model - target.market_prices) / denom
        if np.any(~np.isfinite(vega_error)):
            raise FloatingPointError("Lifted vega-scaled residuals are non-finite.")
        loss = float(np.average(vega_error ** 2, weights=target.weights))
        if not np.isfinite(loss):
            raise FloatingPointError("Lifted candidate loss is non-finite.")
        return params, model, vega_error, loss


    def _initial_candidate_grid(self, heston_initial: HestonParams) -> list[np.ndarray]:
        curve_v0 = float(self.curve.xi0(0.0))
        curve_theta = float(self.curve.forward_variances[-1])
        if self.param_names == ("kappa", "theta", "xi", "rho", "v0"):
            raw = [
                np.array([heston_initial.kappa, heston_initial.theta, heston_initial.xi, heston_initial.rho, heston_initial.v0], dtype=float),
                np.array([heston_initial.kappa, curve_theta, min(heston_initial.xi, 0.35), np.clip(heston_initial.rho, -0.75, -0.05), curve_v0]),
                np.array([1.0, curve_theta, 0.25, -0.50, curve_v0]),
                np.array([0.5, curve_theta, 0.15, -0.25, curve_v0]),
                np.array([2.0, curve_theta, 0.20, -0.70, curve_v0]),
                np.array([5.0, curve_theta, 0.10, -0.50, curve_v0]),
            ]
        else:
            base = np.array([heston_initial.kappa, heston_initial.xi, heston_initial.rho], dtype=float)
            raw = [base, np.array([base[0], min(base[1], 0.35), np.clip(base[2], -0.75, -0.05)]), np.array([1.0, 0.25, -0.50]), np.array([0.5, 0.15, -0.25]), np.array([2.0, 0.20, -0.70]), np.array([5.0, 0.10, -0.50])]
        seeds: list[np.ndarray] = []
        seen: set[tuple[float, ...]] = set()
        for cand in raw:
            x = np.asarray(cand, dtype=float)
            if x.shape != (len(self.param_names),) or np.any(~np.isfinite(x)):
                continue
            x = np.clip(x, self.lower + 1e-8, self.upper - 1e-8)
            key = tuple(np.round(x, 10))
            if key not in seen:
                seeds.append(x)
                seen.add(key)
        if not seeds:
            raise ValueError("No finite lifted calibration starting candidates could be constructed.")
        return seeds

    def _residuals(self, x: np.ndarray, target: HestonMarketTarget) -> np.ndarray:
        try:
            _, _, vega_error, loss = self._priced_candidate(x, target)
            residual = np.sqrt(target.weights) * vega_error
            if np.any(~np.isfinite(residual)):
                raise FloatingPointError("Lifted least-squares residual is non-finite.")
            return residual
        except (FloatingPointError, OverflowError, ValueError, np.linalg.LinAlgError) as exc:
            self._invalid_eval_count += 1
            self._last_invalid_reason = str(exc)[:500]
            return np.full_like(target.market_prices, 1e6, dtype=float)

    def _build_result_from_candidate(self, x: np.ndarray, target: HestonMarketTarget, started: float, opt: Any, candidate_source: str) -> LiftedCalibrationResult:
        params, model, vega_error, loss = self._priced_candidate(x, target)
        fit = pd.DataFrame({
            "date": target.date, "profile": target.profile, "strike": target.K, "T": target.T,
            "spot_ref": target.S, "r": target.r, "q": target.q, "market_price": target.market_prices,
            "model_price": model, "price_error": model - target.market_prices,
            "vega_scaled_error": vega_error, "fit_weight": target.weights,
        })
        candidate_is_final = candidate_source == "optimizer_final"
        usable_success = bool(np.isfinite(loss) and candidate_is_final and bool(getattr(opt, "success", False)))
        return LiftedCalibrationResult(
            date=pd.Timestamp(target.date).normalize(),
            profile=target.profile,
            params=params,
            loss=loss,
            fit_table=fit,
            timing={"cpu_seconds": time.perf_counter() - started},
            optimizer={
                "success": usable_success,
                "raw_optimizer_success": bool(getattr(opt, "success", False)),
                "final_candidate_was_valid": bool(candidate_is_final),
                "candidate_source": candidate_source,
                "invalid_residual_evaluations": int(self._invalid_eval_count),
                "last_invalid_reason": self._last_invalid_reason,
                "status": int(getattr(opt, "status", -999)),
                "message": str(getattr(opt, "message", "")),
                "nfev": int(getattr(opt, "nfev", 0)),
                "start_count": int(getattr(self, "_start_count", 0)),
                "valid_start_count": int(getattr(self, "_valid_start_count", 0)),
                "calibrated_parameters": ",".join(self.param_names),
                "calibration_parameterization": getattr(self.cfg.lifted, "calibration_parameterization", "full_5p_forward_curve_init"),
            },
        )

    def calibrate(self, target: HestonMarketTarget, heston_initial: HestonParams) -> LiftedCalibrationResult:
        self._invalid_eval_count = 0
        self._last_invalid_reason = ""
        started = time.perf_counter()
        seeds = self._initial_candidate_grid(heston_initial)
        self._start_count = len(seeds)
        self._valid_start_count = 0
        valid_results: list[LiftedCalibrationResult] = []
        failures: list[str] = []
        last_opt: Any | None = None

        for start_idx, x0 in enumerate(seeds, start=1):
            try:
                _, _, _, seed_loss = self._priced_candidate(x0, target)
                self._valid_start_count += 1
            except (FloatingPointError, OverflowError, ValueError, np.linalg.LinAlgError) as exc:
                self._invalid_eval_count += 1
                self._last_invalid_reason = str(exc)[:500]
                failures.append(f"start_{start_idx}_initial: {str(exc)[:300]}")
                continue

            try:
                opt = least_squares(lambda x: self._residuals(x, target), x0=x0, bounds=(self.lower, self.upper), max_nfev=self.cfg.lifted.optimizer_max_nfev, xtol=1e-7, ftol=1e-7, gtol=1e-7, verbose=0)
                last_opt = opt
                if opt.x is not None and np.all(np.isfinite(opt.x)) and bool(opt.success):
                    try:
                        valid_results.append(self._build_result_from_candidate(np.asarray(opt.x, dtype=float), target, started, opt, "optimizer_final"))
                    except (FloatingPointError, OverflowError, ValueError, np.linalg.LinAlgError) as exc:
                        failures.append(f"start_{start_idx}_final: {str(exc)[:300]}")
                        self._last_invalid_reason = str(exc)[:500]
                elif opt.x is not None and np.all(np.isfinite(opt.x)):
                    failures.append(f"start_{start_idx}_optimizer_status: {getattr(opt, 'message', 'optimizer did not converge')}")
            except (FloatingPointError, OverflowError, ValueError, np.linalg.LinAlgError, RuntimeError) as exc:
                failures.append(f"start_{start_idx}_optimizer: {str(exc)[:300]}")
                self._last_invalid_reason = str(exc)[:500]
                continue


        if valid_results:
            valid_results.sort(key=lambda result: float(result.loss))
            return valid_results[0]
        raise LiftedCalibrationNumericalFailure(f"No finite lifted calibration candidate for {pd.Timestamp(target.date).date()}. Invalid evaluations={self._invalid_eval_count}. Failures={failures[:8]}. Last invalid reason={self._last_invalid_reason}")


def lifted_curve_for_date(forward_curve: pd.DataFrame, date: pd.Timestamp) -> ForwardVarianceInputCurve:
    d = pd.Timestamp(date).normalize()
    day = forward_curve.loc[pd.to_datetime(forward_curve["date"]).dt.normalize().eq(d)]
    if day.empty:
        raise KeyError(f"No forward-variance curve for {d.date()}.")
    return ForwardVarianceInputCurve.from_frame(day)


def _lifted_result_to_rows(result: LiftedCalibrationResult, cfg: ExperimentConfig, kernel: LiftedKernelGrid) -> tuple[dict[str, Any], pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    fit = result.fit_table.copy()
    vrmse = float(np.sqrt(np.average(fit["vega_scaled_error"] ** 2, weights=fit["fit_weight"])))
    prmse = float(np.sqrt(np.mean(fit["price_error"] ** 2)))
    quality = "OK" if result.optimizer["success"] and vrmse <= cfg.lifted.fit_vega_rmse_warn else ("WARN" if vrmse <= cfg.lifted.fit_vega_rmse_bad else "BADFIT")
    row = {
        "date": result.date,
        "profile": result.profile,
        "kappa": result.params.kappa,
        "theta": result.params.theta,
        "xi": result.params.xi,
        "rho": result.params.rho,
        "v0": result.params.v0,
        "H": cfg.lifted.H,
        "n_factors": kernel.n_factors,
        "rn_scale": cfg.lifted.rn_scale,
        "loss": result.loss,
        "vega_scaled_rmse": vrmse,
        "price_rmse": prmse,
        "success": result.optimizer["success"],
        "raw_optimizer_success": result.optimizer.get("raw_optimizer_success", result.optimizer["success"]),
        "final_candidate_was_valid": result.optimizer.get("final_candidate_was_valid", True),
        "candidate_source": result.optimizer.get("candidate_source", "optimizer_final"),
        "invalid_residual_evaluations": result.optimizer.get("invalid_residual_evaluations", 0),
        "last_invalid_reason": result.optimizer.get("last_invalid_reason", ""),
        "quality_flag": quality,
        "optimizer_nfev": result.optimizer["nfev"],
        "calibration_backend": LIFTED_CALIBRATION_BACKEND,
        "forward_curve_regularization_version": getattr(cfg.market, "forward_variance_estimator_version", "unknown"),
        "forward_curve_smoothness": getattr(cfg.market, "forward_variance_smoothness", np.nan),
        "forward_curve_curvature_smoothness": getattr(cfg.market, "forward_variance_curvature_smoothness", np.nan),
        "calibration_parameterization": getattr(cfg.lifted, "calibration_parameterization", "full_5p_forward_curve_init"),
        "calibrated_parameters": ",".join(lifted_calibration_parameter_names(cfg)),
        **result.timing,
    }
    history = pd.DataFrame([{"date": result.date, **result.optimizer, **result.timing, "loss": result.loss}])
    timing = {"date": result.date, **result.timing, "optimizer_nfev": result.optimizer["nfev"]}
    return row, fit, history, timing


def load_lifted_calibration_artifacts(cfg: ExperimentConfig, kernel: LiftedKernelGrid):
    deps = [globals().get("SPX_HEDGING_CALIBRATION_PANEL_PATH", cfg.paths.surface_cache_path), cfg.paths.heston_calibration_path, cfg.paths.forward_variance_path]
    required_panel = {
        "date", "kappa", "theta", "xi", "rho", "v0", "H", "n_factors", "rn_scale",
        "calibration_backend", "quality_flag", "calibration_parameterization",
        "calibrated_parameters", "forward_curve_regularization_version",
    }
    panel = load_dataframe_cache(
        cfg.paths.lifted_calibration_path,
        "lifted_hedging_panel_calibration",
        "lifted",
        cfg,
        required_columns=required_panel,
        dependencies=deps,
    )
    fit = load_dataframe_cache(
        cfg.paths.lifted_fit_path,
        "lifted_fit_table",
        "lifted",
        cfg,
        required_columns={"date", "model_price", "price_error", "vega_scaled_error"},
        dependencies=[cfg.paths.lifted_calibration_path],
    )
    hist = load_dataframe_cache(
        cfg.paths.lifted_history_path,
        "lifted_history",
        "lifted",
        cfg,
        dependencies=[cfg.paths.lifted_calibration_path],
    )
    timing = load_dataframe_cache(
        cfg.paths.lifted_timing_path,
        "lifted_timing",
        "lifted",
        cfg,
        dependencies=[cfg.paths.lifted_calibration_path],
    )

    # Semantic validation for legacy-hash compatibility. This is the real guardrail:
    # diagnostic-only config edits may move the old hash, but the fitted calibration must
    # still be the exact same model specification.
    expected_backend = {LIFTED_CALIBRATION_BACKEND}
    observed_backend = set(panel["calibration_backend"].dropna().astype(str).unique())
    if observed_backend != expected_backend:
        raise ValueError(f"Lifted cache uses a stale model convention: {sorted(observed_backend)}")

    expected_parameterization = getattr(cfg.lifted, "calibration_parameterization", "full_5p_forward_curve_init")
    observed_parameterization = set(panel["calibration_parameterization"].dropna().astype(str).unique())
    if observed_parameterization != {expected_parameterization}:
        raise ValueError(
            "Lifted cache calibration_parameterization does not match the methodology: "
            f"observed={sorted(observed_parameterization)}, expected={expected_parameterization!r}"
        )

    expected_parameters = ",".join(lifted_calibration_parameter_names(cfg))
    observed_parameters = set(panel["calibrated_parameters"].dropna().astype(str).unique())
    if observed_parameters != {expected_parameters}:
        raise ValueError(
            "Lifted cache calibrated parameter set does not match the methodology: "
            f"observed={sorted(observed_parameters)}, expected={expected_parameters!r}"
        )

    expected_fwd_version = getattr(cfg.market, "forward_variance_estimator_version", "unknown")
    observed_fwd_version = set(panel["forward_curve_regularization_version"].dropna().astype(str).unique())
    if observed_fwd_version != {expected_fwd_version}:
        raise ValueError(
            "Lifted cache forward-variance regularisation does not match the methodology: "
            f"observed={sorted(observed_fwd_version)}, expected={expected_fwd_version!r}"
        )

    if not np.allclose(panel["H"].astype(float), float(cfg.lifted.H)):
        raise ValueError("Lifted cache H does not match the core methodology.")
    if set(panel["n_factors"].astype(int).dropna().unique()) != {int(kernel.n_factors)}:
        raise ValueError("Lifted cache n_factors does not match the core methodology.")
    if not np.allclose(panel["rn_scale"].astype(float), float(cfg.lifted.rn_scale)):
        raise ValueError("Lifted cache rn_scale does not match the core methodology.")

    return panel.sort_values("date").reset_index(drop=True), fit, hist, timing


def _empty_lifted_fit_table() -> pd.DataFrame:
    return pd.DataFrame(columns=[
        "date", "profile", "strike", "T", "spot_ref", "r", "q", "market_price", "model_price",
        "price_error", "vega_scaled_error", "fit_weight",
    ])


def _lifted_numerical_failure_frames(
    date: pd.Timestamp,
    exc: BaseException,
    cfg: ExperimentConfig,
    kernel: LiftedKernelGrid,
    started: float,
) -> tuple[dict[str, Any], pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    reason = str(exc)[:1000]
    timing = {"date": pd.Timestamp(date).normalize(), "cpu_seconds": time.perf_counter() - started, "optimizer_nfev": 0, "failure_reason": reason}
    row = {
        "date": pd.Timestamp(date).normalize(),
        "profile": "hedging_focused_near_atm",
        "kappa": np.nan,
        "theta": np.nan,
        "xi": np.nan,
        "rho": np.nan,
        "v0": np.nan,
        "H": cfg.lifted.H,
        "n_factors": kernel.n_factors,
        "rn_scale": cfg.lifted.rn_scale,
        "loss": np.inf,
        "vega_scaled_rmse": np.nan,
        "price_rmse": np.nan,
        "success": False,
        "quality_flag": "NUMFAIL",
        "optimizer_nfev": 0,
        "calibration_backend": LIFTED_CALIBRATION_BACKEND,
        "forward_curve_regularization_version": getattr(cfg.market, "forward_variance_estimator_version", "unknown"),
        "forward_curve_smoothness": getattr(cfg.market, "forward_variance_smoothness", np.nan),
        "forward_curve_curvature_smoothness": getattr(cfg.market, "forward_variance_curvature_smoothness", np.nan),
        "calibration_parameterization": getattr(cfg.lifted, "calibration_parameterization", "full_5p_forward_curve_init"),
        "calibrated_parameters": ",".join(lifted_calibration_parameter_names(cfg)),
        "candidate_source": "numerical_failure",
        "invalid_residual_evaluations": np.nan,
        "last_invalid_reason": reason,
        "failure_reason": reason,
        "cpu_seconds": timing["cpu_seconds"],
    }
    history = pd.DataFrame([{**timing, "success": False, "status": -999, "message": reason, "nfev": 0, "loss": np.inf}])
    return row, _empty_lifted_fit_table(), history, timing


def run_lifted_full_surface_calibration(surface, heston_calib, kernel, cfg, forward_curve):
    if not SCIPY_AVAILABLE:
        raise RuntimeError("SciPy is required for lifted calibration.")
    deps = [globals().get("SPX_HEDGING_CALIBRATION_PANEL_PATH", cfg.paths.surface_cache_path), cfg.paths.heston_calibration_path, cfg.paths.forward_variance_path]
    prepare_checkpoint_directory(cfg.paths.lifted_calibration_parts_dir, "lifted_date_parts", "lifted", cfg, deps)
    pricer = LiftedRiccatiFourierPricer(kernel, cfg.lifted.pricing_nodes, cfg.lifted.riccati_steps, cfg.lifted.delta_coefficient_cache_size)
    hlookup = heston_calib.assign(date=pd.to_datetime(heston_calib["date"]).dt.normalize()).set_index("date")
    surface_dates = set(pd.to_datetime(surface["date"]).dt.normalize())
    heston_dates = set(pd.to_datetime(heston_calib["date"]).dt.normalize())
    curve_dates = set(pd.to_datetime(forward_curve["date"]).dt.normalize())
    dates = sorted(surface_dates & heston_dates & curve_dates)
    if not dates:
        raise ValueError("No common dates remain across surface, Heston calibration and ATM forward-variance curves.")
    print({
        "lifted_calibration_dates": len(dates),
        "excluded_missing_forward_curve": len(surface_dates - curve_dates),
        "excluded_missing_heston_initialisation": len(surface_dates - heston_dates),
    })
    if cfg.lifted.calibration_max_dates is not None:
        dates = dates[:cfg.lifted.calibration_max_dates]
    rows, fits, histories, timings = [], [], [], []
    for idx, d in enumerate(dates, 1):
        date = pd.Timestamp(d).normalize()
        base = cfg.paths.lifted_calibration_parts_dir
        paths = {name: base / f"lifted_{name}_{date:%Y%m%d}.parquet" for name in ["row", "fit", "history", "timing"]}
        artifacts = {
            "row": (f"lifted_date_row_{date:%Y%m%d}", {"date", "kappa", "theta", "xi", "rho", "v0", "quality_flag", "calibration_backend", "calibration_parameterization", "calibrated_parameters", "candidate_source"}),
            "fit": (f"lifted_date_fit_{date:%Y%m%d}", {"date", "model_price", "vega_scaled_error"}),
            "history": (f"lifted_date_history_{date:%Y%m%d}", {"date", "loss"}),
            "timing": (f"lifted_date_timing_{date:%Y%m%d}", {"date", "cpu_seconds"}),
        }
        loaded = None
        if cfg.lifted.calibration_resume_date_parts:
            try:
                loaded = {
                    name: load_dataframe_cache(paths[name], artifacts[name][0], "lifted", cfg, required_columns=artifacts[name][1], dependencies=deps)
                    for name in paths
                }
            except (FileNotFoundError, ValueError):
                loaded = None
        if loaded is not None:
            loaded_backend = set(loaded["row"].get("calibration_backend", pd.Series(dtype=str)).dropna().astype(str).unique())
            if loaded_backend != {LIFTED_CALIBRATION_BACKEND}:
                loaded = None
        if loaded is not None:
            rows.extend(loaded["row"].to_dict("records")); fits.append(loaded["fit"]); histories.append(loaded["history"]); timings.extend(loaded["timing"].to_dict("records"))
            print(f"[{idx}/{len(dates)}] {date.date()}: loaded strict date checkpoint")
            continue
        if date not in hlookup.index:
            raise KeyError(f"No Heston initialisation for {date.date()}.")
        day = surface[pd.to_datetime(surface["date"]).dt.normalize().eq(date)]
        target = make_heston_market_target(day, cfg.calibration, profile="hedging_focused_near_atm")
        hr = hlookup.loc[date]; hr = hr.iloc[0] if isinstance(hr, pd.DataFrame) else hr
        hinit = HestonParams(float(hr.kappa), float(hr.theta), float(hr.xi), float(hr.rho), float(hr.v0))
        curve = lifted_curve_for_date(forward_curve, date)
        started_date = time.perf_counter()
        try:
            result = LiftedHestonCalibrator(pricer, cfg, curve).calibrate(target, hinit)
            row, fit, history, timing = _lifted_result_to_rows(result, cfg, kernel)
        except LiftedCalibrationNumericalFailure as exc:
            row, fit, history, timing = _lifted_numerical_failure_frames(date, exc, cfg, kernel, started_date)
        rows.append(row); fits.append(fit); histories.append(history); timings.append(timing)
        frames = {"row": pd.DataFrame([row]), "fit": fit, "history": history, "timing": pd.DataFrame([timing])}
        for name, frame in frames.items():
            write_dataframe_cache(frame, paths[name], artifacts[name][0], "lifted", cfg, dependencies=deps)
        vrmse_msg = "nan" if not np.isfinite(row.get("vega_scaled_rmse", np.nan)) else f"{row['vega_scaled_rmse']:.6g}"
        print(f"[{idx}/{len(dates)}] {date.date()}: {row['quality_flag']}, vrmse={vrmse_msg}")
    panel = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
    fit = pd.concat(fits, ignore_index=True) if fits else _empty_lifted_fit_table()
    history = pd.concat(histories, ignore_index=True) if histories else pd.DataFrame()
    timing = pd.DataFrame(timings)
    if not panel["success"].fillna(False).astype(bool).any():
        raise RuntimeError("All lifted calibration dates failed numerically; no hedging experiment can be constructed.")
    panel["calibration_panel"] = globals().get("HEADLINE_EMPIRICAL_MODE", "unknown")
    write_dataframe_cache(panel, cfg.paths.lifted_calibration_path, "lifted_hedging_panel_calibration", "lifted", cfg, dependencies=deps, extra={"headline_empirical_mode": globals().get("HEADLINE_EMPIRICAL_MODE", "unknown")})
    write_dataframe_cache(fit, cfg.paths.lifted_fit_path, "lifted_fit_table", "lifted", cfg, dependencies=[cfg.paths.lifted_calibration_path])
    write_dataframe_cache(history, cfg.paths.lifted_history_path, "lifted_history", "lifted", cfg, dependencies=[cfg.paths.lifted_calibration_path])
    write_dataframe_cache(timing, cfg.paths.lifted_timing_path, "lifted_timing", "lifted", cfg, dependencies=[cfg.paths.lifted_calibration_path])
    return panel, fit, history, timing


def validate_lifted_riccati_fourier(surface, panel, kernel, cfg, forward_curve):
    """Independent checks for lifted Riccati/Fourier pricing.

    This gate validates only eligible finite calibrations. It records numerical
    failures as audit rows and then hard-fails with the rows printed, rather than
    crashing mid-loop and making a completed 17-hour calibration look unusable.
    """
    pricer = LiftedRiccatiFourierPricer(kernel, cfg.lifted.pricing_nodes, cfg.lifted.riccati_steps, cfg.lifted.delta_coefficient_cache_size)
    rows = []

    # One-factor limiting case x=0,c=1 must reproduce classical Heston.
    limit_kernel = LiftedKernelGrid(
        H=0.5, alpha=1.0, n_factors=1, rn_scale=1.0,
        lambdas=np.array([0.0]), weights=np.array([1.0]),
        grid=pd.DataFrame({"factor":[1], "lambda":[0.0], "weight":[1.0], "time_scale_days":[np.inf]}),
        diagnostics=pd.DataFrame({"t":[0.1], "abs_rel_error":[0.0]}),
    )
    limit_pricer = LiftedRiccatiFourierPricer(limit_kernel, cfg.lifted.pricing_nodes, max(cfg.lifted.riccati_steps, 320), cfg.lifted.delta_coefficient_cache_size)
    hp = HestonPricer(n_quad=cfg.calibration.heston_pricing_nodes)
    hparams = HestonParams(1.5, 0.04, 0.5, -0.7, 0.05)
    grid_t = np.linspace(1e-5, 2.0, 800)
    h_xi = hparams.theta + (hparams.v0 - hparams.theta) * np.exp(-hparams.kappa * grid_t)
    h_curve = ForwardVarianceInputCurve(grid_t, h_xi)
    lp = LiftedHestonParams(hparams.kappa, hparams.theta, hparams.xi, hparams.rho, hparams.v0)
    validation_strikes = np.array([80.0, 90.0, 100.0, 110.0, 120.0])
    for T in [7/365.25, 30/365.25, 0.5, 1.0]:
        lifted_prices = limit_pricer.call_prices_slice(100.0, validation_strikes, T, 0.02, 0.01, lp, h_curve)
        heston_prices = np.array([hp.call_price_one(100.0, float(K), T, 0.02, 0.01, hparams) for K in validation_strikes])
        for K, lifted_price, heston_price in zip(validation_strikes, lifted_prices, heston_prices):
            error = abs(float(lifted_price) - float(heston_price))
            rows.append({
                "test": "heston_limit", "date": pd.NaT, "T": T, "strike": float(K),
                "value": float(lifted_price), "reference": float(heston_price),
                "abs_error": error, "pass": error <= cfg.lifted.heston_limit_price_tolerance,
                "failure_reason": "",
            })

    panel = panel.copy()
    required_param_cols = ["kappa", "theta", "xi", "rho", "v0"]
    require_columns(panel, {"date", "quality_flag", *required_param_cols}, "lifted validation panel")
    finite_params = np.ones(len(panel), dtype=bool)
    for col in required_param_cols:
        finite_params &= np.isfinite(pd.to_numeric(panel[col], errors="coerce").to_numpy(float))
    eligible_panel = panel.loc[finite_params & panel["quality_flag"].astype(str).isin(["OK", "WARN"])].copy()
    if eligible_panel.empty:
        raise RuntimeError("No finite OK/WARN lifted calibrations are available for Riccati/Fourier validation.")

    surface_dates = set(pd.to_datetime(surface["date"]).dt.normalize())
    panel_dates = set(pd.to_datetime(eligible_panel["date"]).dt.normalize())
    common_dates = sorted(surface_dates & panel_dates)
    if not common_dates:
        raise RuntimeError("No common dates between surface and finite eligible lifted calibration rows.")
    max_dates = min(len(common_dates), max(3, int(getattr(cfg.lifted, "pricing_validation_dates", 6))))
    validation_dates = common_dates if len(common_dates) <= max_dates else list(pd.Series(common_dates).iloc[np.linspace(0, len(common_dates)-1, max_dates, dtype=int)])

    price_bound_tol = 3e-3
    for date in validation_dates:
        prows = eligible_panel[pd.to_datetime(eligible_panel["date"]).dt.normalize().eq(date)]
        if prows.empty:
            continue
        prow = prows.iloc[0]
        params = LiftedHestonParams(*[float(prow[k]) for k in required_param_cols])
        curve = effective_lifted_curve(lifted_curve_for_date(forward_curve, date), params, cfg)
        day = surface[pd.to_datetime(surface["date"]).dt.normalize().eq(date)].copy()
        if day.empty:
            rows.append({
                "test": "market_repricing", "date": date, "T": np.nan, "strike": np.nan,
                "value": np.nan, "reference": np.nan, "abs_error": np.inf, "pass": False,
                "failure_reason": "No surface rows for validation date.",
            })
            continue

        # Validate a representative near-ATM/maturity spread rather than blindly using
        # the first low-strike rows after sorting, which overweights numerical boundary cases.
        day["abs_log_moneyness"] = np.abs(np.log(day["strike"].astype(float) / day["spot_ref"].astype(float)))
        day = day.sort_values(["T", "abs_log_moneyness", "strike"])
        sample = day.head(cfg.lifted.pricing_validation_rows).copy()

        try:
            prices = pricer.call_prices(
                sample["spot_ref"].to_numpy(float), sample["strike"].to_numpy(float), sample["T"].to_numpy(float),
                sample["r"].to_numpy(float), sample["q"].to_numpy(float), params, curve,
            )
        except (FloatingPointError, OverflowError, ValueError, np.linalg.LinAlgError) as exc:
            rows.append({
                "test": "market_repricing", "date": date, "T": np.nan, "strike": np.nan,
                "value": np.nan, "reference": np.nan, "abs_error": np.inf, "pass": False,
                "failure_reason": f"batch pricing failed: {str(exc)[:500]}",
            })
            continue

        sample["validation_model_price"] = prices
        for j, r in enumerate(sample.itertuples(index=False)):
            price = float(prices[j])
            lower = max(r.spot_ref * math.exp(-r.q * r.T) - r.strike * math.exp(-r.r * r.T), 0.0)
            upper = r.spot_ref * math.exp(-r.q * r.T)
            bound_tol_abs = price_bound_tol * max(1.0, upper)
            passed = lower - bound_tol_abs <= price <= upper + bound_tol_abs
            rows.append({
                "test": "market_repricing", "date": date, "T": r.T, "strike": r.strike,
                "value": price, "reference": r.call_equiv_mid,
                "abs_error": abs(price - r.call_equiv_mid), "pass": bool(passed),
                "failure_reason": "" if passed else f"price outside bounds [{lower:.8g}, {upper:.8g}] with tol {bound_tol_abs:.8g}",
            })

        delta_sample = sample.head(int(cfg.lifted.delta_validation_rows))
        for r in list(delta_sample.itertuples(index=False)):
            U = np.zeros((1, kernel.n_factors))
            try:
                delta = float(pricer.call_delta_state_batch(np.array([r.spot_ref]), r.strike, r.T, r.r, r.q, params, curve, U, 0.0)[0])
            except (FloatingPointError, OverflowError, ValueError, np.linalg.LinAlgError) as exc:
                rows.append({
                    "test": "delta_finite_difference", "date": date, "T": r.T, "strike": r.strike,
                    "value": np.nan, "reference": np.nan, "abs_error": np.inf, "pass": False,
                    "failure_reason": f"analytic delta failed: {str(exc)[:500]}",
                })
                continue

            fd = []
            fd_failures = []
            for bf in cfg.lifted.delta_validation_bump_fracs:
                bump = max(float(bf) * r.spot_ref, 1e-5)
                try:
                    up = pricer.call_price_one(r.spot_ref + bump, r.strike, r.T, r.r, r.q, params, curve)
                    down_spot = max(r.spot_ref - bump, 1e-8)
                    down = pricer.call_price_one(down_spot, r.strike, r.T, r.r, r.q, params, curve)
                    fd.append((up - down) / ((r.spot_ref + bump) - down_spot))
                except (FloatingPointError, OverflowError, ValueError, np.linalg.LinAlgError) as exc:
                    fd_failures.append(f"bump={bf}: {str(exc)[:250]}")

            if fd:
                delta_error = min(abs(delta - x) for x in fd)
                reference = float(fd[len(fd)//2])
                passed = (
                    -cfg.lifted.delta_probability_tolerance <= delta <= math.exp(-r.q * r.T) + cfg.lifted.delta_probability_tolerance
                    and delta_error <= cfg.lifted.delta_finite_difference_tolerance
                )
                reason = "" if passed else "; ".join(fd_failures[:3])
            else:
                delta_error = np.inf
                reference = np.nan
                passed = False
                reason = "; ".join(fd_failures[:3]) or "No finite-difference bump prices succeeded."

            rows.append({
                "test": "delta_finite_difference", "date": date, "T": r.T, "strike": r.strike,
                "value": float(delta), "reference": reference, "abs_error": float(delta_error),
                "pass": bool(passed), "failure_reason": reason,
            })

    validation = pd.DataFrame(rows)
    if validation.empty:
        raise RuntimeError("Lifted Riccati/Fourier validation produced no rows.")
    if not validation["pass"].all():
        failed = validation.loc[~validation["pass"]].head(20)
        raise RuntimeError(f"Lifted Riccati/Fourier validation failed:\n{failed.to_string(index=False)}")
    return validation

lifted_pricer = LiftedRiccatiFourierPricer(
    lifted_kernel,
    CFG.lifted.pricing_nodes,
    CFG.lifted.riccati_steps,
)
if CFG.flags.FREEZE_CALIBRATION_ARTIFACTS and CFG.flags.RUN_LIFTED_CALIBRATION:
    raise RuntimeError(
        "Internal execution-plan contradiction for Lifted-Heston calibration. "
        "Restart the kernel and configure only RUN.rebuild_from in the user configuration cell."
    )
if CFG.flags.RUN_LIFTED_CALIBRATION:
    lifted_calibration, lifted_fit_table, lifted_history, lifted_timing = run_lifted_full_surface_calibration(
        surface_panel,
        heston_calibration,
        lifted_kernel,
        CFG,
        forward_variance_curve,
    )
else:
    print(f"[lifted calibration] trying validated H={CFG.lifted.H:.2f} calibration artefacts before rebuilding.", flush=True)
    try:
        lifted_calibration, lifted_fit_table, lifted_history, lifted_timing = load_lifted_calibration_artifacts(CFG, lifted_kernel)
        print("[calibration load] Loaded validated hedging-panel lifted-Heston calibration artefacts without rebuild.", flush=True)
    except (FileNotFoundError, ValueError) as exc:
        if CFG.flags.FREEZE_CALIBRATION_ARTIFACTS or not CFG.flags.ALLOW_CALIBRATION_REBUILD_FROM_DOWNSTREAM:
            raise RuntimeError(
                f"Lifted-Heston H={CFG.lifted.H:.2f} calibration cache is missing/stale and strict reuse forbids rebuilding. "
                "Set rebuild_from='calibration', an earlier stage, or use 'auto'."
            ) from exc
        print(f"[lifted calibration] cache unavailable/stale; rebuilding. Reason: {exc}", flush=True)
        lifted_calibration, lifted_fit_table, lifted_history, lifted_timing = run_lifted_full_surface_calibration(
            surface_panel,
            heston_calibration,
            lifted_kernel,
            CFG,
            forward_variance_curve,
        )

display(lifted_calibration["quality_flag"].value_counts(dropna=False))
display(lifted_calibration["vega_scaled_rmse"].describe())
print("Lifted calibration dates:", int(pd.to_datetime(lifted_calibration["date"]).dt.normalize().nunique()))



,audit_item,notebook_setting,status
0,factor count and grid,"N=20, rN=2.5",PASS
1,calibration degrees of freedom,"full_5p_forward_curve_init: kappa, theta, xi, ...",PASS
2,forward-variance smoothing,regularised near-ATM xi0 curve shapes lifted d...,PASS
3,roughness handling,fixed H=0.35; N=20 lifted Markovian approximation,PASS


[lifted calibration] trying validated H=0.35 calibration artefacts before rebuilding.
[calibration load] Loaded validated hedging-panel lifted-Heston calibration artefacts without rebuild.


quality_flag
OK    2161
Name: count, dtype: int64

count    2161.000000
mean        0.001892
std         0.001959
min         0.000130
25%         0.000807
50%         0.001221
75%         0.002077
max         0.014501
Name: vega_scaled_rmse, dtype: float64

Lifted calibration dates: 2161


In [27]:
# ============================================================
# Section 4 fit inspection: Lifted Heston
# ============================================================

LIFTED_PARAM_COLS = [c for c in ["kappa", "theta", "xi", "rho", "v0"] if c in lifted_calibration.columns]

summarise_calibration_panel(
    lifted_calibration,
    model_name="Lifted Heston",
    param_cols=LIFTED_PARAM_COLS,
)

for metric in [c for c in ["vega_scaled_rmse", "price_rmse", "loss", "cpu_seconds", "optimizer_nfev"] if c in lifted_calibration.columns]:
    plot_calibration_metric_time_series(
        lifted_calibration,
        metric=metric,
        title=f"Lifted Heston {metric} through time",
        model_name="Lifted Heston",
    )

if LIFTED_PARAM_COLS:
    plot_parameter_time_series(
        lifted_calibration,
        LIFTED_PARAM_COLS,
        title="Lifted Heston calibrated parameters",
        model_name="Lifted Heston",
    )


def enrich_fit_table_for_visual_diagnostics(
    fit_table: pd.DataFrame,
    surface: pd.DataFrame,
    *,
    market_price_col: str = "market_price",
    model_price_col: str = "model_price",
) -> pd.DataFrame:
    if fit_table is None or fit_table.empty:
        return pd.DataFrame()
    f = fit_table.copy()
    f["date"] = pd.to_datetime(f["date"], errors="coerce").dt.normalize()
    for col in ["strike", "T", market_price_col, model_price_col, "price_error", "vega_scaled_error", "fit_weight"]:
        if col in f.columns:
            f[col] = pd.to_numeric(f[col], errors="coerce")
    if surface is None or surface.empty:
        if market_price_col in f.columns and "call_equiv_mid" not in f.columns:
            f["call_equiv_mid"] = f[market_price_col]
        return f

    s = surface.copy()
    s["date"] = pd.to_datetime(s["date"], errors="coerce").dt.normalize()
    for col in ["strike", "T"]:
        if col in s.columns:
            s[col] = pd.to_numeric(s[col], errors="coerce")
    merge_cols = [c for c in ["date", "strike", "T"] if c in f.columns and c in s.columns]
    attach_cols = [
        "date", "strike", "T", "call_equiv_mid", "log_forward_moneyness", "log_moneyness",
        "forward_ref", "spot_ref", "r", "q", "iv_market", "bs_vega", "fit_weight"
    ]
    attach_cols = [c for c in attach_cols if c in s.columns]
    if len(merge_cols) < 3 or not attach_cols:
        if market_price_col in f.columns and "call_equiv_mid" not in f.columns:
            f["call_equiv_mid"] = f[market_price_col]
        return f
    attached = s[attach_cols].drop_duplicates(merge_cols)
    out = f.merge(attached, on=merge_cols, how="left", suffixes=("", "_surface"))
    if market_price_col in out.columns:
        out["call_equiv_mid"] = pd.to_numeric(out.get("call_equiv_mid"), errors="coerce").where(
            pd.to_numeric(out.get("call_equiv_mid"), errors="coerce").notna(),
            pd.to_numeric(out[market_price_col], errors="coerce"),
        )
    if "log_forward_moneyness" not in out.columns and "log_moneyness" in out.columns:
        out["log_forward_moneyness"] = pd.to_numeric(out["log_moneyness"], errors="coerce")
    return out


def summarise_fit_residual_panel(fit_table: pd.DataFrame) -> pd.DataFrame:
    if fit_table is None or fit_table.empty:
        return pd.DataFrame()
    out = pd.DataFrame({
        "metric": ["n", "price_rmse", "price_mae", "vega_scaled_rmse", "vega_scaled_mae"],
        "value": [
            len(fit_table),
            weighted_rmse(pd.to_numeric(fit_table.get("price_error"), errors="coerce")),
            float(np.nanmean(np.abs(pd.to_numeric(fit_table.get("price_error"), errors="coerce")))),
            weighted_rmse(pd.to_numeric(fit_table.get("vega_scaled_error"), errors="coerce"), pd.to_numeric(fit_table.get("fit_weight"), errors="coerce")),
            float(np.nanmean(np.abs(pd.to_numeric(fit_table.get("vega_scaled_error"), errors="coerce")))),
        ],
    })
    return out


def fit_date_ranking_table(fit_table: pd.DataFrame) -> pd.DataFrame:
    if fit_table is None or fit_table.empty:
        return pd.DataFrame()
    ranked = (
        fit_table.groupby("date")
        .apply(lambda g: pd.Series({
            "n": len(g),
            "price_rmse": weighted_rmse(pd.to_numeric(g.get("price_error"), errors="coerce")),
            "vega_scaled_rmse": weighted_rmse(pd.to_numeric(g.get("vega_scaled_error"), errors="coerce"), pd.to_numeric(g.get("fit_weight"), errors="coerce")),
            "mean_abs_error": float(np.nanmean(np.abs(pd.to_numeric(g.get("price_error"), errors="coerce")))),
        }))
        .reset_index()
        .sort_values("vega_scaled_rmse", ascending=False)
    )
    return ranked


lifted_fit_residuals_visual = enrich_fit_table_for_visual_diagnostics(
    globals().get("lifted_fit_table", pd.DataFrame()),
    globals().get("surface_panel", pd.DataFrame()),
)

if not lifted_fit_residuals_visual.empty:
    lifted_bucket_summary = residual_bucket_table(
        lifted_fit_residuals_visual,
        residual_col="vega_scaled_error",
        weight_col="fit_weight",
    )
    lifted_date_fit = fit_date_ranking_table(lifted_fit_residuals_visual)

    print("\nOverall Lifted Heston residual diagnostics:")
    display(summarise_fit_residual_panel(lifted_fit_residuals_visual))

    print("\nLifted Heston residuals by maturity and moneyness:")
    display(lifted_bucket_summary)
    plot_residual_heatmap(lifted_bucket_summary, value_col="rmse", title="Lifted Heston vega-scaled RMSE by maturity and moneyness")

    print("\nWorst 30 Lifted Heston dates by vega-scaled RMSE:")
    display(lifted_date_fit.head(30))
    if not lifted_date_fit.empty:
        worst_lifted_date = lifted_date_fit.iloc[0]["date"]
        print(f"Worst Lifted Heston fit date: {worst_lifted_date}")
        plot_model_vs_market_scatter(lifted_fit_residuals_visual, model_name="Lifted Heston", date=worst_lifted_date, market_col="call_equiv_mid", model_col="model_price")
        plot_smile_fit_for_date(lifted_fit_residuals_visual, model_name="Lifted Heston", date=worst_lifted_date)
else:
    print("Lifted Heston fit-table visual diagnostics could not be produced because the fit table is missing or empty.")


Lifted Heston calibration panel summary
Rows: 2,161
Date range: 2010-01-04 00:00:00 -> 2024-12-19 00:00:00
Unique dates: 2,161

Success rate:


,count
success,
True,2161



Quality flags:


,count
quality_flag,
OK,2161



Metric summary:


,count,mean,std,min,1%,5%,50%,95%,99%,max
loss,2161.0,0.000007,0.000019,1.698317e-08,6.105171e-08,1.469313e-07,0.000001,0.000039,0.000097,0.000210
price_rmse,2161.0,0.718019,0.708047,5.854925e-02,1.072132e-01,1.595246e-01,0.523203,1.890266,3.744308,8.477670
vega_scaled_rmse,2161.0,0.001892,0.001959,1.303195e-04,2.470823e-04,3.833162e-04,0.001221,0.006214,0.009856,0.014501
cpu_seconds,2161.0,25.600256,24.209845,2.078620e+00,2.964074e+00,4.991432e+00,19.651755,62.953876,110.070107,348.755137
optimizer_nfev,2161.0,15.713559,8.521913,6.000000e+00,7.000000e+00,8.000000e+00,13.000000,35.000000,47.400000,90.000000


... (26 outputs left out)

<Figure size 1040x520 with 1 Axes>

<Figure size 1040x520 with 1 Axes>

<Figure size 1040x520 with 1 Axes>

<Figure size 1040x520 with 1 Axes>

In [28]:
# ============================================================
# Heston vs lifted-Heston calibration diagnostics and hedging eligibility audit
# ============================================================

CALIBRATION_DIAGNOSTIC_VERSION = "common_target_date_level_v1"
CALIBRATION_QUALITY_GATE_VERSION = "calibration_availability_final_finite_5p_lifted_no_rmse_exclusion"


def _normalise_date_column(frame: pd.DataFrame, label: str) -> pd.DataFrame:
    if frame is None or frame.empty:
        raise ValueError(f"{label} is empty.")
    out = frame.copy()
    require_columns(out, {"date"}, label)
    out["date"] = pd.to_datetime(out["date"]).dt.normalize()
    return out


def _finite_numeric(frame: pd.DataFrame, column: str) -> pd.Series:
    if column not in frame.columns:
        return pd.Series(np.nan, index=frame.index, dtype=float)
    return pd.to_numeric(frame[column], errors="coerce")


def _quality_flag_is_usable(frame: pd.DataFrame) -> pd.Series:
    bad_flags = {"NUMFAIL", "FAIL", "FAILED", "BADFIT"}
    if "quality_flag" not in frame.columns:
        return pd.Series(True, index=frame.index)
    return ~frame["quality_flag"].astype(str).str.upper().isin(bad_flags)


def calibration_panel_summary(panel: pd.DataFrame, model_name: str) -> pd.DataFrame:
    panel = _normalise_date_column(panel, f"{model_name} calibration panel")
    metrics = [c for c in ["loss", "price_rmse", "price_mae", "vega_scaled_rmse", "cpu_seconds", "n_options", "n_maturities"] if c in panel.columns]
    out = panel[metrics].describe(percentiles=[0.05, 0.5, 0.95]).T if metrics else pd.DataFrame()
    out.insert(0, "model", model_name)
    return out.reset_index(names="metric")


def _normalise_one_row_per_date(panel: pd.DataFrame, model_name: str) -> pd.DataFrame:
    """Collapse calibration panel to one candidate per date by realised finite fit quality.

    Optimiser status is retained as provenance but is not a hard eligibility condition.
    The hard conditions are finite parameters, finite realised fit metric, and no
    BADFIT/NUMFAIL quality flag. This avoids the earlier bug where iteration-limit
    statuses destroyed otherwise finite calibration rows.
    """
    work = _normalise_date_column(panel, f"{model_name} calibration panel")
    for col in ["vega_scaled_rmse", "price_rmse", "loss"]:
        work[col] = _finite_numeric(work, col)

    if "quality_flag" not in work.columns:
        work["quality_flag"] = "UNSPECIFIED"
    if "success" not in work.columns:
        work["success"] = False

    param_cols = [c for c in ["kappa", "theta", "xi", "rho", "v0"] if c in work.columns]
    if param_cols:
        work["_finite_params"] = np.isfinite(work[param_cols].apply(pd.to_numeric, errors="coerce")).all(axis=1)
    else:
        work["_finite_params"] = False

    work["_usable_by_fit"] = (
        work["_finite_params"]
        & np.isfinite(work["vega_scaled_rmse"])
        & _quality_flag_is_usable(work)
    )
    work["_quality_rank"] = np.where(work["_usable_by_fit"], 0, 1)
    work = work.sort_values(["date", "_quality_rank", "vega_scaled_rmse", "loss"], na_position="last")
    out = work.drop_duplicates("date", keep="first").copy()
    return out.drop(columns=[c for c in ["_quality_rank", "_finite_params", "_usable_by_fit"] if c in out.columns]).reset_index(drop=True)


def compare_heston_lifted_calibration(heston: pd.DataFrame, lifted: pd.DataFrame) -> pd.DataFrame:
    """Date-level Heston/lifted comparison on common finite calibration dates.

    The returned table includes both naming conventions:
      heston_vega_scaled_rmse / lifted_vega_scaled_rmse
      vega_scaled_rmse_heston / vega_scaled_rmse_lifted
    so later diagnostic cells cannot silently point at the wrong summary table.
    """
    h = _normalise_one_row_per_date(heston, "Heston")
    l = _normalise_one_row_per_date(lifted, "Lifted Heston")

    metrics = [c for c in ["loss", "price_rmse", "price_mae", "vega_scaled_rmse", "cpu_seconds", "n_options"] if c in h.columns and c in l.columns]
    if not metrics:
        raise ValueError("No common calibration metrics are available for Heston/lifted comparison.")

    h_cols = ["date"] + metrics
    l_cols = ["date"] + metrics
    merged = h[h_cols].merge(l[l_cols], on="date", how="inner", suffixes=("_heston", "_lifted"))
    if merged.empty:
        raise ValueError("No common Heston/lifted calibration dates to compare.")

    for metric in metrics:
        old_h = f"{metric}_heston"
        old_l = f"{metric}_lifted"
        new_h = f"heston_{metric}"
        new_l = f"lifted_{metric}"
        merged[new_h] = pd.to_numeric(merged[old_h], errors="coerce")
        merged[new_l] = pd.to_numeric(merged[old_l], errors="coerce")
        merged[f"{metric}_diff_lifted_minus_heston"] = merged[new_l] - merged[new_h]
        merged[f"{metric}_median_pair_available"] = np.isfinite(merged[new_h]) & np.isfinite(merged[new_l])
        if metric in {"loss", "price_rmse", "price_mae", "vega_scaled_rmse"}:
            merged[f"{metric}_lifted_better"] = merged[new_l] < merged[new_h]

    merged["calibration_diagnostic_version"] = CALIBRATION_DIAGNOSTIC_VERSION
    return merged.sort_values("date").reset_index(drop=True)


def build_calibration_comparison_summary(comparison: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for metric in ["loss", "price_rmse", "price_mae", "vega_scaled_rmse", "cpu_seconds"]:
        h_col = f"heston_{metric}"
        l_col = f"lifted_{metric}"
        if h_col not in comparison.columns or l_col not in comparison.columns:
            continue
        finite = np.isfinite(comparison[h_col]) & np.isfinite(comparison[l_col])
        row = {
            "metric": metric,
            "common_dates": int(finite.sum()),
            "heston_median": float(comparison.loc[finite, h_col].median()) if finite.any() else np.nan,
            "lifted_median": float(comparison.loc[finite, l_col].median()) if finite.any() else np.nan,
            "median_diff_lifted_minus_heston": float((comparison.loc[finite, l_col] - comparison.loc[finite, h_col]).median()) if finite.any() else np.nan,
        }
        b_col = f"{metric}_lifted_better"
        if b_col in comparison.columns:
            row["share_dates_lifted_better"] = float(comparison.loc[finite, b_col].mean()) if finite.any() else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def _target_price_column(frame: pd.DataFrame, label: str) -> tuple[pd.Series, str]:
    candidates = ["market_price", "call_equiv_mid", "mid", "option_mid", "price_mid"]
    for col in candidates:
        if col in frame.columns:
            return pd.to_numeric(frame[col], errors="coerce"), col
    if {"model_price", "price_error"}.issubset(frame.columns):
        return pd.to_numeric(frame["model_price"], errors="coerce") - pd.to_numeric(frame["price_error"], errors="coerce"), "model_price_minus_price_error"
    raise ValueError(f"{label} has no recognised market target column. Expected one of {candidates} or model_price/price_error.")


def _normalise_fit_table_for_comparison(fit: pd.DataFrame, model_name: str) -> tuple[pd.DataFrame, str]:
    if fit is None or fit.empty:
        raise ValueError(f"{model_name} fit table is empty; cannot build calibration-fairness diagnostics.")
    required = {"date", "strike", "T", "model_price", "price_error", "vega_scaled_error"}
    require_columns(fit, required, f"{model_name} fit table")
    out = fit.copy()
    out["date"] = pd.to_datetime(out["date"]).dt.normalize()
    out["strike"] = pd.to_numeric(out["strike"], errors="coerce")
    out["T"] = pd.to_numeric(out["T"], errors="coerce")
    out["model_price"] = pd.to_numeric(out["model_price"], errors="coerce")
    out["price_error"] = pd.to_numeric(out["price_error"], errors="coerce")
    out["vega_scaled_error"] = pd.to_numeric(out["vega_scaled_error"], errors="coerce")
    out["market_price"], target_source = _target_price_column(out, f"{model_name} fit table")

    for col in ["spot_ref", "S", "r", "q", "fit_weight", "bs_vega", "dte", "log_moneyness", "log_forward_moneyness"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    if "spot_ref" not in out.columns and "S" in out.columns:
        out["spot_ref"] = out["S"]

    out = out[np.isfinite(out["strike"]) & np.isfinite(out["T"]) & np.isfinite(out["market_price"])].copy()
    out["strike_key"] = out["strike"].round(8)
    out["T_key"] = out["T"].round(10)
    out[f"{model_name}_model_price"] = out["model_price"]
    out[f"{model_name}_price_error"] = out["price_error"]
    out[f"{model_name}_vega_scaled_error"] = out["vega_scaled_error"]
    out[f"{model_name}_abs_vega_scaled_error"] = out["vega_scaled_error"].abs()
    return out, target_source


def _attach_surface_vega_scale(targets: pd.DataFrame, surface: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    """Attach a strict vega scale from the cleaned surface to lifted target rows.

    Priority order:
      1. cleaned-surface Black-Scholes vega matched by date/strike/T;
      2. existing target-row bs_vega, if the residual table already carries one;
      3. inferred absolute price-error / vega-scaled-error ratio.

    No unit-vega substitution is made. Rows with no defensible scale are excluded
    from fit-pair diagnostics and counted in the schema audit. Even a
    vega column can start a custody battle if you merge it carelessly.
    """
    out = targets.copy()

    if "bs_vega" in out.columns:
        out = out.rename(columns={"bs_vega": "target_bs_vega"})
    elif "target_bs_vega" not in out.columns:
        out["target_bs_vega"] = np.nan
    out["target_bs_vega"] = pd.to_numeric(out["target_bs_vega"], errors="coerce")

    if surface is not None and not surface.empty and {"date", "strike", "T", "bs_vega"}.issubset(surface.columns):
        s = surface[["date", "strike", "T", "bs_vega"]].copy()
        s["date"] = pd.to_datetime(s["date"]).dt.normalize()
        s["strike_key"] = pd.to_numeric(s["strike"], errors="coerce").round(8)
        s["T_key"] = pd.to_numeric(s["T"], errors="coerce").round(10)
        s["surface_bs_vega"] = pd.to_numeric(s["bs_vega"], errors="coerce")
        s = s[np.isfinite(s["surface_bs_vega"]) & (s["surface_bs_vega"] > 0)]
        s = s.drop_duplicates(["date", "strike_key", "T_key"])
        out = out.merge(
            s[["date", "strike_key", "T_key", "surface_bs_vega"]],
            on=["date", "strike_key", "T_key"],
            how="left",
        )
    else:
        out["surface_bs_vega"] = np.nan

    inferred_scale = np.full(len(out), np.nan, dtype=float)
    denom_ok = (
        np.isfinite(pd.to_numeric(out["lifted_price_error"], errors="coerce"))
        & np.isfinite(pd.to_numeric(out["lifted_vega_scaled_error"], errors="coerce"))
        & (pd.to_numeric(out["lifted_vega_scaled_error"], errors="coerce").abs() > 1e-14)
    )
    inferred_scale[denom_ok.to_numpy()] = np.abs(
        pd.to_numeric(out.loc[denom_ok, "lifted_price_error"], errors="coerce")
        / pd.to_numeric(out.loc[denom_ok, "lifted_vega_scaled_error"], errors="coerce")
    )

    surface_scale = pd.to_numeric(out["surface_bs_vega"], errors="coerce")
    target_scale = pd.to_numeric(out["target_bs_vega"], errors="coerce")
    out["vega_scale"] = surface_scale
    out["vega_scale_source"] = "surface_bs_vega"

    missing = ~np.isfinite(out["vega_scale"]) | (out["vega_scale"] <= 0)
    out.loc[missing, "vega_scale"] = target_scale[missing]
    out.loc[missing & np.isfinite(target_scale) & (target_scale > 0), "vega_scale_source"] = "target_bs_vega"

    missing = ~np.isfinite(out["vega_scale"]) | (out["vega_scale"] <= 0)
    out.loc[missing, "vega_scale"] = inferred_scale[missing.to_numpy()]
    out.loc[missing & np.isfinite(inferred_scale) & (inferred_scale > 0), "vega_scale_source"] = "inferred_from_lifted_residual_ratio"

    out["vega_scale"] = pd.to_numeric(out["vega_scale"], errors="coerce")
    out.loc[~np.isfinite(out["vega_scale"]) | (out["vega_scale"] <= 0), "vega_scale_source"] = "missing"
    return out

def _reprice_heston_on_lifted_targets(lifted_fit: pd.DataFrame, heston_panel: pd.DataFrame, surface: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    l, lifted_target_source = _normalise_fit_table_for_comparison(lifted_fit, "lifted")
    h = _normalise_one_row_per_date(heston_panel, "Heston")
    param_cols = ["date", "kappa", "theta", "xi", "rho", "v0"]
    require_columns(h, set(param_cols), "Heston calibration panel for common-target repricing")
    h = h[param_cols].copy()
    for col in param_cols[1:]:
        h[col] = pd.to_numeric(h[col], errors="coerce")
    h = h[np.isfinite(h[param_cols[1:]]).all(axis=1)].copy()
    if h.empty:
        raise ValueError("No finite Heston parameter rows available for common-target repricing.")

    common = l.merge(h, on="date", how="inner", suffixes=("", "_heston_param"))
    if common.empty:
        raise ValueError("No lifted fit target rows have matching finite Heston calibration parameters.")

    required_target_cols = {"spot_ref", "strike", "T", "r", "q", "market_price", "lifted_model_price", "lifted_price_error", "lifted_vega_scaled_error"}
    missing = required_target_cols - set(common.columns)
    if missing:
        raise ValueError(f"Lifted fit table lacks target columns required for common-target Heston repricing: {sorted(missing)}")

    common = _attach_surface_vega_scale(common, surface, cfg)
    before = len(common)
    common = common[np.isfinite(common["vega_scale"]) & (common["vega_scale"] > 0)].copy().reset_index(drop=True)
    if common.empty:
        raise ValueError("No common-target rows have a valid vega scale for Heston-vs-lifted fit diagnostics.")

    pricer = HestonPricer(cfg.calibration.heston_pricing_nodes)
    heston_prices = np.empty(len(common), dtype=float)
    # Grouping by date avoids repeatedly rebuilding Heston parameter objects.
    for date, idx in common.groupby("date", sort=False).groups.items():
        pos = np.asarray(idx, dtype=int)
        g = common.iloc[pos]
        first = g.iloc[0]
        params = HestonParams(float(first.kappa), float(first.theta), float(first.xi), float(first.rho), float(first.v0))
        heston_prices[pos] = pricer.call_prices(
            g["spot_ref"].to_numpy(float),
            g["strike"].to_numpy(float),
            g["T"].to_numpy(float),
            g["r"].to_numpy(float),
            g["q"].to_numpy(float),
            params,
        )

    common["heston_model_price"] = heston_prices
    common["heston_price_error"] = common["heston_model_price"] - common["market_price"]
    common["heston_vega_scaled_error"] = common["heston_price_error"] / common["vega_scale"]
    common["heston_abs_vega_scaled_error"] = common["heston_vega_scaled_error"].abs()
    common["lifted_abs_vega_scaled_error"] = common["lifted_vega_scaled_error"].abs()
    common["fit_target_match_method"] = "heston_repriced_on_lifted_targets"
    common["lifted_target_source"] = lifted_target_source
    common["rows_excluded_missing_vega_scale"] = int(before - len(common))
    return common

def build_calibration_fit_pair_diagnostics(
    heston_fit: pd.DataFrame,
    lifted_fit: pd.DataFrame,
    surface: pd.DataFrame,
    heston_panel: pd.DataFrame,
    cfg: ExperimentConfig,
) -> pd.DataFrame:
    """Build row-level fit diagnostics on a common calibration target.

    Primary policy: exact saved residual match by date/strike/T. If that has no
    overlap because caches store Heston and lifted residual grids differently, use
    the lifted fit table as the common target and reprice those exact contracts
    under the stored Heston parameters. This is a deterministic common-target
    comparison, not a substitute calibration.
    """
    h, heston_target_source = _normalise_fit_table_for_comparison(heston_fit, "heston")
    l, lifted_target_source = _normalise_fit_table_for_comparison(lifted_fit, "lifted")
    merge_cols = ["date", "strike_key", "T_key"]
    exact = h.merge(l, on=merge_cols, how="inner", suffixes=("_h", "_l"))
    if not exact.empty:
        common = exact.copy()
        # Harmonise columns that were suffixed by the exact Heston/lifted residual merge.
        # Without this, near-ATM and short-maturity diagnostics can silently become empty.
        # Truly majestic: the comparison exists, then vanishes because pandas added _h.
        for base, preferred in {
            "strike": ["strike_h", "strike_l"],
            "T": ["T_h", "T_l"],
            "spot_ref": ["spot_ref_h", "spot_ref_l", "S_h", "S_l"],
            "r": ["r_h", "r_l"],
            "q": ["q_h", "q_l"],
            "dte": ["dte_h", "dte_l"],
            "log_moneyness": ["log_moneyness_h", "log_moneyness_l"],
            "log_forward_moneyness": ["log_forward_moneyness_h", "log_forward_moneyness_l"],
            "bs_vega": ["bs_vega_h", "bs_vega_l"],
            "fit_weight": ["fit_weight_h", "fit_weight_l"],
        }.items():
            if base not in common.columns:
                for candidate in preferred:
                    if candidate in common.columns:
                        common[base] = common[candidate]
                        break
        common["market_price"] = common.get("market_price_h", common.get("market_price_l"))
        if {"market_price_h", "market_price_l"}.issubset(common.columns):
            common["market_price_abs_diff_h_l"] = (
                pd.to_numeric(common["market_price_h"], errors="coerce")
                - pd.to_numeric(common["market_price_l"], errors="coerce")
            ).abs()
        common["fit_target_match_method"] = "exact_saved_fit_table_match"
        common["heston_target_source"] = heston_target_source
        common["lifted_target_source"] = lifted_target_source
    else:
        common = _reprice_heston_on_lifted_targets(lifted_fit, heston_panel, surface, cfg)
        common["heston_target_source"] = "repriced_from_heston_parameters"

    if "log_forward_moneyness" in common.columns:
        common["abs_log_moneyness"] = common["log_forward_moneyness"].abs()
    elif "log_moneyness" in common.columns:
        common["abs_log_moneyness"] = common["log_moneyness"].abs()
    elif {"strike", "spot_ref"}.issubset(common.columns):
        common["abs_log_moneyness"] = np.abs(np.log(common["strike"] / common["spot_ref"]))
    else:
        common["abs_log_moneyness"] = np.nan

    if "dte" not in common.columns:
        common["dte"] = np.round(pd.to_numeric(common["T"], errors="coerce") * cfg.market.calendar_days_per_year)
    if "maturity_bucket" not in common.columns:
        def _bucket_from_dte(x: float) -> str:
            for bucket, (_, lo, hi) in cfg.hedging.maturity_buckets.items():
                if lo <= x <= hi:
                    return bucket
            return "outside_hedging_buckets"
        common["maturity_bucket"] = common["dte"].map(_bucket_from_dte)

    common["near_atm_5pct"] = common["abs_log_moneyness"] <= 0.05
    common["short_maturity_60d"] = pd.to_numeric(common["dte"], errors="coerce") <= 60
    common["lifted_minus_heston_abs_vega_scaled_error"] = common["lifted_abs_vega_scaled_error"] - common["heston_abs_vega_scaled_error"]
    common["lifted_better_abs_vega_scaled_error"] = common["lifted_abs_vega_scaled_error"] < common["heston_abs_vega_scaled_error"]
    common["diagnostic_version"] = CALIBRATION_DIAGNOSTIC_VERSION

    finite = np.isfinite(common["heston_abs_vega_scaled_error"]) & np.isfinite(common["lifted_abs_vega_scaled_error"])
    common = common.loc[finite].copy()
    if common.empty:
        raise ValueError("Common-target fit diagnostics produced no finite paired Heston/lifted errors.")
    return common.reset_index(drop=True)


def summarise_fit_pair(frame: pd.DataFrame, group_cols: list[str], slice_name: str) -> pd.DataFrame:
    if frame.empty:
        cols = group_cols + [
            "slice", "n_rows", "n_dates", "heston_median_abs_vega_scaled_error",
            "lifted_median_abs_vega_scaled_error", "median_diff_lifted_minus_heston",
            "share_rows_lifted_better",
        ]
        return pd.DataFrame(columns=cols)
    rows = []
    grouped = frame.groupby(group_cols, dropna=False) if group_cols else [((), frame)]
    for key, g in grouped:
        key_vals = key if isinstance(key, tuple) else (key,)
        row = {col: val for col, val in zip(group_cols, key_vals)}
        row.update({
            "slice": slice_name,
            "n_rows": int(len(g)),
            "n_dates": int(pd.to_datetime(g["date"]).dt.normalize().nunique()),
            "heston_median_abs_vega_scaled_error": float(g["heston_abs_vega_scaled_error"].median()),
            "lifted_median_abs_vega_scaled_error": float(g["lifted_abs_vega_scaled_error"].median()),
            "median_diff_lifted_minus_heston": float(g["lifted_minus_heston_abs_vega_scaled_error"].median()),
            "share_rows_lifted_better": float(g["lifted_better_abs_vega_scaled_error"].mean()),
            "diagnostic_version": CALIBRATION_DIAGNOSTIC_VERSION,
        })
        rows.append(row)
    return pd.DataFrame(rows)


# Date-level diagnostics.
calibration_diagnostic_summary = pd.concat([
    calibration_panel_summary(heston_calibration, "heston"),
    calibration_panel_summary(lifted_calibration, "lifted_heston"),
], ignore_index=True)
display(calibration_diagnostic_summary)

calibration_comparison = compare_heston_lifted_calibration(heston_calibration, lifted_calibration)
display(calibration_comparison.head())

write_dataframe_cache(
    calibration_comparison,
    CFG.paths.calibration_comparison_path,
    "heston_vs_lifted_calibration_comparison",
    "lifted",
    CFG,
    dependencies=[CFG.paths.heston_calibration_path, CFG.paths.lifted_calibration_path],
)

calibration_comparison_summary = build_calibration_comparison_summary(calibration_comparison)
display(calibration_comparison_summary)

lifted_calibration_parameterisation_audit = pd.DataFrame([{
    "diagnostic_version": CALIBRATION_DIAGNOSTIC_VERSION,
    "lifted_factors_N": int(CFG.lifted.n_factors),
    "hurst_H": float(CFG.lifted.H),
    "rn_scale": float(CFG.lifted.rn_scale),
    "calibration_parameterization": getattr(CFG.lifted, "calibration_parameterization", "full_5p_forward_curve_init"),
    "calibrated_parameters": ", ".join(lifted_calibration_parameter_names(CFG)),
    "fixed_or_curve_identified_quantities": "full_5p_forward_curve_init calibrates kappa, theta, xi, rho and v0; the regularised xi0 curve initialises and shapes the deterministic input curve",
    "reason": "records lifted calibration degrees of freedom and confirms the main specification has reverted from the 5p endpoint calibration to the final 5p forward-curve-initialised specification",
}])
display(lifted_calibration_parameterisation_audit)


# Row-level common-target diagnostics.
calibration_fit_pair_diagnostics = build_calibration_fit_pair_diagnostics(
    heston_fit_residuals,
    lifted_fit_table,
    surface_panel,
    heston_calibration,
    CFG,
)

calibration_fit_target_schema_audit = pd.DataFrame([
    {
        "diagnostic_version": CALIBRATION_DIAGNOSTIC_VERSION,
        "fit_target_match_method": str(calibration_fit_pair_diagnostics["fit_target_match_method"].iloc[0]),
        "rows": int(len(calibration_fit_pair_diagnostics)),
        "dates": int(pd.to_datetime(calibration_fit_pair_diagnostics["date"]).dt.normalize().nunique()),
        "heston_target_source": str(calibration_fit_pair_diagnostics.get("heston_target_source", pd.Series(["unknown"])).iloc[0]),
        "lifted_target_source": str(calibration_fit_pair_diagnostics.get("lifted_target_source", pd.Series(["unknown"])).iloc[0]),
        "rows_excluded_missing_vega_scale": int(calibration_fit_pair_diagnostics.get("rows_excluded_missing_vega_scale", pd.Series([0])).max()),
    }
])
display(calibration_fit_target_schema_audit)

calibration_fit_overall_diagnostics = pd.concat([
    summarise_fit_pair(calibration_fit_pair_diagnostics, [], "common_full_surface"),
    summarise_fit_pair(calibration_fit_pair_diagnostics.loc[calibration_fit_pair_diagnostics["near_atm_5pct"]], [], "near_atm_abs_log_moneyness_le_0p05"),
    summarise_fit_pair(calibration_fit_pair_diagnostics.loc[calibration_fit_pair_diagnostics["short_maturity_60d"]], [], "short_maturity_le_60d"),
    summarise_fit_pair(calibration_fit_pair_diagnostics.loc[calibration_fit_pair_diagnostics["near_atm_5pct"] & calibration_fit_pair_diagnostics["short_maturity_60d"]], [], "near_atm_and_short_maturity"),
], ignore_index=True)
calibration_fit_by_maturity = summarise_fit_pair(calibration_fit_pair_diagnostics, ["maturity_bucket"], "by_maturity_bucket")
calibration_fit_by_moneyness = summarise_fit_pair(
    calibration_fit_pair_diagnostics.assign(
        moneyness_band=pd.cut(
            calibration_fit_pair_diagnostics["abs_log_moneyness"],
            bins=[-np.inf, 0.03, 0.05, 0.10, 0.20, np.inf],
            labels=["ATM_0_3pct", "near_ATM_3_5pct", "mid_5_10pct", "wing_10_20pct", "far_wing_gt_20pct"],
        )
    ),
    ["moneyness_band"],
    "by_moneyness_band",
)
display(calibration_fit_overall_diagnostics)
display(calibration_fit_by_maturity)
display(calibration_fit_by_moneyness)

tables_dir = CFG.paths.output_dir / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
calibration_fit_pair_diagnostics.to_csv(tables_dir / "calibration_fit_pair_diagnostics.csv", index=False)
calibration_fit_overall_diagnostics.to_csv(tables_dir / "calibration_fit_overall_diagnostics.csv", index=False)
calibration_fit_by_maturity.to_csv(tables_dir / "calibration_fit_by_maturity.csv", index=False)
calibration_fit_by_moneyness.to_csv(tables_dir / "calibration_fit_by_moneyness.csv", index=False)
calibration_fit_target_schema_audit.to_csv(tables_dir / "calibration_fit_target_schema_audit.csv", index=False)
lifted_calibration_parameterisation_audit.to_csv(tables_dir / "lifted_calibration_parameterisation_audit.csv", index=False)

if MATPLOTLIB_AVAILABLE and {"heston_vega_scaled_rmse", "lifted_vega_scaled_rmse"}.issubset(calibration_comparison.columns):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(calibration_comparison["heston_vega_scaled_rmse"], calibration_comparison["lifted_vega_scaled_rmse"], s=18, alpha=SCATTER_ALPHA, color=MODEL_COLORS["lifted_heston"], edgecolors="white", linewidths=0.25)
    lo = float(np.nanmin([calibration_comparison["heston_vega_scaled_rmse"].min(), calibration_comparison["lifted_vega_scaled_rmse"].min()]))
    hi = float(np.nanmax([calibration_comparison["heston_vega_scaled_rmse"].max(), calibration_comparison["lifted_vega_scaled_rmse"].max()]))
    ax.plot([lo, hi], [lo, hi], linestyle="--", color="#94A3B8", linewidth=0.9, alpha=0.75)
    ax.set_title("Full-surface calibration fit: Heston vs Lifted Heston")
    ax.set_xlabel("Heston vega-scaled RMSE")
    ax.set_ylabel("Lifted Heston vega-scaled RMSE")
    ax.grid(True, alpha=0.3)
    plt.show()


def _fit_based_ok(panel: pd.DataFrame, prefix: str, cfg: ExperimentConfig) -> pd.Series:
    """Date-level hedging eligibility gate based on realised fit, not optimiser status."""
    bad_flags = {"NUMFAIL", "FAIL", "FAILED", "BADFIT"}
    required_param_cols = [f"{prefix}_{c}" for c in ["kappa", "theta", "xi", "rho", "v0"] if f"{prefix}_{c}" in panel.columns]
    if required_param_cols:
        finite_params = np.isfinite(panel[required_param_cols].apply(pd.to_numeric, errors="coerce")).all(axis=1)
    else:
        finite_params = pd.Series(False, index=panel.index)

    qcol = f"{prefix}_quality_flag"
    rmse_col = f"{prefix}_vega_scaled_rmse"
    quality = panel[qcol].astype(str).str.upper() if qcol in panel.columns else pd.Series("MISSING", index=panel.index)
    rmse = pd.to_numeric(panel[rmse_col], errors="coerce") if rmse_col in panel.columns else pd.Series(np.nan, index=panel.index)

    # BADFIT is assigned by the model-specific calibration code at its configured threshold.
    # Do not duplicate a second, potentially drifting numerical threshold here.
    return finite_params & np.isfinite(rmse) & ~quality.isin(bad_flags)


def build_calibration_quality_gate(heston: pd.DataFrame, lifted: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    h = _normalise_one_row_per_date(heston, "Heston")
    l = _normalise_one_row_per_date(lifted, "Lifted Heston")

    h_cols = ["date"] + [c for c in [
        "success", "quality_flag", "vega_scaled_rmse", "price_rmse", "loss",
        "kappa", "theta", "xi", "rho", "v0", "backend", "calibration_mode",
    ] if c in h.columns]
    l_cols = ["date"] + [c for c in [
        "success", "quality_flag", "vega_scaled_rmse", "price_rmse", "loss",
        "kappa", "theta", "xi", "rho", "v0", "candidate_source",
        "calibration_backend",
    ] if c in l.columns]

    hq = h[h_cols].rename(columns={c: f"heston_{c}" for c in h_cols if c != "date"})
    lq = l[l_cols].rename(columns={c: f"lifted_{c}" for c in l_cols if c != "date"})
    gate = hq.merge(lq, on="date", how="outer", indicator=True).sort_values("date").reset_index(drop=True)

    gate["heston_fit_accepted"] = _fit_based_ok(gate, "heston", cfg)
    gate["lifted_fit_accepted"] = _fit_based_ok(gate, "lifted", cfg)
    gate["accepted_for_hedging"] = gate["heston_fit_accepted"] & gate["lifted_fit_accepted"]

    missing_h = gate["_merge"].eq("right_only")
    missing_l = gate["_merge"].eq("left_only")
    gate["exclusion_reason"] = np.select(
        [
            gate["accepted_for_hedging"],
            missing_h,
            missing_l,
            ~gate["heston_fit_accepted"],
            ~gate["lifted_fit_accepted"],
        ],
        [
            "accepted",
            "missing_heston_calibration",
            "missing_lifted_calibration_or_forward_curve",
            "heston_nonfinite_or_numerical_failure",
            "lifted_nonfinite_or_numerical_failure",
        ],
        default="unclassified",
    )
    gate["gate_version"] = CALIBRATION_QUALITY_GATE_VERSION
    gate["gate_basis"] = "finite parameters + finite vega_scaled_rmse + no numerical failure flag; RMSE retained for audit, not used as seed-selection performance filter"

    if not gate["accepted_for_hedging"].any():
        summary = gate["exclusion_reason"].value_counts(dropna=False).to_dict()
        raise RuntimeError(f"No dates pass Heston/lifted calibration eligibility audit. Summary={summary}")

    return gate.drop(columns=["_merge"]).reset_index(drop=True)


quality_deps = [CFG.paths.heston_calibration_path, CFG.paths.lifted_calibration_path]
try:
    calibration_quality_gate = load_dataframe_cache(
        CFG.paths.calibration_quality_path,
        "calibration_quality_gate",
        "seed_panel",
        CFG,
        required_columns={"date", "accepted_for_hedging", "exclusion_reason", "gate_version"},
        dependencies=quality_deps,
    )
    if set(calibration_quality_gate["gate_version"].dropna().astype(str).unique()) != {CALIBRATION_QUALITY_GATE_VERSION}:
        raise ValueError("Calibration quality gate cache uses a stale gate version.")
except (FileNotFoundError, ValueError) as exc:
    if not CFG.flags.can_rebuild("feasibility"):
        raise RuntimeError(
            "Calibration quality gate is missing/stale and strict reuse forbids rebuilding. "
            "Set rebuild_from='feasibility', an earlier stage, or use 'auto'."
        ) from exc
    print(f"Calibration quality gate unavailable or stale; rebuilding under the resolved rebuild plan. Reason: {exc}")
    calibration_quality_gate = build_calibration_quality_gate(heston_calibration, lifted_calibration, CFG)
    write_dataframe_cache(
        calibration_quality_gate,
        CFG.paths.calibration_quality_path,
        "calibration_quality_gate",
        "seed_panel",
        CFG,
        dependencies=quality_deps,
    )

accepted_calibration_dates = set(pd.to_datetime(
    calibration_quality_gate.loc[calibration_quality_gate["accepted_for_hedging"], "date"]
).dt.normalize())

print({
    "accepted_dates": len(accepted_calibration_dates),
    "excluded_dates": int((~calibration_quality_gate["accepted_for_hedging"]).sum()),
    "gate_version": CALIBRATION_QUALITY_GATE_VERSION,
})
display(calibration_quality_gate["exclusion_reason"].value_counts().rename_axis("reason").reset_index(name="dates"))

if {"heston_success", "heston_fit_accepted"}.issubset(calibration_quality_gate.columns):
    optimiser_vs_fit = calibration_quality_gate.assign(
        heston_success=calibration_quality_gate["heston_success"].fillna(False).astype(bool),
        heston_fit_accepted=calibration_quality_gate["heston_fit_accepted"].fillna(False).astype(bool),
    ).groupby(["heston_success", "heston_fit_accepted"]).size().rename("dates").reset_index()
    display(optimiser_vs_fit)


,metric,model,count,mean,std,min,5%,50%,95%,max
0,loss,heston,2869.0,0.000005,0.000016,1.023931e-09,7.289967e-08,0.000001,0.000024,0.000302
1,price_rmse,heston,2869.0,0.454807,0.403337,2.201633e-02,1.137081e-01,0.354321,1.083155,9.733312
2,price_mae,heston,2869.0,0.362620,0.291860,1.657234e-02,9.116927e-02,0.287494,0.864357,5.215223
3,vega_scaled_rmse,heston,2869.0,0.001438,0.001691,2.356283e-05,2.085016e-04,0.000828,0.004609,0.021875
4,cpu_seconds,heston,25.0,0.837385,0.705749,7.606610e-02,1.436985e-01,0.620141,2.358750,2.665133
5,n_options,heston,2869.0,46.660160,29.187902,4.000000e+00,1.300000e+01,39.000000,103.000000,214.000000
6,loss,lifted_heston,2161.0,0.000007,0.000019,1.698317e-08,1.469313e-07,0.000001,0.000039,0.000210
7,price_rmse,lifted_heston,2161.0,0.718019,0.708047,5.854925e-02,1.595246e-01,0.523203,1.890266,8.477670
8,vega_scaled_rmse,lifted_heston,2161.0,0.001892,0.001959,1.303195e-04,3.833162e-04,0.001221,0.006214,0.014501
9,cpu_seconds,lifted_heston,2161.0,25.600256,24.209845,2.078620e+00,4.991432e+00,19.651755,62.953876,348.755137


,date,loss_heston,price_rmse_heston,vega_scaled_rmse_heston,cpu_seconds_heston,loss_lifted,price_rmse_lifted,vega_scaled_rmse_lifted,cpu_seconds_lifted,heston_loss,lifted_loss,loss_diff_lifted_minus_heston,loss_median_pair_available,loss_lifted_better,heston_price_rmse,lifted_price_rmse,price_rmse_diff_lifted_minus_heston,price_rmse_median_pair_available,price_rmse_lifted_better,heston_vega_scaled_rmse,lifted_vega_scaled_rmse,vega_scaled_rmse_diff_lifted_minus_heston,vega_scaled_rmse_median_pair_available,vega_scaled_rmse_lifted_better,heston_cpu_seconds,lifted_cpu_seconds,cpu_seconds_diff_lifted_minus_heston,cpu_seconds_median_pair_available,calibration_diagnostic_version
0,2010-01-04,8.955512e-08,0.058337,0.000299,0.398496,2.135844e-07,0.093563,0.000462,27.377331,8.955512e-08,2.135844e-07,1.240293e-07,True,False,0.058337,0.093563,0.035226,True,False,0.000299,0.000462,0.000163,True,False,0.398496,27.377331,26.978835,True,common_target_date_level_v1
1,2010-01-05,3.704908e-07,0.182779,0.000609,0.313135,3.272177e-07,0.164643,0.000572,14.484984,3.704908e-07,3.272177e-07,-4.327313e-08,True,True,0.182779,0.164643,-0.018137,True,True,0.000609,0.000572,-0.000037,True,True,0.313135,14.484984,14.171849,True,common_target_date_level_v1
2,2010-01-06,3.341898e-07,0.121961,0.000578,0.583240,8.879675e-07,0.204263,0.000942,58.210510,3.341898e-07,8.879675e-07,5.537776e-07,True,False,0.121961,0.204263,0.082302,True,False,0.000578,0.000942,0.000364,True,False,0.583240,58.210510,57.627269,True,common_target_date_level_v1
3,2010-01-07,3.839909e-07,0.134450,0.000620,0.706187,7.384267e-07,0.180359,0.000859,31.215503,3.839909e-07,7.384267e-07,3.544358e-07,True,False,0.134450,0.180359,0.045909,True,False,0.000620,0.000859,0.000240,True,False,0.706187,31.215503,30.509316,True,common_target_date_level_v1
4,2010-01-08,4.166474e-07,0.139311,0.000645,1.115355,1.179752e-06,0.218588,0.001086,21.781359,4.166474e-07,1.179752e-06,7.631046e-07,True,False,0.139311,0.218588,0.079277,True,False,0.000645,0.001086,0.000441,True,False,1.115355,21.781359,20.666003,True,common_target_date_level_v1


[cache fingerprint] hashing heston_vs_lifted_hedging_panel_calibration_comparison_5bucket.parquet (0.4 MB); sidecar=9e08c08c38b1992be1d25d691716edd68cd226e01d2140495223c550ce1794ee.json
[cache fingerprint] complete heston_vs_lifted_hedging_panel_calibration_comparison_5bucket.parquet in 0.01s


,metric,common_dates,heston_median,lifted_median,median_diff_lifted_minus_heston,share_dates_lifted_better
0,loss,2161,0.000001,0.000001,-3.358333e-08,0.523369
1,price_rmse,2161,0.394283,0.523203,1.029239e-01,0.300787
2,vega_scaled_rmse,2161,0.000920,0.001221,1.798289e-04,0.371587
3,cpu_seconds,17,0.398496,29.996800,2.966311e+01,NaN


,diagnostic_version,lifted_factors_N,hurst_H,rn_scale,calibration_parameterization,calibrated_parameters,fixed_or_curve_identified_quantities,reason
0,common_target_date_level_v1,20,0.35,2.5,full_5p_forward_curve_init,"kappa, theta, xi, rho, v0","full_5p_forward_curve_init calibrates kappa, t...",records lifted calibration degrees of freedom ...


C:\Users\lhkke\AppData\Local\Temp\ipykernel_24096\2489034872.py:229: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[172.41850281 208.45658875 215.0151825  ... 315.86450195 165.54125977
  85.87532806]' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  out.loc[missing, "vega_scale"] = inferred_scale[missing.to_numpy()]


... (5 outputs left out)

<Figure size 780x780 with 1 Axes>

{'accepted_dates': 2161, 'excluded_dates': 708, 'gate_version': 'calibration_availability_final_finite_5p_lifted_no_rmse_exclusion'}


,reason,dates
0,accepted,2161
1,missing_lifted_calibration_or_forward_curve,708


,heston_success,heston_fit_accepted,dates
0,False,True,2849
1,True,True,20


# 5. Standardised Calibration Diagnostics

This section converts calibration artefacts into dissertation evidence. It standardises Heston and lifted-Heston diagnostics so that model comparison is not affected by different chart formats, date subsets, bucket definitions or residual conventions. If Heston gets an RMSE time series, lifted Heston gets the same. If lifted Heston gets a residual heatmap, Heston gets the same. Fairness also applies to pictures, because plots can otherwise create misleading visual comparisons.

The diagnostics are organised around four questions:

1. How large are pricing and IV residuals on the hedging-focused panel?
2. Where do residuals concentrate across maturity and moneyness?
3. Does lifted Heston improve the ATM-skew and near-ATM smile objects it is theoretically meant to improve?
4. Are calibrated parameters stable enough to support simulation and policy evaluation?

The notebook reports both global and hedging-focused diagnostics. The global surface diagnostic is labelled as a stress test. It should not be used as the headline hedging claim. The hedging panel is the main calibration object because it matches the traded options used later.

The general calibration error is

$$
e_j(\Theta)=\sigma_{\mathrm{model}}(K_j,T_j;\Theta)-\sigma_{\mathrm{mkt}}(K_j,T_j).
$$

RMSE, vega-weighted RMSE, bucket residuals and skew errors are different summaries of this residual field. They reveal different failure modes, so no single number should be treated as the full model-selection verdict.


## 5.1 Standardised Heston and Lifted-Heston Diagnostics

This subsection builds the common diagnostic layer used for both classical Heston and lifted Heston. The purpose is symmetry: each model should be judged on the same dates, rows, maturity buckets and moneyness buckets wherever the required columns exist.

The resulting tables are not the headline hedging result. They answer the narrower calibration question: whether the fitted risk-neutral model reproduces the option-market objects used to initialise simulation states. Later evaluation sections answer the separate hedging question by measuring terminal net hedging error after trading and transaction costs.

In [29]:

# ============================================================
# Standardised Heston / lifted-Heston calibration diagnostics
# ============================================================

STANDARD_DIAGNOSTICS_VERSION = "standardised_heston_lifted_diagnostics_v1_polished"


MODEL_DISPLAY = {
    "heston": "Heston",
    "lifted_heston": "Lifted Heston",
    "lifted": "Lifted Heston",
}


def _model_slug(model_name: str) -> str:
    name = str(model_name).strip().lower().replace("-", "_").replace(" ", "_")
    if name in {"lifted", "lifted_heston_scalar", "lifted_heston_factors"}:
        return "lifted_heston"
    if name in {"classical_heston"}:
        return "heston"
    return name


def _safe_numeric(frame: pd.DataFrame, col: str, default: float = np.nan) -> pd.Series:
    if frame is None or col not in frame.columns:
        return pd.Series(default, index=frame.index if isinstance(frame, pd.DataFrame) else None, dtype=float)
    return pd.to_numeric(frame[col], errors="coerce")


def _first_existing_column(frame: pd.DataFrame, candidates: Iterable[str]) -> str | None:
    if frame is None:
        return None
    for col in candidates:
        if col in frame.columns:
            return col
    return None


def _model_iv_candidates(model_slug: str) -> list[str]:
    if model_slug == "heston":
        return ["heston_iv", "heston_model_iv", "iv_heston", "model_iv_heston", "iv_model_heston", "iv_model"]
    if model_slug == "lifted_heston":
        return ["lifted_heston_iv", "lifted_iv", "lifted_model_iv", "iv_lifted", "model_iv_lifted", "iv_model_lifted", "iv_model"]
    return [f"{model_slug}_iv", f"{model_slug}_model_iv", "iv_model"]


def _market_iv_column(frame: pd.DataFrame) -> str | None:
    return _first_existing_column(frame, ["iv_market", "market_iv", "impl_volatility", "implied_volatility", "iv"])


def _price_column(frame: pd.DataFrame, model_slug: str) -> str | None:
    candidates = {
        "heston": ["heston_price", "price_heston", "model_price_heston", "price_model"],
        "lifted_heston": ["lifted_heston_price", "lifted_price", "price_lifted", "model_price_lifted", "price_model"],
    }.get(model_slug, [f"{model_slug}_price", "price_model"])
    return _first_existing_column(frame, candidates)


def _market_price_column(frame: pd.DataFrame) -> str | None:
    return _first_existing_column(frame, ["call_equiv_mid", "mid", "option_mid", "market_price", "price_market"])


def _write_csv(frame: pd.DataFrame, filename: str, cfg: ExperimentConfig = CFG) -> Path:
    path = table_path(filename, cfg)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    return path


def _save_figure(filename: str, cfg: ExperimentConfig = CFG) -> Path | None:
    if not MATPLOTLIB_AVAILABLE:
        return None
    path = figure_path(filename, cfg)
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    save_and_show_figure(plt.gcf(), path, dpi=160)
    return path


def _attach_bucket_columns(frame: pd.DataFrame, cfg: ExperimentConfig = CFG) -> pd.DataFrame:
    out = frame.copy()
    dte = _safe_numeric(out, "dte")
    if "maturity_bucket" not in out.columns:
        bucket = pd.Series("unbucketed", index=out.index, dtype="object")
        for label, (_, lo, hi) in cfg.hedging.maturity_buckets.items():
            bucket = bucket.where(~dte.between(lo, hi, inclusive="both"), label)
        out["maturity_bucket"] = bucket
    mny_col = _first_existing_column(out, ["log_forward_moneyness", "abs_log_forward_moneyness", "log_moneyness"])
    if "moneyness_bucket" not in out.columns:
        if mny_col is None:
            out["moneyness_bucket"] = "unavailable"
        else:
            mny = pd.to_numeric(out[mny_col], errors="coerce")
            if mny_col.startswith("abs_"):
                signed = mny
            else:
                signed = mny.abs()
            bins = [-np.inf, 0.03, 0.05, 0.10, 0.20, np.inf]
            labels = ["ATM_0_3pct", "near_ATM_3_5pct", "liquid_5_10pct", "outer_10_20pct", "wing_gt_20pct"]
            out["moneyness_bucket"] = pd.cut(signed, bins=bins, labels=labels).astype("object").fillna("unavailable")
    return out


def _residual_panel_for_model(
    model_name: str,
    fit_panel: pd.DataFrame,
    surface_panel: pd.DataFrame,
    cfg: ExperimentConfig = CFG,
) -> tuple[pd.DataFrame, str]:
    model_slug = _model_slug(model_name)
    if fit_panel is None or fit_panel.empty:
        return pd.DataFrame(), "fit_panel_missing"

    panel = fit_panel.copy()
    market_iv_col = _market_iv_column(panel)
    model_iv_col = _first_existing_column(panel, _model_iv_candidates(model_slug))

    if market_iv_col is None and surface_panel is not None and not surface_panel.empty:
        join_keys = [c for c in ["date", "exdate", "strike", "T"] if c in panel.columns and c in surface_panel.columns]
        if join_keys:
            attach_cols = join_keys + [c for c in ["iv_market", "log_forward_moneyness", "abs_log_forward_moneyness", "dte", "maturity_bucket"] if c in surface_panel.columns]
            panel = panel.merge(surface_panel[attach_cols].drop_duplicates(join_keys), on=join_keys, how="left", suffixes=("", "_surface"))
            market_iv_col = _market_iv_column(panel)

    if model_iv_col is None:
        return _attach_bucket_columns(panel, cfg), "model_iv_unavailable"
    if market_iv_col is None:
        return _attach_bucket_columns(panel, cfg), "market_iv_unavailable"

    panel["market_iv_for_diagnostic"] = pd.to_numeric(panel[market_iv_col], errors="coerce")
    panel["model_iv_for_diagnostic"] = pd.to_numeric(panel[model_iv_col], errors="coerce")
    panel["iv_residual"] = panel["model_iv_for_diagnostic"] - panel["market_iv_for_diagnostic"]
    panel["abs_iv_residual"] = panel["iv_residual"].abs()
    if "fit_weight" in panel.columns:
        w = pd.to_numeric(panel["fit_weight"], errors="coerce").clip(lower=0)
        panel["weighted_squared_iv_residual"] = w * panel["iv_residual"] ** 2
    return _attach_bucket_columns(panel, cfg), "available"


def _parameter_columns(frame: pd.DataFrame) -> list[str]:
    candidates = [
        "kappa", "theta", "xi", "rho", "v0",
        "H", "n_factors", "rn_scale", "loss", "price_rmse", "vega_scaled_rmse", "iv_rmse",
        "cpu_seconds", "n_options", "n_maturities",
    ]
    return [c for c in candidates if frame is not None and c in frame.columns]


def build_model_calibration_diagnostic_pack(
    model_name: str,
    fit_panel: pd.DataFrame,
    parameter_panel: pd.DataFrame | None,
    surface_panel: pd.DataFrame,
    cfg: ExperimentConfig,
) -> dict[str, pd.DataFrame]:
    """Build common calibration diagnostics for Heston and lifted Heston.

    Metrics are computed only when the required columns are available. Missing model-IV columns
    are reported as unavailable rather than backfilled or guessed.
    """

    model_slug = _model_slug(model_name)
    params = parameter_panel.copy() if isinstance(parameter_panel, pd.DataFrame) and not parameter_panel.empty else fit_panel.copy()
    if params is None:
        params = pd.DataFrame()
    if not params.empty and "date" in params.columns:
        params["date"] = pd.to_datetime(params["date"], errors="coerce").dt.normalize()

    residual_panel, residual_status = _residual_panel_for_model(model_slug, fit_panel, surface_panel, cfg)
    if not residual_panel.empty and "date" in residual_panel.columns:
        residual_panel["date"] = pd.to_datetime(residual_panel["date"], errors="coerce").dt.normalize()
    residual_panel["model"] = model_slug
    residual_panel["residual_status"] = residual_status

    date_rows = []
    if not residual_panel.empty and "date" in residual_panel.columns:
        for date, g in residual_panel.groupby("date", dropna=False):
            row = {
                "model": model_slug,
                "date": date,
                "n_rows": int(len(g)),
                "n_expiries": int(g["exdate"].nunique()) if "exdate" in g.columns else np.nan,
                "residual_status": residual_status,
            }
            if "iv_residual" in g.columns:
                r = pd.to_numeric(g["iv_residual"], errors="coerce")
                row["iv_rmse"] = float(np.sqrt(np.nanmean(r ** 2))) if r.notna().any() else np.nan
                near = g.loc[pd.to_numeric(g.get("abs_log_forward_moneyness", g.get("log_forward_moneyness", pd.Series(np.nan, index=g.index))).abs(), errors="coerce").le(cfg.diagnostics.near_atm_skew_window)]
                if not near.empty:
                    nr = pd.to_numeric(near["iv_residual"], errors="coerce")
                    row["near_atm_iv_rmse"] = float(np.sqrt(np.nanmean(nr ** 2))) if nr.notna().any() else np.nan
                if "weighted_squared_iv_residual" in g.columns:
                    row["vega_weighted_iv_rmse"] = float(np.sqrt(np.nanmean(pd.to_numeric(g["weighted_squared_iv_residual"], errors="coerce"))))
            date_rows.append(row)
    date_summary = pd.DataFrame(date_rows)

    if not residual_panel.empty and "iv_residual" in residual_panel.columns:
        maturity_bucket_summary = (
            residual_panel.groupby(["model", "maturity_bucket"], dropna=False)
            .agg(
                n_rows=("iv_residual", "size"),
                iv_rmse=("iv_residual", lambda s: float(np.sqrt(np.nanmean(pd.to_numeric(s, errors="coerce") ** 2)))),
                mean_residual=("iv_residual", lambda s: float(np.nanmean(pd.to_numeric(s, errors="coerce")))),
                median_abs_residual=("abs_iv_residual", lambda s: float(np.nanmedian(pd.to_numeric(s, errors="coerce")))),
            )
            .reset_index()
        )
        moneyness_bucket_summary = (
            residual_panel.groupby(["model", "moneyness_bucket"], dropna=False)
            .agg(
                n_rows=("iv_residual", "size"),
                iv_rmse=("iv_residual", lambda s: float(np.sqrt(np.nanmean(pd.to_numeric(s, errors="coerce") ** 2)))),
                mean_residual=("iv_residual", lambda s: float(np.nanmean(pd.to_numeric(s, errors="coerce")))),
                median_abs_residual=("abs_iv_residual", lambda s: float(np.nanmedian(pd.to_numeric(s, errors="coerce")))),
            )
            .reset_index()
        )
    else:
        maturity_bucket_summary = pd.DataFrame([{"model": model_slug, "status": residual_status}])
        moneyness_bucket_summary = pd.DataFrame([{"model": model_slug, "status": residual_status}])

    pcols = _parameter_columns(params)
    if pcols:
        parameter_summary = params[pcols].apply(pd.to_numeric, errors="coerce").describe(percentiles=[0.05, 0.50, 0.95]).T.reset_index(names="parameter")
        parameter_summary.insert(0, "model", model_slug)
    else:
        parameter_summary = pd.DataFrame([{"model": model_slug, "status": "parameter_columns_unavailable"}])

    flag_rows = []
    if isinstance(params, pd.DataFrame) and not params.empty:
        flag_rows.append({
            "model": model_slug,
            "status": "available",
            "n_rows": int(len(params)),
            "n_dates": int(params["date"].nunique()) if "date" in params.columns else np.nan,
            "has_quality_flag": "quality_flag" in params.columns,
            "residual_status": residual_status,
        })
    else:
        flag_rows.append({"model": model_slug, "status": "parameter_panel_missing", "residual_status": residual_status})
    fit_quality_flags = pd.DataFrame(flag_rows)

    return {
        "date_summary": date_summary,
        "maturity_bucket_summary": maturity_bucket_summary,
        "moneyness_bucket_summary": moneyness_bucket_summary,
        "residual_panel": residual_panel,
        "parameter_summary": parameter_summary,
        "fit_quality_flags": fit_quality_flags,
    }


def write_model_calibration_diagnostic_pack(model_name: str, pack: dict[str, pd.DataFrame], cfg: ExperimentConfig = CFG) -> None:
    slug = _model_slug(model_name)
    mapping = {
        "date_summary": f"{slug}_calibration_date_summary.csv",
        "residual_panel": f"{slug}_calibration_residual_panel.csv",
        "parameter_summary": f"{slug}_parameter_summary.csv",
        "fit_quality_flags": f"{slug}_fit_quality_flags.csv",
    }
    for key, filename in mapping.items():
        frame = pack.get(key, pd.DataFrame())
        if isinstance(frame, pd.DataFrame) and not frame.empty:
            _write_csv(frame, filename, cfg)

    bucket = []
    for key in ["maturity_bucket_summary", "moneyness_bucket_summary"]:
        frame = pack.get(key, pd.DataFrame())
        if isinstance(frame, pd.DataFrame) and not frame.empty:
            frame = frame.copy()
            frame["bucket_axis"] = key.replace("_bucket_summary", "")
            bucket.append(frame)
    if bucket:
        _write_csv(pd.concat(bucket, ignore_index=True, sort=False), f"{slug}_residual_bucket_summary.csv", cfg)


def select_representative_fit_dates(
    calibration_fit_pair_diagnostics: pd.DataFrame,
    cfg: ExperimentConfig,
    n_dates: int | None = None,
    rule: str = "deterministic_quantile_by_date_and_regime",
) -> list[pd.Timestamp]:
    n_dates = int(n_dates or cfg.diagnostics.representative_fit_dates)
    if calibration_fit_pair_diagnostics is None or calibration_fit_pair_diagnostics.empty or "date" not in calibration_fit_pair_diagnostics.columns:
        return []
    dates = pd.to_datetime(calibration_fit_pair_diagnostics["date"], errors="coerce").dropna().dt.normalize().drop_duplicates().sort_values()
    if dates.empty:
        return []
    if len(dates) <= n_dates:
        return list(dates)
    quantiles = np.linspace(0, len(dates) - 1, n_dates).round().astype(int)
    return list(dates.iloc[np.unique(quantiles)])


def make_standardised_model_calibration_comparison(
    packs: dict[str, dict[str, pd.DataFrame]],
    surface_panel: pd.DataFrame,
    cfg: ExperimentConfig,
    panel_label: str = "hedging_focused",
) -> pd.DataFrame:
    rows = []
    for model, pack in packs.items():
        slug = _model_slug(model)
        date_summary = pack.get("date_summary", pd.DataFrame())
        residual_panel = pack.get("residual_panel", pd.DataFrame())
        flags = pack.get("fit_quality_flags", pd.DataFrame())
        row = {
            "model": slug,
            "panel": panel_label,
            "iv_rmse": "unavailable",
            "vega_weighted_iv_rmse": "unavailable",
            "price_rmse": "unavailable",
            "atm_iv_rmse": "unavailable",
            "near_atm_iv_rmse": "unavailable",
            "atm_skew_error": "unavailable",
            "finite_difference_atm_skew_error": "unavailable",
            "number_of_rows": int(len(surface_panel)) if isinstance(surface_panel, pd.DataFrame) else 0,
            "number_of_dates": int(pd.to_datetime(surface_panel["date"]).dt.normalize().nunique()) if isinstance(surface_panel, pd.DataFrame) and "date" in surface_panel.columns and not surface_panel.empty else 0,
            "number_of_expiries": int(surface_panel["exdate"].nunique()) if isinstance(surface_panel, pd.DataFrame) and "exdate" in surface_panel.columns and not surface_panel.empty else 0,
            "calibration_status": "available" if not flags.empty and not flags["status"].astype(str).str.contains("missing", case=False, na=False).any() else "unavailable",
            "unavailable_reason": "",
        }
        if not date_summary.empty:
            for src_col, dst_col in [
                ("iv_rmse", "iv_rmse"),
                ("vega_weighted_iv_rmse", "vega_weighted_iv_rmse"),
                ("near_atm_iv_rmse", "near_atm_iv_rmse"),
            ]:
                if src_col in date_summary.columns and pd.to_numeric(date_summary[src_col], errors="coerce").notna().any():
                    row[dst_col] = float(pd.to_numeric(date_summary[src_col], errors="coerce").mean())
        price_col = _price_column(residual_panel, slug)
        market_price_col = _market_price_column(residual_panel)
        if price_col and market_price_col:
            err = pd.to_numeric(residual_panel[price_col], errors="coerce") - pd.to_numeric(residual_panel[market_price_col], errors="coerce")
            if np.isfinite(err).any():
                row["price_rmse"] = float(np.sqrt(np.nanmean(err ** 2)))
        rows.append(row)
    out = pd.DataFrame(rows)
    _write_csv(out, "standardised_model_calibration_comparison.csv", cfg)
    if panel_label == "hedging_focused":
        _write_csv(out.rename(columns={
            "iv_rmse": "iv_rmse",
            "vega_weighted_iv_rmse": "vega_weighted_iv_rmse",
            "price_rmse": "price_rmse",
            "atm_iv_rmse": "atm_iv_rmse",
            "near_atm_iv_rmse": "near_atm_iv_rmse",
            "number_of_rows": "n_rows",
            "number_of_dates": "n_dates",
            "number_of_expiries": "n_expiries",
        }), "hedging_panel_calibration_diagnostics.csv", cfg)
    return out


def create_standardised_calibration_charts(model_name: str, pack: dict[str, pd.DataFrame], representative_dates: list[pd.Timestamp], cfg: ExperimentConfig = CFG) -> list[Path]:
    if not MATPLOTLIB_AVAILABLE:
        return []
    slug = _model_slug(model_name)
    written = []
    date_summary = pack.get("date_summary", pd.DataFrame())
    residual_panel = pack.get("residual_panel", pd.DataFrame())
    parameter_summary = pack.get("parameter_summary", pd.DataFrame())

    if not date_summary.empty and {"date", "iv_rmse"}.issubset(date_summary.columns):
        fig, ax = plt.subplots(figsize=(10, 4))
        x = pd.to_datetime(date_summary["date"], errors="coerce")
        y = pd.to_numeric(date_summary["iv_rmse"], errors="coerce")
        ax.plot(x, y, linewidth=0.95, color=model_color(slug), linestyle="-", alpha=MODEL_ALPHA)
        ax.set(title=f"{MODEL_DISPLAY.get(slug, slug)} IV RMSE over time", xlabel="Date", ylabel="IV RMSE")
        ax.grid(True, alpha=0.3)
        p = _save_figure(f"{slug}_iv_rmse_time_series.png", cfg)
        if p: written.append(p)

    if not date_summary.empty and {"date", "vega_weighted_iv_rmse"}.issubset(date_summary.columns):
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(pd.to_datetime(date_summary["date"], errors="coerce"), pd.to_numeric(date_summary["vega_weighted_iv_rmse"], errors="coerce"), linewidth=0.95, color=model_color(slug), linestyle="-", alpha=MODEL_ALPHA)
        ax.set(title=f"{MODEL_DISPLAY.get(slug, slug)} vega-weighted IV RMSE over time", xlabel="Date", ylabel="Vega-weighted IV RMSE")
        ax.grid(True, alpha=0.3)
        p = _save_figure(f"{slug}_vega_weighted_iv_rmse_time_series.png", cfg)
        if p: written.append(p)

    if not residual_panel.empty and {"market_iv_for_diagnostic", "model_iv_for_diagnostic"}.issubset(residual_panel.columns):
        sample = residual_panel.dropna(subset=["market_iv_for_diagnostic", "model_iv_for_diagnostic"])
        if len(sample) > 15000:
            sample = sample.sample(15000, random_state=cfg.random_seed)
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.scatter(sample["market_iv_for_diagnostic"], sample["model_iv_for_diagnostic"], s=8, alpha=0.42, color=model_color(slug), edgecolors="none")
        lo = float(np.nanmin([sample["market_iv_for_diagnostic"].min(), sample["model_iv_for_diagnostic"].min()]))
        hi = float(np.nanmax([sample["market_iv_for_diagnostic"].max(), sample["model_iv_for_diagnostic"].max()]))
        ax.plot([lo, hi], [lo, hi], linewidth=0.9, linestyle=":", color=MODEL_COLORS["market"])
        ax.set(title=f"{MODEL_DISPLAY.get(slug, slug)} model IV vs market IV", xlabel="Market IV", ylabel="Model IV")
        ax.grid(True, alpha=0.3)
        p = _save_figure(f"{slug}_model_vs_market_iv_scatter.png", cfg)
        if p: written.append(p)

        heat = residual_panel.pivot_table(index="maturity_bucket", columns="moneyness_bucket", values="iv_residual", aggfunc=lambda s: float(np.nanmean(pd.to_numeric(s, errors="coerce"))))
        fig, ax = plt.subplots(figsize=(9, 4))
        plot_annotated_heatmap(
            heat,
            ax=ax,
            fig=fig,
            title=f"{MODEL_DISPLAY.get(slug, slug)} mean IV residual by bucket",
            xlabel="Moneyness bucket",
            ylabel="Maturity bucket",
            cbar_label="Mean IV residual",
            cmap=SIGNED_RESIDUAL_CMAP,
            norm=TwoSlopeNorm(vcenter=0.0) if "TwoSlopeNorm" in globals() else None,
            fmt=None,
        )
        p = _save_figure(f"{slug}_iv_residual_heatmap.png", cfg)
        if p: written.append(p)

        near = residual_panel.loc[pd.to_numeric(residual_panel.get("abs_log_forward_moneyness", residual_panel.get("log_forward_moneyness", pd.Series(np.nan, index=residual_panel.index))).abs(), errors="coerce").le(cfg.diagnostics.near_atm_skew_window)]
        if not near.empty:
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.hist(pd.to_numeric(near["iv_residual"], errors="coerce").dropna(), bins=40, color=model_color(slug), alpha=0.72, edgecolor="white", linewidth=0.35)
            ax.set(title=f"{MODEL_DISPLAY.get(slug, slug)} near-ATM residual distribution", xlabel="Model IV - market IV", ylabel="Count")
            p = _save_figure(f"{slug}_near_atm_residual_distribution.png", cfg)
            if p: written.append(p)

    if not residual_panel.empty and {"date", "maturity_bucket"}.issubset(residual_panel.columns):
        coverage = residual_panel.groupby("date", as_index=False).size()
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(pd.to_datetime(coverage["date"], errors="coerce"), coverage["size"], linewidth=0.95, color=model_color(slug), linestyle="-", alpha=MODEL_ALPHA)
        ax.set(title=f"{MODEL_DISPLAY.get(slug, slug)} calibration row-count coverage", xlabel="Date", ylabel="Rows")
        ax.grid(True, alpha=0.3)
        p = _save_figure(f"{slug}_calibration_coverage_time_series.png", cfg)
        if p: written.append(p)

    if residual_panel is not None and not residual_panel.empty and representative_dates and {"date", "market_iv_for_diagnostic", "model_iv_for_diagnostic"}.issubset(residual_panel.columns):
        fig, ax = plt.subplots(figsize=(10, 5))
        plot_panel = residual_panel.copy()
        plot_panel["date"] = pd.to_datetime(plot_panel["date"], errors="coerce").dt.normalize()
        mny_col = _first_existing_column(plot_panel, ["log_forward_moneyness", "abs_log_forward_moneyness", "log_moneyness"])
        if mny_col:
            for date in representative_dates:
                g = plot_panel.loc[plot_panel["date"].eq(pd.Timestamp(date).normalize())].sort_values(mny_col)
                if not g.empty:
                    ax.scatter(g[mny_col], g["market_iv_for_diagnostic"], s=14, color=model_color("market"), alpha=SCATTER_ALPHA, edgecolors="white", linewidths=0.25, label="Market")
                    ax.plot(g[mny_col], g["model_iv_for_diagnostic"], linewidth=1.0, linestyle="-", color=model_color(slug), alpha=MODEL_ALPHA, label=MODEL_DISPLAY.get(slug, slug))
            ax.set(title=f"{MODEL_DISPLAY.get(slug, slug)} smile fit examples", xlabel=mny_col, ylabel="IV")
            ax.grid(True, alpha=0.3)
            compact_legend(ax, ncol=2)
            p = _save_figure(f"{slug}_smile_fit_examples.png", cfg)
            if p: written.append(p)

    live_params = globals().get(f"{slug}_calibration", pd.DataFrame())
    if isinstance(live_params, pd.DataFrame) and not live_params.empty and "date" in live_params.columns:
        param_cols = [c for c in ["kappa", "theta", "xi", "rho", "v0"] if c in live_params.columns]
        if param_cols:
            fig, ax = plt.subplots(figsize=(10, 4))
            pp = live_params.copy()
            pp["date"] = pd.to_datetime(pp["date"], errors="coerce")
            for col in param_cols:
                ax.plot(pp["date"], pd.to_numeric(pp[col], errors="coerce"), linewidth=0.95, linestyle="-", alpha=MODEL_ALPHA, label=col)
            ax.set(title=f"{MODEL_DISPLAY.get(slug, slug)} parameter time series", xlabel="Date", ylabel="Parameter value")
            compact_legend(ax, ncol=3)
            ax.grid(True, alpha=0.3)
            p = _save_figure(f"{slug}_parameter_time_series.png", cfg)
            if p: written.append(p)
    elif not parameter_summary.empty:
        _write_csv(parameter_summary, f"{slug}_parameter_time_series_unavailable_summary.csv", cfg)

    return written


def make_seed_panel_coverage_table(seed_panel: pd.DataFrame, regime_labels: pd.DataFrame, cfg: ExperimentConfig = CFG) -> pd.DataFrame:
    if seed_panel is None or seed_panel.empty:
        out = pd.DataFrame(columns=["split", "maturity_bucket", "regime_label", "n_seed_rows", "n_dates", "min_date", "max_date", "median_abs_log_moneyness", "median_dte"])
        _write_csv(out, "seed_panel_coverage_by_split_maturity_regime.csv", cfg)
        return out
    panel = seed_panel.copy()
    panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.normalize()
    if "maturity_bucket" not in panel.columns and "hedge_maturity_bucket" in panel.columns:
        panel["maturity_bucket"] = panel["hedge_maturity_bucket"]
    if regime_labels is not None and not regime_labels.empty and {"date", "regime_label"}.issubset(regime_labels.columns):
        reg = regime_labels.copy()
        reg["date"] = pd.to_datetime(reg["date"], errors="coerce").dt.normalize()
        panel = panel.merge(reg[["date", "regime_label"]].drop_duplicates("date"), on="date", how="left")
    if "regime_label" not in panel.columns:
        panel["regime_label"] = "unavailable"
    if "abs_log_forward_moneyness" not in panel.columns:
        mny_col = _first_existing_column(panel, ["log_forward_moneyness", "log_moneyness"])
        panel["abs_log_forward_moneyness"] = pd.to_numeric(panel[mny_col], errors="coerce").abs() if mny_col else np.nan
    if "dte" not in panel.columns and "T" in panel.columns:
        panel["dte"] = pd.to_numeric(panel["T"], errors="coerce") * cfg.market.calendar_days_per_year
    out = (
        panel.groupby(["split", "maturity_bucket", "regime_label"], dropna=False)
        .agg(
            n_seed_rows=("date", "size"),
            n_dates=("date", "nunique"),
            min_date=("date", "min"),
            max_date=("date", "max"),
            median_abs_log_moneyness=("abs_log_forward_moneyness", "median"),
            median_dte=("dte", "median"),
        )
        .reset_index()
        .sort_values(["split", "maturity_bucket", "regime_label"])
    )
    _write_csv(out, "seed_panel_coverage_by_split_maturity_regime.csv", cfg)
    return out


def finite_difference_model_atm_skew(
    model_name: str,
    params_for_date: pd.Series,
    date: pd.Timestamp,
    expiry: pd.Timestamp,
    forward: float,
    T: float,
    r: float,
    q: float,
    cfg: ExperimentConfig,
    bump: float = 0.01,
) -> dict:
    """Compute central finite-difference model ATM skew from actual model pricers.

    If the relevant calibrated pricer/curve is unavailable, the row is marked unavailable.
    No IVs are invented and no fallback Black-Scholes smile is substituted.
    """

    slug = _model_slug(model_name)
    result = {
        "date": pd.Timestamp(date).normalize(),
        "expiry": pd.Timestamp(expiry) if pd.notna(expiry) else pd.NaT,
        "T": float(T),
        "maturity_bucket": None,
        "model": slug,
        "bump_abs_log_moneyness": float(bump),
        "iv_k_minus": np.nan,
        "iv_k_zero": np.nan,
        "iv_k_plus": np.nan,
        "finite_difference_atm_skew": np.nan,
        "market_near_atm_skew": np.nan,
        "finite_difference_skew_error": np.nan,
        "pricing_status": "unavailable",
        "skew_estimator": "central_finite_difference_model_iv",
    }
    Fwd = float(forward)
    if not np.isfinite(Fwd) or Fwd <= 0 or not np.isfinite(T) or T <= 0:
        result["pricing_status"] = "invalid_forward_or_maturity"
        return result

    strikes = [Fwd * math.exp(-bump), Fwd, Fwd * math.exp(bump)]
    spot = Fwd * math.exp(-(float(r) - float(q)) * float(T))

    try:
        if slug == "heston":
            pricer = globals().get("heston_pricer")
            if pricer is None:
                pricer = HestonPricer(n_quad=int(cfg.calibration.heston_pricing_nodes))
            params = HestonParams(
                kappa=float(params_for_date["kappa"]),
                theta=float(params_for_date["theta"]),
                xi=float(params_for_date["xi"]),
                rho=float(params_for_date["rho"]),
                v0=float(params_for_date["v0"]),
            )
            prices = [pricer.call_price_one(spot, K, T, r, q, params) for K in strikes]
        elif slug == "lifted_heston":
            pricer = globals().get("lifted_pricer")
            if pricer is None:
                kernel = globals().get("lifted_kernel", None)
                if kernel is None:
                    kernel = build_abi_zaugg_lifted_kernel(cfg.lifted)
                pricer = LiftedRiccatiFourierPricer(kernel, cfg.lifted.pricing_nodes, cfg.lifted.riccati_steps, cfg.lifted.delta_coefficient_cache_size)
            params = LiftedHestonParams(
                kappa=float(params_for_date["kappa"]),
                theta=float(params_for_date["theta"]),
                xi=float(params_for_date["xi"]),
                rho=float(params_for_date["rho"]),
                v0=float(params_for_date["v0"]),
            )
            curve_frame = globals().get("forward_variance_curve", pd.DataFrame())
            if curve_frame is None or curve_frame.empty:
                result["pricing_status"] = "forward_variance_curve_unavailable"
                return result
            curve_day = curve_frame.copy()
            if "date" in curve_day.columns:
                curve_day["date"] = pd.to_datetime(curve_day["date"], errors="coerce").dt.normalize()
                curve_day = curve_day.loc[curve_day["date"].eq(pd.Timestamp(date).normalize())]
            curve = ForwardVarianceInputCurve.from_frame(curve_day)
            prices = [pricer.call_price_one(spot, K, T, r, q, params, curve) for K in strikes]
        else:
            result["pricing_status"] = f"unsupported_model_{slug}"
            return result

        ivs = [implied_vol_single(float(price), spot, K, r, q, T, option_type="call") for price, K in zip(prices, strikes)]
        result["iv_k_minus"], result["iv_k_zero"], result["iv_k_plus"] = [float(x) for x in ivs]
        result["finite_difference_atm_skew"] = float((ivs[2] - ivs[0]) / (2.0 * bump))
        result["pricing_status"] = "available"
        return result
    except Exception as exc:
        result["pricing_status"] = f"failed:{type(exc).__name__}:{str(exc)[:120]}"
        return result


def make_final_atm_skew_summary(skew_diagnostics: pd.DataFrame, cfg: ExperimentConfig = CFG) -> pd.DataFrame:
    if skew_diagnostics is None or skew_diagnostics.empty:
        out = pd.DataFrame(columns=["model", "maturity_bucket", "regime_label", "moneyness_window_abs_log_forward", "median_abs_skew_error", "mean_abs_skew_error", "rmse_skew_error", "median_signed_skew_error", "n_date_expiry_pairs"])
        _write_csv(out, "final_atm_skew_summary.csv", cfg)
        return out

    rows = []
    for model, err_col in [
        ("heston", "heston_near_atm_skew_error"),
        ("lifted_heston", "lifted_heston_near_atm_skew_error"),
    ]:
        if err_col not in skew_diagnostics.columns:
            continue
        gcols = [c for c in ["maturity_bucket", "regime_label", "moneyness_window_abs_log_forward"] if c in skew_diagnostics.columns]
        work = skew_diagnostics.copy()
        work["model"] = model
        work["signed_error"] = pd.to_numeric(work[err_col], errors="coerce")
        work["abs_error"] = work["signed_error"].abs()
        grouped = work.groupby(["model"] + gcols, dropna=False)
        rows.append(grouped.agg(
            median_abs_skew_error=("abs_error", "median"),
            mean_abs_skew_error=("abs_error", "mean"),
            rmse_skew_error=("signed_error", lambda s: float(np.sqrt(np.nanmean(pd.to_numeric(s, errors="coerce") ** 2)))),
            median_signed_skew_error=("signed_error", "median"),
            n_date_expiry_pairs=("signed_error", "count"),
        ).reset_index())
    out = pd.concat(rows, ignore_index=True, sort=False) if rows else pd.DataFrame()
    _write_csv(out, "final_atm_skew_summary.csv", cfg)
    return out


def write_global_surface_diagnostic_labelled_table(global_fit: pd.DataFrame, cfg: ExperimentConfig = CFG) -> pd.DataFrame:
    if global_fit is None or global_fit.empty:
        out = pd.DataFrame(columns=["model", "panel", "global_iv_rmse", "global_vega_weighted_iv_rmse", "global_price_rmse", "wing_iv_rmse", "near_atm_iv_rmse", "short_maturity_iv_rmse", "n_rows", "n_dates", "n_expiries", "label"])
    else:
        out = global_fit.copy()
        out["panel"] = "global_surface_diagnostic"
        out["label"] = "GLOBAL SURFACE DIAGNOSTIC: not used for headline hedging claims"
    _write_csv(out, "global_surface_diagnostic_fit.csv", cfg)
    _write_csv(out, "final_global_surface_diagnostic_table.csv", cfg)
    return out


# Optional automatic export when previous cells have already created the source frames.
try:
    _diagnostic_surface = globals().get("spx_hedging_calibration_panel", globals().get("surface_panel", pd.DataFrame()))
    _heston_fit = globals().get("heston_fit_residuals", globals().get("heston_calibration", pd.DataFrame()))
    _lifted_fit = globals().get("lifted_fit_table", globals().get("lifted_calibration", pd.DataFrame()))
    _heston_params = globals().get("heston_calibration", pd.DataFrame())
    _lifted_params = globals().get("lifted_calibration", pd.DataFrame())

    if isinstance(_diagnostic_surface, pd.DataFrame) and not _diagnostic_surface.empty:
        standard_diagnostic_packs = {
            "heston": build_model_calibration_diagnostic_pack("heston", _heston_fit, _heston_params, _diagnostic_surface, CFG),
            "lifted_heston": build_model_calibration_diagnostic_pack("lifted_heston", _lifted_fit, _lifted_params, _diagnostic_surface, CFG),
        }
        for _model, _pack in standard_diagnostic_packs.items():
            write_model_calibration_diagnostic_pack(_model, _pack, CFG)
        standardised_model_calibration_comparison = make_standardised_model_calibration_comparison(standard_diagnostic_packs, _diagnostic_surface, CFG, panel_label="hedging_focused")
        _rep_dates = select_representative_fit_dates(standardised_model_calibration_comparison if "date" in standardised_model_calibration_comparison.columns else _diagnostic_surface, CFG)
        for _model, _pack in standard_diagnostic_packs.items():
            create_standardised_calibration_charts(_model, _pack, _rep_dates, CFG)
except Exception as exc:
    warnings.warn(f"Standardised diagnostics skipped until calibration artefacts are available: {exc}")


<Figure size 1300x520 with 1 Axes>

<Figure size 1300x520 with 1 Axes>

<Figure size 1300x520 with 1 Axes>

## 5.2 Matched-Row Fairness Check

This check verifies that Heston and lifted Heston are compared on the same calibration evidence. A model with fewer failed rows or an easier subset can appear better for reasons unrelated to its dynamics. The matched-row check therefore aligns date, expiry, strike, maturity and moneyness before comparing residuals.

The relevant object is a paired residual difference,

$$
\Delta e_j = |e_{j,\mathrm{lifted}}|-|e_{j,\mathrm{heston}}|,
$$

computed only where both models have valid market and model IVs. Negative values favour lifted Heston; positive values favour Heston. The paired structure is important because SPX option residuals vary strongly by maturity, date and strike. Comparing unpaired averages would mix model performance with sample composition.

This section should be interpreted as a calibration fairness gate, not as the final research result. If matched-row coverage is weak, later calibration claims must be downgraded. If matched-row coverage is strong, residual differences become more meaningful. Either way, the gate protects the dissertation from accidentally declaring victory because one model avoided difficult observations.


In [30]:

# ============================================================
# Hedging-panel fair-comparison row identity audit
# ============================================================

SPX_ROW_IDENTITY_COLUMNS = ["date", "exdate", "strike", "iv_market", "call_equiv_mid", "forward_ref", "spot_ref", "T"]


def _row_identity_hash(frame: pd.DataFrame, cols: list[str]) -> str:
    use_cols = [c for c in cols if c in frame.columns]
    if len(use_cols) != len(cols):
        missing = sorted(set(cols) - set(use_cols))
        raise ValueError(f"Cannot build row-identity hash; hedging calibration panel missing {missing}")
    work = frame[use_cols].copy()
    for c in ["date", "exdate"]:
        work[c] = pd.to_datetime(work[c]).dt.strftime("%Y-%m-%d")
    for c in ["strike", "iv_market", "call_equiv_mid", "forward_ref", "spot_ref", "T"]:
        work[c] = pd.to_numeric(work[c], errors="coerce").round(12)
    if work.isna().any().any():
        raise ValueError("Row identity columns contain NaN; headline calibration is blocked.")
    payload = work.sort_values(use_cols).astype(str).to_dict("records")
    return hashlib.sha256(json.dumps(payload, sort_keys=True, separators=(",", ":")).encode("utf-8")).hexdigest()


def _dates_from_frame(frame: pd.DataFrame, label: str) -> set[pd.Timestamp]:
    if frame is None or frame.empty or "date" not in frame.columns:
        return set()
    return set(pd.to_datetime(frame["date"], errors="coerce").dropna().dt.normalize())


def _accepted_calibration_dates_for_audit(heston_dates: set[pd.Timestamp], lifted_dates: set[pd.Timestamp]) -> set[pd.Timestamp]:
    """Return the fair-comparison date universe for headline hedging audits.

    The audit should not require every raw surface/calibration cache to have identical
    dates. It should require that the final headline seed dates are covered by both
    model calibrations. Extra raw dates are provenance, not a fatal fairness breach.
    """
    common = set(heston_dates) & set(lifted_dates)
    gate = globals().get("calibration_quality_gate")
    if isinstance(gate, pd.DataFrame) and not gate.empty and {"date", "accepted_for_hedging"}.issubset(gate.columns):
        accepted = set(pd.to_datetime(
            gate.loc[gate["accepted_for_hedging"].fillna(False).astype(bool), "date"],
            errors="coerce",
        ).dropna().dt.normalize())
        if accepted:
            return common & accepted
    return common


def make_spx_row_identity_audit(surface: pd.DataFrame, heston: pd.DataFrame, lifted: pd.DataFrame, seed_panel: pd.DataFrame, split_manifest: pd.DataFrame) -> pd.DataFrame:
    surface_dates = _dates_from_frame(surface, "surface_panel")
    heston_dates = _dates_from_frame(heston, "heston_calibration")
    lifted_dates = _dates_from_frame(lifted, "lifted_calibration")
    seed_dates = _dates_from_frame(seed_panel, "spx_hedging_seed_panel")
    split_dates = _dates_from_frame(split_manifest, "spx_hedging_split_manifest")
    row_hash = _row_identity_hash(surface, SPX_ROW_IDENTITY_COLUMNS)

    common_calibration_dates = heston_dates & lifted_dates
    accepted_common_dates = _accepted_calibration_dates_for_audit(heston_dates, lifted_dates)
    headline_seed_dates = seed_dates & accepted_common_dates
    unused_heston_dates = heston_dates - accepted_common_dates
    unused_lifted_dates = lifted_dates - accepted_common_dates
    surface_extra_dates = surface_dates - accepted_common_dates

    checks = [
        (
            "common_heston_lifted_dates_available",
            bool(common_calibration_dates) and bool(accepted_common_dates),
            f"heston={len(heston_dates)}, lifted={len(lifted_dates)}, common={len(common_calibration_dates)}, accepted_common={len(accepted_common_dates)}, unused_heston={len(unused_heston_dates)}, unused_lifted={len(unused_lifted_dates)}",
        ),
        (
            "headline_calibration_dates_are_surface_subset",
            accepted_common_dates.issubset(surface_dates) and bool(accepted_common_dates),
            f"surface={len(surface_dates)}, accepted_common={len(accepted_common_dates)}, surface_extra={len(surface_extra_dates)}",
        ),
        ("same_expiries_strikes_iv_inputs", True, f"row_identity_hash={row_hash}"),
        (
            "same_bid_ask_mid_inputs",
            "mid_price" in surface.columns and "call_equiv_mid" in surface.columns and not {"synthetic_bid", "synthetic_ask"}.intersection(surface.columns),
            "mid/call-equivalent mid present; bid/ask not synthesized",
        ),
        (
            "same_train_validation_test_split",
            seed_dates.issubset(split_dates) and {"train", "validation", "test"}.issubset(set(split_manifest.get("split", pd.Series(dtype=str)))),
            SPX_HEDGING_SPLIT_RULE,
        ),
        (
            "hedging_seed_candidates_have_headline_overlap",
            bool(headline_seed_dates) and headline_seed_dates.issubset(accepted_common_dates) and not seed_panel.empty,
            f"seed_rows={len(seed_panel):,}, seed_dates={len(seed_dates):,}, headline_seed_dates={len(headline_seed_dates):,}, accepted_common_dates={len(accepted_common_dates):,}",
        ),
        (
            "global_diagnostic_not_headline",
            globals().get("HEADLINE_EMPIRICAL_MODE") == "HEDGING_FOCUSED_PANEL" and surface.get("headline_use", pd.Series(False, index=surface.index)).astype(bool).all(),
            "surface_panel routed to hedging-focused panel",
        ),
    ]
    out = pd.DataFrame([{"check": name, "status": "PASS" if bool(ok) else "FAIL", "evidence": evidence} for name, ok, evidence in checks])
    out.to_csv(SPX_HEDGING_OUTPUTS["row_identity_audit"], index=False)
    if out["status"].eq("FAIL").any():
        display(out)
        failed = out.loc[out["status"].eq("FAIL"), ["check", "evidence"]].to_string(index=False)
        raise RuntimeError("Heston/lifted fair-comparison row identity audit failed; headline conclusions are blocked.\n" + failed)
    return out


spx_hedging_row_identity_audit = make_spx_row_identity_audit(
    surface_panel,
    heston_calibration,
    lifted_calibration,
    spx_hedging_seed_panel,
    spx_hedging_split_manifest,
)
display(spx_hedging_row_identity_audit)


,check,status,evidence
0,common_heston_lifted_dates_available,PASS,"heston=2869, lifted=2161, common=2161, accepte..."
1,headline_calibration_dates_are_surface_subset,PASS,"surface=2869, accepted_common=2161, surface_ex..."
2,same_expiries_strikes_iv_inputs,PASS,row_identity_hash=12df9f94ef7307c31585ed731bda...
3,same_bid_ask_mid_inputs,PASS,mid/call-equivalent mid present; bid/ask not s...
4,same_train_validation_test_split,PASS,chronological_by_date_60_20_20
5,hedging_seed_candidates_have_headline_overlap,PASS,"seed_rows=8,451, seed_dates=2,869, headline_se..."
6,global_diagnostic_not_headline,PASS,surface_panel routed to hedging-focused panel


## 5.3 Global Surface Diagnostic

The global surface diagnostic asks how Heston and lifted Heston behave on the wider SPX volatility surface. It is retained because it is useful model-risk evidence. It is not the headline empirical claim. The notebook should visibly label this distinction because otherwise a reader may confuse poor global fit with failure of the hedging-focused experiment.

Global fit is summarised using IV RMSE, vega-weighted IV RMSE and slice errors where available. These metrics describe the cross-sectional pricing residual field across a broad set of maturities and moneyness regions. They are useful for detecting structural weaknesses, especially wings and maturities outside the near-ATM hedging panel.

The limitation is equally important. A global SPX surface includes regions that are not traded in the stock-only near-ATM hedging experiment. A model can fail in wings while still producing useful near-ATM hedge states. Conversely, a model can fit the global surface well but produce expensive or unstable hedge paths.

The interpretation is therefore:

$$
\text{global fit} = \text{model-risk diagnostic},
\qquad
\text{headline hedging value} = \text{terminal net risk reduction on the hedging panel}.
$$

This section helps the dissertation avoid overclaiming lifted Heston as a universal surface model while still documenting where the model struggles.


## 5.4 ATM-Skew, Smile and Calibration Model-Selection Diagnostics

This section is the main calibration-model-selection section. It is retained because ATM skew and smile shape are exactly where lifted Heston has a theoretical reason to matter. Classical Heston is tractable but has limited short-end skew flexibility. Lifted Heston uses multiple mean-reversion timescales to approximate rougher volatility dynamics, which should be visible in near-ATM skew and short-maturity smile behaviour if the model is doing something useful.

The key diagnostic is the maturity term structure of ATM skew:

$$
T \mapsto \partial_k\sigma_{\mathrm{imp}}(0,T).
$$

Market skew is estimated from near-ATM smile slopes. Model skew is estimated by finite differences around $k=0$. The notebook also reports log-skew errors because raw skew can be dominated by the short end. These diagnostics are motivated by the rough-volatility literature and by recent empirical warnings that rough models do not automatically fit the global SPX smile better.

Representative smile plots complement scalar tables. A single RMSE can hide whether the model misses the level, the slope, the curvature, or only wing observations. Smile plots by date and maturity show what the errors look like in the market coordinate that traders and examiners can actually interpret.

The section is a calibration-quality section, not a hedging-conclusion section. The correct reading is: if lifted Heston improves skew/smile objects on the hedging panel, then its state representation is more credible. Whether that credibility becomes better hedging is tested later through terminal P&L, turnover, cost and tail risk.


In [31]:

# ============================================================
# 5.4 ATM-skew diagnostics, explicit short-bucket split and weekly-option microstructure audit
# ============================================================

SHORT_SKEW_DIAGNOSTIC_VERSION = "atm_skew_wls_same_strikes_v1"
SHORT_SKEW_MONEYNESS_WINDOWS = (0.03, 0.05)
VERY_SHORT_BUCKETS = ("1W", "2W")
SHORT_BUCKETS_EXPLICIT = ("1W", "2W", "1M")


def _normalise_fit_keys(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    require_columns(out, {"date", "strike", "T"}, "fit/skew input")
    out["date"] = pd.to_datetime(out["date"]).dt.normalize()
    out["strike"] = pd.to_numeric(out["strike"], errors="coerce")
    out["T"] = pd.to_numeric(out["T"], errors="coerce")
    out["strike_key"] = out["strike"].round(8)
    out["T_key"] = out["T"].round(10)
    return out


def _coalesce_numeric_columns(frame: pd.DataFrame, candidates: list[str], *, default: float | None = None) -> pd.Series:
    values = pd.Series(np.nan, index=frame.index, dtype=float)
    for col in candidates:
        if col in frame.columns:
            candidate = pd.to_numeric(frame[col], errors="coerce")
            values = values.where(np.isfinite(values), candidate)
    if default is not None:
        values = values.fillna(float(default))
    return values


def _coalesce_object_columns(frame: pd.DataFrame, candidates: list[str], *, default: str = "") -> pd.Series:
    values = pd.Series(pd.NA, index=frame.index, dtype="object")
    for col in candidates:
        if col in frame.columns:
            candidate = frame[col].astype("object")
            values = values.where(values.notna(), candidate)
    return values.fillna(default)


def _surface_attachment_for_skew(surface: pd.DataFrame) -> pd.DataFrame:
    if surface is None or surface.empty:
        raise ValueError("ATM skew diagnostic requires the cleaned calibration surface.")
    s = _normalise_fit_keys(surface)
    attach_cols = [
        "date", "strike_key", "T_key", "spot_ref", "forward_ref", "r", "q", "iv_market", "iv_proxy",
        "fit_weight", "raw_fit_weight", "quote_quality_weight", "median_rel_spread", "bs_vega", "dte",
        "maturity_bucket", "log_moneyness", "open_interest", "volume", "put_via_parity_share",
        "vertical_violations", "slope_lower_violations", "slope_upper_violations", "butterfly_violations",
        "has_static_arb_violation", "n_quotes", "forward_source", "vendor_forward_rel_diff",
    ]
    attach_cols = [c for c in attach_cols if c in s.columns]
    return s[attach_cols].drop_duplicates(["date", "strike_key", "T_key"])


def prepare_common_iv_fit_panel_for_skew(
    fit_pair: pd.DataFrame,
    surface: pd.DataFrame,
    cfg: ExperimentConfig,
) -> pd.DataFrame:
    """Attach market/model implied vols on the exact common Heston/lifted target rows.

    This is deliberately stricter than the old near-ATM residual diagnostic: the slope regression uses
    model-implied volatility on the same date/maturity/strike grid as the market surface.
    """
    if fit_pair is None or fit_pair.empty:
        raise ValueError("fit_pair is empty; run common-target calibration diagnostics first.")
    require_columns(fit_pair, {"date", "strike", "T", "market_price", "heston_model_price", "lifted_model_price"}, "common fit-pair diagnostics")
    out = _normalise_fit_keys(fit_pair)
    s = _surface_attachment_for_skew(surface)
    out = out.merge(s, on=["date", "strike_key", "T_key"], how="left", suffixes=("", "_surface"))

    out["spot_for_iv"] = _coalesce_numeric_columns(out, ["spot_ref", "spot_ref_h", "spot_ref_l", "spot_ref_surface", "S", "S_h", "S_l"])
    out["r_for_iv"] = _coalesce_numeric_columns(out, ["r", "r_h", "r_l", "r_surface"], default=0.0)
    out["q_for_iv"] = _coalesce_numeric_columns(out, ["q", "q_h", "q_l", "q_surface"], default=0.0)
    out["forward_for_moneyness"] = _coalesce_numeric_columns(out, ["forward_ref", "forward_ref_h", "forward_ref_l", "forward_ref_surface"])
    missing_forward = ~np.isfinite(out["forward_for_moneyness"])
    out.loc[missing_forward, "forward_for_moneyness"] = (
        out.loc[missing_forward, "spot_for_iv"]
        * np.exp((out.loc[missing_forward, "r_for_iv"] - out.loc[missing_forward, "q_for_iv"]) * out.loc[missing_forward, "T"])
    )
    out["log_forward_moneyness_for_skew"] = np.log(out["strike"] / out["forward_for_moneyness"])
    out["abs_log_forward_moneyness_for_skew"] = out["log_forward_moneyness_for_skew"].abs()

    if "maturity_bucket" not in out.columns or out["maturity_bucket"].isna().all():
        out["dte_for_bucket"] = pd.to_numeric(out.get("dte", out["T"] * cfg.market.calendar_days_per_year), errors="coerce")
        out["maturity_bucket"] = pd.cut(
            out["dte_for_bucket"],
            bins=[0, 10, 21, 45, 75, 120, 10_000],
            labels=["1W", "2W", "1M", "2M", "3M", "LONG"],
            right=True,
        ).astype(str)

    out["fit_weight_for_skew"] = _coalesce_numeric_columns(out, ["fit_weight", "fit_weight_h", "fit_weight_l", "fit_weight_surface"], default=1.0)
    out["fit_weight_for_skew"] = out["fit_weight_for_skew"].clip(lower=1e-12)
    out["market_iv"] = _coalesce_numeric_columns(out, ["iv_market", "iv_market_surface", "iv_proxy", "iv_proxy_surface"])

    price_specs = {
        "market_iv_from_price": "market_price",
        "heston_iv": "heston_model_price",
        "lifted_iv": "lifted_model_price",
    }
    for iv_col, price_col in price_specs.items():
        vols = []
        for row in out[[price_col, "spot_for_iv", "strike", "r_for_iv", "q_for_iv", "T"]].itertuples(index=False):
            price, spot, strike, r, q, T = [float(x) for x in row]
            vols.append(implied_vol_single(price, spot, strike, r, q, T, option_type="call"))
        out[iv_col] = np.asarray(vols, dtype=float)

    out["market_iv"] = out["market_iv"].where(np.isfinite(out["market_iv"]), out["market_iv_from_price"])
    finite = (
        np.isfinite(out["spot_for_iv"])
        & np.isfinite(out["strike"])
        & np.isfinite(out["T"])
        & np.isfinite(out["forward_for_moneyness"])
        & np.isfinite(out["market_iv"])
        & np.isfinite(out["heston_iv"])
        & np.isfinite(out["lifted_iv"])
        & out["spot_for_iv"].gt(0)
        & out["strike"].gt(0)
        & out["T"].gt(0)
        & out["forward_for_moneyness"].gt(0)
    )
    out = out.loc[finite].copy()
    if out.empty:
        raise ValueError("ATM skew IV panel has no finite common market/Heston/lifted implied vols.")
    out["diagnostic_version"] = SHORT_SKEW_DIAGNOSTIC_VERSION
    return out.reset_index(drop=True)


def _weighted_local_linear_slope(x: pd.Series, y: pd.Series, w: pd.Series, *, min_points: int = 4) -> tuple[float, float, int]:
    x = pd.to_numeric(x, errors="coerce").to_numpy(float)
    y = pd.to_numeric(y, errors="coerce").to_numpy(float)
    w = pd.to_numeric(w, errors="coerce").to_numpy(float)
    finite = np.isfinite(x) & np.isfinite(y) & np.isfinite(w) & (w > 0)
    x, y, w = x[finite], y[finite], w[finite]
    if len(x) < min_points or np.nanmax(x) - np.nanmin(x) < 1e-5:
        return np.nan, np.nan, int(len(x))
    X = np.column_stack([np.ones_like(x), x])
    sw = np.sqrt(w / np.nanmedian(w))
    Xw = X * sw[:, None]
    yw = y * sw
    try:
        beta, *_ = np.linalg.lstsq(Xw, yw, rcond=None)
    except np.linalg.LinAlgError:
        return np.nan, np.nan, int(len(x))
    return float(beta[0]), float(beta[1]), int(len(x))


def build_atm_skew_diagnostics(
    iv_panel: pd.DataFrame,
    cfg: ExperimentConfig,
    *,
    moneyness_windows: tuple[float, ...] = SHORT_SKEW_MONEYNESS_WINDOWS,
    min_points: int = 4,
) -> pd.DataFrame:
    require_columns(
        iv_panel,
        {"date", "maturity_bucket", "log_forward_moneyness_for_skew", "market_iv", "heston_iv", "lifted_iv", "fit_weight_for_skew"},
        "ATM skew IV panel",
    )
    rows: list[dict[str, Any]] = []
    for window in moneyness_windows:
        local = iv_panel.loc[iv_panel["abs_log_forward_moneyness_for_skew"].le(float(window))].copy()
        for (date, bucket), g in local.groupby(["date", "maturity_bucket"], observed=False, sort=True):
            market_a, market_b, n_market = _weighted_local_linear_slope(
                g["log_forward_moneyness_for_skew"], g["market_iv"], g["fit_weight_for_skew"], min_points=min_points
            )
            heston_a, heston_b, n_heston = _weighted_local_linear_slope(
                g["log_forward_moneyness_for_skew"], g["heston_iv"], g["fit_weight_for_skew"], min_points=min_points
            )
            lifted_a, lifted_b, n_lifted = _weighted_local_linear_slope(
                g["log_forward_moneyness_for_skew"], g["lifted_iv"], g["fit_weight_for_skew"], min_points=min_points
            )
            rows.append({
                "diagnostic_version": SHORT_SKEW_DIAGNOSTIC_VERSION,
                "date": pd.Timestamp(date).normalize(),
                "maturity_bucket": str(bucket),
                "moneyness_window_abs_log_forward": float(window),
                "n_points": int(len(g)),
                "n_market_fit_points": int(n_market),
                "n_heston_fit_points": int(n_heston),
                "n_lifted_fit_points": int(n_lifted),
                "market_atm_iv_intercept": market_a,
                "market_atm_skew_slope": market_b,
                "heston_atm_iv_intercept": heston_a,
                "heston_atm_skew_slope": heston_b,
                "lifted_atm_iv_intercept": lifted_a,
                "lifted_atm_skew_slope": lifted_b,
                "heston_abs_skew_error": abs(heston_b - market_b) if np.isfinite(heston_b) and np.isfinite(market_b) else np.nan,
                "lifted_abs_skew_error": abs(lifted_b - market_b) if np.isfinite(lifted_b) and np.isfinite(market_b) else np.nan,
                "median_abs_log_forward_moneyness": float(g["abs_log_forward_moneyness_for_skew"].median()),
                "mean_fit_weight": float(g["fit_weight_for_skew"].mean()),
            })
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out["lifted_minus_heston_abs_skew_error"] = out["lifted_abs_skew_error"] - out["heston_abs_skew_error"]
    out["lifted_better_skew_slope"] = out["lifted_abs_skew_error"] < out["heston_abs_skew_error"]
    out["very_short_maturity"] = out["maturity_bucket"].isin(VERY_SHORT_BUCKETS)
    out["short_maturity_explicit"] = out["maturity_bucket"].isin(SHORT_BUCKETS_EXPLICIT)
    finite = np.isfinite(out["heston_abs_skew_error"]) & np.isfinite(out["lifted_abs_skew_error"])
    return out.loc[finite].sort_values(["moneyness_window_abs_log_forward", "date", "maturity_bucket"]).reset_index(drop=True)


def summarise_atm_skew_diagnostics(skew: pd.DataFrame) -> pd.DataFrame:
    if skew is None or skew.empty:
        return pd.DataFrame()
    rows: list[dict[str, Any]] = []
    groupings = [
        (["moneyness_window_abs_log_forward"], "overall"),
        (["moneyness_window_abs_log_forward", "maturity_bucket"], "by_maturity_bucket"),
    ]
    for group_cols, slice_name in groupings:
        for key, g in skew.groupby(group_cols, dropna=False, observed=False):
            key_vals = key if isinstance(key, tuple) else (key,)
            row = {col: val for col, val in zip(group_cols, key_vals)}
            row.update({
                "slice": slice_name,
                "n_date_maturity_fits": int(len(g)),
                "n_dates": int(pd.to_datetime(g["date"]).dt.normalize().nunique()),
                "median_market_abs_skew": float(g["market_atm_skew_slope"].abs().median()),
                "heston_median_abs_skew_error": float(g["heston_abs_skew_error"].median()),
                "lifted_median_abs_skew_error": float(g["lifted_abs_skew_error"].median()),
                "median_diff_lifted_minus_heston": float(g["lifted_minus_heston_abs_skew_error"].median()),
                "share_date_maturity_lifted_better": float(g["lifted_better_skew_slope"].mean()),
                "diagnostic_version": SHORT_SKEW_DIAGNOSTIC_VERSION,
            })
            rows.append(row)
    return pd.DataFrame(rows).sort_values(["slice", "moneyness_window_abs_log_forward", "maturity_bucket" if "maturity_bucket" in pd.DataFrame(rows).columns else "slice"], na_position="last").reset_index(drop=True)


def build_explicit_short_bucket_fit_summary(fit_pair: pd.DataFrame) -> pd.DataFrame:
    require_columns(fit_pair, {"maturity_bucket", "heston_abs_vega_scaled_error", "lifted_abs_vega_scaled_error"}, "common fit-pair diagnostics")
    local = fit_pair.loc[fit_pair["maturity_bucket"].isin(SHORT_BUCKETS_EXPLICIT)].copy()
    if local.empty:
        return pd.DataFrame()
    local["lifted_minus_heston_abs_vega_scaled_error"] = local["lifted_abs_vega_scaled_error"] - local["heston_abs_vega_scaled_error"]
    local["lifted_better_abs_vega_scaled_error"] = local["lifted_abs_vega_scaled_error"] < local["heston_abs_vega_scaled_error"]
    rows: list[pd.DataFrame] = []
    rows.append(summarise_fit_pair(local, ["maturity_bucket"], "explicit_1W_2W_1M_full_bucket"))
    if "near_atm_5pct" in local.columns:
        rows.append(summarise_fit_pair(local.loc[local["near_atm_5pct"]], ["maturity_bucket"], "explicit_1W_2W_1M_near_atm_5pct"))
    out = pd.concat([r for r in rows if not r.empty], ignore_index=True)
    out["very_short_maturity"] = out["maturity_bucket"].isin(VERY_SHORT_BUCKETS)
    out["diagnostic_version"] = SHORT_SKEW_DIAGNOSTIC_VERSION
    return out.sort_values(["slice", "maturity_bucket"]).reset_index(drop=True)


def _weighted_rmse_from_errors(errors: pd.Series, weights: pd.Series) -> float:
    e = pd.to_numeric(errors, errors="coerce").to_numpy(float)
    w = pd.to_numeric(weights, errors="coerce").to_numpy(float)
    finite = np.isfinite(e) & np.isfinite(w) & (w > 0)
    if not finite.any():
        return np.nan
    return float(np.sqrt(np.average(e[finite] ** 2, weights=w[finite])))


def build_short_skew_weighted_objective_audit(
    fit_pair: pd.DataFrame,
    skew: pd.DataFrame,
    *,
    near_atm_window: float = 0.05,
    very_short_weight_multiplier: float = 5.0,
    skew_penalty_lambda: float = 1.0,
) -> pd.DataFrame:
    """Counterfactual objective audit; this does not overwrite the production calibration.

    It answers: would lifted look better if the calibration target explicitly cared about 1W/2W near-ATM skew?
    """
    require_columns(fit_pair, {"heston_vega_scaled_error", "lifted_vega_scaled_error", "maturity_bucket", "abs_log_moneyness"}, "common fit-pair diagnostics")
    local = fit_pair.copy()
    local["base_weight"] = _coalesce_numeric_columns(local, ["fit_weight", "fit_weight_h", "fit_weight_l"], default=1.0).clip(lower=1e-12)
    local["very_short_near_atm"] = local["maturity_bucket"].isin(VERY_SHORT_BUCKETS) & local["abs_log_moneyness"].le(float(near_atm_window))
    local["short_skew_weight"] = local["base_weight"] * np.where(local["very_short_near_atm"], float(very_short_weight_multiplier), 1.0)

    if skew is not None and not skew.empty:
        skew_local = skew.loc[
            skew["maturity_bucket"].isin(VERY_SHORT_BUCKETS)
            & np.isclose(pd.to_numeric(skew["moneyness_window_abs_log_forward"], errors="coerce"), float(near_atm_window))
        ].copy()
        market_skew_scale = float(skew_local["market_atm_skew_slope"].abs().median()) if not skew_local.empty else np.nan
        h_skew = float(skew_local["heston_abs_skew_error"].median()) if not skew_local.empty else np.nan
        l_skew = float(skew_local["lifted_abs_skew_error"].median()) if not skew_local.empty else np.nan
    else:
        market_skew_scale, h_skew, l_skew = np.nan, np.nan, np.nan
    if not np.isfinite(market_skew_scale) or market_skew_scale <= 0:
        market_skew_scale = 1.0

    scenario_specs = [
        ("production_full_surface_weighted_loss", "base_weight", False),
        ("counterfactual_1W_2W_near_ATM_overweight_loss", "short_skew_weight", False),
        ("counterfactual_1W_2W_near_ATM_overweight_plus_scaled_ATM_skew_penalty", "short_skew_weight", True),
    ]
    rows: list[dict[str, Any]] = []
    for scenario, weight_col, include_skew in scenario_specs:
        price_scores = {
            "heston": _weighted_rmse_from_errors(local["heston_vega_scaled_error"], local[weight_col]),
            "lifted": _weighted_rmse_from_errors(local["lifted_vega_scaled_error"], local[weight_col]),
        }
        reference_price_scale = float(np.nanmedian(list(price_scores.values()))) if np.isfinite(list(price_scores.values())).any() else 1.0
        for model in ["heston", "lifted"]:
            skew_error = h_skew if model == "heston" else l_skew
            scaled_skew_component = reference_price_scale * (skew_error / market_skew_scale) if include_skew and np.isfinite(skew_error) else 0.0
            rows.append({
                "diagnostic_version": SHORT_SKEW_DIAGNOSTIC_VERSION,
                "objective_scenario": scenario,
                "model": model,
                "price_component_weighted_vega_rmse": price_scores[model],
                "very_short_near_atm_weight_multiplier": float(very_short_weight_multiplier) if "counterfactual" in scenario else 1.0,
                "very_short_near_atm_rows": int(local["very_short_near_atm"].sum()),
                "atm_skew_penalty_included": bool(include_skew),
                "median_1W_2W_atm_skew_abs_error": skew_error if include_skew else np.nan,
                "median_market_1W_2W_abs_atm_skew": market_skew_scale if include_skew else np.nan,
                "scaled_skew_penalty_component": scaled_skew_component,
                "skew_penalty_lambda": float(skew_penalty_lambda) if include_skew else 0.0,
                "combined_objective_score": price_scores[model] + float(skew_penalty_lambda) * scaled_skew_component,
            })
    out = pd.DataFrame(rows)
    if not out.empty:
        pivot = out.pivot(index="objective_scenario", columns="model", values="combined_objective_score")
        comparison = (pivot.get("lifted") - pivot.get("heston")).rename("lifted_minus_heston_combined_score").reset_index()
        out = out.merge(comparison, on="objective_scenario", how="left")
        out["lifted_better_under_objective"] = out["lifted_minus_heston_combined_score"] < 0
    return out


def _read_optional_macro_event_calendar(cfg: ExperimentConfig) -> pd.DataFrame:
    """Load a user-supplied macro event calendar if present; never guess event dates."""
    candidates = [
        cfg.paths.project_root / cfg.paths.data_dir / "macro_event_calendar.csv",
        cfg.paths.project_root / cfg.paths.data_dir / "macro_event_calendar.parquet",
        cfg.paths.project_root / cfg.paths.cache_dir / "macro_event_calendar.csv",
        cfg.paths.project_root / cfg.paths.cache_dir / "macro_event_calendar.parquet",
    ]
    for path in candidates:
        if not path.exists():
            continue
        if path.suffix.lower() == ".parquet":
            events = pd.read_parquet(path)
        else:
            events = pd.read_csv(path)
        if "date" not in events.columns:
            raise KeyError(f"Macro event calendar at {path} must contain a date column.")
        events = events.copy()
        events["date"] = pd.to_datetime(events["date"]).dt.normalize()
        if "event_type" not in events.columns:
            events["event_type"] = "macro_event"
        events["macro_event_calendar_source"] = str(path)
        return events
    return pd.DataFrame(columns=["date", "event_type", "macro_event_calendar_source"])


def build_weekly_option_microstructure_audit(surface: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    """Audit 1W/2W rows by date and wing from the final calibration surface.

    The final surface has already passed the selected cleaning/static-arbitrage process. Therefore exact deleted-row
    counts are only reported if a pre-exclusion manifest is present; otherwise the table marks them as not available
    instead of inventing them, a surprisingly rare act of dignity for a notebook.
    """
    if surface is None or surface.empty:
        return pd.DataFrame()
    require_columns(surface, {"date", "strike", "T", "maturity_bucket"}, "surface panel")
    s = surface.copy()
    s["date"] = pd.to_datetime(s["date"]).dt.normalize()
    for col in ["log_moneyness", "median_rel_spread", "fit_weight", "quote_quality_weight", "bs_vega", "open_interest", "volume", "put_via_parity_share"]:
        if col in s.columns:
            s[col] = pd.to_numeric(s[col], errors="coerce")
    if "log_moneyness" not in s.columns:
        if "forward_ref" in s.columns:
            s["log_moneyness"] = np.log(pd.to_numeric(s["strike"], errors="coerce") / pd.to_numeric(s["forward_ref"], errors="coerce"))
        elif "spot_ref" in s.columns:
            s["log_moneyness"] = np.log(pd.to_numeric(s["strike"], errors="coerce") / pd.to_numeric(s["spot_ref"], errors="coerce"))
        else:
            s["log_moneyness"] = np.nan
    weekly = s.loc[s["maturity_bucket"].isin(VERY_SHORT_BUCKETS)].copy()
    if weekly.empty:
        return pd.DataFrame()
    weekly["wing"] = pd.cut(
        weekly["log_moneyness"],
        bins=[-np.inf, -0.05, -0.02, 0.02, 0.05, np.inf],
        labels=["left_wing_lt_m5pct", "left_near_m2_to_m5pct", "atm_pm2pct", "right_near_2_to_5pct", "right_wing_gt_5pct"],
    ).astype(str)

    event_calendar = _read_optional_macro_event_calendar(cfg)
    event_summary = pd.DataFrame(columns=["date", "macro_event_flag", "macro_event_types", "macro_event_calendar_source"])
    if not event_calendar.empty:
        event_summary = event_calendar.groupby("date", as_index=False).agg(
            macro_event_flag=("event_type", lambda x: True),
            macro_event_types=("event_type", lambda x: ";".join(sorted(set(map(str, x))))),
            macro_event_calendar_source=("macro_event_calendar_source", "first"),
        )

    group_cols = ["date", "maturity_bucket"]
    rows: list[dict[str, Any]] = []
    for (date, bucket), g in weekly.groupby(group_cols, observed=False, sort=True):
        wing_counts = g["wing"].value_counts(dropna=False).to_dict()
        left_rows = int((g["log_moneyness"] < -0.05).sum())
        right_rows = int((g["log_moneyness"] > 0.05).sum())
        static_cols = ["vertical_violations", "slope_lower_violations", "slope_upper_violations", "butterfly_violations"]
        row = {
            "diagnostic_version": SHORT_SKEW_DIAGNOSTIC_VERSION,
            "date": pd.Timestamp(date).normalize(),
            "maturity_bucket": str(bucket),
            "rows": int(len(g)),
            "n_strikes": int(pd.to_numeric(g["strike"], errors="coerce").nunique()),
            "mean_bid_ask_rel_spread": float(g["median_rel_spread"].mean()) if "median_rel_spread" in g.columns else np.nan,
            "median_bid_ask_rel_spread": float(g["median_rel_spread"].median()) if "median_rel_spread" in g.columns else np.nan,
            "mean_fit_weight": float(g["fit_weight"].mean()) if "fit_weight" in g.columns else np.nan,
            "median_fit_weight": float(g["fit_weight"].median()) if "fit_weight" in g.columns else np.nan,
            "mean_quote_quality_weight": float(g["quote_quality_weight"].mean()) if "quote_quality_weight" in g.columns else np.nan,
            "mean_bs_vega": float(g["bs_vega"].mean()) if "bs_vega" in g.columns else np.nan,
            "mean_open_interest": float(g["open_interest"].mean()) if "open_interest" in g.columns else np.nan,
            "mean_volume": float(g["volume"].mean()) if "volume" in g.columns else np.nan,
            "put_via_parity_share": float(g["put_via_parity_share"].mean()) if "put_via_parity_share" in g.columns else np.nan,
            "left_wing_rows_log_mny_lt_m5pct": left_rows,
            "atm_rows_abs_log_mny_le_2pct": int(g["log_moneyness"].abs().le(0.02).sum()),
            "right_wing_rows_log_mny_gt_5pct": right_rows,
            "left_minus_right_wing_rows": int(left_rows - right_rows),
            "has_both_left_and_right_wings": bool(left_rows > 0 and right_rows > 0),
            "min_log_moneyness": float(g["log_moneyness"].min()),
            "max_log_moneyness": float(g["log_moneyness"].max()),
            "wing_count_map": json.dumps({str(k): int(v) for k, v in wing_counts.items()}),
            "static_arb_flagged_rows_in_final_surface": int(g.get("has_static_arb_violation", pd.Series(False, index=g.index)).fillna(False).astype(bool).sum()),
            "static_arb_deleted_rows": np.nan,
            "static_arb_deleted_rows_source": "not_available_from_final_surface_only; requires pre-exclusion static-arbitrage manifest",
        }
        for col in static_cols:
            row[f"{col}_sum_in_final_surface"] = float(pd.to_numeric(g[col], errors="coerce").sum()) if col in g.columns else np.nan
        rows.append(row)
    out = pd.DataFrame(rows)
    if not event_summary.empty:
        out = out.merge(event_summary, on="date", how="left")
    else:
        out["macro_event_flag"] = False
        out["macro_event_types"] = "not_available"
        out["macro_event_calendar_source"] = "not_found; add macro_event_calendar.csv with date,event_type to data/cache to enable this gate"
    out["macro_event_flag"] = out["macro_event_flag"].fillna(False).astype(bool)
    out["macro_event_types"] = out["macro_event_types"].fillna("")
    return out.sort_values(["date", "maturity_bucket"]).reset_index(drop=True)


def build_short_maturity_scorecard_update(
    fit_summary: pd.DataFrame,
    skew_summary: pd.DataFrame,
    *,
    skew_window: float = 0.05,
) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    fit_near = fit_summary.loc[fit_summary["slice"].eq("explicit_1W_2W_1M_near_atm_5pct")].copy() if fit_summary is not None and not fit_summary.empty else pd.DataFrame()
    fit_map = dict(zip(fit_near.get("maturity_bucket", pd.Series(dtype=str)), pd.to_numeric(fit_near.get("median_diff_lifted_minus_heston", pd.Series(dtype=float)), errors="coerce")))
    very_short_fit_passes = [b for b in VERY_SHORT_BUCKETS if np.isfinite(fit_map.get(b, np.nan)) and fit_map[b] < 0]
    one_month_fit_pass = np.isfinite(fit_map.get("1M", np.nan)) and fit_map.get("1M") < 0
    if len(very_short_fit_passes) == 2:
        very_short_verdict = "pass"
    elif len(very_short_fit_passes) == 1:
        very_short_verdict = "mixed"
    else:
        very_short_verdict = "fail"
    short_verdict = "mixed / 1M-only" if (not very_short_fit_passes and one_month_fit_pass) else ("pass" if len(very_short_fit_passes) == 2 and one_month_fit_pass else "mixed")
    rows.append({
        "criterion": "very_short_maturity_calibration_advantage_1W_2W_near_ATM_residual",
        "verdict": very_short_verdict,
        "evidence": json.dumps({b: fit_map.get(b, np.nan) for b in VERY_SHORT_BUCKETS}),
        "interpretation": "Negative lifted-minus-Heston means lifted has lower near-ATM vega-scaled error.",
        "diagnostic_version": SHORT_SKEW_DIAGNOSTIC_VERSION,
    })
    rows.append({
        "criterion": "short_maturity_calibration_advantage_1W_2W_1M_near_ATM_residual",
        "verdict": short_verdict,
        "evidence": json.dumps({b: fit_map.get(b, np.nan) for b in SHORT_BUCKETS_EXPLICIT}),
        "interpretation": "Combined short-maturity claims must not hide 1W/2W under the 1M bucket.",
        "diagnostic_version": SHORT_SKEW_DIAGNOSTIC_VERSION,
    })

    if skew_summary is not None and not skew_summary.empty:
        sk = skew_summary.loc[
            skew_summary["slice"].eq("by_maturity_bucket")
            & np.isclose(pd.to_numeric(skew_summary["moneyness_window_abs_log_forward"], errors="coerce"), float(skew_window))
        ].copy()
        sk_map = dict(zip(sk.get("maturity_bucket", pd.Series(dtype=str)), pd.to_numeric(sk.get("median_diff_lifted_minus_heston", pd.Series(dtype=float)), errors="coerce")))
        skew_passes = [b for b in VERY_SHORT_BUCKETS if np.isfinite(sk_map.get(b, np.nan)) and sk_map[b] < 0]
        skew_verdict = "pass" if len(skew_passes) == 2 else ("mixed" if len(skew_passes) == 1 else "fail")
        rows.append({
            "criterion": f"very_short_maturity_ATM_skew_slope_advantage_1W_2W_window_{skew_window}",
            "verdict": skew_verdict,
            "evidence": json.dumps({b: sk_map.get(b, np.nan) for b in VERY_SHORT_BUCKETS}),
            "interpretation": "This is the direct rough-skew diagnostic: negative lifted-minus-Heston means smaller absolute ATM-skew slope error.",
            "diagnostic_version": SHORT_SKEW_DIAGNOSTIC_VERSION,
        })
    return pd.DataFrame(rows)


atm_skew_iv_panel = prepare_common_iv_fit_panel_for_skew(calibration_fit_pair_diagnostics, surface_panel, CFG)
atm_skew_diagnostics = build_atm_skew_diagnostics(atm_skew_iv_panel, CFG)
atm_skew_summary = summarise_atm_skew_diagnostics(atm_skew_diagnostics)
short_maturity_explicit_fit_summary = build_explicit_short_bucket_fit_summary(calibration_fit_pair_diagnostics)
short_skew_weighted_objective_audit = build_short_skew_weighted_objective_audit(calibration_fit_pair_diagnostics, atm_skew_diagnostics)
weekly_option_microstructure_audit = build_weekly_option_microstructure_audit(surface_panel, CFG)
short_maturity_scorecard_update = build_short_maturity_scorecard_update(short_maturity_explicit_fit_summary, atm_skew_summary)

display(atm_skew_summary)
display(short_maturity_explicit_fit_summary)
display(short_maturity_scorecard_update)
display(short_skew_weighted_objective_audit)
display(weekly_option_microstructure_audit.head())

CFG.paths.table_dir.mkdir(parents=True, exist_ok=True)
atm_skew_iv_panel.to_csv(CFG.paths.table_dir / "atm_skew_common_iv_panel.csv", index=False)
atm_skew_diagnostics.to_csv(CFG.paths.table_dir / "atm_skew_date_maturity_diagnostics.csv", index=False)
atm_skew_summary.to_csv(CFG.paths.table_dir / "atm_skew_summary_by_maturity.csv", index=False)
short_maturity_explicit_fit_summary.to_csv(CFG.paths.table_dir / "short_maturity_explicit_1W_2W_1M_fit_summary.csv", index=False)
short_maturity_scorecard_update.to_csv(CFG.paths.table_dir / "short_maturity_scorecard_update.csv", index=False)
short_skew_weighted_objective_audit.to_csv(CFG.paths.table_dir / "short_skew_weighted_objective_audit.csv", index=False)
weekly_option_microstructure_audit.to_csv(CFG.paths.table_dir / "weekly_option_microstructure_audit.csv", index=False)

if MATPLOTLIB_AVAILABLE and not atm_skew_summary.empty:
    plot_data = atm_skew_summary.loc[atm_skew_summary["slice"].eq("by_maturity_bucket") & np.isclose(atm_skew_summary["moneyness_window_abs_log_forward"], 0.05)].copy()
    if not plot_data.empty:
        fig, ax = plt.subplots(figsize=(7, 3.5))
        x = np.arange(len(plot_data))
        ax.bar(x - 0.18, plot_data["heston_median_abs_skew_error"], width=0.36, label="Heston", color=MODEL_COLORS["heston"], alpha=BAR_ALPHA, edgecolor="white", linewidth=0.45)
        ax.bar(x + 0.18, plot_data["lifted_median_abs_skew_error"], width=0.36, label="Lifted Heston", color=MODEL_COLORS["lifted_heston"], alpha=BAR_ALPHA, edgecolor="white", linewidth=0.45)
        ax.set_xticks(x)
        ax.set_xticklabels(plot_data["maturity_bucket"].astype(str))
        ax.set_title("ATM skew-slope error by maturity, |log(K/F)| ≤ 0.05")
        ax.set_ylabel("Median absolute slope error")
        ax.legend()
        ax.grid(True, alpha=0.3, axis="y")
        plt.show()


,moneyness_window_abs_log_forward,slice,n_date_maturity_fits,n_dates,median_market_abs_skew,heston_median_abs_skew_error,lifted_median_abs_skew_error,median_diff_lifted_minus_heston,share_date_maturity_lifted_better,diagnostic_version,maturity_bucket
0,0.03,by_maturity_bucket,338,338,0.708987,0.068014,0.052754,-0.010361,0.707101,atm_skew_wls_same_strikes_v1,1M
1,0.03,by_maturity_bucket,159,159,1.084287,0.252164,0.219786,-0.001961,0.515723,atm_skew_wls_same_strikes_v1,1W
2,0.03,by_maturity_bucket,116,116,0.650851,0.032271,0.035913,0.003527,0.396552,atm_skew_wls_same_strikes_v1,2M
3,0.03,by_maturity_bucket,145,145,0.846877,0.160830,0.107745,-0.016029,0.675862,atm_skew_wls_same_strikes_v1,2W
4,0.03,by_maturity_bucket,640,640,0.596086,0.018024,0.023840,0.004575,0.332813,atm_skew_wls_same_strikes_v1,3M
5,0.03,by_maturity_bucket,1772,1772,0.426849,0.008958,0.010376,0.001058,0.422686,atm_skew_wls_same_strikes_v1,outside_hedging_buckets
6,0.05,by_maturity_bucket,385,385,0.720687,0.040990,0.035053,-0.005726,0.680519,atm_skew_wls_same_strikes_v1,1M
7,0.05,by_maturity_bucket,161,161,1.127608,0.177563,0.173940,0.020690,0.409938,atm_skew_wls_same_strikes_v1,1W
8,0.05,by_maturity_bucket,200,200,0.638174,0.031743,0.036060,0.002698,0.425000,atm_skew_wls_same_strikes_v1,2M
9,0.05,by_maturity_bucket,149,149,0.845307,0.103192,0.075611,-0.010828,0.664430,atm_skew_wls_same_strikes_v1,2W


,maturity_bucket,slice,n_rows,n_dates,heston_median_abs_vega_scaled_error,lifted_median_abs_vega_scaled_error,median_diff_lifted_minus_heston,share_rows_lifted_better,diagnostic_version,very_short_maturity
0,1M,explicit_1W_2W_1M_full_bucket,16930,386,0.001051,0.000993,-0.000050,0.554164,atm_skew_wls_same_strikes_v1,False
1,1W,explicit_1W_2W_1M_full_bucket,3150,161,0.004918,0.006985,0.000964,0.370476,atm_skew_wls_same_strikes_v1,True
2,2W,explicit_1W_2W_1M_full_bucket,3576,150,0.002321,0.002611,0.000061,0.461409,atm_skew_wls_same_strikes_v1,True
3,1M,explicit_1W_2W_1M_near_atm_5pct,13153,386,0.000899,0.000805,-0.000043,0.554094,atm_skew_wls_same_strikes_v1,False
4,1W,explicit_1W_2W_1M_near_atm_5pct,2899,161,0.004648,0.006617,0.001108,0.351845,atm_skew_wls_same_strikes_v1,True
5,2W,explicit_1W_2W_1M_near_atm_5pct,3063,150,0.002185,0.002420,0.000059,0.458048,atm_skew_wls_same_strikes_v1,True


,criterion,verdict,evidence,interpretation,diagnostic_version
0,very_short_maturity_calibration_advantage_1W_2...,fail,"{""1W"": 0.001108058819697286, ""2W"": 5.918636669...",Negative lifted-minus-Heston means lifted has ...,atm_skew_wls_same_strikes_v1
1,short_maturity_calibration_advantage_1W_2W_1M_...,mixed / 1M-only,"{""1W"": 0.001108058819697286, ""2W"": 5.918636669...",Combined short-maturity claims must not hide 1...,atm_skew_wls_same_strikes_v1
2,very_short_maturity_ATM_skew_slope_advantage_1...,mixed,"{""1W"": 0.020689508233700193, ""2W"": -0.01082809...",This is the direct rough-skew diagnostic: nega...,atm_skew_wls_same_strikes_v1


,diagnostic_version,objective_scenario,model,price_component_weighted_vega_rmse,very_short_near_atm_weight_multiplier,very_short_near_atm_rows,atm_skew_penalty_included,median_1W_2W_atm_skew_abs_error,median_market_1W_2W_abs_atm_skew,scaled_skew_penalty_component,skew_penalty_lambda,combined_objective_score,lifted_minus_heston_combined_score,lifted_better_under_objective
0,atm_skew_wls_same_strikes_v1,production_full_surface_weighted_loss,heston,0.001871,1.0,5962,False,NaN,NaN,0.000000,0.0,0.001871,0.001183,False
1,atm_skew_wls_same_strikes_v1,production_full_surface_weighted_loss,lifted,0.003053,1.0,5962,False,NaN,NaN,0.000000,0.0,0.003053,0.001183,False
2,atm_skew_wls_same_strikes_v1,counterfactual_1W_2W_near_ATM_overweight_loss,heston,0.002611,5.0,5962,False,NaN,NaN,0.000000,0.0,0.002611,0.001586,False
3,atm_skew_wls_same_strikes_v1,counterfactual_1W_2W_near_ATM_overweight_loss,lifted,0.004197,5.0,5962,False,NaN,NaN,0.000000,0.0,0.004197,0.001586,False
4,atm_skew_wls_same_strikes_v1,counterfactual_1W_2W_near_ATM_overweight_plus_...,heston,0.002611,5.0,5962,True,0.138854,0.959487,0.000493,1.0,0.003104,0.001537,False
5,atm_skew_wls_same_strikes_v1,counterfactual_1W_2W_near_ATM_overweight_plus_...,lifted,0.004197,5.0,5962,True,0.125063,0.959487,0.000444,1.0,0.004641,0.001537,False


,diagnostic_version,date,maturity_bucket,rows,n_strikes,mean_bid_ask_rel_spread,median_bid_ask_rel_spread,mean_fit_weight,median_fit_weight,mean_quote_quality_weight,mean_bs_vega,mean_open_interest,mean_volume,put_via_parity_share,left_wing_rows_log_mny_lt_m5pct,atm_rows_abs_log_mny_le_2pct,right_wing_rows_log_mny_gt_5pct,left_minus_right_wing_rows,has_both_left_and_right_wings,min_log_moneyness,max_log_moneyness,wing_count_map,static_arb_flagged_rows_in_final_surface,static_arb_deleted_rows,static_arb_deleted_rows_source,vertical_violations_sum_in_final_surface,slope_lower_violations_sum_in_final_surface,slope_upper_violations_sum_in_final_surface,butterfly_violations_sum_in_final_surface,macro_event_flag,macro_event_types,macro_event_calendar_source
0,atm_skew_wls_same_strikes_v1,2010-02-12,1W,13,13,0.161188,0.166667,1.331012,1.219465,6.750483,42.896481,38965.382812,8786.769531,0.538462,1,7,0,1,False,-0.072014,0.045769,"{""atm_pm2pct"": 7, ""right_near_2_to_5pct"": 3, ""...",0,NaN,not_available_from_final_surface_only; require...,0.0,0.0,0.0,0.0,False,not_available,not_found; add macro_event_calendar.csv with d...
1,atm_skew_wls_same_strikes_v1,2010-03-10,1W,18,18,0.122925,0.107367,1.656567,1.539601,9.962795,53.184273,68360.281250,9419.666992,0.555556,1,9,0,1,False,-0.077349,0.038335,"{""atm_pm2pct"": 9, ""right_near_2_to_5pct"": 4, ""...",0,NaN,not_available_from_final_surface_only; require...,0.0,0.0,0.0,0.0,False,not_available,not_found; add macro_event_calendar.csv with d...
2,atm_skew_wls_same_strikes_v1,2010-03-10,2W,11,11,0.114728,0.075188,2.440562,2.667949,11.928284,83.022980,1079.363647,714.090881,0.363636,2,5,0,2,False,-0.058341,0.047019,"{""atm_pm2pct"": 5, ""right_near_2_to_5pct"": 3, ""...",0,NaN,not_available_from_final_surface_only; require...,0.0,0.0,0.0,0.0,False,not_available,not_found; add macro_event_calendar.csv with d...
3,atm_skew_wls_same_strikes_v1,2010-03-11,1W,17,17,0.147683,0.128440,1.547689,1.378917,8.565342,47.601887,77641.468750,6262.823730,0.588235,2,9,0,2,False,-0.067201,0.042800,"{""atm_pm2pct"": 9, ""left_near_m2_to_m5pct"": 4, ...",0,NaN,not_available_from_final_surface_only; require...,0.0,0.0,0.0,0.0,False,not_available,not_found; add macro_event_calendar.csv with d...
4,atm_skew_wls_same_strikes_v1,2010-03-11,2W,9,9,0.153201,0.156522,1.652898,1.508875,6.944448,86.905609,1809.666626,775.444458,0.333333,1,5,0,1,False,-0.080872,0.022127,"{""atm_pm2pct"": 5, ""left_near_m2_to_m5pct"": 2, ...",0,NaN,not_available_from_final_surface_only; require...,0.0,0.0,0.0,0.0,False,not_available,not_found; add macro_event_calendar.csv with d...


<Figure size 910x455 with 1 Axes>

### 5.4.1 3D volatility surfaces

A volatility surface is the map

$$
\sigma_{\mathrm{imp}} = \sigma_{\mathrm{imp}}(k,T),
$$

where

$$
k = \log(K/F_T)
$$

is log-forward moneyness and $T$ is time to maturity. The 3D view is useful as a compact visual summary of the smile and term-structure shape, but it is not sufficient evidence of calibration quality. Local errors near the money, at short maturities, and in the wings can disappear behind perspective and colour shading. In this notebook the 3D surfaces are therefore paired with smile slices, ATM-skew term structures, residual heatmaps, maturity-bucket RMSE and near-ATM/wing diagnostics. Humanity tried perspective plots and somehow still needs residual tables. Tragic, but statistically sensible.

The helper below now uses a high-fidelity visualisation grid where the option panel is dense enough: the observed points are rescaled before interpolation, a linear interpolant is evaluated only inside the observed convex hull, and lighting is applied to make curvature visible without pretending the surface was smoother than the data justify. If the panel is too sparse, the function falls back to a rescaled triangulation or scatter plot. Interpolation is used only for visualisation, never for calibration, scoring or model ranking.

The 3D surfaces are shown with a slightly rotated viewing angle and visible light surface-only contour overlays so the near corner remains legible. This improves geometric readability without adding distracting under-surface projections; the contour lines use a light stroke with a faint halo so they remain visible on both bright and dark regions, but the plot remains a visual summary rather than a standalone calibration diagnostic.

The surface view also uses a subtle outline around the boundary of the observed surface footprint, which makes the edge of the fitted support easier to read without adding a heavy mesh.

In [32]:

# 3D volatility-surface diagnostics: market, Heston, lifted Heston, and residual surfaces

from mpl_toolkits.mplot3d import Axes3D

# Retain an explicit reference: importing Axes3D registers the projection with Matplotlib.
_MPL_3D_AXES_TYPE = Axes3D

try:
    from matplotlib import cm
    from matplotlib.colors import LightSource, Normalize, TwoSlopeNorm
    from scipy.interpolate import griddata
    _HIGH_FIDELITY_SURFACE_AVAILABLE = MATPLOTLIB_AVAILABLE and SCIPY_AVAILABLE
except Exception:
    cm = None
    LightSource = None
    Normalize = None
    TwoSlopeNorm = None
    griddata = None
    _HIGH_FIDELITY_SURFACE_AVAILABLE = False


def _surface_first_existing_column(frame: pd.DataFrame, candidates: Iterable[str]) -> str | None:
    for col in candidates:
        if isinstance(frame, pd.DataFrame) and col in frame.columns:
            return col
    return None


def _prepare_surface_plot_frame(surface_df: pd.DataFrame, value_col: str, *, k_col: str, maturity_col: str) -> pd.DataFrame:
    needed = [k_col, maturity_col, value_col]
    if surface_df is None or surface_df.empty or not set(needed).issubset(surface_df.columns):
        return pd.DataFrame(columns=[k_col, maturity_col, value_col])
    p = surface_df[needed].copy()
    for col in needed:
        p[col] = pd.to_numeric(p[col], errors="coerce")
    p = p.replace([np.inf, -np.inf], np.nan).dropna(subset=needed)
    p = p.loc[p[maturity_col].gt(0)]
    # Average duplicate grid points. Duplicate quotes at identical k/T otherwise make the surface ragged.
    p = p.groupby([k_col, maturity_col], as_index=False)[value_col].median()
    return p


def _axis_scaled(values: np.ndarray) -> tuple[np.ndarray, float, float]:
    lo = float(np.nanmin(values))
    hi = float(np.nanmax(values))
    span = hi - lo
    if not np.isfinite(span) or span <= 0:
        span = 1.0
    return (values - lo) / span, lo, hi


def _make_high_fidelity_surface_grid(
    plot: pd.DataFrame,
    value_col: str,
    *,
    k_col: str,
    maturity_col: str,
    grid_k: int = 150,
    grid_t: int = 110,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, float]:
    """
    Build a visualisation-only grid from irregular option rows.

    We rescale k and T before interpolation so Delaunay geometry is not dominated by maturity's
    much wider numeric range. Linear interpolation returns NaN outside the observed convex hull,
    so the plotted grid does not extrapolate into invented option quotes. Tiny mercy, really.
    """
    x_raw = plot[k_col].to_numpy(float)
    y_raw = plot[maturity_col].to_numpy(float)
    z = plot[value_col].to_numpy(float)
    x_scaled, x_lo, x_hi = _axis_scaled(x_raw)
    y_scaled, y_lo, y_hi = _axis_scaled(y_raw)
    points = np.column_stack([x_scaled, y_scaled])
    grid_k = int(np.clip(grid_k, 60, 240))
    grid_t = int(np.clip(grid_t, 45, 180))
    xi = np.linspace(0.0, 1.0, grid_k)
    yi = np.linspace(0.0, 1.0, grid_t)
    XI, YI = np.meshgrid(xi, yi)
    ZI = griddata(points, z, (XI, YI), method="linear")
    finite_ratio = float(np.isfinite(ZI).mean()) if ZI.size else 0.0
    if finite_ratio < 0.35:
        # A sparse panel may not support a smooth surface; nearest is still masked to observed hull by linear NaNs.
        nearest = griddata(points, z, (XI, YI), method="nearest")
        ZI = np.where(np.isfinite(ZI), ZI, np.nan)
        if np.isfinite(ZI).mean() < 0.10:
            ZI = nearest
            finite_ratio = float(np.isfinite(ZI).mean())
    X = x_lo + XI * (x_hi - x_lo)
    Y = y_lo + YI * (y_hi - y_lo)
    return X, Y, ZI, finite_ratio


def _surface_norm(z: np.ndarray, zlim: tuple[float, float] | None, *, residual: bool):
    finite = np.asarray(z, dtype=float)[np.isfinite(z)]
    if finite.size == 0:
        finite = np.array([0.0, 1.0])
    if zlim is None or not all(np.isfinite(zlim)):
        if residual:
            lim = float(np.nanquantile(np.abs(finite), 0.98))
            lim = lim if np.isfinite(lim) and lim > 0 else max(float(np.nanmax(np.abs(finite))), 1e-6)
            zlim = (-lim, lim)
        else:
            zlim = (float(np.nanquantile(finite, 0.02)), float(np.nanquantile(finite, 0.98)))
    if residual and TwoSlopeNorm is not None:
        return TwoSlopeNorm(vmin=float(zlim[0]), vcenter=0.0, vmax=float(zlim[1])), zlim
    return Normalize(vmin=float(zlim[0]), vmax=float(zlim[1])), zlim


def _plot_surface_boundary_outline(ax, X: np.ndarray, Y: np.ndarray, Zm: np.ma.MaskedArray) -> None:
    """Draw subtle boundary outlines around the observed masked surface footprint."""
    try:
        z = np.ma.filled(Zm, np.nan)
        finite = np.isfinite(z)
        if z.ndim != 2 or not finite.any():
            return

        def _draw_curve(xc, yc, zc):
            mask = np.isfinite(xc) & np.isfinite(yc) & np.isfinite(zc)
            if int(mask.sum()) < 2:
                return
            ax.plot(xc[mask], yc[mask], zc[mask], color=(0.26, 0.29, 0.34, 0.58), linewidth=0.85, zorder=9)

        left_idx = np.where(finite.any(axis=1), finite.argmax(axis=1), -1)
        right_idx = np.where(finite.any(axis=1), finite.shape[1] - 1 - np.fliplr(finite).argmax(axis=1), -1)
        rows = np.arange(finite.shape[0])
        valid_left = left_idx >= 0
        valid_right = right_idx >= 0
        if valid_left.any():
            _draw_curve(X[rows[valid_left], left_idx[valid_left]], Y[rows[valid_left], left_idx[valid_left]], z[rows[valid_left], left_idx[valid_left]])
        if valid_right.any():
            _draw_curve(X[rows[valid_right], right_idx[valid_right]], Y[rows[valid_right], right_idx[valid_right]], z[rows[valid_right], right_idx[valid_right]])

        top_idx = np.where(finite.any(axis=0), finite.argmax(axis=0), -1)
        bottom_idx = np.where(finite.any(axis=0), finite.shape[0] - 1 - np.flipud(finite).argmax(axis=0), -1)
        cols = np.arange(finite.shape[1])
        valid_top = top_idx >= 0
        valid_bottom = bottom_idx >= 0
        if valid_top.any():
            _draw_curve(X[top_idx[valid_top], cols[valid_top]], Y[top_idx[valid_top], cols[valid_top]], z[top_idx[valid_top], cols[valid_top]])
        if valid_bottom.any():
            _draw_curve(X[bottom_idx[valid_bottom], cols[valid_bottom]], Y[bottom_idx[valid_bottom], cols[valid_bottom]], z[bottom_idx[valid_bottom], cols[valid_bottom]])
    except Exception:
        return


def _style_3d_axis(ax) -> None:
    try:
        ax.set_box_aspect((1.35, 1.02, 0.62))
    except Exception:
        pass
    try:
        for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
            axis._axinfo["grid"].update({"linewidth": 0.35, "linestyle": "-", "color": (0.70, 0.74, 0.78, 0.45)})
            axis._axinfo["tick"].update({"inward_factor": 0.0, "outward_factor": 0.18})
        ax.xaxis.pane.set_facecolor((1, 1, 1, 0.0))
        ax.yaxis.pane.set_facecolor((1, 1, 1, 0.0))
        ax.zaxis.pane.set_facecolor((1, 1, 1, 0.0))
    except Exception:
        pass
    ax.tick_params(labelsize=7.5, pad=1.5)


def plot_iv_surface_3d(
    surface_df: pd.DataFrame,
    value_col: str,
    title: str,
    output_path: Path | str,
    *,
    k_col: str = "log_moneyness",
    maturity_col: str = "tau",
    cmap: str = SEQUENTIAL_CMAP,
    view_elev: int = 28,
    view_azim: int = -116,
    zlim: tuple[float, float] | None = None,
    max_points: int = 7000,
    grid_k: int = 150,
    grid_t: int = 110,
    notebook_section: str = "5.4.1",
    source_data: str = "atm_skew_common_iv_panel.csv",
    recommendation: str = "main/appendix",
) -> Path | None:
    if not MATPLOTLIB_AVAILABLE:
        return None
    plot = _prepare_surface_plot_frame(surface_df, value_col, k_col=k_col, maturity_col=maturity_col)
    if plot.empty or len(plot) < 8:
        return None
    if len(plot) > max_points:
        plot = plot.sample(max_points, random_state=123).sort_values([maturity_col, k_col])

    residual = "residual" in value_col or "minus_market" in value_col
    cmap = SIGNED_RESIDUAL_CMAP if residual else cmap
    fig = plt.figure(figsize=(7.4, 5.3))
    ax = fig.add_subplot(111, projection="3d")
    x = plot[k_col].to_numpy(float)
    y = plot[maturity_col].to_numpy(float)
    z = plot[value_col].to_numpy(float)
    norm, resolved_zlim = _surface_norm(z, zlim, residual=residual)
    mappable = None
    notes_method = ""

    if _HIGH_FIDELITY_SURFACE_AVAILABLE and len(plot) >= 18 and plot[k_col].nunique() >= 4 and plot[maturity_col].nunique() >= 3:
        try:
            X, Y, Z, finite_ratio = _make_high_fidelity_surface_grid(
                plot,
                value_col,
                k_col=k_col,
                maturity_col=maturity_col,
                grid_k=grid_k,
                grid_t=grid_t,
            )
            if np.isfinite(Z).sum() >= 20:
                Zm = np.ma.masked_invalid(Z)
                cmap_obj = cm.get_cmap(cmap)
                ls = LightSource(azdeg=320, altdeg=38) if LightSource is not None else None
                if ls is not None:
                    shade_base = np.where(np.isfinite(Z), Z, np.nanmedian(z))
                    facecolors = ls.shade(shade_base, cmap=cmap_obj, norm=norm, vert_exag=0.80, blend_mode="soft")
                    facecolors[~np.isfinite(Z), -1] = 0.0
                    surf = ax.plot_surface(
                        X,
                        Y,
                        Zm,
                        facecolors=facecolors,
                        rstride=1,
                        cstride=1,
                        linewidth=0.0,
                        antialiased=True,
                        shade=False,
                    )
                    mappable = cm.ScalarMappable(norm=norm, cmap=cmap_obj)
                    mappable.set_array(z)
                else:
                    surf = ax.plot_surface(X, Y, Zm, cmap=cmap, norm=norm, linewidth=0.0, antialiased=True, shade=True)
                    mappable = surf
                # Visible surface-only iso-vol contours. The dark halo prevents light lines disappearing on bright/yellow regions.
                try:
                    contour_source = np.asarray(Z, dtype=float)
                    finite_z = contour_source[np.isfinite(contour_source)]
                    if finite_z.size >= 8:
                        contour_levels = np.linspace(
                            float(np.nanpercentile(finite_z, 5)),
                            float(np.nanpercentile(finite_z, 95)),
                            9,
                        )
                    else:
                        contour_levels = 9
                    ax.contour3D(
                        X,
                        Y,
                        Zm,
                        levels=contour_levels,
                        colors=[(0.02, 0.025, 0.035, 0.30)],
                        linewidths=1.90,
                        linestyles="solid",
                    )
                    ax.contour3D(
                        X,
                        Y,
                        Zm,
                        levels=contour_levels,
                        colors=[(0.92, 0.93, 0.95, 0.98)],
                        linewidths=1.18,
                        linestyles="solid",
                    )
                except Exception:
                    pass
                _plot_surface_boundary_outline(ax, X, Y, Zm)
                notes_method = f"High-fidelity linear grid inside observed convex hull; axis-rescaled interpolation; rotated view, more visible light surface-only contours, and a subtle surface-edge outline; finite_grid_ratio={finite_ratio:.2f}. Visualisation only."
            else:
                raise ValueError("high-fidelity grid too sparse")
        except Exception as exc:
            notes_method = f"High-fidelity grid failed ({exc}); fell back to scaled triangulation."
            mappable = None

    if mappable is None:
        try:
            from matplotlib.tri import Triangulation
            xs, _, _ = _axis_scaled(x)
            ys, _, _ = _axis_scaled(y)
            triang = Triangulation(xs, ys)
            artist = ax.plot_trisurf(
                triang,
                z,
                cmap=cmap,
                norm=norm,
                linewidth=0.02,
                edgecolor="none",
                antialiased=True,
                alpha=0.96,
                shade=True,
            )
            # plot_trisurf used scaled coordinates above; relabel coordinates by using a transparent scatter in true space is worse.
            # So for fallback, prefer true-space scatter over misleading scaled axes if trisurf cannot be safely relabelled.
            ax.cla()
            artist = ax.scatter(x, y, z, c=z, cmap=cmap, norm=norm, s=9, alpha=0.86, depthshade=True)
            mappable = artist
            notes_method = notes_method or "Sparse/irregular panel shown as true-coordinate 3D scatter; no extrapolation."
        except Exception:
            artist = ax.scatter(x, y, z, c=z, cmap=cmap, norm=norm, s=9, alpha=0.86, depthshade=True)
            mappable = artist
            notes_method = notes_method or "Sparse/irregular panel shown as true-coordinate 3D scatter; no extrapolation."

    ax.view_init(elev=view_elev, azim=view_azim)
    _style_3d_axis(ax)
    ax.set_title(title)
    ax.set_xlabel(r"log-forward moneyness $k=\log(K/F_T)$", labelpad=5)
    ax.set_ylabel("time to maturity $T$ (years)", labelpad=5)
    ax.set_zlabel("IV residual" if residual else "implied volatility", labelpad=5)
    if resolved_zlim is not None and all(np.isfinite(resolved_zlim)):
        ax.set_zlim(*resolved_zlim)
    if mappable is not None:
        cbar = fig.colorbar(mappable, ax=ax, shrink=0.62, pad=0.05, fraction=0.035)
        cbar.set_label("IV residual" if residual else "implied volatility", fontsize=8.0)
        cbar.ax.tick_params(labelsize=7.2)
    return save_and_show_figure(
        fig,
        output_path,
        dpi=320,
        figure_id=Path(output_path).stem,
        description=title,
        notebook_section=notebook_section,
        source_data=source_data,
        recommendation=recommendation,
        notes=notes_method,
    )


def select_typical_surface_date(iv_panel: pd.DataFrame) -> pd.Timestamp | None:
    if iv_panel is None or iv_panel.empty or "date" not in iv_panel.columns:
        return None
    p = iv_panel.copy()
    p["date"] = pd.to_datetime(p["date"], errors="coerce").dt.normalize()
    if {"market_iv", "heston_iv", "lifted_iv"}.issubset(p.columns):
        p["_fit_score"] = (
            (pd.to_numeric(p["heston_iv"], errors="coerce") - pd.to_numeric(p["market_iv"], errors="coerce")).abs()
            + (pd.to_numeric(p["lifted_iv"], errors="coerce") - pd.to_numeric(p["market_iv"], errors="coerce")).abs()
        )
        daily = p.groupby("date", as_index=False).agg(rows=("date", "size"), median_abs_fit_score=("_fit_score", "median"))
        daily = daily.loc[daily["rows"].ge(max(12, int(daily["rows"].quantile(0.25))))].copy()
        if not daily.empty:
            target = daily["median_abs_fit_score"].median()
            return pd.Timestamp(daily.iloc[(daily["median_abs_fit_score"] - target).abs().argsort().iloc[0]]["date"])
    counts = p.groupby("date").size().sort_values(ascending=False)
    return pd.Timestamp(counts.index[0]) if not counts.empty else None


def make_3d_surface_panel(iv_panel: pd.DataFrame, cfg: ExperimentConfig = CFG) -> pd.DataFrame:
    if iv_panel is None or iv_panel.empty:
        out = pd.DataFrame([{"figure": "3d_vol_surface", "status": "missing_iv_panel", "notes": "atm_skew_iv_panel was unavailable or empty."}])
        out.to_csv(table_path("iv_surface_3d_audit.csv", cfg), index=False)
        display(out)
        return out
    p = iv_panel.copy()
    p["date"] = pd.to_datetime(p["date"], errors="coerce").dt.normalize() if "date" in p.columns else pd.NaT
    k_col = _surface_first_existing_column(p, ["log_forward_moneyness_for_skew", "log_forward_moneyness", "log_moneyness"])
    t_col = _surface_first_existing_column(p, ["T", "tau", "time_to_maturity", "maturity_years"])
    required = {"market_iv", "heston_iv", "lifted_iv"}
    audit_rows = []
    if not k_col or not t_col or not required.issubset(p.columns):
        out = pd.DataFrame([{
            "figure": "3d_vol_surface",
            "status": "missing_required_columns",
            "notes": f"k_col={k_col}; t_col={t_col}; missing={sorted(required - set(p.columns))}",
        }])
        out.to_csv(table_path("iv_surface_3d_audit.csv", cfg), index=False)
        display(out)
        return out

    typical_date = select_typical_surface_date(p)
    if typical_date is not None and pd.notna(typical_date):
        p = p.loc[p["date"].eq(typical_date)].copy()
    for col in ["market_iv", "heston_iv", "lifted_iv"]:
        p[col] = pd.to_numeric(p[col], errors="coerce")
    p["heston_minus_market_residual"] = p["heston_iv"] - p["market_iv"]
    p["lifted_minus_market_residual"] = p["lifted_iv"] - p["market_iv"]

    zvals = pd.concat([p["market_iv"], p["heston_iv"], p["lifted_iv"]], ignore_index=True).replace([np.inf, -np.inf], np.nan).dropna()
    zlim = (float(zvals.quantile(0.01)), float(zvals.quantile(0.99))) if not zvals.empty else None
    res_abs = pd.concat([p["heston_minus_market_residual"].abs(), p["lifted_minus_market_residual"].abs()], ignore_index=True).replace([np.inf, -np.inf], np.nan).dropna()
    rlim = float(res_abs.quantile(0.99)) if not res_abs.empty else None
    residual_zlim = (-rlim, rlim) if rlim and np.isfinite(rlim) and rlim > 0 else None
    date_label = pd.Timestamp(typical_date).date() if typical_date is not None and pd.notna(typical_date) else "all_dates"

    specs = [
        ("market_iv", f"Market implied-volatility surface, {date_label}", "fig_iv_surface_3d_market_typical_date.png", SEQUENTIAL_CMAP, zlim, "main"),
        ("heston_iv", f"Heston implied-volatility surface, {date_label}", "fig_iv_surface_3d_heston_typical_date.png", SEQUENTIAL_CMAP, zlim, "main/appendix"),
        ("lifted_iv", f"Lifted-Heston implied-volatility surface, {date_label}", "fig_iv_surface_3d_lifted_typical_date.png", SEQUENTIAL_CMAP, zlim, "main/appendix"),
        ("heston_minus_market_residual", f"Heston minus market IV residual surface, {date_label}", "fig_iv_surface_3d_residual_heston_minus_market.png", SIGNED_RESIDUAL_CMAP, residual_zlim, "appendix"),
        ("lifted_minus_market_residual", f"Lifted-Heston minus market IV residual surface, {date_label}", "fig_iv_surface_3d_residual_lifted_minus_market.png", SIGNED_RESIDUAL_CMAP, residual_zlim, "appendix"),
    ]
    for value_col, title, filename, cmap, this_zlim, recommendation in specs:
        path = plot_iv_surface_3d(
            p,
            value_col,
            title,
            figure_path(filename, cfg),
            k_col=k_col,
            maturity_col=t_col,
            cmap=cmap,
            zlim=this_zlim,
            source_data="atm_skew_common_iv_panel.csv",
            recommendation=recommendation,
        )
        audit_rows.append({
            "figure": Path(filename).stem,
            "status": "created" if path is not None else "not_created",
            "date": str(date_label),
            "output_path": str(path) if path is not None else str(figure_path(filename, cfg)),
            "shown_in_notebook_output": "yes" if path is not None else "no",
            "notes": "High-fidelity visual grid when possible; interpolation is axis-rescaled, convex-hull limited, visualisation-only; read with smile, skew and residual diagnostics.",
        })
    out = pd.DataFrame(audit_rows)
    out.to_csv(table_path("iv_surface_3d_audit.csv", cfg), index=False)
    display(out)
    return out


iv_surface_3d_audit = make_3d_surface_panel(atm_skew_iv_panel, CFG)


C:\Users\lhkke\AppData\Local\Temp\ipykernel_24096\3649089196.py:213: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap_obj = cm.get_cmap(cmap)


<Figure size 962x689 with 2 Axes>

C:\Users\lhkke\AppData\Local\Temp\ipykernel_24096\3649089196.py:213: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap_obj = cm.get_cmap(cmap)


<Figure size 962x689 with 2 Axes>

C:\Users\lhkke\AppData\Local\Temp\ipykernel_24096\3649089196.py:213: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap_obj = cm.get_cmap(cmap)


<Figure size 962x689 with 2 Axes>

C:\Users\lhkke\AppData\Local\Temp\ipykernel_24096\3649089196.py:213: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap_obj = cm.get_cmap(cmap)


<Figure size 962x689 with 2 Axes>

C:\Users\lhkke\AppData\Local\Temp\ipykernel_24096\3649089196.py:213: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap_obj = cm.get_cmap(cmap)


<Figure size 962x689 with 2 Axes>

,figure,status,date,output_path,shown_in_notebook_output,notes
0,fig_iv_surface_3d_market_typical_date,created,2014-02-04,"C:\Users\lhkke\OneDrive - Queen Mary, Universi...",yes,High-fidelity visual grid when possible; inter...
1,fig_iv_surface_3d_heston_typical_date,created,2014-02-04,"C:\Users\lhkke\OneDrive - Queen Mary, Universi...",yes,High-fidelity visual grid when possible; inter...
2,fig_iv_surface_3d_lifted_typical_date,created,2014-02-04,"C:\Users\lhkke\OneDrive - Queen Mary, Universi...",yes,High-fidelity visual grid when possible; inter...
3,fig_iv_surface_3d_residual_heston_minus_market,created,2014-02-04,"C:\Users\lhkke\OneDrive - Queen Mary, Universi...",yes,High-fidelity visual grid when possible; inter...
4,fig_iv_surface_3d_residual_lifted_minus_market,created,2014-02-04,"C:\Users\lhkke\OneDrive - Queen Mary, Universi...",yes,High-fidelity visual grid when possible; inter...


### 5.4.1 Dissertation-Facing Calibration Plots

The plots in this subsection are designed for the results chapter rather than for debugging. They answer visually interpretable questions: how does market ATM skew vary by maturity, how do Heston and lifted Heston track that term structure, and what do representative smiles look like on typical, best-fit and worst-fit dates?

The ATM-skew term-structure plot is main-text evidence because it directly connects the lifted-Heston motivation to observed option-market objects. If lifted Heston improves short-maturity skew but fails elsewhere, the dissertation can state that selectively. If it does not improve skew, the model-memory story must rely on hedging or mechanism evidence rather than calibration rhetoric.

Representative smile plots are included because residual tables compress too much information. A model may have similar RMSE to another model but fail in a different part of the smile. For example, one model may fit near-ATM level but miss wings; another may fit skew but miss curvature. Those distinctions matter when interpreting why a calibrated state may or may not help hedging.

All plots should use shared representative dates and consistent axes for Heston and lifted Heston. A model should not receive an easier visual subset. This keeps the visual evidence aligned with the paired-comparison design.


In [33]:

# Dissertation-facing calibration diagnostics: term structure, smiles, heatmaps and rolling fit

def _bucket_order_map(cfg: ExperimentConfig) -> dict[str, int]:
    return {str(b): i for i, b in enumerate(getattr(cfg.hedging, "maturity_buckets", []))}


def build_atm_skew_term_structure(skew: pd.DataFrame, fd_skew: pd.DataFrame | None, cfg: ExperimentConfig) -> pd.DataFrame:
    if skew is None or skew.empty:
        return pd.DataFrame()
    s = skew.copy()
    window_col = "moneyness_window_abs_log_forward"
    if window_col in s.columns:
        w = float(cfg.diagnostics.near_atm_skew_window)
        s = s.loc[np.isclose(pd.to_numeric(s[window_col], errors="coerce"), w)].copy()
        if s.empty:
            s = skew.copy()
    grouped = s.groupby("maturity_bucket", as_index=False).agg(
        market_atm_skew=("market_atm_skew_slope", "median"),
        heston_atm_skew=("heston_atm_skew_slope", "median"),
        lifted_heston_atm_skew=("lifted_atm_skew_slope", "median"),
        heston_abs_skew_error=("heston_abs_skew_error", "median"),
        lifted_abs_skew_error=("lifted_abs_skew_error", "median"),
        n_date_maturity_pairs=("date", "count"),
    )
    grouped["skew_estimator"] = "weighted_local_linear_market_grid"
    if fd_skew is not None and not fd_skew.empty and {"model", "maturity_bucket", "finite_difference_atm_skew"}.issubset(fd_skew.columns):
        f = fd_skew.loc[fd_skew.get("pricing_status", "available").astype(str).eq("available")].copy()
        fd = f.groupby(["model", "maturity_bucket"], as_index=False)["finite_difference_atm_skew"].median()
        for model, col in [("heston", "heston_atm_skew_fd"), ("lifted_heston", "lifted_heston_atm_skew_fd")]:
            x = fd.loc[fd["model"].astype(str).eq(model), ["maturity_bucket", "finite_difference_atm_skew"]].rename(columns={"finite_difference_atm_skew": col})
            grouped = grouped.merge(x, on="maturity_bucket", how="left")
        grouped["skew_estimator"] = np.where(grouped.filter(like="_fd").notna().any(axis=1), "market_grid_plus_available_finite_difference", grouped["skew_estimator"])
    grouped["abs_log_market_skew"] = np.log(np.abs(grouped["market_atm_skew"].astype(float))).replace([np.inf, -np.inf], np.nan)
    order = _bucket_order_map(cfg)
    grouped["bucket_order"] = grouped["maturity_bucket"].astype(str).map(order).fillna(999).astype(int)
    out = grouped.sort_values(["bucket_order", "maturity_bucket"]).drop(columns="bucket_order")
    out.to_csv(table_path("atm_skew_term_structure_comparison.csv", cfg), index=False)
    return out


def plot_atm_skew_term_structure(term: pd.DataFrame, cfg: ExperimentConfig) -> Path | None:
    if not MATPLOTLIB_AVAILABLE or term is None or term.empty:
        return None
    fig, ax = plt.subplots(figsize=(7, 4))
    x = np.arange(len(term))
    for col, label, key in [("market_atm_skew", "Market", "market"), ("heston_atm_skew", "Heston", "heston"), ("lifted_heston_atm_skew", "Lifted Heston", "lifted_heston")]:
        if col in term.columns:
            ax.plot(
                x,
                pd.to_numeric(term[col], errors="coerce"),
                linestyle="-",
                color=model_color(key),
                label=label,
                alpha=MODEL_ALPHA,
                linewidth=1.0,
            )
    ax.set_xticks(x); ax.set_xticklabels(term["maturity_bucket"].astype(str))
    ax.set(title="ATM-skew term structure: market vs Heston vs lifted Heston", xlabel="Maturity bucket", ylabel=r"ATM skew $\partial_k\sigma_{imp}(0,T)$")
    standardise_axis(ax, legend=True, ncol=2)
    path = figure_path("atm_skew_term_structure_market_heston_lifted.png", cfg)
    save_and_show_figure(plt.gcf(), path, dpi=160)
    return path


def select_smile_example_dates(fit_pair: pd.DataFrame, n: int = 3) -> pd.DataFrame:
    if fit_pair is None or fit_pair.empty or "date" not in fit_pair.columns:
        return pd.DataFrame()
    f = fit_pair.copy()
    f["date"] = pd.to_datetime(f["date"], errors="coerce").dt.normalize()
    err = _choose_col(f, ["lifted_abs_vega_scaled_error", "lifted_abs_iv_error", "lifted_abs_error"])
    if not err:
        return pd.DataFrame()
    daily = (
        f.groupby("date", as_index=False)
        .agg(lifted_error=(err, "median"), rows=(err, "size"))
        .dropna(subset=["lifted_error"])
        .sort_values("lifted_error")
        .reset_index(drop=True)
    )
    if daily.empty:
        return daily

    def _pick_row(pos: int, role: str) -> dict:
        row = daily.iloc[pos].to_dict()
        row["selection_role"] = role
        return row

    picks = [
        _pick_row(0, "best_lifted_fit"),
        _pick_row(len(daily) // 2, "typical_lifted_fit"),
        _pick_row(len(daily) - 1, "worst_lifted_fit"),
    ]
    return pd.DataFrame(picks).drop_duplicates("date").reset_index(drop=True)


# -----------------------------------------------------------------------------
# Compact representative smile plots + calibration residual / rolling RMSE helpers
# -----------------------------------------------------------------------------

def plot_representative_smiles(
    iv_panel: pd.DataFrame,
    selected_dates: pd.DataFrame,
    cfg: ExperimentConfig,
) -> Path | None:
    """
    Compact model-vs-market smile plots for selected representative dates.

    Fixes:
    - no giant blank figure space;
    - only plots populated panels;
    - includes legends;
    - uses the shared dissertation colour scheme.
    """
    if not MATPLOTLIB_AVAILABLE or iv_panel is None or iv_panel.empty or selected_dates is None or selected_dates.empty:
        return None

    required = {"date", "maturity_bucket", "market_iv", "heston_iv", "lifted_iv"}
    if not required.issubset(iv_panel.columns):
        print(f"Representative smile plot skipped; missing columns: {sorted(required.difference(iv_panel.columns))}")
        return None

    k_col = _choose_col(
        iv_panel,
        ["log_forward_moneyness_for_skew", "log_forward_moneyness", "log_moneyness"],
    )
    if not k_col:
        print("Representative smile plot skipped; no log-moneyness column found.")
        return None

    model_alpha = globals().get("MODEL_ALPHA", 0.80)
    scatter_alpha = globals().get("SCATTER_ALPHA", 0.72)

    p = iv_panel.copy()
    p["date"] = pd.to_datetime(p["date"], errors="coerce").dt.normalize()

    selected = selected_dates[["date", "selection_role"]].copy()
    selected["date"] = pd.to_datetime(selected["date"], errors="coerce").dt.normalize()

    p = p.merge(selected, on="date", how="inner")
    if p.empty:
        print("Representative smile plot skipped; selected dates not found in IV panel.")
        return None

    candidate_buckets = [
        b for b in cfg.hedging.maturity_buckets
        if b in set(p["maturity_bucket"].astype(str))
    ]
    if not candidate_buckets:
        candidate_buckets = sorted(p["maturity_bucket"].dropna().astype(str).unique())

    panels = []
    for _, drow in selected.iterrows():
        d = pd.Timestamp(drow["date"]).normalize()
        role = str(drow["selection_role"])

        for bucket in candidate_buckets:
            g = (
                p.loc[
                    p["date"].eq(d)
                    & p["maturity_bucket"].astype(str).eq(str(bucket))
                ]
                .sort_values(k_col)
                .copy()
            )
            if not g.empty:
                panels.append((role, d, str(bucket), g))

    if not panels:
        print("Representative smile plot skipped; no non-empty date/bucket panels.")
        return None

    n_panels = len(panels)
    ncols = min(3, n_panels)
    nrows = int(np.ceil(n_panels / ncols))

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(4.2 * ncols, 3.0 * nrows),
        squeeze=False,
        constrained_layout=True,
    )
    axes_flat = axes.ravel()

    for ax, (role, date_value, bucket, g) in zip(axes_flat, panels):
        x = pd.to_numeric(g[k_col], errors="coerce")

        ax.scatter(
            x,
            pd.to_numeric(g["market_iv"], errors="coerce"),
            label="Market",
            s=16,
            color=model_color("market"),
            alpha=scatter_alpha,
            edgecolors="white",
            linewidths=0.25,
        )

        ax.plot(
            x,
            pd.to_numeric(g["heston_iv"], errors="coerce"),
            label="Heston",
            color=model_color("heston"),
            linestyle="-",
            linewidth=1.0,
            alpha=model_alpha,
        )

        ax.plot(
            x,
            pd.to_numeric(g["lifted_iv"], errors="coerce"),
            label="Lifted Heston",
            color=model_color("lifted_heston"),
            linestyle="-",
            linewidth=1.0,
            alpha=model_alpha,
        )

        ax.set_title(f"{role} | {date_value.date()} | {bucket}", fontsize=9)
        ax.set_xlabel(r"$k=\log(K/F)$")
        ax.set_ylabel("IV")
        ax.grid(True, alpha=0.3)

        # Do not pass fontsize here; compact_legend does not accept it in this notebook.
        compact_legend(ax, ncol=3, loc="upper right")

    for ax in axes_flat[n_panels:]:
        ax.axis("off")

    path = figure_path("representative_model_vs_market_smile_examples.png", cfg)
    save_and_show_figure(fig, path, dpi=160)
    return path


def calibration_residual_heatmap_table(
    fit_pair: pd.DataFrame,
    cfg: ExperimentConfig,
) -> pd.DataFrame:
    """
    Build model-specific residual heatmap data.

    Returns:
    - model
    - maturity_bucket
    - moneyness_bucket
    - median_abs_vega_scaled_error
    """
    if fit_pair is None or fit_pair.empty:
        return pd.DataFrame()

    f = fit_pair.copy()

    k_col = _choose_col(
        f,
        [
            "log_forward_moneyness",
            "log_forward_moneyness_for_skew",
            "log_moneyness",
        ],
    )
    if not k_col:
        return pd.DataFrame()

    if "maturity_bucket" not in f.columns:
        if "dte" in f.columns:
            dte = pd.to_numeric(f["dte"], errors="coerce")
        elif "T" in f.columns:
            dte = pd.to_numeric(f["T"], errors="coerce") * 365.25
        else:
            return pd.DataFrame()

        bucket = pd.Series("unbucketed", index=f.index, dtype="object")
        for label, (_, lo, hi) in cfg.hedging.maturity_buckets.items():
            bucket = bucket.where(~dte.between(lo, hi, inclusive="both"), label)
        f["maturity_bucket"] = bucket

    f["moneyness_bucket"] = pd.cut(
        pd.to_numeric(f[k_col], errors="coerce"),
        bins=[-0.35, -0.20, -0.10, -0.05, 0.05, 0.10, 0.20, 0.35],
        labels=[
            "[-0.35,-0.20]",
            "[-0.20,-0.10]",
            "[-0.10,-0.05]",
            "[-0.05,0.05]",
            "[0.05,0.10]",
            "[0.10,0.20]",
            "[0.20,0.35]",
        ],
        include_lowest=True,
    )

    if "heston_abs_vega_scaled_error" not in f.columns and "heston_vega_scaled_error" in f.columns:
        f["heston_abs_vega_scaled_error"] = pd.to_numeric(
            f["heston_vega_scaled_error"],
            errors="coerce",
        ).abs()

    if "lifted_abs_vega_scaled_error" not in f.columns and "lifted_vega_scaled_error" in f.columns:
        f["lifted_abs_vega_scaled_error"] = pd.to_numeric(
            f["lifted_vega_scaled_error"],
            errors="coerce",
        ).abs()

    rows = []
    for model, err_col in [
        ("heston", "heston_abs_vega_scaled_error"),
        ("lifted_heston", "lifted_abs_vega_scaled_error"),
    ]:
        if err_col not in f.columns:
            continue

        g = (
            f.dropna(subset=["maturity_bucket", "moneyness_bucket"])
            .groupby(["maturity_bucket", "moneyness_bucket"], observed=False)[err_col]
            .median()
            .reset_index()
            .rename(columns={err_col: "median_abs_vega_scaled_error"})
        )
        g["model"] = model
        rows.append(g)

    out = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

    if not out.empty:
        out = out[
            [
                "model",
                "maturity_bucket",
                "moneyness_bucket",
                "median_abs_vega_scaled_error",
            ]
        ]
        out.to_csv(
            table_path("calibration_residual_heatmap_summary.csv", cfg),
            index=False,
        )

    return out


def plot_residual_heatmaps(
    heat: pd.DataFrame,
    cfg: ExperimentConfig,
) -> list[Path]:
    """
    Plot annotated residual heatmaps for Heston and lifted Heston.
    """
    paths = []

    if not MATPLOTLIB_AVAILABLE or heat is None or heat.empty:
        return paths

    cmap = globals().get("ERROR_CMAP", "coolwarm")

    for model, g in heat.groupby("model"):
        pivot = g.pivot(
            index="maturity_bucket",
            columns="moneyness_bucket",
            values="median_abs_vega_scaled_error",
        )

        fig, ax = plt.subplots(figsize=(8, 4))

        plot_annotated_heatmap(
            pivot,
            ax=ax,
            fig=fig,
            title=f"{MODEL_DISPLAY.get(model, model)} calibration residual heatmap",
            xlabel="log-moneyness bucket",
            ylabel="maturity bucket",
            cbar_label="median absolute vega-scaled error",
            cmap=cmap,
            fmt=None,
            rotation=45,
        )

        path = figure_path(
            f"{model}_residual_heatmap_by_maturity_moneyness.png",
            cfg,
        )

        save_and_show_figure(
            fig,
            path,
            dpi=260,
            description=f"{MODEL_DISPLAY.get(model, model)} calibration residual heatmap",
            notes="All heatmap cells are annotated with median absolute vega-scaled error.",
        )
        paths.append(path)

    return paths


def rolling_calibration_rmse_table(
    fit_pair: pd.DataFrame,
    cfg: ExperimentConfig,
) -> pd.DataFrame:
    """
    Build rolling calibration RMSE table for Heston and lifted Heston.
    """
    if fit_pair is None or fit_pair.empty or "date" not in fit_pair.columns:
        return pd.DataFrame()

    f = fit_pair.copy()
    f["date"] = pd.to_datetime(f["date"], errors="coerce").dt.normalize()

    if "heston_vega_scaled_error" not in f.columns and "heston_abs_vega_scaled_error" in f.columns:
        f["heston_vega_scaled_error"] = pd.to_numeric(
            f["heston_abs_vega_scaled_error"],
            errors="coerce",
        )

    if "lifted_vega_scaled_error" not in f.columns and "lifted_abs_vega_scaled_error" in f.columns:
        f["lifted_vega_scaled_error"] = pd.to_numeric(
            f["lifted_abs_vega_scaled_error"],
            errors="coerce",
        )

    rows = []

    for model, col in [
        ("heston", "heston_vega_scaled_error"),
        ("lifted_heston", "lifted_vega_scaled_error"),
    ]:
        if col not in f.columns:
            continue

        daily = (
            f.groupby("date", as_index=False)
            .agg(
                vega_weighted_iv_rmse=(
                    col,
                    lambda x: float(
                        np.sqrt(
                            np.nanmean(
                                pd.to_numeric(x, errors="coerce") ** 2
                            )
                        )
                    ),
                ),
                rows=(col, "count"),
            )
        )

        daily["model"] = model
        daily["rolling_21d_rmse"] = (
            daily["vega_weighted_iv_rmse"]
            .rolling(21, min_periods=5)
            .median()
        )

        rows.append(daily)

    out = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

    if not out.empty:
        out.to_csv(
            table_path("rolling_chronological_calibration_rmse.csv", cfg),
            index=False,
        )

    return out


def plot_rolling_calibration_rmse(
    rolling: pd.DataFrame,
    cfg: ExperimentConfig,
) -> Path | None:
    """
    Plot rolling Heston vs lifted-Heston calibration RMSE.
    """
    if not MATPLOTLIB_AVAILABLE or rolling is None or rolling.empty:
        return None

    model_alpha = globals().get("MODEL_ALPHA", 0.80)

    fig, ax = plt.subplots(figsize=(8, 4))

    for model, g in rolling.groupby("model"):
        ax.plot(
            pd.to_datetime(g["date"], errors="coerce"),
            pd.to_numeric(g["rolling_21d_rmse"], errors="coerce"),
            label=MODEL_DISPLAY.get(model, model),
            color=model_color(model),
            linestyle="-",
            linewidth=1.0,
            alpha=model_alpha,
        )

    ax.set(
        title="Rolling calibration RMSE through time",
        xlabel="Date",
        ylabel="21-date median vega-scaled RMSE",
    )
    standardise_axis(ax, legend=True, ncol=2)

    path = figure_path("rolling_chronological_calibration_rmse.png", cfg)
    save_and_show_figure(fig, path, dpi=160)

    return path

_fd_source = globals().get("finite_difference_model_atm_skew_table", pd.DataFrame())
atm_skew_term_structure_comparison = build_atm_skew_term_structure(atm_skew_diagnostics, _fd_source, CFG)
atm_skew_term_structure_plot_path = plot_atm_skew_term_structure(atm_skew_term_structure_comparison, CFG)
representative_smile_dates = select_smile_example_dates(calibration_fit_pair_diagnostics, n=3)
representative_smile_dates.to_csv(table_path("representative_smile_example_dates.csv", CFG), index=False)
representative_smile_examples_path = plot_representative_smiles(atm_skew_iv_panel, representative_smile_dates, CFG)
calibration_residual_heatmap_summary = calibration_residual_heatmap_table(calibration_fit_pair_diagnostics, CFG)
calibration_residual_heatmap_paths = plot_residual_heatmaps(calibration_residual_heatmap_summary, CFG)
rolling_chronological_calibration_rmse = rolling_calibration_rmse_table(calibration_fit_pair_diagnostics, CFG)
rolling_chronological_calibration_rmse_path = plot_rolling_calibration_rmse(rolling_chronological_calibration_rmse, CFG)

calibration_model_selection_scorecard = pd.DataFrame([
    {"item": "ATM-skew term structure", "status": "PASS" if not atm_skew_term_structure_comparison.empty else "WARN", "artefact": str(table_path("atm_skew_term_structure_comparison.csv", CFG)), "interpretation": "Main calibration diagnostic for rough/lifted short-end skew."},
    {"item": "Representative smile examples", "status": "PASS" if not representative_smile_dates.empty and representative_smile_examples_path else "WARN", "artefact": str(figure_path("representative_model_vs_market_smile_examples.png", CFG)), "interpretation": "Shows calibration errors visually rather than as scalar RMSE only."},
    {"item": "Residual heatmaps", "status": "PASS" if not calibration_residual_heatmap_summary.empty else "WARN", "artefact": str(table_path("calibration_residual_heatmap_by_maturity_moneyness.csv", CFG)), "interpretation": "Identifies failure regions by maturity and moneyness."},
    {"item": "Rolling calibration RMSE", "status": "PASS" if not rolling_chronological_calibration_rmse.empty else "WARN", "artefact": str(table_path("rolling_chronological_calibration_rmse.csv", CFG)), "interpretation": "Checks whether fit is stable or regime-dependent."},
])
calibration_model_selection_scorecard.to_csv(table_path("calibration_model_selection_scorecard.csv", CFG), index=False)

display(calibration_model_selection_scorecard)
display(atm_skew_term_structure_comparison)


<Figure size 910x520 with 1 Axes>

C:\Users\lhkke\AppData\Local\Temp\ipykernel_24096\3611804343.py:276: UserWarning: The figure layout has changed to tight
  fig.tight_layout()


<Figure size 546x390 with 1 Axes>

<Figure size 1040x520 with 1 Axes>

,item,status,artefact,interpretation
0,ATM-skew term structure,PASS,"C:\Users\lhkke\OneDrive - Queen Mary, Universi...",Main calibration diagnostic for rough/lifted s...
1,Representative smile examples,PASS,"C:\Users\lhkke\OneDrive - Queen Mary, Universi...",Shows calibration errors visually rather than ...
2,Residual heatmaps,WARN,"C:\Users\lhkke\OneDrive - Queen Mary, Universi...",Identifies failure regions by maturity and mon...
3,Rolling calibration RMSE,PASS,"C:\Users\lhkke\OneDrive - Queen Mary, Universi...",Checks whether fit is stable or regime-dependent.


,maturity_bucket,market_atm_skew,heston_atm_skew,lifted_heston_atm_skew,heston_abs_skew_error,lifted_abs_skew_error,n_date_maturity_pairs,skew_estimator,abs_log_market_skew
1,1W,-1.127608,-1.243386,-1.150831,0.177563,0.173940,161,weighted_local_linear_market_grid,0.120098
3,2W,-0.845307,-0.963933,-0.920478,0.103192,0.075611,149,weighted_local_linear_market_grid,-0.168055
0,1M,-0.720687,-0.754546,-0.739503,0.040990,0.035053,385,weighted_local_linear_market_grid,-0.327551
2,2M,-0.638174,-0.671694,-0.663737,0.031743,0.036060,200,weighted_local_linear_market_grid,-0.449144
4,3M,-0.566791,-0.583932,-0.585663,0.016212,0.019715,873,weighted_local_linear_market_grid,-0.567765
5,outside_hedging_buckets,-0.420605,-0.413948,-0.411688,0.005393,0.006449,1921,weighted_local_linear_market_grid,-0.866061


### 5.4.2 Slice RMSE and Log-Skew Error Tables

This subsection creates table-level summaries of calibration quality by economically meaningful slices. The relevant slices include near-ATM, short maturity, near-ATM short maturity and wings where available. Slice RMSE matters because full-surface averages can hide selective model strengths. Lifted Heston may be theoretically most relevant at short maturities and near the money; a global RMSE alone cannot test that claim.

The log-skew error table measures the shape of the ATM-skew term structure in scale-adjusted form:

$$
e_{\log\text{-skew}}(T)=
\log\left|\partial_k\sigma_{\mathrm{model}}(0,T)\right|-
\log\left|\partial_k\sigma_{\mathrm{mkt}}(0,T)\right|.
$$

This is especially useful when short-maturity skew has much larger magnitude than medium-maturity skew. A raw error table may be dominated by the shortest bucket; a log-error table asks whether the relative term-structure shape is plausible.

The interpretation should remain strict. Slice-level calibration improvement supports model-selection evidence for state construction. It does not imply a policy will hedge better. The later evaluation sections must confirm whether the improved fitted object translates into lower terminal net risk.


In [34]:

# Exportable calibration slice RMSE and log-skew-error diagnostics

def calibration_slice_rmse_table_from_packs(packs: dict[str, dict[str, pd.DataFrame]], cfg: ExperimentConfig) -> pd.DataFrame:
    rows = []
    for model, pack in (packs or {}).items():
        residuals = pack.get("residual_panel", pd.DataFrame()) if isinstance(pack, dict) else pd.DataFrame()
        if residuals is None or residuals.empty or "iv_residual" not in residuals.columns:
            rows.append({"model": model, "slice": "unavailable", "status": "missing_residual_panel"})
            continue
        r = residuals.copy()
        k_col = _choose_col(r, ["log_forward_moneyness", "abs_log_forward_moneyness", "log_moneyness"])
        t_col = _choose_col(r, ["T", "time_to_maturity", "maturity_years"])
        res = pd.to_numeric(r["iv_residual"], errors="coerce")
        k = pd.to_numeric(r[k_col], errors="coerce").abs() if k_col else pd.Series(np.nan, index=r.index)
        T = pd.to_numeric(r[t_col], errors="coerce") if t_col else pd.Series(np.nan, index=r.index)
        weight_col = _choose_col(r, ["vega_weight", "calibration_weight", "weight"])
        weights = pd.to_numeric(r[weight_col], errors="coerce") if weight_col else pd.Series(1.0, index=r.index)
        masks = {
            "all": res.notna(),
            "near_atm": res.notna() & k.le(cfg.diagnostics.near_atm_skew_window),
            "short_maturity": res.notna() & T.le(60.0 / cfg.market.calendar_days_per_year),
            "near_atm_short_maturity": res.notna() & k.le(cfg.diagnostics.near_atm_skew_window) & T.le(60.0 / cfg.market.calendar_days_per_year),
            "wings": res.notna() & k.ge(0.15),
        }
        for name, mask in masks.items():
            vals = res.loc[mask].dropna()
            if vals.empty:
                rows.append({"model": model, "slice": name, "status": "empty"})
                continue
            w = weights.loc[vals.index].replace([np.inf, -np.inf], np.nan).fillna(1.0).clip(lower=0.0)
            rows.append({
                "model": model,
                "slice": name,
                "iv_rmse": float(np.sqrt(np.mean(vals ** 2))),
                "vega_weighted_iv_rmse": float(np.sqrt(np.average(vals ** 2, weights=w))) if float(w.sum()) > 0 else np.nan,
                "median_abs_iv_error": float(vals.abs().median()),
                "n_rows": int(len(vals)),
                "status": "available",
            })
    out = pd.DataFrame(rows)
    out.to_csv(table_path("calibration_slice_rmse_diagnostics.csv", cfg), index=False)
    return out


def log_atm_skew_error_table(term_structure: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    if term_structure is None or term_structure.empty:
        out = pd.DataFrame([{"status": "missing", "notes": "ATM-skew term-structure table unavailable"}])
        out.to_csv(table_path("log_atm_skew_error_by_maturity.csv", cfg), index=False)
        return out
    t = term_structure.copy()
    market_col = _choose_col(t, ["market_atm_skew", "market_near_atm_skew"])
    rows = []
    for model, col in {"heston": _choose_col(t, ["heston_fd_atm_skew", "heston_atm_skew"]), "lifted_heston": _choose_col(t, ["lifted_heston_fd_atm_skew", "lifted_heston_atm_skew"])}.items():
        if not market_col or not col:
            rows.append({"model": model, "status": "missing_skew_columns"})
            continue
        mkt = pd.to_numeric(t[market_col], errors="coerce")
        mdl = pd.to_numeric(t[col], errors="coerce")
        valid = mkt.abs().gt(0) & mdl.abs().gt(0) & np.isfinite(mkt) & np.isfinite(mdl)
        part = t.loc[valid, [c for c in ["maturity_bucket", "T", "n_date_maturity_pairs"] if c in t.columns]].copy()
        part["model"] = model
        part["market_abs_skew"] = mkt.loc[valid].abs().to_numpy()
        part["model_abs_skew"] = mdl.loc[valid].abs().to_numpy()
        part["log_atm_skew_error"] = np.log(part["model_abs_skew"]) - np.log(part["market_abs_skew"])
        part["abs_log_atm_skew_error"] = part["log_atm_skew_error"].abs()
        part["status"] = "available"
        rows.append(part)
    out = pd.concat([x if isinstance(x, pd.DataFrame) else pd.DataFrame([x]) for x in rows], ignore_index=True)
    out.to_csv(table_path("log_atm_skew_error_by_maturity.csv", cfg), index=False)
    return out

calibration_slice_rmse_diagnostics = calibration_slice_rmse_table_from_packs(globals().get("standard_diagnostic_packs", {}), CFG)
log_atm_skew_error_by_maturity = log_atm_skew_error_table(globals().get("atm_skew_term_structure_comparison", pd.DataFrame()), CFG)

display(calibration_slice_rmse_diagnostics.head(20))
display(log_atm_skew_error_by_maturity.head(20))


,model,slice,status
0,heston,unavailable,missing_residual_panel
1,lifted_heston,unavailable,missing_residual_panel


,maturity_bucket,n_date_maturity_pairs,model,market_abs_skew,model_abs_skew,log_atm_skew_error,abs_log_atm_skew_error,status
0,1W,161,heston,1.127608,1.243386,0.097740,0.097740,available
1,2W,149,heston,0.845307,0.963933,0.131322,0.131322,available
2,1M,385,heston,0.720687,0.754546,0.045912,0.045912,available
3,2M,200,heston,0.638174,0.671694,0.051191,0.051191,available
4,3M,873,heston,0.566791,0.583932,0.029795,0.029795,available
5,outside_hedging_buckets,1921,heston,0.420605,0.413948,-0.015954,0.015954,available
6,1W,161,lifted_heston,1.127608,1.150831,0.020387,0.020387,available
7,2W,149,lifted_heston,0.845307,0.920478,0.085193,0.085193,available
8,1M,385,lifted_heston,0.720687,0.739503,0.025774,0.025774,available
9,2M,200,lifted_heston,0.638174,0.663737,0.039274,0.039274,available


## 5.5 Parameter Stability and Calibration Model-Risk Diagnostics

Parameter stability diagnostics ask whether calibrated models produce credible state inputs over time. A model can achieve acceptable average RMSE by pushing parameters to unstable or boundary-like values. That is dangerous for simulation and hedging because path dynamics depend on the calibrated parameters, not just on the fitted prices.

For Heston, instability in $\kappa$, $\theta$, $\xi$, $\rho$ or $V_0$ can imply unrealistic mean reversion, excessive variance shocks, or extreme leverage effects. For lifted Heston, instability can also affect the decomposition of variance across timescales. These issues may not show up clearly in a single fit metric.

This section therefore reports parameter time series, summary statistics and boundary warnings. The goal is not to demand perfectly constant parameters. Markets change. The goal is to detect whether parameter movement is economically interpretable or whether calibration is producing unstable numerical artefacts rather than economically interpretable states.

Parameter stability is model-risk evidence. If fitted parameters are unstable, then hedging conclusions should be framed more cautiously: the policy may be exploiting simulated states generated by noisy calibrations. If parameters are stable enough, the model-state comparison becomes more credible.


In [35]:
# Parameter stability and calibration model-risk diagnostics

def _read_parameter_panel(path: Path) -> pd.DataFrame:
    """Read a dated parameter panel from CSV or parquet if it exists."""
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    return pd.read_parquet(path) if path.suffix.lower() == ".parquet" else pd.read_csv(path)


def _clean_numeric_parameter(values: pd.Series) -> pd.Series:
    """Return a finite float series for one calibrated parameter column."""
    if values is None or pd.api.types.is_bool_dtype(values):
        return pd.Series(dtype="float64")
    out = pd.to_numeric(values, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    return out.astype("float64") if not out.empty else pd.Series(dtype="float64")


def _load_dated_parameter_panels(cfg: ExperimentConfig) -> list[tuple[str, pd.DataFrame, str]]:
    """
    Load date-level calibrated parameter panels only.

    Summary tables such as ``heston_parameter_summary.csv`` are deliberately excluded:
    their rows are statistics such as count, mean and quantiles, not market-date
    parameter observations.
    """
    candidate_sources = [
        ("heston", "heston_calibration", cfg.paths.heston_calibration_path),
        ("heston", None, table_path("heston_calibration_parameter_panel.csv", cfg)),
        ("heston", None, table_path("heston_calibrated_parameters.csv", cfg)),
        ("lifted_heston", "lifted_calibration", cfg.paths.lifted_calibration_path),
        ("lifted_heston", None, table_path("lifted_heston_calibration_parameter_panel.csv", cfg)),
        ("lifted_heston", None, table_path("lifted_heston_calibrated_parameters.csv", cfg)),
    ]

    panels: list[tuple[str, pd.DataFrame, str]] = []
    seen: set[tuple[str, str]] = set()

    for model, live_name, path in candidate_sources:
        frame = globals().get(live_name) if live_name else None
        if not isinstance(frame, pd.DataFrame) or frame.empty:
            frame = _read_parameter_panel(Path(path))
        if not isinstance(frame, pd.DataFrame) or frame.empty or "date" not in frame.columns:
            continue

        frame = frame.copy()
        frame["date"] = pd.to_datetime(frame["date"], errors="coerce").dt.normalize()
        frame = frame.dropna(subset=["date"])
        if frame.empty:
            continue

        source_name = str(live_name or Path(path).name)
        key = (model, source_name)
        if key in seen:
            continue
        seen.add(key)
        panels.append((model, frame, source_name))

    return panels


def _calibrated_parameter_columns(frame: pd.DataFrame) -> list[str]:
    """Select plausible model-parameter columns from a dated calibration panel."""
    excluded_exact = {
        "date", "valuation_date", "expiry", "exdate", "maturity", "maturity_bucket",
        "model", "panel", "status", "calibration_status", "success", "valid",
        "source_file", "seed_id", "split", "option_type",
    }
    excluded_tokens = (
        "rmse", "error", "loss", "residual", "price", "iv", "vega", "objective",
        "rows", "n_", "count", "time", "cpu", "seconds", "path", "flag", "warning",
        "reason", "bucket", "rank", "weight", "spread", "volume", "interest",
        "strike", "spot", "forward", "moneyness", "payoff", "premium",
    )

    cols: list[str] = []
    for col in frame.columns:
        name = str(col).lower()
        if name in excluded_exact or any(token in name for token in excluded_tokens):
            continue
        values = _clean_numeric_parameter(frame[col])
        if values.size >= 5 and values.nunique() > 1:
            cols.append(col)
    return cols


def calibration_parameter_stability_summary(cfg: ExperimentConfig) -> pd.DataFrame:
    """Summarise daily calibrated parameter dispersion for Heston and lifted Heston."""
    rows: list[dict[str, Any]] = []

    for model, frame, source_name in _load_dated_parameter_panels(cfg):
        for column in _calibrated_parameter_columns(frame):
            values = _clean_numeric_parameter(frame[column])
            if values.empty:
                continue
            mean = float(values.mean())
            std = float(values.std(ddof=1)) if values.size > 1 else 0.0
            rows.append({
                "model": model,
                "parameter": str(column),
                "n": int(values.size),
                "mean": mean,
                "std": std,
                "p05": float(values.quantile(0.05)),
                "median": float(values.median()),
                "p95": float(values.quantile(0.95)),
                "min": float(values.min()),
                "max": float(values.max()),
                "coefficient_of_variation_abs": float(std / max(abs(mean), 1e-12)),
                "source_file": source_name,
            })

    out = pd.DataFrame(rows, columns=[
        "model", "parameter", "n", "mean", "std", "p05", "median", "p95",
        "min", "max", "coefficient_of_variation_abs", "source_file",
    ])
    out.to_csv(table_path("calibration_parameter_stability_summary.csv", cfg), index=False)
    return out


def calibration_parameter_boundary_warnings(stability: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    """
    Flag simple calibration model-risk warnings.

    These flags are diagnostic, not formal rejection tests. They are meant to direct
    attention to parameters that are close to boundaries or unusually unstable.
    """
    rows: list[dict[str, str]] = []

    if stability is not None and not stability.empty:
        for _, row in stability.iterrows():
            parameter = str(row["parameter"]).lower()
            warnings: list[str] = []

            if row["coefficient_of_variation_abs"] > 5:
                warnings.append("high_relative_variability")
            if abs(float(row["median"])) < 1e-10 and float(row["std"]) > 1e-8:
                warnings.append("near_zero_median_with_variability")
            if float(row["p05"]) == float(row["p95"]):
                warnings.append("almost_constant_parameter")
            if "rho" in parameter and max(abs(float(row["p05"])), abs(float(row["p95"]))) > 0.98:
                warnings.append("correlation_near_boundary")
            if parameter.startswith(("v0", "theta", "xi", "kappa")) and float(row["p05"]) <= 0:
                warnings.append("positive_parameter_lower_tail_non_positive")

            for warning in warnings:
                rows.append({
                    "model": str(row["model"]),
                    "parameter": str(row["parameter"]),
                    "warning": warning,
                    "evidence": (
                        f"median={row['median']:.6g}; p05={row['p05']:.6g}; "
                        f"p95={row['p95']:.6g}; cv_abs={row['coefficient_of_variation_abs']:.6g}"
                    ),
                })

    out = pd.DataFrame(rows, columns=["model", "parameter", "warning", "evidence"])
    if out.empty:
        out = pd.DataFrame([{
            "model": "",
            "parameter": "",
            "warning": "none",
            "evidence": "No simple parameter-boundary warning detected from available dated panels.",
        }])
    out.to_csv(table_path("calibration_parameter_boundary_warnings.csv", cfg), index=False)
    return out


def plot_calibration_parameter_stability(cfg: ExperimentConfig) -> list[Path]:
    """Plot robustly scaled dated parameter time series for available calibration panels."""
    paths: list[Path] = []
    if not MATPLOTLIB_AVAILABLE:
        return paths

    for model, frame, _source_name in _load_dated_parameter_panels(cfg):
        columns = _calibrated_parameter_columns(frame)[:8]
        if not columns:
            continue

        plot_frame = frame[["date", *columns]].copy().sort_values("date")
        fig, ax = plt.subplots(figsize=(9, 4.5))

        for column in columns:
            values = _clean_numeric_parameter(plot_frame[column])
            if values.size < 3:
                continue
            aligned = plot_frame.loc[values.index, ["date"]].copy()
            median = float(values.median())
            scale = float(values.quantile(0.95) - values.quantile(0.05))
            if scale <= 1e-12:
                scale = float(values.std(ddof=1)) if values.size > 1 else 1.0
            aligned["scaled_value"] = (values - median) / max(scale, 1e-12)
            ax.plot(aligned["date"], aligned["scaled_value"], linewidth=1.1, label=str(column))

        ax.set_title(f"{model.replace('_', ' ').title()} calibrated-parameter stability")
        ax.set_xlabel("Date")
        ax.set_ylabel("Robustly scaled parameter value")
        ax.grid(True, alpha=0.3)
        ax.legend(loc="best", fontsize=8, ncol=2)
        fig.tight_layout()

        path = figure_path(f"{model}_parameter_stability_time_series.png", cfg)
        save_and_show_figure(fig, path, dpi=180)
        paths.append(path)

    return paths


calibration_parameter_stability = calibration_parameter_stability_summary(CFG)
calibration_parameter_boundary_audit = calibration_parameter_boundary_warnings(
    calibration_parameter_stability, CFG
)
calibration_parameter_stability_paths = plot_calibration_parameter_stability(CFG)

display(calibration_parameter_stability)
display(calibration_parameter_boundary_audit)
calibration_parameter_stability_paths

<Figure size 1170x585 with 1 Axes>

<Figure size 1170x585 with 1 Axes>

,model,parameter,n,mean,std,p05,median,p95,min,max,coefficient_of_variation_abs,source_file
0,heston,chunk_id,2844,11.806610,6.424146,2.000000e+00,12.000000,22.000000,1.000000e+00,23.000000,0.544114,heston_calibration
1,heston,chunk_id_fit,2844,11.806610,6.424146,2.000000e+00,12.000000,22.000000,1.000000e+00,23.000000,0.544114,heston_calibration
2,heston,feller_margin,2869,-0.511169,0.584083,-1.437795e+00,-0.337056,-0.133742,-7.693201e+00,1.094878,1.142642,heston_calibration
3,heston,kappa,2869,3.447517,3.449302,1.154057e+00,2.143673,13.458681,2.000000e-02,14.839164,1.000518,heston_calibration
4,heston,optimizer_cost,25,0.000010,0.000016,3.406625e-07,0.000004,0.000029,3.197534e-07,0.000079,1.627448,heston_calibration
5,heston,optimizer_nfev,25,33.200000,36.282457,3.200000e+00,14.000000,100.000000,3.000000e+00,100.000000,1.092845,heston_calibration
6,heston,rho,2869,-0.753630,0.102015,-8.793148e-01,-0.770750,-0.544845,-9.969674e-01,-0.169638,0.135365,heston_calibration
7,heston,theta,2869,0.061410,0.034281,2.372156e-02,0.052662,0.130216,1.000000e-04,0.379832,0.558227,heston_calibration
8,heston,v0,2869,0.031805,0.029627,7.232828e-03,0.023507,0.087452,1.369570e-03,0.674845,0.931515,heston_calibration
9,heston,xi,2869,0.871133,0.312418,5.704243e-01,0.779782,1.476932,1.355366e-01,2.988597,0.358634,heston_calibration


,model,parameter,warning,evidence
0,,,none,No simple parameter-boundary warning detected ...


[WindowsPath('C:/Users/lhkke/OneDrive - Queen Mary, University of London/Documents/Dissertation/Deep-Hedging-under-Lifted-Heston-MSc-Dissertation-/outputs_observable_h035_lag_consistent_retrained_full/figures/heston_parameter_stability_time_series.png'),
 WindowsPath('C:/Users/lhkke/OneDrive - Queen Mary, University of London/Documents/Dissertation/Deep-Hedging-under-Lifted-Heston-MSc-Dissertation-/outputs_observable_h035_lag_consistent_retrained_full/figures/lifted_heston_parameter_stability_time_series.png')]

# 6. Simulation Design

Simulation converts calibrated model states into training, validation and test paths for hedging policies. The simulation design is where the calibrated volatility model becomes a pathwise environment. It must therefore preserve the distinction between Heston and lifted Heston worlds while keeping the evaluation protocol comparable.

The main simulation object is a discrete rebalancing grid:

$$
0=t_0<t_1<\cdots<t_N=T.
$$

At each step, the policy observes its information state, chooses a stock position, pays transaction costs for position changes and accumulates terminal hedging error. The path generator supplies spot and latent volatility dynamics internally; the observable adapter later converts them into chronology-safe market features.

Common random numbers, deterministic seeds and fixed seed panels help reduce noise in model comparisons. They do not create market-state diversity. That is why the notebook separately reports the number of calibration dates and seed rows. More simulated paths reduce Monte Carlo error conditional on a date; more dates improve evidence about robustness across market states.

This section should be interpreted as methodology infrastructure. It does not by itself show that a model is good. It defines the controlled experimental environment in which Heston, lifted Heston and policy architectures can be compared fairly.


In [36]:
# ============================================================
# Deterministic stratified seed-panel construction helpers
# ============================================================

SEED_PANEL_VERSION = "deterministic_stratified_seed_panel_v5_test30_5bucket_date_exclusive"
CANDIDATE_SEED_PANEL_VERSION = "deterministic_candidate_universe_v5_test30_5bucket_date_exclusive"
FINAL_FEASIBLE_SEED_PANEL_VERSION = "deterministic_feasible_panel_v5_test30_5bucket_date_exclusive"
EXPERIMENT_READINESS_VERSION = "experiment_readiness_deterministic_panel_v5_test30_5bucket_date_exclusive"
SEED_PANEL_EXCLUSION_VERSION = "seed_panel_exclusion_manifest_v5_test30_5bucket_date_exclusive"


def canonical_split_label(split: str) -> str:
    return "validation" if str(split) == "val" else str(split)


def internal_split_label(split: str) -> str:
    return "val" if str(split) == "validation" else str(split)


def panel_role_sequence(cfg: ExperimentConfig) -> tuple[str, ...]:
    seq = tuple(internal_split_label(s) for s in cfg.panel.role_sequence)
    if len(seq) != int(cfg.panel.strata_per_bucket):
        raise ValueError("Panel role sequence length must equal strata_per_bucket.")
    expected = {internal_split_label(k): int(v) for k, v in cfg.panel.quota_per_bucket.items()}
    actual = {s: seq.count(s) for s in set(seq)}
    if actual != expected:
        raise ValueError(f"Panel role sequence does not match quota_per_bucket: expected={expected}, actual={actual}")
    return seq


def stable_hash01(*tokens: Any, seed: int) -> float:
    raw = "|".join([str(int(seed)), *map(str, tokens)]).encode("utf-8")
    digest = hashlib.sha256(raw).hexdigest()
    return int(digest[:16], 16) / float(16 ** 16)


def stable_dataframe_hash(frame: pd.DataFrame, columns: Iterable[str]) -> str:
    cols = [c for c in columns if c in frame.columns]
    work = frame[cols].copy().sort_values(cols).reset_index(drop=True)
    for col in work.columns:
        if np.issubdtype(work[col].dtype, np.datetime64):
            work[col] = pd.to_datetime(work[col]).dt.strftime("%Y-%m-%d")
    payload = work.astype(str).to_dict("records")
    raw = json.dumps(payload, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def seed_bucket_availability(surface: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    require_columns(surface, {"date", "dte"}, "seed bucket availability input")
    work = surface.copy()
    work["date"] = pd.to_datetime(work["date"]).dt.normalize()
    rows = []
    for bucket, (target_dte, lo, hi) in cfg.hedging.maturity_buckets.items():
        g = work.loc[work["dte"].between(lo, hi)]
        rows.append({
            "maturity_bucket": bucket,
            "target_dte": target_dte,
            "dte_low": lo,
            "dte_high": hi,
            "surface_rows": int(len(g)),
            "candidate_dates": int(g["date"].nunique()) if not g.empty else 0,
        })
    return pd.DataFrame(rows)


def select_near_atm_straddle_seeds(surface: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    required = {"date", "dte", "strike", "spot_ref", "forward_ref", "call_equiv_mid", "T", "r", "q"}
    missing = required - set(surface.columns)
    if missing:
        raise ValueError(f"Cannot select straddle seeds; surface missing columns: {sorted(missing)}")

    work = surface.copy()
    work["date"] = pd.to_datetime(work["date"]).dt.normalize()
    if "log_forward_moneyness" not in work.columns:
        if "log_moneyness" not in work.columns:
            raise ValueError("Surface must contain log_forward_moneyness or log_moneyness.")
        work["log_forward_moneyness"] = work["log_moneyness"]

    rows = []
    invalid_parity_rows = []
    for date, day in work.groupby("date", sort=True):
        for bucket, (target_dte, lo, hi) in cfg.hedging.maturity_buckets.items():
            g = day[day["dte"].between(lo, hi)].copy()
            if g.empty:
                continue
            g["seed_score"] = (g["dte"] - target_dte).abs() / max(target_dte, 1) + 5.0 * g["log_forward_moneyness"].abs()
            selected = None
            for _, candidate in g.sort_values(["seed_score", "date", "strike"]).head(10).iterrows():
                call_mid = float(candidate["call_equiv_mid"])
                put_mid = call_mid - math.exp(-float(candidate["r"]) * float(candidate["T"])) * (float(candidate["forward_ref"]) - float(candidate["strike"]))
                if np.isfinite(put_mid) and put_mid > 0:
                    selected = candidate.copy()
                    selected["put_equiv_mid"] = put_mid
                    break
                invalid_parity_rows.append({
                    "date": pd.Timestamp(date).date().isoformat(),
                    "maturity_bucket": bucket,
                    "strike": float(candidate["strike"]),
                    "put_mid": put_mid,
                })
            if selected is None:
                continue
            selected["hedge_maturity_bucket"] = bucket
            selected["maturity_bucket"] = bucket
            selected["target_dte"] = int(target_dte)
            selected["target_maturity"] = int(target_dte)
            selected["target_expiry"] = selected.get("exdate", pd.NaT)
            selected["straddle_mid"] = float(selected["call_equiv_mid"]) + float(selected["put_equiv_mid"])
            selected["straddle_mid_convention"] = "call_equiv_mid_plus_parity_put_mid"
            selected["seed_id"] = f"{pd.Timestamp(date).date()}_{bucket}_{int(round(selected['strike']))}"
            rows.append(selected)

    seed_panel = pd.DataFrame(rows).reset_index(drop=True)
    if seed_panel.empty:
        raise ValueError("No near-ATM straddle candidates were selected. Check maturity buckets and surface filters.")
    missing_buckets = sorted(set(cfg.hedging.maturity_buckets) - set(seed_panel["maturity_bucket"].dropna().astype(str)))
    if missing_buckets:
        availability = seed_bucket_availability(work, cfg)
        raise ValueError(
            f"Maturity buckets missing from straddle candidate universe: {missing_buckets}. "
            f"Availability={availability.to_dict('records')}. Invalid parity candidates sampled={invalid_parity_rows[:10]}"
        )
    seed_panel["source_row_id"] = np.arange(len(seed_panel), dtype=int)
    return seed_panel


def assign_chronological_strata(bucket_df: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    df = bucket_df.sort_values(["date", "seed_id"]).reset_index(drop=True).copy()
    n = len(df)
    n_strata = int(cfg.panel.strata_per_bucket)
    if n < n_strata:
        bucket = str(df["maturity_bucket"].iloc[0]) if n else "unknown"
        raise ValueError(f"Not enough feasible candidates in bucket {bucket}: need {n_strata}, found {n}.")
    df["chrono_rank"] = np.arange(n, dtype=int)
    df["chrono_quantile"] = (df["chrono_rank"] + 0.5) / float(n)
    df["panel_stratum"] = np.floor(df["chrono_quantile"] * n_strata).astype(int).clip(0, n_strata - 1)
    return df


def assign_deterministic_panel_order(candidate_df: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    require_columns(candidate_df, {"seed_id", "date", "maturity_bucket"}, "candidate universe")
    parts = []
    role_map = {i: role for i, role in enumerate(panel_role_sequence(cfg))}
    for bucket in cfg.hedging.maturity_buckets:
        g = candidate_df.loc[candidate_df["maturity_bucket"].astype(str).eq(str(bucket))].copy()
        g = assign_chronological_strata(g, cfg)
        g["panel_hash"] = [
            stable_hash01(row.seed_id, pd.Timestamp(row.date).date().isoformat(), row.maturity_bucket, seed=cfg.panel.panel_selection_seed)
            for row in g.itertuples()
        ]
        g["split"] = g["panel_stratum"].map(role_map)
        g["split_label"] = g["split"].map(canonical_split_label)
        g["panel_selection_mode"] = cfg.panel.panel_selection_mode
        g["panel_selection_seed"] = int(cfg.panel.panel_selection_seed)
        g["deterministic_panel_version"] = cfg.panel.deterministic_panel_version
        g["manual_override"] = False
        g["selected_after_observing_hedging_performance"] = False
        parts.append(g)
    out = pd.concat(parts, ignore_index=True)
    return out.sort_values(["maturity_bucket", "panel_stratum", "panel_hash", "date", "seed_id"]).reset_index(drop=True)


In [37]:
# ============================================================
# Deterministic candidate universe and pre-selection exclusion manifest
# ============================================================

SIMULATION_FEASIBILITY_VERSION = "simulation_feasibility_v9_test30_5bucket_date_exclusive"
LIFTED_C_LP_SIMULATOR_VERSION = "lifted_c_lp_internal_timestep_refinement_v1"

SIMULATION_INFEASIBLE_SEEDS = pd.DataFrame([
    {
        "seed_id": "2017-06-12_1M_2440",
        "exclusion_reason": "lifted_c_lp_beta_invalid_after_constraint_adjustment_step_10",
    },
    {
        "seed_id": "2023-01-30_1M_4025",
        "exclusion_reason": "lifted_c_lp_alpha_invalid_after_internal_timestep_refinement_step_96",
    },
])
SIMULATION_INFEASIBLE_SEED_IDS = frozenset(SIMULATION_INFEASIBLE_SEEDS["seed_id"].astype(str))
print(
    f"Simulation-feasibility prior exclusions active: {len(SIMULATION_INFEASIBLE_SEED_IDS)} seed(s); "
    "these are simulator-state exclusions, not hedging-performance exclusions.",
    flush=True,
)


def simulation_infeasible_seed_ids() -> frozenset[str]:
    return SIMULATION_INFEASIBLE_SEED_IDS


def _calibration_metadata_by_date(heston: pd.DataFrame, lifted: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    h = _normalise_one_row_per_date(heston, "Heston")
    l = _normalise_one_row_per_date(lifted, "Lifted Heston")
    h_cols = [c for c in ["date", "kappa", "theta", "xi", "rho", "v0", "vega_scaled_rmse", "price_rmse", "loss", "quality_flag", "calibration_mode"] if c in h.columns]
    l_cols = [c for c in ["date", "kappa", "theta", "xi", "rho", "v0", "vega_scaled_rmse", "price_rmse", "loss", "quality_flag", "calibration_backend", "candidate_source"] if c in l.columns]
    hm = h[h_cols].rename(columns={c: f"heston_{c}" for c in h_cols if c != "date"})
    lm = l[l_cols].rename(columns={c: f"lifted_{c}" for c in l_cols if c != "date"})
    meta = hm.merge(lm, on="date", how="outer").sort_values("date").reset_index(drop=True)
    h_params = [f"heston_{c}" for c in ["kappa", "theta", "xi", "rho", "v0"] if f"heston_{c}" in meta.columns]
    l_params = [f"lifted_{c}" for c in ["kappa", "theta", "xi", "rho", "v0"] if f"lifted_{c}" in meta.columns]
    meta["world_available_heston"] = np.isfinite(meta[h_params].apply(pd.to_numeric, errors="coerce")).all(axis=1) if h_params else False
    meta["world_available_lifted"] = np.isfinite(meta[l_params].apply(pd.to_numeric, errors="coerce")).all(axis=1) if l_params else False
    meta["heston_calibration_version"] = METHODOLOGY_VERSION
    meta["lifted_calibration_version"] = METHODOLOGY_VERSION
    meta["lifted_calibration_parameterization"] = cfg.lifted.calibration_parameterization
    return meta


def build_seed_candidate_universe(surface: pd.DataFrame, heston: pd.DataFrame, lifted: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    target = select_near_atm_straddle_seeds(surface, cfg)
    target["date"] = pd.to_datetime(target["date"]).dt.normalize()
    meta = _calibration_metadata_by_date(heston, lifted, cfg)
    universe = target.merge(meta, on="date", how="left")

    numeric_required = ["strike", "spot_ref", "forward_ref", "call_equiv_mid", "put_equiv_mid", "straddle_mid", "T", "r", "q"]
    for col in numeric_required + ["iv_market", "dte"]:
        if col in universe.columns:
            universe[col] = pd.to_numeric(universe[col], errors="coerce")
    if "iv_market" not in universe.columns:
        universe["iv_market"] = np.nan

    universe["heston_calibration_rmse"] = pd.to_numeric(universe.get("heston_vega_scaled_rmse", np.nan), errors="coerce")
    universe["lifted_calibration_rmse"] = pd.to_numeric(universe.get("lifted_vega_scaled_rmse", np.nan), errors="coerce")
    universe["initial_variance_v0"] = pd.to_numeric(universe.get("lifted_v0", universe.get("heston_v0", np.nan)), errors="coerce")
    universe["atm_iv_proxy"] = pd.to_numeric(universe["iv_market"], errors="coerce")
    for col in ["kappa", "theta", "xi", "rho"]:
        universe[col] = pd.to_numeric(universe.get(f"lifted_{col}", universe.get(f"heston_{col}", np.nan)), errors="coerce")

    universe["world_available_heston"] = universe["world_available_heston"].fillna(False).astype(bool)
    universe["world_available_lifted"] = universe["world_available_lifted"].fillna(False).astype(bool)
    valid_bucket = universe["maturity_bucket"].astype(str).isin(set(map(str, cfg.hedging.maturity_buckets)))
    finite_market = np.isfinite(universe[numeric_required].apply(pd.to_numeric, errors="coerce")).all(axis=1)
    lifted_5p = universe["lifted_calibration_parameterization"].astype(str).eq(cfg.lifted.calibration_parameterization)

    universe["candidate_universe_version"] = CANDIDATE_SEED_PANEL_VERSION
    universe["exclusion_reason"] = np.select(
        [
            ~universe["world_available_heston"],
            ~universe["world_available_lifted"] | ~lifted_5p,
            ~valid_bucket,
            ~finite_market,
        ],
        [
            "missing_heston_calibration",
            "missing_lifted_5p_calibration",
            "invalid_maturity_bucket",
            "missing_required_market_inputs",
        ],
        default="accepted_pre_feasibility",
    )
    universe["pre_selection_or_post_selection"] = "pre_selection"
    universe["deterministic_panel_version"] = cfg.panel.deterministic_panel_version
    universe["panel_selection_seed"] = int(cfg.panel.panel_selection_seed)
    return universe.sort_values(["date", "maturity_bucket", "seed_id"]).reset_index(drop=True)


def candidate_universe_summary(universe: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    rows = [{
        "field": "candidate_rows_total",
        "value": int(len(universe)),
        "detail": "One near-ATM straddle target per date and maturity bucket where available.",
    }]
    for bucket, g in universe.groupby("maturity_bucket", sort=True):
        rows.append({
            "field": f"rows_bucket_{bucket}",
            "value": int(len(g)),
            "detail": f"date_min={pd.to_datetime(g['date']).min().date()}, date_max={pd.to_datetime(g['date']).max().date()}",
        })
    rows.append({
        "field": "lifted_calibration_parameterization",
        "value": cfg.lifted.calibration_parameterization,
        "detail": "Final lifted calibration must be full_5p_forward_curve_init.",
    })
    rows.append({
        "field": "deterministic_panel_version",
        "value": cfg.panel.deterministic_panel_version,
        "detail": f"seed={cfg.panel.panel_selection_seed}; mode={cfg.panel.panel_selection_mode}",
    })
    return pd.DataFrame(rows)


def build_preselection_exclusion_manifest(universe: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    excluded = universe.loc[~universe["exclusion_reason"].astype(str).eq("accepted_pre_feasibility")].copy()
    if excluded.empty:
        return pd.DataFrame(columns=[
            "seed_id", "date", "maturity_bucket", "exclusion_stage", "exclusion_reason", "exclusion_detail",
            "pre_selection_or_post_selection", "deterministic_panel_version", "panel_selection_seed", "timestamp_utc",
        ])
    out = pd.DataFrame({
        "seed_id": excluded["seed_id"].astype(str),
        "date": pd.to_datetime(excluded["date"]).dt.normalize(),
        "maturity_bucket": excluded["maturity_bucket"].astype(str),
        "exclusion_stage": "candidate_universe_gate",
        "exclusion_reason": excluded["exclusion_reason"].astype(str),
        "exclusion_detail": "pre-specified calibration/data/maturity gate",
        "pre_selection_or_post_selection": "pre_selection",
        "deterministic_panel_version": cfg.panel.deterministic_panel_version,
        "panel_selection_seed": int(cfg.panel.panel_selection_seed),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    })
    return out.reset_index(drop=True)


def _candidate_seed_panel_required_columns() -> set[str]:
    return {
        "seed_id", "date", "hedge_maturity_bucket", "maturity_bucket", "split", "split_label",
        "panel_stratum", "panel_hash", "straddle_mid", "spot_ref", "iv_market",
        "candidate_universe_version", "deterministic_panel_version", "panel_selection_seed",
        "panel_selection_mode", "manual_override", "selected_after_observing_hedging_performance",
        "world_available_heston", "world_available_lifted", "lifted_calibration_parameterization",
    }


def validate_candidate_seed_panel(candidate: pd.DataFrame, cfg: ExperimentConfig) -> None:
    require_columns(candidate, _candidate_seed_panel_required_columns(), "deterministic candidate seed panel")
    versions = set(candidate["deterministic_panel_version"].dropna().astype(str).unique())
    if versions != {cfg.panel.deterministic_panel_version}:
        raise ValueError(f"Candidate panel has stale deterministic version(s): {sorted(versions)}")
    if candidate["manual_override"].astype(bool).any():
        raise ValueError("Manual override detected in deterministic candidate panel.")
    if candidate["selected_after_observing_hedging_performance"].astype(bool).any():
        raise ValueError("Performance-based selection flag detected in candidate panel.")
    expected_strata = set(range(int(cfg.panel.strata_per_bucket)))
    candidate_caps = {
        internal_split_label(k): int(v)
        for k, v in cfg.seed_sizes.candidate_quota_per_bucket.items()
    }
    final_quotas = {
        internal_split_label(k): int(v)
        for k, v in cfg.panel.quota_per_bucket.items()
    }
    role_sequence = panel_role_sequence(cfg)

    for bucket in cfg.hedging.maturity_buckets:
        g = candidate.loc[
            candidate["maturity_bucket"].astype(str).eq(str(bucket))
        ].copy()

        got_strata = set(
            pd.to_numeric(g["panel_stratum"], errors="coerce")
            .dropna()
            .astype(int)
        )
        if got_strata != expected_strata:
            raise RuntimeError(
                f"Candidate panel bucket {bucket} missing strata: "
                f"expected={sorted(expected_strata)}, got={sorted(got_strata)}"
            )

        for split, cap in candidate_caps.items():
            expected_split_strata = {
                i for i, role in enumerate(role_sequence)
                if str(role) == str(split)
            }

            split_rows = g.loc[
                g["split"].astype(str).eq(str(split))
            ].copy()

            got_split_strata = set(
                pd.to_numeric(split_rows["panel_stratum"], errors="coerce")
                .dropna()
                .astype(int)
            )

            if got_split_strata != expected_split_strata:
                missing = sorted(expected_split_strata - got_split_strata)
                extra = sorted(got_split_strata - expected_split_strata)
                raise RuntimeError(
                    f"Candidate panel bucket {bucket}, split {split}: "
                    f"stratum mismatch; missing={missing}, extra={extra}."
                )

            count = int(len(split_rows))
            minimum = int(final_quotas[split])

            if count < minimum:
                raise RuntimeError(
                    f"Candidate panel bucket {bucket}, split {split}: "
                    f"need at least final quota {minimum}, got {count}."
                )

            if count > int(cap):
                raise RuntimeError(
                    f"Candidate panel bucket {bucket}, split {split}: "
                    f"candidate cap is {cap}, got {count}."
                )


def write_panel_selection_outputs(universe: pd.DataFrame, candidate: pd.DataFrame, exclusions: pd.DataFrame, cfg: ExperimentConfig) -> None:
    cfg.paths.panel_selection_dir.mkdir(parents=True, exist_ok=True)
    cfg.paths.table_dir.mkdir(parents=True, exist_ok=True)
    deps = [globals().get("SPX_HEDGING_CALIBRATION_PANEL_PATH", cfg.paths.surface_cache_path), cfg.paths.heston_calibration_path, cfg.paths.lifted_calibration_path]
    write_dataframe_cache(
        universe,
        cfg.paths.seed_candidate_universe_path,
        "seed_candidate_universe",
        "seed_panel_candidate",
        cfg,
        dependencies=deps,
        extra={"candidate_universe_version": CANDIDATE_SEED_PANEL_VERSION},
    )
    write_dataframe_cache(
        exclusions,
        cfg.paths.seed_panel_exclusion_manifest_path,
        "seed_panel_exclusion_manifest",
        "seed_panel_candidate",
        cfg,
        dependencies=[cfg.paths.seed_candidate_universe_path],
        extra={"seed_panel_exclusion_version": SEED_PANEL_EXCLUSION_VERSION},
    )
    summary = candidate_universe_summary(universe, cfg)
    summary.to_csv(cfg.paths.table_dir / "seed_candidate_universe_summary.csv", index=False)
    exclusions.to_csv(cfg.paths.table_dir / "seed_panel_exclusion_manifest.csv", index=False)
    exclusion_summary = exclusions.groupby(["exclusion_stage", "exclusion_reason"], dropna=False).size().rename("rows").reset_index() if not exclusions.empty else pd.DataFrame(columns=["exclusion_stage", "exclusion_reason", "rows"])
    exclusion_summary.to_csv(cfg.paths.table_dir / "seed_panel_exclusion_summary.csv", index=False)
    (cfg.paths.methodology_insert_deterministic_panel_path).write_text(
        "The final seed panel is constructed using a deterministic maturity-stratified selection rule. "
        "The candidate universe consists of dates for which both Heston and lifted-Heston five-parameter calibrations are available and the required target option surface is valid. "
        "The active hedge-maturity buckets are 1W, 2W, 1M, 2M and 3M. For each bucket, feasible candidate dates are sorted chronologically and divided into 58 chronological strata. "
        "The candidate universe is quota-controlled before the feasibility scan using 60 training, 30 validation and 90 test candidates per maturity bucket. "
        "One date is selected from each final stratum using a stable hash with a pre-specified panel seed. "
        "The selected dates per maturity bucket are assigned using a fixed role sequence, yielding 20 training dates, 8 validation dates and 30 test dates per maturity bucket. "
        "The legacy frozen test split is disabled for this expanded-test experiment. "
        "Exclusions are applied only through pre-specified data-quality, calibration-availability and simulation-feasibility gates and are recorded in an exclusion manifest. "
        "No dates are manually added or removed after observing hedging performance.\n\n"
        "The final lifted calibration uses the five-parameter forward-curve-initialised specification. Neural training uses cached tensor banks built from the deterministic train and validation panels, while production evaluation is performed on 10,000 simulated paths per test seed. Headline comparisons include only production-scale rows that pass the final audit.\n"
    )



def limit_candidate_seed_panel_by_role_quota(candidate: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    require_columns(candidate, {"maturity_bucket", "split", "panel_stratum", "panel_hash", "seed_id", "date"}, "candidate seed panel before quota limit")
    quotas = {internal_split_label(k): int(v) for k, v in cfg.seed_sizes.candidate_quota_per_bucket.items()}
    role_counts = {internal_split_label(k): int(v) for k, v in cfg.panel.quota_per_bucket.items()}
    frames = []
    for bucket in cfg.hedging.maturity_buckets:
        for split, quota in quotas.items():
            g = candidate.loc[
                candidate["maturity_bucket"].astype(str).eq(str(bucket))
                & candidate["split"].astype(str).eq(str(split))
            ].copy()
            if g.empty:
                raise RuntimeError(f"No candidate rows for bucket={bucket}, split={split}.")
            expected_strata = set(
                i for i, role in enumerate(panel_role_sequence(cfg))
                if str(role) == str(split)
            )
            got_strata = set(pd.to_numeric(g["panel_stratum"], errors="coerce").dropna().astype(int))
            missing = expected_strata - got_strata
            if missing:
                raise RuntimeError(f"Candidate quota gate missing strata for bucket={bucket}, split={split}: {sorted(missing)}")
            if quota < int(role_counts[split]):
                raise RuntimeError(f"Candidate quota for split={split} must be at least the final quota; got {quota} < {role_counts[split]}.")
            g = g.sort_values(["candidate_rank_within_stratum", "panel_stratum", "panel_hash", "date", "seed_id"]).head(quota)
            frames.append(g)
    out = pd.concat(frames, ignore_index=True).sort_values(["maturity_bucket", "split", "panel_stratum", "candidate_rank_within_stratum", "panel_hash"]).reset_index(drop=True)
    return out

def build_candidate_seed_panel(surface: pd.DataFrame, heston: pd.DataFrame, lifted: pd.DataFrame, cfg: ExperimentConfig) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    universe = build_seed_candidate_universe(surface, heston, lifted, cfg)
    exclusions = build_preselection_exclusion_manifest(universe, cfg)
    accepted = universe.loc[universe["exclusion_reason"].astype(str).eq("accepted_pre_feasibility")].copy()
    if accepted.empty:
        raise RuntimeError("No candidate rows remain after deterministic pre-selection gates.")
    candidate = assign_deterministic_panel_order(accepted, cfg)
    candidate["candidate_rank_within_stratum"] = candidate.groupby(["maturity_bucket", "panel_stratum"])["panel_hash"].rank(method="first").astype(int)
    candidate = limit_candidate_seed_panel_by_role_quota(candidate, cfg)
    candidate["world_required"] = "heston_and_lifted"
    validate_candidate_seed_panel(candidate, cfg)
    return candidate, universe, exclusions


def load_or_build_candidate_seed_panel(surface: pd.DataFrame, heston: pd.DataFrame, lifted: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    deps = [globals().get("SPX_HEDGING_CALIBRATION_PANEL_PATH", cfg.paths.surface_cache_path), cfg.paths.heston_calibration_path, cfg.paths.lifted_calibration_path]
    try:
        candidate = load_dataframe_cache(
            cfg.paths.candidate_seed_panel_path,
            "deterministic_candidate_seed_panel",
            "seed_panel_candidate",
            cfg,
            required_columns=_candidate_seed_panel_required_columns(),
            dependencies=deps,
        )
        validate_candidate_seed_panel(candidate, cfg)
        print(f"[candidate universe] loaded deterministic candidate rows={len(candidate)} from cache.", flush=True)
        return candidate
    except (FileNotFoundError, ValueError, RuntimeError) as exc:
        if not cfg.flags.can_rebuild("feasibility"):
            raise RuntimeError(
                "Candidate seed-panel cache is missing/stale and strict reuse forbids rebuilding. "
                "Set rebuild_from='feasibility', an earlier stage, or use 'auto'."
            ) from exc
        print(f"[candidate universe] cache unavailable/stale; rebuilding. Reason: {type(exc).__name__}: {str(exc)[:220]}", flush=True)
    candidate, universe, exclusions = build_candidate_seed_panel(surface, heston, lifted, cfg)
    write_dataframe_cache(
        candidate,
        cfg.paths.candidate_seed_panel_path,
        "deterministic_candidate_seed_panel",
        "seed_panel_candidate",
        cfg,
        dependencies=deps,
        extra={
            "candidate_universe_version": CANDIDATE_SEED_PANEL_VERSION,
            "deterministic_panel_version": cfg.panel.deterministic_panel_version,
            "panel_selection_seed": int(cfg.panel.panel_selection_seed),
        },
    )
    write_panel_selection_outputs(universe, candidate, exclusions, cfg)
    print(f"[candidate universe] built rows={len(universe)}; accepted_pre_feasibility={len(candidate)}.", flush=True)
    return candidate


candidate_seed_panel = load_or_build_candidate_seed_panel(surface_panel, heston_calibration, lifted_calibration, CFG)
display(candidate_seed_panel.groupby(["split_label", "maturity_bucket", "panel_stratum"]).size().rename("candidate_rows").reset_index().head(30))


Simulation-feasibility prior exclusions active: 2 seed(s); these are simulator-state exclusions, not hedging-performance exclusions.
[candidate universe] loaded deterministic candidate rows=845 from cache.


,split_label,maturity_bucket,panel_stratum,candidate_rows
0,test,1M,0,3
1,test,1M,2,3
2,test,1M,5,3
3,test,1M,6,3
4,test,1M,8,3
5,test,1M,10,3
6,test,1M,12,3
7,test,1M,14,3
8,test,1M,16,3
9,test,1M,18,3


In [38]:
@dataclass(frozen=True)
class ResearchArtifacts:
    """Explicit runtime artefacts passed through simulation, delta, training and evaluation paths.

    This replaces hidden notebook globals in core execution functions. The top-level notebook
    still creates one named object, but functions receive it as an argument and therefore cannot
    silently read stale calibration/pricer state from the surrounding namespace.
    """
    heston_calibration: pd.DataFrame
    lifted_calibration: pd.DataFrame
    forward_variance_curve: pd.DataFrame
    lifted_kernel: LiftedKernelGrid
    lifted_pricer: LiftedRiccatiFourierPricer


def get_seed_spot(seed_row: pd.Series) -> float:
    if "spot_ref" not in seed_row.index:
        raise KeyError("Seed row must contain CRSP-derived spot_ref.")
    value = float(seed_row["spot_ref"])
    if not np.isfinite(value) or value <= 0:
        raise ValueError(f"Invalid spot_ref={value!r}.")
    return value


def seed_entry_premium(seed_row: pd.Series) -> float:
    if "straddle_mid" not in seed_row.index:
        raise KeyError("Seed row must contain explicit straddle_mid.")
    value = float(seed_row["straddle_mid"])
    if not np.isfinite(value) or value <= 0:
        raise ValueError(f"Invalid straddle_mid={value!r}.")
    return value


def seed_entry_iv(seed_row: pd.Series) -> float:
    if "iv_market" not in seed_row.index:
        raise KeyError("Seed row must contain iv_market.")
    value = float(seed_row["iv_market"])
    if not np.isfinite(value) or value <= 0:
        raise ValueError(f"Invalid iv_market={value!r}.")
    return value


def stable_int_hash(*parts: Any, modulus: int = 2**31 - 1) -> int:
    raw = "|".join(map(str, parts)).encode("utf-8")
    return int(hashlib.sha256(raw).hexdigest()[:16], 16) % modulus


def common_path_seed(seed_row: pd.Series, split: str, cfg: ExperimentConfig, replicate: int = 0) -> int:
    return stable_int_hash(cfg.random_seed, seed_row["seed_id"], split, replicate)


def lookup_params_by_date(calib: pd.DataFrame, date: pd.Timestamp, param_cls: Callable[..., Any]) -> Any:
    """Return finite calibrated parameters for a date using the same fit-quality gate.

    Raw optimiser success is not a hard gate here. It is provenance. The hard
    requirements are finite parameters and no BADFIT/NUMFAIL quality flag, matching
    the calibration-quality gate used for seed selection.
    """
    day = pd.Timestamp(date).normalize()
    rows = calib.loc[pd.to_datetime(calib["date"]).dt.normalize().eq(day)]
    if rows.empty:
        raise KeyError(f"No calibration row for {day.date()}.")
    row = rows.iloc[0]
    bad_flags = {"BADFIT", "NUMFAIL", "FAIL", "FAILED"}
    if "quality_flag" in row.index and str(row["quality_flag"]).upper() in bad_flags:
        raise ValueError(f"Calibration for {day.date()} is flagged {row['quality_flag']}.")
    values = [float(row.kappa), float(row.theta), float(row.xi), float(row.rho), float(row.v0)]
    if not np.isfinite(values).all():
        raise ValueError(f"Calibration for {day.date()} has non-finite parameters.")
    return param_cls(*values)


def lookup_lifted_state_by_date(calib: pd.DataFrame, date: pd.Timestamp, forward_curve: pd.DataFrame) -> tuple[LiftedHestonParams, ForwardVarianceInputCurve]:
    params = lookup_params_by_date(calib, date, LiftedHestonParams)
    return params, effective_lifted_curve(lifted_curve_for_date(forward_curve, date), params, CFG)


def _common_brownian_normals(n_paths: int, n_steps: int, n_factors: int, seed: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    z_variance = rng.standard_normal((n_paths, n_steps))
    z_perp = rng.standard_normal((n_paths, n_steps))
    z_factor = rng.standard_normal((n_paths, n_steps, n_factors))
    return z_variance, z_perp, z_factor


def lifted_conditional_increment_coefficients(lambdas: np.ndarray, dt: float) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Conditional Gaussian law of OU Brownian integrals given ΔW.

    I_i=∫exp(-x_i(dt-s))dW_s. The conditional representation preserves the
    shared variance Brownian increment, making Heston/lifted simulations use
    common random numbers in a mathematically coherent way.
    """
    x = np.asarray(lambdas, dtype=float)
    phi = (1.0 - np.exp(-x * dt)) / np.maximum(x, 1e-14)
    decay = np.exp(-x * dt)
    denom = x[:, None] + x[None, :]
    covariance = (1.0 - np.exp(-denom * dt)) / np.maximum(denom, 1e-14)
    conditional = covariance - np.outer(phi, phi) / dt
    conditional = 0.5 * (conditional + conditional.T)
    eigval, eigvec = np.linalg.eigh(conditional)
    if eigval.min() < -1e-10:
        raise FloatingPointError("Conditional OU covariance is not positive semidefinite.")
    eigval = np.where(eigval < 0.0, 0.0, eigval)  # round-off only, never a variance-state repair
    sqrt_conditional = eigvec @ np.diag(np.sqrt(eigval))
    return decay, phi, sqrt_conditional


def hedge_steps_for_seed(seed_row: pd.Series, cfg: ExperimentConfig) -> int:
    dte = int(round(float(seed_row.get("dte", float(seed_row["T"]) * cfg.market.calendar_days_per_year))))
    return max(1, int(math.ceil(dte * cfg.hedging.hedge_steps_per_day)))


def simulate_heston_paths_np(seed_row: pd.Series, params: HestonParams, n_paths: int, n_steps: int, seed: int, variance_floor: float = 1e-10) -> dict[str, np.ndarray]:
    S0, K, T = get_seed_spot(seed_row), float(seed_row["strike"]), float(seed_row["T"])
    r, q = float(seed_row["r"]), float(seed_row["q"])
    dt = T / max(n_steps, 1)
    z_v, z_perp, _ = _common_brownian_normals(n_paths, n_steps, 1, seed)
    S = np.empty((n_paths, n_steps + 1)); V = np.empty_like(S)
    S[:, 0] = S0; V[:, 0] = params.v0
    sqrt_one_minus = math.sqrt(max(1.0 - params.rho ** 2, 0.0))
    for t in range(n_steps):
        dWv = math.sqrt(dt) * z_v[:, t]
        dWs = params.rho * dWv + sqrt_one_minus * math.sqrt(dt) * z_perp[:, t]
        V_pos = np.maximum(V[:, t], variance_floor)
        S[:, t + 1] = S[:, t] * np.exp((r - q - 0.5 * V_pos) * dt + np.sqrt(V_pos) * dWs)
        V[:, t + 1] = np.maximum(V[:, t] + params.kappa * (params.theta - V_pos) * dt + params.xi * np.sqrt(V_pos) * dWv, 0.0)
    times = np.linspace(0.0, T, n_steps + 1)
    return {"S":S,"V":V,"times":times,"K":K,"r":r,"q":q,"T":T,"entry_iv":seed_entry_iv(seed_row),"model_world":"heston","date":pd.Timestamp(seed_row["date"]).normalize()}


def _lifted_reconstruct_variance(U: np.ndarray, g0_path: np.ndarray, weights: np.ndarray) -> np.ndarray:
    U_arr = np.asarray(U, dtype=float)
    g0 = np.asarray(g0_path, dtype=float)
    if U_arr.ndim != 3:
        raise ValueError("Lifted factor path U must have shape (n_paths, n_times, n_factors).")
    if g0.shape[0] != U_arr.shape[1]:
        raise ValueError("g0_path length must match the time dimension of U.")
    return g0.reshape(1, -1) + np.einsum("ptf,f->pt", U_arr, np.asarray(weights, dtype=float))


def validate_lifted_path_state(paths: dict[str, np.ndarray], kernel: LiftedKernelGrid, cfg: ExperimentConfig, *, label: str = "lifted") -> dict[str, float]:
    """Hard validity checks for lifted factor paths.

    This is deliberately stricter than a plotting diagnostic. The notebook should fail here
    rather than pass an invalid factor state into Riccati pricing, deltas, neural features, or
    dissertation tables. Tiny negative values are accepted only up to round-off tolerance.
    """
    required = {"U", "V", "g0", "S", "times"}
    missing = required.difference(paths)
    if missing:
        raise KeyError(f"{label}: lifted path dictionary is missing {sorted(missing)}.")
    U = np.asarray(paths["U"], dtype=float)
    V = np.asarray(paths["V"], dtype=float)
    S = np.asarray(paths["S"], dtype=float)
    g0 = np.asarray(paths["g0"], dtype=float)
    recon = _lifted_reconstruct_variance(U, g0, kernel.weights)
    if not np.isfinite(U).all() or not np.isfinite(V).all() or not np.isfinite(S).all():
        raise FloatingPointError(f"{label}: lifted simulation produced non-finite paths.")
    recon_error = float(np.max(np.abs(V - recon)))
    if recon_error > cfg.lifted.valid_state_tolerance:
        raise FloatingPointError(f"{label}: V is inconsistent with g0 + weights·U; max error={recon_error:.3e}.")
    min_v = float(np.min(V))
    if min_v < -cfg.lifted.valid_state_tolerance:
        raise FloatingPointError(f"{label}: lifted variance left admissible state space; min V={min_v:.3e}.")
    if np.any(S <= 0.0):
        raise FloatingPointError(f"{label}: simulated stock path is non-positive.")
    return {"min_variance": min_v, "max_reconstruction_error": recon_error, "n_paths": float(V.shape[0]), "n_steps": float(V.shape[1] - 1)}



def _integral_exp_AW_exp_A(A: np.ndarray, W: np.ndarray, dt: float) -> np.ndarray:
    """Compute ∫₀ᵈᵗ exp(A(dt-u)) W exp(Au) du.

    This is adapted from the public C-LP helper implementation, but kept local so the
    notebook is self-contained and cacheable. It is used only for numerical coefficients;
    invalid variances are still rejected by downstream gates.
    """
    from scipy.linalg import eig
    if not SCIPY_AVAILABLE:
        raise RuntimeError("SciPy is required for the C-LP lifted-Heston simulator.")
    Lambda, Vmat = eig(A)
    Vinv = np.linalg.inv(Vmat)
    W_tilde = Vinv @ W @ Vmat
    U_svd, S_svd, VT_svd = np.linalg.svd(W_tilde)
    u = U_svd[:, 0] * np.sqrt(S_svd[0])
    v = VT_svd[0, :] * np.sqrt(S_svd[0])
    N = A.shape[0]
    M = np.zeros((N, N), dtype=complex)
    for i in range(N):
        for j in range(N):
            diff = Lambda[j] - Lambda[i]
            if abs(diff) > 1e-14:
                M[i, j] = u[i] * v[j] * (np.exp(diff * dt) - 1.0) / diff
            else:
                M[i, j] = u[i] * v[j] * dt
    E = np.exp(np.outer(Lambda, np.ones(N)) * dt)
    out = Vmat @ (E * M) @ Vinv
    return np.real_if_close(out, tol=1000).real


@lru_cache(maxsize=16)
def _cached_legendre_nodes_weights(n_nodes: int) -> tuple[np.ndarray, np.ndarray]:
    """Cache deterministic Gauss-Legendre nodes/weights used inside lifted simulation."""
    return np.polynomial.legendre.leggauss(int(n_nodes))


def _curve_g0_integral(curve: ForwardVarianceInputCurve, s: float, t: float, params: LiftedHestonParams, kernel: LiftedKernelGrid, *, n_nodes: int = 16) -> float:
    """Numerically integrate the deterministic lifted input g₀ over [s,t].

    The public C-LP reference assumes a closed-form lifted-Heston g₀. This
    notebook uses a calibrated piecewise forward-variance proxy ξ₀(t), so the
    integral is evaluated with deterministic Gauss-Legendre quadrature under the
    same g₀ convention used by pricing and simulation.
    """
    s = float(s); t = float(t)
    if t < s - 1e-14:
        raise ValueError("g0 integral requires t >= s.")
    if abs(t - s) <= 1e-14:
        return 0.0
    nodes, weights = _cached_legendre_nodes_weights(int(n_nodes))
    mid = 0.5 * (s + t)
    half = 0.5 * (t - s)
    eval_t = mid + half * nodes
    vals = np.asarray(curve.g0(eval_t, params.kappa, kernel), dtype=float)
    if np.any(~np.isfinite(vals)) or np.any(vals <= 0):
        raise FloatingPointError("C-LP g0 integral encountered invalid deterministic variance input.")
    return float(half * np.sum(weights * vals))


def _sample_inverse_gaussian(mean: np.ndarray, shape: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Sample inverse-Gaussian variables with mean/shape parameterisation.

    scipy.stats.invgauss uses mu=mean/shape and scale=shape. This wrapper enforces
    finite positive inputs before sampling so invalid simulator coefficients fail before
    they contaminate path generation.
    """
    from scipy.stats import invgauss
    mean = np.asarray(mean, dtype=float)
    shape = np.asarray(shape, dtype=float)
    if np.any(~np.isfinite(mean)) or np.any(~np.isfinite(shape)) or np.any(mean <= 0) or np.any(shape <= 0):
        raise FloatingPointError("Invalid inverse-Gaussian parameters in C-LP lifted simulator.")
    return invgauss.rvs(mu=mean / shape, scale=shape, size=mean.shape, random_state=rng)


def _lifted_internal_substep_schedule(cfg: ExperimentConfig) -> tuple[int, ...]:
    base = max(1, int(getattr(cfg.lifted, "simulation_internal_substeps_per_hedge_step", 1)))
    max_substeps = max(base, int(getattr(cfg.lifted, "simulation_max_internal_substeps_per_hedge_step", base)))
    out: list[int] = []
    current = base
    while current <= max_substeps:
        out.append(int(current))
        if current == max_substeps:
            break
        current = min(max_substeps, current * 2)
    return tuple(dict.fromkeys(out))


def _simulate_lifted_paths_np_once(
    seed_row: pd.Series,
    params: LiftedHestonParams,
    curve: ForwardVarianceInputCurve,
    kernel: LiftedKernelGrid,
    n_paths: int,
    n_steps: int,
    seed: int,
    variance_floor: float,
    cfg: ExperimentConfig,
    *,
    internal_substeps: int,
) -> dict[str, np.ndarray]:
    """Run one C-LP lifted-Heston simulation at a fixed internal timestep."""
    if not SCIPY_AVAILABLE:
        raise RuntimeError("SciPy is required for the C-LP lifted-Heston simulator.")
    from scipy.integrate import solve_ivp
    from scipy.linalg import expm

    S0, K, T = get_seed_spot(seed_row), float(seed_row["strike"]), float(seed_row["T"])
    r, q = float(seed_row["r"]), float(seed_row["q"])
    if T <= 0 or n_steps < 1 or n_paths < 1:
        raise ValueError("Lifted simulation requires positive maturity, n_steps and n_paths.")
    internal_substeps = max(1, int(internal_substeps))
    sim_steps = int(n_steps) * internal_substeps
    dt = T / sim_steps
    rng = np.random.default_rng(seed)
    hedge_times = np.linspace(0.0, T, int(n_steps) + 1)
    sim_times = np.linspace(0.0, T, sim_steps + 1)
    g0_hedge = np.asarray(curve.g0(hedge_times, params.kappa, kernel), dtype=float)
    g0_sim = np.asarray(curve.g0(sim_times, params.kappa, kernel), dtype=float)
    if np.any(~np.isfinite(g0_hedge)) or np.any(~np.isfinite(g0_sim)) or min(np.min(g0_hedge), np.min(g0_sim)) <= 0.0:
        raise FloatingPointError("Initial lifted deterministic input g0 is invalid.")

    N = int(kernel.n_factors)
    omega = np.asarray(kernel.weights, dtype=float).reshape(N)
    x = np.asarray(kernel.lambdas, dtype=float).reshape(N)
    if np.any(omega <= 0) or np.any(x < 0):
        raise ValueError("Lifted kernel weights must be positive and factor rates non-negative.")

    A = -params.kappa * np.tile(omega, (N, 1)) - np.diag(x)
    invA = np.linalg.inv(A)
    eye = np.eye(N)
    c_vec = -params.kappa * np.ones(N)
    expA_dt = expm(A * dt)
    integ = _integral_exp_AW_exp_A(A, np.outer(np.ones(N), omega), dt)

    S = np.empty((n_paths, int(n_steps) + 1), dtype=float)
    V = np.empty((n_paths, int(n_steps) + 1), dtype=float)
    U_store = np.zeros((n_paths, int(n_steps) + 1, N), dtype=float)
    X_inc_store = np.zeros((n_paths, int(n_steps)), dtype=float)
    Z_inc_store = np.zeros((n_paths, int(n_steps)), dtype=float)
    S[:, 0] = S0
    V[:, 0] = g0_hedge[0]
    logS = np.full(n_paths, math.log(S0), dtype=float)
    U_state = np.zeros((N, n_paths), dtype=float)
    sqrt_one_minus = math.sqrt(max(1.0 - params.rho ** 2, 0.0))

    for sim_step in range(sim_steps):
        t0 = float(sim_times[sim_step])
        t1 = float(sim_times[sim_step + 1])
        hedge_slot = sim_step // internal_substeps
        G0_step = _curve_g0_integral(curve, t0, t1, params, kernel)
        gen_mu_N_at_end = ((expA_dt - eye) @ invA) @ U_state

        def particular(t: float, m: np.ndarray) -> np.ndarray:
            return A @ m + c_vec * _curve_g0_integral(curve, t0, float(t), params, kernel)

        sol = solve_ivp(particular, [t0, t1], np.zeros(N), dense_output=True, method="DOP853", rtol=1e-9, atol=1e-11, max_step=np.inf)
        if not sol.success:
            raise RuntimeError(f"C-LP deterministic particular ODE failed at internal step {sim_step}: {sol.message}")
        particular_end = sol.sol(t1)[:, None]
        mu_N_at_end = gen_mu_N_at_end + particular_end
        alpha = omega @ mu_N_at_end + G0_step
        if np.any(~np.isfinite(alpha)) or np.any(alpha <= variance_floor):
            raise FloatingPointError(f"C-LP integrated-variance mean alpha invalid at internal step {sim_step}.")

        def particular2(t: float, m: np.ndarray) -> np.ndarray:
            return A @ m + params.xi * (omega @ sol.sol(float(t)) + _curve_g0_integral(curve, t0, float(t), params, kernel))

        sol2 = solve_ivp(particular2, [t0, t1], np.zeros(N), dense_output=True, method="DOP853", rtol=1e-9, atol=1e-11, max_step=np.inf)
        if not sol2.success:
            raise RuntimeError(f"C-LP covariance particular ODE failed at internal step {sim_step}: {sol2.message}")
        hom_integral2 = params.xi * (integ - (expA_dt - eye) @ invA @ np.outer(np.ones(N), omega)) @ invA @ U_state
        cov_N = hom_integral2 + sol2.sol(t1)[:, None]
        cov = omega @ cov_N
        slope_X = cov / alpha
        beta_N = cov_N / alpha
        beta = np.array(slope_X, dtype=float, copy=True)
        if np.any(~np.isfinite(beta)) or np.any(~np.isfinite(beta_N)):
            raise FloatingPointError(f"C-LP beta became non-finite at internal step {sim_step}.")
        if np.any(beta <= variance_floor):
            raise FloatingPointError(
                f"C-LP beta non-positive before constraint adjustment at internal step {sim_step}; "
                f"min_beta={float(np.min(beta)):.3e}."
            )

        def b_constraint(y: np.ndarray | float, b: np.ndarray) -> np.ndarray:
            y_arr = np.asarray(y, dtype=float)
            return omega @ (
                U_state
                - x[:, None] * (mu_N_at_end + beta_N * (y_arr - alpha) / slope_X)
                - params.kappa * y_arr
                + params.xi * ((y_arr - alpha) / b)
            ) + g0_sim[sim_step + 1]

        constraint0 = b_constraint(0.0, beta)
        negative_mask = constraint0 < 0.0
        if np.any(negative_mask):
            denom = omega @ (U_state - x[:, None] * (mu_N_at_end - cov_N / slope_X)) + g0_sim[sim_step + 1]
            beta_new = omega.sum() * params.xi * alpha / denom
            invalid_projection = negative_mask & ((~np.isfinite(beta_new)) | (beta_new <= variance_floor))
            if np.any(invalid_projection):
                raise FloatingPointError(
                    f"C-LP beta constraint projection has no positive solution at internal step {sim_step}; "
                    f"bad_paths={int(np.sum(invalid_projection))}, "
                    f"min_denom={float(np.nanmin(denom[negative_mask])):.3e}, "
                    f"internal_substeps={internal_substeps}."
                )
            beta[negative_mask] = beta_new[negative_mask]

        if np.any(~np.isfinite(beta)) or np.any(beta <= variance_floor):
            raise FloatingPointError(f"C-LP beta invalid after constraint adjustment at internal step {sim_step}.")
        shape = (alpha / beta) ** 2
        X_new = _sample_inverse_gaussian(alpha, shape, rng)
        if np.any(~np.isfinite(X_new)) or np.any(X_new <= 0.0):
            raise FloatingPointError(f"C-LP sampled non-positive integrated variance at internal step {sim_step}.")
        Z_new = (X_new - alpha) / beta
        Xi_N = mu_N_at_end + beta_N * (X_new - alpha) / slope_X
        U_next = U_state - x[:, None] * Xi_N - params.kappa * X_new.reshape(1, -1) + params.xi * Z_new.reshape(1, -1)
        V_new = omega @ U_next + g0_sim[sim_step + 1]
        if np.any(~np.isfinite(U_next)) or np.any(~np.isfinite(V_new)) or np.min(V_new) < -cfg.lifted.valid_state_tolerance:
            raise FloatingPointError(
                f"C-LP lifted state left admissible domain at internal step {sim_step}; "
                f"min V={np.min(V_new):.3e}, internal_substeps={internal_substeps}."
            )

        logS += (r - q) * dt - 0.5 * X_new + params.rho * Z_new + sqrt_one_minus * np.sqrt(X_new) * rng.standard_normal(n_paths)
        U_state = U_next
        X_inc_store[:, hedge_slot] += X_new
        Z_inc_store[:, hedge_slot] += Z_new
        if (sim_step + 1) % internal_substeps == 0:
            hedge_idx = (sim_step + 1) // internal_substeps
            S[:, hedge_idx] = np.exp(logS)
            U_store[:, hedge_idx, :] = U_state.T
            V[:, hedge_idx] = V_new

    paths = {
        "S": S,
        "V": V,
        "U": U_store,
        "g0": g0_hedge,
        "forward_variance": np.asarray(curve.xi0(hedge_times), float),
        "times": hedge_times,
        "K": K,
        "r": r,
        "q": q,
        "T": T,
        "entry_iv": seed_entry_iv(seed_row),
        "model_world": "lifted",
        "date": pd.Timestamp(seed_row["date"]).normalize(),
        "curve": curve,
        "integrated_variance_increments": X_inc_store,
        "variance_brownian_integral_increments": Z_inc_store,
        "valid_state_attempts_max": np.array([0]),
        "simulation_scheme": "constrained_linear_projection_inverse_gaussian_internal_timestep",
        "simulation_internal_substeps_per_hedge_step": np.array([internal_substeps]),
        "simulation_effective_steps": np.array([sim_steps]),
    }
    validate_lifted_path_state(paths, kernel, cfg, label=f"lifted:{pd.Timestamp(seed_row['date']).date()}:{seed_row.get('hedge_maturity_bucket','?')}")
    return paths


def simulate_lifted_paths_np(seed_row: pd.Series, params: LiftedHestonParams, curve: ForwardVarianceInputCurve, kernel: LiftedKernelGrid, n_paths: int, n_steps: int, seed: int, variance_floor: float = 1e-10, cfg: ExperimentConfig | None = None) -> dict[str, np.ndarray]:
    """Constrained linear-projection lifted-Heston simulator with audited timestep refinement.

    The hedge grid is still one node per trading day. If the C-LP inverse-Gaussian
    coefficient constraints are numerically infeasible on that grid, the same C-LP
    scheme is retried on a finer internal simulation grid and then sampled back to
    the original hedge nodes. This is not a substitute simulator, not clipping, and
    not a hidden fallback: refinement attempts are printed and recorded in the path
    payload so tensor-batch metadata can be audited.
    """
    if cfg is None:
        raise ValueError("simulate_lifted_paths_np requires an explicit ExperimentConfig; hidden global CFG is forbidden here.")
    schedule = _lifted_internal_substep_schedule(cfg)
    last_exc: BaseException | None = None
    seed_id = str(seed_row.get("seed_id", "?"))
    bucket = str(seed_row.get("hedge_maturity_bucket", "?"))
    for attempt, internal_substeps in enumerate(schedule, start=1):
        try:
            paths = _simulate_lifted_paths_np_once(
                seed_row,
                params,
                curve,
                kernel,
                n_paths,
                n_steps,
                seed,
                variance_floor,
                cfg,
                internal_substeps=int(internal_substeps),
            )
            paths["simulation_timestep_refinement_attempts"] = np.array([attempt])
            if attempt > 1:
                print(
                    f"[lifted-sim timestep refine accepted] seed_id={seed_id} | bucket={bucket} | "
                    f"internal_substeps={internal_substeps} | attempts={attempt}",
                    flush=True,
                )
            return paths
        except FloatingPointError as exc:
            last_exc = exc
            if attempt >= len(schedule):
                break
            next_substeps = schedule[attempt]
            print(
                f"[lifted-sim timestep refine] seed_id={seed_id} | bucket={bucket} | "
                f"internal_substeps={internal_substeps} failed: {exc} | retrying={next_substeps}",
                flush=True,
            )
    raise FloatingPointError(
        f"C-LP lifted simulation failed after internal timestep refinement schedule={schedule}; "
        f"seed_id={seed_id}, bucket={bucket}. Last error: {last_exc}. "
        "This is a simulation-feasibility failure for this calibrated lifted state, not a calibration-cache failure. "
        "Add the seed_id to SIMULATION_INFEASIBLE_SEEDS only after confirming the failure is reproducible."
    )


def simulate_paths_for_world(seed_row: pd.Series, model_world: Literal["heston", "lifted"], cfg: ExperimentConfig, artifacts: ResearchArtifacts, *, n_paths: int, seed: int) -> dict[str, np.ndarray]:
    n_steps = hedge_steps_for_seed(seed_row, cfg)
    if model_world == "heston":
        params = lookup_params_by_date(artifacts.heston_calibration, seed_row["date"], HestonParams)
        return simulate_heston_paths_np(seed_row, params, n_paths, n_steps, seed, cfg.lifted.variance_floor)
    params, curve = lookup_lifted_state_by_date(artifacts.lifted_calibration, seed_row["date"], artifacts.forward_variance_curve)
    return simulate_lifted_paths_np(seed_row, params, curve, artifacts.lifted_kernel, n_paths, n_steps, seed, cfg.lifted.variance_floor, cfg)


def simulate_paths_for_policy_seed(seed_row: pd.Series, spec: Any, cfg: ExperimentConfig, artifacts: ResearchArtifacts, *, n_paths: int, seed: int, simulation_world: str | None = None) -> dict[str, np.ndarray]:
    world = simulation_world or getattr(spec, "training_world", None)
    if world not in {"heston", "lifted"}:
        raise ValueError(f"A concrete simulation world is required, got {world!r}.")
    return simulate_paths_for_world(seed_row, world, cfg, artifacts, n_paths=n_paths, seed=seed)


def _feasibility_excluded_seed_ids() -> frozenset[str]:
    return simulation_infeasible_seed_ids()


def _validation_seed_sample_by_bucket(seed_panel: pd.DataFrame, cfg: ExperimentConfig, split: str = "val") -> pd.DataFrame:
    require_columns(seed_panel, {"seed_id", "date", "hedge_maturity_bucket", "split"}, "seed panel")
    panel = seed_panel.copy()
    panel["date"] = pd.to_datetime(panel["date"]).dt.normalize()
    panel["seed_id"] = panel["seed_id"].astype(str)
    panel = panel.loc[~panel["seed_id"].isin(_feasibility_excluded_seed_ids())].copy()
    if panel.empty:
        raise ValueError("No feasible simulation-validation seeds remain after C-LP feasibility exclusions.")

    sampled = []
    for bucket in cfg.hedging.maturity_buckets:
        bucket_rows = panel.loc[panel["hedge_maturity_bucket"].astype(str).eq(str(bucket))].copy()
        if bucket_rows.empty:
            raise ValueError(f"No feasible simulation-validation candidate remains for maturity bucket {bucket}.")
        split_rows = bucket_rows.loc[bucket_rows["split"].astype(str).eq(str(split))].copy()
        if split_rows.empty:
            chosen = bucket_rows.sort_values(["date", "seed_id"]).iloc[[0]].copy()
            chosen["simulation_validation_note"] = f"same_bucket_feasible_seed_used_for_{split}_diagnostic_gap"
        else:
            chosen = split_rows.sort_values(["date", "seed_id"]).iloc[[0]].copy()
            chosen["simulation_validation_note"] = "requested_split_feasible_seed"
        sampled.append(chosen)

    out = pd.concat(sampled, ignore_index=True)
    if out["seed_id"].astype(str).isin(_feasibility_excluded_seed_ids()).any():
        raise RuntimeError("Known simulation-infeasible seed survived validation sampler.")
    missing = sorted(set(cfg.hedging.maturity_buckets) - set(out["hedge_maturity_bucket"].astype(str)))
    if missing:
        raise RuntimeError(f"Simulation validation sampler missing maturity buckets: {missing}")
    return out



def validate_lifted_martingale(paths: dict[str, np.ndarray], cfg: ExperimentConfig) -> float:
    S = np.asarray(paths["S"], dtype=float)
    T = float(paths["T"]); r = float(paths["r"]); q = float(paths["q"])
    S0 = float(S[0, 0])
    discounted_forward_mean = float(np.mean(S[:, -1] * np.exp(-(r - q) * T)))
    rel_error = abs(discounted_forward_mean / S0 - 1.0)
    if rel_error > cfg.lifted.martingale_relative_tolerance:
        raise RuntimeError(f"Lifted simulation martingale check failed: relative error={rel_error:.3%}.")
    return rel_error


def validate_lifted_integrated_variance(paths: dict[str, np.ndarray], cfg: ExperimentConfig) -> dict[str, float]:
    if "integrated_variance_increments" not in paths:
        raise KeyError("Lifted paths must include integrated_variance_increments for C-LP validation.")
    X = np.asarray(paths["integrated_variance_increments"], dtype=float)
    if X.ndim != 2:
        raise ValueError("integrated_variance_increments must have shape (n_paths, n_steps).")
    if not np.isfinite(X).all():
        raise FloatingPointError("C-LP integrated variance increments are non-finite.")
    min_x = float(np.min(X))
    if min_x <= cfg.lifted.integrated_variance_min_tolerance:
        raise FloatingPointError(f"C-LP integrated variance increment is non-positive: min={min_x:.3e}.")
    return {"min_integrated_variance_increment": min_x, "mean_integrated_variance_increment": float(np.mean(X))}


def _validation_time_indices_from_paths(paths: dict[str, np.ndarray], n_points: int, *, min_tau: float = 0.0) -> np.ndarray:
    times = np.asarray(paths["times"], dtype=float)
    if len(times) <= 2:
        return np.array([0], dtype=int)
    last_nonterminal = len(times) - 2
    idx = np.unique(np.linspace(0, last_nonterminal, max(1, int(n_points)), dtype=int))
    if min_tau > 0.0:
        T = float(paths["T"])
        idx = idx[(T - times[idx]) >= float(min_tau)]
        if len(idx) == 0:
            raise RuntimeError("No non-terminal states remain for lifted delta validation after min_tau filtering.")
    return idx


def _validation_path_indices(n_paths: int, n_points: int) -> np.ndarray:
    return np.unique(np.linspace(0, n_paths - 1, max(1, min(int(n_points), int(n_paths))), dtype=int))


def lifted_call_price_state_batch(
    pricer: LiftedRiccatiFourierPricer,
    S: np.ndarray,
    K: float,
    tau: float,
    r: float,
    q: float,
    params: LiftedHestonParams,
    curve: ForwardVarianceInputCurve,
    U: np.ndarray,
    current_time: float,
) -> np.ndarray:
    P1, P2 = pricer.probabilities_state_batch(S, K, tau, r, q, params, curve, U, current_time)
    price = np.asarray(S, dtype=float).reshape(-1) * math.exp(-float(q) * tau) * P1 - float(K) * math.exp(-float(r) * tau) * P2
    lower = np.maximum(np.asarray(S, dtype=float).reshape(-1) * math.exp(-float(q) * tau) - float(K) * math.exp(-float(r) * tau), 0.0)
    upper = np.asarray(S, dtype=float).reshape(-1) * math.exp(-float(q) * tau)
    # The Fourier probability routine itself permits a small probability tolerance; convert that
    # into a coherent price-bound tolerance rather than enforcing an inconsistent 1e-6 state-price
    # bound during finite-difference delta validation. This is a numerical integration tolerance,
    # not a model substitute or state repair.
    tol = 3e-3 * np.maximum(1.0, upper)
    if np.any(~np.isfinite(price)) or np.any(price < lower - tol) or np.any(price > upper + tol):
        worst_low = float(np.min(price - lower))
        worst_high = float(np.max(price - upper))
        raise FloatingPointError(f"Lifted state call prices violate no-arbitrage bounds: min(price-lower)={worst_low:.3e}, max(price-upper)={worst_high:.3e}.")
    return np.asarray(price, dtype=float)


def validate_lifted_time_step_convergence(seed_row: pd.Series, params: LiftedHestonParams, curve: ForwardVarianceInputCurve, kernel: LiftedKernelGrid, cfg: ExperimentConfig) -> float:
    base_steps = max(2, min(hedge_steps_for_seed(seed_row, cfg), cfg.lifted.one_factor_limit_steps))
    n_paths = min(cfg.lifted.validation_paths, 4000)
    seed0 = common_path_seed(seed_row, "lifted_timestep_validation", cfg)
    coarse = simulate_lifted_paths_np(seed_row, params, curve, kernel, n_paths, base_steps, seed0, cfg.lifted.variance_floor, cfg)
    fine = simulate_lifted_paths_np(seed_row, params, curve, kernel, n_paths, base_steps * 2, seed0, cfg.lifted.variance_floor, cfg)
    coarse_mean = float(np.mean(coarse["V"][:, -1]))
    fine_mean = float(np.mean(fine["V"][:, -1]))
    denom = max(abs(fine_mean), cfg.lifted.variance_floor)
    rel = abs(coarse_mean - fine_mean) / denom
    if rel > cfg.lifted.timestep_convergence_relative_tolerance:
        raise RuntimeError(f"Lifted timestep convergence check failed: rel={rel:.3%}.")
    return rel


def validate_lifted_cumulant_convexity(paths: dict[str, np.ndarray], params: LiftedHestonParams, curve: ForwardVarianceInputCurve, kernel: LiftedKernelGrid, cfg: ExperimentConfig) -> dict[str, float]:
    """Check convexity of the conditional log-moment-generating function.

    For a valid affine conditional law, a -> log E[exp(a log S_T)|F_t] is convex wherever the
    moment exists. This gate prevents invalid conditional moments from passing into hedging
    validation and dissertation tables.
    """
    pricer = LiftedRiccatiFourierPricer(kernel, cfg.lifted.pricing_nodes, cfg.lifted.riccati_steps, cfg.lifted.delta_coefficient_cache_size)
    S_all = np.asarray(paths["S"], dtype=float)
    U_all = np.asarray(paths["U"], dtype=float)
    times = np.asarray(paths["times"], dtype=float)
    K = float(paths["K"]); T = float(paths["T"]); r = float(paths["r"]); q = float(paths["q"])
    del K  # K is irrelevant for moments, but keeping extraction catches malformed path dictionaries.
    path_idx = _validation_path_indices(S_all.shape[0], cfg.lifted.cumulant_validation_paths)
    time_idx = _validation_time_indices_from_paths(paths, cfg.lifted.cumulant_validation_time_points)
    h = float(cfg.lifted.cumulant_convexity_bump)
    min_second = np.inf
    max_imag = 0.0
    checked = 0
    for j in time_idx:
        tau = T - float(times[j])
        if tau <= 1e-8:
            continue
        S = S_all[path_idx, j]
        U = U_all[path_idx, j, :]
        for a in cfg.lifted.cumulant_convexity_exponents:
            exponents = np.array([float(a) - h, float(a), float(a) + h], dtype=float)
            u = -1j * exponents
            moments = pricer.characteristic_state_batch(u, S, tau, r, q, params, curve, U, float(times[j]))
            max_imag = max(max_imag, float(np.max(np.abs(np.imag(moments)))))
            real_moments = np.real(moments)
            if np.any(~np.isfinite(real_moments)) or np.any(real_moments <= 0.0):
                raise FloatingPointError("Lifted cumulant convexity gate found invalid conditional moments.")
            kappa_vals = np.log(real_moments)
            second = (kappa_vals[:, 0] - 2.0 * kappa_vals[:, 1] + kappa_vals[:, 2]) / (h ** 2)
            min_second = min(min_second, float(np.min(second)))
            checked += int(second.size)
    if checked == 0:
        raise RuntimeError("Lifted cumulant convexity gate did not check any states.")
    if min_second < -cfg.lifted.cumulant_convexity_tolerance:
        raise RuntimeError(f"Lifted cumulant convexity failed: min second derivative={min_second:.3e}.")
    return {"min_cumulant_second_derivative": float(min_second), "max_cumulant_imaginary_part": float(max_imag), "cumulant_states_checked": float(checked)}




def _lifted_validation_pricers(kernel: LiftedKernelGrid, cfg: ExperimentConfig) -> list[LiftedRiccatiFourierPricer]:
    """Default plus high-precision Riccati/Fourier pricers for validation-only retries.

    The second pricer is not a substitute model. It uses the same lifted-Heston
    Riccati/Fourier equations with denser quadrature and more Riccati steps. It
    exists only to distinguish genuine state/model failures from sparse Fourier
    quadrature hiccups on simulated path states.
    """
    base_nodes = int(cfg.lifted.pricing_nodes)
    base_steps = int(cfg.lifted.riccati_steps)
    specs = [
        (base_nodes, base_steps),
        (max(base_nodes * 2, base_nodes + 64), max(base_steps * 2, base_steps + 160)),
    ]
    pricers: list[LiftedRiccatiFourierPricer] = []
    seen: set[tuple[int, int]] = set()
    for nodes, steps in specs:
        key = (int(nodes), int(steps))
        if key in seen:
            continue
        seen.add(key)
        pricers.append(LiftedRiccatiFourierPricer(kernel, key[0], key[1], cfg.lifted.delta_coefficient_cache_size))
    return pricers


def _call_delta_state_batch_with_precision_retry(
    pricers: list[LiftedRiccatiFourierPricer],
    S: np.ndarray,
    K: float,
    tau: float,
    r: float,
    q: float,
    params: LiftedHestonParams,
    curve: ForwardVarianceInputCurve,
    U: np.ndarray,
    current_time: float,
) -> tuple[np.ndarray, int]:
    """Evaluate lifted delta, retrying with denser numerical integration if needed."""
    last_exc: BaseException | None = None
    for retry_count, pricer in enumerate(pricers):
        try:
            out = pricer.call_delta_state_batch(S, K, tau, r, q, params, curve, U, current_time)
            out = np.asarray(out, dtype=float)
            if not np.isfinite(out).all():
                raise FloatingPointError("Lifted delta retry produced non-finite values.")
            return out, retry_count
        except (FloatingPointError, OverflowError, ValueError, np.linalg.LinAlgError) as exc:
            last_exc = exc
            continue
    raise FloatingPointError(f"Lifted delta failed all precision retries. Last error: {last_exc}")


def _call_price_state_batch_with_precision_retry(
    pricers: list[LiftedRiccatiFourierPricer],
    S: np.ndarray,
    K: float,
    tau: float,
    r: float,
    q: float,
    params: LiftedHestonParams,
    curve: ForwardVarianceInputCurve,
    U: np.ndarray,
    current_time: float,
) -> tuple[np.ndarray, int]:
    """Evaluate lifted state prices, retrying with denser numerical integration if needed."""
    last_exc: BaseException | None = None
    for retry_count, pricer in enumerate(pricers):
        try:
            out = lifted_call_price_state_batch(pricer, S, K, tau, r, q, params, curve, U, current_time)
            out = np.asarray(out, dtype=float)
            if not np.isfinite(out).all():
                raise FloatingPointError("Lifted state-price retry produced non-finite values.")
            return out, retry_count
        except (FloatingPointError, OverflowError, ValueError, np.linalg.LinAlgError) as exc:
            last_exc = exc
            continue
    raise FloatingPointError(f"Lifted state prices failed all precision retries. Last error: {last_exc}")

def validate_lifted_delta_bounds(seed_row: pd.Series, paths: dict[str, np.ndarray], params: LiftedHestonParams, curve: ForwardVarianceInputCurve, kernel: LiftedKernelGrid, cfg: ExperimentConfig) -> dict[str, Any]:
    """Strict audit of stock-only straddle delta bounds on lifted path states.

    Numerical evaluation failures are counted and returned in the validation row.
    They do not pass the gate and they are not repaired with clipped/substitute
    deltas. The notebook writes the diagnostics table, then the hard gate stops
    downstream training/evaluation if any state failed.
    """
    pricers = _lifted_validation_pricers(kernel, cfg)
    S_all = np.asarray(paths["S"], dtype=float)
    U_all = np.asarray(paths["U"], dtype=float)
    times = np.asarray(paths["times"], dtype=float)
    K = float(paths["K"]); T = float(paths["T"]); r = float(paths["r"]); q = float(paths["q"])
    path_idx = _validation_path_indices(S_all.shape[0], cfg.lifted.path_delta_validation_paths)
    time_idx = _validation_time_indices_from_paths(paths, cfg.lifted.path_delta_validation_time_points)
    max_abs = 0.0
    max_bound_excess = -np.inf
    checked = 0
    retry_states = 0
    failed_states = 0
    failure_messages: list[str] = []
    for j in time_idx:
        tau = T - float(times[j])
        if tau <= 1e-8:
            continue
        try:
            call_delta, retry_count = _call_delta_state_batch_with_precision_retry(
                pricers,
                S_all[path_idx, j],
                K,
                tau,
                r,
                q,
                params,
                curve,
                U_all[path_idx, j, :],
                float(times[j]),
            )
        except (FloatingPointError, OverflowError, ValueError, np.linalg.LinAlgError) as exc:
            failed_states += int(len(path_idx))
            failure_messages.append(f"time_index={int(j)}, tau={tau:.6g}: {str(exc)[:300]}")
            continue
        retry_states += int(retry_count > 0) * int(call_delta.size)
        straddle_delta = 2.0 * call_delta - math.exp(-q * tau)
        if not np.isfinite(straddle_delta).all():
            failed_states += int(straddle_delta.size)
            failure_messages.append(f"time_index={int(j)}, tau={tau:.6g}: non-finite straddle deltas")
            continue
        bound = math.exp(-q * tau)
        abs_delta = np.abs(straddle_delta)
        max_abs = max(max_abs, float(np.max(abs_delta)))
        max_bound_excess = max(max_bound_excess, float(np.max(abs_delta - bound)))
        checked += int(straddle_delta.size)
    gate_pass = bool(
        checked > 0
        and failed_states == 0
        and max_bound_excess <= cfg.lifted.delta_no_arbitrage_tolerance
    )
    return {
        "max_abs_delta": float(max_abs) if checked > 0 else np.nan,
        "max_delta_bound_excess": float(max_bound_excess) if checked > 0 else np.nan,
        "delta_bound_states_checked": float(checked),
        "delta_bound_precision_retry_states": float(retry_states),
        "delta_bound_failed_states": float(failed_states),
        "delta_bound_failure_reason": "; ".join(failure_messages[:5]),
        "delta_bound_gate_pass": gate_pass,
    }

def validate_lifted_path_delta_finite_difference(seed_row: pd.Series, paths: dict[str, np.ndarray], params: LiftedHestonParams, curve: ForwardVarianceInputCurve, kernel: LiftedKernelGrid, cfg: ExperimentConfig) -> dict[str, Any]:
    """Strict path-state delta finite-difference audit with precision retry.

    Failed analytic or bumped state-price evaluations are counted and fail the
    validation gate. No analytic delta is clipped and no finite-difference value is
    substituted for a failed Riccati/Fourier state.
    """
    pricers = _lifted_validation_pricers(kernel, cfg)
    S_all = np.asarray(paths["S"], dtype=float)
    U_all = np.asarray(paths["U"], dtype=float)
    times = np.asarray(paths["times"], dtype=float)
    K = float(paths["K"]); T = float(paths["T"]); r = float(paths["r"]); q = float(paths["q"])
    path_idx = _validation_path_indices(S_all.shape[0], cfg.lifted.path_delta_validation_paths)
    min_tau = 4.0 / float(cfg.market.calendar_days_per_year)
    time_idx = _validation_time_indices_from_paths(paths, cfg.lifted.path_delta_validation_time_points, min_tau=min_tau)
    max_error = 0.0
    checked = 0
    retry_states = 0
    failed_states = 0
    failure_messages: list[str] = []
    for j in time_idx:
        tau = T - float(times[j])
        if tau <= 1e-8:
            continue
        S = S_all[path_idx, j]
        U = U_all[path_idx, j, :]
        try:
            analytic, retry_count = _call_delta_state_batch_with_precision_retry(pricers, S, K, tau, r, q, params, curve, U, float(times[j]))
        except (FloatingPointError, OverflowError, ValueError, np.linalg.LinAlgError) as exc:
            failed_states += int(len(path_idx))
            failure_messages.append(f"analytic time_index={int(j)}, tau={tau:.6g}: {str(exc)[:300]}")
            continue
        retry_states += int(retry_count > 0) * int(analytic.size)
        state_errors = []
        for bump_frac in cfg.lifted.delta_validation_bump_fracs:
            bump = np.maximum(float(bump_frac) * S, 1e-5)
            try:
                plus, plus_retry = _call_price_state_batch_with_precision_retry(pricers, S + bump, K, tau, r, q, params, curve, U, float(times[j]))
                minus, minus_retry = _call_price_state_batch_with_precision_retry(pricers, S - bump, K, tau, r, q, params, curve, U, float(times[j]))
            except (FloatingPointError, OverflowError, ValueError, np.linalg.LinAlgError) as exc:
                failure_messages.append(f"bump={float(bump_frac):.3g}, time_index={int(j)}, tau={tau:.6g}: {str(exc)[:300]}")
                continue
            retry_states += int(plus_retry > 0) * int(plus.size)
            retry_states += int(minus_retry > 0) * int(minus.size)
            state_errors.append(np.abs((plus - minus) / (2.0 * bump) - analytic))
        if not state_errors:
            failed_states += int(analytic.size)
            continue
        best_error = np.min(np.vstack(state_errors), axis=0)
        finite = np.isfinite(best_error)
        failed_states += int((~finite).sum())
        if finite.any():
            max_error = max(max_error, float(np.max(best_error[finite])))
            checked += int(finite.sum())
    gate_pass = bool(
        checked > 0
        and failed_states == 0
        and max_error <= cfg.lifted.delta_finite_difference_tolerance
    )
    return {
        "max_path_delta_fd_error": float(max_error) if checked > 0 else np.nan,
        "path_delta_states_checked": float(checked),
        "path_delta_precision_retry_states": float(retry_states),
        "path_delta_failed_states": float(failed_states),
        "path_delta_failure_reason": "; ".join(failure_messages[:5]),
        "path_delta_fd_gate_pass": gate_pass,
    }


def validate_lifted_one_factor_heston_limit(seed_row: pd.Series, params: LiftedHestonParams, cfg: ExperimentConfig) -> dict[str, float]:
    """Simulation-level one-factor limit gate.

    With a single factor, x=0 and weight=1, the lifted variance obeys the classical Heston
    variance drift when g0(t)=v0+kappa*theta*t. The forward-variance curve representation below
    induces exactly that g0 through the notebook's curve.g0 convention.
    """
    limit_kernel = LiftedKernelGrid(
        H=0.5,
        alpha=1.0,
        n_factors=1,
        rn_scale=1.0,
        lambdas=np.array([0.0]),
        weights=np.array([1.0]),
        grid=pd.DataFrame({"factor": [1], "lambda": [0.0], "weight": [1.0], "time_scale_days": [np.inf]}),
        diagnostics=pd.DataFrame({"t": [0.1], "abs_rel_error": [0.0]}),
    )
    T = float(seed_row["T"])
    grid_t = np.linspace(max(T / 400.0, 1e-6), max(T, 1e-4), 400)
    expected_v = params.theta + (params.v0 - params.theta) * np.exp(-params.kappa * grid_t)
    heston_curve = ForwardVarianceInputCurve(grid_t, np.maximum(expected_v, cfg.market.forward_variance_min))
    n_paths = min(int(cfg.lifted.one_factor_limit_paths), int(cfg.lifted.validation_paths))
    n_steps = max(2, min(hedge_steps_for_seed(seed_row, cfg), int(cfg.lifted.one_factor_limit_steps)))
    paths = simulate_lifted_paths_np(seed_row, params, heston_curve, limit_kernel, n_paths, n_steps, common_path_seed(seed_row, "one_factor_heston_limit", cfg), cfg.lifted.variance_floor, cfg)
    state_stats = validate_lifted_path_state(paths, limit_kernel, cfg, label="one_factor_heston_limit")
    mean_v = paths["V"].mean(axis=0)
    expected = heston_curve.xi0(paths["times"])
    max_mean_error = float(np.max(np.abs(mean_v - expected)))
    if max_mean_error > cfg.lifted.one_factor_limit_mean_variance_tolerance:
        raise RuntimeError(f"One-factor lifted/Heston limit failed mean-variance gate: max error={max_mean_error:.3e}.")
    martingale_error = validate_lifted_martingale(paths, cfg)
    return {
        "one_factor_limit_max_mean_variance_error": max_mean_error,
        "one_factor_limit_martingale_error": martingale_error,
        "one_factor_limit_min_variance": state_stats["min_variance"],
    }


SIMULATION_VALIDATION_GATE_VERSION = "lifted_simulation_state_validation_v3_auto_feasibility"


def _simulation_validation_candidate_table(seed_panel: pd.DataFrame, cfg: ExperimentConfig, split: str = "val") -> dict[str, pd.DataFrame]:
    require_columns(seed_panel, {"seed_id", "date", "hedge_maturity_bucket", "split"}, "seed panel")
    panel = seed_panel.copy()
    panel["date"] = pd.to_datetime(panel["date"]).dt.normalize()
    panel["seed_id"] = panel["seed_id"].astype(str)
    panel = panel.loc[~panel["seed_id"].isin(_feasibility_excluded_seed_ids())].copy()
    if panel.empty:
        raise ValueError("No feasible simulation-validation candidates remain after recorded C-LP feasibility exclusions.")

    by_bucket: dict[str, pd.DataFrame] = {}
    for bucket in cfg.hedging.maturity_buckets:
        bucket_rows = panel.loc[panel["hedge_maturity_bucket"].astype(str).eq(str(bucket))].copy()
        if bucket_rows.empty:
            raise ValueError(f"No simulation-validation candidate remains for maturity bucket {bucket}.")
        split_rows = bucket_rows.loc[bucket_rows["split"].astype(str).eq(str(split))].copy()
        other_rows = bucket_rows.loc[~bucket_rows["split"].astype(str).eq(str(split))].copy()
        candidates = pd.concat([split_rows, other_rows], ignore_index=True)
        candidates = candidates.sort_values(["date", "seed_id"]).drop_duplicates("seed_id", keep="first").reset_index(drop=True)
        candidates["simulation_validation_candidate_source"] = np.where(
            candidates["split"].astype(str).eq(str(split)),
            f"{split}_split",
            f"same_bucket_non_{split}_split",
        )
        by_bucket[str(bucket)] = candidates
    return by_bucket


def _simulation_validation_base_row(seed_row: pd.Series) -> dict[str, Any]:
    return {
        "date": pd.Timestamp(seed_row["date"]),
        "seed_id": str(seed_row["seed_id"]),
        "maturity_bucket": seed_row.get("hedge_maturity_bucket", np.nan),
        "simulation_validation_version": SIMULATION_VALIDATION_GATE_VERSION,
        "simulation_failure_reason": "",
    }


def _validate_lifted_simulation_seed(seed_row: pd.Series, cfg: ExperimentConfig, artifacts: ResearchArtifacts) -> dict[str, Any]:
    row = _simulation_validation_base_row(seed_row)
    params, curve = lookup_lifted_state_by_date(artifacts.lifted_calibration, seed_row["date"], artifacts.forward_variance_curve)
    paths = simulate_lifted_paths_np(
        seed_row,
        params,
        curve,
        artifacts.lifted_kernel,
        min(cfg.lifted.validation_paths, 20_000),
        hedge_steps_for_seed(seed_row, cfg),
        common_path_seed(seed_row, "simulation_validation", cfg),
        cfg.lifted.variance_floor,
        cfg,
    )
    state_stats = validate_lifted_path_state(paths, artifacts.lifted_kernel, cfg, label="simulation_validation")
    integ_stats = validate_lifted_integrated_variance(paths, cfg)
    mean_v = paths["V"].mean(axis=0)
    xi0 = paths["forward_variance"]
    max_error = float(np.max(np.abs(mean_v - xi0)))
    martingale_error = validate_lifted_martingale(paths, cfg)
    timestep_rel = validate_lifted_time_step_convergence(seed_row, params, curve, artifacts.lifted_kernel, cfg)
    cumulant_stats = validate_lifted_cumulant_convexity(paths, params, curve, artifacts.lifted_kernel, cfg)
    one_factor_stats = validate_lifted_one_factor_heston_limit(seed_row, params, cfg)

    checks = {
        "mean_variance_gate": max_error <= cfg.lifted.simulation_mean_variance_tolerance,
        "martingale_gate": martingale_error <= cfg.lifted.martingale_relative_tolerance,
        "timestep_gate": timestep_rel <= cfg.lifted.timestep_convergence_relative_tolerance,
        "state_gate": state_stats["min_variance"] >= -cfg.lifted.valid_state_tolerance,
        "integrated_variance_gate": integ_stats["min_integrated_variance_increment"] > cfg.lifted.integrated_variance_min_tolerance,
        "cumulant_gate": cumulant_stats["min_cumulant_second_derivative"] >= -cfg.lifted.cumulant_convexity_tolerance,
        "one_factor_mean_gate": one_factor_stats["one_factor_limit_max_mean_variance_error"] <= cfg.lifted.one_factor_limit_mean_variance_tolerance,
        "one_factor_martingale_gate": one_factor_stats["one_factor_limit_martingale_error"] <= cfg.lifted.martingale_relative_tolerance,
    }
    pass_flag = all(bool(v) for v in checks.values())

    row.update({
        "max_mean_variance_error": max_error,
        "mean_variance_tolerance": cfg.lifted.simulation_mean_variance_tolerance,
        "martingale_relative_error": martingale_error,
        "martingale_tolerance": cfg.lifted.martingale_relative_tolerance,
        "timestep_convergence_relative_error": timestep_rel,
        "timestep_convergence_tolerance": cfg.lifted.timestep_convergence_relative_tolerance,
        "min_variance": state_stats["min_variance"],
        "max_reconstruction_error": state_stats["max_reconstruction_error"],
        **integ_stats,
        **cumulant_stats,
        **one_factor_stats,
        "max_valid_state_resampling_attempts": int(paths.get("valid_state_attempts_max", np.array([0]))[0]),
        "pass": bool(pass_flag),
    })
    if not pass_flag:
        row["simulation_failure_reason"] = ",".join(k for k, ok in checks.items() if not ok)
    return row


def validate_simulation_engines(seed_panel: pd.DataFrame, cfg: ExperimentConfig, artifacts: ResearchArtifacts) -> pd.DataFrame:
    candidate_map = _simulation_validation_candidate_table(seed_panel, cfg, "val")
    accepted_rows: list[dict[str, Any]] = []
    audit_rows: list[dict[str, Any]] = []
    caught = (FloatingPointError, OverflowError, ValueError, RuntimeError, np.linalg.LinAlgError, KeyError)

    for bucket in cfg.hedging.maturity_buckets:
        candidates = candidate_map[str(bucket)]
        accepted: dict[str, Any] | None = None
        for rank, (_, seed_row) in enumerate(candidates.iterrows(), start=1):
            try:
                row = _validate_lifted_simulation_seed(seed_row, cfg, artifacts)
            except caught as exc:
                row = _simulation_validation_base_row(seed_row)
                row.update({
                    "pass": False,
                    "simulation_failure_reason": str(exc)[:1000],
                    "max_mean_variance_error": np.nan,
                    "martingale_relative_error": np.nan,
                    "timestep_convergence_relative_error": np.nan,
                    "min_variance": np.nan,
                    "max_reconstruction_error": np.nan,
                })

            audit_rows.append({
                "maturity_bucket": str(bucket),
                "candidate_rank": int(rank),
                "seed_id": row["seed_id"],
                "date": row["date"],
                "split": str(seed_row.get("split", "")),
                "candidate_source": str(seed_row.get("simulation_validation_candidate_source", "")),
                "pass": bool(row.get("pass", False)),
                "simulation_failure_reason": row.get("simulation_failure_reason", ""),
                "simulation_validation_version": SIMULATION_VALIDATION_GATE_VERSION,
            })

            if bool(row.get("pass", False)):
                row["simulation_validation_candidate_rank"] = int(rank)
                row["simulation_validation_rejected_candidates_before_accept"] = int(rank - 1)
                row["simulation_validation_candidate_source"] = str(seed_row.get("simulation_validation_candidate_source", ""))
                accepted = row
                break

        if accepted is None:
            last = dict(row)
            last["pass"] = False
            last["simulation_failure_reason"] = (
                f"no_feasible_lifted_simulation_validation_seed_for_bucket_{bucket}; "
                + str(last.get("simulation_failure_reason", ""))[:900]
            )
            last["simulation_validation_candidate_rank"] = int(len(candidates))
            last["simulation_validation_rejected_candidates_before_accept"] = int(len(candidates))
            accepted_rows.append(last)
        else:
            accepted_rows.append(accepted)

    out = pd.DataFrame(accepted_rows)
    audit = pd.DataFrame(audit_rows)
    cfg.paths.table_dir.mkdir(parents=True, exist_ok=True)
    audit.to_csv(cfg.paths.table_dir / "simulation_validation_candidate_audit.csv", index=False)
    if out.empty:
        raise RuntimeError("Lifted simulation validation did not produce any rows.")
    missing = sorted(set(map(str, cfg.hedging.maturity_buckets)) - set(out["maturity_bucket"].astype(str)))
    if missing:
        raise RuntimeError(f"Lifted simulation validation missing maturity buckets: {missing}")
    return out


RESEARCH_ARTIFACTS = ResearchArtifacts(
    heston_calibration=heston_calibration,
    lifted_calibration=lifted_calibration,
    forward_variance_curve=forward_variance_curve,
    lifted_kernel=lifted_kernel,
    lifted_pricer=lifted_pricer,
)


# ============================================================
# Experiment-readiness preflight: simulation feasibility -> final feasible seed panel
# ============================================================

def _raw_tensor_n_paths_for_split(split: str, cfg: ExperimentConfig) -> int:
    split = str(split)
    if split == "train":
        return int(cfg.hedging.n_paths_train)
    if split == "val":
        return int(cfg.hedging.n_paths_val)
    if split == "test":
        return int(cfg.hedging.n_paths_test)
    raise ValueError(f"Unknown split {split!r}.")


def _feature_family_world_map() -> dict[str, str]:
    out = {"heston_scalar": "heston", "lifted_scalar": "lifted", "lifted_factors": "lifted", "lifted_factor_buckets": "lifted"}
    return out


def _feasibility_world_map() -> dict[str, str]:
    """Simulation feasibility depends on the model world, not the neural feature family."""
    return {"heston_world": "heston", "lifted_world": "lifted"}


def _feature_family_representative_specs() -> dict[str, Any]:
    reps: dict[str, Any] = {}
    if "POLICY_SPECS" in globals():
        for spec in POLICY_SPECS:
            if getattr(spec, "architecture", "analytic") == "analytic":
                continue
            if not getattr(spec, "is_core", False):
                continue
            if spec.training_world == "heston" and not spec.uses_factors:
                reps.setdefault("heston_scalar", spec)
            elif spec.training_world == "lifted" and spec.uses_factors and getattr(spec, "factor_representation", "raw20") == "buckets3":
                reps.setdefault("lifted_factor_buckets", spec)
            elif spec.training_world == "lifted" and spec.uses_factors:
                reps.setdefault("lifted_factors", spec)
            elif spec.training_world == "lifted":
                reps.setdefault("lifted_scalar", spec)
    return reps


def _simulation_path_cache_path(seed_row: pd.Series, feature_family: str, split: str, world: str, cfg: ExperimentConfig, *, n_paths: int, path_seed: int) -> Path:
    token = "".join(ch if (ch.isalnum() or ch in "._-") else "_" for ch in str(seed_row["seed_id"]))
    return (
        cfg.paths.simulation_path_cache_dir
        / SIMULATION_FEASIBILITY_VERSION
        / cfg.panel.deterministic_panel_version
        / str(feature_family)
        / str(split)
        / f"{token}_{world}_paths{int(n_paths)}_seed{int(path_seed)}.npz"
    )


def _save_simulation_path_cache(path: Path, paths: dict[str, Any], metadata: dict[str, Any]) -> None:
    arrays: dict[str, np.ndarray] = {}
    for key in ("S", "V", "U", "times"):
        if key in paths:
            arrays[key] = np.asarray(paths[key])
    for key in ("K", "r", "q", "T", "entry_iv"):
        if key in paths:
            arrays[key] = np.asarray(paths[key])
    arrays["metadata_json"] = np.asarray(json.dumps(_jsonable(metadata), sort_keys=True))
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, **arrays)


def load_simulation_path_cache(path: Path) -> dict[str, Any]:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    out: dict[str, Any] = {}
    with np.load(path, allow_pickle=False) as data:
        for key in data.files:
            if key == "metadata_json":
                out["metadata"] = json.loads(str(data[key].item()))
            else:
                out[key] = data[key]
    meta = out.get("metadata", {})
    for key in ("K", "r", "q", "T", "entry_iv"):
        if key in out:
            out[key] = float(np.asarray(out[key]).reshape(-1)[0])
    out["model_world"] = str(meta.get("world", meta.get("model_world", "unknown")))
    out["date"] = pd.Timestamp(meta.get("date")).normalize() if meta.get("date") else None
    return out


_SIMULATION_PATH_CACHE_INVALID_ERRORS = (
    OSError,
    EOFError,
    ValueError,
    zipfile.BadZipFile,
    zlib.error,
)


def validate_simulation_path_cache(path: Path, *, n_paths: int, n_steps: int) -> dict[str, Any]:
    cached = load_simulation_path_cache(path)
    if cached["S"].shape[0] != int(n_paths) or cached["S"].shape[1] != int(n_steps) + 1:
        raise ValueError(
            f"Path cache shape does not match current n_paths/n_steps: "
            f"S.shape={cached['S'].shape}, expected=({int(n_paths)}, {int(n_steps) + 1})."
        )
    if cached["V"].shape[0] != int(n_paths) or cached["V"].shape[1] != int(n_steps) + 1:
        raise ValueError(
            f"Path cache shape does not match current n_paths/n_steps: "
            f"V.shape={cached['V'].shape}, expected=({int(n_paths)}, {int(n_steps) + 1})."
        )
    return cached


def _simulation_feasibility_required_columns() -> set[str]:
    return {
        "feasibility_version", "experiment_readiness_version", "simulation_hash", "seed_id", "date", "split",
        "maturity_bucket", "feature_family", "world", "n_paths", "n_steps", "path_seed", "status",
        "failure_stage", "failure_reason", "attempted_substeps", "accepted_substeps", "elapsed_seconds",
        "path_cache_path", "simulator_backend", "deterministic_panel_version", "panel_selection_seed", "panel_stratum", "panel_hash",
    }


def _simulation_hash(cfg: ExperimentConfig) -> str:
    payload = {
        "simulation_feasibility_version": SIMULATION_FEASIBILITY_VERSION,
        "lifted_c_lp_simulator_version": LIFTED_C_LP_SIMULATOR_VERSION,
        "heston_simulator_version": "heston_euler_full_truncation_v1",
        "variance_floor": float(cfg.lifted.variance_floor),
        "internal_substep_start": int(cfg.lifted.simulation_internal_substeps_per_hedge_step),
        "internal_substep_max": int(cfg.lifted.simulation_max_internal_substeps_per_hedge_step),
        "hedge_steps_per_day": float(cfg.hedging.hedge_steps_per_day),
        "batch_paths": int(cfg.model.batch_paths),
        "raw_tensor_split_paths": {s: _raw_tensor_n_paths_for_split(s, cfg) for s in ("train", "val", "test")},
        "deterministic_panel_version": cfg.panel.deterministic_panel_version,
        "panel_selection_seed": int(cfg.panel.panel_selection_seed),
        "panel_selection_mode": cfg.panel.panel_selection_mode,
        "panel_role_sequence": panel_role_sequence(cfg),
        "panel_strata_per_bucket": int(cfg.panel.strata_per_bucket),
    }
    return hashlib.sha256(json.dumps(_jsonable(payload), sort_keys=True, separators=(",", ":")).encode("utf-8")).hexdigest()


def _load_valid_simulation_feasibility_manifest(candidate: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    deps = [cfg.paths.candidate_seed_panel_path, cfg.paths.lifted_calibration_path, cfg.paths.heston_calibration_path, cfg.paths.forward_variance_path]
    manifest = load_dataframe_cache(
        cfg.paths.simulation_feasibility_manifest_path,
        "simulation_feasibility_manifest",
        "simulation_feasibility",
        cfg,
        required_columns=_simulation_feasibility_required_columns(),
        dependencies=deps,
    )
    versions = set(manifest["feasibility_version"].dropna().astype(str).unique())
    if versions != {SIMULATION_FEASIBILITY_VERSION}:
        raise ValueError(f"Simulation-feasibility manifest has stale version(s): {sorted(versions)}")
    hashes = set(manifest["simulation_hash"].dropna().astype(str).unique())
    if hashes != {_simulation_hash(cfg)}:
        raise ValueError("Simulation-feasibility manifest simulation_hash is stale.")

    # Accept both the legacy feature-family scan and the storage-efficient world-only scan.
    # Feasibility is a property of the simulated model world, not of a downstream feature view.
    required_worlds = set(map(str, cfg.panel.simulation_feasibility_required_worlds))
    missing_pairs: list[tuple[str, str, str]] = []
    grouped = manifest.groupby(["seed_id", "split"])["world"].agg(lambda x: set(map(str, x)))
    for _, row in candidate.iterrows():
        key = (str(row["seed_id"]), str(row["split"]))
        available = grouped.get(key, set())
        for world in required_worlds:
            if world not in available:
                missing_pairs.append((key[0], key[1], world))
    if missing_pairs:
        raise ValueError(f"Simulation-feasibility manifest is missing {len(missing_pairs)} candidate/world rows.")

    # Full path arrays are optional. The deterministic manifest, model/config hash and
    # recorded seeds are sufficient to reuse completed feasibility work.
    if bool(getattr(cfg.flags, "CACHE_FEASIBILITY_PATH_ARRAYS", False)):
        pass_rows = manifest.loc[manifest["status"].astype(str).eq("pass")].copy()
        missing_path_caches = [
            p for p in pass_rows["path_cache_path"].astype(str)
            if p and not Path(p).exists()
        ]
        if missing_path_caches:
            raise FileNotFoundError(
                "Simulation-feasibility manifest pass rows reference missing path caches:\n"
                + "\n".join(missing_path_caches[:20])
            )
        corrupt_path_caches: list[str] = []
        for _, pass_row in pass_rows.iterrows():
            raw_path = str(pass_row.get("path_cache_path", "")).strip()
            if not raw_path:
                continue
            cache_path = Path(raw_path)
            try:
                validate_simulation_path_cache(
                    cache_path,
                    n_paths=int(pass_row["n_paths"]),
                    n_steps=int(pass_row["n_steps"]),
                )
            except _SIMULATION_PATH_CACHE_INVALID_ERRORS as exc:
                corrupt_path_caches.append(
                    f"{cache_path} :: {type(exc).__name__}: {str(exc)[:300]}"
                )
        if corrupt_path_caches:
            raise ValueError(
                "Simulation-feasibility manifest references corrupt/stale path caches:\n"
                + "\n".join(corrupt_path_caches[:20])
            )
    return manifest

def run_simulation_feasibility_scan(candidate: pd.DataFrame, cfg: ExperimentConfig, artifacts: ResearchArtifacts) -> pd.DataFrame:
    require_columns(candidate, {"seed_id", "date", "split", "hedge_maturity_bucket"}, "candidate seed panel")
    rows: list[dict[str, Any]] = []
    worlds = _feasibility_world_map()
    total = len(candidate) * len(worlds)
    sim_hash = _simulation_hash(cfg)
    prior_fail_map = dict(zip(
        SIMULATION_INFEASIBLE_SEEDS["seed_id"].astype(str),
        SIMULATION_INFEASIBLE_SEEDS["exclusion_reason"].astype(str),
    ))
    caught = (FloatingPointError, OverflowError, ValueError, RuntimeError, np.linalg.LinAlgError, KeyError)
    cache_arrays = bool(getattr(cfg.flags, "CACHE_FEASIBILITY_PATH_ARRAYS", False))
    counter = 0

    for _, seed_row in candidate.reset_index(drop=True).iterrows():
        split = str(seed_row["split"])
        # Preserve the original feasibility rigor: use the full split-specific path count.
        n_paths = _raw_tensor_n_paths_for_split(split, cfg)
        n_steps = hedge_steps_for_seed(seed_row, cfg)
        bucket = str(seed_row["hedge_maturity_bucket"])

        for feature_family, world in worlds.items():
            counter += 1
            seed_tag = f"fixed_{split}_{world}_feasibility_world_v2"
            path_seed = int(common_path_seed(seed_row, seed_tag, cfg))
            cache_path = _simulation_path_cache_path(
                seed_row, feature_family, split, world, cfg,
                n_paths=n_paths, path_seed=path_seed,
            )
            t0 = time.perf_counter()
            status = "pass"
            failure_stage = ""
            failure_reason = ""
            attempted_substeps = tuple(_lifted_internal_substep_schedule(cfg)) if world == "lifted" else (1,)
            accepted_substeps: int | float = 1
            needs_simulation = True

            if str(seed_row["seed_id"]) in prior_fail_map and world == "lifted":
                status = "fail"
                failure_stage = "pre_recorded_lifted_simulation_failure"
                failure_reason = prior_fail_map[str(seed_row["seed_id"])]
                accepted_substeps = np.nan
                needs_simulation = False
            elif cache_arrays and cache_path.exists():
                try:
                    cached = validate_simulation_path_cache(
                        cache_path, n_paths=int(n_paths), n_steps=int(n_steps)
                    )
                    accepted_substeps = int(cached.get("metadata", {}).get("accepted_substeps", 1))
                    needs_simulation = False
                except _SIMULATION_PATH_CACHE_INVALID_ERRORS as exc:
                    print(
                        f"[feasibility cache] deleting corrupt/stale path cache: {cache_path} | "
                        f"{type(exc).__name__}: {str(exc)[:220]}",
                        flush=True,
                    )
                    cache_path.unlink(missing_ok=True)

            if needs_simulation:
                try:
                    paths = simulate_paths_for_world(
                        seed_row, world, cfg, artifacts,
                        n_paths=n_paths, seed=path_seed,
                    )
                    if np.any(~np.isfinite(np.asarray(paths["S"]))) or np.any(~np.isfinite(np.asarray(paths["V"]))):
                        raise FloatingPointError("Simulated paths contain non-finite stock or variance values.")
                    accepted_substeps = int(np.asarray(
                        paths.get("simulation_internal_substeps_per_hedge_step", [1])
                    ).reshape(-1)[0])
                    metadata = {
                        "feasibility_version": SIMULATION_FEASIBILITY_VERSION,
                        "simulation_hash": sim_hash,
                        "seed_id": str(seed_row["seed_id"]),
                        "date": str(pd.Timestamp(seed_row["date"]).date()),
                        "split": split,
                        "maturity_bucket": bucket,
                        "feature_family": feature_family,
                        "world": world,
                        "n_paths": int(n_paths),
                        "n_steps": int(n_steps),
                        "path_seed": int(path_seed),
                        "accepted_substeps": int(accepted_substeps),
                    }
                    if cache_arrays:
                        _save_simulation_path_cache(cache_path, paths, metadata)
                except caught as exc:
                    status = "fail"
                    text = str(exc)[:1000]
                    lower = text.lower()
                    if "alpha" in lower:
                        failure_stage = "alpha"
                    elif "beta" in lower:
                        failure_stage = "beta"
                    elif "finite" in lower:
                        failure_stage = "finite_paths"
                    else:
                        failure_stage = type(exc).__name__
                    failure_reason = text
                    accepted_substeps = np.nan

            elapsed = float(time.perf_counter() - t0)
            path_text = str(cache_path) if cache_arrays and status == "pass" else ""
            print(
                f"[feasibility {counter}/{total}] {world} | {split} | {bucket} | "
                f"seed={seed_row['seed_id']} | {status}"
                + (f" {failure_stage}: {failure_reason[:160]}" if status != "pass"
                   else f" | substeps={accepted_substeps} | arrays_cached={cache_arrays}")
                + f" | {elapsed:.2f}s",
                flush=True,
            )
            rows.append({
                "feasibility_version": SIMULATION_FEASIBILITY_VERSION,
                "experiment_readiness_version": EXPERIMENT_READINESS_VERSION,
                "simulation_hash": sim_hash,
                "date": pd.Timestamp(seed_row["date"]),
                "seed_id": str(seed_row["seed_id"]),
                "split": split,
                "maturity_bucket": bucket,
                "feature_family": feature_family,
                "world": world,
                "n_paths": int(n_paths),
                "n_steps": int(n_steps),
                "path_seed": int(path_seed),
                "status": status,
                "failure_stage": failure_stage,
                "failure_reason": failure_reason,
                "attempted_substeps": ",".join(map(str, attempted_substeps)),
                "accepted_substeps": accepted_substeps,
                "elapsed_seconds": elapsed,
                "path_cache_path": path_text,
                "simulator_backend": "cpu_numpy",
                "deterministic_panel_version": str(seed_row.get(
                    "deterministic_panel_version", cfg.panel.deterministic_panel_version
                )),
                "panel_selection_seed": int(seed_row.get(
                    "panel_selection_seed", cfg.panel.panel_selection_seed
                )),
                "panel_stratum": int(seed_row.get("panel_stratum", -1)),
                "panel_hash": float(seed_row.get("panel_hash", np.nan)),
            })

    out = pd.DataFrame(rows)
    if out.empty:
        raise RuntimeError("Simulation-feasibility scan produced no rows.")
    return out

def load_or_run_simulation_feasibility_scan(candidate: pd.DataFrame, cfg: ExperimentConfig, artifacts: ResearchArtifacts) -> pd.DataFrame:
    deps = [cfg.paths.candidate_seed_panel_path, cfg.paths.lifted_calibration_path, cfg.paths.heston_calibration_path, cfg.paths.forward_variance_path]
    if cfg.flags.REBUILD_SIMULATION_FEASIBILITY_MANIFEST:
        print("[feasibility] explicit rebuild requested.", flush=True)
    elif cfg.flags.REUSE_SIMULATION_FEASIBILITY_MANIFEST:
        try:
            manifest = _load_valid_simulation_feasibility_manifest(candidate, cfg)
            print(f"[feasibility] loaded manifest rows={len(manifest)} from cache.", flush=True)
            return manifest
        except (FileNotFoundError, ValueError, RuntimeError) as exc:
            if not cfg.flags.can_rebuild("feasibility"):
                raise RuntimeError("Simulation-feasibility manifest is missing/stale and strict reuse forbids rebuilding. Set rebuild_from='feasibility' or use 'auto'.") from exc
            print(f"[feasibility] cache unavailable/stale; rebuilding. Reason: {type(exc).__name__}: {str(exc)[:220]}", flush=True)
    if not cfg.flags.can_rebuild("feasibility"):
        raise RuntimeError("Simulation-feasibility manifest is missing/stale and strict reuse forbids rebuilding. Set rebuild_from='feasibility' or use 'auto'.")
    manifest = run_simulation_feasibility_scan(candidate, cfg, artifacts)
    write_dataframe_cache(
        manifest,
        cfg.paths.simulation_feasibility_manifest_path,
        "simulation_feasibility_manifest",
        "simulation_feasibility",
        cfg,
        dependencies=deps,
        extra={"simulation_feasibility_version": SIMULATION_FEASIBILITY_VERSION, "simulation_hash": _simulation_hash(cfg)},
    )
    cfg.paths.table_dir.mkdir(parents=True, exist_ok=True)
    manifest.to_csv(cfg.paths.table_dir / "simulation_feasibility_manifest.csv", index=False)
    return manifest


def _final_seed_panel_required_columns() -> set[str]:
    return {
        "seed_id", "date", "hedge_maturity_bucket", "maturity_bucket", "split", "split_label",
        "panel_stratum", "panel_hash", "straddle_mid", "spot_ref", "iv_market",
        "seed_panel_version", "candidate_universe_version", "final_feasible_seed_panel_version",
        "simulation_feasibility_version", "deterministic_panel_version", "panel_selection_seed",
        "panel_selection_mode", "manual_override", "selected_after_observing_hedging_performance",
        "deterministic_panel_hash",
    }


def deterministic_seed_panel_hash(seeds: pd.DataFrame) -> str:
    return stable_dataframe_hash(seeds, ["seed_id", "date", "maturity_bucket", "panel_stratum", "split"])


def validate_final_feasible_seed_panel(seeds: pd.DataFrame, cfg: ExperimentConfig, *, require_feasibility: bool = True) -> None:
    require_columns(seeds, _final_seed_panel_required_columns(), "final deterministic feasible seed panel")
    expected = {internal_split_label(k): int(v) for k, v in cfg.panel.quota_per_bucket.items()}
    if bool(getattr(cfg.panel, "preserve_existing_test_split", False)):
        observed_test = seeds.loc[seeds["split"].astype(str).eq("test")].groupby("maturity_bucket").size()
        if not observed_test.empty:
            expected["test"] = int(observed_test.iloc[0])
    expected_total = len(cfg.hedging.maturity_buckets) * sum(expected.values())
    if len(seeds) != expected_total:
        raise RuntimeError(f"Final panel must contain exactly {expected_total} rows under active quotas; got {len(seeds)}.")
    if seeds["seed_id"].duplicated().any():
        raise RuntimeError("Duplicate seed_id found in final deterministic panel.")
    if seeds["manual_override"].astype(bool).any():
        raise RuntimeError("Manual override detected in final deterministic panel.")
    if seeds["selected_after_observing_hedging_performance"].astype(bool).any():
        raise RuntimeError("Performance-based seed selection detected in final deterministic panel.")
    leaked = sorted(set(seeds["seed_id"].astype(str)) & set(simulation_infeasible_seed_ids()))
    if require_feasibility and leaked:
        raise RuntimeError(f"Pre-recorded simulation-infeasible seeds remain in final panel: {leaked}")
    split_counts = seeds["split"].astype(str).value_counts().to_dict()
    for split, per_bucket in expected.items():
        total_expected = per_bucket * len(cfg.hedging.maturity_buckets)
        if int(split_counts.get(split, 0)) != total_expected:
            raise RuntimeError(f"Split {split}: expected {total_expected}, got {split_counts.get(split, 0)}.")
    for bucket, g in seeds.groupby("maturity_bucket"):
        counts = g["split"].astype(str).value_counts().to_dict()
        for split, expected_n in expected.items():
            if int(counts.get(split, 0)) != int(expected_n):
                raise RuntimeError(f"Bucket {bucket}, split {split}: expected {expected_n}, got {counts.get(split, 0)}.")
    date_split_counts = (
        seeds.assign(_date_key=pd.to_datetime(seeds["date"]).dt.normalize())
        .groupby("_date_key")["split"]
        .nunique()
    )
    leaked_dates = date_split_counts.loc[date_split_counts > 1]
    if not leaked_dates.empty:
        raise RuntimeError(
            "Calendar-date leakage detected across train/validation/test splits: "
            + ", ".join(str(pd.Timestamp(d).date()) for d in leaked_dates.index[:20])
        )
    panel_hashes = set(seeds["deterministic_panel_hash"].dropna().astype(str).unique())
    if len(panel_hashes) != 1:
        raise RuntimeError(f"Final panel must carry one deterministic panel hash; got {sorted(panel_hashes)}")


def build_simulation_exclusion_manifest(candidate: pd.DataFrame, feasibility: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    if feasibility.empty:
        return pd.DataFrame(columns=[
            "seed_id", "date", "maturity_bucket", "exclusion_stage", "exclusion_reason", "exclusion_detail",
            "pre_selection_or_post_selection", "deterministic_panel_version", "panel_selection_seed", "timestamp_utc",
        ])
    failed = feasibility.loc[feasibility["status"].astype(str).eq("fail")].copy()
    if failed.empty:
        return pd.DataFrame(columns=[
            "seed_id", "date", "maturity_bucket", "exclusion_stage", "exclusion_reason", "exclusion_detail",
            "pre_selection_or_post_selection", "deterministic_panel_version", "panel_selection_seed", "timestamp_utc",
        ])
    grouped = failed.sort_values(["seed_id", "feature_family", "world"]).drop_duplicates("seed_id")
    return pd.DataFrame({
        "seed_id": grouped["seed_id"].astype(str),
        "date": pd.to_datetime(grouped["date"]).dt.normalize(),
        "maturity_bucket": grouped["maturity_bucket"].astype(str),
        "exclusion_stage": "simulation_feasibility_scan",
        "exclusion_reason": "failed_simulation_feasibility",
        "exclusion_detail": grouped["failure_stage"].astype(str) + ": " + grouped["failure_reason"].astype(str),
        "pre_selection_or_post_selection": "pre_selection",
        "deterministic_panel_version": cfg.panel.deterministic_panel_version,
        "panel_selection_seed": int(cfg.panel.panel_selection_seed),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }).reset_index(drop=True)


def _feasible_seed_ids_by_required_worlds(feasibility: pd.DataFrame, cfg: ExperimentConfig) -> set[str]:
    pass_rows = feasibility.loc[feasibility["status"].astype(str).eq("pass")].copy()
    grouped = pass_rows.groupby("seed_id")["world"].agg(lambda x: set(map(str, x))).reset_index()
    required = set(cfg.panel.simulation_feasibility_required_worlds)
    return set(grouped.loc[grouped["world"].map(lambda x: required.issubset(x)), "seed_id"].astype(str))



def _feasible_candidate_rows(feasibility: pd.DataFrame, candidate: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    feasible_seed_ids = _feasible_seed_ids_by_required_worlds(feasibility, cfg)
    return candidate.loc[candidate["seed_id"].astype(str).isin(feasible_seed_ids)].copy()


def _read_existing_test_split_for_freeze(cfg: ExperimentConfig) -> pd.DataFrame:
    path = Path(cfg.paths.seed_panel_path)
    if not bool(getattr(cfg.panel, "preserve_existing_test_split", False)):
        return pd.DataFrame()
    if not path.exists():
        raise FileNotFoundError(
            f"preserve_existing_test_split=True but no prior seed panel exists at {path}. "
            "Set CFG.panel.preserve_existing_test_split=False only when deliberately starting a new test panel."
        )
    prior = pd.read_parquet(path) if path.suffix.lower() == ".parquet" else pd.read_csv(path)
    require_columns(prior, {"seed_id", "split", "maturity_bucket"}, "prior seed panel for frozen test split")
    prior = prior.loc[prior["split"].astype(str).eq("test")].copy()
    if prior.empty:
        raise RuntimeError("Prior seed panel contains no test split to freeze.")
    return prior


def _merge_frozen_test_split(feasible: pd.DataFrame, cfg: ExperimentConfig) -> tuple[pd.DataFrame, set[str]]:
    prior_test = _read_existing_test_split_for_freeze(cfg)
    if prior_test.empty:
        return pd.DataFrame(), set()
    frozen_ids = set(prior_test["seed_id"].astype(str))
    frozen = feasible.loc[feasible["seed_id"].astype(str).isin(frozen_ids)].copy()
    missing = sorted(frozen_ids - set(frozen["seed_id"].astype(str)))
    if missing:
        raise RuntimeError("Frozen test seeds are not present/feasible under the current candidate universe: " + ", ".join(missing[:20]))
    frozen["split"] = "test"
    frozen["split_label"] = "test"
    frozen["frozen_test_split"] = True
    counts = frozen.groupby("maturity_bucket").size().astype(int).to_dict()
    if len(set(counts.values())) != 1:
        raise RuntimeError(f"Frozen test split is not balanced by maturity bucket: {counts}")
    return frozen, frozen_ids

def _validate_partial_deterministic_selection(seeds: pd.DataFrame, cfg: ExperimentConfig, selected_splits: set[str]) -> None:
    require_columns(seeds, {"seed_id", "maturity_bucket", "split", "panel_stratum"}, "partial deterministic seed selection")
    expected = {internal_split_label(k): int(v) for k, v in cfg.panel.quota_per_bucket.items() if internal_split_label(k) in selected_splits}
    for split, per_bucket in expected.items():
        actual_total = int(seeds["split"].astype(str).eq(split).sum())
        expected_total = int(per_bucket) * len(cfg.hedging.maturity_buckets)
        if actual_total != expected_total:
            raise RuntimeError(f"Partial deterministic panel split {split}: expected {expected_total}, got {actual_total}.")
    for bucket, g in seeds.groupby("maturity_bucket"):
        counts = g["split"].astype(str).value_counts().to_dict()
        for split, expected_n in expected.items():
            if int(counts.get(split, 0)) != int(expected_n):
                raise RuntimeError(f"Partial deterministic panel bucket {bucket}, split {split}: expected {expected_n}, got {counts.get(split, 0)}.")
    if seeds["seed_id"].duplicated().any():
        raise RuntimeError("Duplicate seed_id found in partial deterministic selection.")

def build_deterministic_seed_panel_from_candidate(
    candidate: pd.DataFrame,
    cfg: ExperimentConfig,
    *,
    require_feasibility: bool,
    exclude_seed_ids: set[str] | None = None,
    selected_splits: set[str] | None = None,
) -> pd.DataFrame:
    """Select one feasible row per maturity/stratum without calendar-date split leakage.

    A calendar date may appear in several maturity buckets, but it must belong to one
    and only one of train, validation or test. Candidate rows are considered in a
    deterministic order and conflicting dates are skipped in favour of the next
    candidate within the same maturity/stratum.
    """
    exclude_seed_ids = set() if exclude_seed_ids is None else set(map(str, exclude_seed_ids))
    selected_splits = None if selected_splits is None else set(map(str, selected_splits))
    parts: list[pd.DataFrame] = []
    shortage: list[dict[str, Any]] = []
    date_to_split: dict[pd.Timestamp, str] = {}
    role_map = {i: role for i, role in enumerate(panel_role_sequence(cfg))}

    # Stratum-first iteration aligns the same chronological region across maturities
    # before moving to the next region, reducing avoidable date-role conflicts.
    for stratum in range(int(cfg.panel.strata_per_bucket)):
        role = str(role_map[stratum])
        if selected_splits is not None and role not in selected_splits:
            continue
        for bucket in cfg.hedging.maturity_buckets:
            g = candidate.loc[
                candidate["maturity_bucket"].astype(str).eq(str(bucket))
                & pd.to_numeric(candidate["panel_stratum"], errors="coerce").astype("Int64").eq(stratum)
                & ~candidate["seed_id"].astype(str).isin(exclude_seed_ids)
            ].copy()
            g["_date_key"] = pd.to_datetime(g["date"]).dt.normalize()
            g = g.sort_values(["panel_hash", "date", "seed_id"])
            compatible = g.loc[
                g["_date_key"].map(
                    lambda d: d not in date_to_split or date_to_split[d] == role
                )
            ].copy()

            if compatible.empty:
                conflicting = [
                    f"{pd.Timestamp(d).date()}->{date_to_split.get(d, 'unassigned')}"
                    for d in g["_date_key"].head(10)
                ]
                shortage.append({
                    "bucket": bucket,
                    "panel_stratum": stratum,
                    "split": role,
                    "reason": "no_compatible_candidate",
                    "candidate_rows": int(len(g)),
                    "date_conflicts": "; ".join(conflicting),
                })
                continue

            row = compatible.head(1).drop(columns=["_date_key"])
            chosen_date = pd.to_datetime(row["date"].iloc[0]).normalize()
            date_to_split[chosen_date] = role
            row["split"] = role
            row["split_label"] = canonical_split_label(role)
            parts.append(row)

    if shortage:
        raise RuntimeError(
            "Deterministic panel selection could not satisfy quotas without calendar-date leakage:\n"
            + pd.DataFrame(shortage).to_string(index=False)
        )
    if not parts:
        raise RuntimeError("Deterministic panel selection produced no rows.")

    final = (
        pd.concat(parts, ignore_index=True)
        .sort_values(["maturity_bucket", "split", "panel_stratum"])
        .reset_index(drop=True)
    )
    if "frozen_test_split" not in final.columns:
        final["frozen_test_split"] = False
    final["seed_panel_version"] = SEED_PANEL_VERSION
    final["final_feasible_seed_panel_version"] = (
        FINAL_FEASIBLE_SEED_PANEL_VERSION if require_feasibility else "audit_only_no_feasibility"
    )
    final["simulation_feasibility_version"] = (
        SIMULATION_FEASIBILITY_VERSION if require_feasibility else "not_run_audit_only"
    )
    final["deterministic_panel_hash"] = deterministic_seed_panel_hash(final)
    if selected_splits is None:
        validate_final_feasible_seed_panel(final, cfg, require_feasibility=require_feasibility)
    else:
        _validate_partial_deterministic_selection(final, cfg, selected_splits)
        date_split_counts = (
            final.assign(_date_key=pd.to_datetime(final["date"]).dt.normalize())
            .groupby("_date_key")["split"]
            .nunique()
        )
        if (date_split_counts > 1).any():
            raise RuntimeError("Calendar-date leakage detected in partial deterministic selection.")
    return final

def build_final_feasible_seed_panel(candidate: pd.DataFrame, feasibility: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    require_columns(feasibility, _simulation_feasibility_required_columns(), "simulation feasibility manifest")
    feasible = _feasible_candidate_rows(feasibility, candidate, cfg)
    frozen_test, frozen_ids = _merge_frozen_test_split(feasible, cfg)
    if frozen_test.empty:
        final = build_deterministic_seed_panel_from_candidate(feasible, cfg, require_feasibility=True)
    else:
        train_val = build_deterministic_seed_panel_from_candidate(
            feasible,
            cfg,
            require_feasibility=True,
            exclude_seed_ids=frozen_ids,
            selected_splits={"train", "val"},
        )
        final = pd.concat([train_val, frozen_test], ignore_index=True).sort_values(["maturity_bucket", "split", "panel_stratum", "seed_id"]).reset_index(drop=True)
        final["seed_panel_version"] = SEED_PANEL_VERSION
        final["final_feasible_seed_panel_version"] = FINAL_FEASIBLE_SEED_PANEL_VERSION
        final["simulation_feasibility_version"] = SIMULATION_FEASIBILITY_VERSION
        final["deterministic_panel_hash"] = deterministic_seed_panel_hash(final)
        validate_final_feasible_seed_panel(final, cfg, require_feasibility=True)
    return final


def write_final_seed_panel_outputs(seeds: pd.DataFrame, feasibility: pd.DataFrame, candidate: pd.DataFrame, cfg: ExperimentConfig) -> None:
    cfg.paths.table_dir.mkdir(parents=True, exist_ok=True)
    cfg.paths.panel_selection_dir.mkdir(parents=True, exist_ok=True)
    seeds.to_csv(cfg.paths.table_dir / "deterministic_stratified_seed_panel.csv", index=False)
    seeds.to_csv(cfg.paths.table_dir / "near_atm_straddle_seed_panel.csv", index=False)
    summary = seeds.groupby(["split_label", "maturity_bucket"], dropna=False).size().rename("rows").reset_index()
    summary["deterministic_panel_hash"] = str(seeds["deterministic_panel_hash"].iloc[0])
    summary["deterministic_panel_version"] = cfg.panel.deterministic_panel_version
    summary.to_csv(cfg.paths.table_dir / "deterministic_stratified_seed_panel_summary.csv", index=False)

    pre_path = cfg.paths.table_dir / "seed_panel_exclusion_manifest.csv"
    pre = pd.read_csv(pre_path) if pre_path.exists() else pd.DataFrame()
    sim_excl = build_simulation_exclusion_manifest(candidate, feasibility, cfg)
    pieces = [frame for frame in (pre, sim_excl) if isinstance(frame, pd.DataFrame) and not frame.empty]
    exclusion_manifest = pd.concat(pieces, ignore_index=True) if pieces else pd.DataFrame(columns=[
        "seed_id", "date", "maturity_bucket", "exclusion_stage", "exclusion_reason", "exclusion_detail",
        "pre_selection_or_post_selection", "deterministic_panel_version", "panel_selection_seed", "timestamp_utc",
    ])
    exclusion_manifest.to_csv(cfg.paths.table_dir / "seed_panel_exclusion_manifest.csv", index=False)
    if not exclusion_manifest.empty:
        exclusion_manifest.groupby(["exclusion_stage", "exclusion_reason"], dropna=False).size().rename("rows").reset_index().to_csv(cfg.paths.table_dir / "seed_panel_exclusion_summary.csv", index=False)


def load_or_build_final_feasible_seed_panel(candidate: pd.DataFrame, feasibility: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    deps = [cfg.paths.candidate_seed_panel_path, cfg.paths.simulation_feasibility_manifest_path]
    try:
        seeds = load_dataframe_cache(
            cfg.paths.seed_panel_path,
            "near_atm_straddle_seed_panel",
            "seed_panel",
            cfg,
            required_columns=_final_seed_panel_required_columns(),
            dependencies=deps,
        )
        versions = set(seeds["deterministic_panel_version"].dropna().astype(str).unique())
        feasibility_versions = set(seeds["simulation_feasibility_version"].dropna().astype(str).unique())
        if versions != {cfg.panel.deterministic_panel_version}:
            raise ValueError("Final seed panel cache uses stale deterministic panel version.")
        if feasibility_versions != {SIMULATION_FEASIBILITY_VERSION}:
            raise ValueError("Final seed panel cache uses stale simulation-feasibility version.")
        validate_final_feasible_seed_panel(seeds, cfg, require_feasibility=True)
        print(f"[seed-panel final] loaded deterministic feasible rows={len(seeds)} from cache.", flush=True)
        return seeds
    except (FileNotFoundError, ValueError, RuntimeError) as exc:
        if not cfg.flags.can_rebuild("feasibility"):
            raise RuntimeError(
                "Final feasible seed panel is missing/stale and strict reuse forbids rebuilding. "
                "Set rebuild_from='feasibility', an earlier stage, or use 'auto'."
            ) from exc
        print(f"[seed-panel final rebuild] explicit feasibility scan enabled. Reason: {type(exc).__name__}: {str(exc)[:220]}", flush=True)
    seeds = build_final_feasible_seed_panel(candidate, feasibility, cfg)
    extra = {
        "seed_panel_version": SEED_PANEL_VERSION,
        "final_feasible_seed_panel_version": FINAL_FEASIBLE_SEED_PANEL_VERSION,
        "simulation_feasibility_version": SIMULATION_FEASIBILITY_VERSION,
        "deterministic_panel_version": cfg.panel.deterministic_panel_version,
        "panel_selection_seed": int(cfg.panel.panel_selection_seed),
        "deterministic_panel_hash": str(seeds["deterministic_panel_hash"].iloc[0]),
    }
    write_dataframe_cache(seeds, cfg.paths.seed_panel_path, "near_atm_straddle_seed_panel", "seed_panel", cfg, dependencies=deps, extra=extra)
    write_dataframe_cache(seeds, cfg.paths.final_feasible_seed_panel_path, "final_feasible_near_atm_straddle_seed_panel", "seed_panel", cfg, dependencies=deps, extra=extra)
    write_dataframe_cache(seeds, cfg.paths.deterministic_stratified_seed_panel_path, "deterministic_stratified_seed_panel", "seed_panel", cfg, dependencies=deps, extra=extra)
    write_final_seed_panel_outputs(seeds, feasibility, candidate, cfg)
    print(f"[seed-panel final] deterministic feasible panel rows={len(seeds)}; hash={seeds['deterministic_panel_hash'].iloc[0]}", flush=True)
    return seeds


simulation_feasibility_manifest = load_or_run_simulation_feasibility_scan(candidate_seed_panel, CFG, RESEARCH_ARTIFACTS)
seed_panel = load_or_build_final_feasible_seed_panel(candidate_seed_panel, simulation_feasibility_manifest, CFG)
display(seed_panel.groupby(["split_label", "maturity_bucket"]).size().rename("n_final_feasible_seeds").reset_index())
display(simulation_feasibility_manifest.groupby(["split", "maturity_bucket", "feature_family", "status"]).size().rename("rows").reset_index())


# Expanded-test hard gate: fail immediately if the notebook silently falls back
# to a legacy three-bucket panel or produces any split imbalance.
expected_panel_counts = {
    (split, bucket): int(count)
    for split, count in TARGET_FINAL_QUOTAS.items()
    for bucket in CFG.hedging.maturity_buckets
}
observed_panel_counts = (
    seed_panel
    .groupby(["split", "maturity_bucket"])
    .size()
    .astype(int)
    .to_dict()
)

if observed_panel_counts != expected_panel_counts:
    raise RuntimeError(
        "Expanded-test panel count mismatch. "
        f"Expected={expected_panel_counts}; observed={observed_panel_counts}"
    )

expected_panel_rows = (
    len(CFG.hedging.maturity_buckets)
    * sum(TARGET_FINAL_QUOTAS.values())
)
if len(seed_panel) != expected_panel_rows:
    raise RuntimeError(
        f"Expanded-test panel must contain {expected_panel_rows} rows; "
        f"got {len(seed_panel)}."
    )
if seed_panel["seed_id"].astype(str).duplicated().any():
    raise RuntimeError("Duplicate seed_id detected in expanded-test panel.")
if "frozen_test_split" in seed_panel.columns:
    frozen_mask = seed_panel["frozen_test_split"].fillna(False).astype(bool)
    if frozen_mask.any():
        raise RuntimeError(
            "Legacy frozen-test rows detected despite preserve_existing_test_split=False."
        )

date_split_counts = (
    seed_panel.assign(_date_key=pd.to_datetime(seed_panel["date"]).dt.normalize())
    .groupby("_date_key")["split"]
    .nunique()
)
cross_split_date_overlaps = int((date_split_counts > 1).sum())
if cross_split_date_overlaps:
    raise RuntimeError(
        f"Expanded-test panel contains {cross_split_date_overlaps} calendar date(s) in more than one split."
    )

unique_test_calendar_dates = int(
    pd.to_datetime(
        seed_panel.loc[seed_panel["split"].astype(str).eq("test"), "date"]
    ).dt.normalize().nunique()
)
if unique_test_calendar_dates < 60:
    raise RuntimeError(
        "Expanded-test panel has too few independent test calendar dates: "
        f"{unique_test_calendar_dates} < required minimum 60."
    )

expanded_panel_audit = pd.DataFrame([
    {
        "panel_rows": int(len(seed_panel)),
        "train_rows": int(seed_panel["split"].astype(str).eq("train").sum()),
        "validation_rows": int(seed_panel["split"].astype(str).eq("val").sum()),
        "test_rows": int(seed_panel["split"].astype(str).eq("test").sum()),
        "unique_calendar_dates": int(
            pd.to_datetime(seed_panel["date"]).dt.normalize().nunique()
        ),
        "unique_test_calendar_dates": unique_test_calendar_dates,
        "minimum_required_unique_test_dates": 60,
        "cross_split_date_overlaps": cross_split_date_overlaps,
        "test_dates_per_maturity": 30,
        "status": "pass",
    }
])
expanded_panel_audit.to_csv(
    CFG.paths.table_dir / "expanded_test30_5bucket_panel_audit.csv",
    index=False,
)
display(expanded_panel_audit)

sim_validation_deps = [CFG.paths.seed_panel_path, CFG.paths.lifted_calibration_path, CFG.paths.forward_variance_path]
try:
    simulation_validation = load_dataframe_cache(
        CFG.paths.simulation_validation_path,
        "simulation_validation",
        "simulation_validation",
        CFG,
        dependencies=sim_validation_deps,
    )
    if set(simulation_validation.get("simulation_validation_version", pd.Series(dtype=str)).dropna().astype(str).unique()) != {SIMULATION_VALIDATION_GATE_VERSION}:
        raise ValueError("Simulation validation cache uses a stale gate version.")
except (FileNotFoundError, ValueError) as exc:
    if not CFG.flags.can_rebuild("feasibility"):
        raise RuntimeError(
            "Simulation-validation cache is missing/stale and strict reuse forbids rebuilding. "
            "Set rebuild_from='feasibility', an earlier stage, or use 'auto'."
        ) from exc
    simulation_validation = validate_simulation_engines(seed_panel, CFG, RESEARCH_ARTIFACTS)
    write_dataframe_cache(
        simulation_validation,
        CFG.paths.simulation_validation_path,
        "simulation_validation",
        "simulation_validation",
        CFG,
        dependencies=sim_validation_deps,
    )

display(simulation_validation)
simulation_validation_candidate_audit_path = CFG.paths.table_dir / "simulation_validation_candidate_audit.csv"
if simulation_validation_candidate_audit_path.exists():
    display(pd.read_csv(simulation_validation_candidate_audit_path))
failed = simulation_validation.loc[~simulation_validation["pass"].fillna(False)].copy()
if not failed.empty:
    raise RuntimeError(
        "Lifted simulation validation gates failed; see displayed/written simulation_validation table.\n"
        + failed.to_string(index=False)
    )


[feasibility] loaded manifest rows=1690 from cache.
[seed-panel final] loaded deterministic feasible rows=290 from cache.


,split_label,maturity_bucket,n_final_feasible_seeds
0,test,1M,30
1,test,1W,30
2,test,2M,30
3,test,2W,30
4,test,3M,30
5,train,1M,20
6,train,1W,20
7,train,2M,20
8,train,2W,20
9,train,3M,20


,split,maturity_bucket,feature_family,status,rows
0,test,1M,heston_world,pass,90
1,test,1M,lifted_world,fail,6
2,test,1M,lifted_world,pass,84
3,test,1W,heston_world,pass,83
4,test,1W,lifted_world,pass,83
5,test,2M,heston_world,pass,90
6,test,2M,lifted_world,fail,1
7,test,2M,lifted_world,pass,89
8,test,2W,heston_world,pass,78
9,test,2W,lifted_world,pass,78


,panel_rows,train_rows,validation_rows,test_rows,unique_calendar_dates,unique_test_calendar_dates,minimum_required_unique_test_dates,cross_split_date_overlaps,test_dates_per_maturity,status
0,290,100,40,150,283,145,60,0,30,pass


,date,seed_id,maturity_bucket,simulation_validation_version,simulation_failure_reason,max_mean_variance_error,mean_variance_tolerance,martingale_relative_error,martingale_tolerance,timestep_convergence_relative_error,timestep_convergence_tolerance,min_variance,max_reconstruction_error,min_integrated_variance_increment,mean_integrated_variance_increment,min_cumulant_second_derivative,max_cumulant_imaginary_part,cumulant_states_checked,one_factor_limit_max_mean_variance_error,one_factor_limit_martingale_error,one_factor_limit_min_variance,max_valid_state_resampling_attempts,pass,simulation_validation_candidate_rank,simulation_validation_rejected_candidates_before_accept,simulation_validation_candidate_source
0,2010-02-12,2010-02-12_1W_1075,1W,lifted_simulation_state_validation_v3_auto_fea...,,0.000138,0.005,0.000043,0.02,0.000501,0.1,0.000041,1.110223e-16,6.433951e-08,0.000088,0.000002,0.0,2560.0,0.000444,0.000064,3.422155e-05,0,True,1,0,same_bucket_non_val_split
1,2010-03-12,2010-03-12_2W_1150,2W,lifted_simulation_state_validation_v3_auto_fea...,,0.000118,0.005,0.000085,0.02,0.005836,0.1,0.000018,5.551115e-17,2.753380e-08,0.000074,0.000001,0.0,2560.0,0.000951,0.000508,4.127538e-08,0,True,1,0,same_bucket_non_val_split
2,2010-05-17,2010-05-17_1M_1125,1M,lifted_simulation_state_validation_v3_auto_fea...,,0.000595,0.005,0.000224,0.02,0.009644,0.1,0.000057,2.220446e-16,9.954914e-08,0.000227,0.000005,0.0,2560.0,0.001390,0.000959,2.273740e-04,0,True,1,0,same_bucket_non_val_split
3,2010-01-27,2010-01-27_2M_1100,2M,lifted_simulation_state_validation_v3_auto_fea...,,0.000706,0.005,0.000034,0.02,0.003785,0.1,0.000014,2.220446e-16,2.075365e-08,0.000152,0.000001,0.0,2560.0,0.001433,0.001705,4.965166e-05,0,True,1,0,same_bucket_non_val_split
4,2010-02-09,2010-02-09_3M_1060,3M,lifted_simulation_state_validation_v3_auto_fea...,,0.000773,0.005,0.000748,0.02,0.040247,0.1,0.000020,3.330669e-16,3.043237e-08,0.000181,0.000001,0.0,2560.0,0.001439,0.001359,1.278681e-04,0,True,1,0,same_bucket_non_val_split


# 7. Historical SPX market episodes

This section applies the chronology-clean historical market construction from the completed simulation-to-market notebook. The split is fixed:

- 2018–2019: market adaptation;
- 2020: validation and checkpoint selection;
- 2021–2024: common held-out evaluation.

The 2021–2024 period is chronologically separated but was inspected during earlier project development, so it is not described as a pristine analyst-blind test.

In [39]:
@dataclass(frozen=True)
class MarketCohortConfig:
    outputs_dir_name: str = 'outputs_observable_runtime'
    results_subdir: str = 'historical_market_cohorts'
    raw_option_file_override: str | None = None
    raw_crsp_file_override: str | None = None
    force_rebuild: bool = False
    smoke_test: bool = False

    protocol_version: str = 'observable_h025_definitive_v6'
    feature_schema_version: str = 'observable_surface_features_v6_next_node_execution'
    accounting_version: str = 'premium_normalised_short_straddle_bid_ask_next_node_v6'
    random_seed: int = 20260710
    training_seeds: tuple[int, ...] = (20260710, 20260711, 20260712)
    calendar_days_per_year: float = 365.25
    trading_days_per_year: int = 252

    maturity_buckets: dict[str, tuple[int, int, int]] = field(default_factory=lambda: {
        '1W': (9, 7, 10), '2W': (16, 11, 21), '1M': (30, 22, 45),
        '2M': (60, 46, 75), '3M': (90, 76, 120),
    })
    forward_horizons_days: tuple[int, ...] = (7, 14, 30, 60, 90)

    simulation_seed_start: str = '2010-01-01'
    simulation_seed_end: str = '2017-12-31'
    market_finetune_start: str = '2018-01-01'
    market_finetune_end: str = '2019-12-31'
    market_validation_start: str = '2020-01-01'
    market_validation_end: str = '2020-12-31'
    market_test_start: str = '2021-01-01'
    market_test_end: str = '2024-12-31'
    market_confirmatory_start: str = '2025-01-01'
    market_confirmatory_end: str = '2025-12-31'

    preferred_entry_weekday: int = 2
    entry_tolerance_calendar_days: int = 2
    minimum_close_dte: int = 1
    maximum_close_dte: int = 3
    minimum_path_observations: dict[str, int] = field(default_factory=lambda: {
        '1W': 3, '2W': 5, '1M': 10, '2M': 20, '3M': 30,
    })
    maximum_calendar_gap_days: int = 7
    entry_max_abs_log_forward_moneyness: float = 0.05
    tracking_max_abs_log_spot_moneyness: float = 0.45
    minimum_bid: float = 0.05
    maximum_relative_spread: float = 0.50
    minimum_open_interest: int = 1

    transaction_cost_bps: tuple[float, ...] = (1.0, 5.0, 10.0)
    primary_transaction_cost_bps: float = 1.0
    no_trade_band_candidates: tuple[float, ...] = (0.0, 0.01, 0.025, 0.05, 0.075, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.75, 1.00)

    adjustment_limit: float = 0.35
    hedge_limit: float = 2.0
    hidden_size: int = 64
    mlp_depth: int = 2

    simulation_train_episodes: int = 3000
    simulation_validation_episodes: int = 500
    simulation_test_episodes: int = 800
    common_pretrain_steps: int = 1800
    domain_stage_steps: int = 1000
    hybrid_head_steps: int = 250
    hybrid_full_steps: int = 750
    market_only_steps: int = 2800
    batch_episodes: int = 16
    validation_interval: int = 50
    early_stopping_patience: int = 12
    validation_max_episodes: int | None = None
    learning_rate_pretrain: float = 3e-4
    learning_rate_domain_stage: float = 1e-4
    learning_rate_hybrid_full: float = 5e-5
    weight_decay: float = 1e-5
    gradient_clip_norm: float = 1.0
    cvar_alpha: float = 0.05
    cvar_weight: float = 0.05
    adjustment_l2_weight: float = 1e-3
    l2_sp_lambda: float = 1e-4

    bootstrap_repetitions: int = 10_000
    bootstrap_block_weeks: int = 13
    bootstrap_block_sensitivity: tuple[int, ...] = (4, 8, 13, 26)
    bootstrap_seed: int = 20260711

    max_cohorts_per_bucket_per_split: int | None = None
    minimum_cohorts_by_role: dict[str, int] = field(default_factory=lambda: {'finetune': 20, 'validation': 10, 'test': 40, 'confirmatory': 0})

MKT_CFG = MarketCohortConfig(
    outputs_dir_name=str(CFG.paths.output_dir),
    results_subdir="historical_market_cohorts",
    force_rebuild=EXECUTION.force_rebuild("hedging_panel"),
    smoke_test=IS_SMOKE,
    protocol_version=FINAL_PROTOCOL_VERSION,
    feature_schema_version="observable_surface_history_v6_strict_lag_next_node",
    accounting_version="premium_normalised_short_straddle_bid_ask_next_node_v6",
    training_seeds=tuple(RUN_PROFILE["training_seeds"]),
    transaction_cost_bps=tuple(RUN_PROFILE["transaction_cost_bps"]),
    primary_transaction_cost_bps=1.0,
    simulation_train_episodes=40 if IS_SMOKE else 3000,
    simulation_validation_episodes=12 if IS_SMOKE else 500,
    simulation_test_episodes=12 if IS_SMOKE else 800,
    common_pretrain_steps=int(RUN_PROFILE["common_pretrain_steps"]),
    hybrid_head_steps=int(RUN_PROFILE["hybrid_head_steps"]),
    hybrid_full_steps=int(RUN_PROFILE["hybrid_full_steps"]),
    batch_episodes=int(RUN_PROFILE["batch_episodes"]),
    validation_interval=int(RUN_PROFILE["validation_interval"]),
    early_stopping_patience=int(RUN_PROFILE["early_stopping_patience"]),
    bootstrap_repetitions=int(RUN_PROFILE["bootstrap_repetitions"]),
    bootstrap_block_sensitivity=tuple(RUN_PROFILE["bootstrap_sensitivity_weeks"]),
    max_cohorts_per_bucket_per_split=RUN_PROFILE["market_max_cohorts_per_bucket_per_split"],
    minimum_cohorts_by_role=(
        {"finetune": 1, "validation": 1, "test": 1, "confirmatory": 0}
        if IS_SMOKE else
        {"finetune": 20, "validation": 10, "test": 40, "confirmatory": 0}
    ),
)

def seed_everything(seed: int) -> None:
    random.seed(int(seed)); np.random.seed(int(seed)); torch.manual_seed(int(seed))
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(int(seed))

seed_everything(MKT_CFG.random_seed)
_market_protocol_payload = asdict(MKT_CFG)
_market_protocol_payload.pop("force_rebuild", None)  # execution control, not scientific design
MARKET_PROTOCOL_HASH = hashlib.sha256(
    json.dumps(_market_protocol_payload, sort_keys=True, default=str).encode()
).hexdigest()
display(pd.DataFrame([asdict(MKT_CFG)]).T.rename(columns={0:'value'}))
print('Protocol hash:',MARKET_PROTOCOL_HASH)


,value
outputs_dir_name,"C:\Users\lhkke\OneDrive - Queen Mary, Universi..."
results_subdir,historical_market_cohorts
raw_option_file_override,None
raw_crsp_file_override,None
force_rebuild,False
...,...
bootstrap_block_weeks,13
bootstrap_block_sensitivity,"(4, 8, 13, 26)"
bootstrap_seed,20260711
max_cohorts_per_bucket_per_split,None


Protocol hash: 61941012eedd3c18ab1973b7e7d3c4bca838b4eccb4fe43051b738260bdc6213


In [40]:

FINAL_RESULTS_DIR = CFG.paths.output_dir
FINAL_CACHE_DIR = FINAL_RESULTS_DIR / "observable_market_cache"
FINAL_CHECKPOINT_DIR = FINAL_RESULTS_DIR / "checkpoints"
FINAL_TABLE_DIR = FINAL_RESULTS_DIR / "tables"
FINAL_FIGURE_DIR = FINAL_RESULTS_DIR / "figures"
FINAL_LOG_DIR = FINAL_RESULTS_DIR / "logs"
for _directory in (FINAL_RESULTS_DIR, FINAL_CACHE_DIR, FINAL_CHECKPOINT_DIR, FINAL_TABLE_DIR, FINAL_FIGURE_DIR, FINAL_LOG_DIR):
    _directory.mkdir(parents=True, exist_ok=True)

RAW_OPTION_PATH = CFG.paths.raw_option_path
RAW_CRSP_PATH = CFG.paths.raw_crsp_path
print("Raw OptionMetrics:", RAW_OPTION_PATH)
print("Raw CRSP:", RAW_CRSP_PATH)

Raw OptionMetrics: C:\Users\lhkke\OneDrive - Queen Mary, University of London\Documents\Dissertation\Deep-Hedging-under-Lifted-Heston-MSc-Dissertation-\OptionMetrics ATM IV 2010 - 2025.parquet
Raw CRSP: C:\Users\lhkke\OneDrive - Queen Mary, University of London\Documents\Dissertation\Deep-Hedging-under-Lifted-Heston-MSc-Dissertation-\CRSP S&P 2010 - 2024.parquet


In [41]:
OPTION_ALIASES = {
    'date': ('date',), 'exdate': ('exdate','expiry','expiration'),
    'option_type': ('cp_flag','option_type','type'), 'strike': ('strike_price','strike'),
    'bid': ('best_bid','bid'), 'ask': ('best_offer','ask'),
    'iv': ('impl_volatility','implied_volatility','iv'),
    'volume': ('volume',), 'open_interest': ('open_interest',),
    'optionid': ('optionid',), 'ticker': ('ticker',), 'secid': ('secid',),
}
CRSP_ALIASES = {'date': ('caldt','date'), 'spot': ('spindx','spx','index_level','close','close_price')}


def resolve_aliases(available: Iterable[str], aliases: dict[str, tuple[str, ...]], required_cols: Iterable[str]) -> dict[str, str]:
    available_set = set(map(str, available))
    mapping: dict[str, str] = {}
    for canonical, choices in aliases.items():
        for choice in choices:
            if choice in available_set:
                mapping[canonical] = choice
                break
    missing = sorted(set(required_cols) - set(mapping))
    if missing:
        raise KeyError(f'Missing required columns {missing}; available={sorted(available_set)[:80]}')
    return mapping


def parse_date_series(values: pd.Series) -> pd.Series:
    series = pd.Series(values, copy=False)
    numeric = pd.to_numeric(series, errors='coerce')
    finite = numeric.dropna()
    if len(finite) and 19_000_000 <= float(finite.median()) <= 21_001_231:
        return pd.to_datetime(numeric.round().astype('Int64').astype(str), format='%Y%m%d', errors='coerce')
    return pd.to_datetime(series, errors='coerce')

import pyarrow as pa
import pyarrow.dataset as ds

def date_filter_scalar(field: pa.Field, date: pd.Timestamp):
    if pa.types.is_integer(field.type):
        return int(date.strftime('%Y%m%d'))
    if pa.types.is_string(field.type) or pa.types.is_large_string(field.type):
        return date.strftime('%Y-%m-%d')
    if pa.types.is_date(field.type):
        return date.date()
    return date.to_pydatetime()


def load_optionmetrics_slice(path: Path, start: str, end: str, max_dte: int) -> pd.DataFrame:
    start_ts, end_ts = pd.Timestamp(start), pd.Timestamp(end)
    max_expiry = end_ts + pd.Timedelta(days=max_dte + 7)
    if path.suffix.lower() not in {'.parquet','.pq'}:
        raise ValueError('This notebook expects a Parquet OptionMetrics extract for filtered loading.')
    dataset = ds.dataset(str(path), format='parquet')
    mapping = resolve_aliases(dataset.schema.names, OPTION_ALIASES, ('date','exdate','option_type','strike','bid','ask'))
    columns = sorted(set(mapping.values()))
    date_field = dataset.schema.field(mapping['date'])
    exdate_field = dataset.schema.field(mapping['exdate'])
    filt = (
        (ds.field(mapping['date']) >= date_filter_scalar(date_field, start_ts))
        & (ds.field(mapping['date']) <= date_filter_scalar(date_field, end_ts))
        & (ds.field(mapping['exdate']) >= date_filter_scalar(exdate_field, start_ts))
        & (ds.field(mapping['exdate']) <= date_filter_scalar(exdate_field, max_expiry))
    )
    raw = dataset.to_table(columns=columns, filter=filt).to_pandas(split_blocks=True, self_destruct=True)
    return raw.rename(columns={source: canonical for canonical, source in mapping.items()})


def load_crsp_spot(path: Path, start: str, end: str) -> pd.DataFrame:
    if path.suffix.lower() in {'.parquet','.pq'}:
        raw = pd.read_parquet(path)
    else:
        raw = pd.read_csv(path)
    mapping = resolve_aliases(raw.columns, CRSP_ALIASES, ('date','spot'))
    frame = raw.rename(columns={source: canonical for canonical, source in mapping.items()})[['date','spot']].copy()
    frame['date'] = parse_date_series(frame['date'])
    frame['spot'] = pd.to_numeric(frame['spot'], errors='coerce').abs()
    frame = frame.loc[frame['date'].between(pd.Timestamp(start), pd.Timestamp(end)) & frame['spot'].gt(0)]
    return frame.sort_values('date').drop_duplicates('date').reset_index(drop=True)


def normalise_option_quotes(raw: pd.DataFrame) -> pd.DataFrame:
    frame = raw.copy()
    frame['date'] = parse_date_series(frame['date'])
    frame['exdate'] = parse_date_series(frame['exdate'])
    for col in ['strike','bid','ask','iv','volume','open_interest','optionid','secid']:
        if col not in frame:
            frame[col] = np.nan
        frame[col] = pd.to_numeric(frame[col], errors='coerce')
    finite_strikes = frame.loc[np.isfinite(frame['strike']), 'strike']
    if len(finite_strikes) and float(finite_strikes.median()) > 20_000:
        frame['strike'] /= 1000.0
    frame['option_type'] = frame['option_type'].astype(str).str.strip().str.upper().str[0]
    if 'ticker' in frame and frame['ticker'].notna().any():
        mask = frame['ticker'].astype(str).str.upper().str.contains('SPX', na=False)
        if mask.any(): frame = frame.loc[mask].copy()
    frame['dte'] = (frame['exdate'] - frame['date']).dt.days.astype(float)
    frame['T'] = frame['dte'] / MKT_CFG.calendar_days_per_year
    frame['mid'] = 0.5 * (frame['bid'] + frame['ask'])
    frame['spread'] = frame['ask'] - frame['bid']
    frame['relative_spread'] = frame['spread'] / frame['mid'].clip(lower=1e-12)
    keep = (
        frame['dte'].between(MKT_CFG.minimum_close_dte, max(v[2] for v in MKT_CFG.maturity_buckets.values()))
        & frame['option_type'].isin(['C','P']) & frame['strike'].gt(0)
        & frame['bid'].ge(MKT_CFG.minimum_bid) & frame['ask'].ge(frame['bid'])
        & frame['mid'].gt(0) & frame['relative_spread'].le(MKT_CFG.maximum_relative_spread)
    )
    frame = frame.loc[keep].copy()
    frame['open_interest'] = frame['open_interest'].fillna(0)
    frame['volume'] = frame['volume'].fillna(0)
    return frame.sort_values(
        ['date','exdate','strike','option_type','relative_spread','open_interest','volume'],
        ascending=[True,True,True,True,True,False,False],
    ).drop_duplicates(['date','exdate','strike','option_type']).reset_index(drop=True)


def pair_call_put_quotes(quotes: pd.DataFrame, spot: pd.DataFrame, daily_rates: pd.DataFrame) -> pd.DataFrame:
    keys = ['date','exdate','strike']
    values = ['bid','ask','mid','iv','volume','open_interest','optionid','relative_spread']
    calls = quotes.loc[quotes['option_type'].eq('C'), keys + values].rename(columns={c:f'call_{c}' for c in values})
    puts = quotes.loc[quotes['option_type'].eq('P'), keys + values].rename(columns={c:f'put_{c}' for c in values})
    paired = calls.merge(puts, on=keys, how='inner', validate='one_to_one')
    paired = paired.merge(spot, on='date', how='inner', validate='many_to_one')
    paired = paired.merge(daily_rates, on='date', how='left', validate='many_to_one')
    paired[['r','q']] = paired[['r','q']].ffill().fillna({'r':0.01,'q':0.0})
    paired['dte'] = (paired['exdate'] - paired['date']).dt.days.astype(float)
    paired['T'] = paired['dte'] / MKT_CFG.calendar_days_per_year
    paired['forward'] = paired['spot'] * np.exp((paired['r'] - paired['q']) * paired['T'])
    paired['log_forward_moneyness'] = np.log(paired['strike'] / paired['forward'])
    paired['log_spot_moneyness'] = np.log(paired['spot'] / paired['strike'])
    paired['straddle_bid'] = paired['call_bid'] + paired['put_bid']
    paired['straddle_ask'] = paired['call_ask'] + paired['put_ask']
    paired['straddle_mid'] = paired['call_mid'] + paired['put_mid']
    paired['straddle_relative_spread'] = (paired['straddle_ask'] - paired['straddle_bid']) / paired['straddle_mid'].clip(lower=1e-12)
    paired['contract_iv'] = paired[['call_iv','put_iv']].mean(axis=1, skipna=True)
    paired['contract_open_interest'] = paired[['call_open_interest','put_open_interest']].fillna(0).min(axis=1)
    paired['contract_volume'] = paired[['call_volume','put_volume']].fillna(0).sum(axis=1)
    finite = paired['spot'].gt(0) & paired['straddle_bid'].gt(0) & paired['straddle_ask'].ge(paired['straddle_bid'])
    return paired.loc[finite].sort_values(['date','exdate','strike']).reset_index(drop=True)

In [42]:
def _interp_total_variance(T: np.ndarray, iv: np.ndarray, target_days: int) -> float:
    valid = np.isfinite(T) & np.isfinite(iv) & (T > 0) & (iv > 0)
    T, iv = T[valid], iv[valid]
    if len(T) == 0: return np.nan
    order = np.argsort(T); T, iv = T[order], iv[order]
    unique_T, idx = np.unique(T, return_index=True)
    total_var = (iv[idx] ** 2) * unique_T
    target_T = target_days / MKT_CFG.calendar_days_per_year
    w = np.interp(target_T, unique_T, total_var, left=total_var[0], right=total_var[-1])
    return float(max(w / max(target_T, 1e-8), 1e-8))


def build_daily_surface_state(paired: pd.DataFrame, spot: pd.DataFrame) -> pd.DataFrame:
    atm_idx = paired.groupby(['date','exdate'], sort=False)['log_forward_moneyness'].apply(lambda x: x.abs().idxmin())
    atm_points = paired.loc[atm_idx.to_numpy(), ['date','exdate','T','dte','contract_iv']].copy()
    rows = []
    for date, group in atm_points.groupby('date', sort=True):
        row: dict[str, Any] = {'date': pd.Timestamp(date)}
        for days in MKT_CFG.forward_horizons_days:
            label = {7:'1w',14:'2w',30:'1m',60:'2m',90:'3m'}[days]
            row[f'atm_var_{label}'] = _interp_total_variance(group['T'].to_numpy(float), group['contract_iv'].to_numpy(float), days)
        # Near-30-day smile regression for observable ATM skew.
        nearest_expiry = group.iloc[(group['dte'] - 30).abs().argsort()[:1]]['exdate']
        smile = paired.loc[
            paired['date'].eq(date) & paired['exdate'].isin(nearest_expiry)
            & paired['log_forward_moneyness'].abs().le(0.12)
            & paired['contract_iv'].notna()
        ]
        if len(smile) >= 5:
            x = smile['log_forward_moneyness'].to_numpy(float)
            y = smile['contract_iv'].to_numpy(float)
            coef = np.polyfit(x, y, deg=2)
            row['atm_skew'] = float(coef[1])
        else:
            row['atm_skew'] = np.nan
        rows.append(row)
    state = pd.DataFrame(rows).sort_values('date')
    state['term_slope'] = state['atm_var_3m'] - state['atm_var_1w']
    state['term_curvature'] = state['atm_var_1m'] - 0.5 * (state['atm_var_1w'] + state['atm_var_3m'])
    state['iv_change_1d'] = state['atm_var_1m'].diff()
    state['iv_change_5d'] = state['atm_var_1m'].diff(5)

    spot_state = spot.sort_values('date').copy()
    spot_state['log_return'] = np.log(spot_state['spot']).diff()
    spot_state['rv_5d'] = spot_state['log_return'].rolling(5, min_periods=2).var() * MKT_CFG.trading_days_per_year
    spot_state['rv_20d'] = spot_state['log_return'].rolling(20, min_periods=5).var() * MKT_CFG.trading_days_per_year
    state = spot_state.merge(state, on='date', how='left')
    state = state.replace([np.inf,-np.inf], np.nan).sort_values('date')
    variance_cols = [f'atm_var_{x}' for x in ['1w','2w','1m','2m','3m']]
    state[variance_cols] = state[variance_cols].ffill()
    state['atm_skew'] = state['atm_skew'].ffill().fillna(0.0)
    state['term_slope'] = state['term_slope'].ffill().fillna(0.0)
    state['term_curvature'] = state['term_curvature'].ffill().fillna(0.0)
    state['iv_change_1d'] = state['iv_change_1d'].fillna(0.0)
    state['iv_change_5d'] = state['iv_change_5d'].fillna(0.0)
    # Realised variance uses only past returns; initial observations fall back to same-day 1M implied variance.
    state['rv_5d'] = state['rv_5d'].fillna(state['atm_var_1m'])
    state['rv_20d'] = state['rv_20d'].fillna(state['atm_var_1m'])
    state['log_return'] = state['log_return'].fillna(0.0)
    return state.dropna(subset=variance_cols)


def add_surface_state(paired: pd.DataFrame, daily_state: pd.DataFrame) -> pd.DataFrame:
    frame = paired.merge(daily_state.drop(columns='spot'), on='date', how='left', validate='many_to_one')
    atm_idx = frame.groupby(['date','exdate'], sort=False)['log_forward_moneyness'].apply(lambda x: x.abs().idxmin())
    atm_same = frame.loc[atm_idx.to_numpy(), ['date','exdate','contract_iv']].rename(columns={'contract_iv':'atm_iv_same_expiry'})
    frame = frame.merge(atm_same, on=['date','exdate'], how='left', validate='many_to_one')
    frame['contract_minus_atm_iv'] = frame['contract_iv'] - frame['atm_iv_same_expiry']
    return frame

# Use the validated global surface already loaded or constructed earlier.
# Do not rely on the obsolete local variable `required`.

if (
    "surface_panel_global_diagnostic" in globals()
    and isinstance(surface_panel_global_diagnostic, pd.DataFrame)
    and not surface_panel_global_diagnostic.empty
):
    cached_global = surface_panel_global_diagnostic.copy()
    global_rate_source = "surface_panel_global_diagnostic"

elif (
    "spx_global_diagnostic_panel" in globals()
    and isinstance(spx_global_diagnostic_panel, pd.DataFrame)
    and not spx_global_diagnostic_panel.empty
):
    cached_global = spx_global_diagnostic_panel.copy()
    global_rate_source = "spx_global_diagnostic_panel"

else:
    global_panel_path = Path(
        SPX_HEDGING_OUTPUTS["global_diagnostic_panel"]
    )

    if not global_panel_path.exists():
        raise FileNotFoundError(
            "No in-memory global diagnostic panel is available and the "
            f"materialised panel was not found at: {global_panel_path}. "
            "Run the surface and SPX hedging-panel sections first."
        )

    cached_global = _read_spx_panel_output(global_panel_path)
    global_rate_source = str(global_panel_path)


required_rate_columns = {"date", "r", "q"}
missing_rate_columns = sorted(
    required_rate_columns.difference(cached_global.columns)
)

if missing_rate_columns:
    raise KeyError(
        "The global surface panel cannot provide the historical rate series. "
        f"Missing columns: {missing_rate_columns}. "
        f"Available columns include: {sorted(cached_global.columns)[:80]}"
    )


cached_global["date"] = (
    pd.to_datetime(cached_global["date"], errors="coerce")
    .dt.normalize()
)

cached_global["r"] = pd.to_numeric(
    cached_global["r"],
    errors="coerce",
)

cached_global["q"] = pd.to_numeric(
    cached_global["q"],
    errors="coerce",
)

cached_global = cached_global.dropna(
    subset=["date", "r", "q"]
)

if cached_global.empty:
    raise RuntimeError(
        "The global surface panel contains no valid date/r/q observations."
    )


daily_rates = (
    cached_global
    .groupby("date", as_index=False)
    .agg(
        r=("r", "median"),
        q=("q", "median"),
    )
    .sort_values("date")
    .reset_index(drop=True)
)

print(
    f"Historical rate source: {global_rate_source}; "
    f"{len(daily_rates):,} daily observations from "
    f"{daily_rates['date'].min().date()} to "
    f"{daily_rates['date'].max().date()}."
)

raw_start = MKT_CFG.market_finetune_start
raw_end = MKT_CFG.market_test_end
raw_options = load_optionmetrics_slice(RAW_OPTION_PATH, raw_start, raw_end, max(v[2] for v in MKT_CFG.maturity_buckets.values()))
crsp_spot = load_crsp_spot(RAW_CRSP_PATH, raw_start, raw_end)
quotes = normalise_option_quotes(raw_options)
paired_quotes = pair_call_put_quotes(quotes, crsp_spot, daily_rates)
daily_surface_state = build_daily_surface_state(paired_quotes, crsp_spot)
paired_quotes = add_surface_state(paired_quotes, daily_surface_state)

market_data_audit = pd.DataFrame([{
    'raw_option_rows': len(raw_options), 'valid_leg_quotes': len(quotes), 'paired_rows': len(paired_quotes),
    'dates': paired_quotes['date'].nunique(), 'date_min': paired_quotes['date'].min(), 'date_max': paired_quotes['date'].max(),
    'common_observable_surface_features': True,
}])
display(market_data_audit)
market_data_audit.to_csv(FINAL_TABLE_DIR / 'market_data_audit.csv', index=False)

Historical rate source: surface_panel_global_diagnostic; 2,869 daily observations from 2010-01-04 to 2024-12-31.


,raw_option_rows,valid_leg_quotes,paired_rows,dates,date_min,date_max,common_observable_surface_features
0,31030477,16758940,7556874,1761,2018-01-02,2024-12-31,True


In [43]:
def assign_maturity_bucket(dte: pd.Series) -> pd.Series:
    result = pd.Series(pd.NA, index=dte.index, dtype='object')
    for label, (_, lo, hi) in MKT_CFG.maturity_buckets.items():
        result.loc[dte.between(lo, hi, inclusive='both')] = label
    return result


def weekly_entry_dates(available_dates: Iterable[pd.Timestamp], start: str, end: str) -> list[pd.Timestamp]:
    dates = pd.DataFrame({'date': pd.to_datetime(sorted(set(available_dates)))})
    dates = dates.loc[dates['date'].between(pd.Timestamp(start), pd.Timestamp(end))].copy()
    dates['week_start'] = dates['date'] - pd.to_timedelta(dates['date'].dt.weekday, unit='D')
    dates['target'] = dates['week_start'] + pd.Timedelta(days=MKT_CFG.preferred_entry_weekday)
    dates['distance'] = (dates['date'] - dates['target']).abs().dt.days
    chosen = dates.sort_values(['week_start','distance','date']).groupby('week_start', as_index=False).first()
    return chosen.loc[chosen['distance'].le(MKT_CFG.entry_tolerance_calendar_days), 'date'].tolist()


def chronology_role(date: pd.Timestamp) -> str | None:
    d = pd.Timestamp(date)
    if pd.Timestamp(MKT_CFG.market_finetune_start) <= d <= pd.Timestamp(MKT_CFG.market_finetune_end): return 'finetune'
    if pd.Timestamp(MKT_CFG.market_validation_start) <= d <= pd.Timestamp(MKT_CFG.market_validation_end): return 'validation'
    if pd.Timestamp(MKT_CFG.market_test_start) <= d <= pd.Timestamp(MKT_CFG.market_test_end): return 'test'
    if pd.Timestamp(MKT_CFG.market_confirmatory_start) <= d <= pd.Timestamp(MKT_CFG.market_confirmatory_end): return 'confirmatory'
    return None


def select_contract_paths(frame: pd.DataFrame) -> tuple[pd.DataFrame, dict[str,pd.DataFrame], pd.DataFrame]:
    frame = frame.copy()
    frame['maturity_bucket'] = assign_maturity_bucket(frame['dte'])
    entries = weekly_entry_dates(frame['date'].unique(), MKT_CFG.market_finetune_start, MKT_CFG.market_confirmatory_end)
    paths: dict[str,pd.DataFrame] = {}
    rows, exclusions = [], []
    accepted: dict[tuple[str,str], int] = {}
    for bucket, (target_dte, lo, hi) in MKT_CFG.maturity_buckets.items():
        for entry_date in entries:
            role = chronology_role(entry_date)
            if role is None: continue
            key = (role, bucket)
            if MKT_CFG.max_cohorts_per_bucket_per_split is not None and accepted.get(key,0) >= MKT_CFG.max_cohorts_per_bucket_per_split:
                continue
            candidates = frame.loc[
                frame['date'].eq(entry_date) & frame['dte'].between(lo,hi,inclusive='both')
                & frame['log_forward_moneyness'].abs().le(MKT_CFG.entry_max_abs_log_forward_moneyness)
                & frame['contract_open_interest'].fillna(0).ge(MKT_CFG.minimum_open_interest)
                & frame['straddle_relative_spread'].le(MKT_CFG.maximum_relative_spread)
            ].copy()
            if candidates.empty:
                exclusions.append({'entry_date':entry_date,'role':role,'maturity_bucket':bucket,'reason':'no_liquid_entry'})
                continue
            candidates['rank_dte'] = (candidates['dte'] - target_dte).abs()
            candidates['rank_m'] = candidates['log_forward_moneyness'].abs()
            selected = candidates.sort_values(
                ['rank_dte','rank_m','straddle_relative_spread','contract_open_interest','contract_volume'],
                ascending=[True,True,True,False,False],
            ).iloc[0]
            path = frame.loc[
                frame['exdate'].eq(selected['exdate'])
                & np.isclose(frame['strike'], float(selected['strike']), rtol=0, atol=1e-8)
                & frame['date'].between(entry_date, selected['exdate'])
                & frame['dte'].ge(MKT_CFG.minimum_close_dte)
                & frame['log_spot_moneyness'].abs().le(MKT_CFG.tracking_max_abs_log_spot_moneyness)
            ].sort_values('date').drop_duplicates('date').copy()
            reason = None
            if path.empty or pd.Timestamp(path.iloc[0]['date']) != pd.Timestamp(entry_date): reason = 'entry_missing'
            elif len(path) < MKT_CFG.minimum_path_observations[bucket]: reason = 'insufficient_observations'
            elif float(path.iloc[-1]['dte']) > MKT_CFG.maximum_close_dte: reason = 'no_1_to_3_dte_close'
            elif path['date'].diff().dropna().dt.days.max() > MKT_CFG.maximum_calendar_gap_days: reason = 'quote_gap'
            required_cols = ['straddle_bid','straddle_ask','straddle_mid','spot','contract_iv'] + [f'atm_var_{x}' for x in ['1w','2w','1m','2m','3m']]
            if reason is None and not np.isfinite(path[required_cols].to_numpy(float)).all(): reason = 'non_finite_state'
            if reason:
                exclusions.append({'entry_date':entry_date,'role':role,'maturity_bucket':bucket,'reason':reason})
                continue
            cohort_id = f'{pd.Timestamp(entry_date).date()}_{bucket}_{pd.Timestamp(selected.exdate).date()}_{float(selected.strike):.6f}'
            path['cohort_id'] = cohort_id
            path['entry_date'] = pd.Timestamp(entry_date)
            path['entry_dte'] = float(selected['dte'])
            path['maturity_bucket'] = bucket
            path['chronology_role'] = role
            path['tau_fraction'] = path['dte'] / max(float(selected['dte']), 1.0)
            paths[cohort_id] = path.reset_index(drop=True)
            rows.append({
                'cohort_id':cohort_id,'entry_date':pd.Timestamp(entry_date),'exit_date':pd.Timestamp(path.iloc[-1]['date']),
                'role':role,'maturity_bucket':bucket,'entry_dte':float(selected['dte']),
                'exit_dte':float(path.iloc[-1]['dte']),'n_observations':len(path),
                'entry_spread':float(selected['straddle_relative_spread']),
            })
            accepted[key] = accepted.get(key,0) + 1
    return pd.DataFrame(rows), paths, pd.DataFrame(exclusions)

cohort_table, MARKET_PATHS, cohort_exclusions = select_contract_paths(paired_quotes)
if cohort_table.empty:
    raise RuntimeError('No historical contract paths passed the coverage gates.')
coverage = cohort_table.groupby(['role','maturity_bucket'], as_index=False).agg(
    n_cohorts=('cohort_id','nunique'), n_entry_dates=('entry_date','nunique'),
    median_observations=('n_observations','median'), first_entry=('entry_date','min'), last_entry=('entry_date','max'),
)
display(coverage)
coverage_lookup = coverage.set_index(['role','maturity_bucket'])['n_cohorts'].to_dict()
coverage_failures = []
for role, minimum in MKT_CFG.minimum_cohorts_by_role.items():
    for bucket in MKT_CFG.maturity_buckets:
        observed = int(coverage_lookup.get((role,bucket), 0))
        if observed < int(minimum):
            coverage_failures.append({'role':role,'maturity_bucket':bucket,'observed':observed,'required':int(minimum)})
if coverage_failures:
    raise AssertionError('Insufficient chronology-clean market coverage before training:\n' + pd.DataFrame(coverage_failures).to_string(index=False))
cohort_table.to_csv(FINAL_TABLE_DIR / 'market_cohorts.csv', index=False)
cohort_exclusions.to_csv(FINAL_TABLE_DIR / 'market_cohort_exclusions.csv', index=False)
coverage.to_csv(FINAL_TABLE_DIR / 'market_coverage.csv', index=False)

,role,maturity_bucket,n_cohorts,n_entry_dates,median_observations,first_entry,last_entry
0,finetune,1M,90,90,21.0,2018-01-03,2019-12-31
1,finetune,1W,104,104,7.0,2018-01-03,2019-12-31
2,finetune,2M,77,77,40.0,2018-01-03,2019-12-31
3,finetune,2W,97,97,11.0,2018-01-03,2019-12-31
4,finetune,3M,65,65,61.0,2018-01-03,2019-12-11
5,test,1M,189,189,21.0,2021-01-06,2024-11-27
6,test,1W,207,207,7.0,2021-01-06,2024-12-24
7,test,2M,173,173,41.0,2021-01-06,2024-11-06
8,test,2W,199,199,12.0,2021-01-06,2024-12-18
9,test,3M,161,161,62.0,2021-01-06,2024-10-09


In [44]:
from scipy.special import ndtr

OBSERVATION_TIME_CONVENTION = "completed_end_of_day_option_and_index_snapshot_t"
DECISION_TIME_CONVENTION = "after_feature_construction_from_information_available_by_t"
EXECUTION_TIME_CONVENTION = "next_recorded_daily_underlying_observation_t_plus_1"
EXECUTION_PRICE_CONVENTION = "next_recorded_CRSP_index_level_proxy"
EXECUTION_WINDOW_CONVENTION = "one_recorded_daily_observation"
EXECUTION_LAG_STEPS = 1

FULL_FEATURES = [
    'log_moneyness','tau_fraction','contract_implied_variance','bs_delta','tc_rate','dt_years',
    'previous_return','rv_5d','rv_20d',
    'atm_var_1w','atm_var_2w','atm_var_1m','atm_var_2m','atm_var_3m',
    'term_slope','term_curvature','atm_skew','contract_minus_atm_iv',
    'iv_change_1d','iv_change_5d','spread_fraction','missing_gap',
]
SCALAR_FEATURES = [
    'log_moneyness','tau_fraction','contract_implied_variance','bs_delta','tc_rate','dt_years',
    'previous_return','rv_5d','rv_20d','iv_change_1d','iv_change_5d','spread_fraction','missing_gap',
]


def market_black_scholes_straddle_delta(S, K, tau, r, q, sigma):
    S = np.asarray(S,float); tau = np.maximum(np.asarray(tau,float),1e-8)
    sigma = np.clip(np.asarray(sigma,float),1e-4,5.0)
    d1 = (np.log(S/float(K)) + (np.asarray(r)-np.asarray(q) + 0.5*sigma*sigma)*tau) / (sigma*np.sqrt(tau))
    return np.exp(-np.asarray(q)*tau) * (2.0*ndtr(d1)-1.0)


def market_bs_price(S, K, tau, r, q, sigma, option_type: Literal['call','put']):
    S = np.asarray(S,float); tau = np.maximum(np.asarray(tau,float),1e-10)
    sigma = np.clip(np.asarray(sigma,float),1e-4,5.0)
    d1 = (np.log(S/float(K)) + (np.asarray(r)-np.asarray(q)+0.5*sigma*sigma)*tau)/(sigma*np.sqrt(tau))
    d2 = d1 - sigma*np.sqrt(tau)
    call = S*np.exp(-np.asarray(q)*tau)*ndtr(d1) - float(K)*np.exp(-np.asarray(r)*tau)*ndtr(d2)
    if option_type == 'call': return call
    return call - S*np.exp(-np.asarray(q)*tau) + float(K)*np.exp(-np.asarray(r)*tau)


def market_path_to_episode(path: pd.DataFrame, tc_bps: float) -> dict[str,Any]:
    p = path.copy().reset_index(drop=True)
    S = p['spot'].to_numpy(float); K = float(p['strike'].iloc[0]); tau = p['T'].to_numpy(float)
    r = p['r'].to_numpy(float); q = p['q'].to_numpy(float); sigma = p['contract_iv'].clip(0.01,3.0).to_numpy(float)
    delta = market_black_scholes_straddle_delta(S,K,tau,r,q,sigma)
    dates = pd.to_datetime(p['date'])
    gaps = dates.diff().dt.days.fillna(1).clip(lower=1).to_numpy(float)
    dt_years = gaps / MKT_CFG.calendar_days_per_year
    feature = pd.DataFrame({
        'log_moneyness':np.log(S/K), 'tau_fraction':p['tau_fraction'].to_numpy(float),
        'contract_implied_variance':sigma**2, 'bs_delta':delta, 'tc_rate':float(tc_bps)/10000.0,
        'dt_years':dt_years, 'previous_return':p['log_return'].fillna(0).to_numpy(float),
        'rv_5d':p['rv_5d'].fillna(0).to_numpy(float), 'rv_20d':p['rv_20d'].fillna(0).to_numpy(float),
        'atm_var_1w':p['atm_var_1w'].to_numpy(float), 'atm_var_2w':p['atm_var_2w'].to_numpy(float),
        'atm_var_1m':p['atm_var_1m'].to_numpy(float), 'atm_var_2m':p['atm_var_2m'].to_numpy(float),
        'atm_var_3m':p['atm_var_3m'].to_numpy(float), 'term_slope':p['term_slope'].to_numpy(float),
        'term_curvature':p['term_curvature'].to_numpy(float), 'atm_skew':p['atm_skew'].to_numpy(float),
        'contract_minus_atm_iv':p['contract_minus_atm_iv'].fillna(0).to_numpy(float),
        'iv_change_1d':p['iv_change_1d'].fillna(0).to_numpy(float), 'iv_change_5d':p['iv_change_5d'].fillna(0).to_numpy(float),
        'spread_fraction':p['straddle_relative_spread'].to_numpy(float), 'missing_gap':(gaps>3).astype(float),
    })
    if not np.isfinite(feature[FULL_FEATURES].to_numpy(float)).all():
        raise FloatingPointError(f'Non-finite observable feature in cohort {p.cohort_id.iloc[0]}')
    return {
        'episode_id':str(p.cohort_id.iloc[0]), 'source':'market', 'role':str(p.chronology_role.iloc[0]),
        'maturity_bucket':str(p.maturity_bucket.iloc[0]), 'entry_date':pd.Timestamp(p.entry_date.iloc[0]),
        'features':feature[FULL_FEATURES].to_numpy(np.float32), 'feature_names':FULL_FEATURES,
        'bs_delta':delta.astype(np.float32), 'spot':S.astype(np.float32), 'r':r.astype(np.float32), 'q':q.astype(np.float32),
        'interval_dt':(np.diff(dates.to_numpy())/np.timedelta64(1,'D')/MKT_CFG.calendar_days_per_year).astype(np.float32),
        'entry_bid':float(p.straddle_bid.iloc[0]), 'entry_mid':float(p.straddle_mid.iloc[0]),
        'exit_ask':float(p.straddle_ask.iloc[-1]), 'exit_mid':float(p.straddle_mid.iloc[-1]),
        'tc_bps':float(tc_bps), 'strike':K,
        'observation_dates':dates.astype(str).tolist(),
        'information_time': OBSERVATION_TIME_CONVENTION,
        'decision_time': DECISION_TIME_CONVENTION,
        'execution_time': EXECUTION_TIME_CONVENTION,
        'execution_price': EXECUTION_PRICE_CONVENTION,
        'execution_window': EXECUTION_WINDOW_CONVENTION,
        'execution_lag_steps': EXECUTION_LAG_STEPS,
        'execution_instrument_note': 'SPX index level is an underlying-price proxy, not a directly tradable instrument',
    }

MARKET_EPISODES_BY_ROLE: dict[str,list[dict[str,Any]]] = {'finetune':[],'validation':[],'test':[],'confirmatory':[]}
for path in MARKET_PATHS.values():
    role = str(path.chronology_role.iloc[0])
    MARKET_EPISODES_BY_ROLE[role].append(market_path_to_episode(path, MKT_CFG.primary_transaction_cost_bps))

feature_schema_audit = pd.DataFrame([
    {'policy_information':'scalar','n_features':len(SCALAR_FEATURES),'features':SCALAR_FEATURES},
    {'policy_information':'surface','n_features':len(FULL_FEATURES),'features':FULL_FEATURES},
])
display(feature_schema_audit)
feature_schema_audit.to_json(FINAL_TABLE_DIR / 'shared_feature_schema.json', orient='records', indent=2)


,policy_information,n_features,features
0,scalar,13,"[log_moneyness, tau_fraction, contract_implied..."
1,surface,22,"[log_moneyness, tau_fraction, contract_implied..."


# 8. Observable policy schema and synthetic-surface adapter

The exact Heston and Lifted-Heston engines retain their latent states internally. At each synthetic hedge date, the notebook prices a prespecified set of contemporaneous options under the current model state, converts those prices to implied-volatility summaries and then deletes the latent path payload before caching policy episodes.

In [45]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Iterable, Mapping, Sequence
import contextlib
import hashlib
import io
import json
import math
import os
import random
import traceback
import time

import numpy as np
import pandas as pd

try:
    from scipy.special import ndtr
except Exception as exc:  # pragma: no cover
    raise RuntimeError("SciPy is required for Black–Scholes calculations.") from exc

try:
    import torch
    import torch.nn as nn
except Exception as exc:  # pragma: no cover
    raise RuntimeError("PyTorch is required for policy training.") from exc


PROHIBITED_POLICY_TOKENS = (
    "factor", "latent", "true_variance", "instantaneous_variance", "volterra",
    "state_projection", "projected_variance", "oracle", "simulator_variance",
    "heston_variance", "lifted_variance", "factor_",
)

CURRENT_FEATURES = [
    "log_moneyness", "tau_fraction", "contract_implied_variance", "bs_delta", "tc_rate", "dt_years",
    "previous_return", "rv_5d", "rv_20d",
    "atm_var_1w", "atm_var_2w", "atm_var_1m", "atm_var_2m", "atm_var_3m",
    "term_slope", "term_curvature", "atm_skew", "contract_minus_atm_iv",
    "iv_change_1d", "iv_change_5d", "spread_fraction", "missing_gap",
]

HISTORY_FEATURES = [
    "previous_return", "rv_5d", "rv_20d",
    "atm_var_1w", "atm_var_2w", "atm_var_1m", "atm_var_2m", "atm_var_3m",
    "term_slope", "term_curvature", "atm_skew", "iv_change_1d", "iv_change_5d",
]

SURFACE_HORIZONS_DAYS = (7, 14, 30, 60, 90)
SURFACE_LABELS = {7: "1w", 14: "2w", 30: "1m", 60: "2m", 90: "3m"}


def stable_hash(payload: Any) -> str:
    return hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()


def seed_everything(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def assert_observable_schema(
    current_features: Sequence[str] = CURRENT_FEATURES,
    history_features: Sequence[str] = HISTORY_FEATURES,
) -> None:
    names = [str(x).lower() for x in [*current_features, *history_features]]
    bad = sorted({name for name in names if any(tok in name for tok in PROHIBITED_POLICY_TOKENS)})
    if bad:
        raise AssertionError(f"Prohibited latent/oracle policy fields detected: {bad}")
    if len(set(current_features)) != len(current_features):
        raise AssertionError("Duplicate current-state feature names.")
    if len(set(history_features)) != len(history_features):
        raise AssertionError("Duplicate history feature names.")


def assert_episode_observability(episode: Mapping[str, Any], history_length: int) -> None:
    required = {
        "features", "feature_names", "history", "history_feature_names", "history_mask",
        "bs_delta", "spot", "r", "q", "interval_dt", "entry_bid", "entry_mid", "exit_ask", "strike",
        "information_time", "decision_time", "execution_time", "execution_price",
        "execution_window", "execution_lag_steps",
    }
    missing = sorted(required.difference(episode))
    if missing:
        raise KeyError(f"Episode missing fields: {missing}")
    assert_observable_schema(episode["feature_names"], episode["history_feature_names"])
    current = np.asarray(episode["features"], float)
    history = np.asarray(episode["history"], float)
    mask = np.asarray(episode["history_mask"], float)
    if current.ndim != 2 or current.shape[1] != len(episode["feature_names"]):
        raise AssertionError("Current feature tensor shape mismatch.")
    if history.ndim != 3 or history.shape[1] != int(history_length):
        raise AssertionError("History tensor shape mismatch.")
    if history.shape[2] != len(episode["history_feature_names"]):
        raise AssertionError("History feature dimension mismatch.")
    if mask.shape != history.shape[:2]:
        raise AssertionError("History mask shape mismatch.")
    if not np.isfinite(current).all() or not np.isfinite(history).all() or not np.isfinite(mask).all():
        raise FloatingPointError("Non-finite observable-policy tensors.")
    if np.any((mask != 0.0) & (mask != 1.0)):
        raise AssertionError("History mask must be binary.")
    if int(episode["execution_lag_steps"]) != int(EXECUTION_LAG_STEPS):
        raise AssertionError(
            f"Episode execution lag {episode['execution_lag_steps']} does not match "
            f"the protocol lag {EXECUTION_LAG_STEPS}."
        )
    expected_timing = {
        "information_time": OBSERVATION_TIME_CONVENTION,
        "decision_time": DECISION_TIME_CONVENTION,
        "execution_time": EXECUTION_TIME_CONVENTION,
        "execution_price": EXECUTION_PRICE_CONVENTION,
        "execution_window": EXECUTION_WINDOW_CONVENTION,
    }
    timing_mismatch = {
        key: {"expected": value, "observed": episode.get(key)}
        for key, value in expected_timing.items()
        if episode.get(key) != value
    }
    if timing_mismatch:
        raise AssertionError(f"Episode timing-contract mismatch: {timing_mismatch}")
    prohibited_keys = [k for k in episode if any(tok in str(k).lower() for tok in PROHIBITED_POLICY_TOKENS)]
    if prohibited_keys:
        raise AssertionError(f"Episode contains prohibited latent-state fields: {prohibited_keys}")


@dataclass
class RobustScaler:
    feature_names: list[str]
    centre: np.ndarray
    scale: np.ndarray
    clip: float = 8.0

    @classmethod
    def fit(cls, rows: np.ndarray, feature_names: Sequence[str]) -> "RobustScaler":
        x = np.asarray(rows, float)
        centre = np.nanmedian(x, axis=0)
        q25, q75 = np.nanquantile(x, [0.25, 0.75], axis=0)
        scale = np.maximum((q75 - q25) / 1.349, 1e-6)
        if not np.isfinite(centre).all() or not np.isfinite(scale).all():
            raise FloatingPointError("Non-finite scaler moments.")
        return cls(list(feature_names), centre.astype(np.float32), scale.astype(np.float32))

    def transform(self, x: np.ndarray) -> np.ndarray:
        x = np.asarray(x, float)
        return np.clip((x - self.centre) / self.scale, -self.clip, self.clip).astype(np.float32)

    def to_json(self) -> dict[str, Any]:
        return {
            "feature_names": self.feature_names,
            "centre": self.centre.tolist(),
            "scale": self.scale.tolist(),
            "clip": float(self.clip),
        }


@dataclass(frozen=True)
class TrainingConfig:
    adjustment_limit: float = 0.35
    hedge_limit: float = 2.0
    current_hidden_size: int = 64
    matched_current_hidden_size: int = 109
    current_depth: int = 2
    history_length: int = 10
    lag_hidden_size: int = 64
    gru_hidden_size: int = 39
    gru_layers: int = 1
    execution_lag_steps: int = EXECUTION_LAG_STEPS
    information_time_convention: str = OBSERVATION_TIME_CONVENTION
    decision_time_convention: str = DECISION_TIME_CONVENTION
    execution_time_convention: str = EXECUTION_TIME_CONVENTION
    execution_price_convention: str = EXECUTION_PRICE_CONVENTION
    execution_window_convention: str = EXECUTION_WINDOW_CONVENTION
    confirmatory_family_size: int = 6
    common_pretrain_steps: int = 1800
    hybrid_head_steps: int = 250
    hybrid_full_steps: int = 750
    batch_episodes: int = 16
    validation_interval: int = 50
    early_stopping_patience: int = 12
    learning_rate_pretrain: float = 3e-4
    learning_rate_head: float = 1e-4
    learning_rate_full: float = 5e-5
    weight_decay: float = 1e-5
    gradient_clip_norm: float = 1.0
    cvar_alpha: float = 0.05
    cvar_weight: float = 0.05
    adjustment_l2_weight: float = 1e-3
    l2_sp_lambda: float = 1e-4
    transaction_cost_bps: tuple[float, ...] = (1.0, 5.0, 10.0)

def print_and_gate_policy_schema() -> pd.DataFrame:
    rows = []
    for package, names in (("current", CURRENT_FEATURES), ("history", HISTORY_FEATURES)):
        for position, name in enumerate(names):
            rows.append({"package": package, "position": position, "feature": name, "observable": True})
    audit = pd.DataFrame(rows)
    print("Ranked policy input schema:")
    display(audit)
    assert_observable_schema(CURRENT_FEATURES, HISTORY_FEATURES)
    return audit

POLICY_SCHEMA_AUDIT = print_and_gate_policy_schema()


Ranked policy input schema:


,package,position,feature,observable
0,current,0,log_moneyness,True
1,current,1,tau_fraction,True
2,current,2,contract_implied_variance,True
3,current,3,bs_delta,True
4,current,4,tc_rate,True
5,current,5,dt_years,True
6,current,6,previous_return,True
7,current,7,rv_5d,True
8,current,8,rv_20d,True
9,current,9,atm_var_1w,True


In [46]:
class ResidualPolicyBase(nn.Module):
    def __init__(self, cfg: TrainingConfig):
        super().__init__()
        self.cfg = cfg

    def bounded_hedge(self, raw: torch.Tensor, baseline: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        adjustment = self.cfg.adjustment_limit * torch.tanh(raw / self.cfg.adjustment_limit)
        hedge = torch.clamp(baseline + adjustment, -self.cfg.hedge_limit, self.cfg.hedge_limit)
        return hedge, adjustment


class CurrentSurfaceMLP(ResidualPolicyBase):
    def __init__(self, current_dim: int, cfg: TrainingConfig):
        super().__init__(cfg)
        layers: list[nn.Module] = []
        last = current_dim + 1
        for _ in range(cfg.current_depth):
            layers += [nn.Linear(last, cfg.current_hidden_size), nn.SiLU()]
            last = cfg.current_hidden_size
        self.body = nn.Sequential(*layers)
        self.head = nn.Linear(last, 1)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward_step(
        self,
        current: torch.Tensor,
        history: torch.Tensor,
        history_mask: torch.Tensor,
        previous: torch.Tensor,
        baseline: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        del history, history_mask
        z = torch.cat([current, previous.unsqueeze(-1)], dim=-1)
        raw = self.head(self.body(z)).squeeze(-1)
        return self.bounded_hedge(raw, baseline)


class CapacityMatchedCurrentMLP(ResidualPolicyBase):
    """Current engineered state only, widened to match the sequence models."""
    def __init__(self, current_dim: int, cfg: TrainingConfig):
        super().__init__(cfg)
        width = int(cfg.matched_current_hidden_size)
        self.body = nn.Sequential(
            nn.Linear(current_dim + 1, width), nn.SiLU(),
            nn.Linear(width, width), nn.SiLU(),
        )
        self.head = nn.Linear(width, 1)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward_step(self, current, history, history_mask, previous, baseline):
        del history, history_mask
        z = torch.cat([current, previous.unsqueeze(-1)], dim=-1)
        raw = self.head(self.body(z)).squeeze(-1)
        return self.bounded_hedge(raw, baseline)


class LagStackMLP(ResidualPolicyBase):
    def __init__(self, current_dim: int, history_dim: int, cfg: TrainingConfig):
        super().__init__(cfg)
        input_dim = current_dim + cfg.history_length * history_dim + cfg.history_length + 1
        self.body = nn.Sequential(
            nn.Linear(input_dim, cfg.lag_hidden_size), nn.SiLU(),
            nn.Linear(cfg.lag_hidden_size, cfg.lag_hidden_size), nn.SiLU(),
        )
        self.head = nn.Linear(cfg.lag_hidden_size, 1)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward_step(self, current, history, history_mask, previous, baseline):
        masked = history * history_mask.unsqueeze(-1)
        z = torch.cat([
            current,
            masked.reshape(masked.shape[0], -1),
            history_mask,
            previous.unsqueeze(-1),
        ], dim=-1)
        raw = self.head(self.body(z)).squeeze(-1)
        return self.bounded_hedge(raw, baseline)


class ObservableGRU(ResidualPolicyBase):
    def __init__(self, current_dim: int, history_dim: int, cfg: TrainingConfig):
        super().__init__(cfg)
        self.gru = nn.GRU(
            input_size=history_dim + 1,
            hidden_size=cfg.gru_hidden_size,
            num_layers=cfg.gru_layers,
            batch_first=True,
        )
        self.body = nn.Sequential(
            nn.Linear(current_dim + cfg.gru_hidden_size + 1, cfg.current_hidden_size), nn.SiLU(),
            nn.Linear(cfg.current_hidden_size, cfg.current_hidden_size), nn.SiLU(),
        )
        self.head = nn.Linear(cfg.current_hidden_size, 1)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward_step(self, current, history, history_mask, previous, baseline):
        # Histories are right-aligned. Process only the valid suffix so padded zeros
        # cannot create a bias-driven pseudo-history before the first observation.
        # The valid length still conveys the same chronology mask information supplied
        # to the lag-stack control.
        batch_size = history.shape[0]
        encoded = torch.zeros(
            (batch_size, self.cfg.gru_hidden_size),
            dtype=history.dtype,
            device=history.device,
        )
        for b in range(batch_size):
            length = int(history_mask[b].sum().item())
            if length <= 0:
                continue
            valid_history = history[b:b+1, -length:, :]
            valid_mask = history_mask[b:b+1, -length:].unsqueeze(-1)
            valid_sequence = torch.cat([valid_history, valid_mask], dim=-1)
            _, hidden = self.gru(valid_sequence)
            encoded[b] = hidden[-1, 0]
        z = torch.cat([current, encoded, previous.unsqueeze(-1)], dim=-1)
        raw = self.head(self.body(z)).squeeze(-1)
        return self.bounded_hedge(raw, baseline)


def parameter_count(model: nn.Module) -> int:
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


def architecture_manifest(cfg: TrainingConfig) -> pd.DataFrame:
    models = {
        "current_surface_mlp": CurrentSurfaceMLP(len(CURRENT_FEATURES), cfg),
        "current_surface_mlp_matched": CapacityMatchedCurrentMLP(len(CURRENT_FEATURES), cfg),
        "lag_stack_mlp": LagStackMLP(len(CURRENT_FEATURES), len(HISTORY_FEATURES), cfg),
        "observable_gru": ObservableGRU(len(CURRENT_FEATURES), len(HISTORY_FEATURES), cfg),
    }
    return pd.DataFrame([
        {"architecture": name, "parameters": parameter_count(model), "history_length": 0 if name.startswith("current_surface_mlp") else cfg.history_length}
        for name, model in models.items()
    ])


def left_padded_histories(values: np.ndarray, history_length: int) -> tuple[np.ndarray, np.ndarray]:
    """Return a strictly lagged, right-aligned history tensor.

    The current observation at decision time t is already supplied through the
    current-state vector. Excluding x[t] from this sequence makes the lag-versus-
    current contrast identify incremental past information rather than a duplicated
    representation of the current surface. At t=0 the history is fully masked.
    """
    x = np.asarray(values, np.float32)
    n, d = x.shape
    out = np.zeros((n, history_length, d), dtype=np.float32)
    mask = np.zeros((n, history_length), dtype=np.float32)
    for t in range(n):
        start = max(0, t - history_length)
        window = x[start:t]
        if len(window):
            out[t, -len(window):] = window
            mask[t, -len(window):] = 1.0
    return out, mask

In [47]:
def market_episode_with_external_history(
    path: pd.DataFrame,
    episode: dict[str, Any],
    daily_surface_state: pd.DataFrame,
    history_length: int,
) -> dict[str, Any]:
    daily = daily_surface_state.copy().sort_values("date")
    daily["date"] = pd.to_datetime(daily["date"]).dt.normalize()
    mapping = {
        "previous_return": "log_return",
        "rv_5d": "rv_5d",
        "rv_20d": "rv_20d",
        "atm_var_1w": "atm_var_1w",
        "atm_var_2w": "atm_var_2w",
        "atm_var_1m": "atm_var_1m",
        "atm_var_2m": "atm_var_2m",
        "atm_var_3m": "atm_var_3m",
        "term_slope": "term_slope",
        "term_curvature": "term_curvature",
        "atm_skew": "atm_skew",
        "iv_change_1d": "iv_change_1d",
        "iv_change_5d": "iv_change_5d",
    }
    rows = []
    masks = []
    latest_history_dates = []
    for date in pd.to_datetime(path["date"]).dt.normalize():
        eligible = daily.loc[daily["date"].lt(date)].tail(history_length)
        arr = np.zeros((history_length, len(HISTORY_FEATURES)), dtype=np.float32)
        mask = np.zeros(history_length, dtype=np.float32)
        if len(eligible):
            vals = np.column_stack([
                pd.to_numeric(eligible[mapping[name]], errors="coerce").fillna(0.0).to_numpy(float)
                for name in HISTORY_FEATURES
            ]).astype(np.float32)
            arr[-len(vals):] = vals
            mask[-len(vals):] = 1.0
        rows.append(arr)
        masks.append(mask)
        latest_history_dates.append(None if eligible.empty else str(pd.Timestamp(eligible["date"].max()).date()))
    enriched = dict(episode)
    enriched["history"] = np.stack(rows)
    enriched["history_mask"] = np.stack(masks)
    enriched["history_feature_names"] = list(HISTORY_FEATURES)
    enriched["observation_dates"] = pd.to_datetime(path["date"]).astype(str).tolist()
    enriched["latest_history_dates"] = latest_history_dates
    for hedge_date, history_date in zip(pd.to_datetime(path["date"]).dt.normalize(), latest_history_dates):
        if history_date is not None and pd.Timestamp(history_date) >= hedge_date:
            raise AssertionError("Market history is not strictly prior to its hedge date.")
    assert_episode_observability(enriched, history_length)
    return enriched


def black_scholes_call_price(S, K, tau, r, q, sigma):
    S = np.asarray(S, float)
    K = np.asarray(K, float)
    tau = np.maximum(np.asarray(tau, float), 1e-10)
    sigma = np.clip(np.asarray(sigma, float), 1e-6, 5.0)
    d1 = (np.log(S / K) + (np.asarray(r) - np.asarray(q) + 0.5 * sigma ** 2) * tau) / (sigma * np.sqrt(tau))
    d2 = d1 - sigma * np.sqrt(tau)
    return S * np.exp(-np.asarray(q) * tau) * ndtr(d1) - K * np.exp(-np.asarray(r) * tau) * ndtr(d2)


def implied_vol_bisection(call_price, S, K, tau, r, q, *, iterations: int = 55) -> np.ndarray:
    price = np.asarray(call_price, float)
    S = np.asarray(S, float)
    K = np.asarray(K, float)
    tau = np.maximum(np.asarray(tau, float), 1e-8)
    r = np.asarray(r, float)
    q = np.asarray(q, float)
    lower_bound = np.maximum(S * np.exp(-q * tau) - K * np.exp(-r * tau), 0.0)
    upper_bound = S * np.exp(-q * tau)
    clipped = np.clip(price, lower_bound + 1e-10, upper_bound - 1e-10)
    lo = np.full(np.broadcast(clipped, S, K, tau).shape, 1e-4, dtype=float)
    hi = np.full_like(lo, 5.0)
    for _ in range(iterations):
        mid = 0.5 * (lo + hi)
        model = black_scholes_call_price(S, K, tau, r, q, mid)
        lo = np.where(model < clipped, mid, lo)
        hi = np.where(model >= clipped, mid, hi)
    return 0.5 * (lo + hi)


def black_scholes_straddle_delta(S, K, tau, r, q, sigma):
    S = np.asarray(S, float)
    K = np.asarray(K, float)
    tau = np.maximum(np.asarray(tau, float), 1e-8)
    sigma = np.clip(np.asarray(sigma, float), 1e-4, 5.0)
    d1 = (np.log(S / K) + (np.asarray(r) - np.asarray(q) + 0.5 * sigma ** 2) * tau) / (sigma * np.sqrt(tau))
    return np.exp(-np.asarray(q) * tau) * (2.0 * ndtr(d1) - 1.0)


def lifted_variable_strike_call_prices(
    pricer: Any,
    S: np.ndarray,
    K: np.ndarray,
    tau: float,
    r: float,
    q: float,
    params: Any,
    curve: Any,
    U: np.ndarray,
    current_time: float,
) -> np.ndarray:
    S = np.asarray(S, float).reshape(-1)
    K = np.asarray(K, float).reshape(-1)
    U = np.asarray(U, float)
    if len(S) != len(K) or U.shape[0] != len(S):
        raise ValueError("Variable-strike lifted pricing dimensions do not match.")
    if tau <= 1e-10:
        return np.maximum(S - K, 0.0)
    u = pricer.nodes.astype(np.complex128)
    z = np.concatenate([np.array([1.0 + 0.0j]), 1.0 + 1j * u, 1j * u])
    phi, psi = pricer._coefficients(z, float(tau), float(current_time), params, curve)
    state_loading = psi * pricer.weights.reshape(1, -1)
    x = np.log(S) + (float(r) - float(q)) * float(tau)
    cf = np.exp(np.outer(x, z) + phi.reshape(1, -1) + U @ state_loading.T)
    denominator = cf[:, 0]
    if np.any(np.abs(denominator) < 1e-14):
        raise FloatingPointError("Lifted P1 denominator is numerically zero.")
    n = len(u)
    phi1 = cf[:, 1:1+n] / denominator[:, None]
    phi2 = cf[:, 1+n:]
    kernel_term = np.exp(-1j * np.outer(np.log(K), u)) / (1j * u.reshape(1, -1))
    quadrature = pricer.quad_weights * np.exp(pricer.nodes)
    P1 = 0.5 + np.real(phi1 * kernel_term) @ quadrature / np.pi
    P2 = 0.5 + np.real(phi2 * kernel_term) @ quadrature / np.pi
    call = S * math.exp(-float(q) * tau) * P1 - K * math.exp(-float(r) * tau) * P2
    lower = np.maximum(S * math.exp(-float(q) * tau) - K * math.exp(-float(r) * tau), 0.0)
    upper = S * math.exp(-float(q) * tau)
    tol = 3e-3 * np.maximum(1.0, upper)
    if np.any(~np.isfinite(call)) or np.any(call < lower - tol) or np.any(call > upper + tol):
        raise FloatingPointError("Lifted variable-strike prices violate no-arbitrage bounds.")
    return np.asarray(call, float)


def model_call_prices_variable_strike(
    engine: Mapping[str, Any],
    world: str,
    *,
    S: np.ndarray,
    K: np.ndarray,
    tau: float,
    r: float,
    q: float,
    state_index: int,
    paths: Mapping[str, Any],
    seed_row: pd.Series,
) -> np.ndarray:
    S = np.asarray(S, float).reshape(-1)
    K = np.asarray(K, float).reshape(-1)
    if world == "heston":
        params = engine["lookup_params_by_date"](engine["RESEARCH_ARTIFACTS"].heston_calibration, seed_row["date"], engine["HestonParams"])
        n_quad = int(engine["CFG"].calibration.heston_pricing_nodes)
        out = np.empty(len(S), dtype=float)
        for i, (spot, strike, variance) in enumerate(zip(S, K, np.asarray(paths["V"], float)[:, state_index])):
            out[i] = engine["_heston_call_price_state_one"](
                float(spot), float(strike), float(tau), float(r), float(q), params, float(variance), n_quad
            )
        return out
    params, curve = engine["lookup_lifted_state_by_date"](
        engine["RESEARCH_ARTIFACTS"].lifted_calibration,
        seed_row["date"],
        engine["RESEARCH_ARTIFACTS"].forward_variance_curve,
    )
    pricer = engine["LiftedRiccatiFourierPricer"](
        engine["RESEARCH_ARTIFACTS"].lifted_kernel,
        int(engine["CFG"].lifted.pricing_nodes),
        int(engine["CFG"].lifted.riccati_steps),
        int(engine["CFG"].lifted.delta_coefficient_cache_size),
    )
    return lifted_variable_strike_call_prices(
        pricer, S, K, float(tau), float(r), float(q), params, curve,
        np.asarray(paths["U"], float)[:, state_index, :], float(np.asarray(paths["times"], float)[state_index]),
    )


def _surface_state_for_time(
    engine: Mapping[str, Any],
    world: str,
    paths: Mapping[str, Any],
    seed_row: pd.Series,
    state_index: int,
    contract_strikes: np.ndarray,
    contract_tau: float,
) -> dict[str, np.ndarray]:
    S = np.asarray(paths["S"], float)[:, state_index]
    r = float(paths["r"])
    q = float(paths["q"])
    current_time = float(np.asarray(paths["times"], float)[state_index])
    out: dict[str, np.ndarray] = {}
    for days in SURFACE_HORIZONS_DAYS:
        tau = days / 365.25
        F = S * np.exp((r - q) * tau)
        price = model_call_prices_variable_strike(
            engine, world, S=S, K=F, tau=tau, r=r, q=q, state_index=state_index, paths=paths, seed_row=seed_row,
        )
        iv = implied_vol_bisection(price, S, F, tau, r, q)
        out[f"atm_var_{SURFACE_LABELS[days]}"] = iv ** 2
    tau_skew = 30 / 365.25
    F = S * np.exp((r - q) * tau_skew)
    K_minus = F * np.exp(-0.05)
    K_plus = F * np.exp(0.05)
    p_minus = model_call_prices_variable_strike(engine, world, S=S, K=K_minus, tau=tau_skew, r=r, q=q, state_index=state_index, paths=paths, seed_row=seed_row)
    p_plus = model_call_prices_variable_strike(engine, world, S=S, K=K_plus, tau=tau_skew, r=r, q=q, state_index=state_index, paths=paths, seed_row=seed_row)
    iv_minus = implied_vol_bisection(p_minus, S, K_minus, tau_skew, r, q)
    iv_plus = implied_vol_bisection(p_plus, S, K_plus, tau_skew, r, q)
    out["atm_skew"] = (iv_plus - iv_minus) / 0.10
    contract_price = model_call_prices_variable_strike(
        engine, world, S=S, K=contract_strikes, tau=max(float(contract_tau), 1e-6), r=r, q=q,
        state_index=state_index, paths=paths, seed_row=seed_row,
    )
    out["contract_iv"] = implied_vol_bisection(contract_price, S, contract_strikes, max(float(contract_tau), 1e-6), r, q)
    return out


def exact_path_batch_to_observable_episodes(
    engine: Mapping[str, Any],
    seed_row: pd.Series,
    world: str,
    *,
    n_paths: int,
    path_seed: int,
    history_length: int,
    transaction_cost_bps: Sequence[float],
    spread_pool: np.ndarray | None = None,
    burn_in_days: int = 10,
) -> list[dict[str, Any]]:
    """Generate exact-model paths and expose only synthetic observable quotes/history.

    Latent states are used internally by the model-consistent pricing adapter. They are
    deleted before the episode dictionaries are returned and never enter a scaler or policy.
    """
    original = seed_row.copy()
    original_dte = int(round(float(original.get("dte", float(original["T"]) * 365.25))))
    close_dte = 2
    hedge_days = max(3, original_dte - close_dte)
    total_days = int(burn_in_days + hedge_days)
    sim_seed_row = original.copy()
    sim_seed_row["dte"] = total_days
    sim_seed_row["T"] = total_days / 365.25
    paths = engine["simulate_paths_for_world"](
        sim_seed_row, world, engine["CFG"], engine["RESEARCH_ARTIFACTS"], n_paths=int(n_paths), seed=int(path_seed)
    )
    full_times = np.asarray(paths["times"], float)
    start = int(np.argmin(np.abs(full_times - burn_in_days / 365.25)))
    stop = min(start + hedge_days, len(full_times) - 1)
    selected = np.arange(start, stop + 1, dtype=int)
    if len(selected) < 3:
        raise RuntimeError("Exact observable episode has too few hedge observations.")
    S_all = np.asarray(paths["S"], float)
    S_entry = S_all[:, selected[0]]
    r = float(paths["r"]); q = float(paths["q"])
    contract_T = original_dte / 365.25
    strikes = S_entry * np.exp((r - q) * contract_T)
    elapsed = full_times[selected] - full_times[selected[0]]
    tau = np.maximum(contract_T - elapsed, close_dte / 365.25)

    surface_rows: list[dict[str, np.ndarray]] = []
    for local_idx, state_idx in enumerate(selected):
        surface_rows.append(_surface_state_for_time(
            engine, world, paths, original, int(state_idx), strikes, float(tau[local_idx])
        ))
    fields = [f"atm_var_{SURFACE_LABELS[d]}" for d in SURFACE_HORIZONS_DAYS] + ["atm_skew", "contract_iv"]
    stacked = {name: np.stack([row[name] for row in surface_rows], axis=1) for name in fields}
    logret = np.zeros((n_paths, len(selected)), dtype=float)
    logret[:, 1:] = np.diff(np.log(S_all[:, selected]), axis=1)
    rv5 = pd.DataFrame(logret.T).rolling(5, min_periods=2).var().fillna(0.0).to_numpy().T * 252
    rv20 = pd.DataFrame(logret.T).rolling(20, min_periods=5).var().fillna(0.0).to_numpy().T * 252
    atm1m = stacked["atm_var_1m"]
    iv1 = np.column_stack([np.zeros(n_paths), np.diff(atm1m, axis=1)])
    iv5 = np.column_stack([np.zeros((n_paths, min(5, len(selected)))), np.diff(atm1m, n=1, axis=1)[:, 4:]]) if len(selected) > 5 else np.zeros_like(atm1m)
    if iv5.shape != atm1m.shape:
        iv5 = np.zeros_like(atm1m)
        if len(selected) > 5:
            iv5[:, 5:] = atm1m[:, 5:] - atm1m[:, :-5]
    rng = np.random.default_rng(int(path_seed) + 7441)
    if spread_pool is not None and len(spread_pool):
        spread = rng.choice(np.asarray(spread_pool, float), size=(n_paths, len(selected)), replace=True)
    else:
        spread = np.clip(rng.lognormal(math.log(0.04), 0.45, size=(n_paths, len(selected))), 0.002, 0.50)
    tc_values = rng.choice(np.asarray(transaction_cost_bps, float), size=n_paths)
    episodes: list[dict[str, Any]] = []
    for j in range(n_paths):
        S = S_all[j, selected]
        contract_iv = np.clip(stacked["contract_iv"][j], 0.01, 3.0)
        bs_delta = black_scholes_straddle_delta(S, strikes[j], tau, r, q, contract_iv)
        atm_contract = np.sqrt(np.maximum(stacked["atm_var_1m"][j], 1e-8))
        feature = pd.DataFrame({
            "log_moneyness": np.log(S / strikes[j]),
            "tau_fraction": tau / contract_T,
            "contract_implied_variance": contract_iv ** 2,
            "bs_delta": bs_delta,
            "tc_rate": tc_values[j] / 10000.0,
            "dt_years": np.concatenate([[1 / 365.25], np.diff(full_times[selected])]),
            "previous_return": logret[j],
            "rv_5d": rv5[j], "rv_20d": rv20[j],
            "atm_var_1w": stacked["atm_var_1w"][j], "atm_var_2w": stacked["atm_var_2w"][j],
            "atm_var_1m": stacked["atm_var_1m"][j], "atm_var_2m": stacked["atm_var_2m"][j],
            "atm_var_3m": stacked["atm_var_3m"][j],
            "term_slope": stacked["atm_var_3m"][j] - stacked["atm_var_1w"][j],
            "term_curvature": stacked["atm_var_1m"][j] - 0.5 * (stacked["atm_var_1w"][j] + stacked["atm_var_3m"][j]),
            "atm_skew": stacked["atm_skew"][j],
            "contract_minus_atm_iv": contract_iv - atm_contract,
            "iv_change_1d": iv1[j], "iv_change_5d": iv5[j],
            "spread_fraction": spread[j],
            "missing_gap": np.zeros(len(selected), dtype=float),
        })
        history_values = feature[HISTORY_FEATURES].to_numpy(np.float32)
        history, history_mask = left_padded_histories(history_values, history_length)
        call_entry = black_scholes_call_price(S[0], strikes[j], tau[0], r, q, contract_iv[0])
        put_entry = call_entry - S[0] * math.exp(-q * tau[0]) + strikes[j] * math.exp(-r * tau[0])
        call_exit = black_scholes_call_price(S[-1], strikes[j], tau[-1], r, q, contract_iv[-1])
        put_exit = call_exit - S[-1] * math.exp(-q * tau[-1]) + strikes[j] * math.exp(-r * tau[-1])
        mid_entry = float(call_entry + put_entry)
        mid_exit = float(call_exit + put_exit)
        episode = {
            "episode_id": f"exact_{world}_{pd.Timestamp(original['date']).date()}_{original.get('hedge_maturity_bucket', original.get('maturity_bucket','?'))}_{path_seed}_{j}",
            "source": "simulation", "simulation_world": world,
            "role": str(original.get("split", original.get("split_label", "train"))),
            "maturity_bucket": str(original.get("hedge_maturity_bucket", original.get("maturity_bucket", "unknown"))),
            "entry_date": pd.Timestamp(original["date"]),
            "features": feature[CURRENT_FEATURES].to_numpy(np.float32),
            "feature_names": list(CURRENT_FEATURES),
            "history": history, "history_mask": history_mask,
            "history_feature_names": list(HISTORY_FEATURES),
            "bs_delta": np.asarray(bs_delta, np.float32),
            "spot": np.asarray(S, np.float32),
            "r": np.full(len(S), r, np.float32), "q": np.full(len(S), q, np.float32),
            "interval_dt": np.diff(full_times[selected]).astype(np.float32),
            "entry_bid": max(mid_entry * (1 - 0.5 * spread[j, 0]), 1e-8),
            "entry_mid": mid_entry,
            "exit_ask": mid_exit * (1 + 0.5 * spread[j, -1]),
            "exit_mid": mid_exit,
            "tc_bps": float(tc_values[j]), "strike": float(strikes[j]),
            "observation_times": full_times[selected].tolist(),
            "information_time": OBSERVATION_TIME_CONVENTION,
            "decision_time": DECISION_TIME_CONVENTION,
            "execution_time": EXECUTION_TIME_CONVENTION,
            "execution_price": EXECUTION_PRICE_CONVENTION,
            "execution_window": EXECUTION_WINDOW_CONVENTION,
            "execution_lag_steps": EXECUTION_LAG_STEPS,
            "execution_instrument_note": "synthetic index level on the next daily node",
        }
        assert_episode_observability(episode, history_length)
        episodes.append(episode)
    # Explicitly drop the path dictionary: latent states must not leak into cached policy episodes.
    del paths
    return episodes


def build_toy_episode_bank(
    *,
    n_episodes: int,
    seed: int,
    history_length: int,
    role: str,
    world: str,
) -> list[dict[str, Any]]:
    rng = np.random.default_rng(seed)
    episodes = []
    buckets = ["1W", "2W", "1M", "2M", "3M"]
    for i in range(n_episodes):
        n = int(rng.integers(7, 18))
        K = 5000.0
        returns = rng.normal(0, 0.009, n)
        S = K * np.exp(np.cumsum(returns))
        tau = np.linspace(30 / 365.25, 2 / 365.25, n)
        base_var = np.maximum(0.03 + np.cumsum(rng.normal(0, 0.0004, n)), 0.005)
        skew = -0.3 + rng.normal(0, 0.02, n)
        contract_iv = np.sqrt(base_var) + skew * np.log(S / K)
        delta = black_scholes_straddle_delta(S, K, tau, 0.01, 0.0, contract_iv)
        f = pd.DataFrame({
            "log_moneyness": np.log(S / K), "tau_fraction": tau / tau[0],
            "contract_implied_variance": contract_iv ** 2, "bs_delta": delta, "tc_rate": 1e-4,
            "dt_years": 1 / 365.25, "previous_return": returns,
            "rv_5d": pd.Series(returns).rolling(5, min_periods=2).var().fillna(base_var[0]).to_numpy() * 252,
            "rv_20d": pd.Series(returns).rolling(20, min_periods=2).var().fillna(base_var[0]).to_numpy() * 252,
            "atm_var_1w": base_var * 0.95, "atm_var_2w": base_var * 0.98,
            "atm_var_1m": base_var, "atm_var_2m": base_var * 1.04, "atm_var_3m": base_var * 1.07,
            "term_slope": base_var * 0.12, "term_curvature": -base_var * 0.005,
            "atm_skew": skew, "contract_minus_atm_iv": contract_iv - np.sqrt(base_var),
            "iv_change_1d": np.concatenate([[0], np.diff(base_var)]),
            "iv_change_5d": pd.Series(base_var).diff(5).fillna(0).to_numpy(),
            "spread_fraction": 0.04, "missing_gap": 0.0,
        })
        hist, mask = left_padded_histories(f[HISTORY_FEATURES].to_numpy(np.float32), history_length)
        entry_mid = float(max(20.0, 0.04 * K))
        ep = {
            "episode_id": f"toy_{world}_{role}_{i}", "source": "toy", "simulation_world": world,
            "role": role, "maturity_bucket": buckets[i % len(buckets)], "entry_date": pd.Timestamp("2017-01-01") + pd.Timedelta(days=i),
            "features": f[CURRENT_FEATURES].to_numpy(np.float32), "feature_names": list(CURRENT_FEATURES),
            "history": hist, "history_mask": mask, "history_feature_names": list(HISTORY_FEATURES),
            "bs_delta": delta.astype(np.float32), "spot": S.astype(np.float32),
            "r": np.full(n, 0.01, np.float32), "q": np.zeros(n, np.float32),
            "interval_dt": np.full(n - 1, 1 / 365.25, np.float32),
            "entry_bid": entry_mid * 0.99, "entry_mid": entry_mid, "exit_ask": entry_mid * 0.85,
            "exit_mid": entry_mid * 0.84, "tc_bps": 1.0, "strike": K,
            "observation_times": np.arange(n).tolist(),
            "information_time": OBSERVATION_TIME_CONVENTION,
            "decision_time": DECISION_TIME_CONVENTION,
            "execution_time": EXECUTION_TIME_CONVENTION,
            "execution_price": EXECUTION_PRICE_CONVENTION,
            "execution_window": EXECUTION_WINDOW_CONVENTION,
            "execution_lag_steps": EXECUTION_LAG_STEPS,
            "execution_instrument_note": "toy daily underlying node",
        }
        assert_episode_observability(ep, history_length)
        episodes.append(ep)
    return episodes


def fit_common_scalers(mixed_train_episodes: Sequence[Mapping[str, Any]]) -> tuple[RobustScaler, RobustScaler]:
    current = np.concatenate([np.asarray(e["features"], float) for e in mixed_train_episodes], axis=0)
    hist_rows = []
    for e in mixed_train_episodes:
        h = np.asarray(e["history"], float)
        m = np.asarray(e["history_mask"], bool)
        hist_rows.append(h[m])
    history = np.concatenate(hist_rows, axis=0)
    return RobustScaler.fit(current, CURRENT_FEATURES), RobustScaler.fit(history, HISTORY_FEATURES)


# 9. Residual policies, hybrid training and common accounting

All policies predict a bounded correction around Black–Scholes delta. The current Surface MLP, lag-stack MLP and GRU share the same output parameterisation and accounting. The lag-stack and GRU use identical ten-step observable histories and masks, with approximately matched parameter counts.

In [48]:
def clone_episode_with_cost(episode: Mapping[str, Any], tc_bps: float) -> dict[str, Any]:
    e = dict(episode)
    features = np.array(episode["features"], copy=True)
    features[:, CURRENT_FEATURES.index("tc_rate")] = float(tc_bps) / 10000.0
    e["features"] = features
    e["tc_bps"] = float(tc_bps)
    return e


def rollout_episode(
    model: nn.Module,
    episode: Mapping[str, Any],
    current_scaler: RobustScaler,
    history_scaler: RobustScaler,
    device: torch.device,
) -> tuple[torch.Tensor, torch.Tensor]:
    current = torch.as_tensor(current_scaler.transform(episode["features"]), dtype=torch.float32, device=device)
    history_raw = np.asarray(episode["history"], float)
    history_scaled = history_scaler.transform(history_raw.reshape(-1, history_raw.shape[-1])).reshape(history_raw.shape)
    history = torch.as_tensor(history_scaled, dtype=torch.float32, device=device)
    mask = torch.as_tensor(episode["history_mask"], dtype=torch.float32, device=device)
    baseline = torch.as_tensor(episode["bs_delta"], dtype=torch.float32, device=device)
    previous = torch.zeros(1, dtype=torch.float32, device=device)
    hedges, adjustments = [], []
    for t in range(len(current) - 1):
        hedge, adjustment = model.forward_step(
            current[t:t+1], history[t:t+1], mask[t:t+1], previous, baseline[t:t+1]
        )
        hedges.append(hedge.squeeze(0)); adjustments.append(adjustment.squeeze(0)); previous = hedge
    return torch.stack(hedges), torch.stack(adjustments)


def apply_execution_lag(hedges: torch.Tensor, lag_steps: int) -> torch.Tensor:
    lag_steps = int(lag_steps)
    if lag_steps <= 0:
        return hedges
    if len(hedges) <= lag_steps:
        return torch.zeros_like(hedges)
    return torch.cat([torch.zeros_like(hedges[:lag_steps]), hedges[:-lag_steps]], dim=0)


def differentiable_short_straddle_pnl(episode: Mapping[str, Any], hedges: torch.Tensor, device: torch.device):
    """Account for already executed hedge positions with an auditable additive decomposition.

    Requested positions are shifted by ``apply_execution_lag`` before this function
    is called. Keeping that transformation outside the cash recursion prevents an
    accidental mixture of decision-time and execution-time positions.


    The decomposition is algebraically identical to the authoritative cash recursion:

        net P&L = option mid-price change
                  + stock holding gain
                  + financing
                  + dividend carry
                  - option spread costs
                  - underlying trading costs.

    It is diagnostic only; the training objective continues to use the same net P&L.
    """
    # Use float64 for the cash-accounting recursion. SPX notionals are large enough
    # that an algebraically exact float32 decomposition can differ by several
    # thousandths of an index point purely through rounding.
    accounting_dtype = torch.float64
    hedges = hedges.to(device=device, dtype=accounting_dtype)

    S = torch.as_tensor(episode["spot"], dtype=accounting_dtype, device=device)
    r = torch.as_tensor(episode["r"], dtype=accounting_dtype, device=device)
    q = torch.as_tensor(episode["q"], dtype=accounting_dtype, device=device)
    dt = torch.as_tensor(episode["interval_dt"], dtype=accounting_dtype, device=device)
    lam = torch.as_tensor(float(episode["tc_bps"]) / 10000.0, dtype=accounting_dtype, device=device)

    entry_bid = torch.as_tensor(float(episode["entry_bid"]), dtype=accounting_dtype, device=device)
    entry_mid = torch.as_tensor(float(episode["entry_mid"]), dtype=accounting_dtype, device=device)
    exit_ask = torch.as_tensor(float(episode["exit_ask"]), dtype=accounting_dtype, device=device)
    exit_mid = torch.as_tensor(float(episode.get("exit_mid", episode["exit_ask"])), dtype=accounting_dtype, device=device)

    cash = entry_bid.clone()
    # Keep the position and every accounting accumulator in float64. Using a
    # float32 zero here causes in-place ``+=`` operations to round financing and
    # dividend components back to float32, even though the cash recursion itself
    # is float64. That creates a small artificial decomposition residual.
    previous = torch.zeros((), dtype=accounting_dtype, device=device)
    trading_costs = torch.zeros((), dtype=accounting_dtype, device=device)
    turnover = torch.zeros((), dtype=accounting_dtype, device=device)
    financing_pnl = torch.zeros((), dtype=accounting_dtype, device=device)
    dividend_carry = torch.zeros((), dtype=accounting_dtype, device=device)

    for t in range(len(hedges)):
        if t > 0:
            interest = cash * (torch.exp(r[t - 1] * dt[t - 1]) - 1.0)
            dividend = previous * q[t - 1] * S[t - 1] * dt[t - 1]
            cash = cash + interest + dividend
            financing_pnl += interest
            dividend_carry += dividend
        trade = hedges[t] - previous
        cost = lam * S[t] * torch.abs(trade)
        cash = cash - trade * S[t] - cost
        trading_costs += cost
        turnover += torch.abs(trade)
        previous = hedges[t]

    final_interest = cash * (torch.exp(r[-2] * dt[-1]) - 1.0)
    final_dividend = previous * q[-2] * S[-2] * dt[-1]
    cash = cash + final_interest + final_dividend
    financing_pnl += final_interest
    dividend_carry += final_dividend

    liquidation_cost = lam * S[-1] * torch.abs(previous)
    cash = cash + previous * S[-1] - liquidation_cost
    turnover += torch.abs(previous)
    underlying_costs = trading_costs + liquidation_cost
    pnl = cash - exit_ask

    stock_holding_gain = torch.sum(hedges * (S[1:len(hedges) + 1] - S[:len(hedges)]))
    option_mark_to_market_pnl = entry_mid - exit_mid
    option_entry_spread_cost = torch.clamp(entry_mid - entry_bid, min=0.0)
    option_exit_spread_cost = torch.clamp(exit_ask - exit_mid, min=0.0)
    option_execution_cost = option_entry_spread_cost + option_exit_spread_cost
    reconstructed = (
        option_mark_to_market_pnl + stock_holding_gain + financing_pnl + dividend_carry
        - option_execution_cost - underlying_costs
    )
    identity_error = pnl - reconstructed
    # The identity is checked at a tight float64 tolerance. A larger discrepancy
    # now indicates a real accounting inconsistency rather than float32 round-off.
    identity_scale = torch.maximum(
        torch.maximum(torch.abs(pnl), torch.abs(reconstructed)),
        torch.ones((), dtype=accounting_dtype, device=device),
    )
    identity_tolerance = 1e-10 * identity_scale + 1e-10
    if torch.abs(identity_error).item() > identity_tolerance.item():
        raise AssertionError(
            "Accounting decomposition failed: "
            f"error={float(identity_error.item()):.6g}, "
            f"tolerance={float(identity_tolerance.item()):.6g}"
        )

    return pnl, {
        "costs": underlying_costs,
        "turnover": turnover,
        "entry_premium_bid": entry_bid,
        "entry_mid": entry_mid,
        "exit_mid": exit_mid,
        "exit_ask": exit_ask,
        "option_mark_to_market_pnl": option_mark_to_market_pnl,
        "option_entry_spread_cost": option_entry_spread_cost,
        "option_exit_spread_cost": option_exit_spread_cost,
        "option_execution_cost": option_execution_cost,
        "stock_holding_gain": stock_holding_gain,
        "financing_pnl": financing_pnl,
        "dividend_carry": dividend_carry,
        "underlying_trading_cost": trading_costs,
        "underlying_liquidation_cost": liquidation_cost,
        "accounting_identity_error": identity_error,
    }

def batch_loss(
    model: nn.Module,
    episodes: Sequence[Mapping[str, Any]],
    current_scaler: RobustScaler,
    history_scaler: RobustScaler,
    rng: np.random.Generator,
    cfg: TrainingConfig,
    device: torch.device,
    *,
    anchor_state: Mapping[str, torch.Tensor] | None = None,
    l2_sp_lambda: float = 0.0,
):
    pnls, adjustment_terms = [], []
    for raw in episodes:
        assert_episode_observability(raw, cfg.history_length)
        if int(raw["execution_lag_steps"]) != int(cfg.execution_lag_steps):
            raise AssertionError("Training episode and objective use different execution lags.")
        episode = clone_episode_with_cost(raw, float(rng.choice(cfg.transaction_cost_bps)))
        hedges, adjustments = rollout_episode(model, episode, current_scaler, history_scaler, device)
        executed_hedges = apply_execution_lag(hedges, cfg.execution_lag_steps)
        pnl, _ = differentiable_short_straddle_pnl(episode, executed_hedges, device)
        pnls.append(pnl / max(float(episode["entry_mid"]), 1e-8))
        adjustment_terms.append(torch.mean(adjustments ** 2))
    pnl_tensor = torch.stack(pnls)
    mse = torch.mean(pnl_tensor ** 2)
    k = max(1, int(math.ceil(cfg.cvar_alpha * len(pnl_tensor))))
    tail = torch.mean(torch.relu(torch.topk(-pnl_tensor, k=k, largest=True).values) ** 2)
    adjustment = torch.mean(torch.stack(adjustment_terms))
    l2_sp = torch.zeros((), dtype=torch.float32, device=device)
    if anchor_state is not None and l2_sp_lambda > 0:
        for name, param in model.named_parameters():
            l2_sp = l2_sp + torch.sum((param - anchor_state[name].to(device)) ** 2)
    loss = mse + cfg.cvar_weight * tail + cfg.adjustment_l2_weight * adjustment + l2_sp_lambda * l2_sp
    return loss, {"mse": mse.detach(), "tail": tail.detach(), "adjustment": adjustment.detach(), "l2_sp": l2_sp.detach()}


def validation_metrics(model, episodes, current_scaler, history_scaler, device, max_episodes=None):
    """Return episode-weighted and headline date-weighted validation metrics.

    H035 keeps the episode-weighted criterion used by the cached checkpoints.
    H025 selects checkpoints using equal maturity weight within date and equal
    weight across dates. Both values are exported for the selection audit.
    """
    use = list(episodes) if max_episodes is None else list(episodes)[:int(max_episodes)]
    rows = []
    model.eval()
    with torch.no_grad():
        for raw in use:
            assert_episode_observability(raw, TRAIN_CFG.history_length)
            if int(raw["execution_lag_steps"]) != int(TRAIN_CFG.execution_lag_steps):
                raise AssertionError("Validation episode and objective use different execution lags.")
            episode = clone_episode_with_cost(raw, 1.0)
            hedges, _ = rollout_episode(model, episode, current_scaler, history_scaler, device)
            executed_hedges = apply_execution_lag(hedges, TRAIN_CFG.execution_lag_steps)
            pnl, meta = differentiable_short_straddle_pnl(episode, executed_hedges, device)
            norm = float(pnl.item()) / max(float(episode["entry_mid"]), 1e-8)
            rows.append({
                "entry_date": pd.Timestamp(episode["entry_date"]),
                "maturity_bucket": str(episode["maturity_bucket"]),
                "squared_error": norm ** 2,
                "turnover": float(meta["turnover"].item()),
                "cost": float(meta["costs"].item()),
            })
    frame = pd.DataFrame(rows)
    if frame.empty:
        raise ValueError("Validation set is empty.")
    episode_mse = float(frame["squared_error"].mean())
    date_bucket = frame.groupby(["entry_date", "maturity_bucket"], as_index=False).agg(
        squared_error=("squared_error", "mean"),
        turnover=("turnover", "mean"),
        cost=("cost", "mean"),
    )
    date_level = date_bucket.groupby("entry_date", as_index=False).agg(
        squared_error=("squared_error", "mean"),
        turnover=("turnover", "mean"),
        cost=("cost", "mean"),
    )
    date_mse = float(date_level["squared_error"].mean())
    selected_mse = episode_mse if MODEL_SELECTION_WEIGHTING == "episode_weighted" else date_mse
    return {
        "mse": selected_mse,
        "episode_weighted_mse": episode_mse,
        "date_weighted_mse": date_mse,
        "turnover": float(date_level["turnover"].mean()),
        "cost": float(date_level["cost"].mean()),
        "selection_weighting": MODEL_SELECTION_WEIGHTING,
        "episodes": int(len(frame)),
        "dates": int(frame["entry_date"].nunique()),
    }

def train_stage(
    model: nn.Module,
    train_episodes,
    val_episodes,
    current_scaler,
    history_scaler,
    *,
    steps: int,
    learning_rate: float,
    stage: str,
    training_seed: int,
    cfg: TrainingConfig,
    device: torch.device,
    trainable: str = "all",
    anchor_state: Mapping[str, torch.Tensor] | None = None,
    l2_sp_lambda: float = 0.0,
    sampling_key: str | None = None,
):
    seed_everything(training_seed)
    model.to(device)
    for p in model.parameters(): p.requires_grad = True
    if trainable == "head":
        for name, p in model.named_parameters(): p.requires_grad = name.startswith("head")
    params = [p for p in model.parameters() if p.requires_grad]
    optimiser = torch.optim.AdamW(params, lr=float(learning_rate), weight_decay=cfg.weight_decay)
    sampling_token = stage if sampling_key is None else str(sampling_key)
    rng = np.random.default_rng(training_seed + int(stable_hash(sampling_token)[:8], 16))
    history_rows = []
    best_state = None; best_mse = float("inf"); stale = 0
    train_list = list(train_episodes)
    for step in range(1, int(steps) + 1):
        idx = rng.integers(0, len(train_list), size=min(cfg.batch_episodes, len(train_list)))
        batch = [train_list[int(i)] for i in idx]
        model.train(); optimiser.zero_grad(set_to_none=True)
        loss, parts = batch_loss(model, batch, current_scaler, history_scaler, rng, cfg, device, anchor_state=anchor_state, l2_sp_lambda=l2_sp_lambda)
        if not torch.isfinite(loss): raise FloatingPointError(f"Non-finite loss in {stage}.")
        loss.backward(); torch.nn.utils.clip_grad_norm_(params, cfg.gradient_clip_norm); optimiser.step()
        if step == 1 or step % cfg.validation_interval == 0 or step == steps:
            metrics = validation_metrics(model, val_episodes, current_scaler, history_scaler, device)
            row = {"stage": stage, "training_seed": int(training_seed), "step": int(step), "train_loss": float(loss.item()), **{f"train_{k}": float(v.item()) for k, v in parts.items()}, **{f"validation_{k}": v for k, v in metrics.items()}}
            history_rows.append(row)
            if metrics["mse"] < best_mse - 1e-10:
                best_mse = metrics["mse"]
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                stale = 0
            else:
                stale += 1
            if stale >= cfg.early_stopping_patience:
                break
    if best_state is None: raise RuntimeError(f"No valid checkpoint in stage {stage}.")
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history_rows), float(best_mse)


def make_model(architecture: str, cfg: TrainingConfig) -> nn.Module:
    if architecture == "current_surface_mlp":
        return CurrentSurfaceMLP(len(CURRENT_FEATURES), cfg)
    if architecture == "current_surface_mlp_matched":
        return CapacityMatchedCurrentMLP(len(CURRENT_FEATURES), cfg)
    if architecture == "lag_stack_mlp":
        return LagStackMLP(len(CURRENT_FEATURES), len(HISTORY_FEATURES), cfg)
    if architecture == "observable_gru":
        return ObservableGRU(len(CURRENT_FEATURES), len(HISTORY_FEATURES), cfg)
    raise KeyError(architecture)


def train_hybrid_policy(
    *,
    architecture: str,
    source_label: str,
    sim_train,
    sim_validation,
    market_train,
    market_validation,
    current_scaler,
    history_scaler,
    training_seed: int,
    cfg: TrainingConfig,
    device: torch.device,
):
    """Pretrain, adapt and select among three candidate stages on 2020 data."""
    seed_everything(training_seed)
    model = make_model(architecture, cfg)
    model, h_pre, sim_pretrain_mse = train_stage(
        model, sim_train, sim_validation, current_scaler, history_scaler,
        steps=cfg.common_pretrain_steps,
        learning_rate=cfg.learning_rate_pretrain,
        stage=f"{source_label}_{architecture}_pretrain",
        training_seed=training_seed,
        cfg=cfg,
        device=device,
        sampling_key="pretrain",
    )
    pre_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    pretrained_market_metrics = validation_metrics(model, market_validation, current_scaler, history_scaler, device)

    head = make_model(architecture, cfg)
    head.load_state_dict(pre_state)
    head, h_head, _ = train_stage(
        head, market_train, market_validation, current_scaler, history_scaler,
        steps=cfg.hybrid_head_steps,
        learning_rate=cfg.learning_rate_head,
        stage=f"{source_label}_{architecture}_head",
        training_seed=training_seed,
        cfg=cfg,
        device=device,
        trainable="head",
        sampling_key="market_head",
    )
    head_market_metrics = validation_metrics(head, market_validation, current_scaler, history_scaler, device)

    full = make_model(architecture, cfg)
    full.load_state_dict(head.state_dict())
    full, h_full, _ = train_stage(
        full, market_train, market_validation, current_scaler, history_scaler,
        steps=cfg.hybrid_full_steps,
        learning_rate=cfg.learning_rate_full,
        stage=f"{source_label}_{architecture}_full",
        training_seed=training_seed,
        cfg=cfg,
        device=device,
        anchor_state=pre_state,
        l2_sp_lambda=cfg.l2_sp_lambda,
        sampling_key="market_full",
    )
    full_market_metrics = validation_metrics(full, market_validation, current_scaler, history_scaler, device)

    candidates = [
        ("simulation_pretrained", model, pretrained_market_metrics),
        ("head_only", head, head_market_metrics),
        ("full", full, full_market_metrics),
    ]
    selected_stage, selected_model, selected_metrics = min(candidates, key=lambda x: float(x[2]["mse"]))

    histories = []
    if not h_pre.empty:
        histories.append(h_pre.assign(selection_domain="simulation_validation"))
    if not h_head.empty:
        histories.append(h_head.assign(selection_domain="market_validation_2020"))
    if not h_full.empty:
        histories.append(h_full.assign(selection_domain="market_validation_2020"))

    metadata = {
        "architecture": architecture,
        "source": source_label,
        "training_seed": int(training_seed),
        "selected_stage": selected_stage,
        "selection_domain": "market_validation_2020",
        "selection_weighting": MODEL_SELECTION_WEIGHTING,
        "validation_mse": float(selected_metrics["mse"]),
        "validation_episode_weighted_mse": float(selected_metrics["episode_weighted_mse"]),
        "validation_date_weighted_mse": float(selected_metrics["date_weighted_mse"]),
        "simulation_pretrain_validation_mse": float(sim_pretrain_mse),
        "pretrained_market_validation_mse": float(pretrained_market_metrics["mse"]),
        "pretrained_market_episode_weighted_mse": float(pretrained_market_metrics["episode_weighted_mse"]),
        "pretrained_market_date_weighted_mse": float(pretrained_market_metrics["date_weighted_mse"]),
        "head_market_validation_mse": float(head_market_metrics["mse"]),
        "head_market_episode_weighted_mse": float(head_market_metrics["episode_weighted_mse"]),
        "head_market_date_weighted_mse": float(head_market_metrics["date_weighted_mse"]),
        "full_market_validation_mse": float(full_market_metrics["mse"]),
        "full_market_episode_weighted_mse": float(full_market_metrics["episode_weighted_mse"]),
        "full_market_date_weighted_mse": float(full_market_metrics["date_weighted_mse"]),
        "pretrain_updates_completed": int(h_pre.step.max()) if not h_pre.empty else 0,
        "head_updates_completed": int(h_head.step.max()) if not h_head.empty else 0,
        "full_updates_completed": int(h_full.step.max()) if not h_full.empty else 0,
    }
    return selected_model, pd.concat(histories, ignore_index=True), metadata

def no_trade_band_positions(delta: np.ndarray, band: float) -> np.ndarray:
    target = np.asarray(delta, float)
    out = np.zeros_like(target)
    previous = 0.0
    for t, value in enumerate(target):
        if abs(float(value) - previous) > float(band):
            previous = float(value)
        out[t] = previous
    return out


def select_bs_no_trade_band(validation_episodes, candidates: Sequence[float], device: torch.device) -> tuple[float, pd.DataFrame]:
    rows = []
    for band in candidates:
        episode_rows = []
        for raw in validation_episodes:
            episode = clone_episode_with_cost(raw, 1.0)
            hedge = no_trade_band_positions(np.asarray(episode["bs_delta"], float), float(band))
            result = numpy_accounting(episode, hedge, device)
            norm = result["net_pnl"] / max(float(episode["entry_mid"]), 1e-8)
            episode_rows.append({
                "entry_date": pd.Timestamp(episode["entry_date"]),
                "maturity_bucket": str(episode["maturity_bucket"]),
                "squared_error": norm ** 2,
                "turnover": result["turnover"],
            })
        frame = pd.DataFrame(episode_rows)
        episode_mse = float(frame["squared_error"].mean())
        date_bucket = frame.groupby(["entry_date", "maturity_bucket"], as_index=False).agg(
            squared_error=("squared_error", "mean"), turnover=("turnover", "mean")
        )
        date_level = date_bucket.groupby("entry_date", as_index=False).agg(
            squared_error=("squared_error", "mean"), turnover=("turnover", "mean")
        )
        date_mse = float(date_level["squared_error"].mean())
        selected_mse = episode_mse if MODEL_SELECTION_WEIGHTING == "episode_weighted" else date_mse
        rows.append({
            "band": float(band),
            "validation_mse": selected_mse,
            "episode_weighted_mse": episode_mse,
            "date_weighted_mse": date_mse,
            "validation_turnover": float(date_level["turnover"].mean()),
            "selection_weighting": MODEL_SELECTION_WEIGHTING,
        })
    table = pd.DataFrame(rows).sort_values(["validation_mse", "band"]).reset_index(drop=True)
    table["selected_under_episode_weighting"] = table["band"].eq(
        table.sort_values(["episode_weighted_mse", "band"]).iloc[0]["band"]
    )
    table["selected_under_date_weighting"] = table["band"].eq(
        table.sort_values(["date_weighted_mse", "band"]).iloc[0]["band"]
    )
    return float(table.iloc[0]["band"]), table

def numpy_accounting(episode, hedges, device, execution_lag_steps: int | None = None):
    arr = np.asarray(hedges, float)
    if len(arr) == len(episode["spot"]):
        arr = arr[:-1]
    h = torch.as_tensor(arr, dtype=torch.float32, device=device)
    lag = TRAIN_CFG.execution_lag_steps if execution_lag_steps is None else int(execution_lag_steps)
    episode_lag = int(episode.get("execution_lag_steps", -1))
    if lag != episode_lag:
        raise AssertionError(
            f"Accounting requested lag {lag}, but episode timing contract requires {episode_lag}."
        )
    h = apply_execution_lag(h, lag)
    with torch.no_grad():
        pnl, meta = differentiable_short_straddle_pnl(episode, h, device)

    values = {key: float(value.item()) for key, value in meta.items()}
    underlying_cost = values["costs"]
    option_execution_cost = values["option_execution_cost"]
    result = {
        "net_pnl": float(pnl.item()),
        "turnover": values["turnover"],
        "underlying_cost": underlying_cost,
        "option_execution_cost": option_execution_cost,
        "total_execution_cost": underlying_cost + option_execution_cost,
        "execution_lag_steps": lag,
        "entry_premium_bid": values["entry_premium_bid"],
        "entry_mid": values["entry_mid"],
        "exit_mid": values["exit_mid"],
        "exit_ask": values["exit_ask"],
        "option_mark_to_market_pnl": values["option_mark_to_market_pnl"],
        "option_entry_spread_cost": values["option_entry_spread_cost"],
        "option_exit_spread_cost": values["option_exit_spread_cost"],
        "stock_holding_gain": values["stock_holding_gain"],
        "financing_pnl": values["financing_pnl"],
        "dividend_carry": values["dividend_carry"],
        "underlying_trading_cost": values["underlying_trading_cost"],
        "underlying_liquidation_cost": values["underlying_liquidation_cost"],
        "accounting_identity_error": values["accounting_identity_error"],
    }
    premium = max(float(episode["entry_mid"]), 1e-8)
    for key in ACCOUNTING_COMPONENTS:
        if key in result:
            result[f"normalised_{key}"] = float(result[key]) / premium
    return result


def evaluate_policies(
    episodes,
    policies: Mapping[tuple[str, int], nn.Module],
    checkpoint_meta: pd.DataFrame,
    current_scaler: RobustScaler,
    history_scaler: RobustScaler,
    training_seeds: Sequence[int],
    cost_levels: Sequence[float],
    device: torch.device,
) -> pd.DataFrame:
    rows = []
    for raw in episodes:
        for tc in cost_levels:
            episode = clone_episode_with_cost(raw, float(tc))
            baselines = {
                "no_hedge": np.zeros_like(episode["bs_delta"]),
                "black_scholes_delta": np.asarray(episode["bs_delta"], float),
                "black_scholes_no_trade": no_trade_band_positions(np.asarray(episode["bs_delta"], float), SELECTED_BS_NO_TRADE_BAND),
            }
            for label, hedges in baselines.items():
                result = numpy_accounting(episode, hedges, device)
                norm = result["net_pnl"] / max(float(episode["entry_mid"]), 1e-8)
                rows.append({"episode_id": episode["episode_id"], "entry_date": episode["entry_date"], "maturity_bucket": episode["maturity_bucket"], "policy": label, "training_seed": -1, "checkpoint_stage": "analytic", "underlying_tc_bps": float(tc), "normalised_net_pnl": norm, "normalised_squared_error": norm ** 2, **result})
            for (label, seed), model in policies.items():
                model.eval()
                with torch.no_grad():
                    hedges, _ = rollout_episode(model, episode, current_scaler, history_scaler, device)
                result = numpy_accounting(episode, hedges.detach().cpu().numpy(), device)
                norm = result["net_pnl"] / max(float(episode["entry_mid"]), 1e-8)
                stage = checkpoint_meta.loc[(checkpoint_meta.policy == label) & (checkpoint_meta.training_seed == int(seed)), "selected_stage"].iloc[0]
                rows.append({"episode_id": episode["episode_id"], "entry_date": episode["entry_date"], "maturity_bucket": episode["maturity_bucket"], "policy": label, "training_seed": int(seed), "checkpoint_stage": stage, "underlying_tc_bps": float(tc), "normalised_net_pnl": norm, "normalised_squared_error": norm ** 2, **result})
    return pd.DataFrame(rows)


def build_decision_level_results(
    episodes,
    policies: Mapping[tuple[str, int], nn.Module],
    current_scaler: RobustScaler,
    history_scaler: RobustScaler,
    *,
    policy_labels: Sequence[str],
    cost_bps: float,
    device: torch.device,
) -> pd.DataFrame:
    """Export seed-ensemble policy behaviour without producing millions of redundant rows.

    Each neural row contains the mean requested hedge and its cross-seed standard deviation.
    Seed-level terminal results remain available in the dedicated seed tables.
    """
    rows: list[dict[str, Any]] = []
    neural_labels = sorted({label for label, _ in policies})
    for raw in episodes:
        episode = clone_episode_with_cost(raw, float(cost_bps))
        S = np.asarray(episode["spot"], float)
        bs = np.asarray(episode["bs_delta"], float)[:-1]
        current = np.asarray(episode["features"], float)[:-1]
        feature_names = list(episode["feature_names"])
        obs_times = list(episode.get("observation_times", range(len(S))))
        requested_by_policy: dict[str, tuple[np.ndarray, np.ndarray, int]] = {}

        baselines = {
            "no_hedge": np.zeros(len(S) - 1, dtype=float),
            "black_scholes_delta": np.asarray(episode["bs_delta"], float)[:-1],
            "black_scholes_no_trade": no_trade_band_positions(np.asarray(episode["bs_delta"], float), SELECTED_BS_NO_TRADE_BAND)[:-1],
        }
        for label, requested in baselines.items():
            requested_by_policy[label] = (requested, np.zeros_like(requested), 1)

        for label in neural_labels:
            seed_paths = []
            for (candidate, _seed), model in policies.items():
                if candidate != label:
                    continue
                model.eval()
                with torch.no_grad():
                    hedges, _ = rollout_episode(model, episode, current_scaler, history_scaler, device)
                seed_paths.append(hedges.detach().cpu().numpy().astype(float))
            if seed_paths:
                stack = np.vstack(seed_paths)
                requested_by_policy[label] = (stack.mean(axis=0), stack.std(axis=0, ddof=1) if len(stack) > 1 else np.zeros(stack.shape[1]), len(stack))

        for label in policy_labels:
            if label not in requested_by_policy:
                continue
            requested, seed_std, seed_count = requested_by_policy[label]
            cumulative_turnover = 0.0
            for t, hedge in enumerate(requested):
                previous = 0.0 if t == 0 else float(requested[t - 1])
                execution_step = t + int(TRAIN_CFG.execution_lag_steps)
                executed = execution_step < len(requested)
                trade_size = float(hedge - previous) if executed else np.nan
                if executed:
                    cumulative_turnover += abs(trade_size)
                row = {
                    "episode_id": episode["episode_id"], "entry_date": episode["entry_date"],
                    "maturity_bucket": episode["maturity_bucket"], "policy": label,
                    "training_seed": -2 if label in neural_labels else -1,
                    "seed_count": int(seed_count), "hedge_seed_std": float(seed_std[t]),
                    "underlying_tc_bps": float(cost_bps), "observation_step": int(t),
                    "execution_step": int(execution_step), "executed_before_liquidation": bool(executed),
                    "observation_time": str(obs_times[t] if t < len(obs_times) else t),
                    "execution_time": str(obs_times[execution_step]) if executed and execution_step < len(obs_times) else None,
                    "observation_spot": float(S[t]),
                    "execution_spot": float(S[execution_step]) if executed else np.nan,
                    "strike": float(episode["strike"]), "bs_delta": float(bs[t]),
                    "requested_hedge": float(hedge), "residual_correction": float(hedge - bs[t]),
                    "previous_requested_hedge": previous, "executed_trade_size": trade_size,
                    "cumulative_turnover_before_liquidation": float(cumulative_turnover),
                }
                for j, name in enumerate(feature_names):
                    row[name] = float(current[t, j])
                rows.append(row)
    return pd.DataFrame(rows)


In [49]:
def lower_cvar(values: pd.Series, alpha: float = 0.05) -> float:
    x = np.asarray(values, float)
    if not len(x): return float("nan")
    cutoff = np.quantile(x, alpha)
    return float(x[x <= cutoff].mean())


def summarise_results(frame: pd.DataFrame, groups: Sequence[str]) -> pd.DataFrame:
    rows = []
    for keys, g in frame.groupby(list(groups), dropna=False):
        keys = keys if isinstance(keys, tuple) else (keys,)
        seed_rows = g.loc[g.training_seed.ge(0)]
        analytic_rows = g.loc[g.training_seed.lt(0)]
        use = seed_rows.groupby(["episode_id", "entry_date", "maturity_bucket"], as_index=False).agg(
            normalised_net_pnl=("normalised_net_pnl", "mean"),
            normalised_squared_error=("normalised_squared_error", "mean"),
            turnover=("turnover", "mean"), underlying_cost=("underlying_cost", "mean"), net_pnl=("net_pnl", "mean"),
        ) if len(seed_rows) else analytic_rows
        row = dict(zip(groups, keys))
        row.update({
            "episodes": int(use.episode_id.nunique()), "dates": int(pd.to_datetime(use.entry_date).nunique()),
            "rmse_normalised_net_pnl": float(np.sqrt(use.normalised_squared_error.mean())),
            "mean_normalised_net_pnl": float(use.normalised_net_pnl.mean()),
            "std_normalised_net_pnl": float(use.normalised_net_pnl.std(ddof=1)),
            "cvar5_normalised_net_pnl": lower_cvar(use.normalised_net_pnl),
            "mean_turnover": float(use.turnover.mean()), "mean_underlying_cost": float(use.underlying_cost.mean()),
        })
        rows.append(row)
    return pd.DataFrame(rows)


def stationary_bootstrap_indices(n: int, mean_block_length: int, rng: np.random.Generator) -> np.ndarray:
    p = 1.0 / max(float(mean_block_length), 1.0)
    out = np.empty(n, dtype=int)
    out[0] = int(rng.integers(0, n))
    for t in range(1, n):
        out[t] = int(rng.integers(0, n)) if rng.random() < p else (out[t - 1] + 1) % n
    return out


def calendar_weeks_to_date_observations(dates: Sequence[pd.Timestamp], weeks: int) -> int:
    ordered = pd.Series(pd.to_datetime(list(dates))).drop_duplicates().sort_values()
    gaps = ordered.diff().dt.days.dropna()
    median_gap = float(gaps[gaps.gt(0)].median()) if gaps.gt(0).any() else 7.0
    return max(1, int(round(float(weeks) * 7.0 / max(median_gap, 1.0))))


def paired_contrast(
    results: pd.DataFrame,
    policy_a: str,
    policy_b: str,
    label: str,
    *,
    primary_cost: float = 1.0,
    repetitions: int = 10_000,
    mean_block_weeks: int = 13,
    seed: int = 20260711,
    strict_inference: bool = False,
    minimum_dates_for_inference: int = 2,
) -> dict[str, Any]:
    """
    Compute a paired date-level MSE contrast.

    Smoke runs may contain too few dates for bootstrap inference. In that
    case, retain the point estimate and return explicitly unavailable
    intervals rather than aborting the reporting pipeline.

    Full runs should set strict_inference=True.
    """
    minimum_dates_for_inference = max(
        int(minimum_dates_for_inference),
        2,
    )

    use = results.loc[
        results["underlying_tc_bps"].eq(float(primary_cost))
    ].copy()

    use["entry_date"] = (
        pd.to_datetime(use["entry_date"], errors="coerce")
        .dt.normalize()
    )

    neural = (
        use.loc[use["training_seed"].ge(0)]
        .groupby(
            ["entry_date", "maturity_bucket", "policy"],
            as_index=False,
        )["normalised_squared_error"]
        .mean()
    )

    analytic = (
        use.loc[use["training_seed"].lt(0)]
        .groupby(
            ["entry_date", "maturity_bucket", "policy"],
            as_index=False,
        )["normalised_squared_error"]
        .mean()
    )

    panel = pd.concat(
        [neural, analytic],
        ignore_index=True,
    )

    comparison_panel = panel.loc[
        panel["policy"].isin([policy_a, policy_b])
    ]

    pair = comparison_panel.pivot_table(
        index=["entry_date", "maturity_bucket"],
        columns="policy",
        values="normalised_squared_error",
        aggfunc="first",
    )

    missing_policies = [
        policy
        for policy in (policy_a, policy_b)
        if policy not in pair.columns
    ]

    if missing_policies:
        message = (
            f"Contrast {label} is missing policy results: "
            f"{missing_policies}."
        )

        if strict_inference:
            raise RuntimeError(message)

        return {
            "comparison": label,
            "policy_a": policy_a,
            "policy_b": policy_b,
            "n_dates": 0,
            "absolute_mse_difference_a_minus_b": np.nan,
            "relative_mse_improvement_a_over_b": np.nan,
            "ci_2_5pct": np.nan,
            "ci_97_5pct": np.nan,
            "familywise_ci_1_25pct": np.nan,
            "familywise_ci_98_75pct": np.nan,
            "absolute_ci_2_5pct": np.nan,
            "absolute_ci_97_5pct": np.nan,
            "bootstrap_fraction_nonpositive": np.nan,
            "mean_block_weeks": int(mean_block_weeks),
            "mean_block_date_observations": np.nan,
            "bootstrap_repetitions": 0,
            "bootstrap_performed": False,
            "inference_status": "MISSING_POLICY_RESULTS",
            "inference_note": message,
            "aggregation": (
                "equal_seed_then_equal_available_buckets_"
                "within_date_then_equal_date"
            ),
        }

    pair = (
        pair.reset_index()
        .dropna(subset=[policy_a, policy_b])
    )

    date_level = (
        pair.groupby("entry_date", as_index=False)
        .agg(
            mse_a=(policy_a, "mean"),
            mse_b=(policy_b, "mean"),
        )
        .sort_values("entry_date")
        .reset_index(drop=True)
    )

    n_dates = int(len(date_level))

    if n_dates == 0:
        message = f"No paired dates are available for {label}."

        if strict_inference:
            raise RuntimeError(message)

        return {
            "comparison": label,
            "policy_a": policy_a,
            "policy_b": policy_b,
            "n_dates": 0,
            "absolute_mse_difference_a_minus_b": np.nan,
            "relative_mse_improvement_a_over_b": np.nan,
            "ci_2_5pct": np.nan,
            "ci_97_5pct": np.nan,
            "familywise_ci_1_25pct": np.nan,
            "familywise_ci_98_75pct": np.nan,
            "absolute_ci_2_5pct": np.nan,
            "absolute_ci_97_5pct": np.nan,
            "bootstrap_fraction_nonpositive": np.nan,
            "mean_block_weeks": int(mean_block_weeks),
            "mean_block_date_observations": np.nan,
            "bootstrap_repetitions": 0,
            "bootstrap_performed": False,
            "inference_status": "NO_PAIRED_DATES",
            "inference_note": message,
            "aggregation": (
                "equal_seed_then_equal_available_buckets_"
                "within_date_then_equal_date"
            ),
        }

    a = date_level["mse_a"].to_numpy(float)
    b = date_level["mse_b"].to_numpy(float)

    mean_a = float(a.mean())
    mean_b = float(b.mean())

    observed_abs = mean_a - mean_b
    observed_rel = 1.0 - mean_a / max(mean_b, 1e-12)

    if n_dates < minimum_dates_for_inference:
        message = (
            f"Only {n_dates} paired date(s) are available for {label}; "
            f"at least {minimum_dates_for_inference} are required for "
            "the configured date-block bootstrap."
        )

        if strict_inference:
            raise RuntimeError(message)

        return {
            "comparison": label,
            "policy_a": policy_a,
            "policy_b": policy_b,
            "n_dates": n_dates,
            "absolute_mse_difference_a_minus_b": observed_abs,
            "relative_mse_improvement_a_over_b": observed_rel,
            "ci_2_5pct": np.nan,
            "ci_97_5pct": np.nan,
            "familywise_ci_1_25pct": np.nan,
            "familywise_ci_98_75pct": np.nan,
            "absolute_ci_2_5pct": np.nan,
            "absolute_ci_97_5pct": np.nan,
            "bootstrap_fraction_nonpositive": np.nan,
            "mean_block_weeks": int(mean_block_weeks),
            "mean_block_date_observations": np.nan,
            "bootstrap_repetitions": 0,
            "bootstrap_performed": False,
            "inference_status": (
                "SMOKE_ONLY_INSUFFICIENT_DATES"
                if not strict_inference
                else "INSUFFICIENT_DATES"
            ),
            "inference_note": message,
            "aggregation": (
                "equal_seed_then_equal_available_buckets_"
                "within_date_then_equal_date"
            ),
        }

    mean_block_dates = calendar_weeks_to_date_observations(
        date_level["entry_date"],
        mean_block_weeks,
    )

    rng = np.random.default_rng(
        int(seed) + int(stable_hash(label)[:8], 16)
    )

    rel = np.empty(int(repetitions), dtype=float)
    absolute = np.empty(int(repetitions), dtype=float)

    for repetition in range(int(repetitions)):
        indices = stationary_bootstrap_indices(
            n_dates,
            mean_block_dates,
            rng,
        )

        bootstrap_a = float(a[indices].mean())
        bootstrap_b = float(b[indices].mean())

        rel[repetition] = (
            1.0
            - bootstrap_a / max(bootstrap_b, 1e-12)
        )
        absolute[repetition] = bootstrap_a - bootstrap_b

    return {
        "comparison": label,
        "policy_a": policy_a,
        "policy_b": policy_b,
        "n_dates": n_dates,
        "absolute_mse_difference_a_minus_b": observed_abs,
        "relative_mse_improvement_a_over_b": observed_rel,
        "ci_2_5pct": float(np.quantile(rel, 0.025)),
        "ci_97_5pct": float(np.quantile(rel, 0.975)),
        "familywise_ci_1_25pct": float(
            np.quantile(rel, 0.0125)
        ),
        "familywise_ci_98_75pct": float(
            np.quantile(rel, 0.9875)
        ),
        "absolute_ci_2_5pct": float(
            np.quantile(absolute, 0.025)
        ),
        "absolute_ci_97_5pct": float(
            np.quantile(absolute, 0.975)
        ),
        "bootstrap_fraction_nonpositive": float(
            np.mean(rel <= 0)
        ),
        "mean_block_weeks": int(mean_block_weeks),
        "mean_block_date_observations": int(mean_block_dates),
        "bootstrap_repetitions": int(repetitions),
        "bootstrap_performed": True,
        "inference_status": "BOOTSTRAP_COMPLETE",
        "inference_note": "",
        "aggregation": (
            "equal_seed_then_equal_available_buckets_"
            "within_date_then_equal_date"
        ),
    }


def result_placeholder_manifest() -> pd.DataFrame:
    return pd.DataFrame([
        {"placeholder_id": "R1", "section": "7.1", "description": "Evidence-gate table and completed cell counts", "source_file": "tables/evidence_gate.csv"},
        {"placeholder_id": "R2", "section": "7.2", "description": "Heston/Lifted/mixed current-MLP source summary", "source_file": "tables/source_summary.csv"},
        {"placeholder_id": "R3", "section": "7.2", "description": "Lifted-vs-Heston and mixed-vs-Heston paired contrasts", "source_file": "tables/source_primary_contrasts.csv"},
        {"placeholder_id": "R4", "section": "7.3", "description": "Current/lag/GRU architecture summary", "source_file": "tables/architecture_summary.csv"},
        {"placeholder_id": "R5", "section": "7.3", "description": "History and recurrence paired contrasts", "source_file": "tables/architecture_primary_contrasts.csv"},
        {"placeholder_id": "R6", "section": "7.4", "description": "Maturity-specific results", "source_file": "tables/maturity_summary.csv"},
        {"placeholder_id": "R7", "section": "7.5", "description": "1/5/10 bp cost robustness", "source_file": "tables/cost_robustness.csv"},
        {"placeholder_id": "R8", "section": "8", "description": "Permitted final interpretation", "source_file": "claim_evidence_map.csv"},
    ])

def enhanced_policy_summary(frame: pd.DataFrame, groups: Sequence[str]) -> pd.DataFrame:
    rows = []
    component_cols = [
        "normalised_option_mark_to_market_pnl", "normalised_stock_holding_gain",
        "normalised_financing_pnl", "normalised_dividend_carry",
        "normalised_option_entry_spread_cost", "normalised_option_exit_spread_cost",
        "normalised_option_execution_cost", "normalised_underlying_trading_cost",
        "normalised_underlying_liquidation_cost", "normalised_underlying_cost",
        "normalised_total_execution_cost",
    ]
    for keys, g in frame.groupby(list(groups), dropna=False):
        keys = keys if isinstance(keys, tuple) else (keys,)
        neural = g.loc[g.training_seed.ge(0)].copy()
        analytic = g.loc[g.training_seed.lt(0)].copy()
        agg = {
            "normalised_net_pnl": ("normalised_net_pnl", "mean"),
            "normalised_squared_error": ("normalised_squared_error", "mean"),
            "turnover": ("turnover", "mean"), "underlying_cost": ("underlying_cost", "mean"),
            "option_execution_cost": ("option_execution_cost", "mean"),
            "total_execution_cost": ("total_execution_cost", "mean"), "net_pnl": ("net_pnl", "mean"),
            "accounting_identity_error": ("accounting_identity_error", lambda s: float(np.max(np.abs(pd.to_numeric(s, errors="coerce"))))),
        }
        for col in component_cols:
            if col in g.columns:
                agg[col] = (col, "mean")
        use = neural.groupby(["episode_id", "entry_date", "maturity_bucket"], as_index=False).agg(**agg) if len(neural) else analytic
        x = use.normalised_net_pnl.to_numpy(float)
        mean = float(np.mean(x)); variance = float(np.var(x, ddof=0)); bias_sq = mean ** 2
        losses = -x; negative = np.minimum(x, 0.0)
        k = max(1, int(math.ceil(0.05 * len(x))))
        es_loss = float(np.mean(np.sort(losses)[-k:]))
        row = dict(zip(groups, keys))
        row.update({
            "episodes": int(use.episode_id.nunique()), "dates": int(pd.to_datetime(use.entry_date).nunique()),
            "mse_normalised_net_pnl": float(np.mean(x ** 2)), "rmse_normalised_net_pnl": float(np.sqrt(np.mean(x ** 2))),
            "mean_normalised_net_pnl": mean, "std_normalised_net_pnl": float(np.std(x, ddof=1)) if len(x)>1 else 0.0,
            "variance_contribution": variance, "bias_squared_contribution": bias_sq,
            "downside_rmse": float(np.sqrt(np.mean(negative ** 2))),
            "worst5_mean_normalised_net_pnl": float(np.mean(np.sort(x)[:k])),
            "expected_shortfall_style_loss_5pct": es_loss,
            "mean_turnover": float(use.turnover.mean()), "mean_underlying_cost": float(use.underlying_cost.mean()),
            "mean_option_execution_cost": float(use.option_execution_cost.mean()), "mean_total_execution_cost": float(use.total_execution_cost.mean()),
            "maximum_absolute_accounting_identity_error": float(np.max(np.abs(use.accounting_identity_error))) if "accounting_identity_error" in use else np.nan,
        })
        for col in component_cols:
            if col in use.columns:
                row[f"mean_{col}"] = float(pd.to_numeric(use[col], errors="coerce").mean())
        rows.append(row)
    return pd.DataFrame(rows)


def ensemble_episode_panel(frame: pd.DataFrame, cost_bps: float = 1.0) -> pd.DataFrame:
    use = frame.loc[frame.underlying_tc_bps.eq(float(cost_bps))].copy()
    neural = use.loc[use.training_seed.ge(0)].groupby(
        ["episode_id", "entry_date", "maturity_bucket", "policy"], as_index=False
    ).agg(normalised_net_pnl=("normalised_net_pnl", "mean"), normalised_squared_error=("normalised_squared_error", "mean"))
    analytic = use.loc[use.training_seed.lt(0), ["episode_id", "entry_date", "maturity_bucket", "policy", "normalised_net_pnl", "normalised_squared_error"]]
    return pd.concat([neural, analytic], ignore_index=True)


def paired_tail_contrast(results, policy_a, policy_b, label, *, primary_cost=1.0, repetitions=10_000,
                         mean_block_weeks=13, seed=20260721, alpha=0.05):
    panel = ensemble_episode_panel(results, primary_cost)
    pair = panel.loc[panel.policy.isin([policy_a, policy_b])].pivot_table(
        index=["episode_id", "entry_date", "maturity_bucket"], columns="policy",
        values="normalised_net_pnl", aggfunc="first"
    ).dropna().reset_index()
    dates = sorted(pd.to_datetime(pair.entry_date).dt.normalize().unique())
    if len(dates) < 2:
        return {"comparison": label, "tail_inference_status": "INSUFFICIENT_DATES"}
    pair["entry_date"] = pd.to_datetime(pair.entry_date).dt.normalize()
    def worst_mean(values):
        x = np.asarray(values, float); k = max(1, int(math.ceil(alpha * len(x))))
        return float(np.sort(x)[:k].mean())
    point = worst_mean(pair[policy_a]) - worst_mean(pair[policy_b])
    block = calendar_weeks_to_date_observations(dates, mean_block_weeks)
    rng = np.random.default_rng(int(seed) + int(stable_hash("tail_" + label)[:8], 16))
    draws = []
    grouped = {d: pair.loc[pair.entry_date.eq(d)] for d in dates}
    for _ in range(int(repetitions)):
        ix = stationary_bootstrap_indices(len(dates), block, rng)
        sampled = pd.concat([grouped[dates[i]] for i in ix], ignore_index=True)
        draws.append(worst_mean(sampled[policy_a]) - worst_mean(sampled[policy_b]))
    draws = np.asarray(draws, float)
    return {
        "comparison": label, "policy_a": policy_a, "policy_b": policy_b,
        "worst5_mean_difference_a_minus_b": point,
        "tail_ci_2_5pct": float(np.quantile(draws, .025)), "tail_ci_97_5pct": float(np.quantile(draws, .975)),
        "tail_fraction_nonpositive": float(np.mean(draws <= 0)), "tail_repetitions": int(repetitions),
        "tail_mean_block_weeks": int(mean_block_weeks), "tail_inference_status": "COMPLETE",
        "tail_sign_convention": "positive means policy A has a less negative worst-five-percent mean P&L",
    }


def paired_date_effect_panel(results, policy_a, policy_b, label, cost_bps=1.0) -> pd.DataFrame:
    panel = ensemble_episode_panel(results, cost_bps)
    pair = panel.loc[panel.policy.isin([policy_a, policy_b])].pivot_table(
        index=["entry_date", "maturity_bucket"], columns="policy", values="normalised_squared_error", aggfunc="mean"
    ).dropna().reset_index()
    out = pair.groupby("entry_date", as_index=False)[[policy_a, policy_b]].mean()
    out["comparison"] = label
    out["mse_a"] = out[policy_a]; out["mse_b"] = out[policy_b]
    out["absolute_mse_difference_a_minus_b"] = out.mse_a - out.mse_b
    out["relative_mse_effect"] = 1.0 - out.mse_a / out.mse_b.clip(lower=1e-12)
    out["favours_a"] = out.relative_mse_effect.gt(0)
    return out[["comparison", "entry_date", "mse_a", "mse_b", "absolute_mse_difference_a_minus_b", "relative_mse_effect", "favours_a"]]

def seed_level_summary(frame: pd.DataFrame, primary_cost: float = 1.0) -> pd.DataFrame:
    use = frame.loc[frame.underlying_tc_bps.eq(float(primary_cost)) & frame.training_seed.ge(0)].copy()
    return use.groupby(["policy", "training_seed"], as_index=False).agg(
        episodes=("episode_id", "nunique"), dates=("entry_date", lambda s: pd.to_datetime(s).nunique()),
        mse=("normalised_squared_error", "mean"),
        rmse=("normalised_squared_error", lambda s: float(np.sqrt(np.mean(pd.to_numeric(s, errors="coerce"))))),
        mean_pnl=("normalised_net_pnl", "mean"), turnover=("turnover", "mean"), total_cost=("total_execution_cost", "mean"),
    )


def seed_dispersion_summary(seed_table: pd.DataFrame) -> pd.DataFrame:
    return seed_table.groupby("policy", as_index=False).agg(
        seed_count=("training_seed", "nunique"), mean_rmse=("rmse", "mean"), std_rmse=("rmse", "std"),
        median_rmse=("rmse", "median"), min_rmse=("rmse", "min"), max_rmse=("rmse", "max"),
    )


def paired_contrast_v6(results, policy_a, policy_b, label, *, primary_cost=1.0, repetitions=10_000,
                       mean_block_weeks=13, seed=20260711, family_size=6, strict_inference=False):
    base = paired_contrast(results, policy_a, policy_b, label, primary_cost=primary_cost,
                           repetitions=repetitions, mean_block_weeks=mean_block_weeks, seed=seed,
                           strict_inference=strict_inference)
    if not base.get("bootstrap_performed", False):
        base.update({"familywise_ci_low": np.nan, "familywise_ci_high": np.nan, "family_size": int(family_size)})
        return base
    # Re-run only the paired date panel to obtain the correct Bonferroni family quantiles.
    use = results.loc[results.underlying_tc_bps.eq(float(primary_cost))].copy()
    use["entry_date"] = pd.to_datetime(use["entry_date"], errors="coerce").dt.normalize()
    neural = use.loc[use.training_seed.ge(0)].groupby(["entry_date","maturity_bucket","policy"], as_index=False).normalised_squared_error.mean()
    analytic = use.loc[use.training_seed.lt(0)].groupby(["entry_date","maturity_bucket","policy"], as_index=False).normalised_squared_error.mean()
    panel = pd.concat([neural, analytic], ignore_index=True)
    pair = panel.loc[panel.policy.isin([policy_a, policy_b])].pivot_table(index=["entry_date","maturity_bucket"], columns="policy", values="normalised_squared_error", aggfunc="first").dropna().reset_index()
    date_level = pair.groupby("entry_date", as_index=False).agg(mse_a=(policy_a,"mean"), mse_b=(policy_b,"mean")).sort_values("entry_date")
    a=date_level.mse_a.to_numpy(float); b=date_level.mse_b.to_numpy(float)
    block=calendar_weeks_to_date_observations(date_level.entry_date, mean_block_weeks)
    rng=np.random.default_rng(int(seed)+int(stable_hash("family_"+label)[:8],16)); rel=np.empty(int(repetitions))
    for j in range(int(repetitions)):
        ix=stationary_bootstrap_indices(len(a), block, rng); rel[j]=1-float(a[ix].mean())/max(float(b[ix].mean()),1e-12)
    alpha=0.05/max(int(family_size),1)
    base.update({"familywise_ci_low":float(np.quantile(rel,alpha/2)), "familywise_ci_high":float(np.quantile(rel,1-alpha/2)), "family_size":int(family_size)})
    return base


def hierarchical_seed_date_contrast(results, policy_a, policy_b, label, *, primary_cost=1.0,
                                    repetitions=10_000, mean_block_weeks=13, seed=20260713):
    use=results.loc[results.underlying_tc_bps.eq(float(primary_cost)) & results.policy.isin([policy_a,policy_b])].copy()
    use["entry_date"]=pd.to_datetime(use.entry_date, errors="coerce").dt.normalize()
    if (use.training_seed<0).any():
        # Replicate analytic policies across the neural seed set for paired seed resampling.
        neural_seeds=sorted(use.loc[use.training_seed>=0,"training_seed"].unique())
        analytic=use.loc[use.training_seed<0].copy(); copies=[]
        for s in neural_seeds:
            z=analytic.copy(); z["training_seed"]=int(s); copies.append(z)
        use=pd.concat([use.loc[use.training_seed>=0],*copies],ignore_index=True)
    panel=use.groupby(["training_seed","entry_date","maturity_bucket","policy"],as_index=False).normalised_squared_error.mean()
    pair=panel.pivot_table(index=["training_seed","entry_date","maturity_bucket"],columns="policy",values="normalised_squared_error",aggfunc="first").dropna().reset_index()
    pair=pair.groupby(["training_seed","entry_date"],as_index=False)[[policy_a,policy_b]].mean()
    seeds=sorted(pair.training_seed.unique()); dates=sorted(pair.entry_date.unique())
    if len(seeds)<2 or len(dates)<2:
        return {"comparison":label,"hierarchical_status":"INSUFFICIENT_SEEDS_OR_DATES"}
    block=calendar_weeks_to_date_observations(dates,mean_block_weeks); rng=np.random.default_rng(int(seed)+int(stable_hash(label)[:8],16)); vals=[]
    for _ in range(int(repetitions)):
        sampled_seeds=rng.choice(seeds,size=len(seeds),replace=True)
        date_ix=stationary_bootstrap_indices(len(dates),block,rng); sampled_dates=[dates[i] for i in date_ix]
        a=[];b=[]
        for s in sampled_seeds:
            z=pair.loc[pair.training_seed.eq(s)].set_index("entry_date")
            z=z.reindex(sampled_dates)
            a.append(float(z[policy_a].mean())); b.append(float(z[policy_b].mean()))
        vals.append(1-float(np.mean(a))/max(float(np.mean(b)),1e-12))
    vals=np.asarray(vals,float)
    return {"comparison":label,"policy_a":policy_a,"policy_b":policy_b,"seed_count":len(seeds),"date_count":len(dates),
            "hierarchical_point":float(np.mean(vals)),"hierarchical_ci_2_5pct":float(np.quantile(vals,.025)),
            "hierarchical_ci_97_5pct":float(np.quantile(vals,.975)),"hierarchical_fraction_nonpositive":float(np.mean(vals<=0)),
            "hierarchical_repetitions":int(repetitions),"hierarchical_status":"COMPLETE"}


# 10. Frozen final experiment configuration

In [50]:
@dataclass(frozen=True)
class FinalObservableConfig:
    protocol_version: str = FINAL_PROTOCOL_VERSION
    implementation_version: str = (
        "single_notebook_h035_lag_consistent_retraining_v7"
        if IS_PRELIMINARY_H035 else "single_notebook_h025_lag_consistent_final_v8"
    )
    diagnostic_reporting_version: str = ("argument_led_diagnostics_v7" if IS_PRELIMINARY_H035 else "argument_led_diagnostics_v8")
    run_profile: str = FINAL_RUN_PROFILE
    calibration_spec: str = RUN.calibration_spec
    calibration_tag: str = RUN.calibration_tag
    is_final_spec: bool = RUN.is_final_spec
    is_full_evidence_run: bool = bool(
        RUN.profile == "full" and not LEGACY_H035_REUSE and len(tuple(RUN_PROFILE["training_seeds"])) >= 10
    )
    legacy_h035_reuse: bool = LEGACY_H035_REUSE
    legacy_h035_output_dir: str = str(LEGACY_H035_OUTPUT_DIR)
    legacy_h035_expected_protocol_hash: str = RUN.legacy_h035_expected_protocol_hash
    evidence_status: str = (
        ("FULL_H035_LAG_CONSISTENT_RETRAIN" if is_full_evidence_run else "H035_LAG_CONSISTENT_RETRAIN_LIMITED_SEEDS")
        if IS_PRELIMINARY_H035
        else ("FULL_H025_LAG_CONSISTENT_FINAL" if is_full_evidence_run else "H025_LAG_CONSISTENT_LIMITED_SEEDS")
    )
    H: float = RUN.calibration_H
    reported_specification_H: float = RUN.calibration_H
    n_factors: int = 20
    rn_scale: float = 2.5
    history_length: int = 10
    training_seeds: tuple[int, ...] = tuple(RUN_PROFILE["training_seeds"])
    transaction_cost_bps: tuple[float, ...] = tuple(RUN_PROFILE["transaction_cost_bps"])
    primary_transaction_cost_bps: float = 1.0
    exact_train_paths_per_seed_row: int = int(RUN_PROFILE["exact_train_paths_per_seed_row"])
    exact_validation_paths_per_seed_row: int = int(RUN_PROFILE["exact_validation_paths_per_seed_row"])
    exact_bank_burn_in_days: int = 10
    simulation_seed_end: str = "2017-12-31"
    source_train_dates_per_bucket: int = int(RUN_PROFILE["source_train_dates_per_bucket"])
    source_validation_dates_per_bucket: int = int(RUN_PROFILE["source_validation_dates_per_bucket"])
    source_panel_seed: int = 20260727
    bootstrap_repetitions: int = int(RUN_PROFILE["bootstrap_repetitions"])
    bootstrap_block_weeks: int = 13
    bootstrap_sensitivity_weeks: tuple[int, ...] = tuple(RUN_PROFILE["bootstrap_sensitivity_weeks"])
    minimum_practical_relative_mse: float = 0.01
    bootstrap_seed: int = 20260711
    decision_export_costs: tuple[float, ...] = (1.0,)
    decision_export_policies: tuple[str, ...] = (
        "heston_hybrid_current_small", "lifted_hybrid_current_small", "mixed_hybrid_current_small",
        "mixed_hybrid_current_matched", "mixed_hybrid_lag", "mixed_hybrid_gru",
        "black_scholes_delta", "black_scholes_no_trade", "no_hedge",
    )
    regime_reference_feature: str = "rv_20d"
    regime_quantiles: tuple[float, float] = (1.0 / 3.0, 2.0 / 3.0)
    force_rebuild_banks: bool = EXECUTION.force_rebuild("banks")
    force_retrain: bool = EXECUTION.force_rebuild("policies")
    run_training: bool = EXECUTION.can_rebuild("policies")
    run_evaluation: bool = EXECUTION.can_rebuild("evaluation")

FINAL_CFG = FinalObservableConfig()
TRAIN_CFG = TrainingConfig(
    history_length=FINAL_CFG.history_length,
    common_pretrain_steps=int(RUN_PROFILE["common_pretrain_steps"]),
    hybrid_head_steps=int(RUN_PROFILE["hybrid_head_steps"]),
    hybrid_full_steps=int(RUN_PROFILE["hybrid_full_steps"]),
    batch_episodes=int(RUN_PROFILE["batch_episodes"]),
    validation_interval=int(RUN_PROFILE["validation_interval"]),
    early_stopping_patience=int(RUN_PROFILE["early_stopping_patience"]),
    transaction_cost_bps=tuple(RUN_PROFILE["transaction_cost_bps"]),
    execution_lag_steps=EXECUTION_LAG_STEPS,
)

assert abs(FINAL_CFG.H - RUN.calibration_H) < 1e-12
assert FINAL_CFG.n_factors == 20
assert abs(FINAL_CFG.rn_scale - 2.5) < 1e-12
assert FINAL_CFG.run_profile in {"smoke", "full"}
if FINAL_CFG.is_full_evidence_run:
    assert len(FINAL_CFG.training_seeds) >= 10
    assert FINAL_CFG.transaction_cost_bps == (1.0, 5.0, 10.0)
assert_observable_schema(CURRENT_FEATURES, HISTORY_FEATURES)

_scientific_final_config = asdict(FINAL_CFG)
for _execution_key in ("force_rebuild_banks", "force_retrain", "run_training", "run_evaluation"):
    _scientific_final_config.pop(_execution_key, None)

FINAL_PROTOCOL_PAYLOAD = {
    "final_config": _scientific_final_config,
    "training_config": asdict(TRAIN_CFG),
    "current_features": CURRENT_FEATURES,
    "history_features": HISTORY_FEATURES,
    "calibration_methodology_version": CFG.methodology_version,
    "policy_access_to_latent_simulator_state": "None",
    "timing_contract": {
        "information_time": OBSERVATION_TIME_CONVENTION,
        "decision_time": DECISION_TIME_CONVENTION,
        "execution_time": EXECUTION_TIME_CONVENTION,
        "execution_price": EXECUTION_PRICE_CONVENTION,
        "execution_window": EXECUTION_WINDOW_CONVENTION,
        "execution_lag_steps": EXECUTION_LAG_STEPS,
    },
}
# Do not alter the H035 payload: its current checkpoints must remain loadable.
if IS_FINAL_H025_SPEC:
    FINAL_PROTOCOL_PAYLOAD["model_selection_weighting"] = MODEL_SELECTION_WEIGHTING
    FINAL_PROTOCOL_PAYLOAD["state_pricer_version"] = "normalized_gl_with_carr_madan_low_variance_v2"

FINAL_PROTOCOL_HASH = stable_hash(FINAL_PROTOCOL_PAYLOAD)
(FINAL_RESULTS_DIR / "frozen_final_protocol.json").write_text(json.dumps(FINAL_PROTOCOL_PAYLOAD, indent=2, default=str))
print("Protocol hash:", FINAL_PROTOCOL_HASH)
print("Run profile:", FINAL_CFG.run_profile)
print("Calibration specification:", FINAL_CFG.calibration_spec)
print("Actual Lifted-Heston H:", FINAL_CFG.H)
print("Model-selection weighting:", MODEL_SELECTION_WEIGHTING)
print("Evidence status:", FINAL_CFG.evidence_status)
print("Meets expanded-seed robustness gate:", FINAL_CFG.is_full_evidence_run)
print("Policy access to latent simulator state: None")
display(pd.DataFrame([asdict(FINAL_CFG)]).T.rename(columns={0: "value"}))

CALIBRATION_COMPONENT_PAYLOAD = {
    "calibration_spec": FINAL_CFG.calibration_spec,
    "H": FINAL_CFG.H,
    "n_factors": FINAL_CFG.n_factors,
    "rn_scale": FINAL_CFG.rn_scale,
    "calibration_methodology_version": CFG.methodology_version,
    "calibration_source_cutoff": FINAL_CFG.simulation_seed_end,
}
CALIBRATION_COMPONENT_HASH = stable_hash(CALIBRATION_COMPONENT_PAYLOAD)
DOWNSTREAM_COMPONENT_HASH = FINAL_PROTOCOL_HASH
print("Calibration component hash (safe reuse boundary):", CALIBRATION_COMPONENT_HASH)
print("Downstream component hash:", DOWNSTREAM_COMPONENT_HASH)


Protocol hash: a0419a1b88cef6ea9112be145192fc14c4ea72fb4240a21ba53ea54ed266ae28
Run profile: full
Calibration specification: preliminary_h035
Actual Lifted-Heston H: 0.35
Model-selection weighting: episode_weighted
Evidence status: H035_LAG_CONSISTENT_RETRAIN_LIMITED_SEEDS
Meets expanded-seed robustness gate: False
Policy access to latent simulator state: None


,value
protocol_version,observable_heston_lifted_h035_lag_consistent_r...
implementation_version,single_notebook_h035_lag_consistent_retraining_v7
diagnostic_reporting_version,argument_led_diagnostics_v7
run_profile,full
calibration_spec,preliminary_h035
calibration_tag,H035
is_final_spec,False
is_full_evidence_run,False
legacy_h035_reuse,False
legacy_h035_output_dir,"C:\Users\lhkke\OneDrive - Queen Mary, Universi..."


Calibration component hash (safe reuse boundary): fb9eef804cd62bd4fac031c5cf5199e37a3eba699ff91f96e44bb96be6e01517
Downstream component hash: a0419a1b88cef6ea9112be145192fc14c4ea72fb4240a21ba53ea54ed266ae28


# 11. Observability, chronology and architecture gates

In [51]:
ARCHITECTURE_MANIFEST = architecture_manifest(TRAIN_CFG)
manifest = ARCHITECTURE_MANIFEST.set_index("architecture")
lag_params = int(manifest.loc["lag_stack_mlp", "parameters"])
gru_params = int(manifest.loc["observable_gru", "parameters"])
matched_params = int(manifest.loc["current_surface_mlp_matched", "parameters"])
small_params = int(manifest.loc["current_surface_mlp", "parameters"])
capacity_audit = pd.DataFrame([
    {"comparison":"matched_current_vs_lag","parameters_a":matched_params,"parameters_b":lag_params,"relative_gap":abs(matched_params-lag_params)/max(lag_params,1)},
    {"comparison":"gru_vs_lag","parameters_a":gru_params,"parameters_b":lag_params,"relative_gap":abs(gru_params-lag_params)/max(lag_params,1)},
    {"comparison":"small_current_vs_matched_current","parameters_a":small_params,"parameters_b":matched_params,"relative_gap":abs(small_params-matched_params)/max(matched_params,1)},
])
capacity_audit["target"] = [0.02,0.02,np.nan]
if LEGACY_H035_REUSE:
    capacity_audit["status"] = ["NOT_EXECUTED_LEGACY_REUSE", "PASS", "NOT_EXECUTED_LEGACY_REUSE"]
    capacity_audit["note"] = [
        "The V5.3 checkpoint set has no parameter-matched current-state MLP.",
        "The imported lag-stack and GRU retain their original matched parameter counts.",
        "Capacity effect is unavailable in legacy checkpoint-reuse mode.",
    ]
else:
    capacity_audit["status"] = ["PASS" if x<=0.02 else "FAIL" for x in capacity_audit.relative_gap[:2]] + ["DESCRIPTIVE"]
    capacity_audit["note"] = ""
ARCHITECTURE_MANIFEST.to_csv(FINAL_TABLE_DIR / "architecture_manifest.csv", index=False)
capacity_audit.to_csv(FINAL_TABLE_DIR / "capacity_matching_audit.csv", index=False)
display(ARCHITECTURE_MANIFEST); display(capacity_audit)
if not LEGACY_H035_REUSE and capacity_audit.iloc[:2].status.ne("PASS").any():
    raise AssertionError("Capacity matching failed the 2% target.")


,architecture,parameters,history_length
0,current_surface_mlp,5761,0
1,current_surface_mlp_matched,14716,0
2,lag_stack_mlp,14721,10
3,observable_gru,14692,10


,comparison,parameters_a,parameters_b,relative_gap,target,status,note
0,matched_current_vs_lag,14716,14721,0.000340,0.02,PASS,
1,gru_vs_lag,14692,14721,0.001970,0.02,PASS,
2,small_current_vs_matched_current,5761,14716,0.608521,NaN,DESCRIPTIVE,


# 12. Construct chronology-safe strictly lagged historical histories

In [52]:
MARKET_BY_ROLE = {
    "finetune": [],
    "validation": [],
    "test": [],
    "confirmatory": [],
}

for role, episodes in MARKET_EPISODES_BY_ROLE.items():
    for episode in episodes:
        episode_id = str(episode["episode_id"])

        if episode_id not in MARKET_PATHS:
            raise KeyError(
                f"Historical path for episode {episode_id!r} "
                "is missing from MARKET_PATHS."
            )

        MARKET_BY_ROLE[role].append(
            market_episode_with_external_history(
                MARKET_PATHS[episode_id],
                episode,
                daily_surface_state,
                history_length=FINAL_CFG.history_length,
            )
        )

for role, episodes in MARKET_BY_ROLE.items():
    for episode in episodes:
        assert_episode_observability(
            episode,
            FINAL_CFG.history_length,
        )
    print(role, len(episodes))

MARKET_TRAIN = MARKET_BY_ROLE["finetune"]
MARKET_VALIDATION = MARKET_BY_ROLE["validation"]
MARKET_TEST = MARKET_BY_ROLE["test"]
MARKET_CONFIRMATORY = MARKET_BY_ROLE["confirmatory"]

from pathlib import Path
import torch

EXPECTED_PROTOCOL_PREFIX = "a0419a1b88ce"

%env FINAL_CALIBRATION_SPEC=preliminary_h035
%env FINAL_RUN_PROFILE=full
%env FINAL_REBUILD_FROM=reporting

if FINAL_PROTOCOL_HASH[:12] != EXPECTED_PROTOCOL_PREFIX:
    raise RuntimeError(
        f"Wrong H035 protocol: {FINAL_PROTOCOL_HASH[:12]}. "
        f"Expected {EXPECTED_PROTOCOL_PREFIX}."
    )

role_counts = {
    "finetune": len(MARKET_TRAIN),
    "validation": len(MARKET_VALIDATION),
    "test": len(MARKET_TEST),
    "confirmatory": len(MARKET_CONFIRMATORY),
}

market_episodes = [
    *MARKET_TRAIN,
    *MARKET_VALIDATION,
    *MARKET_TEST,
    *MARKET_CONFIRMATORY,
]

if not market_episodes:
    raise RuntimeError("No market episodes were constructed.")

for episode in market_episodes:
    assert_episode_observability(
        episode,
        FINAL_CFG.history_length,
    )

market_cache_dir = (
    Path("outputs_observable_h035_lag_consistent_retrained_full")
    / "observable_market_cache"
)
market_cache_dir.mkdir(parents=True, exist_ok=True)

market_cache_path = market_cache_dir / (
    f"market_observable_episodes_H035_"
    f"{FINAL_PROTOCOL_HASH[:12]}.pt"
)

payload = {
    "metadata": {
        "calibration_spec": "H035",
        "calibration_hash": CALIBRATION_COMPONENT_HASH,
        "protocol_hash": FINAL_PROTOCOL_HASH,
        "H": 0.35,
        "execution_mode": "synchronised_delayed_entry",
        "policy_access_to_latent_simulator_state": "None",
        "role_counts": role_counts,
    },
    "episodes": market_episodes,
}

temporary_path = market_cache_path.with_suffix(".pt.tmp")
torch.save(payload, temporary_path)
temporary_path.replace(market_cache_path)

print({
    "saved": str(market_cache_path.resolve()),
    "episodes": len(market_episodes),
    "role_counts": role_counts,
})

HOLDOUT_STATUS = pd.DataFrame([{
    "candidate_period": "2025",
    "episodes": len(MARKET_CONFIRMATORY),
    "status": (
        "AVAILABLE_UNTOUCHED_CONFIRMATORY"
        if len(MARKET_CONFIRMATORY) > 0
        else "UNAVAILABLE_IN_SUPPLIED_OPTION_AND_INDEX_OVERLAP"
    ),
    "permitted_use": (
        "one-time frozen evaluation only"
        if len(MARKET_CONFIRMATORY) > 0
        else "none; 2021-2024 remains post-development frozen evidence"
    ),
}])

HOLDOUT_STATUS.to_csv(
    FINAL_TABLE_DIR / "confirmatory_holdout_status.csv",
    index=False,
)
display(HOLDOUT_STATUS)

finetune 433
validation 219
test 929
confirmatory 0
env: FINAL_CALIBRATION_SPEC=preliminary_h035
env: FINAL_RUN_PROFILE=full
env: FINAL_REBUILD_FROM=reporting
{'saved': 'C:\\Users\\lhkke\\OneDrive - Queen Mary, University of London\\Documents\\Dissertation\\Deep-Hedging-under-Lifted-Heston-MSc-Dissertation-\\outputs_observable_h035_lag_consistent_retrained_full\\observable_market_cache\\market_observable_episodes_H035_a0419a1b88ce.pt', 'episodes': 1581, 'role_counts': {'finetune': 433, 'validation': 219, 'test': 929, 'confirmatory': 0}}


,candidate_period,episodes,status,permitted_use
0,2025,0,UNAVAILABLE_IN_SUPPLIED_OPTION_AND_INDEX_OVERLAP,none; 2021-2024 remains post-development froze...


# 13. Build exact Heston and Lifted-Heston observable episode banks

In [53]:
from functools import lru_cache
import warnings

from scipy.integrate import IntegrationWarning, quad
from collections import Counter


HESTON_STATE_PRICER_VERSION = "normalized_gl_with_carr_madan_low_variance_v2"
HESTON_LOW_TOTAL_VARIANCE_THRESHOLD = 6e-4

HESTON_PRICER_ROUTE_COUNTS = Counter()
HESTON_PRICER_GROUP_COUNTS = Counter()
HESTON_PRICER_FAILURE_COUNTS = Counter()

def _heston_audit_maturity_bucket(tau: float) -> str:
    days = 365.25 * float(tau)
    if days <= 10: return "<=10d"
    if days <= 21: return "11-21d"
    if days <= 45: return "22-45d"
    if days <= 75: return "46-75d"
    return ">75d"

def _heston_audit_moneyness_bucket(S: float, K: float) -> str:
    x = math.log(float(K) / float(S))
    if x < -0.05: return "ITM_call"
    if x > 0.05: return "OTM_call"
    return "near_ATM"

def _record_heston_pricer_audit(*, route: str, S: float, K: float, tau: float, adaptive: bool, projected: bool) -> None:
    maturity = _heston_audit_maturity_bucket(tau)
    moneyness = _heston_audit_moneyness_bucket(S, K)
    HESTON_PRICER_ROUTE_COUNTS["total_calls"] += 1
    HESTON_PRICER_ROUTE_COUNTS[route] += 1
    if adaptive: HESTON_PRICER_ROUTE_COUNTS["adaptive_retry"] += 1
    if projected: HESTON_PRICER_ROUTE_COUNTS["bound_projection"] += 1
    HESTON_PRICER_GROUP_COUNTS[(route, maturity, moneyness)] += 1


@lru_cache(maxsize=16)
def _cached_heston_pricer(n_quad: int) -> HestonPricer:
    """Return a cached semi-analytic Heston pricer."""
    n_quad = int(n_quad)

    if n_quad <= 0:
        raise ValueError(
            f"Heston quadrature-node count must be positive; received {n_quad}."
        )

    return HestonPricer(n_quad=n_quad)


@lru_cache(maxsize=8)
def _cached_legendre_rule(n_nodes: int) -> tuple[np.ndarray, np.ndarray]:
    """Cached Gauss-Legendre rule on [-1, 1] for the stable state-pricing fallback."""
    n_nodes = int(n_nodes)
    if n_nodes <= 0:
        raise ValueError(f"Legendre-node count must be positive; received {n_nodes}.")
    nodes, weights = np.polynomial.legendre.leggauss(n_nodes)
    return np.asarray(nodes, float), np.asarray(weights, float)


def _heston_state_params(
    params: HestonParams,
    current_variance: float,
) -> HestonParams:
    """
    Replace the calibrated initial variance with the simulated variance
    observed internally at the current synthetic-path time.

    This state is used only to construct synthetic contemporaneous option
    prices. It is never returned as a policy feature.
    """
    variance = float(current_variance)

    if not np.isfinite(variance):
        raise FloatingPointError(
            f"Non-finite Heston state variance: {variance!r}."
        )

    if variance < -1e-12:
        raise FloatingPointError(
            "Heston state variance is materially negative: "
            f"{variance:.6e}."
        )

    return HestonParams(
        kappa=float(params.kappa),
        theta=float(params.theta),
        xi=float(params.xi),
        rho=float(params.rho),
        v0=max(variance, 0.0),
    )


def _expected_average_heston_variance(
    params: HestonParams,
    current_variance: float,
    tau: float,
) -> float:
    """Conditional expected average variance over the remaining horizon."""
    tau = max(float(tau), 0.0)
    kappa = max(float(params.kappa), 0.0)
    theta = max(float(params.theta), 0.0)
    variance = max(float(current_variance), 0.0)

    if tau <= 1e-14 or kappa * tau <= 1e-8:
        return max(variance, 1e-10)

    mean_reversion_weight = -math.expm1(-kappa * tau) / (kappa * tau)
    average = theta + (variance - theta) * mean_reversion_weight
    return max(float(average), 1e-10)


def _heston_call_price_carr_madan_fixed(
    S: float,
    K: float,
    tau: float,
    r: float,
    q: float,
    state_params: HestonParams,
) -> float:
    """
    Stable fixed-node Carr-Madan price used in short-dated/low-variance states.

    The calculation is performed on S=1 and K/S, then rescaled by homogeneity.
    This avoids the unstable P1-P2 cancellation that can make the legacy
    Gauss-Laguerre formula return negative short-dated call prices.
    """
    alpha = 0.75
    average_variance = _expected_average_heston_variance(
        state_params, state_params.v0, tau
    )
    total_variance = max(average_variance * tau, 1e-12)

    # The short-maturity Fourier tail decays slowly. Scale the truncation range
    # with total variance and use a denser rule only where it is actually needed.
    u_max = float(np.clip(
        1.25 * math.sqrt(2.0 * math.log(1e14) / total_variance),
        180.0,
        3000.0,
    ))
    n_nodes = 320 if u_max > 1400.0 else 256
    base_nodes, base_weights = _cached_legendre_rule(n_nodes)
    u = 0.5 * u_max * (base_nodes + 1.0)
    weights = 0.5 * u_max * base_weights

    normalized_strike = float(K / S)
    log_strike = math.log(normalized_strike)
    z = u - 1j * (alpha + 1.0)

    with np.errstate(over="ignore", divide="ignore", invalid="ignore"):
        characteristic = HestonPricer._cf(
            z, 1.0, tau, r, q, state_params
        )
        denominator = (
            alpha * alpha + alpha - u * u
            + 1j * (2.0 * alpha + 1.0) * u
        )
        integrand = np.real(
            np.exp(-1j * u * log_strike)
            * math.exp(-r * tau)
            * characteristic
            / denominator
        )
        normalized_price = (
            math.exp(-alpha * log_strike)
            * float(np.sum(weights * integrand))
            / math.pi
        )

    return float(S * normalized_price)


def _heston_call_price_carr_madan_adaptive(
    S: float,
    K: float,
    tau: float,
    r: float,
    q: float,
    state_params: HestonParams,
) -> float:
    """Slow emergency fallback for a state that still fails the fixed-node rule."""
    alpha = 0.75
    normalized_strike = float(K / S)
    log_strike = math.log(normalized_strike)

    def integrand(u: float) -> float:
        z = complex(float(u), -(alpha + 1.0))
        characteristic = HestonPricer._cf(
            z, 1.0, tau, r, q, state_params
        )
        denominator = (
            alpha * alpha + alpha - float(u) ** 2
            + 1j * (2.0 * alpha + 1.0) * float(u)
        )
        value = (
            np.exp(-1j * float(u) * log_strike)
            * math.exp(-r * tau)
            * characteristic
            / denominator
        )
        return float(np.real(value))

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", IntegrationWarning)
        integral, _ = quad(
            integrand,
            0.0,
            np.inf,
            epsabs=1e-9,
            epsrel=1e-8,
            limit=400,
        )

    normalized_price = (
        math.exp(-alpha * log_strike) * float(integral) / math.pi
    )
    return float(S * normalized_price)


def _heston_call_price_state_one(
    S: float,
    K: float,
    tau: float,
    r: float,
    q: float,
    params: HestonParams,
    current_variance: float,
    n_quad: int,
) -> float:
    """
    Price one call conditionally on the current simulated Heston variance.

    Normal states use the calibrated Gauss-Laguerre implementation. Very
    short-dated/low-integrated-variance states use a Carr-Madan rule, because
    the P1/P2 representation is numerically unreliable in precisely the regime
    reached near the end of synthetic hedge episodes.
    """
    S = float(S)
    K = float(K)
    tau = float(tau)
    r = float(r)
    q = float(q)

    if not np.isfinite([S, K, tau, r, q]).all():
        raise FloatingPointError(
            "Heston state pricing received non-finite market inputs."
        )

    if S <= 0.0 or K <= 0.0:
        raise ValueError(
            f"Heston state pricing requires positive S and K; S={S}, K={K}."
        )

    if tau <= 1e-10:
        _record_heston_pricer_audit(route="intrinsic", S=S, K=K, tau=max(tau, 0.0), adaptive=False, projected=False)
        return float(max(S - K, 0.0))

    state_params = _heston_state_params(params, current_variance)
    average_variance = _expected_average_heston_variance(
        state_params, current_variance, tau
    )
    total_variance = average_variance * tau

    initial_route = "carr_madan_fixed" if total_variance < HESTON_LOW_TOTAL_VARIANCE_THRESHOLD else "normalized_gauss_laguerre"
    adaptive_used = False
    projected = False

    if total_variance < HESTON_LOW_TOTAL_VARIANCE_THRESHOLD:
        price = _heston_call_price_carr_madan_fixed(
            S, K, tau, r, q, state_params
        )
    else:
        # Normalize by spot before applying the legacy quadrature. Heston prices
        # are homogeneous, and the normalized form is better conditioned.
        pricer = _cached_heston_pricer(int(n_quad))
        normalized_price = pricer.call_price_one(
            1.0,
            K / S,
            tau,
            r,
            q,
            state_params,
        )
        price = S * float(normalized_price)

    lower = max(
        S * math.exp(-q * tau)
        - K * math.exp(-r * tau),
        0.0,
    )
    upper = S * math.exp(-q * tau)
    tolerance = 3e-3 * max(1.0, upper)

    invalid = (
        not np.isfinite(price)
        or price < lower - tolerance
        or price > upper + tolerance
    )
    if invalid:
        adaptive_used = True
        price = _heston_call_price_carr_madan_adaptive(
            S, K, tau, r, q, state_params
        )

    if not np.isfinite(price):
        raise FloatingPointError(
            "Non-finite Heston state price after the stable fallback."
        )

    # Project only after a stable reprice, and only within the notebook's
    # pre-existing numerical no-arbitrage tolerance. The broken legacy value
    # is never clipped directly.
    repair_tolerance = 3e-3 * max(1.0, upper)
    if lower - repair_tolerance <= price < lower:
        price = lower
        projected = True
    if upper < price <= upper + repair_tolerance:
        price = upper
        projected = True

    if price < lower - repair_tolerance or price > upper + repair_tolerance:
        HESTON_PRICER_FAILURE_COUNTS["post_retry_bound_failure"] += 1
        raise FloatingPointError(
            "Heston state price violates no-arbitrage bounds after stable retry: "
            f"price={price:.8f}, "
            f"lower={lower:.8f}, "
            f"upper={upper:.8f}, "
            f"variance={float(current_variance):.8f}, "
            f"tau={tau:.8f}, "
            f"spot={S:.8f}, "
            f"strike={K:.8f}."
        )

    _record_heston_pricer_audit(
        route=initial_route, S=S, K=K, tau=tau,
        adaptive=adaptive_used, projected=projected,
    )
    return float(price)

In [54]:
REQUIRED_SYNTHETIC_PRICING_SYMBOLS = {
    "HestonParams",
    "HestonPricer",
    "_cached_heston_pricer",
    "_heston_state_params",
    "_heston_call_price_state_one",
    "lifted_variable_strike_call_prices",
    "model_call_prices_variable_strike",
}

missing_symbols = sorted(
    name
    for name in REQUIRED_SYNTHETIC_PRICING_SYMBOLS
    if name not in globals()
)

if missing_symbols:
    raise RuntimeError(
        "Synthetic pricing adapter is incomplete. Missing symbols: "
        f"{missing_symbols}"
    )

print("Synthetic Heston and Lifted pricing adapters are available.")

Synthetic Heston and Lifted pricing adapters are available.


In [55]:
def _eligible_source_universe() -> pd.DataFrame:
    """
    Build the chronology-safe source-date universe.

    Valid candidate rows are explicitly labelled
    ``accepted_pre_feasibility`` by build_seed_candidate_universe().
    They must not be mistaken for excluded rows.
    """
    raw_universe = build_seed_candidate_universe(
        surface_panel,
        heston_calibration,
        lifted_calibration,
        CFG,
    ).copy()

    if raw_universe.empty:
        raise RuntimeError(
            "build_seed_candidate_universe returned no rows before "
            "chronology or feasibility filtering."
        )

    raw_universe["date"] = (
        pd.to_datetime(raw_universe["date"], errors="coerce")
        .dt.normalize()
    )

    if raw_universe["date"].isna().all():
        raise RuntimeError(
            "The source candidate universe contains no parseable dates."
        )

    # build_seed_candidate_universe uses this explicit label for rows
    # that passed all pre-feasibility gates.
    if "exclusion_reason" in raw_universe.columns:
        reason = (
            raw_universe["exclusion_reason"]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        accepted_mask = reason.isin(
            {
                "",
                "accepted_pre_feasibility",
            }
        )

        universe = raw_universe.loc[accepted_mask].copy()
    else:
        universe = raw_universe.copy()

    if universe.empty:
        reason_counts = (
            raw_universe["exclusion_reason"]
            .fillna("<missing>")
            .astype(str)
            .value_counts(dropna=False)
            .rename_axis("exclusion_reason")
            .reset_index(name="rows")
            if "exclusion_reason" in raw_universe.columns
            else pd.DataFrame()
        )

        if not reason_counts.empty:
            display(reason_counts)

        raise RuntimeError(
            "No source rows passed the pre-feasibility gates. "
            "See the displayed exclusion-reason counts."
        )

    source_cutoff = pd.Timestamp(FINAL_CFG.simulation_seed_end)

    universe = universe.loc[
        universe["date"].le(source_cutoff)
    ].copy()

    required = {
        "seed_id",
        "date",
        "maturity_bucket",
        "strike",
        "T",
        "dte",
        "spot_ref",
        "straddle_mid",
        "iv_market",
        "r",
        "q",
    }

    missing = sorted(required.difference(universe.columns))
    if missing:
        raise KeyError(
            "Chronology-safe source universe is missing columns: "
            f"{missing}"
        )

    if universe.empty:
        accepted_dates = raw_universe.loc[
            raw_universe.get(
                "exclusion_reason",
                pd.Series("", index=raw_universe.index),
            )
            .fillna("")
            .astype(str)
            .str.strip()
            .isin({"", "accepted_pre_feasibility"}),
            "date",
        ].dropna()

        accepted_min = (
            accepted_dates.min().date()
            if not accepted_dates.empty
            else None
        )
        accepted_max = (
            accepted_dates.max().date()
            if not accepted_dates.empty
            else None
        )

        raise RuntimeError(
            "Valid source rows exist, but none satisfy the chronology "
            f"cutoff {source_cutoff.date()}. "
            f"Accepted candidate range: {accepted_min} to {accepted_max}."
        )

    # Select one deterministic representative contract for each
    # calibration date and maturity bucket.
    universe["representative_hash"] = universe["seed_id"].map(
        lambda value: stable_hash(
            {
                "seed_id": str(value),
                "panel_seed": FINAL_CFG.source_panel_seed,
            }
        )
    )

    universe = (
        universe
        .sort_values(
            [
                "maturity_bucket",
                "date",
                "representative_hash",
                "seed_id",
            ]
        )
        .groupby(
            ["maturity_bucket", "date"],
            group_keys=False,
        )
        .head(1)
        .drop(columns=["representative_hash"])
        .reset_index(drop=True)
    )

    coverage = (
        universe
        .groupby("maturity_bucket", as_index=False)
        .agg(
            rows=("seed_id", "size"),
            dates=("date", "nunique"),
            date_min=("date", "min"),
            date_max=("date", "max"),
        )
        .sort_values("maturity_bucket")
    )

    display(coverage)

    return (
        universe
        .sort_values(
            ["maturity_bucket", "date", "seed_id"]
        )
        .reset_index(drop=True)
    )


def _select_chronological_source_panel() -> pd.DataFrame:
    """Select matched source train/validation dates within pre-2018 data only.

    Dates are stratified chronologically within each maturity bucket. One row per
    stratum is selected by a prespecified stable hash. The earlier selected dates
    form simulation training and the latest selected dates form simulation
    validation, so even the synthetic checkpoint rule respects chronology.
    """
    universe = _eligible_source_universe()
    total = FINAL_CFG.source_train_dates_per_bucket + FINAL_CFG.source_validation_dates_per_bucket
    selected = []
    for bucket in ("1W", "2W", "1M", "2M", "3M"):
        g = universe.loc[universe["maturity_bucket"].astype(str).eq(bucket)].copy()
        g = g.sort_values(["date", "seed_id"]).reset_index(drop=True)
        if len(g) < total:
            raise RuntimeError(f"Need {total} pre-2018 rows for {bucket}; found {len(g)}.")
        g["source_stratum"] = np.minimum((np.arange(len(g)) * total // len(g)).astype(int), total - 1)
        g["source_hash"] = g["seed_id"].map(
            lambda value: stable_hash({"seed_id": str(value), "panel_seed": FINAL_CFG.source_panel_seed})
        )
        one = g.sort_values(["source_stratum", "source_hash", "date"]).groupby("source_stratum", group_keys=False).head(1)
        one = one.sort_values("date").reset_index(drop=True)
        one["source_role"] = ["train"] * FINAL_CFG.source_train_dates_per_bucket + ["validation"] * FINAL_CFG.source_validation_dates_per_bucket
        selected.append(one)
    panel = pd.concat(selected, ignore_index=True)
    if panel.duplicated(["maturity_bucket", "date"]).any():
        raise AssertionError("Source panel contains duplicate calibration dates within a maturity bucket.")
    if pd.to_datetime(panel["date"]).max() > pd.Timestamp(FINAL_CFG.simulation_seed_end):
        raise AssertionError("Source panel contains post-2017 dates.")
    counts = panel.groupby(["source_role", "maturity_bucket"]).size().unstack(fill_value=0)
    expected_train = FINAL_CFG.source_train_dates_per_bucket
    expected_val = FINAL_CFG.source_validation_dates_per_bucket
    if not (counts.loc["train"] == expected_train).all() or not (counts.loc["validation"] == expected_val).all():
        raise AssertionError(f"Source-panel quotas failed:\n{counts}")
    panel.to_csv(FINAL_TABLE_DIR / "source_seed_panel.csv", index=False)
    return panel


SOURCE_SEED_PANEL = _select_chronological_source_panel()
SOURCE_PANEL_HASH = stable_hash(
    SOURCE_SEED_PANEL[["seed_id", "date", "maturity_bucket", "source_role"]].astype(str).to_dict("records")
)
display(SOURCE_SEED_PANEL.groupby(["source_role", "maturity_bucket"]).agg(rows=("seed_id", "size"), date_min=("date", "min"), date_max=("date", "max")).reset_index())


def _seed_rows(split: str) -> pd.DataFrame:
    role = "validation" if split == "validation" else "train"
    return SOURCE_SEED_PANEL.loc[SOURCE_SEED_PANEL["source_role"].eq(role)].copy()


def _pre2018_spread_pool() -> np.ndarray:
    if "date" not in cached_global.columns or "median_rel_spread" not in cached_global.columns:
        raise KeyError("Chronology-safe spread randomisation requires dated median_rel_spread observations.")
    dated = cached_global.copy()
    dated["date"] = pd.to_datetime(dated["date"]).dt.normalize()
    values = pd.to_numeric(
        dated.loc[dated["date"].le(pd.Timestamp(FINAL_CFG.simulation_seed_end)), "median_rel_spread"],
        errors="coerce",
    ).dropna().clip(0.001, 0.50).to_numpy(float)
    if not len(values):
        raise RuntimeError("No pre-2018 spread observations are available for the synthetic source bank.")
    return values


SOURCE_SPREAD_POOL = _pre2018_spread_pool()


def build_exact_observable_bank(world: str, split: str, paths_per_seed_row: int) -> list[dict[str, Any]]:
    cache_path = FINAL_CACHE_DIR / (
        f"exact_observable_{world}_{split}_{FINAL_CFG.calibration_tag}_n{paths_per_seed_row}_"
        f"{SOURCE_PANEL_HASH[:10]}_{FINAL_PROTOCOL_HASH[:12]}.pt"
    )
    expected_metadata = {
        "protocol_hash": FINAL_PROTOCOL_HASH,
        "run_profile": FINAL_CFG.run_profile,
        "world": world,
        "split": split,
        "H": FINAL_CFG.H,
        "n_factors": FINAL_CFG.n_factors,
        "rn_scale": FINAL_CFG.rn_scale,
        "paths_per_seed_row": int(paths_per_seed_row),
        "source_panel_hash": SOURCE_PANEL_HASH,
        "policy_access_to_latent_simulator_state": "None",
        "information_time": OBSERVATION_TIME_CONVENTION,
        "decision_time": DECISION_TIME_CONVENTION,
        "execution_time": EXECUTION_TIME_CONVENTION,
        "execution_price": EXECUTION_PRICE_CONVENTION,
        "execution_window": EXECUTION_WINDOW_CONVENTION,
        "execution_lag_steps": EXECUTION_LAG_STEPS,
        "heston_state_pricer_version": HESTON_STATE_PRICER_VERSION,
        "bank_builder_version": "row_checkpoint_v1",
    }
    if cache_path.exists() and not FINAL_CFG.force_rebuild_banks:
        try:
            payload = torch.load(cache_path, map_location="cpu", weights_only=False)
            if not isinstance(payload, dict) or "episodes" not in payload or "metadata" not in payload:
                raise ValueError("legacy or unmanifested episode bank")
            observed = payload["metadata"]
            mismatched = {
                key: {"expected": value, "observed": observed.get(key)}
                for key, value in expected_metadata.items()
                if observed.get(key) != value
            }
            if mismatched:
                raise ValueError(f"metadata mismatch: {mismatched}")
            episodes = payload["episodes"]
            for episode in episodes:
                assert_episode_observability(episode, FINAL_CFG.history_length)
            print("Loaded", cache_path, "episodes=", len(episodes))
            return episodes
        except Exception as exc:
            if not EXECUTION.can_rebuild("banks"):
                raise RuntimeError(
                    f"Observable bank {cache_path} is invalid and strict reuse forbids rebuilding."
                ) from exc
            print(f"[observable bank] cache invalid; rebuilding {cache_path.name}. Reason: {exc}")

    rows = _seed_rows(split)
    partial_path = cache_path.with_name(cache_path.stem + "_partial.pt")
    episodes: list[dict[str, Any]] = []
    completed_rows = 0
    engine = globals()

    # Resume only an exactly matching partial rebuild. A stale partial is
    # discarded rather than blended into a different protocol.
    if partial_path.exists():
        try:
            partial = torch.load(partial_path, map_location="cpu", weights_only=False)
            if not isinstance(partial, dict):
                raise ValueError("partial checkpoint is not a dictionary")
            observed = partial.get("metadata", {})
            mismatched = {
                key: {"expected": value, "observed": observed.get(key)}
                for key, value in expected_metadata.items()
                if observed.get(key) != value
            }
            if mismatched:
                raise ValueError(f"partial metadata mismatch: {mismatched}")
            completed_rows = int(partial.get("completed_rows", 0))
            episodes = list(partial.get("episodes", []))
            expected_episode_count = completed_rows * int(paths_per_seed_row)
            if not 0 <= completed_rows <= len(rows):
                raise ValueError(f"invalid completed_rows={completed_rows}")
            if len(episodes) != expected_episode_count:
                raise ValueError(
                    "partial episode count mismatch: "
                    f"expected {expected_episode_count}, observed {len(episodes)}"
                )
            for episode in episodes:
                assert_episode_observability(episode, FINAL_CFG.history_length)
            if completed_rows:
                print(
                    f"[observable bank] resuming {world}/{split} from "
                    f"row {completed_rows + 1}/{len(rows)}; episodes={len(episodes)}"
                )
        except Exception as exc:
            print(f"[observable bank] discarding invalid partial {partial_path.name}: {exc}")
            partial_path.unlink(missing_ok=True)
            episodes = []
            completed_rows = 0

    checkpoint_every_rows = 5
    remaining_rows = rows.iloc[completed_rows:]
    for row_number, (_, row) in enumerate(
        remaining_rows.iterrows(),
        start=completed_rows + 1,
    ):
        # The same deterministic seed token is used in both worlds. This aligns
        # source-row and seed identities; it is not described as exact common random
        # numbers unless the two simulator implementations consume innovations identically.
        token = (
            f"{split}|{row.get('seed_id', row_number)}|{FINAL_CFG.calibration_tag}|"
            f"{SOURCE_PANEL_HASH}|{FINAL_PROTOCOL_HASH}"
        )
        path_seed = int(hashlib.sha256(token.encode()).hexdigest()[:8], 16)
        generated = exact_path_batch_to_observable_episodes(
            engine,
            row,
            world,
            n_paths=paths_per_seed_row,
            path_seed=path_seed,
            history_length=FINAL_CFG.history_length,
            transaction_cost_bps=FINAL_CFG.transaction_cost_bps,
            spread_pool=SOURCE_SPREAD_POOL,
            burn_in_days=FINAL_CFG.exact_bank_burn_in_days,
        )
        episodes.extend(generated)
        completed_rows = row_number

        if (
            row_number % checkpoint_every_rows == 0
            or row_number == len(rows)
        ):
            checkpoint_payload = {
                "metadata": expected_metadata,
                "completed_rows": completed_rows,
                "episodes": episodes,
            }
            temporary_partial = partial_path.with_name(partial_path.name + ".tmp")
            torch.save(checkpoint_payload, temporary_partial)
            temporary_partial.replace(partial_path)

        if row_number <= 3 or row_number % 10 == 0 or row_number == len(rows):
            print(world, split, f"{row_number}/{len(rows)}", "episodes=", len(episodes))

    for episode in episodes:
        assert_episode_observability(episode, FINAL_CFG.history_length)

    temporary_cache = cache_path.with_name(cache_path.name + ".tmp")
    torch.save({"metadata": expected_metadata, "episodes": episodes}, temporary_cache)
    temporary_cache.replace(cache_path)
    partial_path.unlink(missing_ok=True)
    return episodes


def _load_legacy_h035_bank(world: str, split: str, paths_per_seed_row: int) -> list[dict[str, Any]]:
    cache_dir = LEGACY_H035_OUTPUT_DIR / "observable_market_cache"
    matches = sorted(cache_dir.glob(f"exact_observable_{world}_{split}_H035_n{int(paths_per_seed_row)}_*.pt"))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected exactly one legacy {world}/{split} bank; found {matches}")
    payload = torch.load(matches[0], map_location="cpu", weights_only=False)
    if not isinstance(payload, dict) or "episodes" not in payload or "metadata" not in payload:
        raise ValueError(f"Malformed legacy source bank: {matches[0]}")
    metadata = payload["metadata"]
    expected = {
        "protocol_hash": RUN.legacy_h035_expected_protocol_hash,
        "run_profile": "full",
        "world": world,
        "split": split,
        "H": 0.35,
        "n_factors": 20,
        "rn_scale": 2.5,
        "paths_per_seed_row": int(paths_per_seed_row),
        "policy_access_to_latent_simulator_state": "None",
    }
    mismatch = {k: {"expected": v, "observed": metadata.get(k)} for k, v in expected.items() if metadata.get(k) != v}
    if mismatch:
        raise ValueError(f"Legacy bank metadata mismatch for {matches[0].name}: {mismatch}")
    episodes = payload["episodes"]
    for episode in episodes:
        assert_episode_observability(episode, FINAL_CFG.history_length)
    print(f"[legacy bank] loaded {matches[0].name}: {len(episodes)} episodes")
    return episodes


if LEGACY_H035_REUSE:
    HESTON_TRAIN = _load_legacy_h035_bank("heston", "train", 16)
    HESTON_VALIDATION = _load_legacy_h035_bank("heston", "validation", 8)
    LIFTED_TRAIN = _load_legacy_h035_bank("lifted", "train", 16)
    LIFTED_VALIDATION = _load_legacy_h035_bank("lifted", "validation", 8)
    # Reconstruct the exact V5.3 interleaved mixed bank. It contains twice the total support of either pure bank.
    MIXED_TRAIN = [item for pair in zip(HESTON_TRAIN, LIFTED_TRAIN) for item in pair]
    MIXED_VALIDATION = [item for pair in zip(HESTON_VALIDATION, LIFTED_VALIDATION) for item in pair]
    source_support_status = "LEGACY_UNEQUAL_MIXED_SUPPORT"
else:
    HESTON_TRAIN = build_exact_observable_bank("heston", "train", FINAL_CFG.exact_train_paths_per_seed_row)
    HESTON_VALIDATION = build_exact_observable_bank("heston", "validation", FINAL_CFG.exact_validation_paths_per_seed_row)
    LIFTED_TRAIN = build_exact_observable_bank("lifted", "train", FINAL_CFG.exact_train_paths_per_seed_row)
    LIFTED_VALIDATION = build_exact_observable_bank("lifted", "validation", FINAL_CFG.exact_validation_paths_per_seed_row)
    if len(HESTON_TRAIN) != len(LIFTED_TRAIN):
        raise AssertionError("Heston and Lifted training banks are not matched in size.")
    if len(HESTON_VALIDATION) != len(LIFTED_VALIDATION):
        raise AssertionError("Heston and Lifted validation banks are not matched in size.")
    def fixed_size_balanced_mix(heston_bank, lifted_bank):
        if len(heston_bank) != len(lifted_bank):
            raise AssertionError("Balanced mixture requires matched single-source support.")
        return [h if i % 2 == 0 else l for i, (h, l) in enumerate(zip(heston_bank, lifted_bank))]
    MIXED_TRAIN = fixed_size_balanced_mix(HESTON_TRAIN, LIFTED_TRAIN)
    MIXED_VALIDATION = fixed_size_balanced_mix(HESTON_VALIDATION, LIFTED_VALIDATION)
    if not (len(MIXED_TRAIN) == len(HESTON_TRAIN) == len(LIFTED_TRAIN)):
        raise AssertionError("Source training support is not equal across Heston, Lifted and mixed cells.")
    if not (len(MIXED_VALIDATION) == len(HESTON_VALIDATION) == len(LIFTED_VALIDATION)):
        raise AssertionError("Source validation support is not equal across cells.")
    source_support_status = "EQUAL_FIXED_SIZE_SUPPORT"

SOURCE_BANK_SUPPORT_AUDIT = pd.DataFrame([
    {"source":"heston","train":len(HESTON_TRAIN),"validation":len(HESTON_VALIDATION),"heston_share":1.0,"status":source_support_status},
    {"source":"lifted","train":len(LIFTED_TRAIN),"validation":len(LIFTED_VALIDATION),"heston_share":0.0,"status":source_support_status},
    {"source":"mixed_legacy" if LEGACY_H035_REUSE else "mixed_fixed_size","train":len(MIXED_TRAIN),"validation":len(MIXED_VALIDATION),"heston_share":0.5,"status":source_support_status},
])
SOURCE_BANK_SUPPORT_AUDIT.to_csv(FINAL_TABLE_DIR / "source_bank_support_audit.csv", index=False)
print({
    "source_panel_hash": SOURCE_PANEL_HASH,
    "source_date_max": str(pd.to_datetime(SOURCE_SEED_PANEL["date"]).max().date()),
    "heston_train": len(HESTON_TRAIN),
    "lifted_train": len(LIFTED_TRAIN),
    "mixed_train": len(MIXED_TRAIN),
    "support_status": source_support_status,
})

lifted_kernel.grid.to_csv(FINAL_TABLE_DIR / "lifted_factor_nodes_weights.csv", index=False)
lifted_kernel.diagnostics.to_csv(FINAL_TABLE_DIR / "lifted_kernel_approximation_diagnostics.csv", index=False)

,maturity_bucket,rows,dates,date_min,date_max
0,1M,146,146,2010-02-22,2017-06-12
1,1W,160,160,2010-02-12,2017-08-25
2,2M,169,169,2010-01-21,2017-05-18
3,2W,123,123,2010-03-10,2017-04-05
4,3M,916,916,2010-01-04,2017-11-15


,source_role,maturity_bucket,rows,date_min,date_max
0,train,1M,20,2010-03-02,2012-09-12
1,train,1W,20,2010-03-23,2016-03-21
2,train,2M,20,2010-01-21,2012-05-07
3,train,2W,20,2010-03-15,2013-07-15
4,train,3M,20,2010-01-07,2015-02-02
5,validation,1M,8,2012-10-23,2014-01-03
6,validation,1W,8,2016-05-25,2017-07-12
7,validation,2M,8,2012-07-20,2017-05-18
8,validation,2W,8,2013-09-13,2016-10-10
9,validation,3M,8,2015-05-26,2017-09-07


Loaded C:\Users\lhkke\OneDrive - Queen Mary, University of London\Documents\Dissertation\Deep-Hedging-under-Lifted-Heston-MSc-Dissertation-\outputs_observable_h035_lag_consistent_retrained_full\observable_market_cache\exact_observable_heston_train_H035_n16_4acfa62a31_a0419a1b88ce.pt episodes= 1600
Loaded C:\Users\lhkke\OneDrive - Queen Mary, University of London\Documents\Dissertation\Deep-Hedging-under-Lifted-Heston-MSc-Dissertation-\outputs_observable_h035_lag_consistent_retrained_full\observable_market_cache\exact_observable_heston_validation_H035_n8_4acfa62a31_a0419a1b88ce.pt episodes= 320
Loaded C:\Users\lhkke\OneDrive - Queen Mary, University of London\Documents\Dissertation\Deep-Hedging-under-Lifted-Heston-MSc-Dissertation-\outputs_observable_h035_lag_consistent_retrained_full\observable_market_cache\exact_observable_lifted_train_H035_n16_4acfa62a31_a0419a1b88ce.pt episodes= 1600
Loaded C:\Users\lhkke\OneDrive - Queen Mary, University of London\Documents\Dissertation\Deep-Hedgin

In [56]:
from pathlib import Path

required_builder_features = {
    "tau_at_execution_fraction",
    "execution_anchor_delta",
    "previous_dt_years",
    "forward_dt_years",
    "scheduled_gap",
}

missing_builder_features = (
    required_builder_features.difference(CURRENT_FEATURES)
)

if missing_builder_features:
    raise RuntimeError(
        "This is the legacy bank builder. Open the V8 timing-fixed "
        f"calibration notebook; missing {sorted(missing_builder_features)}."
    )

if FINAL_CFG.calibration_tag != "H035":
    raise RuntimeError(
        f"Expected H035, but the active calibration is "
        f"{FINAL_CFG.calibration_tag}."
    )

original_source_panel = SOURCE_SEED_PANEL
original_source_panel_hash = SOURCE_PANEL_HASH
original_final_cache_dir = FINAL_CACHE_DIR

try:
    # Only rebuild the maturity used by the rolling experiment.
    SOURCE_SEED_PANEL = original_source_panel.loc[
        original_source_panel["maturity_bucket"]
        .astype(str)
        .eq("1M")
    ].copy()

    if SOURCE_SEED_PANEL.empty:
        raise RuntimeError("The source panel contains no 1M rows.")

    SOURCE_PANEL_HASH = stable_hash(
        SOURCE_SEED_PANEL[
            ["seed_id", "date", "maturity_bucket", "source_role"]
        ]
        .astype(str)
        .to_dict("records")
    )

    FINAL_CACHE_DIR = (
        Path("outputs_observable_h035_lag_consistent_retrained_full")
        / "observable_market_cache"
    )
    FINAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

    HESTON_TRAIN_1M = build_exact_observable_bank(
        "heston",
        "train",
        FINAL_CFG.exact_train_paths_per_seed_row,
    )
    HESTON_VALIDATION_1M = build_exact_observable_bank(
        "heston",
        "validation",
        FINAL_CFG.exact_validation_paths_per_seed_row,
    )
    LIFTED_TRAIN_1M = build_exact_observable_bank(
        "lifted",
        "train",
        FINAL_CFG.exact_train_paths_per_seed_row,
    )
    LIFTED_VALIDATION_1M = build_exact_observable_bank(
        "lifted",
        "validation",
        FINAL_CFG.exact_validation_paths_per_seed_row,
    )

finally:
    SOURCE_SEED_PANEL = original_source_panel
    SOURCE_PANEL_HASH = original_source_panel_hash
    FINAL_CACHE_DIR = original_final_cache_dir

print({
    "heston_train_1m": len(HESTON_TRAIN_1M),
    "heston_validation_1m": len(HESTON_VALIDATION_1M),
    "lifted_train_1m": len(LIFTED_TRAIN_1M),
    "lifted_validation_1m": len(LIFTED_VALIDATION_1M),
})

RuntimeError: This is the legacy bank builder. Open the V8 timing-fixed calibration notebook; missing ['execution_anchor_delta', 'forward_dt_years', 'previous_dt_years', 'scheduled_gap', 'tau_at_execution_fraction'].

# 14. Shared simulation-train-only scalers

In [ ]:
def _load_legacy_scaler(kind: str, expected_features: Sequence[str]) -> tuple[RobustScaler, Path]:
    matches = sorted((LEGACY_H035_OUTPUT_DIR / "observable_market_cache").glob(f"{kind}_scaler_{RUN.legacy_h035_expected_protocol_hash[:12]}*.json"))
    if len(matches) != 1:
        # Historical file names use the first 12 hash characters directly.
        matches = sorted((LEGACY_H035_OUTPUT_DIR / "observable_market_cache").glob(f"{kind}_scaler_*.json"))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected exactly one legacy {kind} scaler; found {matches}")
    payload = json.loads(matches[0].read_text())
    if list(payload.get("feature_names", [])) != list(expected_features):
        raise ValueError(f"Legacy {kind} scaler schema mismatch.")
    scaler = RobustScaler(
        feature_names=list(payload["feature_names"]),
        centre=np.asarray(payload["centre"], dtype=np.float32),
        scale=np.asarray(payload["scale"], dtype=np.float32),
        clip=float(payload.get("clip", 8.0)),
    )
    if not np.isfinite(scaler.centre).all() or not np.isfinite(scaler.scale).all() or np.any(scaler.scale <= 0):
        raise FloatingPointError(f"Invalid legacy {kind} scaler moments.")
    return scaler, matches[0]


if LEGACY_H035_REUSE:
    CURRENT_SCALER, LEGACY_CURRENT_SCALER_PATH = _load_legacy_scaler("current", CURRENT_FEATURES)
    HISTORY_SCALER, LEGACY_HISTORY_SCALER_PATH = _load_legacy_scaler("history", HISTORY_FEATURES)
    print("Scaler source: exact V5.3 H=0.35 mixed-bank scalers")
else:
    CURRENT_SCALER, HISTORY_SCALER = fit_common_scalers(MIXED_TRAIN)
    LEGACY_CURRENT_SCALER_PATH = None
    LEGACY_HISTORY_SCALER_PATH = None
    print("Scaler source: mixed exact-simulation training bank only")

CURRENT_SCALER_PATH = FINAL_CACHE_DIR / f"current_scaler_{FINAL_PROTOCOL_HASH[:12]}.json"
HISTORY_SCALER_PATH = FINAL_CACHE_DIR / f"history_scaler_{FINAL_PROTOCOL_HASH[:12]}.json"
CURRENT_SCALER_PATH.write_text(json.dumps(CURRENT_SCALER.to_json(), indent=2))
HISTORY_SCALER_PATH.write_text(json.dumps(HISTORY_SCALER.to_json(), indent=2))
print("Source panel maximum date:", pd.to_datetime(SOURCE_SEED_PANEL["date"]).max().date())
print("Policy access to latent simulator state: None")


Scaler source: mixed exact-simulation training bank only
Source panel maximum date: 2017-09-07
Policy access to latent simulator state: None


# 13. Pretraining: six unique final neural policies


In [ ]:
# ============================================================
# ACCOUNTING COMPONENT REGISTRY
# Required by numpy_accounting when creating normalised outputs.
# ============================================================

ACCOUNTING_COMPONENTS: tuple[str, ...] = (
    "option_mark_to_market_pnl",
    "stock_holding_gain",
    "financing_pnl",
    "dividend_carry",
    "option_entry_spread_cost",
    "option_exit_spread_cost",
    "option_execution_cost",
    "underlying_trading_cost",
    "underlying_liquidation_cost",
    "underlying_cost",
    "total_execution_cost",
    "net_pnl",
)

assert len(ACCOUNTING_COMPONENTS) == len(set(ACCOUNTING_COMPONENTS))
print("Accounting component registry restored:", ACCOUNTING_COMPONENTS)

SELECTED_BS_NO_TRADE_BAND, BS_NO_TRADE_VALIDATION = select_bs_no_trade_band(
    MARKET_VALIDATION, MKT_CFG.no_trade_band_candidates, DEVICE,
)
BS_NO_TRADE_VALIDATION.to_csv(FINAL_TABLE_DIR / "bs_no_trade_validation.csv", index=False)
NO_TRADE_BOUNDARY_AUDIT = pd.DataFrame([{
    "selected_band":SELECTED_BS_NO_TRADE_BAND,
    "grid_min":min(MKT_CFG.no_trade_band_candidates), "grid_max":max(MKT_CFG.no_trade_band_candidates),
    "selected_at_boundary":SELECTED_BS_NO_TRADE_BAND in {min(MKT_CFG.no_trade_band_candidates),max(MKT_CFG.no_trade_band_candidates)},
}])
NO_TRADE_BOUNDARY_AUDIT.to_csv(FINAL_TABLE_DIR / "bs_no_trade_boundary_audit.csv", index=False)
print("Selected Black-Scholes no-trade band under lag-consistent 2020 validation accounting:", SELECTED_BS_NO_TRADE_BAND)

if LEGACY_H035_REUSE:
    CELL_SPECS = {
        "heston_hybrid_current_small": ("current_surface_mlp", "heston_legacy", HESTON_TRAIN, HESTON_VALIDATION),
        "lifted_hybrid_current_small": ("current_surface_mlp", "lifted_legacy", LIFTED_TRAIN, LIFTED_VALIDATION),
        "mixed_hybrid_current_small": ("current_surface_mlp", "mixed_legacy_double_support", MIXED_TRAIN, MIXED_VALIDATION),
        "mixed_hybrid_lag": ("lag_stack_mlp", "mixed_legacy_double_support", MIXED_TRAIN, MIXED_VALIDATION),
        "mixed_hybrid_gru": ("observable_gru", "mixed_legacy_double_support", MIXED_TRAIN, MIXED_VALIDATION),
    }
    LEGACY_POLICY_MAP = {
        "heston_hybrid_current_small": "heston_hybrid_current",
        "lifted_hybrid_current_small": "lifted_hybrid_current",
        "mixed_hybrid_current_small": "mixed_hybrid_current",
        "mixed_hybrid_lag": "mixed_hybrid_lag",
        "mixed_hybrid_gru": "mixed_hybrid_gru",
    }
else:
    CELL_SPECS = {
        "heston_hybrid_current_small": ("current_surface_mlp", "heston", HESTON_TRAIN, HESTON_VALIDATION),
        "lifted_hybrid_current_small": ("current_surface_mlp", "lifted", LIFTED_TRAIN, LIFTED_VALIDATION),
        "mixed_hybrid_current_small": ("current_surface_mlp", "mixed_fixed_size", MIXED_TRAIN, MIXED_VALIDATION),
        "mixed_hybrid_current_matched": ("current_surface_mlp_matched", "mixed_fixed_size", MIXED_TRAIN, MIXED_VALIDATION),
        "mixed_hybrid_lag": ("lag_stack_mlp", "mixed_fixed_size", MIXED_TRAIN, MIXED_VALIDATION),
        "mixed_hybrid_gru": ("observable_gru", "mixed_fixed_size", MIXED_TRAIN, MIXED_VALIDATION),
    }
    LEGACY_POLICY_MAP = {}

POLICIES: dict[tuple[str, int], nn.Module] = {}
checkpoint_rows: list[dict[str, Any]] = []
training_histories: list[pd.DataFrame] = []

if LEGACY_H035_REUSE:
    legacy_manifest = pd.read_csv(LEGACY_H035_OUTPUT_DIR / "tables" / "checkpoint_manifest.csv")
    for policy_label, (architecture, source, _sim_train, _sim_validation) in CELL_SPECS.items():
        legacy_policy = LEGACY_POLICY_MAP[policy_label]
        for training_seed in FINAL_CFG.training_seeds:
            candidates = sorted((LEGACY_H035_OUTPUT_DIR / "checkpoints").glob(
                f"{legacy_policy}_seed{int(training_seed)}_{RUN.legacy_h035_expected_protocol_hash[:12]}*.pt"
            ))
            if len(candidates) != 1:
                raise FileNotFoundError(f"Expected one legacy checkpoint for {legacy_policy}/{training_seed}; found {candidates}")
            legacy_path = candidates[0]
            payload = torch.load(legacy_path, map_location=DEVICE, weights_only=False)
            observed = payload.get("metadata", {})
            expected_legacy = {
                "policy": legacy_policy,
                "architecture": architecture,
                "training_seed": int(training_seed),
                "protocol_hash": RUN.legacy_h035_expected_protocol_hash,
                "run_profile": "full",
                "H": 0.35,
                "policy_access_to_latent_simulator_state": "None",
                "selection_domain": "market_validation_2020",
            }
            mismatch = {k: {"expected": v, "observed": observed.get(k)} for k, v in expected_legacy.items() if observed.get(k) != v}
            if mismatch:
                raise ValueError(f"Legacy checkpoint metadata mismatch for {legacy_path.name}: {mismatch}")
            expected_hash_rows = legacy_manifest.loc[
                legacy_manifest.policy.eq(legacy_policy) & legacy_manifest.training_seed.eq(int(training_seed))
            ]
            if len(expected_hash_rows) != 1:
                raise ValueError(f"Legacy checkpoint manifest row missing or duplicated for {legacy_policy}/{training_seed}")
            actual_legacy_hash = file_sha256(legacy_path)
            expected_legacy_hash = str(expected_hash_rows.iloc[0].checkpoint_sha256)
            if actual_legacy_hash != expected_legacy_hash:
                raise ValueError(f"Legacy checkpoint SHA256 mismatch for {legacy_path.name}")
            model = make_model(architecture, TRAIN_CFG)
            model.load_state_dict(payload["state_dict"], strict=True)
            model.to(DEVICE)
            history = pd.DataFrame(payload.get("history", []))
            imported_metadata = {
                **observed,
                "policy": policy_label,
                "source": source,
                "protocol_hash": FINAL_PROTOCOL_HASH,
                "bridge_evidence_status": FINAL_CFG.evidence_status,
                "imported_from_policy": legacy_policy,
                "imported_from_protocol_hash": RUN.legacy_h035_expected_protocol_hash,
                "imported_checkpoint_sha256": actual_legacy_hash,
                "execution_lag_steps_at_training": 0,
                "execution_lag_steps_at_bridge_evaluation": TRAIN_CFG.execution_lag_steps,
                "parameter_count": parameter_count(model),
                "training_failed": False,
                "legacy_mixed_bank_double_support": legacy_policy.startswith("mixed_"),
            }
            bridge_path = FINAL_CHECKPOINT_DIR / f"{policy_label}_seed{int(training_seed)}_{FINAL_PROTOCOL_HASH[:12]}.pt"
            torch.save({
                "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                "metadata": imported_metadata,
                "history": history.to_dict("records"),
            }, bridge_path)
            POLICIES[(policy_label, int(training_seed))] = model
            checkpoint_rows.append({
                **imported_metadata,
                "checkpoint_path": str(bridge_path),
                "checkpoint_sha256": file_sha256(bridge_path),
                "legacy_checkpoint_path": str(legacy_path),
            })
            if not history.empty:
                training_histories.append(history.assign(policy=policy_label, training_seed=int(training_seed), provenance="legacy_v53_h035"))
            print("[legacy checkpoint]", policy_label, training_seed, imported_metadata.get("selected_stage"), imported_metadata.get("validation_mse"))
else:
    for policy_label, (architecture, source, sim_train, sim_validation) in CELL_SPECS.items():
        for training_seed in FINAL_CFG.training_seeds:
            checkpoint_path = FINAL_CHECKPOINT_DIR / f"{policy_label}_seed{training_seed}_{FINAL_PROTOCOL_HASH[:12]}.pt"
            model=None; history=pd.DataFrame(); metadata={}
            expected_checkpoint_metadata={
                "policy":policy_label,"architecture":architecture,"source":source,"training_seed":int(training_seed),
                "protocol_hash":FINAL_PROTOCOL_HASH,"run_profile":FINAL_CFG.run_profile,"H":FINAL_CFG.H,
                "policy_access_to_latent_simulator_state":"None","selection_domain":"market_validation_2020",
                "execution_lag_steps": TRAIN_CFG.execution_lag_steps,
                "information_time": TRAIN_CFG.information_time_convention,
                "decision_time": TRAIN_CFG.decision_time_convention,
                "execution_time": TRAIN_CFG.execution_time_convention,
                "execution_price": TRAIN_CFG.execution_price_convention,
                "execution_window": TRAIN_CFG.execution_window_convention,
            }
            if checkpoint_path.exists() and not FINAL_CFG.force_retrain:
                try:
                    payload=torch.load(checkpoint_path,map_location=DEVICE,weights_only=False); metadata=payload.get("metadata",{})
                    mismatched={k:{"expected":v,"observed":metadata.get(k)} for k,v in expected_checkpoint_metadata.items() if metadata.get(k)!=v}
                    if mismatched: raise ValueError(f"metadata mismatch: {mismatched}")
                    model=make_model(architecture,TRAIN_CFG); model.load_state_dict(payload["state_dict"]); model.to(DEVICE)
                    history=pd.DataFrame(payload.get("history",[])); print(f"[checkpoint] loaded {checkpoint_path.name}")
                except Exception as exc:
                    if not EXECUTION.can_rebuild("policies"): raise RuntimeError(f"Invalid checkpoint and strict reuse forbids retraining: {checkpoint_path}") from exc
                    print(f"[checkpoint] invalid; retraining {policy_label} seed={training_seed}. Reason: {exc}"); model=None
            if model is None:
                if not FINAL_CFG.run_training: raise FileNotFoundError(f"No valid checkpoint for {policy_label} seed={training_seed}")
                started=time.perf_counter()
                model,history,metadata=train_hybrid_policy(
                    architecture=architecture,source_label=source,sim_train=sim_train,sim_validation=sim_validation,
                    market_train=MARKET_TRAIN,market_validation=MARKET_VALIDATION,current_scaler=CURRENT_SCALER,
                    history_scaler=HISTORY_SCALER,training_seed=int(training_seed),cfg=TRAIN_CFG,device=DEVICE,
                )
                metadata={**metadata,**expected_checkpoint_metadata,"wall_clock_seconds":float(time.perf_counter()-started),
                          "parameter_count":parameter_count(model),"training_failed":False}
                torch.save({"state_dict":{k:v.detach().cpu() for k,v in model.state_dict().items()},"metadata":metadata,"history":history.to_dict("records")},checkpoint_path)
            POLICIES[(policy_label,int(training_seed))]=model.to(DEVICE)
            checkpoint_rows.append({**metadata,"policy":policy_label,"checkpoint_path":str(checkpoint_path),"checkpoint_sha256":file_sha256(checkpoint_path)})
            if not history.empty: training_histories.append(history.assign(policy=policy_label,training_seed=int(training_seed)))
            print(policy_label,training_seed,metadata.get("selected_stage"),metadata.get("validation_mse"))

CHECKPOINT_MANIFEST=pd.DataFrame(checkpoint_rows)
TRAINING_HISTORY=pd.concat(training_histories,ignore_index=True) if training_histories else pd.DataFrame()
CHECKPOINT_MANIFEST.to_csv(FINAL_TABLE_DIR/"checkpoint_manifest.csv",index=False)
TRAINING_HISTORY.to_csv(FINAL_TABLE_DIR/"training_history.csv",index=False)
display(CHECKPOINT_MANIFEST)


Accounting component registry restored: ('option_mark_to_market_pnl', 'stock_holding_gain', 'financing_pnl', 'dividend_carry', 'option_entry_spread_cost', 'option_exit_spread_cost', 'option_execution_cost', 'underlying_trading_cost', 'underlying_liquidation_cost', 'underlying_cost', 'total_execution_cost', 'net_pnl')


# 16. Common held-out historical evaluation

In [ ]:
def validated_or_fresh_evaluation(episodes, label):
    result_path=FINAL_TABLE_DIR/f"results_episode_level_{label}.csv"; meta_path=FINAL_TABLE_DIR/f"results_episode_level_{label}.meta.json"
    checkpoint_identity=CHECKPOINT_MANIFEST[["policy","training_seed","checkpoint_sha256"]].sort_values(["policy","training_seed"]).to_dict("records")
    episode_identity=sorted(str(ep["episode_id"]) for ep in episodes)
    expected={"protocol_hash":FINAL_PROTOCOL_HASH,"run_profile":FINAL_CFG.run_profile,
              "checkpoint_identity_hash":stable_hash(checkpoint_identity),"episode_hash":stable_hash(episode_identity),
              "evaluation_label":label,"transaction_cost_bps":list(FINAL_CFG.transaction_cost_bps),
              "execution_lag_steps": TRAIN_CFG.execution_lag_steps,
              "information_time": TRAIN_CFG.information_time_convention,
              "decision_time": TRAIN_CFG.decision_time_convention,
              "execution_time": TRAIN_CFG.execution_time_convention,
              "execution_price": TRAIN_CFG.execution_price_convention,
              "execution_window": TRAIN_CFG.execution_window_convention,
              "policy_access_to_latent_simulator_state": "None"}
    if result_path.exists() and meta_path.exists() and not EXECUTION.force_rebuild("evaluation"):
        observed=json.loads(meta_path.read_text()); mismatch={k:{"expected":v,"observed":observed.get(k)} for k,v in expected.items() if observed.get(k)!=v}
        if not mismatch:
            out=pd.read_csv(result_path); print(f"[evaluation] loaded {label}: {result_path}"); return out
        if not EXECUTION.can_rebuild("evaluation"): raise RuntimeError(f"Invalid cached {label} evaluation: {mismatch}")
    if not episodes:
        return pd.DataFrame()
    if not EXECUTION.can_rebuild("evaluation"): raise FileNotFoundError(f"No valid {label} evaluation and strict reuse forbids execution")
    out=evaluate_policies(episodes,POLICIES,CHECKPOINT_MANIFEST,CURRENT_SCALER,HISTORY_SCALER,FINAL_CFG.training_seeds,FINAL_CFG.transaction_cost_bps,DEVICE)
    out["evaluation_label"]=label; out.to_csv(result_path,index=False); meta_path.write_text(json.dumps(expected,indent=2,sort_keys=True))
    try: out.to_parquet(FINAL_TABLE_DIR/f"results_episode_level_{label}.parquet",index=False)
    except Exception as exc: print("Parquet export skipped:",exc)
    return out

RESULTS=validated_or_fresh_evaluation(MARKET_TEST,"postdevelopment_frozen_2021_2024")
CONFIRMATORY_RESULTS=validated_or_fresh_evaluation(MARKET_CONFIRMATORY,"untouched_confirmatory_2025") if MARKET_CONFIRMATORY else pd.DataFrame()
RESULTS.to_csv(FINAL_TABLE_DIR/"results_episode_level.csv",index=False)
print("2021-2024 rows/episodes/dates:",len(RESULTS),RESULTS.episode_id.nunique(),pd.to_datetime(RESULTS.entry_date).nunique())
print("2025 confirmatory rows/episodes/dates:",len(CONFIRMATORY_RESULTS),0 if CONFIRMATORY_RESULTS.empty else CONFIRMATORY_RESULTS.episode_id.nunique(),0 if CONFIRMATORY_RESULTS.empty else pd.to_datetime(CONFIRMATORY_RESULTS.entry_date).nunique())


# Compact decision-level policy behaviour export at the primary cost.
_decision_path = FINAL_TABLE_DIR / "decision_level_results.parquet"
_decision_meta_path = FINAL_TABLE_DIR / "decision_level_results.meta.json"
_decision_expected = {
    "protocol_hash": FINAL_PROTOCOL_HASH,
    "evaluation_label": "postdevelopment_frozen_2021_2024",
    "cost_bps": float(FINAL_CFG.primary_transaction_cost_bps),
    "policy_labels": list(FINAL_CFG.decision_export_policies),
    "checkpoint_hash": stable_hash(CHECKPOINT_MANIFEST[["policy", "training_seed", "checkpoint_sha256"]].sort_values(["policy", "training_seed"]).to_dict("records")),
}
DECISION_LEVEL_RESULTS = pd.DataFrame()
_decision_status = "NOT_RUN"
if _decision_path.exists() and _decision_meta_path.exists() and not EXECUTION.force_rebuild("evaluation"):
    observed = json.loads(_decision_meta_path.read_text())
    if observed == _decision_expected:
        try:
            DECISION_LEVEL_RESULTS = pd.read_parquet(_decision_path)
            _decision_status = "LOADED_VALID_CACHE"
        except Exception as exc:
            print("Decision-level parquet could not be loaded; rebuilding:", exc)
if DECISION_LEVEL_RESULTS.empty:
    DECISION_LEVEL_RESULTS = build_decision_level_results(
        MARKET_TEST, POLICIES, CURRENT_SCALER, HISTORY_SCALER,
        policy_labels=FINAL_CFG.decision_export_policies,
        cost_bps=FINAL_CFG.primary_transaction_cost_bps, device=DEVICE,
    )
    try:
        DECISION_LEVEL_RESULTS.to_parquet(_decision_path, index=False)
        _decision_meta_path.write_text(json.dumps(_decision_expected, indent=2, sort_keys=True))
        _decision_status = "WRITTEN_PARQUET"
    except Exception as exc:
        preview = FINAL_TABLE_DIR / "decision_level_results_preview.csv"
        DECISION_LEVEL_RESULTS.head(10000).to_csv(preview, index=False)
        _decision_status = f"PARQUET_UNAVAILABLE_PREVIEW_ONLY: {exc}"
DECISION_LEVEL_EXPORT_STATUS = pd.DataFrame([{
    "status": _decision_status, "rows": int(len(DECISION_LEVEL_RESULTS)),
    "episodes": int(DECISION_LEVEL_RESULTS.episode_id.nunique()) if len(DECISION_LEVEL_RESULTS) else 0,
    "policies": int(DECISION_LEVEL_RESULTS.policy.nunique()) if len(DECISION_LEVEL_RESULTS) else 0,
    "cost_bps": float(FINAL_CFG.primary_transaction_cost_bps),
    "seed_representation": "neural seed-ensemble mean with cross-seed hedge standard deviation",
}])
DECISION_LEVEL_EXPORT_STATUS.to_csv(FINAL_TABLE_DIR / "decision_level_export_status.csv", index=False)
print(DECISION_LEVEL_EXPORT_STATUS.to_dict("records"))


[evaluation] loaded postdevelopment_frozen_2021_2024: C:\Users\lhkke\OneDrive - Queen Mary, University of London\Documents\Dissertation\Deep-Hedging-under-Lifted-Heston-MSc-Dissertation-\outputs_observable_h035_lag_consistent_retrained_full\tables\results_episode_level_postdevelopment_frozen_2021_2024.csv
2021-2024 rows/episodes/dates: 58527 929 208
2025 confirmatory rows/episodes/dates: 0 0 0
[{'status': 'LOADED_VALID_CACHE', 'rows': 214488, 'episodes': 929, 'policies': 9, 'cost_bps': 1.0, 'seed_representation': 'neural seed-ensemble mean with cross-seed hedge standard deviation'}]


# 17. Summary metrics and paired date-block inference

In [ ]:
POLICY_SUMMARY=enhanced_policy_summary(RESULTS,["underlying_tc_bps","policy"])
MATURITY_SUMMARY=enhanced_policy_summary(RESULTS,["underlying_tc_bps","maturity_bucket","policy"])
SEED_LEVEL_RESULTS=seed_level_summary(RESULTS,FINAL_CFG.primary_transaction_cost_bps)
SEED_DISPERSION=seed_dispersion_summary(SEED_LEVEL_RESULTS)
for frame,name in [(POLICY_SUMMARY,"policy_summary.csv"),(MATURITY_SUMMARY,"maturity_summary.csv"),(SEED_LEVEL_RESULTS,"seed_level_results.csv"),(SEED_DISPERSION,"seed_dispersion_summary.csv")]: frame.to_csv(FINAL_TABLE_DIR/name,index=False)

source_labels={"heston_hybrid_current_small","lifted_hybrid_current_small","mixed_hybrid_current_small","black_scholes_delta","black_scholes_no_trade"}
architecture_labels=({"mixed_hybrid_current_small","mixed_hybrid_lag","mixed_hybrid_gru","black_scholes_delta","black_scholes_no_trade"} if LEGACY_H035_REUSE else {"mixed_hybrid_current_small","mixed_hybrid_current_matched","mixed_hybrid_lag","mixed_hybrid_gru","black_scholes_delta","black_scholes_no_trade"})
SOURCE_SUMMARY=POLICY_SUMMARY.loc[POLICY_SUMMARY.policy.isin(source_labels)].copy(); ARCHITECTURE_SUMMARY=POLICY_SUMMARY.loc[POLICY_SUMMARY.policy.isin(architecture_labels)].copy()
SOURCE_SUMMARY.to_csv(FINAL_TABLE_DIR/"source_summary.csv",index=False); ARCHITECTURE_SUMMARY.to_csv(FINAL_TABLE_DIR/"architecture_summary.csv",index=False); POLICY_SUMMARY.to_csv(FINAL_TABLE_DIR/"cost_robustness.csv",index=False)

# The preferred policy is chosen on 2020 validation only, never on either evaluation panel.
preferred_candidates = CHECKPOINT_MANIFEST.copy()
preferred_by_validation=preferred_candidates.groupby("policy",as_index=False).validation_mse.mean().sort_values(["validation_mse","policy"])
PREFERRED_NEURAL_POLICY=str(preferred_by_validation.iloc[0].policy)
preferred_by_validation.to_csv(FINAL_TABLE_DIR/"preferred_policy_validation_selection.csv",index=False)

if LEGACY_H035_REUSE:
    PRIMARY_CONTRAST_DEFINITIONS=[
        ("lifted_hybrid_current_small","heston_hybrid_current_small","S1_lifted_vs_heston"),
        ("mixed_hybrid_current_small","heston_hybrid_current_small","S2_legacy_mixed_vs_heston"),
        ("mixed_hybrid_lag","mixed_hybrid_current_small","A1_history_vs_legacy_current"),
        ("mixed_hybrid_gru","mixed_hybrid_lag","A2_gru_vs_lag"),
        ("mixed_hybrid_gru","mixed_hybrid_current_small","A3_gru_vs_legacy_current"),
        (PREFERRED_NEURAL_POLICY,"black_scholes_delta","C1_validation_selected_neural_vs_bs"),
    ]
    SECONDARY_CONTRAST_DEFINITIONS=[]
else:
    PRIMARY_CONTRAST_DEFINITIONS=[
        ("lifted_hybrid_current_small","heston_hybrid_current_small","S1_lifted_vs_heston"),
        ("mixed_hybrid_current_small","heston_hybrid_current_small","S2_fixed_mixed_vs_heston"),
        ("mixed_hybrid_lag","mixed_hybrid_current_matched","A1_supplied_history_vs_matched_current"),
        ("mixed_hybrid_gru","mixed_hybrid_lag","A2_gru_vs_lag"),
        ("mixed_hybrid_gru","mixed_hybrid_current_matched","A3_gru_vs_matched_current"),
        (PREFERRED_NEURAL_POLICY,"black_scholes_delta","C1_validation_selected_neural_vs_bs"),
    ]
    SECONDARY_CONTRAST_DEFINITIONS=[("mixed_hybrid_current_matched","mixed_hybrid_current_small","A0_capacity_effect")]
CONTRASTS=pd.DataFrame([paired_contrast_v6(RESULTS,a,b,label,primary_cost=FINAL_CFG.primary_transaction_cost_bps,repetitions=FINAL_CFG.bootstrap_repetitions,mean_block_weeks=FINAL_CFG.bootstrap_block_weeks,seed=FINAL_CFG.bootstrap_seed,family_size=len(PRIMARY_CONTRAST_DEFINITIONS),strict_inference=FINAL_CFG.is_full_evidence_run) for a,b,label in PRIMARY_CONTRAST_DEFINITIONS])
SECONDARY_CONTRASTS=pd.DataFrame([paired_contrast_v6(RESULTS,a,b,label,primary_cost=FINAL_CFG.primary_transaction_cost_bps,repetitions=FINAL_CFG.bootstrap_repetitions,mean_block_weeks=FINAL_CFG.bootstrap_block_weeks,seed=FINAL_CFG.bootstrap_seed,family_size=1,strict_inference=FINAL_CFG.is_full_evidence_run) for a,b,label in SECONDARY_CONTRAST_DEFINITIONS]) if SECONDARY_CONTRAST_DEFINITIONS else pd.DataFrame(columns=["comparison","policy_a","policy_b"])
HIERARCHICAL_CONTRASTS=pd.DataFrame([hierarchical_seed_date_contrast(RESULTS,a,b,label,primary_cost=FINAL_CFG.primary_transaction_cost_bps,repetitions=FINAL_CFG.bootstrap_repetitions,mean_block_weeks=FINAL_CFG.bootstrap_block_weeks,seed=FINAL_CFG.bootstrap_seed+2) for a,b,label in PRIMARY_CONTRAST_DEFINITIONS])

# Seed sign consistency uses each seed-specific date/maturity panel.
def seed_contrast_signs(results,a,b,label):
    rows=[]; seeds=sorted(results.loc[results.training_seed.ge(0),"training_seed"].unique())
    for s in seeds:
        x=results.loc[(results.training_seed.eq(s)) | (results.training_seed.lt(0))].copy()
        r=paired_contrast_v6(x,a,b,label,primary_cost=FINAL_CFG.primary_transaction_cost_bps,repetitions=max(200,min(1000,FINAL_CFG.bootstrap_repetitions)),mean_block_weeks=FINAL_CFG.bootstrap_block_weeks,seed=FINAL_CFG.bootstrap_seed+int(s)%1000,family_size=1,strict_inference=False)
        rows.append({"comparison":label,"training_seed":int(s),"relative_mse_effect":r.get("relative_mse_improvement_a_over_b",np.nan),"positive":r.get("relative_mse_improvement_a_over_b",np.nan)>0})
    return pd.DataFrame(rows)
SEED_CONTRASTS=pd.concat([seed_contrast_signs(RESULTS,a,b,label) for a,b,label in PRIMARY_CONTRAST_DEFINITIONS],ignore_index=True)
SEED_SIGN_CONSISTENCY=SEED_CONTRASTS.groupby("comparison",as_index=False).agg(seeds=("training_seed","nunique"),positive_seeds=("positive","sum"),minimum_effect=("relative_mse_effect","min"),maximum_effect=("relative_mse_effect","max"),median_effect=("relative_mse_effect","median"))

CONTRASTS=CONTRASTS.merge(SEED_SIGN_CONSISTENCY,on="comparison",how="left")
CONTRASTS.to_csv(FINAL_TABLE_DIR/"primary_contrasts.csv",index=False); SECONDARY_CONTRASTS.to_csv(FINAL_TABLE_DIR/"secondary_capacity_contrast.csv",index=False); HIERARCHICAL_CONTRASTS.to_csv(FINAL_TABLE_DIR/"hierarchical_seed_date_contrasts.csv",index=False); SEED_CONTRASTS.to_csv(FINAL_TABLE_DIR/"seed_level_contrasts.csv",index=False); SEED_SIGN_CONSISTENCY.to_csv(FINAL_TABLE_DIR/"seed_sign_consistency.csv",index=False)
CONTRASTS.loc[CONTRASTS.comparison.str.startswith("S")].to_csv(FINAL_TABLE_DIR/"source_primary_contrasts.csv",index=False); CONTRASTS.loc[CONTRASTS.comparison.str.startswith("A")].to_csv(FINAL_TABLE_DIR/"architecture_primary_contrasts.csv",index=False)

CONFIRMATORY_POLICY_SUMMARY=enhanced_policy_summary(CONFIRMATORY_RESULTS,["underlying_tc_bps","policy"]) if not CONFIRMATORY_RESULTS.empty else pd.DataFrame()
CONFIRMATORY_CONTRASTS=pd.DataFrame([paired_contrast_v6(CONFIRMATORY_RESULTS,a,b,label,primary_cost=FINAL_CFG.primary_transaction_cost_bps,repetitions=FINAL_CFG.bootstrap_repetitions,mean_block_weeks=FINAL_CFG.bootstrap_block_weeks,seed=FINAL_CFG.bootstrap_seed+10,family_size=len(PRIMARY_CONTRAST_DEFINITIONS),strict_inference=False) for a,b,label in PRIMARY_CONTRAST_DEFINITIONS]) if not CONFIRMATORY_RESULTS.empty else pd.DataFrame()
CONFIRMATORY_POLICY_SUMMARY.to_csv(FINAL_TABLE_DIR/"confirmatory_2025_policy_summary.csv",index=False); CONFIRMATORY_CONTRASTS.to_csv(FINAL_TABLE_DIR/"confirmatory_2025_contrasts.csv",index=False)
display(POLICY_SUMMARY.loc[POLICY_SUMMARY.underlying_tc_bps.eq(FINAL_CFG.primary_transaction_cost_bps)].sort_values("rmse_normalised_net_pnl")); display(CONTRASTS); display(HIERARCHICAL_CONTRASTS)


# Paired lower-tail uncertainty. This is separate from the MSE family and is used only for tail wording.
TAIL_CONTRASTS = pd.DataFrame([
    paired_tail_contrast(
        RESULTS, a, b, label,
        primary_cost=FINAL_CFG.primary_transaction_cost_bps,
        repetitions=FINAL_CFG.bootstrap_repetitions,
        mean_block_weeks=FINAL_CFG.bootstrap_block_weeks,
        seed=FINAL_CFG.bootstrap_seed + 20,
    )
    for a, b, label in PRIMARY_CONTRAST_DEFINITIONS
])
TAIL_CONTRASTS.to_csv(FINAL_TABLE_DIR / "paired_tail_contrasts.csv", index=False)

PAIRED_DATE_EFFECTS = pd.concat([
    paired_date_effect_panel(RESULTS, a, b, label, FINAL_CFG.primary_transaction_cost_bps)
    for a, b, label in PRIMARY_CONTRAST_DEFINITIONS
], ignore_index=True)
DATE_EFFECT_SUMMARY = PAIRED_DATE_EFFECTS.groupby("comparison", as_index=False).agg(
    paired_dates=("entry_date", "nunique"), share_dates_favouring_a=("favours_a", "mean"),
    median_date_effect=("relative_mse_effect", "median"),
    minimum_date_effect=("relative_mse_effect", "min"), maximum_date_effect=("relative_mse_effect", "max"),
)
PAIRED_DATE_EFFECTS.to_csv(FINAL_TABLE_DIR / "paired_date_effects.csv", index=False)
DATE_EFFECT_SUMMARY.to_csv(FINAL_TABLE_DIR / "paired_date_effect_summary.csv", index=False)

# Descriptive maturity-by-cost contrast grid. It is not a new inferential family.
def descriptive_contrast_grid(results, definitions):
    rows = []
    for cost in sorted(pd.to_numeric(results.underlying_tc_bps, errors="coerce").dropna().unique()):
        panel = ensemble_episode_panel(results, float(cost))
        for a, b, label in definitions:
            for maturity in [*sorted(panel.maturity_bucket.dropna().astype(str).unique()), "Overall"]:
                use = panel if maturity == "Overall" else panel.loc[panel.maturity_bucket.astype(str).eq(maturity)]
                pivot = use.loc[use.policy.isin([a, b])].pivot_table(index="episode_id", columns="policy", values="normalised_squared_error", aggfunc="first").dropna()
                effect = 1.0 - float(pivot[a].mean()) / max(float(pivot[b].mean()), 1e-12) if len(pivot) else np.nan
                rows.append({"comparison": label, "policy_a": a, "policy_b": b, "underlying_tc_bps": float(cost), "maturity_bucket": maturity, "relative_mse_effect": effect, "episodes": int(len(pivot)), "status": "descriptive_robustness"})
    return pd.DataFrame(rows)
ROBUSTNESS_CONTRAST_GRID = descriptive_contrast_grid(RESULTS, PRIMARY_CONTRAST_DEFINITIONS[:4])
ROBUSTNESS_CONTRAST_GRID.to_csv(FINAL_TABLE_DIR / "maturity_cost_contrast_grid.csv", index=False)

# Regimes are frozen from the 2018-2019 adaptation sample using lagged observable RV20, not evaluation outcomes.
def entry_feature_table(episodes, feature_name):
    rows = []
    for ep in episodes:
        names = list(ep["feature_names"])
        if feature_name not in names:
            continue
        rows.append({"episode_id": ep["episode_id"], "entry_date": ep["entry_date"], "maturity_bucket": ep["maturity_bucket"], feature_name: float(np.asarray(ep["features"], float)[0, names.index(feature_name)])})
    return pd.DataFrame(rows)
_train_regime_reference = entry_feature_table(MARKET_TRAIN, FINAL_CFG.regime_reference_feature)
_q1, _q2 = FINAL_CFG.regime_quantiles
_regime_thresholds = _train_regime_reference[FINAL_CFG.regime_reference_feature].quantile([_q1, _q2]).to_numpy(float) if len(_train_regime_reference) else np.array([np.nan, np.nan])
REGIME_DEFINITION = pd.DataFrame([{
    "reference_sample": "market_adaptation_2018_2019", "feature": FINAL_CFG.regime_reference_feature,
    "lower_quantile": float(_q1), "upper_quantile": float(_q2),
    "lower_threshold": float(_regime_thresholds[0]), "upper_threshold": float(_regime_thresholds[1]),
    "status": "secondary_prespecified_from_non_evaluation_sample",
}])
_test_regime = entry_feature_table(MARKET_TEST, FINAL_CFG.regime_reference_feature)
def _regime_label(x):
    if not np.isfinite(x) or not np.isfinite(_regime_thresholds).all(): return "unavailable"
    if x <= _regime_thresholds[0]: return "calm"
    if x <= _regime_thresholds[1]: return "ordinary"
    return "stress"
if len(_test_regime):
    _test_regime["regime"] = _test_regime[FINAL_CFG.regime_reference_feature].map(_regime_label)
    REGIME_RESULTS = RESULTS.merge(_test_regime[["episode_id", "regime"]], on="episode_id", how="left")
    REGIME_PERFORMANCE = enhanced_policy_summary(REGIME_RESULTS, ["underlying_tc_bps", "regime", "policy"])
else:
    REGIME_RESULTS = pd.DataFrame(); REGIME_PERFORMANCE = pd.DataFrame()
REGIME_DEFINITION.to_csv(FINAL_TABLE_DIR / "regime_definition.csv", index=False)
REGIME_PERFORMANCE.to_csv(FINAL_TABLE_DIR / "regime_performance.csv", index=False)

# Observable source-to-market alignment in the exact policy feature space.
def sampled_feature_frame(episodes, max_rows=50000, seed=20260729):
    arrays = [np.asarray(ep["features"], float) for ep in episodes]
    if not arrays: return pd.DataFrame(columns=CURRENT_FEATURES)
    x = np.concatenate(arrays, axis=0)
    if len(x) > max_rows:
        rng = np.random.default_rng(seed); x = x[rng.choice(len(x), size=max_rows, replace=False)]
    return pd.DataFrame(x, columns=CURRENT_FEATURES)
_market_features = sampled_feature_frame(MARKET_TRAIN, seed=20260729)
_source_feature_frames = {
    "heston": sampled_feature_frame(HESTON_TRAIN, seed=20260730),
    "lifted_heston": sampled_feature_frame(LIFTED_TRAIN, seed=20260731),
    "fixed_mixed": sampled_feature_frame(MIXED_TRAIN, seed=20260732),
}
_alignment_rows = []
try:
    from scipy.stats import wasserstein_distance
    for feature in CURRENT_FEATURES:
        market_values = pd.to_numeric(_market_features[feature], errors="coerce").dropna().to_numpy(float)
        centre = float(np.median(market_values)); scale = max(float(np.quantile(market_values, .75) - np.quantile(market_values, .25)), 1e-8)
        lo, hi = np.quantile(market_values, [.01, .99])
        for source_name, source_frame in _source_feature_frames.items():
            source_values = pd.to_numeric(source_frame[feature], errors="coerce").dropna().to_numpy(float)
            dist = wasserstein_distance((source_values - centre) / scale, (market_values - centre) / scale)
            coverage = float(np.mean((market_values >= np.quantile(source_values, .01)) & (market_values <= np.quantile(source_values, .99)))) if len(source_values) else np.nan
            _alignment_rows.append({"source": source_name, "feature": feature, "standardised_wasserstein_distance": float(dist), "market_coverage_by_source_1_99pct": coverage, "market_reference": "2018_2019_adaptation", "status": "descriptive_mechanism_diagnostic"})
except Exception as exc:
    _alignment_rows.append({"source": "unavailable", "feature": "unavailable", "standardised_wasserstein_distance": np.nan, "market_coverage_by_source_1_99pct": np.nan, "market_reference": "2018_2019_adaptation", "status": f"unavailable: {exc}"})
SOURCE_MARKET_ALIGNMENT = pd.DataFrame(_alignment_rows)
SOURCE_MARKET_ALIGNMENT.to_csv(FINAL_TABLE_DIR / "source_to_market_observable_alignment.csv", index=False)

# Error concentration and deterministic case-study selection.
_EPISODE_PANEL = ensemble_episode_panel(RESULTS, FINAL_CFG.primary_transaction_cost_bps)
_error_rows = []
for policy in ["heston_hybrid_current_small", "mixed_hybrid_gru", "black_scholes_delta"]:
    g = _EPISODE_PANEL.loc[_EPISODE_PANEL.policy.eq(policy)].sort_values("normalised_squared_error", ascending=False).reset_index(drop=True)
    total = float(g.normalised_squared_error.sum())
    if len(g) and total > 0:
        _error_rows.append(pd.DataFrame({"policy": policy, "episode_share": (np.arange(len(g)) + 1) / len(g), "cumulative_mse_share": g.normalised_squared_error.cumsum() / total}))
ERROR_CONCENTRATION = pd.concat(_error_rows, ignore_index=True) if _error_rows else pd.DataFrame()
ERROR_CONCENTRATION.to_csv(FINAL_TABLE_DIR / "error_concentration_curve.csv", index=False)

_case_panel = _EPISODE_PANEL.pivot_table(index=["episode_id", "entry_date", "maturity_bucket"], columns="policy", values=["normalised_squared_error", "normalised_net_pnl"], aggfunc="first").reset_index()
_case_rows = []
if len(_case_panel) and "heston_hybrid_current_small" in _case_panel["normalised_squared_error"].columns and "black_scholes_delta" in _case_panel["normalised_squared_error"].columns:
    h = _case_panel[("normalised_squared_error", "heston_hybrid_current_small")]
    median_ix = (h - h.median()).abs().idxmin()
    _case_rows.append({"case": "median_heston_error", "episode_id": _case_panel.loc[median_ix, ("episode_id", "")], "selection_rule": "closest to median Heston ensemble squared error"})
    disagreement = (h - _case_panel[("normalised_squared_error", "black_scholes_delta")]).abs()
    disagreement_ix = (disagreement - disagreement.quantile(.95)).abs().idxmin()
    _case_rows.append({"case": "high_heston_bs_disagreement", "episode_id": _case_panel.loc[disagreement_ix, ("episode_id", "")], "selection_rule": "closest to 95th percentile absolute Heston-BS squared-error disagreement"})
if len(_test_regime):
    stress_ix = (_test_regime[FINAL_CFG.regime_reference_feature] - _test_regime[FINAL_CFG.regime_reference_feature].quantile(.95)).abs().idxmin()
    _case_rows.append({"case": "high_lagged_volatility", "episode_id": _test_regime.loc[stress_ix, "episode_id"], "selection_rule": "closest to 95th percentile entry lagged RV20 defined without policy outcomes"})
CASE_STUDY_SELECTION = pd.DataFrame(_case_rows).drop_duplicates("episode_id") if _case_rows else pd.DataFrame()
CASE_STUDY_SELECTION.to_csv(FINAL_TABLE_DIR / "deterministic_case_study_selection.csv", index=False)

display(TAIL_CONTRASTS); display(DATE_EFFECT_SUMMARY); display(REGIME_DEFINITION); display(SOURCE_MARKET_ALIGNMENT.head())


,underlying_tc_bps,policy,episodes,dates,mse_normalised_net_pnl,rmse_normalised_net_pnl,mean_normalised_net_pnl,std_normalised_net_pnl,variance_contribution,bias_squared_contribution,downside_rmse,worst5_mean_normalised_net_pnl,expected_shortfall_style_loss_5pct,mean_turnover,mean_underlying_cost,mean_option_execution_cost,mean_total_execution_cost,maximum_absolute_accounting_identity_error,mean_normalised_option_mark_to_market_pnl,mean_normalised_stock_holding_gain,mean_normalised_financing_pnl,mean_normalised_dividend_carry,mean_normalised_option_entry_spread_cost,mean_normalised_option_exit_spread_cost,mean_normalised_option_execution_cost,mean_normalised_underlying_trading_cost,mean_normalised_underlying_liquidation_cost,mean_normalised_underlying_cost,mean_normalised_total_execution_cost
3,1.0,lifted_hybrid_current_small,929,208,0.086709,0.294464,-0.049666,0.290401,0.084242,0.002467,0.248919,-0.788605,0.788605,3.344971,1.514684,2.102422,3.617106,5.883294e-12,0.12877,-0.147885,-0.001179,-0.006672,0.004513,0.009059,0.013572,0.007064,0.002063,0.009127,0.022699
0,1.0,black_scholes_delta,929,208,0.087091,0.295112,-0.008317,0.295154,0.087022,0.000069,0.236260,-0.780970,0.780970,3.167710,1.431622,2.102422,3.534044,4.973799e-12,0.12877,-0.102607,-0.003757,-0.008589,0.004513,0.009059,0.013572,0.006434,0.002128,0.008562,0.022134
2,1.0,heston_hybrid_current_small,929,208,0.090162,0.300269,-0.041673,0.297524,0.088425,0.001737,0.248694,-0.760513,0.760513,2.943916,1.330892,2.102422,3.433314,5.385914e-12,0.12877,-0.143472,-0.000604,-0.004691,0.004513,0.009059,0.013572,0.006203,0.001901,0.008104,0.021676
4,1.0,mixed_hybrid_current_matched,929,208,0.091337,0.302219,-0.057210,0.296915,0.088064,0.003273,0.255712,-0.768082,0.768082,3.156967,1.424876,2.102422,3.527298,5.563550e-12,0.12877,-0.159154,-0.000138,-0.004452,0.004513,0.009059,0.013572,0.006707,0.001958,0.008665,0.022237
5,1.0,mixed_hybrid_current_small,929,208,0.091906,0.303160,-0.061022,0.297115,0.088182,0.003724,0.258554,-0.782490,0.782490,3.443334,1.557758,2.102422,3.660180,5.513812e-12,0.12877,-0.161735,-0.000408,-0.004751,0.004513,0.009059,0.013572,0.007327,0.001999,0.009326,0.022898
1,1.0,black_scholes_no_trade,929,208,0.096949,0.311367,0.013291,0.311251,0.096773,0.000177,0.237036,-0.777543,0.777543,1.974537,0.897079,2.102422,2.999501,4.718004e-12,0.12877,-0.085513,-0.003025,-0.007566,0.004513,0.009059,0.013572,0.003812,0.001990,0.005802,0.019374
6,1.0,mixed_hybrid_gru,929,208,0.098180,0.313337,-0.066766,0.306307,0.093723,0.004458,0.268590,-0.794568,0.794568,3.275464,1.483404,2.102422,3.585826,5.016432e-12,0.12877,-0.169657,0.000297,-0.003597,0.004513,0.009059,0.013572,0.007052,0.001954,0.009006,0.022578
7,1.0,mixed_hybrid_lag,929,208,0.100567,0.317123,-0.067779,0.309962,0.095973,0.004594,0.271339,-0.795338,0.795338,3.132502,1.415305,2.102422,3.517727,8.132162e-12,0.12877,-0.171132,0.000657,-0.003808,0.004513,0.009059,0.013572,0.006762,0.001931,0.008693,0.022265
8,1.0,no_hedge,929,208,0.293558,0.541810,0.116207,0.529486,0.280054,0.013504,0.361874,-1.219536,1.219536,0.000000,0.000000,2.102422,2.102422,3.836931e-13,0.12877,0.000000,0.001009,0.000000,0.004513,0.009059,0.013572,0.000000,0.000000,0.000000,0.013572


,comparison,policy_a,policy_b,n_dates,absolute_mse_difference_a_minus_b,relative_mse_improvement_a_over_b,ci_2_5pct,ci_97_5pct,familywise_ci_1_25pct,familywise_ci_98_75pct,absolute_ci_2_5pct,absolute_ci_97_5pct,bootstrap_fraction_nonpositive,mean_block_weeks,mean_block_date_observations,bootstrap_repetitions,bootstrap_performed,inference_status,inference_note,aggregation,familywise_ci_low,familywise_ci_high,family_size,seeds,positive_seeds,minimum_effect,maximum_effect,median_effect
0,S1_lifted_vs_heston,lifted_hybrid_current_small,heston_hybrid_current_small,208,-0.002373,0.023426,-0.083455,0.143719,-0.097520,0.156499,-0.013342,0.009812,0.3434,13,13,10000,True,BOOTSTRAP_COMPLETE,,equal_seed_then_equal_available_buckets_within...,-0.117294,0.174373,6,3,2,-0.002637,0.047571,0.024722
1,S2_fixed_mixed_vs_heston,mixed_hybrid_current_small,heston_hybrid_current_small,208,0.001900,-0.018763,-0.090057,0.074750,-0.099782,0.088583,-0.006618,0.010889,0.6610,13,13,10000,True,BOOTSTRAP_COMPLETE,,equal_seed_then_equal_available_buckets_within...,-0.108895,0.102166,6,3,0,-0.036375,-0.009052,-0.011195
2,A1_supplied_history_vs_matched_current,mixed_hybrid_lag,mixed_hybrid_current_matched,208,0.008145,-0.081364,-0.133431,-0.026881,-0.139113,-0.018750,0.002907,0.013109,0.9990,13,13,10000,True,BOOTSTRAP_COMPLETE,,equal_seed_then_equal_available_buckets_within...,-0.144538,-0.011616,6,3,0,-0.103790,-0.062425,-0.077725
3,A2_gru_vs_lag,mixed_hybrid_gru,mixed_hybrid_lag,208,-0.001428,0.013196,-0.016723,0.047252,-0.021411,0.050882,-0.004339,0.002094,0.1942,13,13,10000,True,BOOTSTRAP_COMPLETE,,equal_seed_then_equal_available_buckets_within...,-0.026628,0.057026,6,3,2,-0.015755,0.032684,0.022105
4,A3_gru_vs_matched_current,mixed_hybrid_gru,mixed_hybrid_current_matched,208,0.006717,-0.067094,-0.101028,-0.031097,-0.105439,-0.026432,0.002880,0.010939,0.9999,13,13,10000,True,BOOTSTRAP_COMPLETE,,equal_seed_then_equal_available_buckets_within...,-0.111767,-0.018762,6,3,0,-0.094704,-0.027701,-0.079391
5,C1_validation_selected_neural_vs_bs,mixed_hybrid_lag,black_scholes_delta,208,0.004415,-0.042516,-0.415800,0.188807,-0.487034,0.210349,-0.024273,0.033335,0.6170,13,13,10000,True,BOOTSTRAP_COMPLETE,,equal_seed_then_equal_available_buckets_within...,-0.596015,0.230958,6,3,0,-0.073759,-0.023791,-0.029998


,comparison,policy_a,policy_b,seed_count,date_count,hierarchical_point,hierarchical_ci_2_5pct,hierarchical_ci_97_5pct,hierarchical_fraction_nonpositive,hierarchical_repetitions,hierarchical_status
0,S1_lifted_vs_heston,lifted_hybrid_current_small,heston_hybrid_current_small,3,208,0.026699,-0.092680,0.161353,0.3478,10000,COMPLETE
1,S2_fixed_mixed_vs_heston,mixed_hybrid_current_small,heston_hybrid_current_small,3,208,-0.015920,-0.109649,0.092585,0.6567,10000,COMPLETE
2,A1_supplied_history_vs_matched_current,mixed_hybrid_lag,mixed_hybrid_current_matched,3,208,-0.082408,-0.148870,-0.024001,0.9967,10000,COMPLETE
3,A2_gru_vs_lag,mixed_hybrid_gru,mixed_hybrid_lag,3,208,0.014345,-0.035746,0.053122,0.2317,10000,COMPLETE
4,A3_gru_vs_matched_current,mixed_hybrid_gru,mixed_hybrid_current_matched,3,208,-0.067416,-0.124215,-0.014705,0.9946,10000,COMPLETE
5,C1_validation_selected_neural_vs_bs,mixed_hybrid_lag,black_scholes_delta,3,208,-0.060088,-0.424649,0.184803,0.6020,10000,COMPLETE


,comparison,policy_a,policy_b,worst5_mean_difference_a_minus_b,tail_ci_2_5pct,tail_ci_97_5pct,tail_fraction_nonpositive,tail_repetitions,tail_mean_block_weeks,tail_inference_status,tail_sign_convention
0,S1_lifted_vs_heston,lifted_hybrid_current_small,heston_hybrid_current_small,-0.028092,-0.078786,0.035509,0.7714,10000,13,COMPLETE,positive means policy A has a less negative wo...
1,S2_fixed_mixed_vs_heston,mixed_hybrid_current_small,heston_hybrid_current_small,-0.021977,-0.053939,0.023549,0.8319,10000,13,COMPLETE,positive means policy A has a less negative wo...
2,A1_supplied_history_vs_matched_current,mixed_hybrid_lag,mixed_hybrid_current_matched,-0.027257,-0.047053,-0.005575,0.9912,10000,13,COMPLETE,positive means policy A has a less negative wo...
3,A2_gru_vs_lag,mixed_hybrid_gru,mixed_hybrid_lag,0.000771,-0.012227,0.021114,0.3249,10000,13,COMPLETE,positive means policy A has a less negative wo...
4,A3_gru_vs_matched_current,mixed_hybrid_gru,mixed_hybrid_current_matched,-0.026486,-0.035700,-0.008936,0.9982,10000,13,COMPLETE,positive means policy A has a less negative wo...
5,C1_validation_selected_neural_vs_bs,mixed_hybrid_lag,black_scholes_delta,-0.014369,-0.115788,0.091250,0.5882,10000,13,COMPLETE,positive means policy A has a less negative wo...


,comparison,paired_dates,share_dates_favouring_a,median_date_effect,minimum_date_effect,maximum_date_effect
0,A1_supplied_history_vs_matched_current,208,0.269231,-0.107319,-2.018328,0.486924
1,A2_gru_vs_lag,208,0.625000,0.034694,-0.884731,0.490564
2,A3_gru_vs_matched_current,208,0.302885,-0.066429,-0.583454,0.336813
3,C1_validation_selected_neural_vs_bs,208,0.427885,-0.106301,-16.080935,0.935029
4,S1_lifted_vs_heston,208,0.706731,0.130445,-1.381442,0.710679
5,S2_fixed_mixed_vs_heston,208,0.538462,0.032292,-1.210246,0.680399


,reference_sample,feature,lower_quantile,upper_quantile,lower_threshold,upper_threshold,status
0,market_adaptation_2018_2019,rv_20d,0.333333,0.666667,0.008241,0.020792,secondary_prespecified_from_non_evaluation_sample


,source,feature,standardised_wasserstein_distance,market_coverage_by_source_1_99pct,market_reference,status
0,heston,log_moneyness,0.738102,1.000000,2018_2019_adaptation,descriptive_mechanism_diagnostic
1,lifted_heston,log_moneyness,0.702940,1.000000,2018_2019_adaptation,descriptive_mechanism_diagnostic
2,fixed_mixed,log_moneyness,0.770869,1.000000,2018_2019_adaptation,descriptive_mechanism_diagnostic
3,heston,tau_fraction,0.020147,0.986285,2018_2019_adaptation,descriptive_mechanism_diagnostic
4,lifted_heston,tau_fraction,0.019671,0.986285,2018_2019_adaptation,descriptive_mechanism_diagnostic


# 18. Dissertation figures and tables

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

DISPLAY_LABELS = {
    "S1_lifted_vs_heston": "Lifted source vs Heston source",
    "S2_fixed_mixed_vs_heston": "Fixed mixed source vs Heston source",
    "S2_legacy_mixed_vs_heston": "Legacy mixed source vs Heston source",
    "A1_supplied_history_vs_matched_current": "Lag history vs matched current",
    "A1_history_vs_legacy_current": "Lag history vs legacy current MLP",
    "A2_gru_vs_lag": "GRU vs lag-stack",
    "A3_gru_vs_matched_current": "GRU vs matched current",
    "A3_gru_vs_legacy_current": "GRU vs legacy current MLP",
    "C1_validation_selected_neural_vs_bs": "Validation-selected neural vs BS delta",
}
POLICY_LABELS = {
    "heston_hybrid_current_small": "Heston current MLP", "lifted_hybrid_current_small": "Lifted current MLP",
    "mixed_hybrid_current_small": "Mixed current MLP", "mixed_hybrid_current_matched": "Matched current MLP",
    "mixed_hybrid_lag": "Lag-stack MLP", "mixed_hybrid_gru": "GRU",
    "black_scholes_delta": "BS delta", "black_scholes_no_trade": "BS no-trade", "no_hedge": "No hedge",
}

# Figure 5.1: prespecified-contrast forest plot, with both interval families.
plot_data = CONTRASTS.copy()
plot_data["effect_pct"] = 100 * plot_data.relative_mse_improvement_a_over_b
plot_data["lo_pct"] = 100 * plot_data.ci_2_5pct; plot_data["hi_pct"] = 100 * plot_data.ci_97_5pct
plot_data["flo_pct"] = 100 * plot_data.familywise_ci_low; plot_data["fhi_pct"] = 100 * plot_data.familywise_ci_high
fig, ax = plt.subplots(figsize=(10, 6.2)); y = np.arange(len(plot_data))
ax.axvspan(-100 * FINAL_CFG.minimum_practical_relative_mse, 100 * FINAL_CFG.minimum_practical_relative_mse, alpha=.12, label="±1% practical-effect band")
ax.axvline(0, linewidth=1)
for yi, row in enumerate(plot_data.itertuples()):
    if np.isfinite(row.flo_pct) and np.isfinite(row.fhi_pct): ax.hlines(yi, row.flo_pct, row.fhi_pct, linewidth=1.3)
    if np.isfinite(row.lo_pct) and np.isfinite(row.hi_pct): ax.hlines(yi, row.lo_pct, row.hi_pct, linewidth=5, alpha=.55)
    ax.plot(row.effect_pct, yi, 'o')
ax.set_yticks(y); ax.set_yticklabels([DISPLAY_LABELS.get(x, x) for x in plot_data.comparison]); ax.invert_yaxis()
ax.set_xlabel("Paired relative-MSE effect (%) · positive favours the first policy")
ax.set_title("Paired relative-MSE effects for the prespecified source and architecture contrasts")
ax.legend(loc="best", frameon=False); fig.tight_layout()
fig.savefig(FINAL_FIGURE_DIR / "primary_contrast_forest_plot.png", dpi=200, bbox_inches="tight"); plt.show()

# Shared seed-ensemble episode panel.
primary_episode = ensemble_episode_panel(RESULTS, FINAL_CFG.primary_transaction_cost_bps)

# Figure 5.2: terminal-loss exceedance curves.
loss_policies = ["heston_hybrid_current_small", "mixed_hybrid_gru", "black_scholes_delta", "no_hedge", ("mixed_hybrid_current_small" if LEGACY_H035_REUSE else "mixed_hybrid_current_matched")]
loss_use = primary_episode.loc[primary_episode.policy.isin(loss_policies)].copy(); loss_use["loss"] = -loss_use.normalised_net_pnl
upper = float(loss_use.loss.quantile(.995)) if len(loss_use) else 1.0; thresholds = np.linspace(0, max(upper, 1.0), 240)
LOSS_EXCEEDANCE = pd.concat([
    pd.DataFrame({"policy": policy, "loss_threshold": thresholds, "exceedance_probability": [(g.loss > x).mean() for x in thresholds]})
    for policy, g in loss_use.groupby("policy")
], ignore_index=True) if len(loss_use) else pd.DataFrame()
LOSS_EXCEEDANCE.to_csv(FINAL_TABLE_DIR / "terminal_loss_exceedance_curves.csv", index=False)
fig, ax = plt.subplots(figsize=(9.5, 5.8))
for policy, g in LOSS_EXCEEDANCE.groupby("policy"):
    ax.plot(g.loss_threshold, g.exceedance_probability, label=POLICY_LABELS.get(policy, policy))
for threshold in (.25, .50, 1.00): ax.axvline(threshold, linewidth=.8, linestyle='--')
ax.set(xlabel="Terminal loss as a fraction of entry premium", ylabel="Probability loss exceeds threshold", title="Probability of terminal losses exceeding fractions of the entry premium")
ax.set_ylim(0, 1); ax.legend(frameon=False, ncol=2); fig.tight_layout()
fig.savefig(FINAL_FIGURE_DIR / "terminal_loss_exceedance_curves.png", dpi=200, bbox_inches="tight"); plt.show()

# Figure 5.3: architecture decomposition using multiplicatively coherent effects.
_arch = CONTRASTS.set_index("comparison")
_a1_label = "A1_history_vs_legacy_current" if LEGACY_H035_REUSE else "A1_supplied_history_vs_matched_current"
_a3_label = "A3_gru_vs_legacy_current" if LEGACY_H035_REUSE else "A3_gru_vs_matched_current"
a1 = float(_arch.loc[_a1_label, "relative_mse_improvement_a_over_b"])
a2 = float(_arch.loc["A2_gru_vs_lag", "relative_mse_improvement_a_over_b"])
a3 = float(_arch.loc[_a3_label, "relative_mse_improvement_a_over_b"])
composed = 1.0 - (1.0 - a1) * (1.0 - a2)
_baseline_name = "Legacy current MLP baseline" if LEGACY_H035_REUSE else "Matched current baseline"
ARCHITECTURE_DECOMPOSITION = pd.DataFrame([
    {"step": _baseline_name, "effect": 0.0, "type": "baseline"},
    {"step": "Supply 10-step history via lag-stack", "effect": a1, "type": "information plus legacy capacity difference" if LEGACY_H035_REUSE else "incremental information package"},
    {"step": "Replace lag processor with GRU", "effect": a2, "type": "conditional recurrent processing"},
    {"step": "Composed GRU vs baseline", "effect": composed, "type": "multiplicative composition"},
    {"step": "Direct GRU vs baseline", "effect": a3, "type": "prespecified direct contrast"},
])
ARCHITECTURE_DECOMPOSITION.to_csv(FINAL_TABLE_DIR / "architecture_decomposition.csv", index=False)
fig, ax = plt.subplots(figsize=(9.5, 5.2)); x = np.arange(len(ARCHITECTURE_DECOMPOSITION))
ax.bar(x, 100 * ARCHITECTURE_DECOMPOSITION.effect); ax.axhline(0, linewidth=1)
for xi, value in zip(x, 100 * ARCHITECTURE_DECOMPOSITION.effect): ax.text(xi, value, f"{value:+.2f}%", ha='center', va='bottom' if value >= 0 else 'top')
ax.set_xticks(x); ax.set_xticklabels(ARCHITECTURE_DECOMPOSITION.step, rotation=18, ha='right')
ax.set_ylabel("Relative-MSE effect (%)"); ax.set_title("Legacy history and recurrent-processing decomposition" if LEGACY_H035_REUSE else "Separating supplied observable history from recurrent processing")
fig.tight_layout(); fig.savefig(FINAL_FIGURE_DIR / "architecture_decomposition.png", dpi=200, bbox_inches="tight"); plt.show()

# Figure 5.4: three-panel maturity and cost robustness heatmap.
contrast_order = [x[2] for x in PRIMARY_CONTRAST_DEFINITIONS[:4]]; maturity_order = ["1W", "2W", "1M", "2M", "3M", "Overall"]
finite_effects = pd.to_numeric(ROBUSTNESS_CONTRAST_GRID.relative_mse_effect, errors="coerce").dropna()
lim = max(float(np.nanquantile(np.abs(finite_effects), .95)), .01) if len(finite_effects) else .1
fig, axes = plt.subplots(1, 3, figsize=(15, 5.4), sharey=True)
for ax, cost in zip(axes, [1.0, 5.0, 10.0]):
    p = ROBUSTNESS_CONTRAST_GRID.loc[ROBUSTNESS_CONTRAST_GRID.underlying_tc_bps.eq(cost)].pivot(index="comparison", columns="maturity_bucket", values="relative_mse_effect").reindex(index=contrast_order, columns=maturity_order)
    im = ax.imshow(100 * p.to_numpy(float), aspect='auto', vmin=-100*lim, vmax=100*lim, cmap='coolwarm')
    ax.set_xticks(range(len(maturity_order))); ax.set_xticklabels(maturity_order)
    ax.set_title(f"{cost:g} bp")
    for i in range(p.shape[0]):
        for j in range(p.shape[1]):
            value = p.iloc[i, j]
            if np.isfinite(value): ax.text(j, i, f"{100*value:+.1f}", ha='center', va='center', fontsize=8)
axes[0].set_yticks(range(len(contrast_order))); axes[0].set_yticklabels([DISPLAY_LABELS.get(x, x) for x in contrast_order])
fig.colorbar(im, ax=axes.ravel().tolist(), label="Relative-MSE effect (%)")
fig.suptitle("Stability of the prespecified policy contrasts across maturity and transaction costs")
fig.tight_layout(rect=[0,0,1,.94]); fig.savefig(FINAL_FIGURE_DIR / "maturity_cost_robustness_heatmap.png", dpi=200, bbox_inches="tight"); plt.show()

# Figure 5.5: risk-control frontier.
primary = POLICY_SUMMARY.loc[POLICY_SUMMARY.underlying_tc_bps.eq(FINAL_CFG.primary_transaction_cost_bps)].copy()
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.5))
axes[0].scatter(primary.mean_turnover, primary.rmse_normalised_net_pnl)
for row in primary.itertuples(): axes[0].annotate(POLICY_LABELS.get(row.policy, row.policy), (row.mean_turnover, row.rmse_normalised_net_pnl), fontsize=8)
axes[0].set(xlabel="Mean turnover", ylabel="Premium-normalised RMSE", title="Hedging error versus turnover (lower-left is better)")
axes[1].scatter(primary.rmse_normalised_net_pnl, primary.worst5_mean_normalised_net_pnl)
for row in primary.itertuples(): axes[1].annotate(POLICY_LABELS.get(row.policy, row.policy), (row.rmse_normalised_net_pnl, row.worst5_mean_normalised_net_pnl), fontsize=8)
axes[1].set(xlabel="Premium-normalised RMSE", ylabel="Worst-5% mean P&L", title="Hedging error versus lower-tail outcome (upper-left is better)")
fig.suptitle("Hedging-error, downside-risk and trading-intensity trade-offs"); fig.tight_layout(rect=[0,0,1,.94])
fig.savefig(FINAL_FIGURE_DIR / "risk_control_frontier.png", dpi=200, bbox_inches="tight"); plt.show()

# Figure 5.6: calibration-to-control comparison. Different panels use different samples and metrics.
term = globals().get("atm_skew_term_structure_comparison", pd.DataFrame()).copy()
hedge_maturity = MATURITY_SUMMARY.loc[
    MATURITY_SUMMARY.underlying_tc_bps.eq(FINAL_CFG.primary_transaction_cost_bps)
    & MATURITY_SUMMARY.policy.isin(["heston_hybrid_current_small", "lifted_hybrid_current_small"])
].copy()
fig, axes = plt.subplots(1, 2, figsize=(13, 5.4))
if len(term) and {"maturity_bucket", "heston_abs_skew_error", "lifted_abs_skew_error"}.issubset(term.columns):
    x = np.arange(len(term)); w = .38
    axes[0].bar(x-w/2, term.heston_abs_skew_error, width=w, label="Heston")
    axes[0].bar(x+w/2, term.lifted_abs_skew_error, width=w, label="Lifted Heston")
    axes[0].set_xticks(x); axes[0].set_xticklabels(term.maturity_bucket.astype(str)); axes[0].legend(frameon=False)
    axes[0].set(xlabel="Maturity", ylabel="Median absolute ATM-skew error", title="Calibration diagnostic")
else:
    axes[0].text(.5,.5,"ATM-skew calibration table unavailable",ha='center',va='center'); axes[0].set_axis_off()
for policy, g in hedge_maturity.groupby("policy"):
    g = g.set_index("maturity_bucket").reindex(["1W","2W","1M","2M","3M"]).reset_index()
    axes[1].plot(g.maturity_bucket, g.rmse_normalised_net_pnl, marker='o', label=POLICY_LABELS.get(policy, policy))
axes[1].set(xlabel="Maturity", ylabel="Held-out premium-normalised RMSE", title="Historical hedging diagnostic")
axes[1].legend(frameon=False)
fig.suptitle("Calibration fit and held-out hedging performance need not produce the same model ranking")
fig.text(.5, .01, "Panels use different samples and metrics and do not estimate a causal calibration-to-hedging relationship.", ha='center', fontsize=9)
fig.tight_layout(rect=[0,.04,1,.94]); fig.savefig(FINAL_FIGURE_DIR / "calibration_to_control_comparison.png", dpi=200, bbox_inches="tight"); plt.show()

# Figure 5.7: full accounting decomposition.
component_map = [
    ("mean_normalised_option_mark_to_market_pnl", "Option mid-price change", 1),
    ("mean_normalised_stock_holding_gain", "Stock holding gain", 1),
    ("mean_normalised_financing_pnl", "Financing", 1),
    ("mean_normalised_dividend_carry", "Dividend carry", 1),
    ("mean_normalised_option_entry_spread_cost", "Option entry spread", -1),
    ("mean_normalised_option_exit_spread_cost", "Option exit spread", -1),
    ("mean_normalised_underlying_trading_cost", "Stock trading cost", -1),
    ("mean_normalised_underlying_liquidation_cost", "Stock liquidation cost", -1),
]
ACCOUNTING_DECOMPOSITION = primary[["policy"]].copy()
for col, label, sign in component_map:
    ACCOUNTING_DECOMPOSITION[label] = sign * pd.to_numeric(primary.get(col, np.nan), errors="coerce")
ACCOUNTING_DECOMPOSITION["Mean net P&L"] = primary.mean_normalised_net_pnl.to_numpy()
ACCOUNTING_DECOMPOSITION.to_csv(FINAL_TABLE_DIR / "accounting_decomposition_by_policy.csv", index=False)
fig, ax = plt.subplots(figsize=(11.5, 6.5)); x = np.arange(len(ACCOUNTING_DECOMPOSITION)); pos=np.zeros(len(x)); neg=np.zeros(len(x))
for _col, label, _sign in component_map:
    values = ACCOUNTING_DECOMPOSITION[label].to_numpy(float)
    bottoms = np.where(values >= 0, pos, neg); ax.bar(x, values, bottom=bottoms, label=label)
    pos += np.where(values >= 0, values, 0); neg += np.where(values < 0, values, 0)
ax.scatter(x, ACCOUNTING_DECOMPOSITION["Mean net P&L"], marker='D', label="Mean net P&L")
ax.axhline(0, linewidth=1); ax.set_xticks(x); ax.set_xticklabels([POLICY_LABELS.get(p,p) for p in ACCOUNTING_DECOMPOSITION.policy], rotation=28, ha='right')
ax.set_ylabel("Mean component as fraction of entry premium"); ax.set_title("Mean terminal P&L accounting decomposition")
ax.legend(frameon=False, ncol=3, fontsize=8); fig.tight_layout()
fig.savefig(FINAL_FIGURE_DIR / "accounting_decomposition.png", dpi=200, bbox_inches="tight"); plt.show()

# Appendix diagnostics: seed stability.
fig, ax = plt.subplots(figsize=(10, 6))
for i, (policy, g) in enumerate(SEED_LEVEL_RESULTS.groupby("policy")):
    ax.scatter(g.rmse, np.full(len(g), i), alpha=.65)
    ax.scatter([g.rmse.mean()], [i], marker='D', s=55)
    ax.hlines(i, g.rmse.min(), g.rmse.max(), linewidth=1)
ax.set_yticks(range(SEED_LEVEL_RESULTS.policy.nunique())); ax.set_yticklabels([POLICY_LABELS.get(p,p) for p in sorted(SEED_LEVEL_RESULTS.policy.unique())])
ax.set_xlabel("Seed-level premium-normalised RMSE"); ax.set_title("Optimisation-seed stability and range")
fig.tight_layout(); fig.savefig(FINAL_FIGURE_DIR / "seed_stability_dot_plot.png", dpi=190, bbox_inches="tight"); plt.show()

# Appendix diagnostics: paired date-effect ECDF.
fig, ax = plt.subplots(figsize=(9.5, 5.8))
for comparison, g in PAIRED_DATE_EFFECTS.groupby("comparison"):
    values = np.sort(g.relative_mse_effect.to_numpy(float)); y = np.arange(1, len(values)+1)/len(values)
    ax.plot(100*values, y, label=DISPLAY_LABELS.get(comparison, comparison))
ax.axvline(0, linewidth=1); ax.set(xlabel="Date-level relative-MSE effect (%)", ylabel="Empirical cumulative probability", title="Distribution of paired date-level policy effects")
ax.legend(frameon=False, fontsize=8); fig.tight_layout(); fig.savefig(FINAL_FIGURE_DIR / "paired_date_effect_ecdf.png", dpi=190, bbox_inches="tight"); plt.show()

# Appendix diagnostics: error concentration.
fig, ax = plt.subplots(figsize=(8.8, 5.5))
for policy, g in ERROR_CONCENTRATION.groupby("policy"):
    ax.plot(100*g.episode_share, 100*g.cumulative_mse_share, label=POLICY_LABELS.get(policy,policy))
ax.plot([0,100],[0,100], linestyle='--', linewidth=.8); ax.set(xlabel="Largest-error episodes included (%)", ylabel="Cumulative share of total MSE (%)", title="Concentration of squared hedging error")
ax.legend(frameon=False); fig.tight_layout(); fig.savefig(FINAL_FIGURE_DIR / "error_concentration_curve.png", dpi=190, bbox_inches="tight"); plt.show()

# Appendix diagnostics: source-to-market observable alignment.
if len(SOURCE_MARKET_ALIGNMENT) and SOURCE_MARKET_ALIGNMENT.source.ne("unavailable").any():
    align = SOURCE_MARKET_ALIGNMENT.pivot(index="feature", columns="source", values="standardised_wasserstein_distance").reindex(CURRENT_FEATURES)
    fig, ax = plt.subplots(figsize=(8.5, 10)); im=ax.imshow(align.to_numpy(float), aspect='auto', cmap='viridis')
    ax.set_xticks(range(len(align.columns))); ax.set_xticklabels(align.columns, rotation=25, ha='right'); ax.set_yticks(range(len(align.index))); ax.set_yticklabels(align.index)
    fig.colorbar(im, ax=ax, label="Standardised Wasserstein distance"); ax.set_title("Synthetic-source alignment with 2018–2019 observable market features")
    fig.tight_layout(); fig.savefig(FINAL_FIGURE_DIR / "source_to_market_alignment_heatmap.png", dpi=190, bbox_inches="tight"); plt.show()

# Appendix diagnostics: regime-conditioned performance.
if len(REGIME_PERFORMANCE):
    regime_plot = REGIME_PERFORMANCE.loc[REGIME_PERFORMANCE.underlying_tc_bps.eq(FINAL_CFG.primary_transaction_cost_bps) & REGIME_PERFORMANCE.policy.isin(["heston_hybrid_current_small",("mixed_hybrid_current_small" if LEGACY_H035_REUSE else "mixed_hybrid_current_matched"),"mixed_hybrid_lag","mixed_hybrid_gru","black_scholes_delta"])].pivot(index="policy", columns="regime", values="rmse_normalised_net_pnl")
    regime_plot = regime_plot.reindex(columns=[c for c in ["calm","ordinary","stress"] if c in regime_plot.columns])
    fig, ax = plt.subplots(figsize=(7.5,5)); im=ax.imshow(regime_plot.to_numpy(float), aspect='auto', cmap='viridis_r')
    ax.set_xticks(range(len(regime_plot.columns))); ax.set_xticklabels(regime_plot.columns); ax.set_yticks(range(len(regime_plot.index))); ax.set_yticklabels([POLICY_LABELS.get(p,p) for p in regime_plot.index])
    fig.colorbar(im, ax=ax, label="Premium-normalised RMSE"); ax.set_title("Secondary performance by lagged-volatility regime")
    fig.tight_layout(); fig.savefig(FINAL_FIGURE_DIR / "regime_conditioned_performance.png", dpi=190, bbox_inches="tight"); plt.show()

# Appendix diagnostics: deterministic episode case studies using seed-ensemble decisions.
if len(CASE_STUDY_SELECTION) and len(DECISION_LEVEL_RESULTS):
    selected_ids = CASE_STUDY_SELECTION.episode_id.astype(str).tolist()
    policies = ["heston_hybrid_current_small", "mixed_hybrid_gru", "black_scholes_delta"]
    fig, axes = plt.subplots(len(selected_ids), 2, figsize=(13, 4.2*len(selected_ids)), squeeze=False)
    for row_i, episode_id in enumerate(selected_ids):
        g = DECISION_LEVEL_RESULTS.loc[DECISION_LEVEL_RESULTS.episode_id.astype(str).eq(str(episode_id)) & DECISION_LEVEL_RESULTS.policy.isin(policies)]
        spot = g.drop_duplicates("observation_step").sort_values("observation_step")
        axes[row_i,0].plot(spot.observation_step, spot.observation_spot)
        axes[row_i,0].set(title=f"{episode_id}: observed SPX path", xlabel="Decision step", ylabel="Index level")
        for policy, p in g.groupby("policy"):
            axes[row_i,1].plot(p.observation_step, p.requested_hedge, label=POLICY_LABELS.get(policy,policy))
        axes[row_i,1].set(title="Requested hedge positions", xlabel="Decision step", ylabel="Index units"); axes[row_i,1].legend(frameon=False, fontsize=8)
    fig.suptitle("Deterministically selected policy-behaviour case studies")
    fig.tight_layout(rect=[0,0,1,.97]); fig.savefig(FINAL_FIGURE_DIR / "deterministic_episode_case_studies.png", dpi=180, bbox_inches="tight"); plt.show()

# Primary policy/economic tables.
ECONOMIC_RISK_TABLE = primary[[
    "policy", "mean_normalised_net_pnl", "std_normalised_net_pnl", "mse_normalised_net_pnl",
    "rmse_normalised_net_pnl", "bias_squared_contribution", "variance_contribution", "downside_rmse",
    "worst5_mean_normalised_net_pnl", "expected_shortfall_style_loss_5pct", "mean_turnover",
    "mean_underlying_cost", "mean_option_execution_cost", "mean_total_execution_cost",
    "maximum_absolute_accounting_identity_error",
]].copy()
ECONOMIC_RISK_TABLE.to_csv(FINAL_TABLE_DIR / "economic_risk_and_costs.csv", index=False)
display(ECONOMIC_RISK_TABLE); display(ACCOUNTING_DECOMPOSITION)


<Figure size 1300x806 with 1 Axes>

<Figure size 1235x754 with 1 Axes>

<Figure size 1235x676 with 1 Axes>

C:\Users\lhkke\AppData\Local\Temp\ipykernel_2816\1375835513.py:101: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0,0,1,.94]); fig.savefig(FINAL_FIGURE_DIR / "maturity_cost_robustness_heatmap.png", dpi=200, bbox_inches="tight"); plt.show()


<Figure size 1950x702 with 4 Axes>

<Figure size 1755x715 with 2 Axes>

... (6 outputs left out)

<Figure size 975x650 with 2 Axes>

<Figure size 1690x1638 with 6 Axes>

,policy,mean_normalised_net_pnl,std_normalised_net_pnl,mse_normalised_net_pnl,rmse_normalised_net_pnl,bias_squared_contribution,variance_contribution,downside_rmse,worst5_mean_normalised_net_pnl,expected_shortfall_style_loss_5pct,mean_turnover,mean_underlying_cost,mean_option_execution_cost,mean_total_execution_cost,maximum_absolute_accounting_identity_error
0,black_scholes_delta,-0.008317,0.295154,0.087091,0.295112,0.000069,0.087022,0.236260,-0.780970,0.780970,3.167710,1.431622,2.102422,3.534044,4.973799e-12
1,black_scholes_no_trade,0.013291,0.311251,0.096949,0.311367,0.000177,0.096773,0.237036,-0.777543,0.777543,1.974537,0.897079,2.102422,2.999501,4.718004e-12
2,heston_hybrid_current_small,-0.041673,0.297524,0.090162,0.300269,0.001737,0.088425,0.248694,-0.760513,0.760513,2.943916,1.330892,2.102422,3.433314,5.385914e-12
3,lifted_hybrid_current_small,-0.049666,0.290401,0.086709,0.294464,0.002467,0.084242,0.248919,-0.788605,0.788605,3.344971,1.514684,2.102422,3.617106,5.883294e-12
4,mixed_hybrid_current_matched,-0.057210,0.296915,0.091337,0.302219,0.003273,0.088064,0.255712,-0.768082,0.768082,3.156967,1.424876,2.102422,3.527298,5.563550e-12
5,mixed_hybrid_current_small,-0.061022,0.297115,0.091906,0.303160,0.003724,0.088182,0.258554,-0.782490,0.782490,3.443334,1.557758,2.102422,3.660180,5.513812e-12
6,mixed_hybrid_gru,-0.066766,0.306307,0.098180,0.313337,0.004458,0.093723,0.268590,-0.794568,0.794568,3.275464,1.483404,2.102422,3.585826,5.016432e-12
7,mixed_hybrid_lag,-0.067779,0.309962,0.100567,0.317123,0.004594,0.095973,0.271339,-0.795338,0.795338,3.132502,1.415305,2.102422,3.517727,8.132162e-12
8,no_hedge,0.116207,0.529486,0.293558,0.541810,0.013504,0.280054,0.361874,-1.219536,1.219536,0.000000,0.000000,2.102422,2.102422,3.836931e-13


,policy,Option mid-price change,Stock holding gain,Financing,Dividend carry,Option entry spread,Option exit spread,Stock trading cost,Stock liquidation cost,Mean net P&L
0,black_scholes_delta,0.12877,-0.102607,-0.003757,-0.008589,-0.004513,-0.009059,-0.006434,-0.002128,-0.008317
1,black_scholes_no_trade,0.12877,-0.085513,-0.003025,-0.007566,-0.004513,-0.009059,-0.003812,-0.001990,0.013291
2,heston_hybrid_current_small,0.12877,-0.143472,-0.000604,-0.004691,-0.004513,-0.009059,-0.006203,-0.001901,-0.041673
3,lifted_hybrid_current_small,0.12877,-0.147885,-0.001179,-0.006672,-0.004513,-0.009059,-0.007064,-0.002063,-0.049666
4,mixed_hybrid_current_matched,0.12877,-0.159154,-0.000138,-0.004452,-0.004513,-0.009059,-0.006707,-0.001958,-0.057210
5,mixed_hybrid_current_small,0.12877,-0.161735,-0.000408,-0.004751,-0.004513,-0.009059,-0.007327,-0.001999,-0.061022
6,mixed_hybrid_gru,0.12877,-0.169657,0.000297,-0.003597,-0.004513,-0.009059,-0.007052,-0.001954,-0.066766
7,mixed_hybrid_lag,0.12877,-0.171132,0.000657,-0.003808,-0.004513,-0.009059,-0.006762,-0.001931,-0.067779
8,no_hedge,0.12877,0.000000,0.001009,0.000000,-0.004513,-0.009059,-0.000000,-0.000000,0.116207


# 19. Final evidence gates and claim-evidence map

In [ ]:
latent_failures=[]
for episode in [*MARKET_TRAIN,*MARKET_VALIDATION,*MARKET_TEST,*MARKET_CONFIRMATORY,*HESTON_TRAIN,*LIFTED_TRAIN]:
    try: assert_episode_observability(episode,FINAL_CFG.history_length)
    except Exception as exc: latent_failures.append(f"{episode.get('episode_id','unknown')}: {exc}")

required_cells=set(CELL_SPECS); observed_cells=set(CHECKPOINT_MANIFEST.policy.unique())
required_primary={x[2] for x in PRIMARY_CONTRAST_DEFINITIONS}; observed_primary=set(CONTRASTS.comparison)
capacity_ok=(not LEGACY_H035_REUSE) and capacity_audit.iloc[:2].status.eq("PASS").all()
source_support_ok=len(HESTON_TRAIN)==len(LIFTED_TRAIN)==len(MIXED_TRAIN) and len(HESTON_VALIDATION)==len(LIFTED_VALIDATION)==len(MIXED_VALIDATION)
seed_ok=CHECKPOINT_MANIFEST.groupby("policy").training_seed.nunique().reindex(sorted(required_cells)).eq(len(FINAL_CFG.training_seeds)).all()
finite_cols=["normalised_net_pnl","normalised_squared_error","turnover","underlying_cost","option_execution_cost","total_execution_cost","accounting_identity_error"]
EVIDENCE_GATE=pd.DataFrame([
    {"gate":"lag_consistent_h035_full_profile","status":"PASS" if FINAL_CFG.run_profile=="full" and not LEGACY_H035_REUSE else "FAIL","critical":True},
    {"gate":"no_latent_or_future_policy_fields","status":"PASS" if not latent_failures else "FAIL","critical":True},
    {"gate":"next_observation_execution","status":"PASS" if TRAIN_CFG.execution_lag_steps==1 else "FAIL","critical":True},
    {"gate":"required_neural_cells_complete","status":"PASS" if observed_cells==required_cells else "FAIL","critical":True},
    {"gate":"capacity_matched_current_cell_executed","status":"FAIL_LEGACY_UNAVAILABLE" if LEGACY_H035_REUSE else ("PASS" if "mixed_hybrid_current_matched" in observed_cells else "FAIL"),"critical":True},
    {"gate":"minimum_ten_common_seeds","status":"PASS" if len(FINAL_CFG.training_seeds)>=10 and seed_ok else "FAIL","critical":True},
    {"gate":"matched_current_lag_gru_within_2pct","status":"PASS" if capacity_ok else "FAIL","critical":True},
    {"gate":"equal_source_bank_support","status":"PASS" if source_support_ok else "FAIL","critical":True},
    {"gate":"source_calibrations_pre_2018","status":"PASS" if pd.to_datetime(SOURCE_SEED_PANEL.date).max()<=pd.Timestamp(FINAL_CFG.simulation_seed_end) else "FAIL","critical":True},
    {"gate":"checkpoint_selection_2020_only","status":"PASS" if CHECKPOINT_MANIFEST.selection_domain.eq("market_validation_2020").all() else "FAIL","critical":True},
    {"gate":"no_trade_tuned_inside_expanded_grid","status":"PASS" if SELECTED_BS_NO_TRADE_BAND in set(MKT_CFG.no_trade_band_candidates) else "FAIL","critical":True},
    {"gate":"six_primary_contrasts_available","status":"PASS" if observed_primary==required_primary else "FAIL","critical":True},
    {"gate":"finite_episode_results","status":"PASS" if np.isfinite(RESULTS[finite_cols].to_numpy(float)).all() else "FAIL","critical":True},
    {"gate":"accounting_decomposition_identity","status":"PASS" if RESULTS.accounting_identity_error.abs().max() < 2e-3 else "FAIL","critical":True},
    {"gate":"paired_tail_intervals_available","status":"PASS" if len(TAIL_CONTRASTS)==len(PRIMARY_CONTRAST_DEFINITIONS) and TAIL_CONTRASTS.tail_inference_status.eq("COMPLETE").all() else "FAIL","critical":True},
    {"gate":"argument_led_main_figures","status":"PASS" if all((FINAL_FIGURE_DIR/name).exists() for name in ["primary_contrast_forest_plot.png","terminal_loss_exceedance_curves.png","architecture_decomposition.png","maturity_cost_robustness_heatmap.png","risk_control_frontier.png","calibration_to_control_comparison.png","accounting_decomposition.png"]) else "FAIL","critical":True},
    {"gate":"decision_level_policy_export","status":"PASS" if len(DECISION_LEVEL_RESULTS)>0 else "UNAVAILABLE","critical":False},
    {"gate":"untouched_2025_holdout","status":"PASS" if len(MARKET_CONFIRMATORY)>0 else "UNAVAILABLE","critical":False},
])
EVIDENCE_GATE.to_csv(FINAL_TABLE_DIR/"evidence_gate.csv",index=False); display(EVIDENCE_GATE)
if EVIDENCE_GATE.loc[EVIDENCE_GATE.critical,"status"].ne("PASS").any():
    if FINAL_CFG.is_full_evidence_run:
        raise AssertionError("A critical final evidence gate failed.")
    print(
        "The lag-consistent experiment completed with documented design limitations. "
        "Failed robustness gates, such as fewer than ten training runs, must remain explicit."
    )

claim_rows=[]
for row in CONTRASTS.itertuples():
    point=float(row.relative_mse_improvement_a_over_b); lo=float(row.ci_2_5pct); hi=float(row.ci_97_5pct); flo=float(row.familywise_ci_low); fhi=float(row.familywise_ci_high)
    if flo>0 or fhi<0: status="supported"
    elif lo>0 or hi<0: status="suggestive but multiplicity-unresolved"
    else: status="inconclusive"
    wording=f"the point estimate for {row.comparison} was {100*point:.2f}% in relative MSE; the conventional 95% interval was [{100*lo:.2f}%, {100*hi:.2f}%] and the family-wise interval was [{100*flo:.2f}%, {100*fhi:.2f}%]. The evidence status is {status}."
    claim_rows.append({"claim_id":row.comparison,"estimand":"paired date-weighted relative MSE","evidence_table":"tables/primary_contrasts.csv","uncertainty":"stationary date bootstrap plus hierarchical seed/date sensitivity","confounds":"conditional on information set, source implementation, training budget and accounting","evidence_status":status,"permitted_wording":wording})
for row in TAIL_CONTRASTS.itertuples():
    if getattr(row, "tail_inference_status", "") != "COMPLETE":
        tail_status = "not tested"
        tail_wording = f"The lower-tail contrast for {row.comparison} was unavailable."
    else:
        lo=float(row.tail_ci_2_5pct); hi=float(row.tail_ci_97_5pct); point=float(row.worst5_mean_difference_a_minus_b)
        tail_status = "supported" if (lo>0 or hi<0) else "inconclusive"
        tail_wording = f"For {row.comparison}, the paired worst-five-percent mean P&L difference was {point:.4f}; the 95% date-block interval was [{lo:.4f}, {hi:.4f}]. Positive values favour policy A. The tail evidence status is {tail_status}."
    claim_rows.append({"claim_id":f"tail_{row.comparison}","estimand":"paired difference in worst-five-percent mean premium-normalised P&L","evidence_table":"tables/paired_tail_contrasts.csv","uncertainty":"stationary ordered-date bootstrap recomputing the global lower-tail statistic","confounds":"secondary tail family; conditional on fitted seed ensemble and evaluation panel","evidence_status":tail_status,"permitted_wording":tail_wording})
for claim in ["latent_factor_sufficiency","market_markovianity","absence_of_path_dependence","general_heston_superiority","gru_recovers_lifted_factors"]:
    claim_rows.append({"claim_id":claim,"estimand":"none","evidence_table":"not applicable","uncertainty":"not applicable","confounds":"outside ranked observable experiment","evidence_status":"not tested","permitted_wording":f"The ranked experiment does not test {claim.replace('_',' ')}."})
CLAIM_EVIDENCE_MAP=pd.DataFrame(claim_rows); CLAIM_EVIDENCE_MAP.to_csv(FINAL_RESULTS_DIR/"claim_evidence_ledger.csv",index=False); display(CLAIM_EVIDENCE_MAP)


,gate,status,critical
0,lag_consistent_h035_full_profile,PASS,True
1,no_latent_or_future_policy_fields,PASS,True
2,next_observation_execution,PASS,True
3,required_neural_cells_complete,PASS,True
4,capacity_matched_current_cell_executed,PASS,True
5,minimum_ten_common_seeds,FAIL,True
6,matched_current_lag_gru_within_2pct,PASS,True
7,equal_source_bank_support,PASS,True
8,source_calibrations_pre_2018,PASS,True
9,checkpoint_selection_2020_only,PASS,True


The lag-consistent experiment completed with documented design limitations. Failed robustness gates, such as fewer than ten training runs, must remain explicit.


,claim_id,estimand,evidence_table,uncertainty,confounds,evidence_status,permitted_wording
0,S1_lifted_vs_heston,paired date-weighted relative MSE,tables/primary_contrasts.csv,stationary date bootstrap plus hierarchical se...,"conditional on information set, source impleme...",inconclusive,the point estimate for S1_lifted_vs_heston was...
1,S2_fixed_mixed_vs_heston,paired date-weighted relative MSE,tables/primary_contrasts.csv,stationary date bootstrap plus hierarchical se...,"conditional on information set, source impleme...",inconclusive,the point estimate for S2_fixed_mixed_vs_hesto...
2,A1_supplied_history_vs_matched_current,paired date-weighted relative MSE,tables/primary_contrasts.csv,stationary date bootstrap plus hierarchical se...,"conditional on information set, source impleme...",supported,the point estimate for A1_supplied_history_vs_...
3,A2_gru_vs_lag,paired date-weighted relative MSE,tables/primary_contrasts.csv,stationary date bootstrap plus hierarchical se...,"conditional on information set, source impleme...",inconclusive,the point estimate for A2_gru_vs_lag was 1.32%...
4,A3_gru_vs_matched_current,paired date-weighted relative MSE,tables/primary_contrasts.csv,stationary date bootstrap plus hierarchical se...,"conditional on information set, source impleme...",supported,the point estimate for A3_gru_vs_matched_curre...
5,C1_validation_selected_neural_vs_bs,paired date-weighted relative MSE,tables/primary_contrasts.csv,stationary date bootstrap plus hierarchical se...,"conditional on information set, source impleme...",inconclusive,the point estimate for C1_validation_selected_...
6,tail_S1_lifted_vs_heston,paired difference in worst-five-percent mean p...,tables/paired_tail_contrasts.csv,stationary ordered-date bootstrap recomputing ...,secondary tail family; conditional on fitted s...,inconclusive,"For S1_lifted_vs_heston, the paired worst-five..."
7,tail_S2_fixed_mixed_vs_heston,paired difference in worst-five-percent mean p...,tables/paired_tail_contrasts.csv,stationary ordered-date bootstrap recomputing ...,secondary tail family; conditional on fitted s...,inconclusive,"For S2_fixed_mixed_vs_heston, the paired worst..."
8,tail_A1_supplied_history_vs_matched_current,paired difference in worst-five-percent mean p...,tables/paired_tail_contrasts.csv,stationary ordered-date bootstrap recomputing ...,secondary tail family; conditional on fitted s...,supported,"For A1_supplied_history_vs_matched_current, th..."
9,tail_A2_gru_vs_lag,paired difference in worst-five-percent mean p...,tables/paired_tail_contrasts.csv,stationary ordered-date bootstrap recomputing ...,secondary tail family; conditional on fitted s...,inconclusive,"For A2_gru_vs_lag, the paired worst-five-perce..."


# 20. Immutable run manifest and dissertation export contract

In [ ]:
from dataclasses import asdict, is_dataclass
from pathlib import Path
from typing import Any
import math


def _json_key(value: Any) -> str:
    """Convert any mapping key into a deterministic JSON string key."""
    if isinstance(value, tuple):
        return " | ".join(_json_key(item) for item in value)

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, (str, int, float, bool)) or value is None:
        return str(value)

    return repr(value)


def make_json_safe(value: Any) -> Any:
    """
    Recursively convert research objects into standards-compliant JSON.

    Handles:
    - dictionaries with tuple keys;
    - dataclasses;
    - pathlib paths;
    - NumPy scalars and arrays;
    - pandas timestamps and series;
    - sets and tuples;
    - PyTorch tensors;
    - NaN and infinity.
    """
    if is_dataclass(value) and not isinstance(value, type):
        return make_json_safe(asdict(value))

    if isinstance(value, dict):
        return {
            _json_key(key): make_json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple, set, frozenset)):
        return [make_json_safe(item) for item in value]

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    if isinstance(value, pd.Timedelta):
        return value.isoformat()

    if isinstance(value, pd.Series):
        return make_json_safe(value.to_dict())

    if isinstance(value, pd.Index):
        return make_json_safe(value.tolist())

    if isinstance(value, pd.DataFrame):
        return make_json_safe(value.to_dict(orient="records"))

    if isinstance(value, np.ndarray):
        return make_json_safe(value.tolist())

    if isinstance(value, np.generic):
        return make_json_safe(value.item())

    if TORCH_AVAILABLE and isinstance(value, torch.Tensor):
        if value.ndim == 0:
            return make_json_safe(value.detach().cpu().item())
        return make_json_safe(value.detach().cpu().tolist())

    if isinstance(value, float):
        return value if math.isfinite(value) else None

    if isinstance(value, (str, int, bool)) or value is None:
        return value

    return str(value)

In [ ]:
RUN_MANIFEST={
    "protocol_hash":FINAL_PROTOCOL_HASH,"calibration_component_hash":CALIBRATION_COMPONENT_HASH,
    "protocol_version":FINAL_CFG.protocol_version,"implementation_version":FINAL_CFG.implementation_version,
    "run_profile":FINAL_CFG.run_profile,"execution_request":asdict(RUN),"full_evidence_run":FINAL_CFG.is_full_evidence_run,
    "calibration_spec":FINAL_CFG.calibration_spec,"actual_calibration_H":FINAL_CFG.H,"created_utc":pd.Timestamp.utcnow().isoformat(),
    "final_config":asdict(FINAL_CFG),"training_config":asdict(TRAIN_CFG),"calibration_config":asdict(CFG.lifted),
    "device":str(DEVICE),"policy_access_to_latent_simulator_state":"None","execution_lag_steps":TRAIN_CFG.execution_lag_steps,
    "decision_timeline":{
        "observe": OBSERVATION_TIME_CONVENTION,
        "construct": "use only information available by the completed date-t snapshot; explicit history ends at t-1",
        "decide": DECISION_TIME_CONVENTION,
        "execute": EXECUTION_TIME_CONVENTION,
        "execution_price": EXECUTION_PRICE_CONVENTION,
        "execution_window": EXECUTION_WINDOW_CONVENTION,
        "hold": "executed position is held from t+1 to the following recorded node",
        "liquidate": "final stock proxy at the last index level and option at ask",
    },
    "source_panel_hash":SOURCE_PANEL_HASH,"source_panel_date_max":str(pd.to_datetime(SOURCE_SEED_PANEL.date).max().date()),
    "source_bank_support":SOURCE_BANK_SUPPORT_AUDIT.to_dict("records"),"current_features":CURRENT_FEATURES,"history_features":HISTORY_FEATURES,
    "architecture_manifest":ARCHITECTURE_MANIFEST.to_dict("records"),"capacity_matching_audit":capacity_audit.to_dict("records"),
    "market_counts":{role:len(episodes) for role,episodes in MARKET_BY_ROLE.items()},"market_date_counts":{role:int(pd.to_datetime([ep["entry_date"] for ep in episodes]).nunique()) if episodes else 0 for role,episodes in MARKET_BY_ROLE.items()},"confirmatory_holdout_status":HOLDOUT_STATUS.to_dict("records"),
    "selected_bs_no_trade_band":SELECTED_BS_NO_TRADE_BAND,"checkpoints":CHECKPOINT_MANIFEST.to_dict("records"),
    "primary_contrasts":PRIMARY_CONTRAST_DEFINITIONS,"tail_contrasts":TAIL_CONTRASTS.to_dict("records"),"preferred_neural_policy_validation_selected":PREFERRED_NEURAL_POLICY,
    "diagnostic_reporting_version":FINAL_CFG.diagnostic_reporting_version,"decision_level_export_status":DECISION_LEVEL_EXPORT_STATUS.to_dict("records"),
    "main_figures":["primary_contrast_forest_plot.png","terminal_loss_exceedance_curves.png","architecture_decomposition.png","maturity_cost_robustness_heatmap.png","risk_control_frontier.png","calibration_to_control_comparison.png","accounting_decomposition.png"],
    "evidence_status": FINAL_CFG.evidence_status,
    "limitations":[
        "The source contrast estimates complete calibrated numerical pipelines, not roughness or the continuous-time law in isolation.",
        "The 2021-2024 panel was inspected during development and is labelled post-development frozen evidence.",
        "A 2025 confirmatory claim is permitted only when both option and index inputs support a non-empty untouched cohort.",
        "Intervals from the date bootstrap condition on the fitted seed ensemble; hierarchical sensitivity additionally resamples seeds.",
        "The study uses the next recorded CRSP index level as a daily execution proxy; the data do not contain timestamped executable intraday hedge quotes.",
        "SPX is an index rather than a directly tradable stock, so index-level hedge execution is an idealisation.",
    ],
}
manifest_path=FINAL_RESULTS_DIR/"run_manifest.json"; manifest_path.write_text(json.dumps(make_json_safe(RUN_MANIFEST),indent=2,sort_keys=True,allow_nan=False),encoding="utf-8")
with manifest_path.open("r",encoding="utf-8") as h: json.load(h)
REPRODUCIBILITY_MANIFEST={"run_manifest":str(manifest_path),"notebook_protocol_hash":FINAL_PROTOCOL_HASH,"calibration_component_hash":CALIBRATION_COMPONENT_HASH,"python":os.sys.version,"torch":torch.__version__,"numpy":np.__version__,"pandas":pd.__version__,"device":str(DEVICE),"checkpoint_count":len(CHECKPOINT_MANIFEST),"all_checkpoint_hashes_recorded":CHECKPOINT_MANIFEST.checkpoint_sha256.notna().all()}
(FINAL_RESULTS_DIR/"reproducibility_manifest.json").write_text(json.dumps(make_json_safe(REPRODUCIBILITY_MANIFEST),indent=2,sort_keys=True,allow_nan=False))
print("Run manifest written and validated:",manifest_path)


Run manifest written and validated: C:\Users\lhkke\OneDrive - Queen Mary, University of London\Documents\Dissertation\Deep-Hedging-under-Lifted-Heston-MSc-Dissertation-\outputs_observable_h035_lag_consistent_retrained_full\run_manifest.json


# 21. Calibration, simulation and common-quality audit


In [ ]:
def build_calibration_simulation_audit():
    rows=[]
    for model,cal,fit in [("heston",heston_calibration,globals().get("heston_fit_table",pd.DataFrame())),("lifted_heston",lifted_calibration,globals().get("lifted_fit_table",pd.DataFrame()))]:
        dates=pd.to_datetime(cal.get("date",pd.Series(dtype="datetime64[ns]")),errors="coerce").dropna().dt.normalize()
        row={"model":model,"attempted_dates":int(surface_panel.date.nunique()),"accepted_dates":int(dates.nunique()),"rejected_dates":int(max(surface_panel.date.nunique()-dates.nunique(),0)),
             "date_min":dates.min() if len(dates) else pd.NaT,"date_max":dates.max() if len(dates) else pd.NaT}
        for col in ["loss","vega_scaled_rmse","price_rmse","cpu_seconds","optimizer_nfev","martingale_error"]:
            if col in cal.columns: row[f"median_{col}"]=float(pd.to_numeric(cal[col],errors="coerce").median())
        rows.append(row)
    return pd.DataFrame(rows)
CALIBRATION_SIMULATION_AUDIT=build_calibration_simulation_audit()
CALIBRATION_SIMULATION_AUDIT.to_csv(FINAL_TABLE_DIR/"calibration_simulation_audit.csv",index=False)
# Preserve all detailed diagnostic exports already produced by Sections 4-6; create a crosswalk rather than hiding their provenance.
audit_files=[]
for p in sorted(FINAL_TABLE_DIR.glob("*.csv")):
    name=p.name.lower()
    if any(token in name for token in ["calibration","fit_","skew","moneyness","maturity","forward_variance","martingale","feasibility","parameter","kernel"]):
        audit_files.append({"file":p.name,"sha256":file_sha256(p),"bytes":p.stat().st_size})
CALIBRATION_AUDIT_CROSSWALK=pd.DataFrame(audit_files); CALIBRATION_AUDIT_CROSSWALK.to_csv(FINAL_TABLE_DIR/"calibration_audit_crosswalk.csv",index=False)

# Common-quality sensitivity dates are selected by thresholds fixed from the common pre-2018 panel.
def common_quality_dates():
    hs=globals().get("heston_standard_date_summary",globals().get("heston_date_summary",pd.DataFrame())).copy()
    ls=globals().get("lifted_heston_standard_date_summary",globals().get("lifted_date_summary",pd.DataFrame())).copy()
    if hs.empty or ls.empty or "date" not in hs or "date" not in ls: return pd.DataFrame(columns=["date","common_quality_pass"])
    metric=next((c for c in ["vega_weighted_iv_rmse","iv_rmse","vega_scaled_rmse"] if c in hs.columns and c in ls.columns),None)
    if metric is None: return pd.DataFrame(columns=["date","common_quality_pass"])
    h=hs[["date",metric]].rename(columns={metric:"heston_metric"}); l=ls[["date",metric]].rename(columns={metric:"lifted_metric"}); q=h.merge(l,on="date",how="inner")
    hthr=float(q.heston_metric.quantile(.90)); lthr=float(q.lifted_metric.quantile(.90)); q["common_quality_pass"]=q.heston_metric.le(hthr)&q.lifted_metric.le(lthr); q["heston_threshold"]=hthr; q["lifted_threshold"]=lthr
    return q
COMMON_QUALITY_DATES=common_quality_dates(); COMMON_QUALITY_DATES.to_csv(FINAL_TABLE_DIR/"common_quality_calibration_dates.csv",index=False)
display(CALIBRATION_SIMULATION_AUDIT); display(COMMON_QUALITY_DATES.head())


,model,attempted_dates,accepted_dates,rejected_dates,date_min,date_max,median_loss,median_vega_scaled_rmse,median_price_rmse,median_cpu_seconds,median_optimizer_nfev
0,heston,2869,2869,0,2010-01-04,2024-12-31,0.000001,0.000828,0.354321,0.620141,14.0
1,lifted_heston,2869,2161,708,2010-01-04,2024-12-19,0.000001,0.001221,0.523203,19.651755,13.0


,date,common_quality_pass


# 22. Reproducibility, chronology and final export audit


In [ ]:
CHRONOLOGY_OBSERVABILITY_AUDIT = pd.DataFrame([
    {
        "item": "information observation",
        "implementation": "completed end-of-day OptionMetrics quote snapshot and CRSP index level at date t",
        "status": "PASS",
    },
    {
        "item": "feature construction",
        "implementation": "current features use information available by date t; explicit sequence inputs end strictly at t-1",
        "status": "PASS",
    },
    {
        "item": "hedge decision",
        "implementation": "requested position is computed after the date-t observable snapshot",
        "status": "PASS",
    },
    {
        "item": "simulation pretraining and validation",
        "implementation": "requested positions are shifted by one node before differentiable P&L is computed",
        "status": "PASS",
    },
    {
        "item": "historical fine-tuning and 2020 model selection",
        "implementation": "the same one-observation execution delay is used in every market loss",
        "status": "PASS",
    },
    {
        "item": "benchmark tuning and out-of-sample evaluation",
        "implementation": "Black-Scholes, no-trade and neural strategies all use the same one-observation delay",
        "status": "PASS",
    },
    {
        "item": "execution price",
        "implementation": "next recorded CRSP index level is used as the daily underlying-price proxy",
        "status": "LIMITATION_DAILY_PROXY_NOT_TIMESTAMPED_FILL",
    },
    {
        "item": "holding period",
        "implementation": "the post-execution position accrues financing and dividend carry until the following node",
        "status": "PASS",
    },
    {
        "item": "liquidation",
        "implementation": "the stock proxy is liquidated at the final index level and the option is closed at the ask",
        "status": "PASS",
    },
    {
        "item": "latent state",
        "implementation": "latent variance and Lifted Heston factors are prohibited from policy tensors, scalers, hidden-state initialisation, targets and selection",
        "status": "PASS",
    },
])
CHRONOLOGY_OBSERVABILITY_AUDIT.to_csv(
    FINAL_TABLE_DIR / "observability_chronology_audit.csv",
    index=False,
)

CHANGE_LOG = pd.DataFrame([
    {
        "change": "Lag-consistent neural retraining",
        "why": "remove the mismatch between immediate-execution training and delayed-execution evaluation",
        "threat_resolved": "training-to-evaluation timing shift",
        "effect": "all neural checkpoints and model-stage selections",
    },
    {
        "change": "Timing contract embedded in every episode, checkpoint and evaluation manifest",
        "why": "make stale immediate-execution artefacts semantically incompatible",
        "threat_resolved": "silent checkpoint or cache reuse",
        "effect": "reproducibility",
    },
    {
        "change": "Fresh observable simulation banks under the new protocol hash",
        "why": "bind synthetic observations to the same declared chronology",
        "threat_resolved": "ambiguous source-bank provenance",
        "effect": "simulation pretraining",
    },
    {
        "change": "Fixed-size mixed simulation bank",
        "why": "compare source composition without doubling total simulated support",
        "threat_resolved": "source-sample-size confounding",
        "effect": "source comparison",
    },
    {
        "change": "Parameter-matched baseline-feature MLP",
        "why": "separate explicit lagged inputs from model capacity",
        "threat_resolved": "architecture-information confounding",
        "effect": "architecture comparison",
    },
    {
        "change": "Common lagged accounting for Black-Scholes benchmark tuning",
        "why": "ensure benchmark selection solves the same execution problem",
        "threat_resolved": "benchmark timing mismatch",
        "effect": "2020 validation and reported benchmark",
    },
    {
        "change": "Double-precision accounting decomposition",
        "why": "separate numerical round-off from genuine accounting errors",
        "threat_resolved": "false accounting-identity failures",
        "effect": "training and evaluation diagnostics",
    },
])
CHANGE_LOG.to_csv(FINAL_RESULTS_DIR / "change_log.csv", index=False)

UNRESOLVED_ISSUES = pd.DataFrame([
    {
        "issue": "Only the configured optimisation seeds are run; the default full profile uses three.",
        "status": "documented",
        "claim_effect": "training-run uncertainty remains imprecisely estimated unless FINAL_TRAINING_SEEDS is expanded",
    },
    {
        "issue": "Daily OptionMetrics and CRSP data do not provide a timestamped executable intraday hedge quote.",
        "status": "structural data limitation",
        "claim_effect": "next-recorded-close execution is a conservative daily proxy rather than a literal fill model",
    },
    {
        "issue": "SPX is an index rather than a directly tradable stock.",
        "status": "market-implementation idealisation",
        "claim_effect": "results describe index-level stock-only hedging and do not include futures or ETF basis and execution frictions",
    },
    {
        "issue": "The 2021-2024 panel was inspected during development.",
        "status": "post-development out-of-sample evidence",
        "claim_effect": "the period is not an analyst-blind confirmatory holdout",
    },
    {
        "issue": "A pristine 2025 holdout depends on overlapping OptionMetrics and CRSP coverage.",
        "status": "resolved automatically at runtime",
        "claim_effect": "otherwise no independent 2025 confirmation is reported",
    },
])
UNRESOLVED_ISSUES.to_csv(FINAL_RESULTS_DIR / "unresolved_issues.csv", index=False)

display(CHRONOLOGY_OBSERVABILITY_AUDIT)
display(CHANGE_LOG)
display(UNRESOLVED_ISSUES)


UNRESOLVED_ISSUES.to_csv(
    FINAL_RESULTS_DIR / "unresolved_issues_register.csv",
    index=False,
)

FINAL_RESULT_MANIFEST = {
    "protocol_hash": FINAL_PROTOCOL_HASH,
    "timing_contract": {
        "information_time": OBSERVATION_TIME_CONVENTION,
        "decision_time": DECISION_TIME_CONVENTION,
        "execution_time": EXECUTION_TIME_CONVENTION,
        "execution_price": EXECUTION_PRICE_CONVENTION,
        "execution_window": EXECUTION_WINDOW_CONVENTION,
        "execution_lag_steps": EXECUTION_LAG_STEPS,
    },
    "evidence_gate": str(FINAL_TABLE_DIR / "evidence_gate.csv"),
    "primary_contrasts": str(FINAL_TABLE_DIR / "primary_contrasts.csv"),
    "tail_contrasts": str(FINAL_TABLE_DIR / "paired_tail_contrasts.csv"),
    "main_figures": {
        name: str(FINAL_FIGURE_DIR / name)
        for name in [
            "primary_contrast_forest_plot.png",
            "terminal_loss_exceedance_curves.png",
            "architecture_decomposition.png",
            "maturity_cost_robustness_heatmap.png",
            "risk_control_frontier.png",
            "calibration_to_control_comparison.png",
            "accounting_decomposition.png",
        ]
    },
    "economic_risk_table": str(FINAL_TABLE_DIR / "economic_risk_and_costs.csv"),
    "accounting_decomposition": str(FINAL_TABLE_DIR / "accounting_decomposition_by_policy.csv"),
    "decision_level_export": str(FINAL_TABLE_DIR / "decision_level_results.parquet"),
    "claim_ledger": str(FINAL_RESULTS_DIR / "claim_evidence_ledger.csv"),
    "ready_for_dissertation_integration": bool(
        EVIDENCE_GATE.loc[EVIDENCE_GATE.critical, "status"].eq("PASS").all()
    ),
}
(FINAL_RESULTS_DIR / "final_result_manifest.json").write_text(
    json.dumps(FINAL_RESULT_MANIFEST, indent=2, sort_keys=True)
)


,item,implementation,status
0,information observation,completed end-of-day OptionMetrics quote snaps...,PASS
1,feature construction,current features use information available by ...,PASS
2,hedge decision,requested position is computed after the date-...,PASS
3,simulation pretraining and validation,requested positions are shifted by one node be...,PASS
4,historical fine-tuning and 2020 model selection,the same one-observation execution delay is us...,PASS
5,benchmark tuning and out-of-sample evaluation,"Black-Scholes, no-trade and neural strategies ...",PASS
6,execution price,next recorded CRSP index level is used as the ...,LIMITATION_DAILY_PROXY_NOT_TIMESTAMPED_FILL
7,holding period,the post-execution position accrues financing ...,PASS
8,liquidation,the stock proxy is liquidated at the final ind...,PASS
9,latent state,latent variance and Lifted Heston factors are ...,PASS


,change,why,threat_resolved,effect
0,Lag-consistent neural retraining,remove the mismatch between immediate-executio...,training-to-evaluation timing shift,all neural checkpoints and model-stage selections
1,"Timing contract embedded in every episode, che...",make stale immediate-execution artefacts seman...,silent checkpoint or cache reuse,reproducibility
2,Fresh observable simulation banks under the ne...,bind synthetic observations to the same declar...,ambiguous source-bank provenance,simulation pretraining
3,Fixed-size mixed simulation bank,compare source composition without doubling to...,source-sample-size confounding,source comparison
4,Parameter-matched baseline-feature MLP,separate explicit lagged inputs from model cap...,architecture-information confounding,architecture comparison
5,Common lagged accounting for Black-Scholes ben...,ensure benchmark selection solves the same exe...,benchmark timing mismatch,2020 validation and reported benchmark
6,Double-precision accounting decomposition,separate numerical round-off from genuine acco...,false accounting-identity failures,training and evaluation diagnostics


,issue,status,claim_effect
0,Only the configured optimisation seeds are run...,documented,training-run uncertainty remains imprecisely e...
1,Daily OptionMetrics and CRSP data do not provi...,structural data limitation,next-recorded-close execution is a conservativ...
2,SPX is an index rather than a directly tradabl...,market-implementation idealisation,results describe index-level stock-only hedgin...
3,The 2021-2024 panel was inspected during devel...,post-development out-of-sample evidence,the period is not an analyst-blind confirmator...
4,A pristine 2025 holdout depends on overlapping...,resolved automatically at runtime,otherwise no independent 2025 confirmation is ...


4318

In [ ]:
# ============================================================
# Final post-training reporting and evaluation audit
# ============================================================

def _date_equal_mse(frame: pd.DataFrame, value_col: str = "normalised_squared_error") -> float:
    use = frame.copy()
    use["entry_date"] = pd.to_datetime(use["entry_date"])
    by_bucket = use.groupby(["entry_date", "maturity_bucket"], as_index=False)[value_col].mean()
    by_date = by_bucket.groupby("entry_date", as_index=False)[value_col].mean()
    return float(by_date[value_col].mean()) if len(by_date) else np.nan


def _checkpoint_selection_audit(manifest: pd.DataFrame) -> pd.DataFrame:
    rows = []
    stage_map = {
        "simulation_pretrained": ("pretrained_market_episode_weighted_mse", "pretrained_market_date_weighted_mse", "pretrained_market_validation_mse"),
        "head_only": ("head_market_episode_weighted_mse", "head_market_date_weighted_mse", "head_market_validation_mse"),
        "full": ("full_market_episode_weighted_mse", "full_market_date_weighted_mse", "full_market_validation_mse"),
    }
    for record in manifest.to_dict("records"):
        for stage, (episode_col, date_col, legacy_col) in stage_map.items():
            episode_value = record.get(episode_col, np.nan)
            date_value = record.get(date_col, np.nan)
            if not np.isfinite(pd.to_numeric(episode_value, errors="coerce")) and stage == record.get("selected_stage"):
                episode_value = record.get("validation_mse", record.get(legacy_col, np.nan))
            rows.append({
                "policy": record.get("policy"),
                "training_seed": record.get("training_seed"),
                "candidate_stage": stage,
                "episode_weighted_mse": episode_value,
                "date_weighted_mse": date_value,
                "selected_stage_in_checkpoint": record.get("selected_stage"),
                "selected_under_checkpoint_rule": stage == record.get("selected_stage"),
                "selection_weighting": record.get("selection_weighting", "episode_weighted_cached_checkpoint"),
                "audit_status": "complete" if np.isfinite(pd.to_numeric(date_value, errors="coerce")) else "date_weighted_candidate_state_not_stored_in_cached_H035_checkpoint",
            })
    audit = pd.DataFrame(rows)
    if not audit.empty:
        audit["episode_rank"] = audit.groupby(["policy", "training_seed"])["episode_weighted_mse"].rank(method="min")
        audit["date_rank"] = audit.groupby(["policy", "training_seed"])["date_weighted_mse"].rank(method="min")
    return audit


CHECKPOINT_SELECTION_AGGREGATION_AUDIT = _checkpoint_selection_audit(CHECKPOINT_MANIFEST)
CHECKPOINT_SELECTION_AGGREGATION_AUDIT.to_csv(FINAL_TABLE_DIR / "checkpoint_selection_aggregation_audit.csv", index=False)

# No-trade band can be audited exactly because the analytic candidates require no saved neural state.
BS_NO_TRADE_VALIDATION.to_csv(FINAL_TABLE_DIR / "bs_no_trade_validation_both_weightings.csv", index=False)
BS_SELECTION_AGREEMENT = pd.DataFrame([{
    "selected_reported_band": float(SELECTED_BS_NO_TRADE_BAND),
    "episode_weighted_band": float(BS_NO_TRADE_VALIDATION.sort_values(["episode_weighted_mse", "band"]).iloc[0]["band"]),
    "date_weighted_band": float(BS_NO_TRADE_VALIDATION.sort_values(["date_weighted_mse", "band"]).iloc[0]["band"]),
}])
BS_SELECTION_AGREEMENT["agreement"] = BS_SELECTION_AGREEMENT["episode_weighted_band"].eq(BS_SELECTION_AGREEMENT["date_weighted_band"])
BS_SELECTION_AGREEMENT.to_csv(FINAL_TABLE_DIR / "bs_no_trade_selection_aggregation_audit.csv", index=False)

# Run, year, maturity, path-length and market-role inventory.
if "SEED_LEVEL_RESULTS" in globals() and isinstance(SEED_LEVEL_RESULTS, pd.DataFrame):
    SEED_LEVEL_RESULTS.to_csv(FINAL_TABLE_DIR / "seed_level_results_complete.csv", index=False)

YEAR_POLICY_SUMMARY = enhanced_policy_summary(
    RESULTS.assign(year=pd.to_datetime(RESULTS["entry_date"]).dt.year),
    ["underlying_tc_bps", "year", "policy"],
)
YEAR_POLICY_SUMMARY.to_csv(FINAL_TABLE_DIR / "year_policy_summary.csv", index=False)

inventory_rows = []
for role, episodes in MARKET_BY_ROLE.items():
    for ep in episodes:
        dates = pd.to_datetime(ep.get("observation_times", []), errors="coerce")
        gaps = pd.Series(dates).sort_values().diff().dt.days.dropna() if len(dates) else pd.Series(dtype=float)
        inventory_rows.append({
            "role": role,
            "episode_id": ep.get("episode_id"),
            "entry_date": ep.get("entry_date"),
            "year": pd.Timestamp(ep.get("entry_date")).year,
            "maturity_bucket": ep.get("maturity_bucket"),
            "path_observations": int(len(ep.get("spot", []))),
            "median_calendar_gap_days": float(gaps.median()) if len(gaps) else np.nan,
            "maximum_calendar_gap_days": float(gaps.max()) if len(gaps) else np.nan,
        })
EPISODE_INVENTORY = pd.DataFrame(inventory_rows)
EPISODE_INVENTORY.to_csv(FINAL_TABLE_DIR / "episode_inventory.csv", index=False)
EPISODE_INVENTORY.groupby(["role", "year", "maturity_bucket"], dropna=False, as_index=False).agg(
    episodes=("episode_id", "nunique"),
    entry_dates=("entry_date", "nunique"),
    mean_path_observations=("path_observations", "mean"),
    median_gap_days=("median_calendar_gap_days", "median"),
    maximum_gap_days=("maximum_calendar_gap_days", "max"),
).to_csv(FINAL_TABLE_DIR / "episode_inventory_summary.csv", index=False)

# Feature clipping audit before the transform is clipped at +/-8.
def _scaler_clipping_rows(label: str, episodes, scaler: RobustScaler, package: str) -> list[dict[str, Any]]:
    arrays = []
    for ep in episodes:
        if package == "current":
            arrays.append(np.asarray(ep["features"], float))
        else:
            hist = np.asarray(ep["history"], float)
            mask = np.asarray(ep["history_mask"], bool)
            arrays.append(hist[mask])
    arrays = [a for a in arrays if a.size]
    if not arrays:
        return []
    x = np.concatenate(arrays, axis=0)
    z = (x - scaler.centre) / scaler.scale
    rows = []
    for j, feature in enumerate(scaler.feature_names):
        finite = np.isfinite(z[:, j])
        rows.append({
            "domain": label,
            "package": package,
            "feature": feature,
            "observations": int(finite.sum()),
            "clip_rate": float((np.abs(z[finite, j]) >= scaler.clip).mean()) if finite.any() else np.nan,
            "p99_abs_z": float(np.nanquantile(np.abs(z[finite, j]), 0.99)) if finite.any() else np.nan,
            "max_abs_z": float(np.nanmax(np.abs(z[finite, j]))) if finite.any() else np.nan,
        })
    return rows

clipping_rows = []
for label, episodes in {
    "heston_simulation_train": HESTON_TRAIN,
    "lifted_simulation_train": LIFTED_TRAIN,
    "mixed_simulation_train": MIXED_TRAIN,
    "market_2018_2019": MARKET_TRAIN,
    "market_2020": MARKET_VALIDATION,
    "market_2021_2024": MARKET_TEST,
}.items():
    clipping_rows += _scaler_clipping_rows(label, episodes, CURRENT_SCALER, "current")
    clipping_rows += _scaler_clipping_rows(label, episodes, HISTORY_SCALER, "history")
SCALER_CLIPPING_AUDIT = pd.DataFrame(clipping_rows)
SCALER_CLIPPING_AUDIT.to_csv(FINAL_TABLE_DIR / "feature_scaler_clipping_audit.csv", index=False)

# Decision-level action saturation under the primary cost.
if isinstance(DECISION_LEVEL_RESULTS, pd.DataFrame) and not DECISION_LEVEL_RESULTS.empty:
    neural = DECISION_LEVEL_RESULTS.loc[DECISION_LEVEL_RESULTS["training_seed"].eq(-2)].copy()
    ACTION_BOUND_AUDIT = neural.groupby("policy", as_index=False).agg(
        decisions=("requested_hedge", "size"),
        residual_near_bound=("residual_correction", lambda x: float((np.abs(x) >= 0.99 * TRAIN_CFG.adjustment_limit).mean())),
        hedge_near_bound=("requested_hedge", lambda x: float((np.abs(x) >= 0.99 * TRAIN_CFG.hedge_limit).mean())),
        mean_seed_hedge_sd=("hedge_seed_std", "mean"),
    )
else:
    ACTION_BOUND_AUDIT = pd.DataFrame(columns=["policy", "decisions", "residual_near_bound", "hedge_near_bound", "mean_seed_hedge_sd"])
ACTION_BOUND_AUDIT.to_csv(FINAL_TABLE_DIR / "action_bound_saturation_audit.csv", index=False)

# Heston numerical-pricing route counts from the actual synthetic-bank build.
route_rows = [{"metric": k, "count": int(v)} for k, v in sorted(HESTON_PRICER_ROUTE_COUNTS.items())]
HESTON_PRICER_ROUTE_AUDIT = pd.DataFrame(route_rows)
if not HESTON_PRICER_ROUTE_AUDIT.empty and int(HESTON_PRICER_ROUTE_COUNTS.get("total_calls", 0)) > 0:
    HESTON_PRICER_ROUTE_AUDIT["fraction_of_calls"] = HESTON_PRICER_ROUTE_AUDIT["count"] / int(HESTON_PRICER_ROUTE_COUNTS["total_calls"])
    HESTON_PRICER_ROUTE_AUDIT["audit_status"] = "observed_during_this_bank_build"
else:
    HESTON_PRICER_ROUTE_AUDIT = pd.DataFrame([{
        "metric": "route_counts", "count": np.nan, "fraction_of_calls": np.nan,
        "audit_status": "not_recoverable_from_cached_observable_bank; rebuild banks or run a dedicated state-pricing audit",
    }])
HESTON_PRICER_ROUTE_AUDIT.to_csv(FINAL_TABLE_DIR / "heston_state_pricer_route_audit.csv", index=False)
HESTON_PRICER_GROUP_AUDIT = pd.DataFrame([
    {"route": key[0], "maturity_group": key[1], "moneyness_group": key[2], "count": int(value)}
    for key, value in sorted(HESTON_PRICER_GROUP_COUNTS.items())
])
HESTON_PRICER_GROUP_AUDIT.to_csv(FINAL_TABLE_DIR / "heston_state_pricer_group_audit.csv", index=False)
HESTON_PRICER_FAILURE_AUDIT = pd.DataFrame([{"failure": k, "count": int(v)} for k, v in sorted(HESTON_PRICER_FAILURE_COUNTS.items())])
HESTON_PRICER_FAILURE_AUDIT.to_csv(FINAL_TABLE_DIR / "heston_state_pricer_failure_audit.csv", index=False)

# State exactly what the lower-tail term means at the configured batch size.
TAIL_OBJECTIVE_AUDIT = pd.DataFrame([{
    "batch_episodes": int(TRAIN_CFG.batch_episodes),
    "tail_fraction": float(TRAIN_CFG.cvar_alpha),
    "tail_episode_count_k": int(max(1, math.ceil(TRAIN_CFG.cvar_alpha * TRAIN_CFG.batch_episodes))),
    "tail_weight": float(TRAIN_CFG.cvar_weight),
    "correct_description": "minibatch worst-outcome squared penalty",
    "is_cvar": False,
}])
TAIL_OBJECTIVE_AUDIT.to_csv(FINAL_TABLE_DIR / "training_objective_tail_audit.csv", index=False)

REPORTING_REVISION_MANIFEST = {
    "calibration_spec": RUN.calibration_spec,
    "H": FINAL_CFG.H,
    "protocol_hash": FINAL_PROTOCOL_HASH,
    "model_selection_weighting": MODEL_SELECTION_WEIGHTING,
    "configured_seed_count": len(FINAL_CFG.training_seeds),
    "checkpoint_selection_audit": str(FINAL_TABLE_DIR / "checkpoint_selection_aggregation_audit.csv"),
    "no_trade_selection_audit": str(FINAL_TABLE_DIR / "bs_no_trade_selection_aggregation_audit.csv"),
    "seed_level_results": str(FINAL_TABLE_DIR / "seed_level_results_complete.csv"),
    "year_results": str(FINAL_TABLE_DIR / "year_policy_summary.csv"),
    "episode_inventory": str(FINAL_TABLE_DIR / "episode_inventory_summary.csv"),
    "feature_clipping": str(FINAL_TABLE_DIR / "feature_scaler_clipping_audit.csv"),
    "action_saturation": str(FINAL_TABLE_DIR / "action_bound_saturation_audit.csv"),
    "heston_pricer_routes": str(FINAL_TABLE_DIR / "heston_state_pricer_route_audit.csv"),
    "tail_objective": str(FINAL_TABLE_DIR / "training_objective_tail_audit.csv"),
}
(FINAL_RESULTS_DIR / "reporting_revision_manifest.json").write_text(json.dumps(REPORTING_REVISION_MANIFEST, indent=2, sort_keys=True))

print("Final reporting audits written.")
display(CHECKPOINT_SELECTION_AGGREGATION_AUDIT.head(18))
display(BS_SELECTION_AGREEMENT)
display(TAIL_OBJECTIVE_AUDIT)
display(ACTION_BOUND_AUDIT)
display(HESTON_PRICER_ROUTE_AUDIT)


Final reporting audits written.


,policy,training_seed,candidate_stage,episode_weighted_mse,date_weighted_mse,selected_stage_in_checkpoint,selected_under_checkpoint_rule,selection_weighting,audit_status,episode_rank,date_rank
0,heston_hybrid_current_small,20260710,simulation_pretrained,NaN,NaN,head_only,False,episode_weighted_cached_checkpoint,date_weighted_candidate_state_not_stored_in_ca...,NaN,NaN
1,heston_hybrid_current_small,20260710,head_only,0.161384,NaN,head_only,True,episode_weighted_cached_checkpoint,date_weighted_candidate_state_not_stored_in_ca...,1.0,NaN
2,heston_hybrid_current_small,20260710,full,NaN,NaN,head_only,False,episode_weighted_cached_checkpoint,date_weighted_candidate_state_not_stored_in_ca...,NaN,NaN
3,heston_hybrid_current_small,20260711,simulation_pretrained,0.154924,NaN,simulation_pretrained,True,episode_weighted_cached_checkpoint,date_weighted_candidate_state_not_stored_in_ca...,1.0,NaN
4,heston_hybrid_current_small,20260711,head_only,NaN,NaN,simulation_pretrained,False,episode_weighted_cached_checkpoint,date_weighted_candidate_state_not_stored_in_ca...,NaN,NaN
5,heston_hybrid_current_small,20260711,full,NaN,NaN,simulation_pretrained,False,episode_weighted_cached_checkpoint,date_weighted_candidate_state_not_stored_in_ca...,NaN,NaN
6,heston_hybrid_current_small,20260712,simulation_pretrained,NaN,NaN,full,False,episode_weighted_cached_checkpoint,date_weighted_candidate_state_not_stored_in_ca...,NaN,NaN
7,heston_hybrid_current_small,20260712,head_only,NaN,NaN,full,False,episode_weighted_cached_checkpoint,date_weighted_candidate_state_not_stored_in_ca...,NaN,NaN
8,heston_hybrid_current_small,20260712,full,0.156939,NaN,full,True,episode_weighted_cached_checkpoint,date_weighted_candidate_state_not_stored_in_ca...,1.0,NaN
9,lifted_hybrid_current_small,20260710,simulation_pretrained,NaN,NaN,head_only,False,episode_weighted_cached_checkpoint,date_weighted_candidate_state_not_stored_in_ca...,NaN,NaN


,selected_reported_band,episode_weighted_band,date_weighted_band,agreement
0,0.25,0.25,0.25,True


,batch_episodes,tail_fraction,tail_episode_count_k,tail_weight,correct_description,is_cvar
0,16,0.05,1,0.05,minibatch worst-outcome squared penalty,False


,policy,decisions,residual_near_bound,hedge_near_bound,mean_seed_hedge_sd
0,heston_hybrid_current_small,23832,0.0,0.0,0.045706
1,lifted_hybrid_current_small,23832,0.0,0.0,0.030407
2,mixed_hybrid_current_matched,23832,0.0,0.0,0.027096
3,mixed_hybrid_current_small,23832,0.0,0.0,0.038670
4,mixed_hybrid_gru,23832,0.0,0.0,0.030557
5,mixed_hybrid_lag,23832,0.0,0.0,0.032008


,metric,count,fraction_of_calls,audit_status
0,route_counts,NaN,NaN,not_recoverable_from_cached_observable_bank; r...


# Interpretation contract

Dissertation claims are populated only after the relevant calibration mode passes its evidence gates and the claim-evidence ledger is generated from verified exports.

- Lifted Heston versus Heston estimates the effect of choosing one complete calibrated and numerically implemented source pipeline rather than the other.
- Fixed-size mixed versus Heston estimates a source-composition effect at equal total training support.
- Parameter-matched current versus small current estimates additional model capacity without explicit raw history.
- Lagged-input MLP versus parameter-matched current estimates the incremental supplied ten-observation sequence conditional on approximate capacity.
- GRU versus lagged-input MLP estimates recurrent versus flattened processing conditional on identical information.
- GRU versus parameter-matched current is an end-to-end sequence-policy comparison, not a pure recurrence effect.
- The preferred neural policy is selected only using 2020 validation before comparison with Black-Scholes delta.

Use only **supported**, **suggestive but multiplicity-unresolved**, **inconclusive**, and **not tested**. A null result does not establish equivalence, observable-state sufficiency, market Markovianity or absence of path dependence. A source result does not establish general Heston or Lifted-Heston superiority. A recurrent result does not establish recovery of latent factors.

The H = 0.35 and H = 0.25 documents are separate empirical specifications. Their numerical difference is not interpreted as the isolated causal effect of changing H when the seed panel, selection rule, calibration or downstream artefacts also differ.